# GoA Run #15 / Run-21 (v18.0.0) — OBJECT VISION (core audit fixes 1-4)
Deep-mode trigger + IL v1.6 + Transport v26.3.4 + dual-server + Object Vision

**Certified:** my_agent v1.0.21 (Object Vision) | CM v1.6 | artifact adb470eae95e94d557fd110420676096
**Core fixes:** (1) OBJECTS block — model ko seedha id/color/rows/cols dikhta
hai, RLE column-arithmetic khatam; (2) extract_objects data ab prompt mein
jaata hai (pehle compute karke phenk dete the); (3) coordinate targeting
direct; (4) _obs_fallback dual-format fix (grid_text "W8" RLE pehle
<malformed> deta tha). Obs advisory cap 1500->2500 (noise fix).
**Budget:** 17 min/game, 7.5h global (25x17=7.08h + infra < 7.5h; Kaggle 9h)
**Cells:** 36 (coder-conformance runs BEFORE THE RUN — FIX-V)


In [ ]:
# PREFLIGHT v8 (FINAL): paths, GPU(96GB hard), host RAM, parquet, disk — v8: VRAM GiB unit fix (external audit round 2)
# Locked facts (Chief-confirmed):
#   - RTX PRO 6000 = 96 GB GDDR7 hard limit → VRAM < 90 GB = FAIL
#   - Offline eval, single GPU, host RAM ~30-60 GB
#   - Python 3.12 mandatory, output = submission.parquet
#   - ARC framework import test is DEFERRED to Cell 2 (post-pip-install)
import os, sys, shutil, subprocess, json, glob
import importlib, importlib.metadata

# ---------- helpers ----------
def check(label, path, min_entries=1):
    if not os.path.isdir(path):
        print(f"MISS {label}: {path}")
        return False
    try:
        entries = os.listdir(path)
    except Exception as e:
        print(f"ERR  {label}: {path} — {e}")
        return False
    ok = len(entries) >= min_entries
    print(("OK   " if ok else "MISS ") + f"{label}: {path} ({len(entries)} entries)")
    return ok

def _norm(v):
    return str(v).split("+")[0] if v else None

def _mod_version(modname):
    try:
        return importlib.metadata.version(modname), "metadata"
    except Exception:
        try:
            m = importlib.import_module(modname)
            v = getattr(m, "__version__", None)
            return (v, "attribute") if v else (None, None)
        except Exception:
            return None, None

# ---------- /kaggle/input discovery ----------
print("=== /kaggle/input ===")
try:
    for p in sorted(os.listdir("/kaggle/input")):
        print(" ", p)
        full = "/kaggle/input/" + p
        if os.path.isdir(full):
            try:
                for sub in sorted(os.listdir(full))[:5]:
                    print("    └─", sub)
            except PermissionError:
                print("    └─ <permission denied>")
except Exception as e:
    print("  ERR listing /kaggle/input:", e)

# ---------- Python version ----------
print(f"INFO python: {sys.version.split()[0]} | exec: {sys.executable}")
if sys.version_info[:2] != (3, 12):
    print(f"WARN Python 3.12 expected, mila {sys.version.split()[0]} — ARC runtime check karo")

# ---------- Inputs ----------
REASONER_PATH = "/kaggle/input/datasets/gray2488/qwen38-27b-fp8"
CODER_PATH    = "/kaggle/input/datasets/gray2488/qwen3-coder-30b-a3b-instruct"
COMP          = "/kaggle/input/competitions/arc-prize-2026-arc-agi-3"
FRAMEWORK_DIR = COMP + "/ARC-AGI-3-Agents"
WHEELS        = COMP + "/arc_agi_3_wheels"
ENVFILES      = COMP + "/environment_files"

checks = [
    check("reasoner weights", REASONER_PATH),
    check("coder weights",    CODER_PATH),
    check("framework",        FRAMEWORK_DIR),
    check("wheels",           WHEELS),
    check("env files",        ENVFILES),
]
# BUG-1 FIX: hard assert wapas — inputs ke bina aage jana = 9-hour session waste
assert all(checks), "Input missing — Add Input panel check karo (upar /kaggle/input listing dekho)"

# ---------- Working dir writability ----------
if os.access("/kaggle/working", os.W_OK):
    print("OK   /kaggle/working writable")
else:
    print("FAIL /kaggle/working not writable — submission save nahi hoga")

SUBMISSION_PATH = "/kaggle/working/submission.parquet"
print(f"INFO submission target: {SUBMISSION_PATH}")

# ---------- nvidia-smi ----------
out = None
try:
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"],
        capture_output=True, text=True, check=False,
    )
    if out.returncode == 0 and out.stdout.strip():
        print(f"OK   nvidia-smi: {out.stdout.strip()}")
    else:
        print(f"WARN nvidia-smi exit={out.returncode} | "
              f"stdout={out.stdout.strip()!r} | stderr={out.stderr.strip()!r}")
except FileNotFoundError:
    print("WARN nvidia-smi not found — CPU-only environment?")
except Exception as e:
    print(f"WARN nvidia-smi failed: {e}")

# ---------- Host RAM ----------
ram_gb = None
try:
    with open("/proc/meminfo") as f:
        for line in f:
            if line.startswith("MemTotal:"):
                ram_gb = int(line.split()[1]) / 1e6
                print(f"INFO host RAM: {ram_gb:.1f} GB")
                if ram_gb < 60:
                    print(f"WARN host RAM {ram_gb:.1f} GB < 60 — direct safetensors load "
                          f"fail ho sakta hai. low_cpu_mem_usage=True + device_map='auto' use karo")
                else:
                    print("OK   host RAM sufficient for streaming safetensors load")
                break
except Exception as e:
    print(f"WARN host RAM check failed: {e}")

# ---------- torch + GPU (96 GB hard target) ----------
vram_gb = None
gpu_name = None
try:
    import torch as torch_mod  # noqa
    if torch_mod.cuda.is_available():
        props = torch_mod.cuda.get_device_properties(0)
        vram_gb = props.total_memory / (1024 ** 3)   # GiB (v8: /1e9 decimal tha -> 102.7, locked-fact 90-100 GiB range fail karta tha; 97887 MiB = 95.6 GiB)
        gpu_name = props.name
        print(f"OK   torch {torch_mod.__version__} | {gpu_name} | {vram_gb:.0f} GB")

        # Chief-confirmed: RTX PRO 6000 = 96 GB GDDR7 hard limit
        if vram_gb < 90:
            print(f"FAIL VRAM {vram_gb:.0f} GB < 90 — RTX PRO 6000 (96 GB) expect tha. "
                  f"Actual: {gpu_name}. Accelerator='rtx6000' select karo")
        elif vram_gb > 100:
            print(f"WARN VRAM {vram_gb:.0f} GB > 100 — unexpected GPU: {gpu_name}")
        else:
            # COSMETIC FIX: 27B FP8 (~27GB) + 30B MoE FP8 (~31GB) = ~58 GB baseline
            print(f"OK   VRAM {vram_gb:.0f} GB — RTX PRO 6000 confirmed "
                  f"(~{vram_gb - 58:.0f} GB headroom after 27B FP8 + 30B MoE)")

        if "RTX" in gpu_name and "6000" in gpu_name:
            print(f"OK   GPU name matches RTX PRO 6000: {gpu_name}")
        else:
            print(f"WARN GPU name '{gpu_name}' — RTX PRO 6000 expect tha")

        # nvidia-smi vs torch cross-check
        if out is not None and out.returncode == 0:
            try:
                smi_mem = out.stdout.split(",")[1].strip()
                smi_mb = int("".join(c for c in smi_mem if c.isdigit()))
                smi_gb = smi_mb / 1024
                if abs(smi_gb - vram_gb) > 5:
                    print(f"WARN nvidia-smi VRAM {smi_gb:.1f} GB vs torch VRAM "
                          f"{vram_gb:.1f} GB mismatch (>5 GB)")
            except Exception:
                pass
    else:
        print(f"WARN torch {torch_mod.__version__} | CUDA unavailable — GPU enable karo")

    # ---------- wheels manifest match ----------
    _m = (glob.glob("/kaggle/input/datasets/*/goa-vllm-wheels/manifest.json")
          or glob.glob("/kaggle/input/datasets/*/goa-vllm-wheels/wheels_manifest.json")
          or glob.glob("/kaggle/input/models/*/goa-vllm-wheels/manifest.json")
          or glob.glob("/kaggle/input/*/goa-vllm-wheels/manifest.json"))
    if _m:
        with open(_m[0]) as f:
            _manifest = json.load(f)
        _wt = _manifest.get("torch_version")
        if _wt is None:
            print("WARN manifest mila par 'torch_version' key nahi")
        elif not isinstance(_wt, str):
            print(f"WARN manifest torch_version non-string: {type(_wt).__name__}")
        elif _norm(_wt) == _norm(torch_mod.__version__):
            print(f"OK   wheels torch {_wt} MATCH (normalized: {_norm(_wt)})")
        else:
            print(f"WARN wheels torch {_wt} != image torch {torch_mod.__version__} — LEAN risk")
    else:
        print("WARN wheels manifest nahi mila — torch check skip")
except Exception as e:
    print(f"WARN torch/GPU check skipped: {e}")

# ---------- vLLM / httpx ----------
# COSMETIC FIX: 'v == "?"' dead branch hataya (importlib.metadata "?" nahi return karta)
for mod in ("vllm", "httpx"):
    v, src = _mod_version(mod)
    if v:
        print(f"OK   {mod} {v} (via {src})")
    else:
        print(f"WARN {mod} missing — bootstrap cell se install hoga")

# BUG-2 FIX: ARC framework import test REMOVED from preflight.
# Reasons:
#   (a) arc-agi ka pip install cell 2 mein hota hai — preflight us se pehle chalta hai
#   (b) fallback 'import agents' unrelated pip package ko pick kar sakta hai (false positive)
#   (c) FRAMEWORK_DIR se import → sys.modules cache → cell 2 ka pip-installed version SHADOW
# Preflight sirf structure verify karta hai; actual import test Cell 2 (post-install) mein.
if os.path.isdir(FRAMEWORK_DIR):
    try:
        _fw_entries = sorted(os.listdir(FRAMEWORK_DIR))
        print(f"OK   FRAMEWORK_DIR structure: {_fw_entries[:10]}"
              + (" ..." if len(_fw_entries) > 10 else ""))
    except Exception as e:
        print(f"WARN FRAMEWORK_DIR listing failed: {e}")
else:
    print(f"FAIL FRAMEWORK_DIR exist nahi karta: {FRAMEWORK_DIR}")

# BUG-3 FIX: env vars section — sirf MAX_ACTIONS rakha (verified = 80).
# GAME_TIME_LIMIT_S / MESSAGE_LIMIT expected values unverified the — hata diye.
_ma = os.environ.get("MAX_ACTIONS")
if _ma is None:
    print("INFO env MAX_ACTIONS: <unset> — framework default 80 use hoga")
else:
    try:
        print(f"INFO env MAX_ACTIONS: {int(_ma)} (framework cap = 80)")
    except ValueError:
        print(f"WARN env MAX_ACTIONS: '{_ma}' non-numeric — int() fail hoga")

# ---------- Parquet deps + roundtrip ----------
_parquet_deps_ok = True
for mod in ("pandas", "pyarrow"):
    v, src = _mod_version(mod)
    if v:
        print(f"OK   {mod} {v}")
    else:
        print(f"WARN {mod} missing — bootstrap cell install karega (submission.parquet ke liye)")
        _parquet_deps_ok = False

_parquet_roundtrip_ok = False
if _parquet_deps_ok:
    _test_path = "/kaggle/working/.preflight_write_test.parquet"
    try:
        import pandas as pd
        _test_df = pd.DataFrame({"a": [1, 2, 3], "b": ["x", "y", "z"]})
        _test_df.to_parquet(_test_path, index=False)
        _back = pd.read_parquet(_test_path)
        assert _back.shape == (3, 2), f"roundtrip shape mismatch: {_back.shape}"
        _parquet_roundtrip_ok = True
        print("OK   parquet write+read roundtrip PASSED")
    except Exception as e:
        print(f"FAIL parquet write test: {e}")
    finally:
        try:
            if os.path.exists(_test_path):
                os.remove(_test_path)
        except Exception:
            pass

# ---------- Disk ----------
try:
    du = shutil.disk_usage("/kaggle/working")
    print(f"OK   disk /kaggle/working: free {du.free/1e9:.1f} GB / total {du.total/1e9:.1f} GB")
except Exception as e:
    print(f"WARN disk check failed: {e}")

# ---------- Final summary ----------
# BUG-1 FIX: 'Inputs attached' fact add; COSMETIC FIX: Host RAM check ab >= 60
print("\n=== LOCKED-FACTS COMPLIANCE ===")
_facts = [
    ("Inputs attached",          all(checks)),
    ("Python 3.12",              sys.version_info[:2] == (3, 12)),
    ("Working writable",         os.access("/kaggle/working", os.W_OK)),
    ("Parquet deps",             _parquet_deps_ok),
    ("Parquet roundtrip",        _parquet_roundtrip_ok),
    ("GPU = RTX PRO 6000 96GB",  vram_gb is not None and 90 <= vram_gb <= 100),
    ("Host RAM >= 60 GB",        ram_gb is not None and ram_gb >= 60),
]
for name, ok in _facts:
    print(f"  {'PASS' if ok else 'FAIL'}  {name}")

# ARC framework reachability + symbols ab Cell 2 verify karega (post pip-install)
print("INFO ARC framework/symbol import test Cell 2 mein hoga (post-install)")

In [ ]:
# Cell 2 v4: Offline installs (verified spec) + games cache + env vars
# Verified spec (DIR/067 W-4): vllm + httpx (bundle) + arc-agi
import subprocess, os, shutil, sys, glob, importlib

# ---------- Offline env vars ----------
# vLLM subprocesses tak propagate honge — real improvement
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("HF_DATASETS_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
print("OK   offline env vars set")

# ---------- Resolve ALL wheels sources ----------
WHEELS_SOURCES = []
if 'WHEELS' in globals() and os.path.isdir(WHEELS):
    WHEELS_SOURCES.append(WHEELS)
WHEELS_SOURCES += glob.glob("/kaggle/input/datasets/*/goa-vllm-wheels/wheels")
WHEELS_SOURCES += glob.glob("/kaggle/input/datasets/*/goa-vllm-wheels")
WHEELS_SOURCES += glob.glob("/kaggle/input/goa-vllm-wheels/wheels")
WHEELS_SOURCES += glob.glob("/kaggle/input/*/goa-vllm-wheels/wheels")
WHEELS_SOURCES = list(dict.fromkeys(WHEELS_SOURCES))  # dedupe, preserve order
WHEELS_SOURCES = [d for d in WHEELS_SOURCES if os.path.isdir(d)]
print(f"OK   wheel sources ({len(WHEELS_SOURCES)}): {WHEELS_SOURCES}")
if not WHEELS_SOURCES:
    raise RuntimeError("Koi wheels source nahi mila")

# Inventory: debug aid
_all_wheels = []
for d in WHEELS_SOURCES:
    _all_wheels += glob.glob(os.path.join(d, "*.whl"))
print(f"INFO total wheel files: {len(_all_wheels)}")

# ---------- BUG-D FIX: verified spec only ----------
# Verified: vllm + httpx (bundle) + arc-agi + python-dotenv (bundle).
# transformers/safetensors/tokenizers/accelerate vllm ke transitive deps hain — auto aayenge.
# 'arcengine' pip se framework ke engine ko shadow karega — REMOVED.
# 'flask', 'verifiers' bundle mein nahi hain → bulk-FAIL trigger karte the — REMOVED.
PACKAGES_WANTED = ["vllm", "httpx", "arc-agi", "python-dotenv"]

_find_links_args = []
for d in WHEELS_SOURCES:
    _find_links_args += ["--find-links", d]

# ---------- Bulk install ----------
def _pip_install(pkgs):
    return subprocess.run(
        [sys.executable, "-m", "pip", "install",
         "--no-index", *_find_links_args, *pkgs],
        capture_output=True, text=True, check=False,
    )

print(f"INFO installing {len(PACKAGES_WANTED)} packages: {PACKAGES_WANTED}")
r = _pip_install(PACKAGES_WANTED)

if r.returncode == 0:
    print("OK   bulk install complete")
else:
    print(f"WARN bulk install exit={r.returncode} — per-package fallback chalu")
    print("  STDOUT tail:", r.stdout[-800:])
    print("  STDERR tail:", r.stderr[-800:])
    _ok, _fail = [], []
    for pkg in PACKAGES_WANTED:
        rr = _pip_install([pkg])
        if rr.returncode == 0:
            _ok.append(pkg)
            print(f"  OK   {pkg}")
        else:
            _fail.append(pkg)
            _err = rr.stderr.strip().splitlines()[-1] if rr.stderr.strip() else "?"
            print(f"  FAIL {pkg} — {_err}")
    print(f"\nSummary: {len(_ok)} installed, {len(_fail)} failed")
    if _fail:
        print(f"WARN ye packages install nahi hue: {_fail}")

# ---------- BUG-C FIX: vllm kernel import avoid ----------
# vllm kernel import libcudart error se FAIL hota hai (verified).
# Subprocess check Cell 3 (bootstrap v5) mein authoritative hai — wahan verify hoga.
_verify_required = ["pandas", "pyarrow", "httpx"]  # submission + HTTP — must-have
_failed_required = []
for mod in _verify_required:
    try:
        _m = importlib.import_module(mod)
        print(f"OK   {mod} {getattr(_m, '__version__', '?')}")
    except ImportError as e:
        print(f"FAIL {mod} (required): {e}")
        _failed_required.append(mod)
if _failed_required:
    raise RuntimeError(f"Required imports fail: {_failed_required}")

# vllm installed hai ya nahi — pip metadata se check (kernel import nahi)
try:
    _vllm_v = importlib.metadata.version("vllm")
    print(f"OK   vllm {_vllm_v} (metadata only — kernel import Cell 3 subprocess mein)")
except Exception:
    print("WARN vllm metadata nahi mila — bootstrap cell check karega")

# ---------- BUG-B FIX: sirf 'arc_agi' import (koi fallback nahi) ----------
# Verified: import name poori chain mein sirf 'arc_agi' hai.
# Fallbacks ('agents', 'arc_agi_agents') unrelated pip packages ko pick kar sakte the
# → false-positive "OK ARC package". Wahi risk jo preflight v6 BUG-2b tha.
try:
    _arc_module = importlib.import_module("arc_agi")
    _v = getattr(_arc_module, "__version__", None)
    print(f"OK   arc_agi @ {getattr(_arc_module, '__file__', '?')}"
          + (f" (v{_v})" if _v else ""))
except ImportError as e:
    # Fallback: FRAMEWORK_DIR ko sys.path mein daal ke try karo (framework source path)
    _fw = globals().get("FRAMEWORK_DIR")
    if _fw and os.path.isdir(_fw):
        if _fw not in sys.path:
            sys.path.insert(0, _fw)
        try:
            _arc_module = importlib.import_module("arc_agi")
            print(f"OK   arc_agi @ {getattr(_arc_module, '__file__', '?')} (via FRAMEWORK_DIR)")
        except ImportError as e2:
            raise RuntimeError(f"arc_agi import fail (pip + framework dir dono se): {e2}")
    else:
        raise RuntimeError(f"arc_agi import fail: {e}")

# ---------- Games cache ----------
# BUG-A FIX: hard gate hata diya. Asli verified gate run cell ka `assert len(game_ids) == 25` hai.
# Public set = 25 games (verified). environment_files ka file-count games se alag metric hai
# (per-game multiple files) — koi verified number nahi. Isliye yahan sirf WARN/INFO.
CACHE_DIR = "/kaggle/working/environment_files"

if os.path.exists(CACHE_DIR):
    try:
        _n = len(os.listdir(CACHE_DIR))
    except Exception:
        _n = 0
    # Incomplete-cache re-copy logic rakha — bas hard number-gate nahi
    if _n == 0:
        print(f"WARN cache exists but empty (0 entries) — re-copy")
        shutil.rmtree(CACHE_DIR, ignore_errors=True)
        shutil.copytree(ENVFILES, CACHE_DIR)
    else:
        print(f"OK   cache exists ({_n} entries)")
else:
    print(f"INFO copying env files → {CACHE_DIR} (may take minutes)...")
    shutil.copytree(ENVFILES, CACHE_DIR)

_n = len(os.listdir(CACHE_DIR))
if _n == 0:
    raise RuntimeError(f"games cache empty after copy — ENVFILES check karo: {ENVFILES}")
print(f"OK   games cached: {_n} entries (game-count verify run cell karega: assert == 25)")

_free_gb = shutil.disk_usage("/kaggle/working").free / 1e9
print(f"{'WARN' if _free_gb < 5 else 'OK'}   disk free after copy: {_free_gb:.1f} GB")

# ---------- Working dirs ----------
for d in ("/kaggle/working/goa/agent", "/kaggle/working/logs"):
    try:
        os.makedirs(d, exist_ok=True)
        print(f"OK   mkdir {d}")
    except Exception as e:
        print(f"FAIL mkdir {d}: {e}")
        raise

print("OK   Cell 2 complete — ready for vLLM bootstrap")

In [ ]:
# vLLM BOOTSTRAP v7: skip-if-installed + exact vllm wheel match + LD_LIBRARY_PATH propagated

import glob, subprocess, sys, os, shutil
import importlib.metadata as _im

_free = shutil.disk_usage("/").free / 1e9
if _free < 5:
    raise SystemExit(f"Disk free {_free:.1f} GB < 5 GB — vLLM install fail ho sakta hai")
print(f"OK   disk free: {_free:.1f} GB")

_vllm_preinstalled = None
try:
    _vllm_preinstalled = _im.version("vllm")
    print(f"OK   vllm {_vllm_preinstalled} already installed — install skip")
except Exception:
    print("INFO vllm not installed yet — wheels se install hoga")

_cands = []
if 'WHEELS_SOURCES' in globals():
    _cands += list(WHEELS_SOURCES)
_cands += glob.glob("/kaggle/input/datasets/*/goa-vllm-wheels/wheels")
_cands += glob.glob("/kaggle/input/datasets/*/goa-vllm-wheels")
_cands += glob.glob("/kaggle/input/datasets/*/wheels/wheels")
_cands += glob.glob("/kaggle/input/goa-vllm-wheels/wheels")
_cands += glob.glob("/kaggle/input/*/goa-vllm-wheels/wheels")
_cands = list(dict.fromkeys(_cands))

_valid = []
for d in _cands:
    if not os.path.isdir(d):
        continue
    try:
        whls = glob.glob(os.path.join(d, "*.whl"))
        vllm_whls = [w for w in whls if os.path.basename(w).lower().startswith("vllm-")]
        if vllm_whls:
            _valid.append((d, len(whls), len(vllm_whls), vllm_whls))
    except Exception:
        pass

if _vllm_preinstalled is None:
    if _valid:
        _valid.sort(key=lambda x: (-x[2], -x[1]))
        W = _valid[0][0]
        _vllm_wheel = os.path.basename(sorted(_valid[0][3])[0])
        print(f"OK   wheels: {W} ({_valid[0][1]} total, {_valid[0][2]} vllm)")
        print(f"     picked: {_vllm_wheel}")

        _vllm_ver = None
        _parts = _vllm_wheel.split("-")
        if len(_parts) >= 2 and _parts[0].lower() == "vllm":
            _vllm_ver = _parts[1]
        _pin = f"vllm=={_vllm_ver}" if _vllm_ver else "vllm"
        print(f"INFO pip install {_pin} httpx (LIVE, 2-5 min):", flush=True)

        rc = subprocess.run(
            [sys.executable, "-m", "pip", "install", "--no-index",
             "--find-links", W, _pin, "httpx"],
        ).returncode

        if rc != 0 and _vllm_ver:
            print(f"WARN pinned install fail (rc={rc}) — unpinned retry:", flush=True)
            rc = subprocess.run(
                [sys.executable, "-m", "pip", "install", "--no-index",
                 "--find-links", W, "vllm", "httpx"],
            ).returncode

        if rc != 0:
            raise SystemExit(
                "pip install FAIL — torch mismatch (S-3)? ya wheels incomplete. "
                f"Wheels dir: {W}"
            )
        print("OK   pip install complete")
    else:
        print("WARN wheels dataset nahi mila (koi candidate mein 'vllm-*.whl' nahi tha)")
        print(f"     checked dirs: {_cands[:5]}{' ...' if len(_cands) > 5 else ''}")
        print("     preinstalled vLLM pe bharosa — missing ho to subprocess import fail karega")

try:
    _hv = _im.version("httpx")
    print(f"OK   httpx {_hv}")
except Exception as e:
    raise SystemExit(f"httpx missing — wheels dataset attach karo: {e}")

import sysconfig, site
_site_dirs = set()
for _p in ("purelib", "platlib"):
    _d = sysconfig.get_paths().get(_p)
    if _d and os.path.isdir(_d):
        _site_dirs.add(_d)
try:
    for _d in site.getsitepackages():
        if _d and os.path.isdir(_d):
            _site_dirs.add(_d)
except Exception:
    pass

_libs = []
for base in _site_dirs:
    _libs += glob.glob(f"{base}/nvidia/*/lib")
    _libs += glob.glob(f"{base}/cuda/*/lib")
_libs += glob.glob("/usr/local/cuda*/lib64")
_libs += glob.glob("/usr/local/cuda*/lib")
_libs = list(dict.fromkeys(_libs))

print(f"INFO site dirs: {sorted(_site_dirs)}")
print(f"INFO nvidia/cuda lib dirs found: {len(_libs)}")
if not _libs:
    print("WARN koi nvidia/cuda lib dir nahi mila — subprocess import fail ho sakta hai")

_existing_ld = os.environ.get("LD_LIBRARY_PATH", "")
_new_ld = ":".join(_libs) + (":" + _existing_ld if _existing_ld else "")
os.environ["LD_LIBRARY_PATH"] = _new_ld
_env = dict(os.environ)

_r = subprocess.run(
    [sys.executable, "-c", "import vllm; print(vllm.__version__)"],
    env=_env, capture_output=True, text=True,
)
if _r.returncode == 0:
    print(f"OK   vLLM ready (subprocess): {_r.stdout.strip()}")
else:
    print("FAIL vllm subprocess import")
    print("  LD_LIBRARY_PATH:", _new_ld[:500] + ("..." if len(_new_ld) > 500 else ""))
    print("  STDOUT:", _r.stdout.strip() or "(empty)")
    print("  STDERR tail:", _r.stderr.strip()[-1200:] if _r.stderr.strip() else "(empty)")
    raise SystemExit("vllm subprocess import FAIL — upar ka error dekho")

print("OK   BOOTSTRAP complete: vllm (subprocess) + httpx ready")


In [ ]:
%%writefile goa/agent/context_manager.py
"""agent/context_manager.py — 3-Zone Context Manager + Grid Diff Encoder. v1.6.

v1.6 — Martial-law core audit (Chief order, 2026-09-30): (1) OBJECTS
        rendering — set_observation(objects=...) structured object list
        (id/color/size/bbox) observation mein render karta hai, taaki model
        ko RLE rows se column-arithmetic karke target nikalna na pade.
        (2) _obs_fallback dual-format — grid_text ka "char+count" RLE
        ("W8") pehle NONZERO_CELLS=<malformed> deta tha; ab dono formats
        chalete hain. + T-29/T-30.

INTEGRATED BUILD — module code plus an inline self-test suite under
``if __name__ == "__main__":``.  Import this file as a normal module; run
it directly to execute the full v1.4 test battery.

Pure data-structure module.  No LLM calls, no network, no I/O, no threads,
no globals mutated across instances.  Stdlib only.

Design (per DIR/2026-028, research facts F-1..F-7): full grids are too
big for the window (F-1) so diffs are used; static-first ordering (F-2,
F-4); ~2400-token working target is a quality decision (F-3); Zone 2 is
written by CODER output routed through the harness, the Reasoner never
writes it (F-5); warn at 70%, flush at 100% (F-6); silent truncation is
the worst failure mode (F-7) — fail loud.

Per-zone token caps (``zone1_max_tokens``, ``zone2_max_tokens``,
``zone3_max_tokens``, ``observation_max_tokens``) are ADVISORY: exceeding
them logs a WARNING at build time but does not raise.  The sole hard
enforcement gate is ``total_hard_cap``, which runs the eviction cascade
and raises ``ContextOverflowError`` if the packet cannot be reduced.
Zone 2 additionally enforces four per-field sub-caps via
``_ZONE2_FIELD_CAPS`` (see ``update_zone2``).

JSON snapshot encoding
----------------------
``snapshot()`` returns a dict that ``json.dumps`` can serialise.  Values
that JSON cannot represent natively are wrapped in single-key marker
dicts (``__set__``, ``__frozenset__``, ``__bytes__``, ``__tuple__``).
``load_snapshot()`` inverts these via ``_json_restore()`` so sets,
frozensets, bytes, and tuples round-trip.  Reserved marker keys: a user
dict with exactly one key equal to one of those markers will be
misinterpreted.  No such dict is expected in the ARC-AGI-3
action/observation domain.

Changelog
---------
v1.0  Initial (DIR/2026-028).
v1.1  Post-audit fixes (F-01..F-06 first independent audit).
v1.2  Post-v1.1 audit fixes (F-01..F-05 v1.1 audit).
v1.3  Post-v1.2 audit fixes (F-01..F-03 v1.2 audit) + inline test suite.
        F-01  _rle_decode_rows validates 0 <= value <= GRID_MAX_VALUE.
        F-02  apply_delta verifies ch["old"] against current cell.
        F-03  Per-zone advisory caps emit WARNINGs at build time.
        INT   Self-test suite integrated under __main__.
v1.5  Per DIR/2026-068 (Memory & Coder Awakening, Chief directive):
        Field caps raised after Run #13 evidence — reasoner ~250-450
        tokens likhta tha, sirf 80 yaad rehte the (memory starvation,
        81-action blind reflex). confirmed_rules 120->200,
        hypotheses 80->350, world_model_status 60->200,
        goal_notes 40->100. Zone budgets / total_hard_cap module
        defaults UNCHANGED (runtime injection ke through badhte hain,
        T-suite determinism ke liye). T-3 version-proof ban gaya
        (cap constant se derive hota hai). Server budget: actual
        context_window 16384; truncation ab bhi advisory-warning only.
v1.4  Per DIR/2026-030:
        T-19 tuple roundtrip: __tuple__ marker added; contract and test
        aligned.  Previously tuples were serialised to lists and not
        restored, so an (x, y) tuple in available_actions rendered as
        [x, y] after a mid-run snapshot/restore — the Reasoner saw a
        different prompt than it had before.  With the marker, tuple
        round-trip is byte-stable.  Reserved marker count is now four.
"""
from __future__ import annotations

import hashlib
import json
import logging
import random
from collections import deque
from dataclasses import dataclass
from typing import Any, Callable, Deque, Dict, List, Optional, Tuple

logger = logging.getLogger(__name__)

GRID_SIZE = 64
GRID_MAX_VALUE = 15
DEFAULT_REFRESH_EVERY = 10
MAX_DELTA_LIST = 400
_MAX_SUMMARY_LINES = 20
_MAX_DEAD_SIGS_PER_LEVEL = 200
__version__ = "1.6"
_MAX_FALLBACK_CELLS = 500
# v1.6: OBJECTS rendering cap (observation budget protection)
_MAX_OBS_OBJECTS = 24
# v1.6: grid_text "char+count" RLE decode table — PALETTE order (digits 0..15).
# my_agent T-suite isko grid_text.CHAR_TO_DIGIT ke against cross-check
# karta hai (integration drift guard).
_FALLBACK_PALETTE = "WwgGcBMPRbSYOrNp"
_FALLBACK_CHAR_TO_DIGIT = {c: i for i, c in enumerate(_FALLBACK_PALETTE)}

_ZONE2_FIELD_CAPS: Dict[str, int] = {
    "confirmed_rules": 200,
    "hypotheses": 350,
    "world_model_status": 200,
    "goal_notes": 100,
}
_ZONE2_FIELDS: Tuple[str, ...] = tuple(_ZONE2_FIELD_CAPS.keys())

__all__ = [
    "GridDiffEncoder", "ContextManager", "PromptPacket",
    "ContextOverflowError", "Zone2OverflowError", "default_token_counter",
    "GRID_SIZE",
]


def default_token_counter(text: str) -> int:
    """Conservative upper bound: UTF-8 byte count.  Never undercounts."""
    return len(text.encode("utf-8"))


class ContextOverflowError(RuntimeError):
    """Raised when the packet cannot fit under total_hard_cap even after
    the full eviction cascade."""


class Zone2OverflowError(ValueError):
    """Raised when a Zone 2 update exceeds its per-field cap and the
    caller did not opt into truncation via ``truncate=True``."""


def _json_safe(v: Any) -> Any:
    """Coerce ``v`` to a JSON-serialisable form, recursively.

    Sets/frozensets/bytes/tuples are wrapped in single-key marker dicts
    so they survive round-trip; other objects become reprs.
    """
    if v is None or isinstance(v, (bool, int, float, str)):
        return v
    if isinstance(v, list):
        return [_json_safe(x) for x in v]
    if isinstance(v, tuple):
        return {"__tuple__": [_json_safe(x) for x in v]}
    if isinstance(v, dict):
        return {str(k): _json_safe(val) for k, val in v.items()}
    if isinstance(v, frozenset):
        return {"__frozenset__":
                sorted((_json_safe(x) for x in v), key=repr)}
    if isinstance(v, set):
        return {"__set__": sorted((_json_safe(x) for x in v), key=repr)}
    if isinstance(v, bytes):
        return {"__bytes__": v.hex()}
    return repr(v)


def _json_restore(v: Any) -> Any:
    """Inverse of :func:`_json_safe`.

    Marker dicts (``__set__`` / ``__frozenset__`` / ``__bytes__`` /
    ``__tuple__``) are decoded back into their original Python types.
    Ordinary dicts and lists are recursed.
    """
    if isinstance(v, dict):
        if set(v.keys()) == {"__set__"}:
            return set(_json_restore(x) for x in v["__set__"])
        if set(v.keys()) == {"__frozenset__"}:
            return frozenset(_json_restore(x) for x in v["__frozenset__"])
        if set(v.keys()) == {"__bytes__"}:
            return bytes.fromhex(v["__bytes__"])
        if set(v.keys()) == {"__tuple__"}:
            return tuple(_json_restore(x) for x in v["__tuple__"])
        return {k: _json_restore(val) for k, val in v.items()}
    if isinstance(v, list):
        return [_json_restore(x) for x in v]
    return v


# --------------------------------------------------------------------------- #
# GridDiffEncoder
# --------------------------------------------------------------------------- #

class GridDiffEncoder:
    """Compact encoder for 64x64 integer grids (values 0-15).

    Full encoding: one row per list element, each row RLE-encoded as
    ``<value_hex>:<count_decimal>`` pairs joined by ``,``.  Delta
    encoding: a list of ``{"r","c","old","new"}`` cell changes plus
    base_checksum, new checksum, new version.

    ``apply_delta`` is atomic: all changes are staged on a candidate
    copy and committed only after every change validates (coordinate
    range, value range, old-value match) and the post-apply checksum
    matches.  On any rejection the internal grid is left byte-identical
    and ``needs_refresh`` is set True.

    ``decode`` is a reconstruction routine.  It always validates
    coordinate range and value range (both in the full RLE path and in
    every delta change).  Pass ``verify=True`` to also verify base and
    post-delta checksums against the recorded checksum fields.
    """

    def __init__(self, refresh_every: int = DEFAULT_REFRESH_EVERY) -> None:
        self.refresh_every = int(refresh_every)
        self.needs_refresh: bool = True
        self._version: int = 0
        self._frame_count: int = 0
        self._delta_count: int = 0
        self._checksum: Optional[str] = None
        self._grid: Optional[List[List[int]]] = None

    def encode_full(self, grid: List[List[int]]) -> Dict[str, Any]:
        """Encode ``grid`` as a full snapshot; clears refresh state."""
        self._validate_grid(grid)
        cs = self._compute_checksum(grid)
        self._version += 1
        self._checksum = cs
        self._grid = [list(r) for r in grid]
        self._frame_count = 0
        self._delta_count = 0
        self.needs_refresh = False
        return {"type": "full", "version": self._version,
                "checksum": cs, "rows": [self._rle_row(r) for r in grid]}

    def encode_delta(self, old_grid: List[List[int]],
                     new_grid: List[List[int]]) -> Dict[str, Any]:
        """Encode the cell-level diff between two same-shape grids."""
        self._validate_grid(old_grid)
        self._validate_grid(new_grid)
        changes: List[Dict[str, int]] = []
        for r in range(GRID_SIZE):
            orow = old_grid[r]
            nrow = new_grid[r]
            for c in range(GRID_SIZE):
                if orow[c] != nrow[c]:
                    changes.append({"r": r, "c": c,
                                    "old": orow[c], "new": nrow[c]})
        base_cs = self._compute_checksum(old_grid)
        new_cs = self._compute_checksum(new_grid)
        self._version += 1
        self._checksum = new_cs
        self._grid = [list(r) for r in new_grid]
        self._delta_count += 1
        if self._delta_count > MAX_DELTA_LIST:
            self.needs_refresh = True
        return {"type": "delta", "version": self._version,
                "base_checksum": base_cs, "checksum": new_cs,
                "changes": changes}

    def apply_delta(self, delta: Dict[str, Any]) -> bool:
        """Apply a delta to internal state; return False on any failure.

        Atomic.  Validates base checksum, coordinate range, cell-value
        range, old-value match (F-02 v1.3), and post-apply checksum.
        On any rejection the internal grid is left byte-identical.
        """
        if self.needs_refresh:
            return False
        if delta.get("type") != "delta":
            self.needs_refresh = True
            return False
        if delta.get("base_checksum") != self._checksum:
            logger.warning("grid drift: base checksum mismatch")
            self.needs_refresh = True
            return False
        if self._grid is None:
            self.needs_refresh = True
            return False

        candidate = [row[:] for row in self._grid]
        for ch in delta.get("changes", []):
            r = int(ch["r"])
            c = int(ch["c"])
            nv = int(ch["new"])
            if not (0 <= r < GRID_SIZE and 0 <= c < GRID_SIZE):
                logger.warning(
                    "grid drift: change out of bounds (%d,%d)", r, c)
                self.needs_refresh = True
                return False
            if not (0 <= nv <= GRID_MAX_VALUE):
                logger.warning(
                    "grid drift: change value out of range %d", nv)
                self.needs_refresh = True
                return False
            # F-02 (v1.3): verify the delta's stated old value matches reality.
            ov = int(ch["old"])
            if candidate[r][c] != ov:
                logger.warning(
                    "grid drift: old value mismatch at (%d,%d): "
                    "delta says %d, grid has %d",
                    r, c, ov, candidate[r][c])
                self.needs_refresh = True
                return False
            candidate[r][c] = nv

        if self._compute_checksum(candidate) != delta.get("checksum"):
            logger.warning("grid drift: post-apply checksum mismatch")
            self.needs_refresh = True
            return False

        self._grid = candidate
        self._checksum = delta["checksum"]
        self._version = int(delta["version"])
        self._delta_count += 1
        if self._delta_count > MAX_DELTA_LIST:
            self.needs_refresh = True
        return True

    @classmethod
    def decode(cls, state: Dict[str, Any], verify: bool = False
               ) -> List[List[int]]:
        """Reconstruct a 64x64 grid from full or {full, deltas} bundle.

        Always validates coordinate and value range (including the
        values embedded in the full RLE).  ``verify=True`` additionally
        checks base and post-delta checksums.
        """
        if "full" in state:
            full = state["full"]
            grid = cls._rle_decode_rows(full["rows"])
            if verify and cls._compute_checksum(grid) != full.get("checksum"):
                raise ValueError("decode: full checksum mismatch")
            for d in state.get("deltas", []):
                if verify:
                    if cls._compute_checksum(grid) != d.get("base_checksum"):
                        raise ValueError("decode: base checksum mismatch")
                for ch in d.get("changes", []):
                    r = int(ch["r"])
                    c = int(ch["c"])
                    nv = int(ch["new"])
                    if not (0 <= r < GRID_SIZE and 0 <= c < GRID_SIZE):
                        raise ValueError(
                            f"decode: coordinate out of range ({r},{c})")
                    if not (0 <= nv <= GRID_MAX_VALUE):
                        raise ValueError(
                            f"decode: value out of range ({nv})")
                    grid[r][c] = nv
                if verify and cls._compute_checksum(grid) != d.get("checksum"):
                    raise ValueError("decode: post-delta checksum mismatch")
            return grid
        if state.get("type") == "full":
            grid = cls._rle_decode_rows(state["rows"])
            if verify and cls._compute_checksum(grid) != state.get("checksum"):
                raise ValueError("decode: full checksum mismatch")
            return grid
        raise ValueError(
            "decode expects a full-encoding dict or a {full,deltas} bundle")

    def mark_new_game(self) -> None:
        self.needs_refresh = True

    def mark_new_level(self) -> None:
        self.needs_refresh = True

    def mark_death(self) -> None:
        self.needs_refresh = True

    def tick(self) -> None:
        self._frame_count += 1
        if self.refresh_every > 0 and self._frame_count >= self.refresh_every:
            self.needs_refresh = True

    def snapshot_state(self) -> Dict[str, Any]:
        return {"refresh_every": self.refresh_every,
                "needs_refresh": self.needs_refresh,
                "version": self._version,
                "frame_count": self._frame_count,
                "delta_count": self._delta_count,
                "checksum": self._checksum,
                "grid": self._grid}

    def load_state(self, snap: Dict[str, Any]) -> None:
        self.refresh_every = int(snap.get("refresh_every",
                                          DEFAULT_REFRESH_EVERY))
        self.needs_refresh = bool(snap.get("needs_refresh", True))
        self._version = int(snap.get("version", 0))
        self._frame_count = int(snap.get("frame_count", 0))
        self._delta_count = int(snap.get("delta_count", 0))
        self._checksum = snap.get("checksum")
        grid = snap.get("grid")
        self._grid = [list(r) for r in grid] if grid else None

    @staticmethod
    def _validate_grid(grid: List[List[int]]) -> None:
        if len(grid) != GRID_SIZE:
            raise ValueError(f"grid must have {GRID_SIZE} rows")
        for r, row in enumerate(grid):
            if len(row) != GRID_SIZE:
                raise ValueError(f"row {r}: must have {GRID_SIZE} cells")
            for c, v in enumerate(row):
                if not isinstance(v, int) or not (0 <= v <= GRID_MAX_VALUE):
                    raise ValueError(
                        f"cell ({r},{c}) = {v!r} not an int 0-{GRID_MAX_VALUE}")

    @staticmethod
    def _compute_checksum(grid: List[List[int]]) -> str:
        h = hashlib.md5()
        for row in grid:
            h.update(bytes(row))
        return h.hexdigest()

    @staticmethod
    def _rle_row(row: List[int]) -> str:
        parts: List[str] = []
        cur = row[0]
        n = 1
        for v in row[1:]:
            if v == cur:
                n += 1
            else:
                parts.append(f"{cur:x}:{n}")
                cur = v
                n = 1
        parts.append(f"{cur:x}:{n}")
        return ",".join(parts)

    @staticmethod
    def _rle_decode_rows(rows: List[str]) -> List[List[int]]:
        grid: List[List[int]] = []
        for s in rows:
            out: List[int] = []
            for chunk in s.split(","):
                if not chunk:
                    continue
                vhex, sep, cdec = chunk.partition(":")
                if not sep:
                    raise ValueError(f"malformed RLE chunk: {chunk!r}")
                v = int(vhex, 16)
                n = int(cdec)
                # F-01 (v1.3): enforce the grid value domain in the full path.
                if not (0 <= v <= GRID_MAX_VALUE):
                    raise ValueError(
                        f"RLE value out of range: {v} (chunk {chunk!r})")
                out.extend([v] * n)
            if len(out) != GRID_SIZE:
                raise ValueError("decoded row wrong length")
            grid.append(out)
        if len(grid) != GRID_SIZE:
            raise ValueError("decoded grid wrong number of rows")
        return grid


# --------------------------------------------------------------------------- #
# PromptPacket
# --------------------------------------------------------------------------- #

@dataclass
class PromptPacket:
    """One assembled prompt for the inference layer."""

    text: str
    token_estimate: int
    per_zone_tokens: Dict[str, int]
    needs_refresh: bool
    version: int


# --------------------------------------------------------------------------- #
# ContextManager
# --------------------------------------------------------------------------- #

class ContextManager:
    """Three-zone context manager for the ARC-AGI-3 agent loop.

    Zone 1 (system) set once, never evicted.  Zone 2 is per-game
    knowledge: four named fields, each with its own token cap, written
    WHOLESALE.  Zone 3 is a rolling FIFO of the last ten actions plus a
    bounded summary slot.  Observation carries the grid encoding,
    available actions, levels completed, and game state.

    ``build_prompt`` counts tokens BEFORE returning; if the total exceeds
    ``total_hard_cap`` it runs an eviction cascade: (1) drop oldest Zone 3
    entries, (2) force observation coordinate-only fallback, (3) drop
    Zone 2 hypotheses then goal_notes.  If still over cap, raises
    ``ContextOverflowError`` — never silently truncates.

    Per-zone caps are ADVISORY: exceeding them logs a WARNING at build
    time but does not raise.  The sole hard gate is ``total_hard_cap``.
    Zone 2 additionally enforces four per-field sub-caps via
    ``_ZONE2_FIELD_CAPS``.

    Contract (F-5): Zone 2 is written via update_zone2 / promote_to_zone2
    calls the HARNESS routes from CODER output.
    """

    def __init__(
        self,
        zone1_max_tokens: int = 200,
        zone2_max_tokens: int = 300,
        zone3_max_tokens: int = 200,
        observation_max_tokens: int = 2500,
        total_hard_cap: int = 7800,
        refresh_every: int = DEFAULT_REFRESH_EVERY,
        token_counter: Optional[Callable[[str], int]] = None,
    ) -> None:
        self.zone1_max_tokens = int(zone1_max_tokens)
        self.zone2_max_tokens = int(zone2_max_tokens)
        self.zone3_max_tokens = int(zone3_max_tokens)
        self.observation_max_tokens = int(observation_max_tokens)
        self.total_hard_cap = int(total_hard_cap)
        self.refresh_every = int(refresh_every)
        self._tc: Callable[[str], int] = token_counter or default_token_counter

        self._zone1: str = ""
        self._zone2: Dict[str, str] = {f: "" for f in _ZONE2_FIELDS}
        self._zone3_summary: Deque[str] = deque(maxlen=_MAX_SUMMARY_LINES)
        self._zone3_entries: Deque[Dict[str, Any]] = deque(maxlen=10)

        self._obs_encoding: Optional[Dict[str, Any]] = None
        self._obs_actions: List[Any] = []
        self._obs_levels: int = 0
        self._obs_state: str = "unknown"
        self._obs_objects: List[Any] = []
        self._obs_force_fallback: bool = False
        self._obs_fallback_truncated: int = 0

        self._dead_sigs: Deque[str] = deque(maxlen=_MAX_DEAD_SIGS_PER_LEVEL)
        self._level_num: int = 0

        self._build_version: int = 0
        self._last_total_tokens: int = 0
        self.needs_refresh: bool = True

    # ---- Zone 1 ---- #

    def set_system_prompt(self, text: str) -> None:
        """Set Zone 1.  Never evicted, never rewritten by build_prompt."""
        self._zone1 = text or ""
        t = self._tc(self._zone1)
        if t > self.zone1_max_tokens:
            logger.warning(
                "zone1 set to %d tokens (declared cap %d); zone1 is "
                "never evicted and this may force overflow at build time",
                t, self.zone1_max_tokens)

    # ---- Zone 2 ---- #

    def update_zone2(self, field: str, new_text: str,
                     truncate: bool = False) -> None:
        """Wholesale-replace a Zone 2 field; enforce its per-field cap.

        Raises ``Zone2OverflowError`` by default.  Pass ``truncate=True``
        to opt into prefix-keep truncation (logs a WARNING).
        """
        if field not in self._zone2:
            raise ValueError(f"unknown zone2 field: {field!r}")
        cap = _ZONE2_FIELD_CAPS[field]
        text = new_text or ""
        if self._tc(text) <= cap:
            self._zone2[field] = text
            return
        if not truncate:
            raise Zone2OverflowError(
                f"zone2 field {field!r}: {self._tc(text)} tokens > cap "
                f"{cap}; pass truncate=True to accept truncation")
        truncated = self._truncate_prefix_to_cap(text, cap)
        logger.warning(
            "zone2 field %r truncated %d -> %d tokens (cap %d)",
            field, self._tc(text), self._tc(truncated), cap)
        self._zone2[field] = truncated

    def reset_zone2(self) -> None:
        """Clear all four Zone 2 fields (call on game change)."""
        for f in _ZONE2_FIELDS:
            self._zone2[f] = ""

    def promote_to_zone2(self, line: str, field: str,
                         truncate: bool = False) -> None:
        """Promote one distilled line into a Zone 2 field immediately.

        F-02 (v1.2): the ``truncate`` flag is caller-controlled and
        defaults to False — an over-cap composed field raises
        ``Zone2OverflowError`` rather than silently dropping the tail.
        """
        if field not in self._zone2:
            raise ValueError(f"unknown zone2 field: {field!r}")
        existing = self._zone2.get(field, "")
        composed = f"{line}\n{existing}".strip() if existing else line
        self.update_zone2(field, composed, truncate=truncate)

    # ---- Zone 3 ---- #

    def append_action(self, action: Any, result: str,
                      delta_count: int) -> None:
        """Append one executed action to the Zone 3 FIFO (max 10)."""
        dc = int(delta_count)
        self._zone3_entries.append({
            "action": action, "result": str(result),
            "delta_count": dc, "changed": dc > 0})

    def add_summary_line(self, line: str) -> None:
        """Append a distilled failing-attempt summary line.

        The deque has maxlen=_MAX_SUMMARY_LINES; oldest evicted at cap.
        """
        self._zone3_summary.append(str(line))

    # ---- Observation ---- #

    def set_observation(
        self,
        encoding_dict: Optional[Dict[str, Any]],
        available_actions: Optional[List[Any]] = None,
        levels_completed: int = 0,
        state: str = "unknown",
        objects: Optional[List[Any]] = None,
    ) -> None:
        """Set the observation slot (grid encoding + metadata).

        v1.6: ``objects`` = list of dicts {id, color_char, color_digit,
        size, bbox[r0, r1, c0, c1]}.  Observation block mein "OBJECTS:"
        section render hota hai (cap _MAX_OBS_OBJECTS).
        """
        self._obs_encoding = encoding_dict
        self._obs_actions = list(available_actions or [])
        self._obs_objects = list(objects or [])
        self._obs_levels = int(levels_completed)
        self._obs_state = str(state)
        self._obs_force_fallback = False
        self._obs_fallback_truncated = 0

    # ---- Dead signatures ---- #

    def register_dead_signature(self, sig: str) -> None:
        """Record an object signature that never changed the grid.

        FIFO-bounded at _MAX_DEAD_SIGS_PER_LEVEL.
        """
        s = str(sig)
        if s and s not in self._dead_sigs:
            self._dead_sigs.append(s)

    def clear_dead_signatures(self) -> None:
        self._dead_sigs.clear()

    def set_level(self, level_num: int) -> None:
        """Notify a level transition; clears the dead-signature registry."""
        n = int(level_num)
        if n != self._level_num:
            self._dead_sigs.clear()
            self._level_num = n

    # ---- Memory pressure (F-6) ---- #

    def memory_pressure(self) -> str:
        """Return 'ok' / 'warn' / 'flush' from the last build total."""
        if self.total_hard_cap <= 0:
            return "flush"
        r = self._last_total_tokens / self.total_hard_cap
        if r >= 1.0:
            return "flush"
        if r >= 0.7:
            return "warn"
        return "ok"

    # ---- build_prompt + cascade ---- #

    def build_prompt(self) -> PromptPacket:
        """Assemble the packet; enforce total_hard_cap via cascade."""
        text, tokens, total = self._assemble_with_tokens()
        if total > self.total_hard_cap:
            text, tokens, total = self._evict_and_rebuild()
        if total > self.total_hard_cap:
            self._last_total_tokens = total
            self._build_version += 1
            raise ContextOverflowError(
                f"cannot fit under total_hard_cap={self.total_hard_cap}; "
                f"packet={total} tokens")
        self._last_total_tokens = total
        self._build_version += 1
        return PromptPacket(
            text=text, token_estimate=total, per_zone_tokens=tokens,
            needs_refresh=self.needs_refresh, version=self._build_version)

    def _evict_and_rebuild(self) -> Tuple[str, Dict[str, int], int]:
        """4-step eviction cascade; return (text, tokens, total)."""
        _, _, total = self._assemble_with_tokens()
        while total > self.total_hard_cap and len(self._zone3_entries) > 3:
            self._zone3_entries.popleft()
            _, _, total = self._assemble_with_tokens()
        if total > self.total_hard_cap and not self._obs_force_fallback:
            self._obs_force_fallback = True
            _, _, total = self._assemble_with_tokens()
        if total > self.total_hard_cap and self._zone2.get("hypotheses"):
            self._zone2["hypotheses"] = ""
            _, _, total = self._assemble_with_tokens()
        if total > self.total_hard_cap and self._zone2.get("goal_notes"):
            self._zone2["goal_notes"] = ""
            _, _, total = self._assemble_with_tokens()
        return self._assemble_with_tokens()

    def _assemble_with_tokens(self) -> Tuple[str, Dict[str, int], int]:
        sections = (
            ("zone1", "=== SYSTEM ===", self._zone1),
            ("zone2", "=== GAME KNOWLEDGE ===", self._zone2_block()),
            ("zone3", "=== RECENT ACTIONS ===", self._zone3_block()),
            ("observation", "=== OBSERVATION ===", self._obs_block()),
        )
        chunks: List[str] = []
        tokens: Dict[str, int] = {}
        for key, header, body in sections:
            chunk = f"{header}\n{body}"
            chunks.append(chunk)
            tokens[key] = self._tc(chunk)
        text = "\n\n".join(chunks)
        total = self._tc(text)
        sum_zones = sum(tokens.values())
        if total > sum_zones:
            tokens["overhead"] = total - sum_zones

        # F-03 (v1.3): advisory per-zone cap warnings.
        for key, cap in (
            ("zone1", self.zone1_max_tokens),
            ("zone2", self.zone2_max_tokens),
            ("zone3", self.zone3_max_tokens),
            ("observation", self.observation_max_tokens),
        ):
            if tokens.get(key, 0) > cap:
                logger.warning(
                    "zone %s at %d tokens exceeds advisory cap %d",
                    key, tokens[key], cap)

        return text, tokens, total

    def _zone2_block(self) -> str:
        parts: List[str] = []
        for f in _ZONE2_FIELDS:
            v = self._zone2.get(f, "")
            if v:
                parts.append(f"[{f}]\n{v}")
        return "\n".join(parts)

    def _zone3_block(self) -> str:
        parts: List[str] = []
        if self._zone3_summary:
            parts.append("SUMMARY:\n" + "\n".join(self._zone3_summary))
        for e in self._zone3_entries:
            tag = "changed" if e.get("changed") else "no-change"
            parts.append(
                f"- {e.get('action')} -> {e.get('result')} ({tag})")
        return "\n".join(parts)

    def _objects_lines(self) -> List[str]:
        """v1.6: structured OBJECTS rendering — model ko seedha
        rows/cols dikhta hai, RLE column-arithmetic nahi karna padta."""
        out: List[str] = []
        if not self._obs_objects:
            return out
        out.append("OBJECTS (0-indexed rows/cols; ACTION r/c targets):")
        for ob in self._obs_objects[:_MAX_OBS_OBJECTS]:
            try:
                r0, r1, c0, c1 = ob["bbox"]
                out.append(
                    "  [{}] color={} size={} rows={}-{} cols={}-{}".format(
                        ob["id"], ob["color_char"], ob["size"],
                        int(r0), int(r1), int(c0), int(c1)))
            except (KeyError, TypeError, ValueError, IndexError):
                continue
        if len(self._obs_objects) > _MAX_OBS_OBJECTS:
            out.append("  <{} more objects hidden>".format(
                len(self._obs_objects) - _MAX_OBS_OBJECTS))
        return out

    def _obs_block(self) -> str:
        if self._obs_force_fallback:
            return self._obs_fallback()
        lines = [
            f"levels_completed={self._obs_levels}",
            f"state={self._obs_state}",
            f"available_actions={self._obs_actions}",
        ]
        if self._dead_sigs:
            lines.append(f"do_not_retry={list(self._dead_sigs)}")
        lines.extend(self._objects_lines())
        enc = self._obs_encoding
        if enc is None:
            lines.append("grid=none")
        elif enc.get("type") == "full":
            lines.append("GRID_FULL:")
            for i, r in enumerate(enc.get("rows", [])):
                lines.append(f"  {i:02d}|{r}")
        elif enc.get("type") == "delta":
            lines.append(f"GRID_DELTA v{enc.get('version')}:")
            for ch in enc.get("changes", []):
                lines.append(
                    f"  ({ch['r']},{ch['c']}) {ch['old']}->{ch['new']}")
        else:
            lines.append("grid=unknown")
        return "\n".join(lines)

    @staticmethod
    def _decode_rle_chunk(chunk: str):
        """v1.6: dono RLE formats — legacy "hex:dec" ("5:12") aur
        grid_text "char+count" ("W8").  Returns (value, count) ya
        (None, None) on malformed input."""
        if ":" in chunk:
            vhex, _, cdec = chunk.partition(":")
            try:
                return int(vhex, 16), int(cdec)
            except ValueError:
                return None, None
        if len(chunk) < 2 or chunk[0] not in _FALLBACK_CHAR_TO_DIGIT:
            return None, None
        try:
            return _FALLBACK_CHAR_TO_DIGIT[chunk[0]], int(chunk[1:])
        except ValueError:
            return None, None

    def _obs_fallback(self) -> str:
        """Coordinate-only summary.  Handles full and delta encodings."""
        self._obs_fallback_truncated = 0
        lines = [
            f"levels_completed={self._obs_levels}",
            f"state={self._obs_state}",
            f"available_actions={self._obs_actions}",
        ]
        if self._dead_sigs:
            lines.append(f"do_not_retry={list(self._dead_sigs)}")
        lines.extend(self._objects_lines())

        enc = self._obs_encoding or {}
        etype = enc.get("type")

        if etype == "full":
            total_nonzero = 0
            pending: List[str] = []
            for r, s in enumerate(enc.get("rows", [])):
                c = 0
                for chunk in s.split(","):
                    if not chunk:
                        continue
                    v, n = self._decode_rle_chunk(chunk)
                    if v is None:
                        logger.warning(
                            "fallback: malformed RLE chunk %r at row %d",
                            chunk, r)
                        lines.append("NONZERO_CELLS=<malformed>")
                        return "\n".join(lines)
                    if v != 0:
                        for k in range(n):
                            total_nonzero += 1
                            if total_nonzero <= _MAX_FALLBACK_CELLS:
                                pending.append(f"  ({r},{c + k},{v})")
                    c += n
            if total_nonzero == 0:
                lines.append("NONZERO_CELLS=0")
                return "\n".join(lines)
            lines.append("NONZERO_CELLS (r,c,v):")
            lines.extend(pending)
            if total_nonzero > _MAX_FALLBACK_CELLS:
                hidden = total_nonzero - _MAX_FALLBACK_CELLS
                self._obs_fallback_truncated = hidden
                lines.append(
                    f"  <TRUNCATED: {hidden} more non-zero cells>")
                logger.warning(
                    "observation fallback truncated: %d non-zero cells "
                    "hidden (cap %d)", hidden, _MAX_FALLBACK_CELLS)
            return "\n".join(lines)

        if etype == "delta":
            changes = enc.get("changes", [])
            if not changes:
                lines.append("GRID_DELTA (no changes)")
                return "\n".join(lines)
            lines.append(f"GRID_DELTA v{enc.get('version')} "
                         f"({len(changes)} changes):")
            shown = changes[:_MAX_FALLBACK_CELLS]
            for ch in shown:
                lines.append(f"  ({ch['r']},{ch['c']},{ch['new']})")
            if len(changes) > _MAX_FALLBACK_CELLS:
                hidden = len(changes) - _MAX_FALLBACK_CELLS
                self._obs_fallback_truncated = hidden
                lines.append(f"  <TRUNCATED: {hidden} more changes>")
                logger.warning(
                    "observation fallback truncated: %d delta changes "
                    "hidden (cap %d)", hidden, _MAX_FALLBACK_CELLS)
            return "\n".join(lines)

        lines.append("grid=unknown")
        return "\n".join(lines)

    # ---- snapshot / load ---- #

    def snapshot(self) -> Dict[str, Any]:
        """Return a JSON-serialisable snapshot of all internal state."""
        return {
            "schema": "context_manager/1.4",
            "zone1": self._zone1,
            "zone2": _json_safe(dict(self._zone2)),
            "zone3_summary": list(self._zone3_summary),
            "zone3_entries": _json_safe(list(self._zone3_entries)),
            "obs_encoding": _json_safe(self._obs_encoding),
            "obs_actions": _json_safe(self._obs_actions),
            "obs_levels": self._obs_levels,
            "obs_state": self._obs_state,
            "obs_force_fallback": self._obs_force_fallback,
            "obs_fallback_truncated": self._obs_fallback_truncated,
            "dead_sigs": list(self._dead_sigs),
            "level_num": self._level_num,
            "build_version": self._build_version,
            "last_total_tokens": self._last_total_tokens,
            "needs_refresh": self.needs_refresh,
        }

    def load_snapshot(self, snap: Dict[str, Any]) -> None:
        """Restore state from a snapshot produced by :meth:`snapshot`."""
        self._zone1 = snap.get("zone1", "")
        z2 = _json_restore(snap.get("zone2", {}))
        self._zone2 = {f: z2.get(f, "") for f in _ZONE2_FIELDS}
        self._zone3_summary = deque(
            snap.get("zone3_summary", []), maxlen=_MAX_SUMMARY_LINES)
        self._zone3_entries = deque(
            _json_restore(snap.get("zone3_entries", [])), maxlen=10)
        self._obs_encoding = _json_restore(snap.get("obs_encoding"))
        self._obs_actions = _json_restore(list(snap.get("obs_actions", [])))
        self._obs_levels = int(snap.get("obs_levels", 0))
        self._obs_state = snap.get("obs_state", "unknown")
        self._obs_force_fallback = bool(
            snap.get("obs_force_fallback", False))
        self._obs_fallback_truncated = int(
            snap.get("obs_fallback_truncated", 0))
        self._dead_sigs = deque(
            snap.get("dead_sigs", []), maxlen=_MAX_DEAD_SIGS_PER_LEVEL)
        self._level_num = int(snap.get("level_num", 0))
        self._build_version = int(snap.get("build_version", 0))
        self._last_total_tokens = int(snap.get("last_total_tokens", 0))
        self.needs_refresh = bool(snap.get("needs_refresh", True))

    # ---- helpers ---- #

    def _truncate_prefix_to_cap(self, text: str, cap: int) -> str:
        """Keep the longest prefix of ``text`` whose token count <= cap."""
        if self._tc(text) <= cap:
            return text
        lo, hi = 0, len(text)
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if self._tc(text[:mid]) <= cap:
                lo = mid
            else:
                hi = mid - 1
        return text[:lo]


# =========================================================================== #
# INLINE TEST SUITE (v1.4) — runs only under __main__
# =========================================================================== #

def _grid(value: int = 0) -> List[List[int]]:
    return [[value] * GRID_SIZE for _ in range(GRID_SIZE)]


def _random_grid(rng: random.Random) -> List[List[int]]:
    return [[rng.randrange(0, 16) for _ in range(GRID_SIZE)]
            for _ in range(GRID_SIZE)]


class _Capture(logging.Handler):
    def __init__(self) -> None:
        super().__init__()
        self.records: List[logging.LogRecord] = []

    def emit(self, record: logging.LogRecord) -> None:
        self.records.append(record)


def _t1_zone2_reset():
    cm = ContextManager()
    cm.set_system_prompt("sys")
    cm.update_zone2("confirmed_rules", "rule A")
    cm.update_zone2("hypotheses", "h1")
    cm.update_zone2("world_model_status", "wip")
    cm.update_zone2("goal_notes", "goal")
    cm.reset_zone2()
    for f in ("confirmed_rules", "hypotheses",
              "world_model_status", "goal_notes"):
        assert cm._zone2[f] == "", f"{f} not cleared"
    assert cm._zone1 == "sys", "zone1 lost on reset"


def _t2_whole_field_replacement():
    cm = ContextManager()
    cm.update_zone2("confirmed_rules", "old text here")
    cm.update_zone2("confirmed_rules", "new")
    assert cm._zone2["confirmed_rules"] == "new"
    assert "old" not in cm._zone2["confirmed_rules"]


def _t3_field_cap_and_warning():
    # v1.5: version-proof — cap constant se derive hota hai (80 tha, ab 350)
    _cap = _ZONE2_FIELD_CAPS["hypotheses"]
    cm = ContextManager(token_counter=lambda s: len(s))
    try:
        cm.update_zone2("hypotheses", "X" * (_cap + 120))
    except Zone2OverflowError:
        pass
    else:
        raise AssertionError("expected Zone2OverflowError")
    cap = _Capture()
    logger.addHandler(cap)
    try:
        cm.update_zone2("hypotheses", "X" * (_cap + 120), truncate=True)
    finally:
        logger.removeHandler(cap)
    assert cm._tc(cm._zone2["hypotheses"]) <= _cap
    assert any("truncated" in r.getMessage() for r in cap.records)


def _t4_grid_diff_roundtrip():
    rng = random.Random(1)
    enc = GridDiffEncoder()
    g0 = _random_grid(rng)
    full = enc.encode_full(g0)
    bundle = {"full": full, "deltas": []}
    for _ in range(5):
        g1 = [row[:] for row in g0]
        for _ in range(20):
            r = rng.randrange(GRID_SIZE)
            c = rng.randrange(GRID_SIZE)
            g1[r][c] = rng.randrange(16)
        bundle["deltas"].append(enc.encode_delta(g0, g1))
        g0 = g1
    decoded = GridDiffEncoder.decode(bundle)
    assert decoded == g0, "roundtrip mismatch"


def _t5_drift_guard():
    enc = GridDiffEncoder()
    g0 = _grid(0)
    enc.encode_full(g0)
    g1 = [row[:] for row in g0]
    g1[0][0] = 5
    delta = enc.encode_delta(g0, g1)
    g3 = _grid(3)
    enc.encode_full(g3)
    before = [r[:] for r in enc._grid]
    ok = enc.apply_delta(delta)
    assert ok is False
    assert enc.needs_refresh is True
    assert enc._grid == before, "internal state mutated on drift"
    assert enc._grid == g3


def _t6_refresh_triggers():
    enc = GridDiffEncoder(refresh_every=10)
    enc.encode_full(_grid(0))
    assert enc.needs_refresh is False
    enc.mark_new_game()
    assert enc.needs_refresh is True
    enc.encode_full(_grid(0))
    enc.mark_new_level()
    assert enc.needs_refresh is True
    enc.encode_full(_grid(0))
    enc.mark_death()
    assert enc.needs_refresh is True
    enc.encode_full(_grid(0))
    for _ in range(9):
        enc.tick()
    assert enc.needs_refresh is False
    enc.tick()
    assert enc.needs_refresh is True


def _t7_build_order():
    cm = ContextManager()
    cm.set_system_prompt("SYS")
    cm.update_zone2("confirmed_rules", "R")
    cm.append_action("A1", "ok", 1)
    cm.set_observation(None, ["a"], 0, "RUNNING")
    t = cm.build_prompt().text
    i1 = t.index("=== SYSTEM ===")
    i2 = t.index("=== GAME KNOWLEDGE ===")
    i3 = t.index("=== RECENT ACTIONS ===")
    i4 = t.index("=== OBSERVATION ===")
    assert i1 < i2 < i3 < i4, f"order wrong: {i1},{i2},{i3},{i4}"


def _t8_hard_gate_cascade():
    cm = ContextManager(
        zone1_max_tokens=100, zone2_max_tokens=200, zone3_max_tokens=200,
        observation_max_tokens=500, total_hard_cap=800,
        token_counter=lambda s: len(s))
    cm.set_system_prompt("S" * 60)
    cm.update_zone2("confirmed_rules", "R" * 60)
    cm.update_zone2("hypotheses", "H" * 60)
    cm.update_zone2("goal_notes", "G" * 30)
    for i in range(10):
        cm.append_action(f"a{i}", "res" * 10, delta_count=1)
    cm.set_observation(
        {"type": "full", "rows": ["0:64"] * GRID_SIZE}, [], 0, "RUNNING")
    n_before = len(cm._zone3_entries)
    pkt = cm.build_prompt()
    n_after = len(cm._zone3_entries)
    assert n_after <= 3, f"zone3 not shrunk to <=3: {n_after}"
    assert n_after < n_before
    assert pkt.token_estimate <= 800


def _t9_cascade_exhaustion():
    cm = ContextManager(
        zone1_max_tokens=200, zone2_max_tokens=300,
        zone3_max_tokens=200, observation_max_tokens=500,
        total_hard_cap=100, token_counter=lambda s: len(s))
    cm.set_system_prompt("A" * 300)
    cm.update_zone2("confirmed_rules", "R" * 100)
    try:
        cm.build_prompt()
    except ContextOverflowError:
        return
    raise AssertionError("expected ContextOverflowError")


def _t10_promote_to_zone2():
    cm = ContextManager()
    for i in range(10):
        cm.append_action(f"a{i}", "r", 1)
    assert len(cm._zone3_entries) == 10
    cm.promote_to_zone2("hypothesis violated!", "hypotheses")
    assert "hypothesis violated" in cm._zone2["hypotheses"]


def _t11_dead_signature():
    cm = ContextManager()
    cm.register_dead_signature("obj-42")
    cm.set_observation(None, ["a"], 0, "RUNNING")
    pkt1 = cm.build_prompt()
    assert "obj-42" in pkt1.text
    cm.set_level(1)
    pkt2 = cm.build_prompt()
    assert "obj-42" not in pkt2.text


def _t12_token_counting():
    cm = ContextManager(token_counter=lambda s: len(s))
    cm.set_system_prompt("hello world")
    cm.append_action("a", "b", 1)
    cm.set_observation(None, [], 0, "RUNNING")
    pkt = cm.build_prompt()
    assert pkt.token_estimate == len(pkt.text)
    samples = ["hello world foo", "a", "", "   ", "hi\nthere",
               "unicode: हिन्दी"]
    for s in samples:
        assert default_token_counter(s) >= len(s.split()), s


def _t13_snapshot_roundtrip():
    cm = ContextManager()
    cm.set_system_prompt("SYS")
    cm.update_zone2("confirmed_rules", "R")
    cm.append_action("a1", "ok", 1)
    cm.add_summary_line("attempt 1: failed because X")
    cm.set_observation(
        {"type": "full", "rows": ["0:64"] * GRID_SIZE}, [], 0, "RUNNING")
    cm.register_dead_signature("d1")
    pkt1 = cm.build_prompt()
    snap = cm.snapshot()
    js = json.dumps(snap)
    cm2 = ContextManager()
    cm2.load_snapshot(json.loads(js))
    pkt2 = cm2.build_prompt()
    assert pkt1.text == pkt2.text, "text mismatch after roundtrip"
    assert pkt1.token_estimate == pkt2.token_estimate


def _t14_fuzz():
    rng = random.Random(99)
    fields = ("confirmed_rules", "hypotheses",
              "world_model_status", "goal_notes")
    for _ in range(200):
        cm = ContextManager(
            zone1_max_tokens=100, zone2_max_tokens=300,
            zone3_max_tokens=200, observation_max_tokens=500,
            total_hard_cap=3000, token_counter=lambda s: len(s))
        cm.set_system_prompt("S" * 50)
        try:
            for _ in range(20):
                op = rng.randrange(6)
                if op == 0:
                    cm.append_action(
                        f"a{rng.randrange(10)}",
                        "r" * rng.randrange(20),
                        delta_count=rng.randrange(3))
                elif op == 1:
                    try:
                        cm.update_zone2(rng.choice(fields),
                                        "X" * rng.randrange(100))
                    except Zone2OverflowError:
                        pass
                elif op == 2:
                    try:
                        cm.promote_to_zone2(
                            f"evt{rng.randrange(100)}",
                            rng.choice(fields))
                    except Zone2OverflowError:
                        pass
                elif op == 3:
                    if rng.random() < 0.5:
                        enc = {"type": "delta", "version": 1,
                               "changes": [{"r": 0, "c": 0,
                                            "old": 0, "new": 1}]}
                    else:
                        enc = {"type": "full",
                               "rows": ["0:64"] * GRID_SIZE}
                    cm.set_observation(enc, ["a"], 0, "RUNNING")
                elif op == 4:
                    cm.register_dead_signature(
                        f"sig{rng.randrange(20)}")
                else:
                    cm.set_level(rng.randrange(5))
            try:
                cm.build_prompt()
            except ContextOverflowError:
                pass
            assert cm._zone1 == "S" * 50, "zone1 mutated"
        except Exception as e:
            raise AssertionError(f"fuzz raised: {e!r}")


def _t15_apply_delta_partial_reject():
    enc = GridDiffEncoder()
    enc.encode_full(_grid(0))
    crafted = {
        "type": "delta", "version": enc._version + 1,
        "base_checksum": enc._checksum, "checksum": "deadbeef",
        "changes": [
            {"r": 0, "c": 0, "old": 0, "new": 7},
            {"r": 999, "c": 0, "old": 0, "new": 7},
        ],
    }
    before = [row[:] for row in enc._grid]
    ok = enc.apply_delta(crafted)
    assert ok is False
    assert enc._grid == before, "grid mutated despite rejection"
    assert enc._grid[0][0] == 0


def _t16_apply_delta_bad_checksum():
    enc = GridDiffEncoder()
    enc.encode_full(_grid(0))
    crafted = {
        "type": "delta", "version": enc._version + 1,
        "base_checksum": enc._checksum, "checksum": "deadbeef",
        "changes": [{"r": 5, "c": 5, "old": 0, "new": 9}],
    }
    before = [row[:] for row in enc._grid]
    ok = enc.apply_delta(crafted)
    assert ok is False
    assert enc._grid == before, "grid mutated on checksum reject"
    assert enc._grid[5][5] == 0


def _t17_zone3_summary_bounded():
    cm = ContextManager()
    for i in range(1000):
        cm.add_summary_line(f"line {i}")
    assert len(cm._zone3_summary) <= _MAX_SUMMARY_LINES
    assert "line 999" in list(cm._zone3_summary)


def _t18_dead_sigs_bounded():
    cm = ContextManager()
    for i in range(5000):
        cm.register_dead_signature(f"sig {i}")
    assert len(cm._dead_sigs) <= _MAX_DEAD_SIGS_PER_LEVEL
    assert "sig 4999" in list(cm._dead_sigs)


def _t19_snapshot_arbitrary_any():
    cm = ContextManager()
    cm.set_system_prompt("SYS")
    cm.append_action({"a", "b"}, "ok", 1)
    cm.append_action(b"\x00\x01", "ok", 0)
    cm.set_observation(None, [{"x": 1}, (1, 2)], 0, "RUNNING")
    snap = cm.snapshot()
    js = json.dumps(snap)

    entry0 = snap["zone3_entries"][0]["action"]
    assert entry0 == {"__set__": ["a", "b"]} or \
           entry0 == {"__set__": ["b", "a"]}, \
        f"set encoding lost: {entry0!r}"
    entry1 = snap["zone3_entries"][1]["action"]
    assert entry1 == {"__bytes__": "0001"}, \
        f"bytes encoding lost: {entry1!r}"
    assert snap["obs_actions"][0] == {"x": 1}, \
        f"dict obs_action corrupted: {snap['obs_actions'][0]!r}"
    # v1.4: tuples are now preserved via the __tuple__ marker.
    assert snap["obs_actions"][1] == {"__tuple__": [1, 2]}, \
        f"tuple marker missing: {snap['obs_actions'][1]!r}"

    cm2 = ContextManager()
    cm2.load_snapshot(json.loads(js))
    # Round-trip: the tuple must come back as a tuple, not a list.
    assert cm2._obs_actions[1] == (1, 2), \
        f"tuple not restored: {cm2._obs_actions[1]!r}"
    assert isinstance(cm2._obs_actions[1], tuple), \
        f"restored value is {type(cm2._obs_actions[1]).__name__}, not tuple"

    p1 = cm.build_prompt()
    p2 = cm2.build_prompt()
    assert p1.text == p2.text, "text mismatch after roundtrip"


def _t20_fallback_truncation_marker():
    cm = ContextManager()
    rows = ["1:64"] * 30 + ["0:64"] * (GRID_SIZE - 30)
    cm.set_observation({"type": "full", "rows": rows}, [], 0, "RUNNING")
    cm._obs_force_fallback = True
    text = cm.build_prompt().text
    assert "TRUNCATED" in text, "no truncation marker emitted"
    expected_hidden = 30 * GRID_SIZE - _MAX_FALLBACK_CELLS
    assert cm._obs_fallback_truncated == expected_hidden, \
        (f"counter wrong: got {cm._obs_fallback_truncated}, "
         f"expected {expected_hidden}")
    assert f"<TRUNCATED: {expected_hidden} more non-zero cells>" in text, \
        "marker does not report exact hidden count"


def _t21_promote_fail_loud():
    # v1.5: version-proof — cap constant se derive (80 tha, ab 350)
    _cap = _ZONE2_FIELD_CAPS["hypotheses"]
    cm = ContextManager(token_counter=lambda s: len(s))
    cm.update_zone2("hypotheses", "x" * (_cap - 10), truncate=True)
    try:
        cm.promote_to_zone2("surprising event!", "hypotheses")
    except Zone2OverflowError:
        pass
    else:
        raise AssertionError("expected Zone2OverflowError from promote")
    assert cm._zone2["hypotheses"] == "x" * (_cap - 10)
    cm.promote_to_zone2("surprising event!", "hypotheses", truncate=True)
    assert cm._tc(cm._zone2["hypotheses"]) <= _cap


def _t22_fallback_handles_delta():
    cm = ContextManager()
    delta = {"type": "delta", "version": 3,
             "base_checksum": "x", "checksum": "y",
             "changes": [{"r": 1, "c": 2, "old": 0, "new": 5},
                         {"r": 3, "c": 4, "old": 0, "new": 7}]}
    cm.set_observation(delta, [], 0, "RUNNING")
    cm._obs_force_fallback = True
    text = cm.build_prompt().text
    assert "GRID_DELTA" in text, "delta not surfaced in fallback"
    assert "(1,2,5)" in text and "(3,4,7)" in text
    assert "NONZERO_CELLS=0" not in text, \
        "delta silently lost as NONZERO_CELLS=0"


def _t23_apply_delta_bad_value():
    enc = GridDiffEncoder()
    enc.encode_full(_grid(0))
    before = [row[:] for row in enc._grid]
    crafted = {
        "type": "delta", "version": enc._version + 1,
        "base_checksum": enc._checksum, "checksum": "deadbeef",
        "changes": [{"r": 0, "c": 0, "old": 0, "new": 99}],
    }
    ok = enc.apply_delta(crafted)
    assert ok is False
    assert enc._grid == before, "grid mutated on value-range reject"


def _t24_decode_verify():
    enc = GridDiffEncoder()
    g0 = _grid(0)
    full = enc.encode_full(g0)
    g1 = [row[:] for row in g0]
    g1[0][0] = 5
    d = enc.encode_delta(g0, g1)
    d_bad = dict(d)
    d_bad["checksum"] = "deadbeef"
    bundle = {"full": full, "deltas": [d_bad]}
    g = GridDiffEncoder.decode(bundle, verify=False)
    assert g[0][0] == 5
    try:
        GridDiffEncoder.decode(bundle, verify=True)
    except ValueError as exc:
        assert "checksum" in str(exc).lower()
    else:
        raise AssertionError("expected ValueError with verify=True")


def _t25_decode_range_validation():
    enc = GridDiffEncoder()
    full = enc.encode_full(_grid(0))
    bad_bundle = {"full": full, "deltas": [
        {"type": "delta", "version": 2,
         "base_checksum": full["checksum"], "checksum": "x",
         "changes": [{"r": 999, "c": 0, "old": 0, "new": 1}]}]}
    try:
        GridDiffEncoder.decode(bad_bundle, verify=False)
    except ValueError as exc:
        assert "range" in str(exc).lower() or "coordinate" in str(exc).lower()
    else:
        raise AssertionError("expected range ValueError without verify")


def _t26_decode_bad_rle_value():
    bad_full = {"type": "full", "version": 1, "checksum": "x",
                "rows": ["10:64"] + ["0:64"] * (GRID_SIZE - 1)}
    try:
        GridDiffEncoder.decode(bad_full, verify=False)
    except ValueError as exc:
        assert "out of range" in str(exc).lower(), str(exc)
    else:
        raise AssertionError(
            "decode accepted RLE value 16 (GRID_MAX_VALUE is 15)")


def _t27_apply_delta_bad_old():
    enc = GridDiffEncoder()
    enc.encode_full(_grid(0))
    before = [row[:] for row in enc._grid]
    crafted = {
        "type": "delta", "version": enc._version + 1,
        "base_checksum": enc._checksum, "checksum": "deadbeef",
        "changes": [{"r": 0, "c": 0, "old": 14, "new": 5}],
    }
    ok = enc.apply_delta(crafted)
    assert ok is False
    assert enc._grid == before, "grid mutated on old-value mismatch"


def _t28_per_zone_cap_warns():
    cm = ContextManager(
        zone1_max_tokens=10, total_hard_cap=7800,
        token_counter=lambda s: len(s))
    cap = _Capture()
    logger.addHandler(cap)
    try:
        cm.set_system_prompt("S" * 50)
        cm.build_prompt()
    finally:
        logger.removeHandler(cap)
    assert any("advisory cap" in r.getMessage() for r in cap.records), \
        "no advisory-cap warning emitted"
    assert len(cm._zone1) == 50


def _t29_objects_rendered():
    cm = ContextManager()
    cm.set_system_prompt("s")
    obs = [{"id": 1, "color_char": "g", "color_digit": 2, "size": 24,
            "bbox": [5, 10, 30, 33]},
           {"id": 2, "color_char": "B", "color_digit": 5, "size": 25,
            "bbox": [30, 34, 8, 12]}]
    cm.set_observation({"type": "full", "dim": [64, 64],
                        "checksum": "x", "rows": ["W64"] * 64},
                       ["ACTION1"], 0, "RUNNING", objects=obs)
    text = cm.build_prompt().text
    assert "OBJECTS" in text, "objects block missing from prompt"
    assert "[1] color=g size=24 rows=5-10 cols=30-33" in text, \
        f"object 1 render wrong"
    assert "[2] color=B size=25 rows=30-34 cols=8-12" in text, \
        f"object 2 render wrong"
    # objects=None => no block (backward compat, v1.5 behavior)
    cm.set_observation({"type": "full", "dim": [64, 64],
                        "checksum": "x", "rows": ["W64"] * 64},
                       ["ACTION1"], 0, "RUNNING")
    assert "OBJECTS" not in cm.build_prompt().text, \
        "objects=None par bhi block render hua"
    # cap: 30 objects => sirf 24 render + hidden marker
    many = [{"id": i, "color_char": "g", "size": 1, "bbox": [0, 0, i, i]}
            for i in range(1, 31)]
    cm.set_observation({"type": "full", "dim": [64, 64],
                        "checksum": "x", "rows": ["W64"] * 64},
                       [], 0, "RUNNING", objects=many)
    t2 = cm.build_prompt().text
    assert "<6 more objects hidden>" in t2, "cap marker missing"
    # fallback mode mein bhi objects render hone chahiye
    cm._obs_force_fallback = True
    t3 = cm.build_prompt().text
    assert "OBJECTS" in t3 and "[1] color=g" in t3, \
        "fallback mode mein objects missing"


def _t30_fallback_accepts_char_format():
    # v1.6 regression: grid_text ka "char+count" RLE pehle <malformed> deta tha
    cm = ContextManager()
    rows = ["W8,g4,W52"] * 30 + ["W64"] * 34
    cm.set_observation({"type": "full", "dim": [64, 64],
                        "checksum": "x", "rows": rows}, [], 0, "RUNNING")
    cm._obs_force_fallback = True
    text = cm.build_prompt().text
    assert "<malformed>" not in text, "char-format RLE malformed!"
    assert "NONZERO_CELLS (r,c,v):" in text, "nonzero listing missing"
    assert "(0,8,2)" in text and "(0,9,2)" in text and "(0,11,2)" in text, \
        "char-format coordinates wrong"
    expected_nonzero = 30 * 4
    assert cm._obs_fallback_truncated == max(
        0, expected_nonzero - _MAX_FALLBACK_CELLS), \
        cm._obs_fallback_truncated
    # legacy hex:dec format ab bhi chale (v1.4 contract preserved)
    cm2 = ContextManager()
    cm2.set_observation({"type": "full", "dim": [64, 64],
                         "checksum": "x",
                         "rows": ["1:64"] * 30 + ["0:64"] * 34},
                        [], 0, "RUNNING")
    cm2._obs_force_fallback = True
    t2 = cm2.build_prompt().text
    assert "<malformed>" not in t2, "legacy format broken!"
    assert "NONZERO_CELLS (r,c,v):" in t2
    assert "(0,0,1)" in t2, "legacy coordinates wrong"
    # malformed chunk ab bhi fail-loud
    cm3 = ContextManager()
    cm3.set_observation({"type": "full", "dim": [64, 64],
                         "checksum": "x", "rows": ["zz9"]},
                        [], 0, "RUNNING")
    cm3._obs_force_fallback = True
    assert "NONZERO_CELLS=<malformed>" in cm3.build_prompt().text


def _run_self_tests() -> int:
    """Execute the full v1.6 test battery.  Returns process exit code."""
    tests = [
        ("T-1  zone2 reset", _t1_zone2_reset),
        ("T-2  whole-field replacement", _t2_whole_field_replacement),
        ("T-3  field cap + warning", _t3_field_cap_and_warning),
        ("T-4  grid diff roundtrip", _t4_grid_diff_roundtrip),
        ("T-5  drift guard", _t5_drift_guard),
        ("T-6  refresh triggers", _t6_refresh_triggers),
        ("T-7  build order", _t7_build_order),
        ("T-8  hard gate cascade", _t8_hard_gate_cascade),
        ("T-9  cascade exhaustion", _t9_cascade_exhaustion),
        ("T-10 promote to zone2", _t10_promote_to_zone2),
        ("T-11 dead signature", _t11_dead_signature),
        ("T-12 token counting", _t12_token_counting),
        ("T-13 snapshot roundtrip", _t13_snapshot_roundtrip),
        ("T-14 fuzz", _t14_fuzz),
        ("T-15 apply_delta partial-reject atomic",
         _t15_apply_delta_partial_reject),
        ("T-16 apply_delta bad-checksum atomic",
         _t16_apply_delta_bad_checksum),
        ("T-17 zone3 summary bounded", _t17_zone3_summary_bounded),
        ("T-18 dead sigs bounded", _t18_dead_sigs_bounded),
        ("T-19 snapshot arbitrary Any", _t19_snapshot_arbitrary_any),
        ("T-20 fallback truncation marker",
         _t20_fallback_truncation_marker),
        ("T-21 promote fail-loud", _t21_promote_fail_loud),
        ("T-22 fallback handles delta", _t22_fallback_handles_delta),
        ("T-23 apply_delta value-range reject", _t23_apply_delta_bad_value),
        ("T-24 decode verify", _t24_decode_verify),
        ("T-25 decode range validation", _t25_decode_range_validation),
        ("T-26 decode rejects bad RLE value", _t26_decode_bad_rle_value),
        ("T-27 apply_delta rejects bad old", _t27_apply_delta_bad_old),
        ("T-28 per-zone advisory cap warns", _t28_per_zone_cap_warns),
        ("T-29 objects rendered in observation", _t29_objects_rendered),
        ("T-30 fallback accepts char+count RLE", _t30_fallback_accepts_char_format),
    ]
    failed = 0
    for name, fn in tests:
        try:
            fn()
            print(f"PASS  {name}")
        except Exception as e:
            failed += 1
            print(f"FAIL  {name}: {e}")
    print()
    print(f"{len(tests) - failed}/{len(tests)} passed")
    return 0 if failed == 0 else 1


if __name__ == "__main__":
    logging.basicConfig(level=logging.WARNING,
                        format="%(levelname)s %(name)s %(message)s")
    import sys as _sys
    _sys.exit(_run_self_tests())

In [ ]:
%%writefile goa/agent/inference_layer.py
"""agent/inference_layer.py — Inference Layer (IL). v1.5.

Changelog:
  v1.5 — DIR/2026/070 (D-3 single source of truth):
         _call_transport forwards `reasoning_mode=mode` to the transport.
         Transport is sole decision point for DEEP/FAST wire params.
  v1.4 — DIR/2026/060 lineage.
  v1.3 — DIR/2026/055 lineage.

  F-2 (P2, INDUS-DIR-070 final-audit closure):
    `ask_reasoner` signature default changed from `mode="deep"` (v1.0.17
    Cell 5 L55, S-3) to `mode=IL_MODE_FAST` (== "fast") in v1.5.  This is
    a second change beyond DIR-070 Task 2's minimal-diff scope (which was
    only `_call_transport` + `reasoning_mode`).  Harmless in production:
    my_agent always passes `mode=` explicitly at the call site.  The new
    default reflects the intended fail-closed FAST posture when a caller
    omits `mode`.  Covered by new test T-IL-DEFAULT below.
"""
from __future__ import annotations
import json, logging, os
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional, Sequence

logger = logging.getLogger(__name__)
__version__ = "1.6"  # v1.6 certified upgrade (DIR-078): +call_log, CODER_CODE_CAP, build_spec

REASONER_ANSWER_CAP      = 200
REASONER_THINKING_BUDGET = 1000
IL_MODE_FAST = "fast"
IL_MODE_DEEP = "deep"

_DEFAULT_PARAMS = {"temperature": 0.0, "top_p": 1.0, "top_k": -1,
                   "presence_penalty": 0.0, "repetition_penalty": 1.0}


@dataclass
class ReasonerResult:
    observation_summary: str
    hypothesis_updates: List[str]
    instruction: str
    prediction: str
    raw: Optional[Dict[str, Any]] = None

@dataclass
class WorldModelResult:
    code: str
    notes: str = ""

@dataclass
class DiagnoseResult:
    root_cause: str
    confidence: float = 0.0
    suggested_fix: str = ""

@dataclass
class PromptPacket:
    text: str
    meta: Dict[str, Any] = field(default_factory=dict)


class InferenceLayer:
    def __init__(self, transport, *, reasoner_model=None,
                 coder_model=None, context_window=16384,
                 default_mode=IL_MODE_FAST):
        self._transport = transport
        self._reasoner_model = (reasoner_model
                                or os.environ.get("VLLM_REASONER_NAME")
                                or "qwen3.8-27b")
        self._coder_model = (coder_model
                             or os.environ.get("VLLM_CODER_NAME")
                             or "qwen3-coder-30b-a3b")
        self._context_window = int(context_window)
        self._default_mode = default_mode

    @property
    def version(self): return __version__

    def call_log(self):
        """LA-3b FIX (Indus, 2026-09-29 final logic audit): THE RUN reads
        telemetry via the IL *instance* (_il.call_log()); call_log was
        module-level only, so hasattr(_il, "call_log") was False and the
        run's fast/deep/coder counts silently read 0. Instance passthrough
        to the shared ring buffer."""
        return call_log()

    def ask_reasoner(self, prompt_packet, mode=IL_MODE_FAST, *, params=None):
        # F-2 (P2): default is IL_MODE_FAST ("fast") as of v1.5.
        # v1.0.17 Cell 5 used mode="deep" — the change is intentional and
        # documented in the module changelog.  my_agent always passes
        # `mode=` explicitly; this default governs only direct callers.
        text = getattr(prompt_packet, "text", None)
        if text is None and isinstance(prompt_packet, dict):
            text = prompt_packet.get("text", "")
        if text is None: text = str(prompt_packet)
        merged = dict(_DEFAULT_PARAMS)
        if params: merged.update(params)
        rj = self._call_transport(
            model=self._reasoner_model, system=None,
            messages=[{"role": "user", "content": text}],
            max_tokens=REASONER_ANSWER_CAP, mode=mode, params=merged,
            chat_template_kwargs={})
        return self._parse_reasoner_result(rj)

    def write_world_model(self, brief):
        rj = self._call_transport(
            model=self._coder_model, system=None,
            messages=[{"role": "user", "content": brief}],
            max_tokens=REASONER_ANSWER_CAP, mode=None,
            params=dict(_DEFAULT_PARAMS), chat_template_kwargs={})
        return WorldModelResult(code=self._extract_content(rj), notes="")

    def diagnose_failure(self, function_name, failure_trace, current_code=""):
        prompt = (f"Root-cause diagnosis for function {function_name!r}.\n"
                  f"Trace:\n{failure_trace}\nCurrent code:\n{current_code}\n"
                  "Return a short root-cause sentence.")
        rj = self._call_transport(
            model=self._coder_model, system=None,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=REASONER_ANSWER_CAP, mode=None,
            params=dict(_DEFAULT_PARAMS), chat_template_kwargs={})
        return DiagnoseResult(root_cause=self._extract_content(rj).strip())

    def _call_transport(self, *, model, system, messages, max_tokens,
                        mode, params, chat_template_kwargs):
        ctk = dict(chat_template_kwargs or {})
        call_kwargs = {
            "model": model, "system": system, "messages": list(messages),
            "max_tokens": max_tokens,
            "temperature": params["temperature"],
            "top_p": params["top_p"], "top_k": params["top_k"],
            "stop_sequences": [], "chat_template_kwargs": ctk,
            "presence_penalty": params.get("presence_penalty", 0.0),
            "repetition_penalty": params.get("repetition_penalty", 1.0),
            "reasoning_mode": mode,
        }
        _t0 = _time_mod.time()
        rj = self._transport(**call_kwargs)
        # LA-3 FIX (Indus, 2026-09-29 final logic audit): PR-1 §3.2
        # telemetry — _log_call existed but was never wired, so call_log()
        # was always empty and THE RUN's fast/deep/coder telemetry read 0.
        try:
            _u = rj.get("usage") if isinstance(rj, dict) else None
            _u = _u or {}
            _ch = rj.get("choices") if isinstance(rj, dict) else None
            _fr = None
            if _ch:
                _m0 = _ch[0]
                _fr = (_m0.get("finish_reason") if isinstance(_m0, dict)
                       else getattr(_m0, "finish_reason", None))
            _log_call(model=model, mode=mode,
                      prompt_tokens=_u.get("prompt_tokens", 0),
                      completion_tokens=_u.get("completion_tokens", 0),
                      latency_ms=(_time_mod.time() - _t0) * 1000.0,
                      finish_reason=_fr)
        except Exception:
            pass  # fail-open telemetry
        return rj

    @staticmethod
    def _extract_content(rj):
        try:
            choices = rj.get("choices") if isinstance(rj, dict) else rj.choices
            msg = choices[0].get("message") if isinstance(choices[0], dict) \
                  else choices[0].message
            return msg.get("content", "") if isinstance(msg, dict) \
                   else getattr(msg, "content", "")
        except Exception:
            return ""

    def _parse_reasoner_result(self, rj):
        content = self._extract_content(rj)
        try: obj = json.loads(content)
        except Exception: obj = {}
        return ReasonerResult(
            observation_summary=str(obj.get("observation_summary", "")),
            hypothesis_updates=list(obj.get("hypothesis_updates", []) or []),
            instruction=str(obj.get("instruction", content.strip())),
            prediction=str(obj.get("prediction", "")),
            raw=rj if isinstance(rj, dict) else None)


def _t_il_rm_1():
    spy = []
    def _fake(**kw):
        spy.append(kw.get("reasoning_mode"))
        return {"choices": [{"message": {"content": "{}"},
                             "finish_reason": "stop"}],
                "usage": {"completion_tokens": 1}}
    il = InferenceLayer(transport=_fake)
    il.ask_reasoner({"text": "x"}, mode="deep")
    assert spy and spy[-1] == "deep"

def _t_il_rm_2():
    spy = []
    def _fake(**kw):
        spy.append(kw.get("reasoning_mode"))
        return {"choices": [{"message": {"content": "{}"},
                             "finish_reason": "stop"}],
                "usage": {"completion_tokens": 1}}
    il = InferenceLayer(transport=_fake)
    il.ask_reasoner({"text": "x"}, mode="fast")
    assert spy and spy[-1] == "fast"

def _t_il_rm_3():
    spy = []
    def _fake(**kw):
        spy.append(kw.get("reasoning_mode"))
        return {"choices": [{"message": {"content": "```python\npass\n```"},
                             "finish_reason": "stop"}],
                "usage": {"completion_tokens": 1}}
    il = InferenceLayer(transport=_fake)
    il.write_world_model("dummy"); assert spy and spy[-1] is None
    il.diagnose_failure("plan", "trace", "code"); assert spy[-1] is None

def _t_il_env_coder():
    os.environ["VLLM_CODER_NAME"] = "custom-coder-x"
    try:
        il = InferenceLayer(transport=lambda **kw: {})
        assert il._coder_model == "custom-coder-x"
    finally:
        os.environ.pop("VLLM_CODER_NAME", None)


def _t_il_default():
    """F-2 (P2) explicit coverage: signature default is IL_MODE_FAST."""
    import inspect
    sig = inspect.signature(InferenceLayer.ask_reasoner)
    assert sig.parameters["mode"].default == IL_MODE_FAST
    assert IL_MODE_FAST == "fast"
    # Behavioural check: caller omits mode → transport sees "fast"
    spy = []
    def _fake(**kw):
        spy.append(kw.get("reasoning_mode"))
        return {"choices": [{"message": {"content": "{}"},
                             "finish_reason": "stop"}],
                "usage": {"completion_tokens": 1}}
    il = InferenceLayer(transport=_fake)
    il.ask_reasoner({"text": "x"})   # no mode kwarg
    assert spy and spy[-1] == "fast", spy


def _run_self_tests():
    tests = [("T-IL-RM-1", _t_il_rm_1), ("T-IL-RM-2", _t_il_rm_2),
             ("T-IL-RM-3", _t_il_rm_3), ("T-IL-ENV", _t_il_env_coder),
             ("T-IL-DEFAULT", _t_il_default)]
    failed = 0
    for name, fn in tests:
        try: fn(); print(f"PASS  {name}")
        except Exception as e:
            failed += 1; print(f"FAIL  {name}: {type(e).__name__}: {e}")
    print(f"\n{len(tests) - failed}/{len(tests)} passed")
    return 0 if failed == 0 else 1


if __name__ == "__main__":
    import sys
    sys.exit(_run_self_tests())


# =========================================================================
# v1.6 ADDITIONS — PR-1 (DIR-2026-076 §3): call_log + CODER_CODE_CAP +
# spec-builder. Base v1.5 above is UNCHANGED (certified).
# =========================================================================
import threading as _threading
import time as _time_mod

_CALL_LOG = []
_CALL_LOG_CAP = 5000
_CALL_LOG_LOCK = _threading.Lock()

# PR-1 §3.3: hard generation cap for coder output.
CODER_CODE_CAP = 2048
CODER_CODE_CAP_EXCEEDED = "coder_code_cap_exceeded"

_SPEC_TEMPLATE = """You are writing Python that will be executed in a sandbox.
TASK: {task}
HYPOTHESIS: {hypothesis}
REPLAY_EVIDENCE:
{replay_evidence}
IO_CONTRACT:
{io_contract}
CONSTRAINTS:
{constraints}
Write the function {entry_signature}. Be conservative: prefer
conservative-interpretation. List your assumptions as comments.
Return ONLY fenced Python code."""


def _log_call(model, mode, prompt_tokens, completion_tokens,
              latency_ms, finish_reason):
    """Fail-open telemetry append (PR-1 §3.2). Never breaks inference."""
    try:
        with _CALL_LOG_LOCK:
            _CALL_LOG.append({
                "ts": _time_mod.time(), "model": model, "mode": mode,
                "prompt_tokens": prompt_tokens,
                "completion_tokens": completion_tokens,
                "latency_ms": latency_ms, "finish_reason": finish_reason,
            })
            if len(_CALL_LOG) > _CALL_LOG_CAP:
                del _CALL_LOG[: len(_CALL_LOG) - _CALL_LOG_CAP]
    except Exception:
        pass


def call_log() -> list:
    """Shallow copy of the telemetry ring buffer (caller-safe)."""
    with _CALL_LOG_LOCK:
        return list(_CALL_LOG)


def build_spec(task_ctx: dict) -> str:
    """PR-1 §3.4: typed SPEC prompt from task_ctx (frozen template §4).

    Keys: task, hypothesis, replay_evidence, io_contract, constraints,
    alternatives (2-4), mode ("action"|"world"), entry_signature.
    """
    entry = task_ctx.get("entry_signature",
                         "next_action(frame, history)" if task_ctx.get("mode") != "world"
                         else "next_grid(state, action)")
    alts = task_ctx.get("alternatives") or []
    ctx = dict(task_ctx)
    ctx.setdefault("task", "")
    ctx.setdefault("hypothesis", "")
    ctx.setdefault("replay_evidence", "(none)")
    ctx.setdefault("io_contract", "pure function; no IO")
    ctx.setdefault("constraints", "stdlib + numpy only; deterministic")
    ctx.setdefault("entry_signature", entry)
    spec = _SPEC_TEMPLATE.format(**ctx)
    if alts:
        spec += "\nADMISSIBLE_ALTERNATIVES:\n" + "\n".join(
            f"- {a}" for a in alts[:4])
    return spec


In [ ]:
%%writefile goa/agent/grid_text.py
"""agent/grid_text.py — Grid → Text converter. v1.0.8.

Pure module: stdlib only, no I/O, no threads, no print(), deterministic.
Converts ARC-AGI-3 engine frames (list of 64x64 int grids) into compact
observation text for the Reasoner.

Changelog: v1.0 initial (DIR/2026/033); v1.0.1 (DIR/035); v1.0.2-v1.0.4
audit F-01..F-08, H-2, M-2, L-2; v1.0.5 R-1..R-7; v1.0.6 mask_scope;
v1.0.7 T-29 recolor -> move fixture; v1.0.8 T-29 assertions corrected
to match the actual code contract (r5c-only for intersects; multi-row
bbox form for contains).

Interop: rle_anchor(grid)["checksum"] == md5(concat(row bytes)) — same
algorithm as context_manager.GridDiffEncoder._compute_checksum.
"""
from __future__ import annotations
import hashlib, json, logging, operator
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Sequence, Set, Tuple

logger = logging.getLogger(__name__)

__all__ = [
    "PALETTE", "DIGIT_TO_CHAR", "DIGIT_TO_NAME", "DIGIT_TO_HEX",
    "CHAR_TO_DIGIT", "GRID_DIM", "MAX_REGION_CELLS", "DIFF_CELL_CAP",
    "MAX_TEXT_CHARS", "LEGEND_MAX_CHARS", "MAX_SUMMARY_OBJECTS",
    "GridTextError", "RegionTooLargeError", "DimensionMismatchError",
    "InvalidGridError", "InvalidArgumentError",
    "ObjectRec", "GridTextConverter",
    "legend_text", "extract_objects", "adjacency", "cell_diff",
    "object_delta", "auto_hud_mask", "rle_anchor", "decode_rle_anchor",
    "render_region", "object_summary_text", "diff_text",
]

PALETTE = (
    (0, "white", "#FFFFFF", "W"), (1, "off-white", "#CCCCCC", "w"),
    (2, "light gray", "#999999", "g"), (3, "gray", "#666666", "G"),
    (4, "charcoal", "#333333", "c"), (5, "black", "#000000", "B"),
    (6, "magenta", "#E53AA3", "M"), (7, "pink", "#FF7BCC", "P"),
    (8, "red", "#F93C31", "R"), (9, "blue", "#1E93FF", "b"),
    (10, "light blue", "#88D8F1", "S"), (11, "yellow", "#FFDC00", "Y"),
    (12, "orange", "#FF851B", "O"), (13, "maroon", "#921231", "r"),
    (14, "green", "#4FCC30", "N"), (15, "purple", "#A356D6", "p"),
)
DIGIT_TO_CHAR = {d: c for d, _, _, c in PALETTE}
DIGIT_TO_NAME = {d: n for d, n, _, _ in PALETTE}
DIGIT_TO_HEX = {d: h for d, _, h, _ in PALETTE}
CHAR_TO_DIGIT = {c: d for d, _, _, c in PALETTE}

GRID_DIM = 64
MAX_REGION_CELLS = 576
DIFF_CELL_CAP = 400
MAX_TEXT_CHARS = 4000
LEGEND_MAX_CHARS = 400
MAX_SUMMARY_OBJECTS = 40
_MASK_SCOPES = ("intersects", "contains")
_TRUNC = "\n...[renderer capped]"


class GridTextError(RuntimeError):
    """Base for typed failures."""


class RegionTooLargeError(GridTextError):
    """Crop exceeds MAX_REGION_CELLS."""


class DimensionMismatchError(GridTextError):
    """Shape mismatch."""


class InvalidGridError(GridTextError):
    """Non-rectangular / zero-width / OOR grid, or malformed anchor."""


class InvalidArgumentError(GridTextError):
    """Bad caller argument (connectivity, OOB region, scope)."""


def _validate_grid(grid: Any, name: str = "grid") -> None:
    if not isinstance(grid, (list, tuple)) or len(grid) == 0:
        raise InvalidGridError(f"{name}: non-empty sequence required")
    w0 = len(grid[0])
    if w0 == 0:
        raise InvalidGridError(f"{name}: zero-width rows invalid")
    for r, row in enumerate(grid):
        if len(row) != w0:
            raise InvalidGridError(
                f"{name}: row {r} len {len(row)} != {w0}")
        for c, v in enumerate(row):
            if isinstance(v, bool):
                raise InvalidGridError(
                    f"{name}: ({r},{c}) bool not int 0..15")
            try:
                iv = operator.index(v)
            except TypeError:
                raise InvalidGridError(
                    f"{name}: ({r},{c}) {v!r} not integer")
            if not 0 <= iv <= 15:
                raise InvalidGridError(
                    f"{name}: ({r},{c})={iv} not in 0..15")


def _md5_grid(grid) -> str:
    h = hashlib.md5()
    for row in grid:
        h.update(bytes(row))
    return h.hexdigest()


def _detect_background(grid) -> int:
    c: Counter = Counter()
    for row in grid:
        for v in row:
            c[v] += 1
    if not c:
        return 0
    best_d, best_c = 0, -1
    for d in sorted(c.keys()):
        if c[d] > best_c:
            best_c, best_d = c[d], d
    return int(best_d)


def _canonical_cells(cells, color_digit: int) -> str:
    if not cells:
        return f"{color_digit}\n"
    mr = min(r for r, _ in cells)
    mc = min(c for _, c in cells)
    pts = sorted((r - mr, c - mc) for r, c in cells)
    return f"{color_digit}\n" + "\n".join(f"{r},{c}" for r, c in pts)


def _compute_ohash(cells, color_digit: int) -> str:
    return hashlib.md5(_canonical_cells(cells, color_digit)
                       .encode("utf-8")).hexdigest()


def _bbox_str(b) -> str:
    r0, r1, c0, c1 = b
    if r0 == r1 and c0 == c1:
        return f"r{r0}c{c0}"
    if r0 == r1:
        return f"r{r0}c{c0}-{c1}"
    if c0 == c1:
        return f"r{r0}-{r1}c{c0}"
    return f"r{r0}-{r1}c{c0}-{c1}"


def _key(ohash: str, b) -> str:
    return f"{ohash}|{int(b[0])},{int(b[1])},{int(b[2])},{int(b[3])}"


def _hard_cap(text: str) -> str:
    if len(text) <= MAX_TEXT_CHARS:
        return text
    keep = MAX_TEXT_CHARS - len(_TRUNC)
    return (text[:keep] + _TRUNC) if keep > 0 else _TRUNC[:MAX_TEXT_CHARS]


def _bbox_center(b) -> Tuple[float, float]:
    r0, r1, c0, c1 = b
    return ((r0 + r1) / 2.0, (c0 + c1) / 2.0)


def _bbox_fully_masked(b, mask_rows, mask_cols) -> bool:
    r0, r1, c0, c1 = b
    mr, mc = set(mask_rows), set(mask_cols)
    return (all(r in mr for r in range(r0, r1 + 1))
            or all(c in mc for c in range(c0, c1 + 1)))


def _bbox_intersects_mask(b, mask_rows, mask_cols) -> bool:
    r0, r1, c0, c1 = b
    mr, mc = set(mask_rows), set(mask_cols)
    return (any(r in mr for r in range(r0, r1 + 1))
            or any(c in mc for c in range(c0, c1 + 1)))


def _entry_is_masked(b, mr, mc, scope: str) -> bool:
    if not mr and not mc:
        return False
    if scope == "intersects":
        return _bbox_intersects_mask(b, mr, mc)
    return _bbox_fully_masked(b, mr, mc)


def legend_text() -> str:
    parts = " ".join(f"{c}={n}" for _, n, _, c in PALETTE)
    out = (f"colors: {parts} "
           "(names are palette labels only, not game meanings)")
    return out[:LEGEND_MAX_CHARS]


@dataclass
class ObjectRec:
    id: int
    color_digit: int
    color_char: str
    size: int
    bbox: Tuple[int, int, int, int]
    ohash: str
    pixels: Optional[List[List[int]]] = None
    cells: List[Tuple[int, int]] = field(default_factory=list, repr=False)

    def to_dict(self) -> Dict[str, Any]:
        return {"id": self.id, "color_digit": self.color_digit,
                "color_char": self.color_char, "size": self.size,
                "bbox": list(self.bbox), "ohash": self.ohash,
                "pixels": self.pixels}


def extract_objects(grid, background=None, connectivity=4):
    if connectivity not in (4, 8):
        raise InvalidArgumentError(f"connectivity {connectivity} not 4/8")
    _validate_grid(grid)
    if background is None:
        background = _detect_background(grid)
    h, w = len(grid), len(grid[0])
    neigh = (((-1, 0), (1, 0), (0, -1), (0, 1)) if connectivity == 4
             else ((-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1),
                   (1, -1), (1, 0), (1, 1)))
    seen = [[False] * w for _ in range(h)]
    comps = []
    for r0 in range(h):
        for c0 in range(w):
            v = grid[r0][c0]
            if v == background or seen[r0][c0]:
                continue
            stack = [(r0, c0)]; seen[r0][c0] = True; cells = []
            while stack:
                r, c = stack.pop(); cells.append((r, c))
                for dr, dc in neigh:
                    nr, nc = r + dr, c + dc
                    if (0 <= nr < h and 0 <= nc < w
                            and not seen[nr][nc]
                            and grid[nr][nc] == v):
                        seen[nr][nc] = True; stack.append((nr, nc))
            comps.append((v, cells))
    comps.sort(key=lambda vc: (min(r for r, _ in vc[1]),
                               min(c for _, c in vc[1])))
    objs = []
    for i, (v, cells) in enumerate(comps, start=1):
        rs = [r for r, _ in cells]; cs = [c for _, c in cells]
        bbox = (min(rs), max(rs), min(cs), max(cs))
        pix = ([[r, c] for r, c in sorted(cells)]
               if len(cells) <= 16 else None)
        objs.append(ObjectRec(i, int(v), DIGIT_TO_CHAR[int(v)],
                              len(cells), bbox,
                              _compute_ohash(cells, int(v)), pix,
                              list(cells)))
    return objs


def adjacency(objects, grid):
    _validate_grid(grid, "adjacency.grid")
    h, w = len(grid), len(grid[0])
    owner = [[0] * w for _ in range(h)]
    for o in objects:
        for r, c in o.cells:
            owner[r][c] = o.id
    pairs = set()
    for r in range(h):
        for c in range(w):
            a = owner[r][c]
            if a == 0:
                continue
            if c + 1 < w:
                b = owner[r][c + 1]
                if b and b != a:
                    pairs.add((min(a, b), max(a, b)))
            if r + 1 < h:
                b = owner[r + 1][c]
                if b and b != a:
                    pairs.add((min(a, b), max(a, b)))
    return [list(p) for p in sorted(pairs)]


def cell_diff(old_grid, new_grid, mask_rows=(), mask_cols=()):
    _validate_grid(old_grid, "cell_diff.old_grid")
    _validate_grid(new_grid, "cell_diff.new_grid")
    if (len(old_grid) != len(new_grid)
            or len(old_grid[0]) != len(new_grid[0])):
        raise DimensionMismatchError(
            f"cell_diff: {len(old_grid)}x{len(old_grid[0])} vs "
            f"{len(new_grid)}x{len(new_grid[0])}")
    h, w = len(old_grid), len(old_grid[0])
    mr, mc = set(mask_rows), set(mask_cols)
    out = []
    for r in range(h):
        if r in mr:
            continue
        for c in range(w):
            if c in mc:
                continue
            if old_grid[r][c] != new_grid[r][c]:
                out.append((r, c, int(old_grid[r][c]),
                            int(new_grid[r][c])))
    return out


def _render_cell_runs(changes) -> List[str]:
    bp = defaultdict(lambda: defaultdict(list))
    for r, c, o, n in changes:
        bp[(o, n)][r].append(c)
    lines = []
    for (o, n) in sorted(bp.keys()):
        parts = []
        for r in sorted(bp[(o, n)].keys()):
            cols = sorted(bp[(o, n)][r]); i = 0
            while i < len(cols):
                j = i
                while j + 1 < len(cols) and cols[j + 1] == cols[j] + 1:
                    j += 1
                a, b = cols[i], cols[j]
                parts.append(f"r{r}c{a}" if a == b else f"r{r}c{a}-{b}")
                i = j + 1
        lines.append(f"{DIGIT_TO_CHAR[o]}->{DIGIT_TO_CHAR[n]}: "
                     + ",".join(parts))
    return lines


def _pair_nearest(olds, news):
    pairs, used_o, used_n = [], set(), set()
    for n in sorted(news, key=lambda x: x.id):
        best_o, best_d = None, None
        nc = _bbox_center(n.bbox)
        for o in sorted(olds, key=lambda x: x.id):
            if o.id in used_o:
                continue
            oc = _bbox_center(o.bbox)
            d = (oc[0] - nc[0]) ** 2 + (oc[1] - nc[1]) ** 2
            if best_d is None or d < best_d:
                best_d, best_o = d, o
        if best_o is not None:
            used_o.add(best_o.id); used_n.add(n.id)
            pairs.append((best_o, n))
    return pairs


def object_delta(old_objects, new_objects):
    old_h = defaultdict(list); new_h = defaultdict(list)
    for o in old_objects:
        old_h[o.ohash].append(o)
    for n in new_objects:
        new_h[n.ohash].append(n)
    used_o, used_n, moved = set(), set(), []
    for h, news in new_h.items():
        for o, n in _pair_nearest(old_h.get(h, []), news):
            used_o.add(o.id); used_n.add(n.id)
            if o.bbox != n.bbox:
                moved.append([n.id, n.ohash,
                              list(o.bbox), list(n.bbox)])
    rem_o = [o for o in old_objects if o.id not in used_o]
    rem_n = [n for n in new_objects if n.id not in used_n]
    by_bbox = defaultdict(list)
    for o in rem_o:
        by_bbox[o.bbox].append(o)
    recolored = []
    for n in rem_n:
        for o in by_bbox.get(n.bbox, []):
            if set(o.cells) == set(n.cells) and \
                    o.color_digit != n.color_digit:
                used_o.add(o.id); used_n.add(n.id)
                recolored.append([n.id, n.ohash,
                                  o.color_digit, n.color_digit,
                                  list(n.bbox)])
                break
    new_l = [[n.id, n.ohash, list(n.bbox), n.color_char, n.size]
             for n in new_objects if n.id not in used_n]
    gone_l = [[o.id, o.ohash, list(o.bbox)]
              for o in old_objects if o.id not in used_o]
    return {"moved": moved, "recolored": recolored,
            "new": new_l, "gone": gone_l}


def auto_hud_mask(frame_history, threshold=0.5):
    if not frame_history or len(frame_history) < 2:
        return (), ()
    for i, f in enumerate(frame_history):
        _validate_grid(f, f"frame_history[{i}]")
    fh, fw = len(frame_history[0]), len(frame_history[0][0])
    for i in range(1, len(frame_history)):
        fi = frame_history[i]
        if len(fi) != fh or len(fi[0]) != fw:
            raise DimensionMismatchError(
                f"frame {i} shape {len(fi)}x{len(fi[0])} != {fh}x{fw}")
    total = len(frame_history) - 1
    rh, ch = Counter(), Counter()
    for i in range(total):
        a, b = frame_history[i], frame_history[i + 1]
        rc = [False] * fh; cc = [False] * fw
        for r in range(fh):
            for c in range(fw):
                if a[r][c] != b[r][c]:
                    rc[r] = True; cc[c] = True
        for r in range(fh):
            if rc[r]:
                rh[r] += 1
        for c in range(fw):
            if cc[c]:
                ch[c] += 1
    return (tuple(sorted(r for r, k in rh.items() if k / total > threshold)),
            tuple(sorted(c for c, k in ch.items() if k / total > threshold)))


def rle_anchor(grid) -> Dict[str, Any]:
    _validate_grid(grid)
    rows = []
    for row in grid:
        parts = []; cur, cnt = row[0], 1
        for v in row[1:]:
            if v == cur:
                cnt += 1
            else:
                parts.append(f"{DIGIT_TO_CHAR[int(cur)]}{cnt}")
                cur, cnt = v, 1
        parts.append(f"{DIGIT_TO_CHAR[int(cur)]}{cnt}")
        rows.append(",".join(parts))
    return {"type": "full", "dim": [len(grid), len(grid[0])],
            "checksum": _md5_grid(grid), "rows": rows}


def decode_rle_anchor(anchor) -> List[List[int]]:
    grid = []
    try:
        for s in anchor["rows"]:
            row = []
            if s:
                for ch in s.split(","):
                    if not ch:
                        raise InvalidGridError("empty chunk")
                    if ch[0] not in CHAR_TO_DIGIT:
                        raise InvalidGridError(f"char {ch[0]!r}")
                    row.extend([CHAR_TO_DIGIT[ch[0]]] * int(ch[1:]))
            grid.append(row)
    except (KeyError, ValueError, IndexError) as e:
        raise InvalidGridError(f"malformed anchor ({e})") from e
    _validate_grid(grid, "decode.result")
    if "dim" in anchor:
        got = [len(grid), len(grid[0]) if grid else 0]
        if got != list(anchor["dim"]):
            raise InvalidGridError(
                f"dim={anchor['dim']} but decoded {got}")
    return grid


def render_region(grid, r0, r1, c0, c1):
    _validate_grid(grid)
    h, w = len(grid), len(grid[0])
    if not (0 <= r0 <= r1 < h) or not (0 <= c0 <= c1 < w):
        raise InvalidArgumentError(
            f"region r{r0}-{r1} c{c0}-{c1} OOB in {h}x{w}")
    rh, rw = r1 - r0 + 1, c1 - c0 + 1
    if rh * rw > MAX_REGION_CELLS:
        raise RegionTooLargeError(f"{rh*rw} > {MAX_REGION_CELLS}")
    sub = [list(row[c0:c1 + 1]) for row in grid[r0:r1 + 1]]
    lines = [f"region r{r0}-{r1} c{c0}-{c1} dim={rh}x{rw} "
             f"checksum={_md5_grid(sub)}"]
    for i, row in enumerate(sub):
        lines.append(f"{r0 + i} " + "".join(DIGIT_TO_CHAR[v] for v in row))
    return _hard_cap("\n".join(lines))


def object_summary_text(packet) -> str:
    objs = packet.get("objects", [])
    top = sorted(objs, key=lambda o: (-o["size"], o["id"]))[
        :MAX_SUMMARY_OBJECTS]
    lines = []
    for o in top:
        r0, r1, c0, c1 = o["bbox"]
        lines.append(f"#{o['id']} {DIGIT_TO_NAME[o['color_digit']]} "
                     f"{c1-c0+1}x{r1-r0+1} "
                     f"r{r0}-{r1} c{c0}-{c1} h={o['ohash'][:6]}")
    if len(objs) > MAX_SUMMARY_OBJECTS:
        k = len(objs) - MAX_SUMMARY_OBJECTS
        lines.append(f"+{k} more objects "
                     f"(total {len(objs)}; needs_full)")
    return _hard_cap("\n".join(lines))


def _object_delta_lines(delta, mr=(), mc=(), scope="intersects"):
    out = []
    for _id, _h, fb, tb in delta.get("moved", []):
        if _entry_is_masked(fb, mr, mc, scope) \
                or _entry_is_masked(tb, mr, mc, scope):
            continue
        out.append(f"obj#{_id} moved {_bbox_str(tuple(fb))} -> "
                   f"{_bbox_str(tuple(tb))}")
    for _id, _h, od, nd, bb in delta.get("recolored", []):
        if _entry_is_masked(bb, mr, mc, scope):
            continue
        out.append(f"obj#{_id} recolored "
                   f"{DIGIT_TO_CHAR[od]}->{DIGIT_TO_CHAR[nd]}")
    for _id, _h, bb, ch, _sz in delta.get("new", []):
        if _entry_is_masked(bb, mr, mc, scope):
            continue
        r0, r1, c0, c1 = bb
        out.append(f"new obj#{_id} ({ch}, {c1-c0+1}x{r1-r0+1}, "
                   f"r{r0}c{c0})")
    for _id, _h, bb in delta.get("gone", []):
        if _entry_is_masked(bb, mr, mc, scope):
            continue
        out.append(f"gone: obj#{_id}")
    return out


def _explained_cells(delta, cur, prev):
    s = set()
    for _id, h, fb, tb in delta.get("moved", []):
        for cell in prev.get(_key(h, fb), []):
            s.add((cell[0], cell[1]))
        for cell in cur.get(_key(h, tb), []):
            s.add((cell[0], cell[1]))
    for _id, h, _od, _nd, bb in delta.get("recolored", []):
        for cell in cur.get(_key(h, bb), []):
            s.add((cell[0], cell[1]))
    for _id, h, bb, _ch, _sz in delta.get("new", []):
        for cell in cur.get(_key(h, bb), []):
            s.add((cell[0], cell[1]))
    for _id, h, bb in delta.get("gone", []):
        for cell in prev.get(_key(h, bb), []):
            s.add((cell[0], cell[1]))
    return s


def diff_text(packet) -> str:
    delta = packet.get("object_delta", {})
    mr = packet.get("_mask_rows", ())
    mc = packet.get("_mask_cols", ())
    scope = packet.get("_mask_scope", "intersects")
    obj_lines = _object_delta_lines(delta, mr, mc, scope)
    if len(obj_lines) > MAX_SUMMARY_OBJECTS:
        total = len(obj_lines)
        k = total - MAX_SUMMARY_OBJECTS
        obj_lines = obj_lines[:MAX_SUMMARY_OBJECTS]
        obj_lines.append(
            f"+{k} more objects (total {total}; needs_full)")
    lines = list(obj_lines)
    raw = packet.get("cell_changes_raw")
    if obj_lines and raw is not None:
        expl = _explained_cells(
            delta, packet.get("_obj_cells_cur", {}) or {},
            packet.get("_obj_cells_prev", {}) or {})
        res = [tuple(ch) for ch in raw
               if (ch[0], ch[1]) not in expl]
        lines.extend(_render_cell_runs(res))
    elif packet.get("cell_diff"):
        lines.extend(packet["cell_diff"])
    return _hard_cap("\n".join(lines))


class GridTextConverter:
    """Stateful converter.  mask_scope: "intersects" (default, suppresses
    any entry whose bbox touches a mask) or "contains" (suppresses only
    fully-masked bboxes)."""

    def __init__(self, mask_rows=(), mask_cols=(), connectivity=4,
                 mask_scope="intersects"):
        if mask_scope not in _MASK_SCOPES:
            raise InvalidArgumentError(
                f"mask_scope {mask_scope!r} not in {_MASK_SCOPES}")
        self._mask_rows = tuple(int(r) for r in mask_rows)
        self._mask_cols = tuple(int(c) for c in mask_cols)
        self._connectivity = int(connectivity)
        self._mask_scope = mask_scope
        self._version = 0
        self._prev_grid = None
        self._prev_objects = None
        self._prev_cells: Dict[str, List[List[int]]] = {}
        self._last_packet = None

    def set_frame(self, frame):
        if not frame:
            raise InvalidArgumentError("frame must contain a grid")
        last = [list(row) for row in frame[-1]]
        _validate_grid(last, "frame[-1]")
        self._version += 1
        bg = _detect_background(last)
        objects = extract_objects(
            last, background=bg, connectivity=self._connectivity)
        adj = adjacency(objects, last)
        cells_cur: Dict[str, List[List[int]]] = {}
        for o in objects:
            cells_cur[_key(o.ohash, o.bbox)] = [[r, c]
                                                for r, c in o.cells]
        needs_full = False
        raw, cdl = [], []
        delta = {"moved": [], "recolored": [], "new": [], "gone": []}
        if self._prev_grid is not None:
            if (len(self._prev_grid) != len(last)
                    or len(self._prev_grid[0]) != len(last[0])):
                needs_full = True
            else:
                raw = cell_diff(self._prev_grid, last,
                                self._mask_rows, self._mask_cols)
                if len(raw) > DIFF_CELL_CAP:
                    needs_full = True; raw = []
                else:
                    cdl = _render_cell_runs(raw)
                if self._prev_objects is not None:
                    delta = object_delta(self._prev_objects, objects)
        if len(objects) > MAX_SUMMARY_OBJECTS:
            needs_full = True
        dc = (len(delta["moved"]) + len(delta["recolored"])
              + len(delta["new"]) + len(delta["gone"]))
        if dc > MAX_SUMMARY_OBJECTS:
            needs_full = True
        if needs_full:
            raw, cdl = [], []
        packet = {
            "version": self._version, "checksum": _md5_grid(last),
            "dim": [len(last), len(last[0])], "bg": bg,
            "masked": f"rows={len(self._mask_rows)} "
                      f"cols={len(self._mask_cols)}",
            "objects": [o.to_dict() for o in objects],
            "adjacency": adj, "cell_diff": cdl, "object_delta": delta,
            "cell_changes_raw": [list(ch) for ch in raw],
            "needs_full": needs_full, "legend": legend_text(),
            "_obj_cells_cur": cells_cur,
            "_obj_cells_prev": self._prev_cells,
            "_mask_rows": list(self._mask_rows),
            "_mask_cols": list(self._mask_cols),
            "_mask_scope": self._mask_scope,
        }
        self._prev_grid = last; self._prev_objects = objects
        self._prev_cells = cells_cur; self._last_packet = packet
        return packet

    def object_summary_text(self):
        return object_summary_text(self._last_packet or {})

    def diff_text(self):
        return diff_text(self._last_packet or {})

    def last_packet(self):
        return self._last_packet


# ==========================================================================
# Inline self-tests (v1.0.8, 29 tests)
# ==========================================================================

def _eg(v=0):
    return [[v] * GRID_DIM for _ in range(GRID_DIM)]


def _pb(g, r0, c0, h, w, v):
    for r in range(r0, r0 + h):
        for c in range(c0, c0 + w):
            g[r][c] = v


def _t1():
    assert len(PALETTE) == 16
    assert [p[0] for p in PALETTE] == list(range(16))
    assert len({p[3] for p in PALETTE}) == 16
    assert PALETTE[0] == (0, "white", "#FFFFFF", "W")
    assert PALETTE[15] == (15, "purple", "#A356D6", "p")


def _t2():
    L = legend_text()
    assert len(L) <= LEGEND_MAX_CHARS
    for _d, n, _hx, ch in PALETTE:
        assert f"{ch}={n}" in L
    assert "palette labels only" in L


def _t3():
    g = _eg(0)
    _pb(g, 1, 1, 1, 3, 5); _pb(g, 10, 10, 2, 2, 8)
    _pb(g, 20, 20, 1, 1, 15)
    o = extract_objects(g, 0)
    assert [x.id for x in o] == [1, 2, 3]
    assert o[0].bbox == (1, 1, 1, 3) and o[0].size == 3
    assert o[1].bbox == (10, 11, 10, 11) and o[1].size == 4
    assert o[2].bbox == (20, 20, 20, 20)


def _t4():
    g = [[0] * 4 for _ in range(4)]
    g[1][1] = 5; g[2][2] = 5
    assert _detect_background(g) == 0
    assert _detect_background([[0, 0, 5, 5]] * 4) == 0


def _t5():
    g = _eg(0); g[1][1] = 5; g[2][2] = 5
    assert len(extract_objects(g, 0, 4)) == 2
    assert len(extract_objects(g, 0, 8)) == 1


def _t6():
    g1 = _eg(0); _pb(g1, 1, 1, 1, 3, 5)
    o1 = extract_objects(g1, 0)[0]
    g2 = _eg(0); _pb(g2, 40, 40, 1, 3, 5)
    assert extract_objects(g2, 0)[0].ohash == o1.ohash
    g3 = _eg(0); _pb(g3, 1, 1, 1, 3, 8)
    assert extract_objects(g3, 0)[0].ohash != o1.ohash
    g4 = _eg(0); _pb(g4, 1, 1, 1, 4, 5)
    assert extract_objects(g4, 0)[0].ohash != o1.ohash


def _t7():
    g = _eg(0)
    _pb(g, 1, 1, 1, 3, 5); _pb(g, 10, 10, 5, 5, 8)
    o = extract_objects(g, 0)
    assert [x for x in o if x.size == 3][0].pixels is not None
    assert [x for x in o if x.size == 25][0].pixels is None


def _t8():
    g = _eg(0)
    _pb(g, 1, 1, 1, 2, 5); _pb(g, 1, 3, 1, 2, 8)
    _pb(g, 5, 5, 1, 1, 15)
    adj = adjacency(extract_objects(g, 0), g)
    assert [1, 2] in adj and [1, 3] not in adj and [2, 3] not in adj


def _t9():
    a = _eg(0); b = _eg(0)
    for c in range(4):
        b[0][c] = 5
    b[0][7] = 5
    for c in range(3, 6):
        b[2][c] = 5
    lines = _render_cell_runs(cell_diff(a, b))
    assert lines[0] == "W->B: r0c0-3,r0c7,r2c3-5", lines[0]
    assert "r2" not in _render_cell_runs(cell_diff(a, b, [2]))[0]
    assert "r0c7" not in _render_cell_runs(cell_diff(a, b, (), [7]))[0]


def _t10():
    a = _eg(0); _pb(a, 1, 1, 1, 3, 5)
    b = _eg(0); _pb(b, 10, 10, 1, 3, 5)
    d = object_delta(extract_objects(a, 0), extract_objects(b, 0))
    assert d["moved"][0][2] == [1, 1, 1, 3]
    assert d["moved"][0][3] == [10, 10, 10, 12]
    a = _eg(0); _pb(a, 1, 1, 1, 3, 5)
    b = _eg(0); _pb(b, 1, 1, 1, 3, 8)
    d = object_delta(extract_objects(a, 0), extract_objects(b, 0))
    assert d["recolored"][0][2] == 5 and d["recolored"][0][3] == 8
    a = _eg(0); b = _eg(0); _pb(b, 5, 5, 2, 2, 15)
    d = object_delta(extract_objects(a, 0), extract_objects(b, 0))
    assert d["new"][0][3] == "p" and d["new"][0][4] == 4
    a = _eg(0); _pb(a, 5, 5, 2, 2, 15); b = _eg(0)
    d = object_delta(extract_objects(a, 0), extract_objects(b, 0))
    assert len(d["gone"]) == 1


def _t11():
    a = _eg(0); _pb(a, 1, 1, 1, 3, 5)
    b = _eg(0); _pb(b, 10, 10, 1, 3, 5); b[40][40] = 8
    c = GridTextConverter(); c.set_frame([a]); pkt = c.set_frame([b])
    txt = c.diff_text()
    assert txt.startswith("obj#")
    assert "obj#1 moved r1c1-3 -> r10c10-12" in txt
    assert "new obj#2 (R, 1x1, r40c40)" in txt
    for p in ("W->B:", "B->W:", "W->R:"):
        assert p not in txt
    assert any(l == "B->W: r1c1-3" for l in pkt["cell_diff"])
    assert any(l == "W->B: r10c10-12" for l in pkt["cell_diff"])
    assert any(l == "W->R: r40c40" for l in pkt["cell_diff"])


def _t12():
    a = _eg(0); b = _eg(0)
    for i in range(500):
        b[i // 64][i % 64] = 5
    c = GridTextConverter(); c.set_frame([a])
    assert c.set_frame([b])["needs_full"] is True


def _t13():
    base = _eg(0); frames = []
    for i in range(10):
        g = [row[:] for row in base]; g[0][i] = 5; frames.append(g)
    rows, _ = auto_hud_mask(frames, 0.5)
    assert 0 in rows and 30 not in rows


def _t14():
    import random as _r
    rng = _r.Random(20260918)
    g = [[rng.randrange(16) for _ in range(GRID_DIM)]
         for _ in range(GRID_DIM)]
    anc = rle_anchor(g)
    assert anc["dim"] == [GRID_DIM, GRID_DIM]
    assert anc["checksum"] == _md5_grid(g)
    assert decode_rle_anchor(anc) == g


def _t15():
    g = _eg(0); _pb(g, 5, 5, 3, 4, 8)
    lines = render_region(g, 4, 8, 4, 9).split("\n")
    assert lines[0].startswith("region r4-8 c4-9 dim=5x6 checksum=")
    assert lines[1][2:] == "W" * 6
    assert lines[2][2:] == "WRRRRW"
    try:
        render_region(g, 0, 24, 0, 24)
    except RegionTooLargeError:
        pass
    else:
        raise AssertionError("expected RegionTooLargeError")


def _t16():
    a = _eg(0); _pb(a, 1, 1, 2, 2, 5)
    b = _eg(0); _pb(b, 10, 10, 2, 2, 5)
    c = GridTextConverter(); p1 = c.set_frame([a]); p2 = c.set_frame([b])
    assert p1["version"] == 1 and p2["version"] == 2
    for k in ("version", "checksum", "dim", "bg", "masked", "objects",
              "adjacency", "cell_diff", "object_delta",
              "cell_changes_raw", "needs_full", "legend"):
        assert k in p2
    assert json.loads(json.dumps(p2))["version"] == 2


def _t17():
    g = _eg(0)
    _pb(g, 1, 1, 1, 3, 15); _pb(g, 5, 5, 2, 2, 8)
    c = GridTextConverter(); c.set_frame([g])
    lines = c.object_summary_text().split("\n")
    assert lines[0].startswith("#2 red 2x2 r5-6 c5-6 h=")
    assert lines[1].startswith("#1 purple 3x1 r1-1 c1-3 h=")


def _t18():
    g = _eg(0)
    for r in range(GRID_DIM):
        for c in range(GRID_DIM):
            g[r][c] = 0 if (r + c) % 2 == 0 else 5
    try:
        render_region(g, 0, 63, 0, 63)
    except RegionTooLargeError:
        pass
    else:
        raise AssertionError("full-board must raise")
    assert len(render_region(g, 0, 11, 0, 11)) < MAX_TEXT_CHARS


def _t19():
    a = _eg(0); _pb(a, 1, 1, 2, 2, 5)
    b = _eg(0); _pb(b, 10, 10, 2, 2, 5); b[40][40] = 8
    c1 = GridTextConverter(); c1.set_frame([a]); p1 = c1.set_frame([b])
    c2 = GridTextConverter(); c2.set_frame([a]); p2 = c2.set_frame([b])
    assert json.dumps(p1, sort_keys=True) == json.dumps(p2, sort_keys=True)


def _t20():
    import random as _r
    rng = _r.Random(33)
    for _ in range(200):
        a = [[rng.randrange(16) for _ in range(32)] for _ in range(32)]
        b = [[rng.randrange(16) for _ in range(32)] for _ in range(32)]
        c = GridTextConverter(); c.set_frame([a]); pkt = c.set_frame([b])
        for o in pkt["objects"]:
            r0, r1, c0, c1 = o["bbox"]
            assert 0 <= r0 <= r1 < 32 and 0 <= c0 <= c1 < 32
        if not pkt["needs_full"]:
            for ch in pkt["cell_changes_raw"]:
                assert a[ch[0]][ch[1]] != b[ch[0]][ch[1]]


def _t21():
    import random as _r
    rng = _r.Random(35)
    g = [[rng.randrange(16) for _ in range(GRID_DIM)]
         for _ in range(GRID_DIM)]
    c = GridTextConverter(); pkt = c.set_frame([g])
    assert len(pkt["objects"]) > MAX_SUMMARY_OBJECTS
    txt = c.object_summary_text()
    assert len(txt) <= MAX_TEXT_CHARS
    assert "more objects" in txt and "needs_full" in txt
    assert pkt["needs_full"] is True


def _t22():
    import random as _r
    rng = _r.Random(41)
    dense = [[rng.randrange(16) for _ in range(GRID_DIM)]
             for _ in range(GRID_DIM)]
    pkt = GridTextConverter().set_frame([dense])
    assert pkt["needs_full"] and pkt["cell_changes_raw"] == []
    assert pkt["cell_diff"] == []
    a = _eg(0)
    b = [[rng.randrange(16) for _ in range(GRID_DIM)]
         for _ in range(GRID_DIM)]
    c2 = GridTextConverter(); c2.set_frame([a]); p2 = c2.set_frame([b])
    assert p2["needs_full"] and p2["cell_changes_raw"] == []


def _t23():
    g0 = [[0] * 4 for _ in range(3)]
    g1 = [row[:] for row in g0]; g1[0][0] = 5; g1[0][1] = 5
    g2 = [row[:] for row in g1]
    rows, _ = auto_hud_mask([g0, g1, g2], 0.5)
    assert 0 not in rows
    rows2, _ = auto_hud_mask([g0, g1, g2], 0.4)
    assert 0 in rows2


def _t24():
    a = _eg(0)
    a[1][1] = 5; a[2][1] = 5; a[2][2] = 5
    b = _eg(0)
    b[10][10] = 5; b[11][10] = 5; b[11][11] = 5; b[1][2] = 8
    c = GridTextConverter(); c.set_frame([a]); c.set_frame([b])
    assert "r1c2" in c.diff_text()


def _t25():
    a = [[0] * 4 for _ in range(4)]
    for bad in ([[0] * 5 for _ in range(4)],
                [[0] * 4 for _ in range(5)]):
        try:
            cell_diff(a, bad)
        except DimensionMismatchError:
            pass
        else:
            raise AssertionError("expected DimensionMismatchError")


def _t26():
    a = _eg(0)
    for r in (1, 2):
        for c in (1, 2):
            a[r][c] = 5
    for r in (10, 11):
        for c in (10, 11):
            a[r][c] = 5
    b = _eg(0)
    for r in (40, 41):
        for c in (40, 41):
            b[r][c] = 5
    b[10][10] = 8
    b[10][11] = 5; b[11][10] = 5; b[11][11] = 5
    c = GridTextConverter(); c.set_frame([a]); c.set_frame([b])
    assert "r10c10" in c.diff_text()


def _t27():
    for bad in ([[]], [[0, 0], [0]], [[0, 99]], [[0, -1]], "x"):
        try:
            _validate_grid(bad)
        except (InvalidGridError, TypeError):
            pass
        else:
            raise AssertionError(f"expected error on {bad!r}")
    for bad in ([[True, 0], [0, 0]], [[1.0, 0], [0, 0]]):
        try:
            _validate_grid(bad)
        except InvalidGridError:
            pass
        else:
            raise AssertionError("bool/float cells must be rejected")
    good = [[0, 0], [0, 0]]
    cases = [
        lambda: cell_diff(good, [[0, 0], [0]]),
        lambda: cell_diff([[0, 99], [0, 0]], good),
        lambda: adjacency([], [[0, 0], [0]]),
        lambda: auto_hud_mask([[[0, 0]], [[]]], 0.5),
        lambda: decode_rle_anchor({"type": "full", "dim": [1, 2],
                                   "checksum": "x", "rows": ["W1"]}),
        lambda: extract_objects([[0, 0], [0, 0]], connectivity=5),
        lambda: render_region([[0, 0], [0, 0]], 5, 6, 0, 1),
    ]
    for call in cases:
        try:
            call()
        except GridTextError:
            pass
        else:
            raise AssertionError("expected typed GridTextError")
    try:
        auto_hud_mask([[[0, 0]], [[0, 0, 0]]], 0.5)
    except DimensionMismatchError:
        pass
    else:
        raise AssertionError("shape mismatch must raise")


def _t28():
    a = _eg(0); a[1][1] = 5; a[1][2] = 5
    c = GridTextConverter(mask_rows=[1]); c.set_frame([a])
    b = [row[:] for row in a]; b[1][1] = 8
    c.set_frame([b])
    assert "r1c" not in c.diff_text()


def _t29():
    """R-2 v1.0.8: two-part test corrected.
       Part 1: 'intersects' scope suppresses the OBJECT line, but rows
       outside the mask (r4, r6) legitimately appear as residual cell
       runs.  Only the masked row coordinate (r5c) must be absent.
       Part 2: 'contains' scope renders the moved line; the fixture
       bbox spans 3 rows so the multi-row form 'r4-6c10-11' is what
       _bbox_str actually returns.
    """
    a = _eg(0)
    for r in (4, 5, 6):
        for c in (10, 11):
            a[r][c] = 5
    b = _eg(0)
    for r in (20, 21, 22):
        for c in (10, 11):
            b[r][c] = 5

    # Part 1: default 'intersects'.
    c1 = GridTextConverter(mask_rows=[5])
    c1.set_frame([a]); c1.set_frame([b])
    t1 = c1.diff_text()
    assert "r5c" not in t1, f"masked row leaked: {t1}"
    assert "obj#" not in t1, f"object line should be suppressed: {t1}"

    # Part 2: 'contains'.
    c2 = GridTextConverter(mask_rows=[5], mask_scope="contains")
    c2.set_frame([a]); c2.set_frame([b])
    t2 = c2.diff_text()
    assert "moved" in t2, t2
    assert "r4-6c10-11" in t2, t2
    assert "r20-22c10-11" in t2, t2


def _run_self_tests() -> int:
    tests = [
        ("T-1 palette", _t1), ("T-2 legend", _t2),
        ("T-3 objects", _t3), ("T-4 background", _t4),
        ("T-5 connectivity", _t5), ("T-6 ohash", _t6),
        ("T-7 pixels", _t7), ("T-8 adjacency", _t8),
        ("T-9 cell_diff", _t9), ("T-10 object_delta", _t10),
        ("T-11 priority", _t11), ("T-12 diff cap", _t12),
        ("T-13 HUD mask", _t13), ("T-14 RLE", _t14),
        ("T-15 region", _t15), ("T-16 pipeline", _t16),
        ("T-17 formats", _t17), ("T-18 H-5", _t18),
        ("T-19 determinism", _t19), ("T-20 fuzz", _t20),
        ("T-21 obj cap", _t21), ("T-22 needs_full clears raw", _t22),
        ("T-23 HUD transitions", _t23), ("T-24 bbox residual", _t24),
        ("T-25 dim mismatch", _t25),
        ("T-26 dup-ohash", _t26), ("T-27 validation", _t27),
        ("T-28 mask contains", _t28),
        ("T-29 mask intersects / contains (v1.0.8)", _t29),
    ]
    failed = 0
    for name, fn in tests:
        try:
            fn()
            logging.info("PASS  %s", name)
        except Exception as e:
            failed += 1
            logging.info("FAIL  %s: %s: %s", name, type(e).__name__, e)
    logging.info("")
    logging.info("%d/%d passed", len(tests) - failed, len(tests))
    return 0 if failed == 0 else 1


if __name__ == "__main__":
    import sys as _sys
    logging.basicConfig(level=logging.INFO,
                        format="%(message)s", stream=_sys.stdout)
    _sys.exit(_run_self_tests())

In [ ]:
%%writefile goa/agent/action_pipeline.py
"""agent/action_pipeline.py — Action Pipeline + Delta Comparator. v1.0.5.

Two-tier watchdog: no-op gate + debounced object-level divergence, with
patch trigger and premise-failure abort.

Sources: DIR/2026/046 (directive), 043/044/045 (frozen constants),
DIR/2026/047 (R-series audit).

Changelog:
  v1.0    initial — DIR/2026/046.
  v1.0.1..v1.0.4  static-audit closure (see prior changelogs).
  v1.0.5  DIR/2026/047 R-series closure:
     R-F1 (HIGH) Delta packet was diffed against _prev_objects,
          which during a persistent divergence is already the
          diverging state.  Both frames identical -> empty packet.
          New field _last_stable_objects tracks the last matching
          frame's objects; survival and premise-failure branches
          diff against it (fallback _prev_objects).  New T-22
          exercises a persistent stable-jump divergence.
     L-1 (LOW)  step() resets _retry_count when the incoming action
          differs from the previously dispatched one.  _retry_history
          capped at RETRY_HISTORY_CAP (50) entries.

Determinism: no threads, no I/O, no print(), no wall-clock, no
unseeded random.  Double run produces byte-equal event-log JSON.

Deviations from DIR/2026/046:
  D-1  §3.1 per-game values not in directive text; clean/reject
       globals (21+4).
  D-2  §4.3 retry-budget phrasing self-contradictory; parenthetical
       reading chosen (retries after original: clean=2, reject=3).
  D-3  §4.4 fingerprint includes a redundant total-object count.
  D-4  observe() raises DivergenceError on first survival.
  D-5  §4.8 diagnose-then-patch implemented as injectable
       patch_hook; harness supplies the wire.
  D-6  T-14 try/except wrapper added in v1.0.1.
  D-7  Debounce ticks on held frames per §4.5 "hold for
       debounce_steps".
  D-8  Post-patch reset semantics: PASS resets the tracker (fix
       worked); FAIL / no-prediction keeps it open so persistent
       divergence escalates.
  D-9  §4.8 delta baseline was unspecified; v1.0.5 uses the last
       stable frame's objects (R-F1 fix).
"""
from __future__ import annotations
import hashlib, json, logging, statistics
from collections import Counter
from dataclasses import dataclass
from types import MappingProxyType
from typing import Any, Callable, Dict, List, Mapping, Optional, Sequence, Tuple

logger = logging.getLogger(__name__)

try:
    from agent import grid_text as _gt
except ImportError:
    try:
        import grid_text as _gt
    except ImportError:
        _gt = None

try:
    from agent import inference_layer as _il
except ImportError:
    _il = None

try:
    from agent import context_manager as _cm
except ImportError:
    _cm = None

__all__ = [
    "PipelineError", "NoOpDetected", "DivergenceError",
    "PremiseFailureError", "CalibrationError", "StaleObservationError",
    "ActionResult", "ObserveResult", "CalibrationRow", "EventRecord",
    "DeltaPacket", "CLEAN_SET", "REJECT_LIST", "REJECT_LIST_IDS",
    "GLOBAL_DYNAMICS", "ALL_GAME_IDS", "DEFAULT_CALIBRATION",
    "LiveCalibration", "NoOpPredictor", "ActionPipeline",
    "MAX_DELTA_PACKET_CHARS", "RETRY_HISTORY_CAP",
]

CLEAN_SET: Mapping[str, Any] = MappingProxyType({
    "games_count": 21, "noop_pct": 3.78, "exact_repeat_pct": 1.00,
    "jitter_pct": 2.35, "retry_success_pct": 59.1,
    "switch_after_double_fail_pct": 3.9, "pause_retry_s": 1.10,
    "pause_fresh_s": 0.57, "median_revert_distance": 5,
})

REJECT_LIST_IDS: Tuple[str, ...] = (
    "lp85-305b61c3", "s5i5-18d95033", "vc33-5430563c",
    "dc22-fdcac232")

REJECT_LIST: Mapping[str, Any] = MappingProxyType({
    "games_count": 4, "noop_pct": 10.1, "exact_repeat_pct": 45.3,
    "retry_success_pct": 68.2, "retry_budget": 3,
})

# p90=200 verbatim from DIR/2026/046 §2.3.
GLOBAL_DYNAMICS: Mapping[str, Any] = MappingProxyType({
    "oscillation_pct": 57.8, "changeset_median": 54,
    "changeset_p90": 200,
})

ALL_GAME_IDS: Tuple[str, ...] = (
    "ar25-0c556536", "bp35-0a0ad940", "cd82-fb555c5d",
    "cn04-2fe56bfb", "dc22-fdcac232", "ft09-0d8bbf25",
    "g50t-5849a774", "ka59-38d34dbb", "lf52-271a04aa",
    "lp85-305b61c3", "ls20-9607627b", "m0r0-492f87ba",
    "r11l-495a7899", "re86-8af5384d", "s5i5-18d95033",
    "sb26-7fbdac44", "sc25-635fd71a", "sk48-d8078629",
    "sp80-589a99af", "su15-1944f8ab", "tn36-ef4dde99",
    "tr87-cd924810", "tu93-0768757b", "vc33-5430563c",
    "wa30-ee6fef47")

MAX_DELTA_PACKET_CHARS = 4000
LIVE_CALIB_N_TARGET = 30
# REVERT_WINDOW_MAX=15 verbatim from DIR/2026/046 §3.2 ("k in 1..15").
REVERT_WINDOW_MAX = 15
RETRY_HISTORY_CAP = 50


class PipelineError(RuntimeError):
    """Base typed failure for the action pipeline."""


class NoOpDetected(PipelineError):
    """A dispatched action produced zero cell change."""


class DivergenceError(PipelineError):
    """First surviving divergence; carries the DeltaPacket as .packet."""

    def __init__(self, message: str, packet: "DeltaPacket") -> None:
        super().__init__(message)
        self.packet = packet


class PremiseFailureError(PipelineError):
    """Two consecutive surviving divergences — abort the plan."""


class CalibrationError(PipelineError):
    """Calibration table lookup or live estimator failure."""


class StaleObservationError(PipelineError):
    """Checksum mismatch in the patch packet reference."""


@dataclass(frozen=True)
class CalibrationRow:
    game_id: str
    noop_rate: float
    retry_pct: float
    debounce_steps: int
    oscillation_pct: float
    is_reject: bool
    retry_budget: int

    def to_dict(self) -> Dict[str, Any]:
        return {"game_id": self.game_id, "noop_rate": self.noop_rate,
                "retry_pct": self.retry_pct,
                "debounce_steps": self.debounce_steps,
                "oscillation_pct": self.oscillation_pct,
                "is_reject": self.is_reject,
                "retry_budget": self.retry_budget}


def _build_default_calibration() -> Dict[str, CalibrationRow]:
    rows: Dict[str, CalibrationRow] = {}
    for gid in ALL_GAME_IDS:
        rj = gid in REJECT_LIST_IDS
        rows[gid] = CalibrationRow(
            game_id=gid,
            noop_rate=(REJECT_LIST["noop_pct"] if rj
                       else CLEAN_SET["noop_pct"]) / 100.0,
            retry_pct=(REJECT_LIST["retry_success_pct"] if rj
                       else CLEAN_SET["retry_success_pct"]) / 100.0,
            debounce_steps=CLEAN_SET["median_revert_distance"],
            oscillation_pct=GLOBAL_DYNAMICS["oscillation_pct"] / 100.0,
            is_reject=rj,
            retry_budget=REJECT_LIST["retry_budget"] if rj else 2)
    return rows


DEFAULT_CALIBRATION: Mapping[str, CalibrationRow] = MappingProxyType(
    _build_default_calibration())


class LiveCalibration:
    """Running estimator over the first N=30 transitions (§3.2)."""

    def __init__(self, n_target: int = LIVE_CALIB_N_TARGET) -> None:
        self.n_target = int(n_target)
        self.transitions = 0
        self.noop_count = 0
        self.revert_distances: List[int] = []
        self._history: List[List[List[int]]] = []

    def observe(self, old_frame, new_frame) -> None:
        self.transitions += 1
        if old_frame == new_frame:
            self.noop_count += 1
        if self._history:
            for k in range(1, min(REVERT_WINDOW_MAX,
                                  len(self._history)) + 1):
                ref = self._history[-k]
                if len(ref) != len(new_frame):
                    continue
                if any(ref[r] == new_frame[r]
                       for r in range(len(new_frame))):
                    self.revert_distances.append(k)
                    break
        self._history.append([list(r) for r in new_frame])
        if len(self._history) > REVERT_WINDOW_MAX:
            self._history.pop(0)

    @property
    def is_ready(self) -> bool:
        return self.transitions >= self.n_target

    @property
    def noop_rate(self) -> Optional[float]:
        return self.noop_count / self.transitions \
            if self.transitions else None

    @property
    def median_revert_distance(self) -> Optional[int]:
        return (int(statistics.median(self.revert_distances))
                if self.revert_distances else None)

    def to_dict(self) -> Dict[str, Any]:
        return {"transitions": self.transitions,
                "n_target": self.n_target,
                "is_ready": self.is_ready,
                "noop_rate": self.noop_rate,
                "median_revert_distance": self.median_revert_distance}


@dataclass
class EventRecord:
    kind: str
    tick: int
    data: Dict[str, Any]

    def to_dict(self) -> Dict[str, Any]:
        return {"kind": self.kind, "tick": self.tick, "data": self.data}


@dataclass
class ActionResult:
    action: Any
    attempt: int
    no_op: bool
    switch_flag: bool
    tick: int
    frame_before_checksum: Optional[str]

    def to_dict(self) -> Dict[str, Any]:
        return {"action": repr(self.action), "attempt": self.attempt,
                "no_op": self.no_op, "switch_flag": self.switch_flag,
                "tick": self.tick,
                "frame_before_checksum": self.frame_before_checksum}


@dataclass
class ObserveResult:
    first: bool = False
    no_op: bool = False
    retry_queued: bool = False
    switch_flag_set: bool = False
    debounce_started: bool = False
    debounce_decremented: bool = False
    divergence: bool = False
    premise_failure: bool = False
    match: bool = False
    prediction_pass: bool = False
    prediction_fail: bool = False

    def to_dict(self) -> Dict[str, Any]:
        return dict(self.__dict__)


@dataclass
class DeltaPacket:
    game_id: str
    moved: List[list]
    recolored: List[list]
    new: List[list]
    gone: List[list]
    retry_history: List[Dict[str, Any]]
    calibration_row: Dict[str, Any]
    previous_checksums: Dict[str, str]
    needs_full: bool
    char_count: int
    predicted_post_state: Optional[Dict[str, Any]] = None

    def to_dict(self) -> Dict[str, Any]:
        return {"game_id": self.game_id, "moved": self.moved,
                "recolored": self.recolored, "new": self.new,
                "gone": self.gone, "retry_history": self.retry_history,
                "calibration_row": self.calibration_row,
                "previous_checksums": self.previous_checksums,
                "needs_full": self.needs_full,
                "char_count": self.char_count,
                "predicted_post_state": self.predicted_post_state}


class NoOpPredictor:
    """Default hook — no prediction; comparator runs in drift mode."""

    def __call__(self, state, action):
        return None


def _require_grid_text() -> None:
    if _gt is None:
        raise PipelineError(
            "grid_text module unavailable — cannot extract objects "
            "or compute checksums")


def _extract(frame):
    _require_grid_text()
    return _gt.extract_objects(frame, background=0)


def _rle_checksum(frame) -> str:
    _require_grid_text()
    return _gt.rle_anchor(frame)["checksum"]


def _fingerprint(objects) -> Tuple[Tuple[Tuple[str, int], ...], int]:
    counts = Counter(o.ohash for o in objects)
    return (tuple(sorted(counts.items())), len(objects))


def _fp_to_dict(fp) -> Dict[str, Any]:
    return {"ohash_counts": [list(x) for x in fp[0]], "total": fp[1]}


def _dict_to_fp(d) -> Tuple[Tuple[Tuple[str, int], ...], int]:
    return (tuple((str(k), int(v)) for k, v in d["ohash_counts"]),
            int(d["total"]))


def _diff_objects(prev_objs, curr_objs) -> Dict[str, List[list]]:
    if _gt is None:
        return {"moved": [], "recolored": [], "new": [], "gone": []}
    return _gt.object_delta(prev_objs, curr_objs)


class ActionPipeline:
    """Two-tier action watchdog.  Lifecycle:
        submit_plan(actions); step(); observe(settled_frame)

    No-op gate and divergence check run in parallel.  On survival the
    DivergenceError fires with a DeltaPacket diffed against the last
    STABLE frame (not the previous observation).  The next observation
    resets (prediction PASS) or keeps the tracker open (FAIL / none).
    """

    def __init__(
        self,
        game_id: str,
        dispatch: Optional[Callable[[Any], None]] = None,
        predict_hook: Optional[Callable[[Any, Any], Any]] = None,
        patch_hook: Optional[Callable[["DeltaPacket"], Any]] = None,
        calibration: Optional[Mapping[str, CalibrationRow]] = None,
    ) -> None:
        self.game_id = game_id
        if calibration is not None:
            self._calib: Dict[str, CalibrationRow] = dict(calibration)
        else:
            self._calib = dict(DEFAULT_CALIBRATION)
        row = self._calib.get(game_id)
        if row is None:
            row = CalibrationRow(
                game_id=game_id,
                noop_rate=CLEAN_SET["noop_pct"] / 100.0,
                retry_pct=CLEAN_SET["retry_success_pct"] / 100.0,
                debounce_steps=CLEAN_SET["median_revert_distance"],
                oscillation_pct=GLOBAL_DYNAMICS["oscillation_pct"] / 100.0,
                is_reject=False, retry_budget=2)
        self._row = row
        self._dispatch = dispatch
        self._predict = predict_hook or NoOpPredictor()
        self._patch_hook = patch_hook
        self._live = LiveCalibration()
        self._queue: List[Any] = []
        self._prev_settled: Optional[List[List[int]]] = None
        self._prev_checksum: Optional[str] = None
        self._prev_objects: Optional[List] = None
        # R-F1: baseline for the delta packet — objects of the last
        # frame whose fingerprint matched.  Kept alongside
        # _last_stable_fp so survival can diff against it.
        self._last_stable_objects: Optional[List] = None
        self._last_stable_fp: Optional[Tuple] = None
        self._events: List[EventRecord] = []
        self._tick = 0
        self._retry_count = 0
        self._current_action: Optional[Any] = None
        self._current_predicted_fp: Optional[Tuple] = None
        self._switch_flag = False
        self._debounce_remaining = 0
        self._consecutive_divergences = 0
        self._retry_history: List[Dict[str, Any]] = []
        self._last_patch: Optional[DeltaPacket] = None
        self._pending_prediction: Optional[Tuple] = None
        self._post_patch_pending = False

    def check_integration_deps(self) -> Dict[str, bool]:
        return {
            "grid_text": _gt is not None,
            "context_manager": _cm is not None,
            "inference_layer": _il is not None,
        }

    def _record(self, kind: str,
                data: Optional[Dict[str, Any]] = None) -> None:
        self._events.append(EventRecord(kind=kind, tick=self._tick,
                                        data=dict(data or {})))

    def event_log(self) -> List[EventRecord]:
        return list(self._events)

    def event_log_json(self) -> str:
        return json.dumps([e.to_dict() for e in self._events],
                          sort_keys=True)

    def last_patch(self) -> Optional[DeltaPacket]:
        return self._last_patch

    @property
    def switch_flag(self) -> bool:
        return self._switch_flag

    def submit_plan(self, actions: Sequence[Any]) -> None:
        self._queue = list(actions)
        self._record("plan_submitted", {"count": len(actions)})

    def step(self) -> Optional[ActionResult]:
        if not self._queue:
            return None
        action = self._queue.pop(0)
        # L-1: reset the retry counter when a NEW action is
        # dispatched.  A quiescent streak that previously exhausted
        # its retry budget would otherwise deny a freshly-issued
        # action its retry budget.
        if self._current_action is not None and action != self._current_action:
            self._retry_count = 0
        self._current_action = action
        self._tick += 1
        if self._prev_settled is not None:
            self._current_predicted_fp = self._predict(
                self._prev_settled, action)
        if self._dispatch is not None:
            self._dispatch(action)
        self._record("action_dispatched",
                     {"action": repr(action),
                      "queue_remaining": len(self._queue)})
        return ActionResult(
            action=action, attempt=self._retry_count + 1, no_op=False,
            switch_flag=self._switch_flag, tick=self._tick,
            frame_before_checksum=self._prev_checksum)

    def observe(self, settled_frame) -> ObserveResult:
        last = [list(row) for row in settled_frame[-1]]
        checksum = _rle_checksum(last)

        if self._prev_settled is None:
            self._prev_settled = last
            self._prev_checksum = checksum
            self._prev_objects = _extract(last)
            self._last_stable_fp = _fingerprint(self._prev_objects)
            self._last_stable_objects = self._prev_objects
            self._record("first_observation", {"checksum": checksum})
            return ObserveResult(first=True)

        pred_result = self._verify_pending_prediction(last)

        if self._post_patch_pending:
            self._post_patch_pending = False
            if pred_result == "PASS":
                actual_fp = _fingerprint(_extract(last))
                self._last_stable_fp = actual_fp
                self._last_stable_objects = _extract(last)
                self._debounce_remaining = 0
                self._consecutive_divergences = 0
                self._record("post_patch_reset",
                             {"kept_open": False, "prediction": "PASS"})
            else:
                self._record("post_patch_reset", {
                    "kept_open": True,
                    "prediction": pred_result or "none"})

        state_changed = (checksum != self._prev_checksum)
        retry_queued = False
        switch_set = False
        if state_changed:
            self._retry_count = 0
            self._switch_flag = False
        else:
            retry_queued, switch_set = self._handle_no_op(checksum)

        new_objects = _extract(last)
        div_res = self._check_divergence(new_objects)

        self._live.observe(self._prev_settled, last)
        self._prev_settled = last
        self._prev_checksum = checksum
        self._prev_objects = new_objects

        div_res.no_op = (not state_changed)
        div_res.retry_queued = retry_queued
        div_res.switch_flag_set = switch_set
        if pred_result is not None:
            div_res.prediction_pass = (pred_result == "PASS")
            div_res.prediction_fail = (pred_result == "FAIL")

        if div_res.premise_failure:
            self._clear_queue()
            raise PremiseFailureError(
                f"two consecutive surviving divergences on "
                f"{self.game_id!r}")
        if div_res.divergence:
            raise DivergenceError(
                f"divergence survived debounce on {self.game_id!r}",
                self._last_patch)
        return div_res

    def _verify_pending_prediction(self, frame) -> Optional[str]:
        if self._pending_prediction is None:
            return None
        actual_fp = _fingerprint(_extract(frame))
        passed = (actual_fp == self._pending_prediction)
        result = "PASS" if passed else "FAIL"
        self._record("prediction_verified", {
            "result": result,
            "expected": _fp_to_dict(self._pending_prediction),
            "actual": _fp_to_dict(actual_fp),
        })
        self._pending_prediction = None
        return result

    def _handle_no_op(self, checksum: str) -> Tuple[bool, bool]:
        self._retry_count += 1
        self._retry_history.append({
            "action": repr(self._current_action),
            "attempt": self._retry_count, "no_op": True,
            "checksum": checksum})
        # L-1: cap the history.  The packet carries the tail.
        if len(self._retry_history) > RETRY_HISTORY_CAP:
            self._retry_history = \
                self._retry_history[-RETRY_HISTORY_CAP:]
        self._record("no_op_detected",
                     {"attempt": self._retry_count,
                      "action": repr(self._current_action)})
        if self._retry_count > self._row.retry_budget:
            self._switch_flag = True
            self._record("switch_flag_set",
                         {"after_attempts": self._retry_count})
            return False, True
        self._queue.insert(0, self._current_action)
        return True, False

    def _delta_baseline_objects(self):
        """R-F1: the objects the patch delta should be measured
        against.  Prefer the last stable frame; fall back to the
        previous observation when no stable baseline has been
        captured yet."""
        if self._last_stable_objects is not None:
            return self._last_stable_objects
        return self._prev_objects or []

    def _check_divergence(self, new_objects) -> ObserveResult:
        predicted = self._current_predicted_fp
        actual_fp = _fingerprint(new_objects)
        if predicted is None:
            if self._last_stable_fp is None:
                self._last_stable_fp = actual_fp
                self._last_stable_objects = new_objects
                return ObserveResult(match=True)
            mismatch = (actual_fp != self._last_stable_fp)
        else:
            mismatch = (actual_fp != predicted)

        if not mismatch:
            self._debounce_remaining = 0
            self._consecutive_divergences = 0
            self._last_stable_fp = actual_fp
            self._last_stable_objects = new_objects
            self._record("object_match",
                         {"fingerprint_size": len(actual_fp[0])})
            return ObserveResult(match=True)

        if self._consecutive_divergences >= 1:
            self._consecutive_divergences += 1
            # R-F1: diff against the last STABLE objects.
            baseline = self._delta_baseline_objects()
            delta = _diff_objects(baseline, new_objects)
            self._last_patch = self._build_delta_packet(delta, None)
            self._record("premise_failure",
                         {"consecutive": self._consecutive_divergences})
            return ObserveResult(divergence=True, premise_failure=True)

        if self._debounce_remaining == 0:
            window = self._current_debounce_window()
            self._debounce_remaining = window
            self._record("debounce_started", {"window": window})
            return ObserveResult(debounce_started=True)

        self._debounce_remaining -= 1
        if self._debounce_remaining > 0:
            self._record("debounce_decremented",
                         {"remaining": self._debounce_remaining})
            return ObserveResult(debounce_decremented=True)

        # Survival.  R-F1: diff against the last STABLE objects.
        self._consecutive_divergences = 1
        baseline = self._delta_baseline_objects()
        delta = _diff_objects(baseline, new_objects)
        prediction = None
        if self._patch_hook is not None:
            provisional = self._build_delta_packet(delta, None)
            try:
                prediction = self._patch_hook(provisional)
            except Exception as exc:  # noqa: BLE001
                self._record("patch_hook_error", {"error": str(exc)})
                prediction = None
        self._last_patch = self._build_delta_packet(delta, prediction)
        if self._last_patch.predicted_post_state is not None:
            try:
                self._pending_prediction = _dict_to_fp(
                    self._last_patch.predicted_post_state)
            except (KeyError, TypeError, ValueError):
                self._pending_prediction = None
        elif prediction is not None:
            self._record("prediction_dropped_for_budget", {})
        self._post_patch_pending = True
        self._record("patch_triggered", {
            "char_count": self._last_patch.char_count,
            "has_prediction": self._last_patch.predicted_post_state
                              is not None,
            "delta_moved": len(self._last_patch.moved),
            "delta_new": len(self._last_patch.new),
            "delta_gone": len(self._last_patch.gone),
        })
        self._record("divergence_survived", {"consecutive": 1})
        return ObserveResult(divergence=True)

    def _current_debounce_window(self) -> int:
        if self._live.is_ready:
            med = self._live.median_revert_distance
            if med is not None:
                return int(med)
        return self._row.debounce_steps

    def _build_delta_packet(self, delta,
                            prediction: Optional[Dict[str, Any]]
                            ) -> DeltaPacket:
        packet = DeltaPacket(
            game_id=self.game_id,
            moved=list(delta.get("moved", [])),
            recolored=list(delta.get("recolored", [])),
            new=list(delta.get("new", [])),
            gone=list(delta.get("gone", [])),
            retry_history=list(self._retry_history),
            calibration_row=self._row.to_dict(),
            previous_checksums={
                "prev_settled": self._prev_checksum or ""},
            needs_full=False, char_count=0,
            predicted_post_state=prediction)
        js = json.dumps(packet.to_dict(), sort_keys=True)
        if len(js) > MAX_DELTA_PACKET_CHARS:
            packet.moved = []
            packet.recolored = []
            packet.new = []
            packet.gone = []
            packet.needs_full = True
            js = json.dumps(packet.to_dict(), sort_keys=True)
            if len(js) > MAX_DELTA_PACKET_CHARS:
                packet.predicted_post_state = None
                js = json.dumps(packet.to_dict(), sort_keys=True)
        packet.char_count = len(js)
        return packet

    def verify_observation(self, packet: DeltaPacket,
                           current_checksum: str) -> bool:
        prev = packet.previous_checksums.get("prev_settled")
        if prev and prev != current_checksum:
            raise StaleObservationError(
                f"packet referenced {prev!r}, current is "
                f"{current_checksum!r}")
        return True

    def _clear_queue(self) -> None:
        self._queue = []
        self._record("queue_cleared", {})

    def compare(self, prev_frame, curr_frame) -> Dict[str, Any]:
        prev_objs = _extract(prev_frame)
        curr_objs = _extract(curr_frame)
        prev_fp = _fingerprint(prev_objs)
        curr_fp = _fingerprint(curr_objs)
        return {"prev_fp": prev_fp, "curr_fp": curr_fp,
                "delta": _diff_objects(prev_objs, curr_objs),
                "match": prev_fp == curr_fp}


# ======================================================================
# Inline self-tests (v1.0.5, T-1..T-22)
# ======================================================================

def _eg(v=0, h=64, w=64):
    return [[v] * w for _ in range(h)]


def _pb(g, r0, c0, h, w, v):
    for r in range(r0, r0 + h):
        for c in range(c0, c0 + w):
            g[r][c] = v


def _fast_cal(game_id="ls20-9607627b", debounce=1):
    cal = dict(DEFAULT_CALIBRATION)
    cal[game_id] = CalibrationRow(
        game_id, 0.038, 0.591, debounce, 0.578, False, 2)
    return cal


def _t1():
    assert CLEAN_SET["games_count"] == 21
    assert CLEAN_SET["noop_pct"] == 3.78
    assert CLEAN_SET["exact_repeat_pct"] == 1.00
    assert CLEAN_SET["jitter_pct"] == 2.35
    assert CLEAN_SET["retry_success_pct"] == 59.1
    assert CLEAN_SET["switch_after_double_fail_pct"] == 3.9
    assert CLEAN_SET["pause_retry_s"] == 1.10
    assert CLEAN_SET["pause_fresh_s"] == 0.57
    assert CLEAN_SET["median_revert_distance"] == 5
    assert REJECT_LIST_IDS == ("lp85-305b61c3", "s5i5-18d95033",
                               "vc33-5430563c", "dc22-fdcac232")
    assert REJECT_LIST["noop_pct"] == 10.1
    assert REJECT_LIST["exact_repeat_pct"] == 45.3
    assert REJECT_LIST["retry_success_pct"] == 68.2
    assert REJECT_LIST["retry_budget"] == 3
    assert GLOBAL_DYNAMICS["oscillation_pct"] == 57.8
    assert GLOBAL_DYNAMICS["changeset_median"] == 54
    assert GLOBAL_DYNAMICS["changeset_p90"] == 200
    assert len(ALL_GAME_IDS) == 25
    assert len(DEFAULT_CALIBRATION) == 25


def _t2():
    p = ActionPipeline("ls20-9607627b")
    g = _eg(0)
    p.submit_plan(["right"]); p.step(); p.observe([g])
    p.submit_plan(["right"]); p.step()
    r = p.observe([g])
    assert r.no_op is True
    assert any(e.kind == "no_op_detected" for e in p.event_log())


def _t3():
    p = ActionPipeline("ls20-9607627b")
    g = _eg(0)
    p.submit_plan(["right"]); p.step(); p.observe([g])
    p.submit_plan(["right"]); p.step()
    r = p.observe([g]); assert r.no_op and r.retry_queued
    p.step(); r = p.observe([g]); assert r.no_op and r.retry_queued
    p.step(); r = p.observe([g])
    assert r.no_op and r.switch_flag_set
    assert p.switch_flag


def _t4():
    p = ActionPipeline("ls20-9607627b")
    g = _eg(0)
    p.submit_plan(["right"]); p.step(); p.observe([g])
    p.submit_plan(["right"]); p.step(); p.observe([g])
    assert p._retry_count >= 1
    g2 = _eg(0); _pb(g2, 10, 10, 2, 2, 5)
    p.submit_plan(["right"]); p.step(); p.observe([g2])
    assert p._retry_count == 0


def _t5():
    p = ActionPipeline("ls20-9607627b")
    a = _eg(0); _pb(a, 5, 5, 2, 2, 5)
    b = _eg(0); _pb(b, 20, 20, 2, 2, 5)
    cmp = p.compare(a, b)
    assert cmp["match"] is True
    assert len(cmp["delta"]["moved"]) >= 1


def _t6():
    p = ActionPipeline("ls20-9607627b")
    a = _eg(0)
    p.submit_plan(["x"]); p.step(); p.observe([a])
    b = _eg(0); _pb(b, 5, 5, 2, 2, 5)
    p.submit_plan(["x"]); p.step(); p.observe([b])
    p.submit_plan(["x"]); p.step(); r = p.observe([a])
    assert r.match is True
    assert p.last_patch() is None


def _t7():
    p = ActionPipeline("ls20-9607627b")
    a = _eg(0)
    p.submit_plan(["x"]); p.step(); p.observe([a])
    b = _eg(0); _pb(b, 5, 5, 2, 2, 5)
    p.submit_plan(["x"]); p.step(); r = p.observe([b])
    assert r.debounce_started is True
    assert p.last_patch() is None
    assert any(e.kind == "debounce_started" for e in p.event_log())


def _t8():
    p = ActionPipeline("ls20-9607627b")
    a = _eg(0)
    p.submit_plan(["x"]); p.step(); p.observe([a])
    for i in range(LIVE_CALIB_N_TARGET):
        c = _eg(0); c[i % 60][i % 60] = 5
        p.submit_plan(["x"]); p.step()
        try:
            p.observe([c])
        except (DivergenceError, PremiseFailureError):
            pass
    assert p._live.is_ready is True
    assert p._live.median_revert_distance is not None


def _t9():
    p = ActionPipeline("ls20-9607627b", calibration=_fast_cal())
    a = _eg(0)
    p.submit_plan(["x"]); p.step(); p.observe([a])
    b1 = _eg(0); _pb(b1, 5, 5, 2, 2, 5)
    p.submit_plan(["x"]); p.step()
    r = p.observe([b1])
    assert r.debounce_started
    p.submit_plan(["x"]); p.step()
    try:
        p.observe([b1])
    except DivergenceError:
        pass
    else:
        raise AssertionError("expected DivergenceError on first survival")
    assert p._consecutive_divergences == 1
    b2 = _eg(0); _pb(b2, 10, 10, 2, 2, 5)
    p.submit_plan(["x", "y"]); p.step()
    try:
        p.observe([b2])
    except PremiseFailureError:
        pass
    else:
        raise AssertionError("expected PremiseFailureError")
    assert p._queue == []
    assert any(e.kind == "post_patch_reset"
               and e.data.get("kept_open") is True
               for e in p.event_log())


def _t10():
    p = ActionPipeline("ls20-9607627b")
    a = _eg(0)
    p.submit_plan(["x"]); p.step(); p.observe([a])
    b = _eg(0); _pb(b, 5, 5, 2, 2, 5)
    p.submit_plan(["x"]); p.step(); p.observe([b])
    p.submit_plan(["x"]); p.step(); r = p.observe([a])
    assert r.match is True
    assert p.last_patch() is None


def _t11():
    p = ActionPipeline("ls20-9607627b")
    a = _eg(0); _pb(a, 5, 5, 2, 2, 5)
    b = _eg(0); _pb(b, 20, 20, 2, 2, 5)
    delta = _diff_objects(_extract(a), _extract(b))
    assert len(delta["moved"]) >= 1
    entry = delta["moved"][0]
    assert isinstance(entry[1], str) and len(entry[1]) == 32


def _t12():
    p = ActionPipeline("ls20-9607627b")
    huge = {"moved": [[i, "h" * 40, [i] * 4, [i] * 4]
                      for i in range(500)],
            "recolored": [], "new": [], "gone": []}
    packet = p._build_delta_packet(huge, None)
    assert packet.char_count <= MAX_DELTA_PACKET_CHARS
    assert packet.needs_full is True
    assert packet.moved == []


def _t13():
    p = ActionPipeline("ls20-9607627b")
    packet = DeltaPacket(
        game_id="ls20-9607627b", moved=[], recolored=[], new=[], gone=[],
        retry_history=[], calibration_row={},
        previous_checksums={"prev_settled": "abc123"},
        needs_full=False, char_count=50)
    try:
        p.verify_observation(packet, "different")
    except StaleObservationError:
        return
    raise AssertionError("expected StaleObservationError")


def _t14():
    p = ActionPipeline("ls20-9607627b", calibration=_fast_cal())
    a = _eg(0)
    p.submit_plan(["x"]); p.step(); p.observe([a])
    b = _eg(0); _pb(b, 5, 5, 2, 2, 5)
    p.submit_plan(["x"]); p.step(); p.observe([b])
    p.submit_plan(["x"]); p.step()
    try:
        p.observe([b])
    except DivergenceError:
        pass
    packet = p.last_patch()
    assert packet is not None
    assert packet.calibration_row["game_id"] == "ls20-9607627b"
    assert "noop_rate" in packet.calibration_row
    assert "debounce_steps" in packet.calibration_row
    assert isinstance(packet.retry_history, list)


def _t15():
    a = _eg(0); _pb(a, 5, 5, 2, 2, 5)
    b = _eg(0); _pb(b, 20, 20, 2, 2, 5)
    fp_b = _fingerprint(_extract(b))
    p = ActionPipeline("ls20-9607627b",
                       predict_hook=lambda s, act: fp_b)
    p.submit_plan(["x"]); p.step(); p.observe([a])
    p.submit_plan(["x"]); p.step(); r = p.observe([b])
    assert r.match is True
    fp_empty = ((), 0)
    p2 = ActionPipeline("ls20-9607627b",
                        predict_hook=lambda s, act: fp_empty)
    p2.submit_plan(["x"]); p2.step(); p2.observe([a])
    p2.submit_plan(["x"]); p2.step(); r2 = p2.observe([b])
    assert r2.debounce_started is True


def _t16():
    p1 = ActionPipeline("ls20-9607627b")
    assert p1._row.retry_budget == 2
    assert p1._row.is_reject is False
    p2 = ActionPipeline("lp85-305b61c3")
    assert p2._row.retry_budget == 3
    assert p2._row.is_reject is True


def _t17():
    def run():
        p = ActionPipeline("ls20-9607627b")
        g = _eg(0)
        for _ in range(5):
            p.submit_plan(["x"]); p.step(); p.observe([g])
        return p.event_log_json()
    assert run() == run()


def _t18():
    import random as _r
    rng = _r.Random(2026)
    for _ in range(200):
        p = ActionPipeline("ls20-9607627b", calibration=_fast_cal())
        for i in range(20):
            g = [[rng.randrange(16) for _ in range(8)]
                 for _ in range(8)]
            p.submit_plan([f"a{i}"]); p.step()
            try:
                p.observe([g])
            except (DivergenceError, PremiseFailureError):
                pass
        assert p._retry_count <= p._row.retry_budget + 1
        for e in p.event_log():
            assert isinstance(e.kind, str)
            assert isinstance(e.tick, int)


def _t19():
    post = _eg(0); _pb(post, 30, 30, 2, 2, 5)
    pred_pass = _fp_to_dict(_fingerprint(_extract(post)))

    p1 = ActionPipeline(
        "ls20-9607627b", calibration=_fast_cal(),
        patch_hook=lambda packet: pred_pass)
    a = _eg(0)
    p1.submit_plan(["x"]); p1.step(); p1.observe([a])
    b = _eg(0); _pb(b, 5, 5, 2, 2, 5)
    p1.submit_plan(["x"]); p1.step(); p1.observe([b])
    p1.submit_plan(["x"]); p1.step()
    try:
        p1.observe([b])
    except DivergenceError:
        pass
    assert p1.last_patch().predicted_post_state == pred_pass
    p1.submit_plan(["x"]); p1.step()
    r1 = p1.observe([post])
    assert r1.prediction_pass is True
    assert any(e.kind == "prediction_verified"
               and e.data.get("result") == "PASS"
               for e in p1.event_log())
    assert p1._consecutive_divergences == 0
    assert p1._post_patch_pending is False

    pred_fail = _fp_to_dict(_fingerprint(_extract(a)))
    p2 = ActionPipeline(
        "ls20-9607627b", calibration=_fast_cal(),
        patch_hook=lambda packet: pred_fail)
    p2.submit_plan(["x"]); p2.step(); p2.observe([a])
    b2 = _eg(0); _pb(b2, 5, 5, 2, 2, 5)
    p2.submit_plan(["x"]); p2.step(); p2.observe([b2])
    p2.submit_plan(["x"]); p2.step()
    try:
        p2.observe([b2])
    except DivergenceError:
        pass
    p2.submit_plan(["x"]); p2.step()
    escalated = False
    try:
        p2.observe([post])
    except PremiseFailureError:
        escalated = True
    assert escalated, "failed patch must escalate to PremiseFailureError"
    assert any(e.kind == "prediction_verified"
               and e.data.get("result") == "FAIL"
               for e in p2.event_log())
    assert any(e.kind == "post_patch_reset"
               and e.data.get("kept_open") is True
               for e in p2.event_log())


def _t20():
    p = ActionPipeline("ls20-9607627b", calibration=_fast_cal())
    a = _eg(0)
    p.submit_plan(["x"]); p.step(); p.observe([a])
    b = _eg(0); _pb(b, 5, 5, 2, 2, 5)
    p.submit_plan(["x"]); p.step()
    r0 = p.observe([b])
    assert r0.debounce_started
    assert p._debounce_remaining == 1
    p.submit_plan(["x"]); p.step()
    raised = False
    try:
        p.observe([b])
    except DivergenceError:
        raised = True
    assert raised
    kinds = [e.kind for e in p.event_log()]
    assert "no_op_detected" in kinds
    assert "divergence_survived" in kinds


def _t21():
    post = _eg(0); _pb(post, 30, 30, 2, 2, 5)
    pred_pass = _fp_to_dict(_fingerprint(_extract(post)))
    p = ActionPipeline(
        "ls20-9607627b", calibration=_fast_cal(),
        patch_hook=lambda packet: pred_pass)
    a = _eg(0)
    p.submit_plan(["x"]); p.step(); p.observe([a])
    b = _eg(0); _pb(b, 5, 5, 2, 2, 5)
    p.submit_plan(["x"]); p.step(); p.observe([b])
    p.submit_plan(["x"]); p.step()
    try:
        p.observe([b])
    except DivergenceError:
        pass
    assert p._consecutive_divergences == 1
    assert p._post_patch_pending is True
    p.submit_plan(["x"]); p.step()
    p.observe([post])
    assert p._consecutive_divergences == 0
    assert p._post_patch_pending is False
    assert p._last_stable_fp == _fingerprint(_extract(post))


def _t22():
    """R-F1: a persistent stable-jump divergence must produce a
    NON-empty delta packet.  The bug in v1.0.4 diffed against
    _prev_objects which, on a persistent divergence, is already the
    diverging frame."""
    p = ActionPipeline("ls20-9607627b", calibration=_fast_cal())
    # Stable baseline: one object at (5,5).
    a = _eg(0); _pb(a, 5, 5, 3, 3, 5)
    p.submit_plan(["x"]); p.step(); p.observe([a])
    # Jump to a stable-but-divergent state: one object at (40,40).
    b = _eg(0); _pb(b, 40, 40, 3, 3, 8)
    p.submit_plan(["x"]); p.step()
    r1 = p.observe([b])   # debounce_started
    assert r1.debounce_started
    # Hold the divergent state; survival fires on the next observe.
    p.submit_plan(["x"]); p.step()
    try:
        p.observe([b])   # held -> survival -> DivergenceError
    except DivergenceError as exc:
        packet = exc.packet
    else:
        raise AssertionError("expected DivergenceError")
    assert packet is not None
    # R-F1: the delta MUST describe the difference vs the stable
    # baseline (a), NOT vs b (which would be empty).
    total = len(packet.moved) + len(packet.recolored) + \
            len(packet.new) + len(packet.gone)
    assert total >= 1, (
        f"R-F1 regression: empty delta at survival. "
        f"moved={packet.moved} new={packet.new} "
        f"gone={packet.gone} recolored={packet.recolored}")


def _run_self_tests() -> int:
    tests = [
        ("T-1  frozen constants", _t1),
        ("T-2  no-op gate fires", _t2),
        ("T-3  retry policy max attempts", _t3),
        ("T-4  retry counter resets", _t4),
        ("T-5  moved object not divergence", _t5),
        ("T-6  oscillation tolerance", _t6),
        ("T-7  debounce first mismatch transient", _t7),
        ("T-8  debounce live estimate", _t8),
        ("T-9  divergence -> premise failure", _t9),
        ("T-10 transient self-corrects", _t10),
        ("T-11 delta packet structure", _t11),
        ("T-12 delta packet budget", _t12),
        ("T-13 stale observation", _t13),
        ("T-14 patch contains retry + calibration", _t14),
        ("T-15 falsifiable prediction (forward hook)", _t15),
        ("T-16 reject-list retry budget", _t16),
        ("T-17 determinism", _t17),
        ("T-18 fuzz", _t18),
        ("T-19 patch prediction PASS/FAIL logged", _t19),
        ("T-20 held frame advances debounce", _t20),
        ("T-21 post-patch reset (D-8)", _t21),
        ("T-22 stable-baseline delta (R-F1)", _t22),
    ]
    failed = 0
    for name, fn in tests:
        try:
            fn()
            logging.info("PASS  %s", name)
        except Exception as e:
            failed += 1
            logging.info("FAIL  %s: %s: %s", name, type(e).__name__, e)
    logging.info("")
    logging.info("%d/%d passed", len(tests) - failed, len(tests))
    return 0 if failed == 0 else 1


if __name__ == "__main__":
    import sys as _sys
    logging.basicConfig(level=logging.INFO,
                        format="%(message)s", stream=_sys.stdout)
    _sys.exit(_run_self_tests())

In [ ]:
%%writefile goa/agent/sandbox_v6.py
# sandbox_v6.py
"""sandbox_v6.py — 3-Layer Execution Defense. DIR-2026-076 §2. R11.

Fixes applied across rounds:
  A-P0-01 (CRITICAL)  no module-level non-stdlib import; lazy guarded
  A-P0-03 (HIGH)      safe __import__ in child builtins
  A-P0-04 (HIGH)      numpy I/O blocked across every alias
  A-P0-07 (MEDIUM)    canonical SAFE_BUILTIN_NAMES (drift-tested)
  A-P0-10 (LOW)       delattr retained as documented hardening
  A-P0-11 (LOW)       extended np I/O list documented as hardening
  A-P0-14 (CRITICAL)  full-AST numpy alias collector
  A-P0-20 (LOW)       from numpy import * -> disallowed_construct
  A-P0-26 (HIGH)      nested attribute + from-submodule bypass closed
  A-P0-28 (LOW)       module-level docstrings/constants allowed
  A-P0-32 (LOW)       collector stores root binding name only
  A-P0-33 (LOW)       dead `star_imported` field removed
  A-P0-34 (LOW)       _is_module_node_allowed narrowed to scalar constants
  A-P0-37..39         open_memmap / genfromtxt / fromregex / fromstring /
                      fromiter added to _NP_IO_ATTRS
  A-FN-01/02/03 (LOW) T_SUITE docstring traceability
  R11 (LABEL)         revision label bumped from R10 for traceability

FD-3: DIR §2.1 vs §2.3 self-contradiction satisfied by lazy loading.
`time` is stdlib, flagged for DIR correction.
"""
from __future__ import annotations
import ast
import json
import os
import subprocess
import sys
import time


class SandboxViolation(Exception):
    """Raised when candidate code fails Layer-1 analysis. .reason in
    {forbidden_import, dunder_access, dynamic_exec, global_stmt,
     io_access, disallowed_construct, namespace_violation}."""

    def __init__(self, reason: str, detail: str = ""):
        super().__init__(f"{reason}: {detail}" if detail else reason)
        self.reason = reason
        self.detail = detail


_ALLOWED_MODULE_NODES = (
    ast.FunctionDef, ast.AsyncFunctionDef, ast.Assign, ast.AnnAssign,
    ast.ClassDef, ast.Import, ast.ImportFrom,
)

_ALLOWED_IMPORTS = {"numpy", "math", "itertools", "collections"}

_BLACKLIST_NAMES = {
    "eval", "exec", "compile", "__import__",
    "getattr", "setattr", "delattr", "globals", "locals", "vars",
    "open", "input",
}

_NP_IO_ATTRS = frozenset({
    "save", "load", "fromfile", "tofile", "savetxt", "loadtxt",
    "savez", "savez_compressed", "memmap",
    "open_memmap", "genfromtxt", "fromregex", "fromstring", "fromiter",
})

_ALLOWED_CONSTANT_TYPES = (str, int, float, bool, type(None))


def _is_module_node_allowed(node: ast.AST) -> bool:
    if isinstance(node, _ALLOWED_MODULE_NODES):
        return True
    if isinstance(node, ast.Expr) and isinstance(node.value, ast.Constant):
        return isinstance(node.value.value, _ALLOWED_CONSTANT_TYPES)
    return False


def _check_import(node: ast.AST) -> None:
    if isinstance(node, ast.Import):
        for alias in node.names:
            top = alias.name.split(".")[0]
            if top not in _ALLOWED_IMPORTS:
                raise SandboxViolation("forbidden_import", alias.name)
    elif isinstance(node, ast.ImportFrom):
        if node.level != 0 or node.module is None:
            raise SandboxViolation("forbidden_import", "relative import")
        top = node.module.split(".")[0]
        if top not in _ALLOWED_IMPORTS:
            raise SandboxViolation("forbidden_import", node.module)


class _NumpyAliasCollector(ast.NodeVisitor):
    def __init__(self) -> None:
        self.numpy_names: set = {"numpy"}

    def visit_Import(self, node: ast.Import) -> None:
        for alias in node.names:
            top = alias.name.split(".")[0]
            if top == "numpy":
                self.numpy_names.add(alias.asname or "numpy")
        self.generic_visit(node)

    def visit_ImportFrom(self, node: ast.ImportFrom) -> None:
        if node.module and node.module.split(".")[0] == "numpy":
            for alias in node.names:
                if alias.name == "*":
                    raise SandboxViolation(
                        "disallowed_construct",
                        "from numpy import * is disallowed")
                self.numpy_names.add(alias.asname or alias.name)
                if alias.name in _NP_IO_ATTRS:
                    raise SandboxViolation(
                        "io_access",
                        f"from numpy import {alias.name} is blocked")
        self.generic_visit(node)


def _attr_root_name(node: ast.AST):
    cur = node
    while isinstance(cur, ast.Attribute):
        cur = cur.value
    if isinstance(cur, ast.Name):
        return cur.id
    return None


def analyze(source: str) -> ast.Module:
    try:
        tree = ast.parse(source)
    except SyntaxError as e:
        raise SandboxViolation("disallowed_construct", f"syntax error: {e}")

    for node in tree.body:
        if not _is_module_node_allowed(node):
            raise SandboxViolation(
                "disallowed_construct",
                f"module-level {type(node).__name__} not allowed")
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            _check_import(node)

    collector = _NumpyAliasCollector()
    collector.visit(tree)

    for node in ast.walk(tree):
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            _check_import(node)

        if isinstance(node, ast.Attribute):
            if node.attr.startswith("__") and node.attr.endswith("__"):
                raise SandboxViolation(
                    "dunder_access", f"attribute {node.attr!r}")
            if node.attr in _NP_IO_ATTRS:
                root = _attr_root_name(node)
                if root in collector.numpy_names:
                    raise SandboxViolation(
                        "io_access",
                        f"{root}.{node.attr} is blocked")

        if isinstance(node, ast.Name):
            if node.id in _BLACKLIST_NAMES:
                raise SandboxViolation(
                    "dynamic_exec", f"name {node.id!r} is blocked")
            if (node.id.startswith("__") and node.id.endswith("__")
                    and node.id != "__name__"):
                raise SandboxViolation(
                    "dunder_access", f"dunder name {node.id!r}")

        if isinstance(node, ast.Global):
            raise SandboxViolation("global_stmt", "global statement")
        if isinstance(node, ast.Nonlocal):
            raise SandboxViolation("global_stmt", "nonlocal statement")

    return tree


SAFE_BUILTIN_NAMES = (
    "abs", "all", "any", "bool", "dict", "divmod", "enumerate",
    "filter", "float", "frozenset", "int", "isinstance", "len", "list",
    "map", "max", "min", "next", "range", "reversed", "round", "set",
    "sorted", "str", "sum", "tuple", "zip", "print",
)


def _make_safe_builtins() -> dict:
    b = __builtins__
    return {n: (b[n] if isinstance(b, dict) else getattr(b, n))
            for n in SAFE_BUILTIN_NAMES}


def _lazy_import(name: str):
    try:
        return __import__(name)
    except Exception:
        return None


def build_namespace() -> dict:
    return {
        "__builtins__": _make_safe_builtins(),
        "np": _lazy_import("numpy"),
        "math": _lazy_import("math"),
        "itertools": _lazy_import("itertools"),
        "collections": _lazy_import("collections"),
    }


# IN-01 FIX (Indus, 2026-09-29): RLIMIT_NPROC=(1,1) blocks OpenBLAS
# pthread_create during numpy import in the child (verified: child dies at
# ~200ms with KeyboardInterrupt). Pinning thread counts to 1 via env vars
# fixes it with all rlimits intact (verified rc=0). Threads pinned to 1
# also keeps candidate execution deterministic.
_CHILD_THREAD_ENV = {
    "OPENBLAS_NUM_THREADS": "1",
    "OMP_NUM_THREADS": "1",
    "MKL_NUM_THREADS": "1",
    "NUMEXPR_NUM_THREADS": "1",
}


def _child_env() -> dict:
    env = dict(os.environ)
    env.update(_CHILD_THREAD_ENV)
    return env


def _set_limits(timeout_s: float, mem_mb: int) -> None:
    import resource
    try:
        hard = mem_mb * 1024 * 1024
        resource.setrlimit(resource.RLIMIT_AS, (hard, hard))
    except Exception:
        pass
    try:
        cpu = int(timeout_s) + 1
        resource.setrlimit(resource.RLIMIT_CPU, (cpu, cpu))
    except Exception:
        pass
    try:
        resource.setrlimit(resource.RLIMIT_NPROC, (1, 1))
    except Exception:
        pass
    try:
        resource.setrlimit(resource.RLIMIT_FSIZE, (0, 0))
    except Exception:
        pass


_CHILD_WRAPPER = r'''
import sys, json
sys.setrecursionlimit(200)

def _safe_default(o):
    try:
        import numpy as _np
        if isinstance(o, _np.ndarray):
            return o.tolist()
        if isinstance(o, _np.generic):
            return o.item()
    except Exception:
        pass
    try:
        return str(o)
    except Exception:
        return "<unserializable>"

def _emit(d):
    try:
        sys.stdout.write(json.dumps(d, default=_safe_default) + "\n")
        sys.stdout.flush()
    except Exception:
        sys.stdout.write(
            '{"ok": false, "result": null, "error": '
            '"serialization failure", "error_class": "ProtocolError"}\n')
        sys.stdout.flush()

_SAFE_NAMES = (
    "abs", "all", "any", "bool", "dict", "divmod", "enumerate",
    "filter", "float", "frozenset", "int", "isinstance", "len", "list",
    "map", "max", "min", "next", "range", "reversed", "round", "set",
    "sorted", "str", "sum", "tuple", "zip", "print",
)

_b = __builtins__
_SAFE = {n: (_b[n] if isinstance(_b, dict) else getattr(_b, n))
         for n in _SAFE_NAMES}

_ALLOWED_MODULES = {"numpy", "math", "itertools", "collections"}

def _safe_import(name, globals=None, locals=None, fromlist=(), level=0):
    top = (name or "").split(".")[0]
    if level != 0 or top not in _ALLOWED_MODULES:
        raise ImportError(f"module {name!r} not whitelisted in sandbox")
    real = _b["__import__"] if isinstance(_b, dict) else _b.__import__
    return real(name, globals, locals, fromlist, level)

_SAFE["__import__"] = _safe_import

try:
    raw = sys.stdin.read()
    payload = json.loads(raw)
    source = payload["source"]
    entry = payload["entry"]
    args = payload.get("args", {}) or {}
except Exception as e:
    _emit({"ok": False, "result": None,
           "error": f"payload error: {e}",
           "error_class": "ProtocolError"})
    sys.exit(0)

def _lazy(name):
    try:
        return _safe_import(name)
    except Exception:
        return None

try:
    ns = {
        "__builtins__": dict(_SAFE),
        "np": _lazy("numpy"),
        "math": _lazy("math"),
        "itertools": _lazy("itertools"),
        "collections": _lazy("collections"),
    }
    code = compile(source, "<candidate>", "exec")
    exec(code, ns)
    fn = ns.get(entry)
    if fn is None:
        raise NameError(f"entry {entry!r} not defined")
    result = fn(**args)
    _emit({"ok": True, "result": result, "error": None,
           "error_class": None})
except MemoryError as e:
    _emit({"ok": False, "result": None,
           "error": f"MemoryError: {e}",
           "error_class": "MemoryError"})
except RecursionError as e:
    _emit({"ok": False, "result": None,
           "error": f"RecursionError: {e}",
           "error_class": "RecursionError"})
except Exception as e:
    _emit({"ok": False, "result": None,
           "error": f"{type(e).__name__}: {e}",
           "error_class": type(e).__name__})
'''


def run_candidate(source: str, entry: str, args: dict,
                  timeout_s: float = 3.0, mem_mb: int = 512) -> dict:
    t0 = time.time()
    try:
        analyze(source)
    except SandboxViolation as e:
        return {
            "ok": False, "result": None, "error": str(e),
            "error_class": "SandboxViolation",
            "elapsed_ms": int((time.time() - t0) * 1000),
            "timeout": False, "mem_limit_hit": False,
            "stderr_tail": "",
        }
    payload = json.dumps({"source": source, "entry": entry, "args": args})
    popen_kwargs = dict(
        stdin=subprocess.PIPE, stdout=subprocess.PIPE,
        stderr=subprocess.PIPE, text=True,
        env=_child_env(),
    )
    if os.name == "posix":
        popen_kwargs["preexec_fn"] = lambda: _set_limits(timeout_s, mem_mb)
    proc = subprocess.Popen(
        [sys.executable, "-I", "-c", _CHILD_WRAPPER], **popen_kwargs)
    timed_out = False
    out, err = "", ""
    try:
        out, err = proc.communicate(input=payload, timeout=timeout_s)
    except subprocess.TimeoutExpired:
        timed_out = True
        try: proc.kill()
        except Exception: pass
        try: out, err = proc.communicate(timeout=2.0)
        except Exception: pass
    elapsed_ms = int((time.time() - t0) * 1000)
    stderr_tail = (err or "")[-512:]
    if timed_out:
        return {
            "ok": False, "result": None,
            "error": f"timeout after {timeout_s}s",
            "error_class": "TimeoutError",
            "elapsed_ms": elapsed_ms,
            "timeout": True, "mem_limit_hit": False,
            "stderr_tail": stderr_tail,
        }
    if not out or not out.strip():
        return {
            "ok": False, "result": None,
            "error": "child produced no output",
            "error_class": "ProtocolError",
            "elapsed_ms": elapsed_ms,
            "timeout": False, "mem_limit_hit": False,
            "stderr_tail": stderr_tail,
        }
    last_line = out.strip().splitlines()[-1]
    try:
        res = json.loads(last_line)
    except Exception as e:
        return {
            "ok": False, "result": None,
            "error": f"bad child output: {e}; line={last_line[:120]!r}",
            "error_class": "ProtocolError",
            "elapsed_ms": elapsed_ms,
            "timeout": False, "mem_limit_hit": False,
            "stderr_tail": stderr_tail,
        }
    res.setdefault("ok", False)
    res.setdefault("result", None)
    res.setdefault("error", None)
    res.setdefault("error_class", None)
    res["elapsed_ms"] = elapsed_ms
    res["timeout"] = False
    res["stderr_tail"] = stderr_tail
    res["mem_limit_hit"] = (res.get("error_class") == "MemoryError")
    return res


In [ ]:
%%writefile goa/agent/counterexample_schema.json
{
  "$schema": "goa/coder-solver/counterexample/v1.0",
  "type": "object",
  "required": [
    "failure_location",
    "observed",
    "model_predicted",
    "diff_summary",
    "admissible_alternatives",
    "taxonomy_hint",
    "repair_scope"
  ],
  "properties": {
    "failure_location": {
      "type": "object",
      "required": [
        "function",
        "step",
        "action"
      ],
      "properties": {
        "function": {
          "type": "string",
          "enum": [
            "extract_objects",
            "transition",
            "reward",
            "next_action"
          ]
        },
        "step": {
          "type": "integer",
          "minimum": 0
        },
        "action": {
          "type": [
            "string",
            "null"
          ]
        },
        "object_ref": {
          "type": [
            "string",
            "null"
          ]
        }
      }
    },
    "observed": {
      "type": "object",
      "additionalProperties": {
        "type": "string"
      }
    },
    "model_predicted": {
      "type": "object",
      "additionalProperties": {
        "type": "string"
      }
    },
    "diff_summary": {
      "type": "object",
      "required": [
        "cells_differ",
        "first_mismatches",
        "shape_ok",
        "dtype_ok"
      ],
      "properties": {
        "cells_differ": {
          "type": "integer",
          "minimum": 0
        },
        "first_mismatches": {
          "type": "array",
          "items": {
            "type": "string"
          },
          "minItems": 0,
          "maxItems": 3
        },
        "shape_ok": {
          "type": "boolean"
        },
        "dtype_ok": {
          "type": "boolean"
        }
      }
    },
    "admissible_alternatives": {
      "type": "array",
      "items": {
        "type": "string"
      },
      "minItems": 2,
      "maxItems": 4
    },
    "taxonomy_hint": {
      "type": "string",
      "enum": [
        "IndexError",
        "ValueError_shape",
        "timeout_rlimit",
        "replay_mismatch_few",
        "replay_mismatch_many"
      ]
    },
    "repair_scope": {
      "type": "string",
      "enum": [
        "change_only_rule_touching_counterexample",
        "replan_hypothesis",
        "bounds_guard",
        "reshape_guard",
        "complexity_reduction"
      ]
    }
  }
}


In [ ]:
%%writefile goa/agent/replay_verifier.py
# replay_verifier.py — GoA coder-solver. DIR-2026-076/078 (Indus).
"""Transition buffer + exact-replay admission (blueprint §C-1, §8).

Verification is free: the judge is deterministic Python replay, not an LLM.
Fail-closed: every helper degrades to a non-admitting verdict on any error.
"""
from __future__ import annotations
import json
from collections import deque

BUFFER_MAX = 64
MISMATCH_FEW_MAX = 3  # <=3 differing cells -> repair; >3 -> replan (frozen)


def _norm(grid) -> str:
    """Canonical serialization for exact comparison (astype-safe)."""
    if hasattr(grid, "tolist"):
        grid = grid.tolist()
    return json.dumps(grid, sort_keys=True)


class TransitionBuffer:
    """Bounded FIFO of (state, action, next_state)."""

    def __init__(self, maxlen: int = BUFFER_MAX):
        self._q = deque(maxlen=maxlen)

    def append(self, state, action, next_state) -> None:
        self._q.append((state, action, next_state))

    def as_list(self):
        return list(self._q)

    def __len__(self):
        return len(self._q)

    def clear(self) -> None:
        self._q.clear()


def replay_verify(source: str, run_candidate, buffer: "TransitionBuffer") -> dict:
    """Exact replay of `source`'s next_grid(state, action) over the buffer.

    Returns {"admitted": bool, "mismatches": int, "first_mismatches": [str],
             "error": str|None, "error_class": str|None}.
    """
    transitions = buffer.as_list()
    if not transitions:
        return {"admitted": False, "mismatches": 0, "first_mismatches": [],
                "error": "empty buffer", "error_class": "EmptyBuffer"}
    mismatches, first_mismatches = 0, []
    for state, action, next_state in transitions:
        r = run_candidate(source, "next_grid",
                          {"state": state, "action": action},
                          timeout_s=3.0)
        if not r.get("ok"):
            return {"admitted": False, "mismatches": 0, "first_mismatches": [],
                    "error": r.get("error"), "error_class": r.get("error_class")}
        try:
            if _norm(r.get("result")) != _norm(next_state):
                mismatches += 1
                if len(first_mismatches) < MISMATCH_FEW_MAX:
                    first_mismatches.append(f"t{mismatches}")
        except Exception as e:
            return {"admitted": False, "mismatches": 0, "first_mismatches": [],
                    "error": f"compare failed: {e}", "error_class": "CompareError"}
    return {"admitted": mismatches == 0, "mismatches": mismatches,
            "first_mismatches": first_mismatches, "error": None, "error_class": None}


def build_counterexample(function: str, step: int, action, observed: dict,
                         model_predicted: dict, cells_differ: int,
                         first_mismatches: list, shape_ok: bool = True,
                         dtype_ok: bool = True, taxonomy_hint: str = "replay_mismatch_few",
                         repair_scope: str = "change_only_rule_touching_counterexample",
                         alternatives: list = None) -> dict:
    """Schema-conformant counterexample (counterexample_schema.json, frozen)."""
    if alternatives is None:
        alternatives = ["player can push blue blocks",
                        "blue block is static + player moves diagonally",
                        "frame-diff encoding error"][:2]
    return {
        "failure_location": {"function": function, "step": step,
                             "action": action, "object_ref": None},
        "observed": {k: str(v) for k, v in observed.items()},
        "model_predicted": {k: str(v) for k, v in model_predicted.items()},
        "diff_summary": {"cells_differ": cells_differ,
                         "first_mismatches": [str(m) for m in first_mismatches[:3]],
                         "shape_ok": shape_ok, "dtype_ok": dtype_ok},
        "admissible_alternatives": [str(a) for a in alternatives[:4]],
        "taxonomy_hint": taxonomy_hint,
        "repair_scope": repair_scope,
    }


In [ ]:
%%writefile goa/agent/cegis_taxonomy.py
# cegis_taxonomy.py — GoA coder-solver. DIR-2026-076/078 (Indus).
"""Error taxonomy (blueprint §C-3): failure -> (taxonomy_hint, repair_scope)."""
from __future__ import annotations

TAXONOMY = ("IndexError", "ValueError_shape", "timeout_rlimit",
            "replay_mismatch_few", "replay_mismatch_many")

_SCOPE = {
    "IndexError": "bounds_guard",
    "ValueError_shape": "reshape_guard",
    "timeout_rlimit": "complexity_reduction",
    "replay_mismatch_few": "change_only_rule_touching_counterexample",
    "replay_mismatch_many": "replan_hypothesis",
}


def classify(error_class: str, mismatches: int = 0) -> tuple:
    """Map a runtime failure onto the frozen taxonomy.

    Rules (frozen):
      timeout / RLIMIT hits          -> timeout_rlimit
      MemoryError                    -> timeout_rlimit (resource exhaustion)
      IndexError                     -> IndexError
      ValueError w/ shape semantics   -> ValueError_shape
      replay mismatch count <= 3      -> replay_mismatch_few
      replay mismatch count >  3      -> replay_mismatch_many
    """
    ec = (error_class or "")
    if ec in ("TimeoutError", "MemoryError", "SandboxViolation"):
        hint = "timeout_rlimit" if ec != "SandboxViolation" else "IndexError"
        return (hint, _SCOPE[hint])
    if ec == "IndexError":
        return ("IndexError", _SCOPE["IndexError"])
    if ec == "ValueError":
        return ("ValueError_shape", _SCOPE["ValueError_shape"])
    if ec in ("ReplayMismatch", "CompareError", "EmptyBuffer"):
        hint = ("replay_mismatch_few" if mismatches <= 3
                else "replay_mismatch_many")
        return (hint, _SCOPE[hint])
    # unknown -> conservative default: replan
    return ("replay_mismatch_many", _SCOPE["replay_mismatch_many"])


In [ ]:
%%writefile goa/agent/cegis_router.py
# cegis_router.py — GoA coder-solver. DIR-2026-076/078 (Indus).
"""CEGIS routing (blueprint §C-3, frozen): repair vs replan decision."""
from __future__ import annotations

try:
    import cegis_taxonomy
except ImportError:
    from agent import cegis_taxonomy

REPAIR, REPLAN = "repair", "replan"


def route(error_class: str, mismatches: int = 0) -> str:
    """few-mismatch -> repair (touched rule only); everything else -> replan.

    A-CEGIS-13/16: replan forces a fresh reasoner hypothesis (Zone-2 reset
    + jolt). Repair only rewrites the rule touching the counterexample.
    """
    hint, _scope = cegis_taxonomy.classify(error_class, mismatches)
    if hint == "replay_mismatch_few":
        return REPAIR
    if hint == "replay_mismatch_many":
        return REPLAN
    # IndexError / ValueError_shape / timeout_rlimit: repair once, then
    # replan if the repair does not stick (guard-based scopes).
    return REPAIR


def max_repairs(taxonomy_hint: str) -> int:
    return 2 if taxonomy_hint in ("replay_mismatch_few", "IndexError",
                                  "ValueError_shape") else 0


In [ ]:
%%writefile goa/agent/adversarial_critic.py
# adversarial_critic.py — GoA coder-solver. DIR-2026-076/078 (Indus).
"""Structural spec critic (blueprint §1.5 VeriHarness pattern, zero-LLM).

Checks the spec the reasoner emits BEFORE any generation is spent:
signature-first, micro-example, assumptions listed, conservative
interpretation flag, alternatives 2-4. Structural only — fail-closed:
issues are advisory to the caller; an empty issue list never forces a
replan by itself.
"""
from __future__ import annotations

REQUIRED_MARKERS = ("def ",)          # signature-first
MAX_ALTERNATIVES = 4
MIN_ALTERNATIVES = 2


def critique_spec(spec: str, alternatives=None) -> list:
    issues = []
    if not spec or not spec.strip():
        return ["empty spec"]
    if not any(m in spec for m in REQUIRED_MARKERS):
        issues.append("spec does not lead with a function signature")
    if "return" not in spec:
        issues.append("spec does not state a return contract")
    if "assum" not in spec.lower():
        issues.append("spec does not list assumptions")
    if "conservative" not in spec.lower():
        issues.append("spec does not carry a conservative-interpretation directive")
    if alternatives is not None:
        n = len(alternatives)
        if not (MIN_ALTERNATIVES <= n <= MAX_ALTERNATIVES):
            issues.append(f"alternatives count {n} outside [2, 4]")
    return issues


In [ ]:
%%writefile goa/agent/inference_layer_v1_6.py
# inference_layer_v1_6.py — deployed-IL shim (DIR-078).
"""In the Run-21 notebook the deployed module IS v1.6
(goa/agent/inference_layer.py). This shim exists so that
`from inference_layer_v1_6 import ...` (diagnose extension, offline
tests) resolves against the deployed module without duplication."""
try:
    from inference_layer import *  # noqa: F401,F403
    from inference_layer import _log_call, call_log, build_spec, \
        CODER_CODE_CAP, CODER_CODE_CAP_EXCEEDED, _DEFAULT_PARAMS, \
        REASONER_ANSWER_CAP, DiagnoseResult
except ImportError:
    from agent.inference_layer import *  # noqa: F401,F403
    from agent.inference_layer import _log_call, call_log, build_spec, \
        CODER_CODE_CAP, CODER_CODE_CAP_EXCEEDED, _DEFAULT_PARAMS, \
        REASONER_ANSWER_CAP, DiagnoseResult


In [ ]:
%%writefile goa/agent/inference_layer_v1_6_additions.py
# inference_layer_v1_6_additions.py — GoA coder-solver. DIR-2026-076/078.
"""Thin re-export surface used by my_agent v1.0.19 (_il16_module).

Keeps my_agent decoupled from the full IL v1.6 module: only the
coder-facing constants and helpers cross this boundary. In the Run-21
notebook the deployed module IS v1.6 (goa/agent/inference_layer.py),
so we import from there first; the standalone inference_layer_v1_6
module name is the fallback for offline tests.
"""
from __future__ import annotations

try:
    from inference_layer import (CODER_CODE_CAP,
                                 CODER_CODE_CAP_EXCEEDED,
                                 build_spec, call_log, _log_call,
                                 _DEFAULT_PARAMS)
    _AVAILABLE = True
except ImportError:
    try:
        from agent.inference_layer import (CODER_CODE_CAP,
                                           CODER_CODE_CAP_EXCEEDED,
                                           build_spec, call_log, _log_call,
                                           _DEFAULT_PARAMS)
        _AVAILABLE = True
    except ImportError:
        from inference_layer_v1_6 import (CODER_CODE_CAP,
                                          CODER_CODE_CAP_EXCEEDED,
                                          build_spec, call_log, _log_call,
                                          _DEFAULT_PARAMS)
        _AVAILABLE = True
    except ImportError:
        _AVAILABLE = False
        CODER_CODE_CAP = 2048
        CODER_CODE_CAP_EXCEEDED = "coder_code_cap_exceeded"
        build_spec = None
        call_log = None
        _log_call = None
        _DEFAULT_PARAMS = {"temperature": 0.0, "top_p": 1.0, "top_k": -1}


In [ ]:
%%writefile goa/agent/coder_solver.py
# coder_solver.py — GoA coder-solver core. DIR-2026-076/078 (Indus).
"""Blueprint §8 core architecture, implemented fail-closed:

  TRIGGER (my_agent) -> reasoner emits TYPED SPEC -> CoderSolver:
    spec -> sandbox_v6 exec -> REPLAY VERIFY (pure Python, zero LLM)
      PASS -> admit + GuardedExecutor (1 action at a time)
      FAIL -> counterexample (frozen schema) -> repair (<=2/attempt)
              -> still FAIL -> replan signal -> my_agent reasoner path
  COUNTERS (frozen): 1 attempt = 1 synthesis + <=2 repairs;
                      <=3 attempts/level, <=6/game (18 generations worst).
  FALLBACK: any exception anywhere -> non-admitting result; the game
            NEVER aborts because of this module.
"""
from __future__ import annotations
import logging

def _dual(modname):
    """Flat import first (offline), then package import (notebook: goa/agent)."""
    try:
        import importlib
        return importlib.import_module(modname)
    except ImportError:
        return importlib.import_module("agent." + modname)

cegis_taxonomy = _dual("cegis_taxonomy")
cegis_router = _dual("cegis_router")
replay_verifier = _dual("replay_verifier")

logger = logging.getLogger(__name__)

try:
    _sb = _dual("sandbox_v6")
except ImportError:
    _sb = None

try:
    _build_spec = _dual("inference_layer_v1_6_additions").build_spec
except (ImportError, AttributeError):
    _build_spec = None

ATTEMPTS_PER_LEVEL_CAP = 3
ATTEMPTS_PER_GAME_CAP = 6
REPAIRS_PER_ATTEMPT_CAP = 2


class AttemptCounters:
    """Frozen attempt arithmetic (T-ATTEMPT-1 semantics)."""

    def __init__(self, max_apl=ATTEMPTS_PER_LEVEL_CAP,
                 max_apg=ATTEMPTS_PER_GAME_CAP,
                 max_rep=REPAIRS_PER_ATTEMPT_CAP):
        self.max_apl, self.max_apg, self.max_rep = max_apl, max_apg, max_rep
        self.apl = 0
        self.apg = 0
        self.repairs = 0
        self.generations = 0

    def begin_attempt(self):
        if self.apl >= self.max_apl or self.apg >= self.max_apg:
            return False
        self.apl += 1; self.apg += 1; self.repairs = 0
        self.generations += 1
        return True

    def begin_repair(self):
        if self.repairs >= self.max_rep:
            return False
        self.repairs += 1
        self.generations += 1
        return True

    def new_level(self):
        self.apl = 0

    def new_game(self):
        self.apl = 0; self.apg = 0; self.generations = 0


class SolveResult:
    """Fields consumed by my_agent v1.0.19 (see its mock tests T-64/T-65):
    admitted, replan_requested, mode, source (+ diagnostics)."""

    def __init__(self, admitted=False, replan_requested=False, mode="world",
                 source="", counterexample=None, error=None, error_class=None,
                 generations=0):
        self.admitted = admitted
        self.replan_requested = replan_requested
        self.mode = mode
        self.source = source
        self.counterexample = counterexample
        self.error = error
        self.error_class = error_class
        self.generations = generations


def _extract_code(raw: str) -> str:
    """Strip markdown fences from the coder's reply."""
    if not raw:
        return ""
    text = raw.strip()
    if "```" in text:
        parts = text.split("```")
        for p in parts:
            if "def " in p:
                body = p
                if body.startswith("python"):
                    body = body[6:]
                return body.strip()
    return text


def _spec_text(task_ctx: dict) -> str:
    if _build_spec is not None:
        try:
            return _build_spec(task_ctx)
        except Exception as e:
            logger.info("build_spec failed: %s", e)
    # minimal internal fallback (fail-closed, keeps chain alive)
    return (f"TASK: {task_ctx.get('task', '')}\n"
            f"HYPOTHESIS: {task_ctx.get('hypothesis', '')}\n"
            f"Return ONLY fenced Python defining "
            f"{task_ctx.get('entry_signature', 'next_action(frame, history)')}.")


class CoderSolver:
    """Spec -> sandbox exec -> replay verify -> admit / counterexample loop."""

    def __init__(self, call_llm, run_candidate=None):
        # call_llm(spec: str, mode: str) -> str  (my_agent._call_coder)
        self._call_llm = call_llm
        # LA-1 FIX (Indus, 2026-09-29 recheck audit): my_agent constructs
        # CoderSolver(call_llm=...) WITHOUT run_candidate; the old default
        # (NoSandbox lambda) silently killed every admit. Default to the
        # real sandbox when importable; explicit arg still wins; offline
        # without sandbox stays fail-closed NoSandbox.
        if run_candidate is not None:
            self._run = run_candidate
        elif _sb is not None:
            self._run = _sb.run_candidate
        else:
            self._run = (lambda *a, **k: {"ok": False,
                                         "error": "sandbox unavailable",
                                         "error_class": "NoSandbox"})
        self.counters = AttemptCounters()

    # ---------- mode 2: world model ----------
    def solve_world_model(self, task_ctx, buffer):
        return self._solve(task_ctx, buffer, "world", "next_grid")

    # ---------- mode 1: action program ----------
    def solve_action_program(self, task_ctx, buffer):
        return self._solve(task_ctx, buffer, "action", "next_action")

    # ---------- shared loop ----------
    def _solve(self, task_ctx, buffer, mode, entry):
        ctx = dict(task_ctx or {})
        ctx["mode"] = mode
        ctx.setdefault("entry_signature",
                        "next_grid(state, action)" if mode == "world"
                        else "next_action(frame, history)")
        # counters-before-cleverness: refuse before generating anything
        if not self.counters.begin_attempt():
            return SolveResult(mode=mode,
                               error="attempt cap reached",
                               error_class="CapExceeded")

        raw = ""
        try:
            raw = self._call_llm(_spec_text(ctx), mode)
        except Exception as e:
            logger.info("coder call failed: %s", e)
            return SolveResult(mode=mode, error=str(e), error_class="CallFailed")

        source = _extract_code(raw)
        if not source or "def " not in source:
            return SolveResult(mode=mode, error="no code in coder reply",
                               error_class="NoCode", generations=1)

        # static layer-1 first: never exec un-analyzed code
        if _sb is not None:
            try:
                _sb.analyze(source)
            except Exception as e:
                return SolveResult(mode=mode, error=f"L1: {e}",
                                   error_class="SandboxViolation", generations=1)

        # replay verify (mode "world") / dry exec (mode "action")
        if mode == "world":
            verdict = replay_verifier.replay_verify(source, self._run, buffer)
            if verdict.get("admitted"):
                return SolveResult(admitted=True, mode=mode, source=source,
                                   generations=1)
            return self._repair_loop(ctx, buffer, mode, entry, source, verdict)

        # action mode: single smoke exec, then admit (guarded per action)
        r = self._run(source, entry, {"frame": [[0]], "history": []},
                      timeout_s=3.0)
        if r.get("ok"):
            return SolveResult(admitted=True, mode=mode, source=source,
                               generations=1)
        return self._repair_loop(ctx, buffer, mode, entry, source, r)

    def _repair_loop(self, ctx, buffer, mode, entry, source, verdict):
        error_class = verdict.get("error_class") or "ReplayMismatch"
        mismatches = verdict.get("mismatches", 0)
        hint, scope = cegis_taxonomy.classify(error_class, mismatches)
        cx = replay_verifier.build_counterexample(
            function=entry, step=0, action=None,
            observed={"verdict": "recorded transitions"},
            model_predicted={"verdict": "candidate prediction"},
            cells_differ=mismatches,
            first_mismatches=verdict.get("first_mismatches") or [],
            taxonomy_hint=hint, repair_scope=scope,
            alternatives=ctx.get("alternatives"))
        gens = 1
        # repair only when routing says so, and only <= max_repairs
        if cegis_router.route(error_class, mismatches) == cegis_router.REPAIR:
            for _ in range(cegis_router.max_repairs(hint)):
                if not self.counters.begin_repair():
                    break
                gens += 1
                try:
                    repair_prompt = (_spec_text(ctx) +
                        f"\nREPAIR ONLY the rule touching this counterexample"
                        f" (scope: {scope}).\nCOUNTEREXAMPLE: {cx}"
                        f"\nPrevious code:\n{source}")
                    raw = self._call_llm(repair_prompt, mode)
                except Exception as e:
                    logger.info("repair call failed: %s", e)
                    break
                cand = _extract_code(raw)
                if not cand or "def " not in cand:
                    continue
                if _sb is not None:
                    try:
                        _sb.analyze(cand)
                    except Exception:
                        continue
                if mode == "world":
                    v2 = replay_verifier.replay_verify(cand, self._run, buffer)
                    if v2.get("admitted"):
                        return SolveResult(admitted=True, mode=mode,
                                           source=cand, generations=gens)
                else:
                    r2 = self._run(cand, entry,
                                   {"frame": [[0]], "history": []}, timeout_s=3.0)
                    if r2.get("ok"):
                        return SolveResult(admitted=True, mode=mode,
                                           source=cand, generations=gens)
        # exhausted: replan signal ONLY on many-mismatch (A-CEGIS-13/16)
        replan = hint in ("replay_mismatch_many", "timeout_rlimit")
        return SolveResult(admitted=False, replan_requested=replan,
                           mode=mode, counterexample=cx,
                           error=verdict.get("error") or f"{hint}",
                           error_class=error_class, generations=gens)


class GuardedExecutor:
    """Admitted program, one action at a time (Tycho guarded execution).

    my_agent records the transition and discards the route on mismatch;
    any exception here degrades to None -> reasoner path.
    """

    def __init__(self, source: str, mode: str = "action"):
        if _sb is None:
            raise RuntimeError("sandbox unavailable")
        self._source = source
        self._mode = mode
        self._entry = "next_action" if mode != "world" else "next_grid"
        _sb.analyze(source)  # raises SandboxViolation -> admit fails upstream

    def next_action(self, grid, history):
        try:
            r = _sb.run_candidate(self._source, self._entry,
                                   {"frame": grid, "history": list(history)},
                                   timeout_s=3.0)
            if r.get("ok"):
                return r.get("result")
            logger.info("guarded exec failed: %s", r.get("error"))
            return None
        except Exception as e:
            logger.info("guarded exec raised: %s", e)
            return None


In [ ]:
%%writefile goa/agent/my_agent.py
"""agent/my_agent.py — Kaggle entry point + single-file compiler, v1.0.21.

Changelog:
  v1.0.21 — Martial-law core audit (Chief order, 2026-09-30): Object Vision.
            extract_objects() ka structured output (id/color/size/bbox) ab
            cm.set_observation(objects=...) se observation block mein render
            hota hai — model ko RLE rows se column-arithmetic karke target
            nikalna nahi padta (core audit finding #1-#3). CERTIFIED_VERSIONS:
            context_manager 1.5 -> 1.6. + T-68/T-69 (render + palette
            cross-check).
  v1.0.20 — Run-23 postmortem (Chief order, 2026-09-30): proactive coder
            trigger. v1.0.19 mein coder SIRF reactive tha (jolt/divergence)
            — Run-23 mein reasoner ne alag actions try karte rahe, loop
            kabhi nahi pada, isliye 4 games mein ZERO coder calls. Naya
            config seam: coder_proactive_actions_at_level = N => N actions
            same level par bina advance ke par bhi coder attempt hota hai.
            Default None = v1.0.19 behavior (backward-compatible, T-suite).
            + T-66/T-67 tests.
  v1.0.19 — DIR/2026-076 (Operation Coder=Solver) + AGA fixes
            A-BH-01..A-BH-14 + A-CEGIS-13 + A-CEGIS-16 + A-CEGIS-17 +
            A-CEGIS-18 + A-CEGIS-19:
            (1)  Coder-solver wired via CoderSolver + ReplayVerifier.
            (2)  Persistent guarded execution (A-BH-04).
            (3)  _record_transition feeds the replay buffer.
            (4)  reset_level clears pending state (A-BH-03).
            (5)  AR-08: coder attempts vs deep-calls separate counters.
            (6)  Guarded-success clears stale jolt injections (A-BH-06).
            (7)  CERTIFIED_VERSIONS: inference_layer 1.5 -> 1.6.
            (8)  Lazy solver init — fail-closed additivity.
            (9)  coder_actions / coder_actions_level telemetry.
           (10)  Empty-buffer guard prevents budget drain (A-BH-10).
           (11)  Level-start trigger removed (A-BH-13).
           (12)  A-CEGIS-13: REPLAN signal consumed by agent loop.
           (13)  A-CEGIS-16: world-model REPLAN short-circuits BEFORE
                 action-program fallback; signal never overwritten.
           (14)  A-CEGIS-17: _handle_coder_replan sets _jolt_pending=True
                 to force a fresh deep reasoner call.
           (15)  A-CEGIS-18: defensive _cs_module / GuardedExecutor guard
                 at the top of _maybe_admit_coder — opaque AttributeError
                 avoided if the coder chain is partially importable.
           (16)  A-CEGIS-19: T-65 dead lines removed.
  v1.0.18 — DIR/2026-070 + DIR/2026-071 + DIR/2026-072 + INDUS-DIR-070.
"""
from __future__ import annotations
import argparse, hashlib, json, logging, re, statistics, sys, textwrap
from collections import Counter, deque
from dataclasses import dataclass
from pathlib import Path
from typing import (Any, Callable, Dict, FrozenSet, Iterable, List,
                    Mapping, Optional, Sequence, Tuple)

logger = logging.getLogger(__name__)

CERTIFIED_VERSIONS = {
    "context_manager": "1.6",
    "inference_layer": "1.6",
    "grid_text": "1.0.8",
    "action_pipeline": "1.0.5",
}
HARNESS_OWNED: FrozenSet[str] = frozenset({"arcengine", "agents.agent"})
DISPATCHED_LOG_CAP = 200
PROMPT_LOG_CAP = 50
JOLT_HISTORY_CAP = 50
CODER_ATTEMPTS_PER_GAME_CAP = 6
_WARNED_UNKNOWN_IDS: set = set()
_TACTICS_JOURNAL: List[Dict[str, Any]] = []


def _journal_brief(max_entries=6, max_chars=400):
    if not _TACTICS_JOURNAL: return ""
    parts = []
    for e in _TACTICS_JOURNAL[-max_entries:]:
        parts.append("[{g}:lv={lv}:{r}]".format(
            g=e.get("game", "?"), lv=e.get("levels", 0),
            r=str(e.get("rules", ""))[:60]))
    return " ".join(parts)[:max_chars]


try:
    from agent import context_manager as _cm_module
except ImportError:
    try: import context_manager as _cm_module
    except ImportError: _cm_module = None
try:
    from agent import inference_layer as _il_module
except ImportError:
    try: import inference_layer as _il_module
    except ImportError: _il_module = None
try:
    from agent import grid_text as _gt_module
except ImportError:
    try: import grid_text as _gt_module
    except ImportError: _gt_module = None
try:
    from agent import action_pipeline as _ap_module
except ImportError:
    try: import action_pipeline as _ap_module
    except ImportError: _ap_module = None

try:
    import coder_solver as _cs_module
except ImportError:
    try:
        from agent import coder_solver as _cs_module
    except ImportError:
        _cs_module = None
try:
    import replay_verifier as _rv_module
except ImportError:
    try:
        from agent import replay_verifier as _rv_module
    except ImportError:
        _rv_module = None
try:
    import inference_layer_v1_6_additions as _il16_module
except ImportError:
    try:
        from agent import inference_layer_v1_6_additions as _il16_module
    except ImportError:
        _il16_module = None

try:
    from arcengine import FrameData, GameAction, GameState  # noqa
    _HAS_ARCENGINE = True
except ImportError:
    FrameData = GameAction = GameState = None
    _HAS_ARCENGINE = False
try:
    from agents.agent import Agent as _FrameworkAgent  # noqa
    _HAS_FRAMEWORK_AGENT = True
except ImportError:
    class _FrameworkAgent:
        def __init__(self, *a, **k): pass
    _HAS_FRAMEWORK_AGENT = False


DEFAULT_CONFIG: Dict[str, Any] = {
    "reasoner_endpoint": "qwen3.8-27b",
    "coder_endpoint": "qwen3-coder-30b-a3b",
    "offline_fp8": False,
    "reasoner_mode": "fast",
    "connectivity": 4,
    "coder_solver_enabled": True,
}


class AgentError(RuntimeError): ...
class VersionMismatch(AgentError): ...
class VersionMissing(AgentError): ...
class ComponentUnavailable(AgentError): ...


def check_certified_versions(cm, il, gt, ap, expected=None,
                              version_manifest=None):
    expected = expected or CERTIFIED_VERSIONS
    manifest = version_manifest or {}
    verified = {}
    for name, mod in (("context_manager", cm), ("inference_layer", il),
                      ("grid_text", gt), ("action_pipeline", ap)):
        exp = expected[name]
        declared = getattr(mod, "__version__", None) or manifest.get(name)
        if declared is None:
            raise VersionMissing(
                f"certified module {name!r} has no __version__ or "
                f"manifest entry (expected {exp!r})")
        if declared != exp:
            raise VersionMismatch(f"{name}: {declared!r} != {exp!r}")
        verified[name] = True
    return verified


@dataclass(frozen=True)
class LocalCalibration:
    game_id: str
    is_reject: bool
    retry_budget: int
    debounce_steps: int
    pause_before_retry_s: float
    pause_before_fresh_s: float
    resolved_id: Optional[str] = None
    known: bool = True


def _prefix(game_id):
    s = str(game_id)
    return s.split("-", 1)[0] if "-" in s else s


def _require_attr(mod, name, *, reject_none=True):
    if not hasattr(mod, name):
        raise AgentError(f"action_pipeline missing {name!r}")
    v = getattr(mod, name)
    if reject_none and v is None:
        raise AgentError(f"action_pipeline.{name} is None")
    return v


def _require_clean_key(clean, key):
    try:
        if isinstance(clean, Mapping):
            if key not in clean:
                raise AgentError(f"CLEAN_SET missing {key!r}")
            return clean[key]
        if not hasattr(clean, key):
            raise AgentError(f"CLEAN_SET missing {key!r}")
        return getattr(clean, key)
    except AgentError: raise
    except Exception as e:
        raise AgentError(f"CLEAN_SET[{key!r}] unreadable: {e}") from e


def _row_get(row, key, default=None):
    if row is None: return default
    if isinstance(row, Mapping): return row.get(key, default)
    return getattr(row, key, default)


def _row_get_strict(row, key):
    if row is None:
        raise AgentError(f"calibration row is None (reading {key!r})")
    if isinstance(row, Mapping):
        if key not in row:
            raise AgentError(f"calibration row missing {key!r}")
        return row[key]
    if not hasattr(row, key):
        raise AgentError(f"calibration row missing {key!r}")
    return getattr(row, key)


def _aggregate_clean_defaults(rows):
    budgets, debounces = [], []
    for r in rows:
        if _row_get_strict(r, "is_reject"): continue
        b = _row_get(r, "retry_budget"); d = _row_get(r, "debounce_steps")
        if b is not None: budgets.append(int(b))
        if d is not None: debounces.append(int(d))
    if not budgets or not debounces:
        raise AgentError("DEFAULT_CALIBRATION has no clean rows to aggregate")
    return min(budgets), int(round(statistics.median(debounces)))


def lookup_calibration(game_id, ap_module):
    table = _require_attr(ap_module, "DEFAULT_CALIBRATION")
    clean = _require_attr(ap_module, "CLEAN_SET")
    reject_ids = tuple(_require_attr(ap_module, "REJECT_LIST_IDS"))

    resolved_id = None
    if game_id in table:
        resolved_id = game_id
    else:
        pref = _prefix(game_id)
        candidates = [k for k in table if _prefix(k) == pref]
        if len(candidates) == 1:
            resolved_id = candidates[0]

    effective_id = resolved_id if resolved_id is not None else game_id
    is_reject = effective_id in reject_ids

    if resolved_id is None:
        default_budget, default_debounce = _aggregate_clean_defaults(
            table.values())
        if game_id not in _WARNED_UNKNOWN_IDS:
            _WARNED_UNKNOWN_IDS.add(game_id)
            logger.warning(
                "unknown game_id %r; using aggregated clean defaults "
                "(retry_budget=%d, debounce=%d)", game_id,
                default_budget, default_debounce)
        return LocalCalibration(
            game_id=game_id, is_reject=False,
            retry_budget=default_budget, debounce_steps=default_debounce,
            pause_before_retry_s=float(_require_clean_key(clean, "pause_retry_s")),
            pause_before_fresh_s=float(_require_clean_key(clean, "pause_fresh_s")),
            resolved_id=None, known=False)

    row = table[resolved_id]
    b = _row_get(row, "retry_budget"); d = _row_get(row, "debounce_steps")
    if b is None or d is None:
        default_budget, default_debounce = _aggregate_clean_defaults(
            table.values())
        retry_budget = int(b) if b is not None else default_budget
        debounce = int(d) if d is not None else default_debounce
    else:
        retry_budget, debounce = int(b), int(d)

    def _pause(attr, key):
        v = _row_get(row, attr)
        if v is not None: return float(v)
        return float(_require_clean_key(clean, key))

    return LocalCalibration(
        game_id=game_id, is_reject=is_reject,
        retry_budget=retry_budget, debounce_steps=debounce,
        pause_before_retry_s=_pause("pause_before_retry_s", "pause_retry_s"),
        pause_before_fresh_s=_pause("pause_before_fresh_s", "pause_fresh_s"),
        resolved_id=resolved_id, known=True)


STUCK_CLEAN = 3
STUCK_MECHANIC = 4
JITTER_RADIUS = 2
DEADLOCK_WINDOWS = 2

_INSTRUCTION_RE = re.compile(
    r"^(?P<aid>[A-Za-z0-9_\-]+)"
    r"(?:\s+r(?P<r>\d+)\s+c(?P<c>\d+))?$")


@dataclass
class StuckSig:
    action_id: str
    target_xy: Optional[Tuple[int, int]]
    target_hash: Optional[str]
    target_bbox: Optional[Tuple[int, int, int, int]] = None


@dataclass
class JoltRequest:
    kind: str
    blacklist_sig: Optional[StuckSig] = None

    def to_dict(self):
        return {"kind": self.kind,
                "blacklist_sig": None if self.blacklist_sig is None else {
                    "action_id": self.blacklist_sig.action_id,
                    "target_xy": (None if self.blacklist_sig.target_xy is None
                                  else list(self.blacklist_sig.target_xy)),
                    "target_hash": self.blacklist_sig.target_hash,
                    "target_bbox": (None if self.blacklist_sig.target_bbox
                                    is None
                                    else list(self.blacklist_sig.target_bbox))}}


class LoopDetector:
    def __init__(self, game_id, is_reject, context_mgr, debounce_window,
                 allow_repetition_while=None, on_repetition_allowed=None):
        self.game_id = game_id
        self._threshold = STUCK_MECHANIC if is_reject else STUCK_CLEAN
        self._cm = context_mgr
        self._debounce_window = int(debounce_window)
        self._streak: List[StuckSig] = []
        self._frozen_count = 0
        self._last_objset_fp = None
        self._jolt_history = deque(maxlen=JOLT_HISTORY_CAP)
        self._allow_repetition = allow_repetition_while or (lambda: False)
        self._on_repetition_allowed = on_repetition_allowed or (lambda: None)

    def reset(self):
        self._streak.clear()
        self._frozen_count = 0
        self._last_objset_fp = None

    def tick(self, *, action_id, target_xy, target_hash, target_bbox,
             state_changed, objset_fp):
        if not state_changed:
            new_sig = StuckSig(action_id, target_xy, target_hash, target_bbox)
            if self._streak and self._matches(new_sig, self._streak[-1]):
                self._streak.append(new_sig)
            else:
                self._streak = [new_sig]
            if len(self._streak) > self._threshold:
                if self._allow_repetition():
                    self._on_repetition_allowed()
                else:
                    req = self._fire_stuck(new_sig)
                    self._jolt_history.append(req)
                    self._frozen_count = 0
                    self._last_objset_fp = None
                    return req
        else:
            self._streak.clear()

        if self._last_objset_fp is None:
            self._last_objset_fp = objset_fp
            self._frozen_count = 1
        elif objset_fp == self._last_objset_fp:
            self._frozen_count += 1
        else:
            self._last_objset_fp = objset_fp
            self._frozen_count = 1

        if self._frozen_count >= DEADLOCK_WINDOWS * self._debounce_window:
            self._frozen_count = 0
            self._last_objset_fp = None
            self._streak.clear()
            req = JoltRequest(kind="deadlock")
            self._jolt_history.append(req)
            return req
        return None

    def _matches(self, a, b):
        if a.action_id != b.action_id: return False
        if a.target_hash != b.target_hash: return False
        if a.target_xy is None and b.target_xy is None: return True
        if a.target_xy is None or b.target_xy is None: return False
        return (abs(a.target_xy[0] - b.target_xy[0]) <= JITTER_RADIUS
                and abs(a.target_xy[1] - b.target_xy[1]) <= JITTER_RADIUS)

    def _fire_stuck(self, sig):
        self._cm.register_dead_signature(self._format_blacklist(sig))
        self._streak.clear()
        return JoltRequest(kind="stuck", blacklist_sig=sig)

    @staticmethod
    def _format_blacklist(sig):
        if sig is None:
            return ("SYSTEM OVERRIDE: repeated action failed with no state "
                    "change.  DO NOT REPEAT.")
        sh = (sig.target_hash or "")[:6] or "<unknown>"
        if sig.target_bbox is not None:
            r0, r1, c0, c1 = sig.target_bbox
            bb = f"r{r0}-{r1} c{c0}-{c1}"
        elif sig.target_xy is not None:
            bb = f"r{sig.target_xy[0]} c{sig.target_xy[1]}"
        else:
            bb = "r? c?"
        return (f"SYSTEM OVERRIDE: action {sig.action_id} on object "
                f"<{sh}, {bb}> failed. DO NOT REPEAT.")

    def streak_length(self): return len(self._streak)
    def frozen_count(self): return self._frozen_count
    def jolt_history(self): return list(self._jolt_history)


@dataclass
class CRLFIntakeEntry:
    key: str
    cr_byte_count: int
    md5_original: str
    md5_normalized: str


class Agent:
    _DEEP_TRIGGER_MAX_PER_GAME = 8

    def __init__(self, game_id, config=None, *, cm=None, il=None,
                 gt=None, ap=None, model_config_hook=None,
                 version_manifest=None):
        self.game_id = game_id
        self.config = dict(DEFAULT_CONFIG)
        self._crlf_log = []
        if config:
            self.config.update(self._normalize_crlf(config))
        self._deep_cap()
        self._cm = cm if cm is not None else self._require(_cm_module, "context_manager")
        self._il = il if il is not None else self._require(_il_module, "inference_layer")
        self._gt = gt if gt is not None else self._require(_gt_module, "grid_text")
        self._ap_mod = ap if ap is not None else self._require(_ap_module, "action_pipeline")
        self.version_verified = check_certified_versions(
            self._cm, self._il, self._gt, self._ap_mod,
            version_manifest=version_manifest)
        self._pipeline = self._ap_mod.ActionPipeline(
            game_id=game_id, dispatch=self._on_dispatch, patch_hook=None)
        self._row = lookup_calibration(game_id, self._ap_mod)
        self._deep_calls_this_game = 0
        self._actions_at_level = 0
        self._jolt_pending = False
        self._divergence_pending = False
        self._switch_decision_pending = False
        self._p5_budget_remaining = 2
        self._loop = LoopDetector(
            game_id=game_id, is_reject=self._row.is_reject,
            context_mgr=self._cm, debounce_window=self._row.debounce_steps,
            allow_repetition_while=(lambda: self._p5_budget_remaining > 0),
            on_repetition_allowed=self._consume_p5_budget)
        self._model_config_hook = model_config_hook
        self._apply_model_config()
        self._prev_checksum = None
        self._prev_objset_fp = None
        self._last_anchor = None
        self._pending_action = None
        self._pending_injections = []
        self._last_reasoner_result = None
        self._dispatched_log = deque(maxlen=DISPATCHED_LOG_CAP)
        self._prompt_log = deque(maxlen=PROMPT_LOG_CAP)
        self._total_dispatches = 0
        self._force_anchor = False
        self._resync = False
        self._level_num = 0
        self._levels_completed = 0
        self._available_actions = []
        self._last_packet = None

        # ---------- v1.0.19: Coder-solver state ---------- #
        self._coder_attempts_this_game = 0
        self._coder_attempts_this_level = 0
        self._coder_actions_this_game = 0
        self._coder_actions_this_level = 0
        self._coder_solve_pending = False
        self._coder_solve_result = None
        self._coder_guarded = None
        self._transition_buffer = None
        self._solver = None
        self._prev_settled_grid = None
        # A-CEGIS-13: REPLAN consumption state.
        self._coder_replan_pending = False
        self._coder_replan_count = 0
        self._init_coder_solver()

    # ---------- v1.0.19: Coder-solver helpers ---------- #

    def _init_coder_solver(self) -> None:
        if not self.config.get("coder_solver_enabled", True):
            logger.info("coder_solver disabled by config")
            return
        if _cs_module is None or _rv_module is None:
            logger.info("coder_solver chain not importable — reasoner-only")
            return
        try:
            self._transition_buffer = _rv_module.TransitionBuffer()
            self._solver = _cs_module.CoderSolver(call_llm=self._call_coder)
        except Exception as e:
            logger.warning("coder_solver init failed: %s", e)
            self._solver = None
            self._transition_buffer = None

    def _call_coder(self, spec: str, mode: str) -> str:
        if _il16_module is None:
            return ""
        cap = getattr(_il16_module, "CODER_CODE_CAP", 2048)
        sentinel = getattr(_il16_module, "CODER_CODE_CAP_EXCEEDED",
                           "coder_code_cap_exceeded")
        # LA-4 FIX (Indus, 2026-09-29 final logic audit): _DEFAULT_PARAMS
        # is a MODULE global of inference_layer, never an instance attr —
        # getattr(il, ...) always returned {} and _call_transport raised
        # KeyError('temperature'), silently returning "" for every coder
        # call. Source params from the additions re-export with a literal
        # fallback.
        _dp = getattr(self._il, "_DEFAULT_PARAMS", None) \
            or getattr(_il16_module, "_DEFAULT_PARAMS", None) \
            or {"temperature": 0.0, "top_p": 1.0, "top_k": -1}
        try:
            rj = self._il._call_transport(
                model=self._il._coder_model, system=None,
                messages=[{"role": "user", "content": spec}],
                max_tokens=cap, mode=None,
                params=dict(_dp),
                chat_template_kwargs={})
        except Exception as e:
            logger.info("coder transport failed: %s", e)
            return ""
        try:
            fr = rj["choices"][0].get("finish_reason")
            content = rj["choices"][0]["message"]["content"]
        except Exception:
            return ""
        enf = getattr(_il16_module, "enforce_coder_cap", None)
        if callable(enf):
            if enf(fr, content):
                return sentinel
        elif fr == "length":
            return sentinel
        return content

    def _should_attempt_coder_solve(self) -> bool:
        if self._solver is None or self._transition_buffer is None:
            return False
        if self._coder_attempts_this_game >= CODER_ATTEMPTS_PER_GAME_CAP:
            return False
        # A-CEGIS-13: while a REPLAN is pending, skip coder attempts so
        # the next action goes through the reasoner for a fresh hypothesis.
        if self._coder_replan_pending:
            return False
        if len(self._transition_buffer) < 1:
            return False
        if self._jolt_pending or self._divergence_pending:
            return True
        # v1.0.20: proactive coder trigger (Run-23 postmortem). Reactive-only
        # path mein coder tabhi fire hota tha jab loopdetector ya AP failure
        # ho — blind exploration mein wo kabhi nahi hota. Ab harness config
        # se: N actions at-level bina level-advance ke => coder attempt.
        # Default None => v1.0.19 semantics (zero behavior change).
        _proactive = self.config.get("coder_proactive_actions_at_level", None)
        if (_proactive is not None
                and isinstance(_proactive, int)
                and not isinstance(_proactive, bool)
                and _proactive > 0
                and self._actions_at_level >= _proactive):
            return True
        return False

    def _maybe_admit_coder(self, task_ctx: Dict[str, Any]) -> bool:
        # A-CEGIS-18: defensive guard — the coder chain may be partially
        # importable in some deployments. Fail-closed with an explicit
        # return instead of an opaque AttributeError on GuardedExecutor.
        if _cs_module is None or getattr(_cs_module, "GuardedExecutor", None) is None:
            return False
        if self._coder_solve_pending:
            return False
        if self._transition_buffer is None or len(self._transition_buffer) < 1:
            return False
        self._coder_solve_pending = True
        try:
            self._coder_attempts_this_game += 1
            self._coder_attempts_this_level += 1

            # A-CEGIS-16: world-model REPLAN short-circuits BEFORE the
            # action-program fallback, so the signal is never overwritten.
            res = self._solver.solve_world_model(
                task_ctx, self._transition_buffer)
            if getattr(res, "replan_requested", False):
                self._coder_solve_result = res
                self._handle_coder_replan(res)
                return False

            if not res.admitted:
                res = self._solver.solve_action_program(
                    task_ctx, self._transition_buffer)

            self._coder_solve_result = res
            if getattr(res, "replan_requested", False):
                self._handle_coder_replan(res)

            if not res.admitted:
                return False
            self._coder_guarded = _cs_module.GuardedExecutor(
                res.source, res.mode)
            return True
        except Exception as e:
            logger.info("coder admit failed: %s", e)
            self._coder_guarded = None
            return False
        finally:
            self._coder_solve_pending = False

    # ---------- A-CEGIS-13 / A-CEGIS-17: REPLAN semantics ---------- #
    def _handle_coder_replan(self, res) -> None:
        """Blueprint §C-3: `replay_mismatch_many → REPLAN to reasoner;
        do not repair`.

        Consequences:
        1. Clear Zone 2 `hypotheses` and `world_model_status` so the
           reasoner is not re-anchored to the stale hypothesis.
        2. Set `_coder_replan_pending = True`. The next
           `_reason_and_plan` call skips coder attempts and goes
           directly to the reasoner for a fresh hypothesis.
        3. Reset the guarded executor — the previous admitted model is
           void.
        4. Bump `_coder_replan_count` for telemetry.
        5. A-CEGIS-17: set `_jolt_pending = True` so `_should_deep`
           returns True and the reasoner runs in deep mode on the next
           action. This makes the "force a fresh hypothesis" contract
           explicit rather than relying on incidental trigger state.
        """
        try:
            if hasattr(self._cm, "update_zone2"):
                self._cm.update_zone2("hypotheses", "", truncate=True)
                self._cm.update_zone2("world_model_status",
                                      "replan: prior hypothesis rejected",
                                      truncate=True)
        except Exception as e:
            logger.info("replan cm update failed: %s", type(e).__name__)
        self._coder_guarded = None
        self._coder_replan_pending = True
        self._coder_replan_count += 1
        # A-CEGIS-17: explicit deep-call trigger for the next action.
        self._jolt_pending = True
        logger.info("coder REPLAN requested; fresh hypothesis forced "
                    "(count=%d)", self._coder_replan_count)

    def _try_guarded_action(self) -> Optional[str]:
        if self._coder_guarded is None:
            return None
        if self._prev_settled_grid is None:
            return None
        try:
            history = (list(self._transition_buffer.as_list())
                       if self._transition_buffer else [])
            action = self._coder_guarded.next_action(
                self._prev_settled_grid, history)
        except Exception as e:
            logger.info("guarded next_action failed: %s", e)
            self._coder_guarded = None
            return None
        if action is None:
            return None
        self._coder_actions_this_game += 1
        self._coder_actions_this_level += 1
        self._actions_at_level += 1
        self._jolt_pending = False
        self._divergence_pending = False
        self._pending_injections.clear()
        self._force_anchor = False
        self._resync = False
        return action if isinstance(action, str) else str(action)

    def _record_transition(self, state, action, next_state) -> None:
        if self._transition_buffer is None:
            return
        try:
            self._transition_buffer.append(state, action, next_state)
        except Exception as e:
            logger.info("buffer append failed: %s", e)

    def _on_level_advance_coder(self) -> None:
        self._coder_attempts_this_level = 0
        self._coder_actions_this_level = 0
        self._coder_replan_pending = False
        if self._solver is not None:
            self._solver.counters.new_level()
        if _rv_module is not None:
            try:
                self._transition_buffer = _rv_module.TransitionBuffer()
            except Exception:
                self._transition_buffer = None
        self._coder_guarded = None
        self._coder_solve_result = None

    def _deep_cap(self):
        raw = self.config.get("deep_call_cap", self._DEEP_TRIGGER_MAX_PER_GAME)
        if isinstance(raw, bool) or not isinstance(raw, int):
            raise AgentError(f"deep_call_cap must be a positive int; got {raw!r}")
        if raw <= 0:
            raise AgentError(f"deep_call_cap must be > 0; got {raw}")
        return raw

    def _should_deep(self):
        if self._actions_at_level == 0: return True
        if self._jolt_pending: return True
        if self._divergence_pending: return True
        if self._switch_decision_pending: return True
        return False

    def _consume_p5_budget(self):
        if self._p5_budget_remaining > 0:
            self._p5_budget_remaining -= 1

    @property
    def pause_before_retry_s(self): return self._row.pause_before_retry_s
    @property
    def pause_before_fresh_s(self): return self._row.pause_before_fresh_s
    @property
    def is_reject_game(self): return self._row.is_reject

    def model_config(self):
        return {"reasoner_endpoint": self.config["reasoner_endpoint"],
                "coder_endpoint": self.config["coder_endpoint"],
                "offline_fp8": bool(self.config["offline_fp8"])}

    def crlf_intake_log(self): return list(self._crlf_log)
    def last_packet(self): return self._last_packet
    def total_dispatches(self): return self._total_dispatches
    def coder_attempts(self): return self._coder_attempts_this_game
    def coder_result(self): return self._coder_solve_result
    def coder_actions(self): return self._coder_actions_this_game
    def coder_actions_level(self): return self._coder_actions_this_level
    def coder_replan_count(self): return self._coder_replan_count
    def coder_replan_pending(self): return self._coder_replan_pending

    def _apply_model_config(self):
        mc = self.model_config()
        if self._model_config_hook is None:
            logger.warning("model_config_hook not supplied (payload=%r)", mc)
            return
        try:
            self._model_config_hook(mc)
        except Exception as e:
            raise AgentError(f"model_config_hook raised: {e}") from e

    @staticmethod
    def _require(mod, name):
        if mod is None:
            raise ComponentUnavailable(f"certified module {name!r} not importable")
        return mod

    def _normalize_crlf(self, d):
        out = dict(d)
        for k, v in d.items():
            if isinstance(v, str) and ("\r\n" in v or "\r" in v):
                cr = v.count("\r")
                orig = hashlib.md5(v.encode("utf-8")).hexdigest()
                fixed = v.replace("\r\n", "\n").replace("\r", "\n")
                norm = hashlib.md5(fixed.encode("utf-8")).hexdigest()
                self._crlf_log.append(CRLFIntakeEntry(k, cr, orig, norm))
                out[k] = fixed
        return out

    @staticmethod
    def _objset_fp(objects):
        return (tuple(sorted(Counter(o.ohash for o in objects).items())),
                len(objects))

    @staticmethod
    def _object_at(objects, r, c):
        for o in objects:
            for (rr, cc) in o.cells:
                if rr == r and cc == c: return o
        return None

    def _parse_instruction(self, instruction):
        s = (instruction or "").strip()
        m = _INSTRUCTION_RE.match(s)
        if not m: return s, None
        aid, r, c = m.group("aid"), m.group("r"), m.group("c")
        return (aid, None) if r is None or c is None else (aid, (int(r), int(c)))

    def _on_dispatch(self, action):
        self._dispatched_log.append(action)
        self._total_dispatches += 1
        self._cm.append_action(action, "dispatched", 0)

    def act(self, frame):
        if not isinstance(frame, (list, tuple)) or not frame:
            raise AgentError("frame must be a non-empty sequence")
        settled = frame[-1]

        if self._pending_action is not None and self._prev_settled_grid is not None:
            self._record_transition(self._prev_settled_grid,
                                    self._pending_action, settled)

        if self._pending_action is not None:
            try:
                self._pipeline.observe([settled])
            except self._ap_mod.DivergenceError as e:
                self._last_packet = e.packet
                self._divergence_pending = True
                self._coder_diagnose("divergence")
            except self._ap_mod.PremiseFailureError:
                self._divergence_pending = True
                self._coder_diagnose("premise_failure")
            self._pending_action = None

        anchor = self._gt.rle_anchor(settled)
        self._last_anchor = anchor
        checksum = anchor["checksum"]
        objects = self._gt.extract_objects(settled, background=0)
        objset_fp = self._objset_fp(objects)
        state_changed = (self._prev_checksum is not None
                         and checksum != self._prev_checksum)
        # v1.0.21 Object Vision: structured objects CM tak — prompt mein
        # "[id] color=.. rows=r0-r1 cols=c0-c1" render hote hain, model ko
        # RLE column-arithmetic nahi karna padta.
        obs_objects = [
            {"id": o.id, "color_char": o.color_char,
             "color_digit": int(o.color_digit), "size": int(o.size),
             "bbox": [int(v) for v in o.bbox]}
            for o in objects]
        self._cm.set_observation(
            encoding_dict=anchor,
            available_actions=list(self._available_actions),
            levels_completed=self._levels_completed, state="RUNNING",
            objects=obs_objects)
        if self._prev_objset_fp is not None and objset_fp != self._prev_objset_fp:
            self._cm.clear_dead_signatures()

        action_id, target_xy = (None, None)
        target_hash, target_bbox = (None, None)
        if self._dispatched_log:
            aid, txy = self._parse_instruction(str(self._dispatched_log[-1]))
            action_id, target_xy = aid, txy
            if txy is not None:
                o = self._object_at(objects, txy[0], txy[1])
                if o is not None:
                    target_hash, target_bbox = o.ohash, o.bbox
        if action_id is not None:
            jolt = self._loop.tick(
                action_id=action_id, target_xy=target_xy,
                target_hash=target_hash, target_bbox=target_bbox,
                state_changed=state_changed, objset_fp=objset_fp)
            if jolt is not None:
                self._handle_jolt(jolt)

        self._prev_checksum = checksum
        self._prev_objset_fp = objset_fp
        self._prev_settled_grid = settled

        next_action = self._reason_and_plan()
        self._pipeline.submit_plan([next_action])
        result = self._pipeline.step()
        if result is None: return None
        self._pending_action = result.action
        return result.action

    def reset_level(self, level_num=None, advanced=False):
        if level_num is not None: n = int(level_num)
        elif advanced: n = self._level_num + 1
        else: n = self._level_num
        self._cm.set_level(n)
        self._cm.clear_dead_signatures()
        self._loop.reset()
        self._level_num = n
        if advanced: self._levels_completed += 1
        if advanced: self._coder_world_model()
        self._pending_injections.clear()
        self._prev_objset_fp = None
        self._actions_at_level = 0
        self._p5_budget_remaining = 2
        self._jolt_pending = False
        self._divergence_pending = False
        self._pending_action = None
        self._prev_settled_grid = None
        self._on_level_advance_coder()

    def set_available_actions(self, actions):
        self._available_actions = list(actions)

    def _handle_jolt(self, jolt):
        self._jolt_pending = True
        if jolt.kind == "stuck":
            self._force_anchor = True
            self._pending_injections.append(
                "=== CHANGE OF PLAN ===\n"
                "The following action keeps failing with no state change.  "
                "Do not repeat it in this plan.\n"
                + LoopDetector._format_blacklist(jolt.blacklist_sig)
                + "\nConsider a completely different strategy.")
        elif jolt.kind == "deadlock":
            self._force_anchor = True
            self._resync = True
            self._pending_injections.append(
                "=== RESYNC ===\nNo object-level change over two debounce "
                "windows.  Fresh anchor attached below.  Re-derive the plan "
                "from scratch.")

    def _reason_and_plan(self):
        action = self._try_guarded_action()
        if action is not None:
            return action

        if self._should_attempt_coder_solve():
            task_ctx = {
                "task": ("Reproduce every buffered transition exactly. "
                         "Return the required functions."),
                "hypothesis": {
                    "game": self.game_id,
                    "level": self._level_num,
                    "actions_at_level": self._actions_at_level,
                },
                "replay_evidence": {
                    "transitions": len(self._transition_buffer or []),
                },
                "io_contract": {
                    "frame_format": "64x64 int8",
                    "must_replay": True,
                },
                "constraints": {"no_imports": True, "deterministic": True,
                                "max_lines": 80},
                "alternatives": [],
                "mode": "world",
            }
            if self._maybe_admit_coder(task_ctx):
                action = self._try_guarded_action()
                if action is not None:
                    return action
                self._coder_guarded = None

        pp = self._cm.build_prompt()
        text = getattr(pp, "text", str(pp))
        if self._force_anchor:
            text += ("\n\n[FRESH ANCHOR]\n"
                     + json.dumps(self._last_anchor or {}, sort_keys=True))
            self._force_anchor = False
        if self._resync:
            text = "[FULL RESYNC]\n" + text
            self._resync = False
        if self._pending_injections:
            text += "\n\n" + "\n\n".join(self._pending_injections)
            self._pending_injections.clear()
        self._prompt_log.append(text)

        cap = self._deep_cap()
        if self._deep_calls_this_game >= cap:
            mode = "fast"
        elif self._should_deep():
            mode = "deep"
        else:
            mode = self.config.get("reasoner_mode", "fast")
        self._jolt_pending = False
        self._divergence_pending = False

        result = self._il.ask_reasoner({"text": text}, mode=mode)
        if mode == "deep":
            self._deep_calls_this_game += 1
        self._last_reasoner_result = result
        self._cm.update_zone2("world_model_status",
                              str(result.observation_summary), truncate=True)
        if result.hypothesis_updates:
            self._cm.update_zone2("hypotheses",
                                  "\n".join(result.hypothesis_updates),
                                  truncate=True)
        self._cm.update_zone2("goal_notes", str(result.prediction),
                              truncate=True)
        self._actions_at_level += 1

        if self._coder_replan_pending:
            self._coder_replan_pending = False

        aid, txy = self._parse_instruction(result.instruction)
        return f"{aid} r{txy[0]} c{txy[1]}" if txy is not None else aid

    def _coder_world_model(self):
        try:
            r0 = self._last_reasoner_result
            hyp = "; ".join(getattr(r0, "hypothesis_updates", []) or [])
            brief = (
                "Write a short pure-python function world_model() returning a "
                "one-line summary of the current game model. Evidence: "
                f"game={self.game_id}; levels_completed={self._levels_completed}; "
                f"observation_summary={getattr(r0, 'observation_summary', '')}; "
                f"hypotheses={hyp}; prediction={getattr(r0, 'prediction', '')}. "
                "Only python code in one fenced block, no prose.")
            r = self._il.write_world_model(brief)
            digest = " ".join((getattr(r, "code", "") or "").split())[:120]
            self._cm.update_zone2("world_model_status", "wm-code: " + digest,
                                  truncate=True)
        except Exception as e:
            logger.info("coder world-model skipped: %s", type(e).__name__)

    def _coder_diagnose(self, kind, detail=""):
        try:
            r0 = self._last_reasoner_result
            trace = (f"{kind}: {detail}; last_prediction="
                     f"{getattr(r0, 'prediction', '')!r}")
            ce = None
            if self._last_packet is not None and hasattr(self._last_packet, "to_dict"):
                try:
                    ce = self._last_packet.to_dict()
                except Exception:
                    ce = None
            try:
                if ce is not None:
                    r = self._il.diagnose_failure(
                        function_name="plan", failure_trace=trace,
                        current_code="def plan(): pass",
                        counterexample=ce)
                else:
                    r = self._il.diagnose_failure(
                        function_name="plan", failure_trace=trace,
                        current_code="def plan(): pass")
            except TypeError:
                r = self._il.diagnose_failure(
                    function_name="plan", failure_trace=trace,
                    current_code="def plan(): pass")
            note = "{} (conf={})".format(
                getattr(r, "root_cause", ""), getattr(r, "confidence", "?"))
            self._cm.update_zone2("world_model_status", "diag: " + note,
                                  truncate=True)
        except Exception as e:
            logger.info("coder diagnose skipped: %s", type(e).__name__)

    def framework_choose(self, latest_frame, prev_state):
        state = _state_name(latest_frame)
        if state != prev_state:
            if state == "WIN": self.reset_level(advanced=True)
            elif state == "GAME_OVER": self.reset_level(advanced=False)
        if state in (None, "", "NOT_PLAYED", "GAME_OVER"):
            return {"name": "RESET", "x": None, "y": None,
                    "reasoning": f"reset:{state or 'init'}"}
        self.set_available_actions(_frame_available_actions(latest_frame))
        grid = _frame_grid(latest_frame)
        if grid is None:
            return {"name": "RESET", "x": None, "y": None,
                    "reasoning": "no-grid"}
        instr = self.act([grid])
        return _instruction_to_action_dict(instr)

    def prompt_log(self): return list(self._prompt_log)
    def dispatched_log(self): return list(self._dispatched_log)
    def jolt_history(self): return self._loop.jolt_history()


def _state_name(frame):
    s = getattr(frame, "state", None) if frame is not None else None
    if s is None: return None
    return str(s.name) if hasattr(s, "name") else str(s)


def _frame_grid(frame):
    f = getattr(frame, "frame", None) if frame is not None else None
    if f is None: return None
    if hasattr(f, "tolist"): f = f.tolist()
    if not isinstance(f, list) or not f: return None
    first = f[0]
    if isinstance(first, list) and first and isinstance(first[0], list):
        return [[int(cell[0]) for cell in row] for row in f]
    return [[int(c) for c in row] for row in f]


def _frame_available_actions(frame):
    aa = getattr(frame, "available_actions", None) if frame is not None else None
    return list(aa) if aa else []


def _instruction_to_action_dict(instruction):
    if not instruction:
        return {"name": "RESET", "x": None, "y": None,
                "reasoning": "no-instruction"}
    parts = str(instruction).split()
    aid = parts[0]
    x = None; y = None
    for p in parts[1:]:
        if p.startswith("r"):
            try: y = int(p[1:])
            except ValueError: pass
        elif p.startswith("c"):
            try: x = int(p[1:])
            except ValueError: pass
    return {"name": aid, "x": x, "y": y, "reasoning": str(instruction)}


class MyAgent(_FrameworkAgent):
    MAX_ACTIONS = 80

    def __init__(self, game_id, _orchestrator=None, **kwargs):
        super().__init__(**kwargs)
        self.game_id = game_id
        self._agent = _orchestrator or Agent(
            game_id=game_id, version_manifest=CERTIFIED_VERSIONS)
        self._prev_state = None
        _jb = _journal_brief()
        if _jb:
            try:
                _cm = getattr(self._agent, "_cm", None)
                if _cm is not None:
                    _cm.update_zone2("confirmed_rules",
                                     "cross-game: " + _jb, truncate=True)
            except Exception:
                pass

    def is_done(self, frames, latest_frame):
        return _state_name(latest_frame) == "WIN"

    def choose_action(self, frames, latest_frame):
        spec = self._agent.framework_choose(latest_frame, self._prev_state)
        self._prev_state = _state_name(latest_frame)
        return _make_framework_action(
            spec["name"], spec["x"], spec["y"], spec["reasoning"])

    @classmethod
    def _default_manifest(cls):
        return dict(CERTIFIED_VERSIONS)

    def cleanup(self, scorecard=None):
        try:
            lv = 0
            _fr = getattr(self, "frames", None)
            if _fr:
                lv = int(getattr(_fr[-1], "levels_completed", 0) or 0)
            _rules = ""
            _cm = getattr(self._agent, "_cm", None)
            if _cm is not None and hasattr(_cm, "snapshot"):
                try:
                    _snap = _cm.snapshot() or {}
                    _z2 = _snap.get("zone2", {}) if isinstance(_snap, dict) else {}
                    _rules = str(_z2.get("confirmed_rules", ""))[:200]
                except Exception:
                    pass
            _TACTICS_JOURNAL.append(
                {"game": self.game_id, "levels": lv, "rules": _rules})
        except Exception:
            pass
        _cleanup = getattr(super(), "cleanup", None)
        if callable(_cleanup):
            try: _cleanup(scorecard)
            except Exception: pass


_FRAMEWORK_ACTION_FACTORY = None


class _FallbackAction:
    def __init__(self, name, x=None, y=None, reasoning=""):
        self.name = name; self.x = x; self.y = y; self.reasoning = reasoning
        self.data = {"x": x, "y": y} if name == "ACTION6" else {}
    def __repr__(self):
        return f"_FallbackAction({self.name!r}, x={self.x}, y={self.y})"


def _make_framework_action(name, x=None, y=None, reasoning=""):
    if _FRAMEWORK_ACTION_FACTORY is not None:
        return _FRAMEWORK_ACTION_FACTORY(name, x, y, reasoning)
    if _HAS_ARCENGINE:
        try:
            act = getattr(GameAction, name)
        except AttributeError:
            act = GameAction.RESET
        if name == "ACTION6":
            try: act.set_data({"x": int(x) if x is not None else 0,
                               "y": int(y) if y is not None else 0})
            except Exception: pass
        try: act.reasoning = reasoning
        except Exception: pass
        return act
    return _FallbackAction(name, x=x, y=y, reasoning=reasoning)


MODULE_ORDER = ("grid_text", "context_manager", "inference_layer",
                "action_pipeline", "my_agent")
ARTIFACT_NAME = "my_agent.py"
DEFAULT_OUT_DIR = "submission"
_ARTIFACT_FIRST_LINE_PREFIX = '"""AUTO-GENERATED artifact.'

_ARTIFACT_HEADER = '''\
"""AUTO-GENERATED artifact.  Do not edit by hand.

Source of truth: the modular agent/ package.
Rebuild with: python agent/my_agent.py build

Public surface (what Kaggle imports):
    MyAgent, Agent, CERTIFIED_VERSIONS, check_certified_versions, ...
"""
'''


def read_modular_sources(root):
    sources = {}
    for name in MODULE_ORDER:
        path = root / "agent" / f"{name}.py"
        if not path.is_file():
            raise FileNotFoundError(f"missing modular source: {path}")
        text = path.read_text(encoding="utf-8")
        if name == "my_agent":
            first_line = text.split("\n", 1)[0].lstrip()
            if first_line.startswith(_ARTIFACT_FIRST_LINE_PREFIX):
                raise AgentError(
                    f"refusing to compile {path}: first line identifies "
                    f"a compiled artifact.")
        sources[name] = text
    return sources


def compile_sources(sources):
    parts = [_ARTIFACT_HEADER]
    parts.append("# ---- loader " + "-" * 52)
    parts.append("import types as _types, sys as _sys")
    parts.append("")
    parts.append("_agent_pkg = _sys.modules.get('agent')")
    parts.append("if _agent_pkg is None:")
    parts.append("    _agent_pkg = _types.ModuleType('agent')")
    parts.append("    _agent_pkg.__path__ = []")
    parts.append("    _sys.modules['agent'] = _agent_pkg")
    parts.append("")
    for name in ("grid_text", "context_manager",
                 "inference_layer", "action_pipeline"):
        if name not in sources: continue
        parts.append(f"# ---- {name} " + "-" * max(0, 52 - len(name)))
        parts.append(f"_SRC_{name} = {json.dumps(sources[name])}")
        parts.append(f"_mod_{name} = _types.ModuleType({name!r})")
        parts.append(f"_mod_{name}.__file__ = {name + '.py'!r}")
        parts.append(f"_sys.modules[{name!r}] = _mod_{name}")
        parts.append(f"_sys.modules['agent.{name}'] = _mod_{name}")
        parts.append(f"setattr(_agent_pkg, {name!r}, _mod_{name})")
        parts.append(f"exec(_SRC_{name}, _mod_{name}.__dict__)")
        parts.append("")
    parts.append("# ---- my_agent " + "-" * 45)
    parts.append(f"_SRC_my_agent = {json.dumps(sources['my_agent'])}")
    parts.append("_saved_name = __name__")
    parts.append("_self = _sys.modules[__name__]")
    parts.append("_sys.modules['my_agent'] = _self")
    parts.append("_sys.modules['agent.my_agent'] = _self")
    parts.append("try:")
    parts.append('    __name__ = "my_agent"')
    parts.append("    exec(_SRC_my_agent, globals())")
    parts.append("finally:")
    parts.append("    __name__ = _saved_name")
    parts.append("")
    parts.append("setattr(_agent_pkg, 'my_agent', _self)")
    parts.append("")
    return "\n".join(parts)


def md5_of(text): return hashlib.md5(text.encode("utf-8")).hexdigest()


def write_artifact(artifact, out_path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    tmp = out_path.with_suffix(out_path.suffix + ".tmp")
    tmp.write_text(artifact, encoding="utf-8")
    tmp.replace(out_path)
    return md5_of(artifact)


def verify_artifact(out_path):
    import subprocess
    check = textwrap.dedent(f"""\
        import importlib.util, sys
        spec = importlib.util.spec_from_file_location(
            "_artifact_check", {str(out_path)!r})
        mod = importlib.util.module_from_spec(spec)
        sys.modules["_artifact_check"] = mod
        spec.loader.exec_module(mod)
        assert mod.__name__ == "_artifact_check"
        required = ("MyAgent", "Agent", "CERTIFIED_VERSIONS",
                    "check_certified_versions", "lookup_calibration",
                    "LoopDetector", "DEFAULT_CONFIG")
        for attr in required:
            if not hasattr(mod, attr):
                raise SystemExit(f"verify FAIL: missing {{attr!r}}")
        assert mod.CERTIFIED_VERSIONS == {{
            "context_manager": "1.6",
            "inference_layer": "1.6",
            "grid_text": "1.0.8",
            "action_pipeline": "1.0.5",
        }}, mod.CERTIFIED_VERSIONS
        print("verify OK")
    """)
    r = subprocess.run([sys.executable, "-c", check],
                       capture_output=True, text=True)
    if r.stdout: sys.stdout.write(r.stdout)
    if r.stderr: sys.stderr.write(r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f"artifact verification failed (exit {r.returncode})")


def _resolve_root(explicit):
    if explicit is not None: return explicit.resolve()
    here = Path(__file__).resolve()
    for cand in (here.parent, here.parent.parent, here.parent.parent.parent):
        if (cand / "agent" / "my_agent.py").is_file(): return cand
    return here.parent.parent


def _resolve_out(args, root):
    if args.in_place: return root / "agent" / ARTIFACT_NAME
    if args.out is not None:
        return args.out if args.out.is_absolute() else (root / args.out)
    return root / DEFAULT_OUT_DIR / ARTIFACT_NAME


def build_main(argv):
    p = argparse.ArgumentParser(prog="my_agent.py build")
    p.add_argument("--root", type=Path, default=None)
    p.add_argument("--out", type=Path, default=None)
    p.add_argument("--in-place", action="store_true")
    p.add_argument("--force", action="store_true")
    p.add_argument("--dry-run", action="store_true")
    p.add_argument("--no-verify", action="store_true")
    args = p.parse_args(argv)
    if args.in_place and not args.force:
        print("ERROR: --in-place requires --force", file=sys.stderr)
        return 2
    root = _resolve_root(args.root)
    out = _resolve_out(args, root)
    srcs = read_modular_sources(root)
    artifact = compile_sources(srcs)
    md5 = md5_of(artifact)
    if args.dry_run:
        print(f"dry-run md5: {md5}")
        return 0
    written_md5 = write_artifact(artifact, out)
    if written_md5 != md5:
        raise RuntimeError("write corrupted the artifact")
    print(f"wrote {out}")
    print(f"md5:   {md5}")
    if not args.no_verify:
        verify_artifact(out)
    return 0


class _FakeCM:
    __version__ = "1.6"
    def __init__(self):
        self._zone1 = ""
        self._zone2 = {"confirmed_rules": "", "hypotheses": "",
                       "world_model_status": "", "goal_notes": ""}
        self._zone3 = []; self._obs = None
        self.obs_objects = []
        self.dead_sigs = []; self._level = 0
        self.set_level_called_with = []
    def set_system_prompt(self, t): self._zone1 = t
    def set_observation(self, encoding_dict, available_actions=(),
                        levels_completed=0, state="unknown", objects=None):
        self._obs = (encoding_dict, list(available_actions),
                     levels_completed, state)
        self.obs_objects = list(objects or [])
    def update_zone2(self, f, t, truncate=False):
        if f in self._zone2: self._zone2[f] = t
    def append_action(self, a, r, d): self._zone3.append((a, r, d))
    def register_dead_signature(self, s):
        if s not in self.dead_sigs: self.dead_sigs.append(s)
    def clear_dead_signatures(self): self.dead_sigs.clear()
    def set_level(self, n):
        self._level = n; self.set_level_called_with.append(n)
    def build_prompt(self):
        return _PromptPacket(text=(
            f"[Z1]{self._zone1}\n[Z2]"
            f"{json.dumps(self._zone2, sort_keys=True)}\n[OBS]{self._obs}\n"
            f"do_not_retry={list(self.dead_sigs)}"))


@dataclass
class _PromptPacket: text: str


class _FakeIL:
    __version__ = "1.6"
    def __init__(self, instructions=None):
        self._instructions = list(instructions or []); self._idx = 0
        self.mode_calls = []
        self._coder_model = "qwen3-coder-30b-a3b"
        self._DEFAULT_PARAMS = {"temperature": 0.0}
    def ask_reasoner(self, prompt_packet, mode="fast"):
        self.mode_calls.append(mode)
        instr = ("right" if self._idx >= len(self._instructions)
                 else self._instructions[self._idx])
        self._idx += 1
        return _ReasonerResult("obs", ["h1"], instr, "p")
    def write_world_model(self, brief):
        return _WorldModelResult(code="def world_model(): return 'x'")
    def diagnose_failure(self, function_name, failure_trace, current_code=""):
        return _DiagnoseResult(root_cause="cause", confidence=0.5,
                               suggested_fix="fix")
    def _call_transport(self, **kw):
        return {"choices": [{"message": {"content": "```python\npass\n```"},
                             "finish_reason": "stop"}],
                "usage": {"completion_tokens": 1}}


@dataclass
class _WorldModelResult: code: str; notes: str = ""
@dataclass
class _DiagnoseResult:
    root_cause: str; confidence: float = 0.0; suggested_fix: str = ""


@dataclass
class _ReasonerResult:
    observation_summary: str; hypothesis_updates: list
    instruction: str; prediction: str


@dataclass
class _FakeObj:
    ohash: str; bbox: Tuple[int, int, int, int]
    cells: List[Tuple[int, int]]
    # v1.0.21 Object Vision: real grid_text.ObjectRec ke saath parity
    id: int = 0; color_digit: int = 0; color_char: str = "?"
    size: int = 0


class _FakeGT:
    __version__ = "1.0.8"
    def rle_anchor(self, grid):
        h = hashlib.md5()
        for row in grid: h.update(bytes(row))
        return {"type": "full", "dim": [len(grid), len(grid[0])],
                "checksum": h.hexdigest(), "rows": ["ok"]}
    def extract_objects(self, grid, background=0):
        by = {}
        for r, row in enumerate(grid):
            for c, v in enumerate(row):
                if v != background:
                    by.setdefault((v, "1x1"), []).append((r, c, v))
        objs = []
        for (v, shp), cells in by.items():
            rs = [r for r, _, _ in cells]; cs = [c for _, c, _ in cells]
            ohash = hashlib.md5(f"{v}|{shp}".encode()).hexdigest()
            objs.append(_FakeObj(ohash=ohash,
                bbox=(min(rs), max(rs), min(cs), max(cs)),
                cells=[(r, c) for r, c, _ in cells],
                id=len(objs) + 1, color_digit=v,
                color_char="BMPRgw"[v % 6], size=len(cells)))
        return objs


class _FakeAPModule:
    __version__ = "1.0.5"
    CLEAN_SET = {"games_count": 21, "noop_pct": 3.78, "exact_repeat_pct": 1.00,
                 "jitter_pct": 2.35, "retry_success_pct": 59.1,
                 "switch_after_double_fail_pct": 3.9,
                 "pause_retry_s": 1.10, "pause_fresh_s": 0.57}
    REJECT_LIST_IDS = ("lp85-305b61c3", "s5i5-18d95033",
                       "vc33-5430563c", "dc22-fdcac232")
    class CalibrationRow:
        def __init__(self, game_id, is_reject):
            self.game_id = game_id; self.is_reject = is_reject
            self.retry_budget = 3 if is_reject else 2
            self.debounce_steps = 5
    DEFAULT_CALIBRATION = None
    class DivergenceError(RuntimeError):
        def __init__(self, msg, packet):
            super().__init__(msg); self.packet = packet
    class PremiseFailureError(RuntimeError): pass
    class ActionResult:
        def __init__(self, action): self.action = action
    class ActionPipeline:
        def __init__(self, game_id, dispatch=None, patch_hook=None):
            self.game_id = game_id; self._dispatch = dispatch
            self._queue = []; self.step_calls = 0; self.observe_calls = 0
        def submit_plan(self, actions): self._queue = list(actions)
        def step(self):
            self.step_calls += 1
            if not self._queue: return None
            a = self._queue.pop(0)
            if self._dispatch is not None: self._dispatch(a)
            return _FakeAPModule.ActionResult(a)
        def observe(self, frame):
            self.observe_calls += 1; return None


_FakeAPModule.DEFAULT_CALIBRATION = {
    gid: _FakeAPModule.CalibrationRow(
        gid, gid in _FakeAPModule.REJECT_LIST_IDS)
    for gid in (
        "ar25-0c556536", "bp35-0a0ad940", "cd82-fb555c5d",
        "cn04-2fe56bfb", "dc22-fdcac232", "ft09-0d8bbf25",
        "g50t-5849a774", "ka59-38d34dbb", "lf52-271a04aa",
        "lp85-305b61c3", "ls20-9607627b", "m0r0-492f87ba",
        "r11l-495a7899", "re86-8af5384d", "s5i5-18d95033",
        "sb26-7fbdac44", "sc25-635fd71a", "sk48-d8078629",
        "sp80-589a99af", "su15-1944f8ab", "tn36-ef4dde99",
        "tr87-cd924810", "tu93-0768757b", "vc33-5430563c",
        "wa30-ee6fef47")}


def _frame(v=0, h=8, w=8, block=None):
    g = [[v] * w for _ in range(h)]
    if block is not None:
        r0, c0, rh, rw, vv = block
        for r in range(r0, r0 + rh):
            for c in range(c0, c0 + rw): g[r][c] = vv
    return [g]


def _agent(**kw):
    cfg = dict(kw.pop("config", None) or {})
    cfg.setdefault("coder_solver_enabled", False)
    return Agent(game_id=kw.pop("game_id", "ls20-9607627b"),
                 config=cfg,
                 cm=_FakeCM(), il=_FakeIL(instructions=kw.get("instructions")),
                 gt=_FakeGT(), ap=_FakeAPModule,
                 model_config_hook=kw.get("model_config_hook"),
                 version_manifest=kw.get("version_manifest"))


def _stuck_jolt():
    sig = StuckSig(action_id="right", target_xy=None,
                   target_hash=None, target_bbox=None)
    return JoltRequest(kind="stuck", blacklist_sig=sig)


def _t1():
    a = _agent()
    assert a.version_verified["context_manager"]
    class NoVer(_FakeCM): __version__ = None
    try: Agent(game_id="g", config={"coder_solver_enabled": False},
               cm=NoVer(), il=_FakeIL(), gt=_FakeGT(), ap=_FakeAPModule)
    except VersionMissing: pass
    else: raise AssertionError
    class Bad(_FakeCM): __version__ = "9.9"
    try: Agent(game_id="g", config={"coder_solver_enabled": False},
               cm=Bad(), il=_FakeIL(), gt=_FakeGT(), ap=_FakeAPModule)
    except VersionMismatch: pass
    else: raise AssertionError


def _t2():
    a = _agent(instructions=["right"]*4); f = _frame()
    a.act(f)
    for _ in range(3): a.act(f)
    assert a._row.retry_budget == 2 and a._row.is_reject is False


def _t3():
    class P: pass
    p = P(); err = _FakeAPModule.DivergenceError("d", p)
    assert err.packet is p


def _t4():
    ld = LoopDetector("ls20-9607627b", False, _FakeCM(), 5)
    fp = ((), 0); fired = None
    for i in range(1, 5):
        r = ld.tick(action_id="right", target_xy=None, target_hash=None,
                    target_bbox=None, state_changed=False, objset_fp=fp)
        if r is not None: fired = i
    assert fired == 4


def _t5():
    ld = LoopDetector("lp85-305b61c3", True, _FakeCM(), 5)
    fp = ((), 0); fired = None
    for i in range(1, 6):
        r = ld.tick(action_id="right", target_xy=None, target_hash=None,
                    target_bbox=None, state_changed=False, objset_fp=fp)
        if r is not None: fired = i
    assert fired == 5


def _t6():
    ld = LoopDetector("ls20-9607627b", False, _FakeCM(), 5)
    fp = ((), 0); th = "h"
    ld.tick(action_id="click", target_xy=(10, 10), target_hash=th,
            target_bbox=(10,10,10,10), state_changed=False, objset_fp=fp)
    ld.tick(action_id="click", target_xy=(12, 12), target_hash=th,
            target_bbox=(12,12,12,12), state_changed=False, objset_fp=fp)
    assert ld.streak_length() == 2
    ld.tick(action_id="click", target_xy=(15, 15), target_hash=th,
            target_bbox=(15,15,15,15), state_changed=False, objset_fp=fp)
    assert ld.streak_length() == 1


def _t7():
    a = _agent(instructions=["right"]*6); a.set_available_actions(["right"])
    f = _frame(); a.act(f)
    for _ in range(6): a.act(f)
    assert any(j.kind == "stuck" for j in a.jolt_history())


def _t8():
    cm = _FakeCM(); a = _agent(); a._cm = cm; a._loop._cm = cm
    cm.register_dead_signature("X"); assert len(cm.dead_sigs) == 1
    a.act(_frame(block=(1,1,1,1,5))); a.act(_frame(block=(2,2,1,1,5)))
    assert len(cm.dead_sigs) == 1
    a.act(_frame(h=8, w=8, block=(1,1,1,1,5)))
    f2 = _frame(h=8, w=8, block=(1,1,1,1,5)); f2[0][3][3] = 8
    a.act(f2); assert len(cm.dead_sigs) == 0


def _t9():
    a = _agent(instructions=["right"]*8); a.set_available_actions(["right"])
    f = _frame(); a.act(f)
    for _ in range(6): a.act(f)
    assert any(j.kind == "stuck" for j in a.jolt_history())
    assert a._pipeline.step_calls == a.total_dispatches()


def _t10():
    assert "enable_physical_jolt" not in DEFAULT_CONFIG
    assert "enable_physical_jolt" not in _agent().config


def _t11():
    ld = LoopDetector("ls20-9607627b", False, _FakeCM(), 2); fp = (("x",1),)
    fired = None
    for i in range(1, 6):
        r = ld.tick(action_id="right", target_xy=None, target_hash=None,
                    target_bbox=None, state_changed=True, objset_fp=fp)
        if r is not None: fired = i; break
    assert fired == 4


def _t12():
    ld = LoopDetector("ls20-9607627b", False, _FakeCM(), 2); fp = (("x",1),)
    fire = None
    for aid in ["a","b","c","d","e"]:
        r = ld.tick(action_id=aid, target_xy=None, target_hash=None,
                    target_bbox=None, state_changed=False, objset_fp=fp)
        if r is not None: fire = r; break
    assert fire is not None and fire.kind == "deadlock"


def _t13():
    a = _agent()
    assert a.pause_before_retry_s == 1.10 and a.pause_before_fresh_s == 0.57


def _t14():
    a = _agent()
    assert a._row.retry_budget == 2
    a2 = Agent(game_id="lp85-305b61c3",
               config={"coder_solver_enabled": False},
               cm=_FakeCM(), il=_FakeIL(), gt=_FakeGT(), ap=_FakeAPModule)
    assert a2._row.retry_budget == 3 and a2._row.is_reject is True


def _t15():
    spy = []
    _agent(model_config_hook=lambda p: spy.append(dict(p)))
    assert len(spy) == 1


def _t16():
    ld = LoopDetector("ls20-9607627b", False, _FakeCM(), 5)
    for b in ("dispatch", "step", "observe", "act"):
        assert not hasattr(ld, b)


def _t17():
    def run():
        a = _agent(instructions=["right"]*8); a.set_available_actions(["right"])
        return [a.act(_frame()) for _ in range(4)], a.prompt_log()
    assert run() == run()


def _t18():
    a = Agent(game_id="ls20-9607627b",
              config={"reasoner_endpoint": "q\r\nwen",
                      "coder_solver_enabled": False},
              cm=_FakeCM(), il=_FakeIL(), gt=_FakeGT(), ap=_FakeAPModule)
    assert "\r" not in a.config["reasoner_endpoint"]


def _t19():
    a = _agent(instructions=["right"]*4)
    a._prev_objset_fp = (("x",1),)
    a._cm.register_dead_signature("SEED")
    a.reset_level(level_num=2, advanced=True)
    assert a._levels_completed == 1


def _t20():
    a = _agent(instructions=["right"]*4); a.set_available_actions(["right"])
    f = _frame(); a.act(f)
    a._handle_jolt(JoltRequest(kind="deadlock"))
    assert a._force_anchor and a._resync
    a.act(f)
    assert "[FULL RESYNC]" in "\n".join(a.prompt_log())


def _t21():
    a = _agent(instructions=["click r5 c7"]*4)
    f = _frame(block=(5,7,1,1,8)); a.act(f); a.act(f)
    aid, txy = a._parse_instruction(str(a._dispatched_log[0]))
    assert aid == "click" and txy == (5, 7)


def _t22():
    class NoCM(_FakeCM): __version__ = None
    try: Agent(game_id="x", config={"coder_solver_enabled": False},
               cm=NoCM(), il=_FakeIL(), gt=_FakeGT(), ap=_FakeAPModule)
    except VersionMissing: pass
    else: raise AssertionError


def _t23():
    a = Agent(game_id="ls20-9607627b",
              config={"reasoner_endpoint": "q\r\nwen\rmore",
                      "coder_solver_enabled": False},
              cm=_FakeCM(), il=_FakeIL(), gt=_FakeGT(), ap=_FakeAPModule)
    assert a.config["reasoner_endpoint"] == "q\nwen\nmore"


def _t24():
    a = _agent(instructions=["right"]); f = _frame()
    for _ in range(DISPATCHED_LOG_CAP + 50): a.act(f)
    assert len(a._dispatched_log) <= DISPATCHED_LOG_CAP


def _t25():
    c = {"reasoner_endpoint": "a\r\nb"}; _agent(config=c)
    assert c["reasoner_endpoint"] == "a\r\nb"


def _t26():
    ld = LoopDetector("ls20-9607627b", False, _FakeCM(), 5); fp = (("x",1),)
    for _ in range(DEADLOCK_WINDOWS * 5 - 1):
        ld.tick(action_id="right", target_xy=None, target_hash=None,
                target_bbox=None, state_changed=True, objset_fp=fp)
    fired = None
    for _ in range(5):
        r = ld.tick(action_id="right", target_xy=None, target_hash=None,
                    target_bbox=None, state_changed=False, objset_fp=fp)
        if r is not None and r.kind == "stuck": fired = r; break
    assert fired is not None


def _t27():
    ld = LoopDetector("ls20-9607627b", False, _FakeCM(), 2); fp = (("x",1),)
    for _ in range(DEADLOCK_WINDOWS * 2 - 1):
        ld.tick(action_id="right", target_xy=None, target_hash=None,
                target_bbox=None, state_changed=True, objset_fp=fp)
    r = ld.tick(action_id="right", target_xy=None, target_hash=None,
                target_bbox=None, state_changed=False, objset_fp=fp)
    assert r is not None and r.kind == "deadlock"


def _t28():
    a = Agent(game_id="ls20", config={"coder_solver_enabled": False},
              cm=_FakeCM(), il=_FakeIL(), gt=_FakeGT(), ap=_FakeAPModule)
    assert a._row.resolved_id == "ls20-9607627b"


def _t29():
    _WARNED_UNKNOWN_IDS.discard("zzz-nope")
    a = Agent(game_id="zzz-nope", config={"coder_solver_enabled": False},
              cm=_FakeCM(), il=_FakeIL(), gt=_FakeGT(), ap=_FakeAPModule)
    assert a._row.known is False


def _t30():
    a = Agent(game_id="dc22", config={"coder_solver_enabled": False},
              cm=_FakeCM(), il=_FakeIL(), gt=_FakeGT(), ap=_FakeAPModule)
    assert a._row.is_reject is True


class _FakeFrame:
    def __init__(self, state="NOT_PLAYED", grid=None, avail=None):
        self.state = state
        self.frame = grid if grid is not None else [[0]*8 for _ in range(8)]
        self.available_actions = list(avail or [])


class _GameState:
    def __init__(self, name): self.name = name


def _with_real_state(frame, name):
    frame.state = _GameState(name); return frame


def _t31():
    a = _agent(instructions=["right"])
    w = MyAgent(game_id="ls20-9607627b", _orchestrator=a)
    f = _with_real_state(_FakeFrame("GAME_OVER"), "GAME_OVER")
    assert w.is_done([], f) is False
    f = _with_real_state(_FakeFrame("WIN"), "WIN")
    assert w.is_done([], f) is True


def _t32():
    spec = _instruction_to_action_dict("click r5 c7")
    assert spec["name"] == "click" and spec["x"] == 7


def _t33():
    sub = {"grid_text": "x", "context_manager": "x", "inference_layer": "x",
           "action_pipeline": "x", "my_agent": "x"}
    assert compile_sources(sub) == compile_sources(sub)


def _t34():
    assert MyAgent._default_manifest() == dict(CERTIFIED_VERSIONS)


def _t35():
    class OddAP(_FakeAPModule):
        DEFAULT_CALIBRATION = {
            "a-1": {"is_reject": False, "retry_budget": 2, "debounce_steps": 3},
            "b-2": {"is_reject": False, "retry_budget": 4, "debounce_steps": 4},
            "c-3": {"is_reject": False, "retry_budget": 5, "debounce_steps": 8}}
    budget, debounce = _aggregate_clean_defaults(OddAP.DEFAULT_CALIBRATION.values())
    assert budget == 2 and debounce == 4


def _t36():
    a = Agent(game_id="lp85", config={"coder_solver_enabled": False},
              cm=_FakeCM(), il=_FakeIL(), gt=_FakeGT(), ap=_FakeAPModule)
    assert a._row.is_reject is True


def _t37():
    class DictAP(_FakeAPModule):
        DEFAULT_CALIBRATION = {"ls20-9607627b": {"is_reject": False,
                              "retry_budget": 7, "debounce_steps": 9}}
    a = Agent(game_id="ls20-9607627b",
              config={"coder_solver_enabled": False},
              cm=_FakeCM(), il=_FakeIL(), gt=_FakeGT(), ap=DictAP)
    assert a._row.retry_budget == 7


def _t38():
    class BadClean(_FakeAPModule): CLEAN_SET = {"games_count": 1}
    try: Agent(game_id="unknown-id",
               config={"coder_solver_enabled": False},
               cm=_FakeCM(), il=_FakeIL(), gt=_FakeGT(), ap=BadClean)
    except AgentError: pass
    else: raise AssertionError


def _t39():
    assert callable(build_main)
    try: build_main(["--help"])
    except SystemExit as e: assert e.code == 0


def _t40():
    import ast, inspect
    tree = ast.parse(inspect.getsource(sys.modules[__name__]))
    for n in tree.body:
        if isinstance(n, ast.Import):
            for a in n.names:
                assert a.name.split(".")[0] != "subprocess"


def _t41():
    import tempfile, shutil
    root = Path(tempfile.mkdtemp(prefix="t41_"))
    try:
        (root / "agent").mkdir()
        for name in ("grid_text", "context_manager",
                     "inference_layer", "action_pipeline"):
            (root / "agent" / f"{name}.py").write_text(f"# {name}\n", encoding="utf-8")
        (root / "agent" / "my_agent.py").write_text(
            '"""AUTO-GENERATED artifact.\n"""\nX = 1\n', encoding="utf-8")
        try: read_modular_sources(root)
        except AgentError: pass
        else: raise AssertionError
    finally:
        shutil.rmtree(root, ignore_errors=True)


def _t42():
    il = _FakeIL(instructions=["right"] * 8)
    a = Agent(game_id="ls20-9607627b",
              config={"coder_solver_enabled": False},
              cm=_FakeCM(), il=il, gt=_FakeGT(), ap=_FakeAPModule)
    a.set_available_actions(["right"])
    a.act(_frame()); a.act(_frame()); a.act(_frame())
    assert il.mode_calls[0] == "deep" and il.mode_calls[1] == "fast"


def _t43():
    il = _FakeIL(instructions=["right"] * 12)
    a = Agent(game_id="ls20-9607627b",
              config={"coder_solver_enabled": False},
              cm=_FakeCM(), il=il, gt=_FakeGT(), ap=_FakeAPModule)
    a.set_available_actions(["right"]); a.act(_frame()); a.act(_frame())
    a._handle_jolt(_stuck_jolt()); a.act(_frame())
    assert il.mode_calls[-1] == "deep"


def _t44():
    cap = Agent._DEEP_TRIGGER_MAX_PER_GAME
    il = _FakeIL(instructions=["right"] * (cap + 8))
    a = Agent(game_id="ls20-9607627b",
              config={"coder_solver_enabled": False},
              cm=_FakeCM(), il=il, gt=_FakeGT(), ap=_FakeAPModule)
    a.set_available_actions(["right"])
    for _ in range(cap):
        a._handle_jolt(_stuck_jolt()); a.act(_frame())
    assert a._deep_calls_this_game == cap
    a._handle_jolt(_stuck_jolt()); a.act(_frame())
    assert il.mode_calls[-1] == "fast"


def _t45():
    cm = _FakeCM()
    ld = LoopDetector("ls20-9607627b", False, cm, 5,
                      allow_repetition_while=lambda: True,
                      on_repetition_allowed=lambda: None)
    fp = ((), 0); fired = None
    for i in range(1, 8):
        r = ld.tick(action_id="right", target_xy=None, target_hash=None,
                    target_bbox=None, state_changed=False, objset_fp=fp)
        if r is not None: fired = i; break
    assert fired is None


def _t46():
    cap = Agent._DEEP_TRIGGER_MAX_PER_GAME
    il = _FakeIL(instructions=["right"] * (cap + 8))
    a = Agent(game_id="ls20-9607627b",
              config={"reasoner_mode": "deep",
                      "coder_solver_enabled": False},
              cm=_FakeCM(), il=il, gt=_FakeGT(), ap=_FakeAPModule)
    a.set_available_actions(["right"])
    for _ in range(cap + 4): a.act(_frame())
    assert a._deep_calls_this_game == cap
    assert il.mode_calls[cap] == "fast"


def _t47():
    il = _FakeIL(instructions=["right"] * 16)
    a = Agent(game_id="ls20-9607627b",
              config={"deep_call_cap": 2, "coder_solver_enabled": False},
              cm=_FakeCM(), il=il, gt=_FakeGT(), ap=_FakeAPModule)
    a.set_available_actions(["right"])
    a.act(_frame())
    a._handle_jolt(_stuck_jolt()); a.act(_frame())
    a.act(_frame()); a.act(_frame()); a.act(_frame()); a.act(_frame())
    assert a._deep_calls_this_game == 2


def _t48():
    for bad in [-1, 0, True, False, 3.7, "abc", "8", None, [], {}]:
        try:
            Agent(game_id="ls20-9607627b",
                  config={"deep_call_cap": bad,
                          "coder_solver_enabled": False},
                  cm=_FakeCM(), il=_FakeIL(), gt=_FakeGT(), ap=_FakeAPModule)
        except AgentError: pass
        else: raise AssertionError(f"expected AgentError for {bad!r}")
    for good in (1, 2, 8, 100):
        a = Agent(game_id="ls20-9607627b",
                  config={"deep_call_cap": good,
                          "coder_solver_enabled": False},
                  cm=_FakeCM(), il=_FakeIL(), gt=_FakeGT(), ap=_FakeAPModule)
        assert a._deep_cap() == good


def _t49():
    out = LoopDetector._format_blacklist(None)
    assert isinstance(out, str) and "SYSTEM OVERRIDE" in out
    a = _agent(instructions=["right"] * 4)
    a.set_available_actions(["right"]); a.act(_frame())
    a._handle_jolt(JoltRequest(kind="stuck"))
    assert any("SYSTEM OVERRIDE" in inj for inj in a._pending_injections)


def _t50_coder_flag_default_on():
    a = Agent(game_id="ls20-9607627b",
              cm=_FakeCM(), il=_FakeIL(), gt=_FakeGT(), ap=_FakeAPModule,
              version_manifest={"inference_layer": "1.6"})
    assert a.config.get("coder_solver_enabled") is True


def _t51_coder_flag_off_reasoner_only():
    a = _agent()
    assert a._solver is None
    assert a._transition_buffer is None
    assert a._should_attempt_coder_solve() is False


def _t52_coder_attempt_cap():
    a = _agent()

    class _StubSolver:
        def __init__(self):
            self.counters = type("C", (), {"new_level": lambda self: None})()
    class _StubBuffer:
        def __len__(self): return 5
        def as_list(self): return []

    a._solver = _StubSolver()
    a._transition_buffer = _StubBuffer()
    a._coder_attempts_this_game = CODER_ATTEMPTS_PER_GAME_CAP - 1
    a._jolt_pending = True
    assert a._should_attempt_coder_solve() is True

    a._coder_attempts_this_game = CODER_ATTEMPTS_PER_GAME_CAP
    assert a._should_attempt_coder_solve() is False

    a._coder_attempts_this_game = 0
    a._jolt_pending = False
    a._divergence_pending = False
    assert a._should_attempt_coder_solve() is False


def _t53_il_16_certified():
    assert CERTIFIED_VERSIONS["inference_layer"] == "1.6"


def _t54_verify_artifact_expects_16():
    import inspect
    src = inspect.getsource(verify_artifact)
    assert '"inference_layer": "1.6"' in src


def _t55_coder_attempts_accessor():
    a = _agent()
    a._coder_attempts_this_game = 4
    assert a.coder_attempts() == 4


def _t56_coder_result_accessor():
    a = _agent()
    assert a.coder_result() is None


def _t57_reset_level_clears_pending():
    a = _agent()
    a._pending_action = "ACTION1"
    a._prev_settled_grid = [[1, 2], [3, 4]]
    a.reset_level(level_num=1, advanced=True)
    assert a._pending_action is None
    assert a._prev_settled_grid is None


def _t58_coder_actions_accessor():
    a = _agent()
    a._coder_actions_this_game = 3
    assert a.coder_actions() == 3


def _t59_empty_buffer_no_budget_drain():
    a = _agent()

    class _StubSolver:
        def __init__(self):
            self.counters = type("C", (), {"new_level": lambda self: None})()
        def solve_world_model(self, *_a, **_kw):
            raise AssertionError("solve_world_model must not be called on empty buffer")
        def solve_action_program(self, *_a, **_kw):
            raise AssertionError("solve_action_program must not be called on empty buffer")

    class _EmptyBuffer:
        def __len__(self): return 0
        def as_list(self): return []

    a._solver = _StubSolver()
    a._transition_buffer = _EmptyBuffer()
    a._actions_at_level = 0
    assert a._should_attempt_coder_solve() is False

    for _ in range(6):
        a._maybe_admit_coder({"task": "t", "mode": "world"})

    assert a._coder_attempts_this_game == 0
    assert a._coder_attempts_this_level == 0


def _t66_proactive_default_off():
    """v1.0.20: default config (seam absent) => reactive-only, koi behavior
    change nahi — no jolt + no divergence = coder attempt nahi, chahe
    actions_at_level kuch bhi ho."""
    a = _agent()
    class _StubSolver:
        def __init__(self):
            self.counters = type("C", (), {"new_level": lambda self: None})()
    class _FullBuffer:
        def __len__(self): return 3
        def as_list(self): return []
    a._solver = _StubSolver()
    a._transition_buffer = _FullBuffer()
    a._jolt_pending = False
    a._divergence_pending = False
    a._coder_replan_pending = False
    a._coder_attempts_this_game = 0
    a._actions_at_level = 99  # bohot zyada actions, par seam absent
    assert a._should_attempt_coder_solve() is False
    # bool/invalid values ko bhi ignore karna chahiye (fail-closed)
    a.config["coder_proactive_actions_at_level"] = True
    assert a._should_attempt_coder_solve() is False
    a.config["coder_proactive_actions_at_level"] = -5
    assert a._should_attempt_coder_solve() is False


def _t67_proactive_threshold_fires():
    """v1.0.20: coder_proactive_actions_at_level=N => N actions at-level par
    (bina jolt/divergence ke) coder attempt trigger hota hai."""
    a = _agent()
    class _StubSolver:
        def __init__(self):
            self.counters = type("C", (), {"new_level": lambda self: None})()
    class _FullBuffer:
        def __len__(self): return 3
        def as_list(self): return []
    a._solver = _StubSolver()
    a._transition_buffer = _FullBuffer()
    a._jolt_pending = False
    a._divergence_pending = False
    a._coder_replan_pending = False
    a._coder_attempts_this_game = 0
    a.config["coder_proactive_actions_at_level"] = 12
    a._actions_at_level = 11
    assert a._should_attempt_coder_solve() is False
    a._actions_at_level = 12
    assert a._should_attempt_coder_solve() is True
    # attempts cap abhi bhi respected
    a._coder_attempts_this_game = CODER_ATTEMPTS_PER_GAME_CAP
    assert a._should_attempt_coder_solve() is False


def _t68_objects_passed_to_cm():
    """v1.0.21: extract_objects ka structured output cm.set_observation
    tak objects= kwarg ke through pahunchta hai (Object Vision)."""
    a = _agent(instructions=["right"] * 6)
    a.set_available_actions(["right"])
    a.act(_frame(block=(1, 1, 2, 3, 5)))
    objs = a._cm.obs_objects
    assert objs, "objects CM tak nahi pahunche"
    o = objs[0]
    for k in ("id", "color_char", "color_digit", "size", "bbox"):
        assert k in o, f"object payload missing key {k!r}"
    assert list(o["bbox"]) == [1, 2, 1, 3], o["bbox"]
    assert o["color_digit"] == 5 and o["color_char"] == "w", o
    assert o["size"] == 6, o["size"]


def _t69_palette_crosscheck():
    """v1.0.21 integration guard: CM _FALLBACK_PALETTE == grid_text
    CHAR_TO_DIGIT.  (Run-21 core audit: do certified modules alag format
    par test hue the — ye drift phir kabhi nahi hogi.)"""
    cm_pal = dict(_cm_module._FALLBACK_CHAR_TO_DIGIT)
    gt_pal = dict(_gt_module.CHAR_TO_DIGIT)
    assert cm_pal == gt_pal, (
        f"CM fallback palette drift: CM={cm_pal} GT={gt_pal}")


def _t60_level_start_no_coder_trigger():
    a = _agent()

    class _StubSolver:
        def __init__(self):
            self.counters = type("C", (), {"new_level": lambda self: None})()
    class _EmptyBuffer:
        def __len__(self): return 0
        def as_list(self): return []
    class _FullBuffer:
        def __len__(self): return 3
        def as_list(self): return []

    a._solver = _StubSolver()
    a._transition_buffer = _EmptyBuffer()
    a._actions_at_level = 0
    assert a._should_attempt_coder_solve() is False

    a._jolt_pending = True
    assert a._should_attempt_coder_solve() is False

    a._transition_buffer = _FullBuffer()
    assert a._should_attempt_coder_solve() is True


def _t61_replan_clears_hypotheses():
    a = _agent()
    a._cm.update_zone2("hypotheses", "old stale hypothesis", truncate=True)
    a._cm.update_zone2("world_model_status", "old status", truncate=True)

    class _StubResult:
        admitted = False
        replan_requested = True
        mode = "world"
        source = ""

    a._handle_coder_replan(_StubResult())

    assert a._cm._zone2["hypotheses"] == "", \
        f"hypotheses not cleared: {a._cm._zone2['hypotheses']!r}"
    assert "replan" in a._cm._zone2["world_model_status"].lower()
    assert a._coder_replan_pending is True
    assert a._coder_replan_count == 1
    assert a._coder_guarded is None
    # A-CEGIS-17: explicit deep-call trigger.
    assert a._jolt_pending is True


def _t62_replan_blocks_coder_attempt():
    a = _agent()

    class _StubSolver:
        def __init__(self):
            self.counters = type("C", (), {"new_level": lambda self: None})()
    class _FullBuffer:
        def __len__(self): return 3
        def as_list(self): return []

    a._solver = _StubSolver()
    a._transition_buffer = _FullBuffer()
    a._jolt_pending = True
    assert a._should_attempt_coder_solve() is True

    a._coder_replan_pending = True
    assert a._should_attempt_coder_solve() is False, \
        "coder attempt must be suppressed during REPLAN"


def _t63_replan_flag_cleared_after_reasoner():
    il = _FakeIL(instructions=["right"] * 4)
    a = _agent()
    a._il = il

    class _StubSolver:
        def __init__(self):
            self.counters = type("C", (), {"new_level": lambda self: None})()
    class _FullBuffer:
        def __len__(self): return 3
        def as_list(self): return []

    a._solver = _StubSolver()
    a._transition_buffer = _FullBuffer()
    a._coder_replan_pending = True

    a.act(_frame())

    assert a._coder_replan_pending is False, \
        "pending flag should be cleared after reasoner runs"


# ---------- A-CEGIS-16: world-model REPLAN short-circuits ---------- #

def _t64_world_model_replan_short_circuits():
    """A-CEGIS-16: when solve_world_model returns replan_requested=True,
    _handle_coder_replan is invoked and solve_action_program is NEVER
    called (REPLAN signal is not overwritten by the fallback)."""
    a = _agent()
    a._cm.update_zone2("hypotheses", "old", truncate=True)
    a._cm.update_zone2("world_model_status", "old", truncate=True)

    class _StubResult:
        def __init__(self, admitted=False, replan_requested=False,
                     mode="world", source=""):
            self.admitted = admitted
            self.replan_requested = replan_requested
            self.mode = mode
            self.source = source

    call_log = {"world": 0, "action": 0}

    class _StubSolver:
        def __init__(self):
            self.counters = type("C", (), {"new_level": lambda self: None})()
        def solve_world_model(self, *_a, **_kw):
            call_log["world"] += 1
            return _StubResult(admitted=False, replan_requested=True)
        def solve_action_program(self, *_a, **_kw):
            call_log["action"] += 1
            return _StubResult(admitted=True, replan_requested=False,
                               mode="action")

    class _FullBuffer:
        def __len__(self): return 3
        def as_list(self): return []

    a._solver = _StubSolver()
    a._transition_buffer = _FullBuffer()

    admitted = a._maybe_admit_coder({"task": "t", "mode": "world"})

    assert admitted is False
    assert call_log["world"] == 1
    assert call_log["action"] == 0, \
        f"A-CEGIS-16: solve_action_program must NOT run after REPLAN; got {call_log['action']} calls"
    assert a._coder_replan_pending is True
    assert a._coder_replan_count == 1
    assert a._cm._zone2["hypotheses"] == ""
    assert a._coder_guarded is None
    assert a._jolt_pending is True


def _t65_world_model_not_admitted_falls_through_to_action():
    """A-CEGIS-16 complement: when world-model fails WITHOUT a REPLAN
    (e.g. few-mismatch repair exhausted), the action-program fallback
    still runs — the short-circuit only fires on REPLAN.

    A-CEGIS-19: dead lines removed from the previous revision.
    """
    a = _agent()

    class _StubResult:
        def __init__(self, admitted=False, replan_requested=False,
                     mode="world", source=""):
            self.admitted = admitted
            self.replan_requested = replan_requested
            self.mode = mode
            self.source = source

    call_log = {"world": 0, "action": 0}

    class _StubSolver:
        def __init__(self):
            self.counters = type("C", (), {"new_level": lambda self: None})()
        def solve_world_model(self, *_a, **_kw):
            call_log["world"] += 1
            return _StubResult(admitted=False, replan_requested=False)
        def solve_action_program(self, *_a, **_kw):
            call_log["action"] += 1
            return _StubResult(admitted=True, replan_requested=False,
                               mode="action",
                               source="def next_action(f, h): return 'right'")

    class _FullBuffer:
        def __len__(self): return 3
        def as_list(self): return []

    a._solver = _StubSolver()
    a._transition_buffer = _FullBuffer()

    admitted = a._maybe_admit_coder({"task": "t", "mode": "world"})

    assert call_log["world"] == 1
    assert call_log["action"] == 1, \
        "action-program fallback must run when world-model fails WITHOUT REPLAN"
    assert admitted is True


def _run_self_tests():
    tests = [
        ("T-1", _t1), ("T-2", _t2), ("T-3", _t3), ("T-4", _t4), ("T-5", _t5),
        ("T-6", _t6), ("T-7", _t7), ("T-8", _t8), ("T-9", _t9), ("T-10", _t10),
        ("T-11", _t11), ("T-12", _t12), ("T-13", _t13), ("T-14", _t14),
        ("T-15", _t15), ("T-16", _t16), ("T-17", _t17), ("T-18", _t18),
        ("T-19", _t19), ("T-20", _t20), ("T-21", _t21), ("T-22", _t22),
        ("T-23", _t23), ("T-24", _t24), ("T-25", _t25), ("T-26", _t26),
        ("T-27", _t27), ("T-28", _t28), ("T-29", _t29), ("T-30", _t30),
        ("T-31", _t31), ("T-32", _t32), ("T-33", _t33), ("T-34", _t34),
        ("T-35", _t35), ("T-36", _t36), ("T-37", _t37), ("T-38", _t38),
        ("T-39", _t39), ("T-40", _t40), ("T-41", _t41), ("T-42", _t42),
        ("T-43", _t43), ("T-44", _t44), ("T-45", _t45), ("T-46", _t46),
        ("T-47", _t47), ("T-48", _t48), ("T-49", _t49),
        ("T-50 coder flag default ON", _t50_coder_flag_default_on),
        ("T-51 coder off reasoner-only", _t51_coder_flag_off_reasoner_only),
        ("T-52 coder attempt cap", _t52_coder_attempt_cap),
        ("T-53 IL 1.6 certified", _t53_il_16_certified),
        ("T-54 verify_artifact 1.6", _t54_verify_artifact_expects_16),
        ("T-55 coder_attempts accessor", _t55_coder_attempts_accessor),
        ("T-56 coder_result accessor", _t56_coder_result_accessor),
        ("T-57 reset_level clears pending", _t57_reset_level_clears_pending),
        ("T-58 coder_actions accessor", _t58_coder_actions_accessor),
        ("T-59 empty buffer no budget drain", _t59_empty_buffer_no_budget_drain),
        ("T-60 level-start no coder trigger", _t60_level_start_no_coder_trigger),
        ("T-61 replan clears hypotheses", _t61_replan_clears_hypotheses),
        ("T-62 replan blocks coder attempt", _t62_replan_blocks_coder_attempt),
        ("T-63 replan flag cleared after reasoner", _t63_replan_flag_cleared_after_reasoner),
        ("T-64 world-model REPLAN short-circuits (A-CEGIS-16)",
         _t64_world_model_replan_short_circuits),
        ("T-65 world-model non-REPLAN falls through (A-CEGIS-16)",
         _t65_world_model_not_admitted_falls_through_to_action),
        ("T-66 proactive coder default OFF (v1.0.20)",
         _t66_proactive_default_off),
        ("T-67 proactive coder threshold fires (v1.0.20)",
         _t67_proactive_threshold_fires),
        ("T-68 objects passed to CM (Object Vision v1.0.21)",
         _t68_objects_passed_to_cm),
        ("T-69 CM/grid_text palette cross-check (v1.0.21)",
         _t69_palette_crosscheck),
    ]
    failed = 0
    for name, fn in tests:
        try:
            fn(); logging.info("PASS  %s", name)
        except Exception as e:
            failed += 1
            logging.info("FAIL  %s: %s: %s", name, type(e).__name__, e)
    logging.info("")
    logging.info("%d/%d passed", len(tests) - failed, len(tests))
    return 0 if failed == 0 else 1


def _main(argv=None):
    if argv is None: argv = sys.argv[1:]
    if argv and argv[0] == "build": return build_main(argv[1:])
    if argv and argv[0] == "test": return _run_self_tests()
    if argv and argv[0] in ("-h", "--help", "help"):
        print("Usage: my_agent.py [command]\n\n"
              "Commands:\n  test\n  build [options]\n  help")
        return 0
    if argv:
        print(f"unknown command: {argv[0]!r}", file=sys.stderr)
        return 2
    return _run_self_tests()


if __name__ == "__main__":
    import sys as _sys
    logging.basicConfig(level=logging.INFO,
                        format="%(message)s", stream=_sys.stdout)
    _sys.exit(_main())


In [ ]:
%%writefile goa/agent/lora_datagen.py
#!/usr/bin/env python3
# lora_datagen.py — Task 8 Phase A dataset generator (Rev 6)
# CPU-only, deterministic (process-stable, per-game isolated), no network at runtime.
# Public API only from certified modules (DIR/060 §4.2).
#
# ============================================================
# DEVIATION REGISTER (S-8) — D-1 .. D-10
# ============================================================
# D-1  Zone 1/2 template loaded from external file
#      (--zone12-template / GOA_ZONE12_TEMPLATE). Certified reasoner
#      template not available to this author. OPEN — Chief supplies
#      file before Phase B.
# D-2  parse_replays import per DIR/060 §4.2 (certified). CLOSED.
# D-3  Engine-level seed not exposed by arcengine 0.9.3. Mitigated by
#      sha256-derived per-game seeding. Backend re-run is the arbiter.
# D-4  DG-5 stratified K=20 across episode (not first-K). CLOSED.
# D-5  Pre-flight zero-coverage check covers all 25 games per order
#      §3 STEP 1 literal text. Chief to rule if hold-out coverage may
#      be waived. OPEN.
# D-6  pyarrow required for inline fallback loader only. Not in the
#      §4.1 forbidden list. CLOSED.
# D-7  Manifest self-hash excluded (avoid self-reference). CLOSED.
# D-8  Split tolerance: warn if observed replay-guided fraction
#      deviates from 0.60 by >0.10. Order permits adjustment to replay
#      availability. CLOSED.
# D-9  Certified engine legend fixtures not supplied. Synthetic
#      fixtures prove code-path liveness, not production parity.
#      OPEN — backend to supply certified fixtures.
# D-10 Template validated at load time via dummy render (N-2 fix).
#      CLOSED.
#
# TEMPLATE CONTRACT: placeholders {before_grid}, {action}, {after_grid},
# {what_changed} are REQUIRED. Literal braces must be doubled ({{, }}).
# Any missing required placeholder fails at load time, before any
# rollout runs (N-6).
#
# REV 6 CHANGELOG (DIR/066):
#   FIX 1 (H-F1) — r11l-495a7899, sc25-635fd71a (two IDs corrected).
#   FIX 2 (H-F2) — local valid_grid helper; grid_text.valid_grid import
#                   removed (grid_text v1.0.8 __all__ does not export it).
#   FIX 3 (N-8)  — _load_cert_replays hard-aborts on ImportError unless
#                   --no-cert-replays explicitly passed.
#   FIX 4 (N-9)  — per-game global random module re-seed at game-loop
#                   entry, isolating any engine-side global RNG use.
#   N-11         — _cert_load_attempted guard; no double-log / double-raise.
# ============================================================

from __future__ import annotations

import argparse
import hashlib
import json
import logging
import os
import random
import sys
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Iterator

# --- Certified module imports (public API only, §4.2) ---
# LA-2 FIX (Indus, 2026-09-29 final logic audit): bare flat import breaks
# in the notebook package context (only agent.* is importable there) —
# crashed FRAME PROBE's RGB branch. Dual import, fail-closed on neither.
try:
    from grid_text import extract_objects, object_delta
except ImportError:
    from agent.grid_text import extract_objects, object_delta

# parse_replays IS certified per DIR/060 §4.2 — default path.
_cert_iter_replays = None
_cert_load_attempted = False
_REPLAY_SOURCE = "inline_parquet"

def _load_cert_replays(prefer_cert: bool) -> None:
    """
    FIX 3 (N-8): hard abort on ImportError unless operator passed
    --no-cert-replays.  N-11: no retry / no double log on a second call.
    """
    global _cert_iter_replays, _REPLAY_SOURCE, _cert_load_attempted
    if _cert_iter_replays is not None:
        return
    if _cert_load_attempted:
        if prefer_cert:
            raise RuntimeError(
                "parse_replays previously failed to import; "
                "pass --no-cert-replays to use the inline parquet reader."
            )
        return
    _cert_load_attempted = True
    if not prefer_cert:
        _REPLAY_SOURCE = "inline_parquet"
        return
    try:
        from parse_replays import iter_replay_sessions as _fn  # noqa
        _cert_iter_replays = _fn
        _REPLAY_SOURCE = "parse_replays"
    except ImportError as e:
        raise RuntimeError(
            f"Certified parse_replays unavailable: {e}. "
            "Pass --no-cert-replays to use the inline parquet reader."
        ) from e

try:
    from arc_agi import Arcade
except ImportError:
    try:
        from arcengine import Arcade
    except ImportError:
        Arcade = None


# ============================================================
# Configuration
# ============================================================

HOLD_OUT_GAMES = {
    "ft09-0d8bbf25", "vc33-5430563c", "tu93-0768757b",
    "ka59-38d34dbb", "cd82-fb555c5d",
}

ALL_GAME_IDS = [
    "ar25-0c556536", "bp35-0a0ad940", "cd82-fb555c5d", "cn04-2fe56bfb",
    "dc22-fdcac232", "ft09-0d8bbf25", "g50t-5849a774", "ka59-38d34dbb",
    "lf52-271a04aa", "lp85-305b61c3", "ls20-9607627b", "m0r0-492f87ba",
    "r11l-495a7899", "re86-8af5384d", "s5i5-18d95033", "sb26-7fbdac44",
    "sc25-635fd71a", "sk48-d8078629", "sp80-589a99af", "su15-1944f8ab",
    "tn36-ef4dde99", "tr87-cd924810", "tu93-0768757b", "vc33-5430563c",
    "wa30-ee6fef47",
]

TRAIN_GAMES = [g for g in ALL_GAME_IDS if g not in HOLD_OUT_GAMES]
assert len(TRAIN_GAMES) == 20, f"expected 20 train games, got {len(TRAIN_GAMES)}"

ACTIONS_PER_GAME = 500
REPLAY_TARGET_FRACTION = 0.60
SPLIT_WARN_TOLERANCE = 0.10
DG5_SAMPLE_K = 20
SEED = 20260921
DATASET_VERSION = "1.0.0"
SCHEMA_VERSION = "1.0"

REPLAY_TOTAL_EXPECTED = 340
REPLAY_USABLE_EXPECTED = 336
REPLAY_RANGE_LO, REPLAY_RANGE_HI = 336, 342

UNUSABLE_REPLAYS = {"p0_44", "p1_0", "p1_9", "p2_21"}

ARC_PALETTE = [
    (0, 0, 0), (0, 116, 217), (255, 65, 54), (46, 204, 64),
    (255, 220, 0), (170, 170, 170), (240, 18, 190), (255, 133, 27),
    (127, 219, 255), (135, 12, 37), (255, 255, 255), (0, 255, 0),
    (0, 0, 255), (255, 0, 0), (255, 255, 0), (128, 128, 128),
]
_PALETTE_LOOKUP = {c: i for i, c in enumerate(ARC_PALETTE)}


# ============================================================
# Helpers — deterministic by construction
# ============================================================

def _sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

def _stable_hash(obj: Any) -> str:
    s = json.dumps(obj, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(s.encode()).hexdigest()

def _stable_seed_int(*parts: str) -> int:
    h = hashlib.sha256("|".join(parts).encode()).digest()
    return int.from_bytes(h[:8], "big")

def valid_grid(frame: Any) -> bool:
    """
    FIX 2 (H-F2): local engine-style frame validator, DIR/062 §4a shape.
    Frame is a non-empty list of rectangular lists of int in [0, 15].
    Certified grid_text v1.0.8 does NOT export valid_grid.
    """
    if not isinstance(frame, list) or not frame:
        return False
    if not isinstance(frame[0], list) or not frame[0]:
        return False
    w = len(frame[0])
    for row in frame:
        if not isinstance(row, list) or len(row) != w:
            return False
        for v in row:
            if not isinstance(v, int) or v < 0 or v > 15:
                return False
    return True

def log_warn(msg: str) -> None: logging.warning("[WARN] %s", msg)
def log_info(msg: str) -> None: logging.info("[INFO] %s", msg)


# ============================================================
# Replay loading
# ============================================================

def _inline_iter_replays(replay_dir: Path) -> Iterator[dict]:
    import pyarrow.parquet as pq
    for fp in sorted(replay_dir.glob("*.parquet")):
        pf = pq.ParquetFile(fp)
        for rg in range(pf.num_row_groups):
            for row in pf.read_row_group(rg).to_pylist():
                yield _normalize_replay_row(row)

def _normalize_replay_row(row: dict) -> dict:
    sid = row.get("id") or row.get("session_id") or row.get("name")
    gid = row.get("game_id") or row.get("game")
    actions = row.get("actions") or row.get("action_list") or []
    if isinstance(actions, str):
        actions = json.loads(actions)
    return {"id": sid, "game_id": gid, "actions": list(actions)}

def _iter_all_replays(replay_dir: Path, prefer_cert: bool) -> Iterator[dict]:
    if prefer_cert:
        _load_cert_replays(prefer_cert)
        if _cert_iter_replays is not None:
            yield from _cert_iter_replays(replay_dir)
            return
    yield from _inline_iter_replays(replay_dir)


# ============================================================
# STEP 1 — Replay pre-flight
# ============================================================

def verify_replays(replay_dir: Path, prefer_cert: bool) -> dict:
    if replay_dir is None or not replay_dir.exists():
        raise RuntimeError("Replay directory missing.")

    sessions = list(_iter_all_replays(replay_dir, prefer_cert))
    total = len(sessions)
    usable = [s for s in sessions if s.get("id") not in UNUSABLE_REPLAYS]
    usable_count = len(usable)

    if not (REPLAY_RANGE_LO <= total <= REPLAY_RANGE_HI):
        raise RuntimeError(
            f"Total sessions {total} outside {REPLAY_RANGE_LO}-{REPLAY_RANGE_HI}. "
            "STOP and report to Chief."
        )
    if not (REPLAY_RANGE_LO <= usable_count <= REPLAY_RANGE_HI):
        raise RuntimeError(
            f"Usable sessions {usable_count} outside "
            f"{REPLAY_RANGE_LO}-{REPLAY_RANGE_HI}. STOP and report to Chief."
        )
    if total != REPLAY_TOTAL_EXPECTED:
        log_warn(f"total sessions = {total}, Indus-verified = {REPLAY_TOTAL_EXPECTED}")
    if usable_count != REPLAY_USABLE_EXPECTED:
        log_warn(f"usable sessions = {usable_count}, "
                 f"Indus-verified = {REPLAY_USABLE_EXPECTED}")

    per_game = Counter()
    for s in usable:
        gid = s.get("game_id")
        if gid:
            per_game[gid] += 1

    missing = [g for g in ALL_GAME_IDS if per_game[g] == 0]
    if missing:
        raise RuntimeError(
            f"ZERO replay coverage for games: {missing}. STOP per §3 STEP 1."
        )

    return {
        "total_sessions": total,
        "usable_sessions": usable_count,
        "expected_total": REPLAY_TOTAL_EXPECTED,
        "expected_usable": REPLAY_USABLE_EXPECTED,
        "per_game_counts": dict(per_game),
        "unusable_skipped": sorted(UNUSABLE_REPLAYS),
        "replay_source": _REPLAY_SOURCE,
    }


# ============================================================
# STEP 2 — RGB -> grid adapter
# ============================================================

def _try_engine_legend(game_obj: Any, game_id: str) -> dict | None:
    legend = getattr(game_obj, "legend", None)
    if isinstance(legend, dict):
        return legend
    try:
        from arcengine import get_color_legend  # type: ignore
        lg = get_color_legend(game_id)
        if isinstance(lg, dict):
            return lg
    except Exception:
        pass
    return None

def detect_adapter_mode(game_obj: Any, game_id: str) -> tuple[str, dict | None]:
    legend = _try_engine_legend(game_obj, game_id)
    return ("legend", legend) if legend is not None else ("nearest-match", None)

def _nearest_palette_idx(pixel: tuple[int, int, int]) -> int:
    hit = _PALETTE_LOOKUP.get(pixel)
    if hit is not None:
        return hit
    r, g, b = pixel
    best_i, best_d = 0, 1 << 30
    for i, (pr, pg, pb) in enumerate(ARC_PALETTE):
        d = (r - pr) ** 2 + (g - pg) ** 2 + (b - pb) ** 2
        if d < best_d:
            best_d, best_i = d, i
    return best_i

def rgb_to_grid(frame: Any, legend: dict | None) -> list[list[int]]:
    if legend is not None:
        return [[legend.get(tuple(px), _nearest_palette_idx(tuple(px)))
                 for px in row] for row in frame]
    return [[_nearest_palette_idx(tuple(px)) for px in row] for row in frame]

def dg3_fixture_parity() -> dict:
    """Exercises both adapter paths. Synthetic legend (D-9)."""
    black, white = (0, 0, 0), (255, 255, 255)

    nm_cases = [
        ("nm_black_1x1", [[black]], [[0]]),
        ("nm_white_1x1", [[white]], [[10]]),
        ("nm_palette_id_2x2",
         [[ARC_PALETTE[0], ARC_PALETTE[1]],
          [ARC_PALETTE[2], ARC_PALETTE[3]]],
         [[0, 1], [2, 3]]),
    ]
    nm_p, nm_f, nm_d = 0, 0, []
    for name, rgb, exp in nm_cases:
        got = rgb_to_grid(rgb, None)
        ok = got == exp
        nm_p += int(ok); nm_f += int(not ok)
        nm_d.append({"name": name, "ok": ok})

    legend_identity = {c: i for i, c in enumerate(ARC_PALETTE)}
    legend_shifted = {ARC_PALETTE[i]: (i + 1) % 16 for i in range(16)}

    lg_cases = [
        ("lg_identity_black", legend_identity, [[black]], [[0]]),
        ("lg_identity_white", legend_identity, [[white]], [[10]]),
        ("lg_identity_palette_2x2", legend_identity,
         [[ARC_PALETTE[0], ARC_PALETTE[1]],
          [ARC_PALETTE[2], ARC_PALETTE[3]]],
         [[0, 1], [2, 3]]),
        ("lg_shifted_black", legend_shifted, [[black]], [[1]]),
        ("lg_shifted_white", legend_shifted, [[white]], [[11]]),
    ]
    lg_p, lg_f, lg_d = 0, 0, []
    for name, lg, rgb, exp in lg_cases:
        got = rgb_to_grid(rgb, lg)
        ok = got == exp
        lg_p += int(ok); lg_f += int(not ok)
        lg_d.append({"name": name, "ok": ok})

    return {
        "nearest_match": {"passed": nm_p, "failed": nm_f, "details": nm_d},
        "legend_path":   {"passed": lg_p, "failed": lg_f, "details": lg_d},
        "certified_engine_fixtures": "NOT SUPPLIED (D-9)",
    }


# ============================================================
# STEP 3 — Rollout core
# ============================================================

def _normalize_action(payload: Any) -> dict:
    if isinstance(payload, dict):
        a = int(payload.get("action", -1))
        if a == 6:
            x, y = payload.get("x"), payload.get("y")
            if not (isinstance(x, int) and isinstance(y, int)
                    and 0 <= x <= 63 and 0 <= y <= 63):
                raise ValueError(f"ACTION6 missing/invalid x,y: {payload!r}")
            return {"action": 6, "x": x, "y": y}
        return {"action": a}
    a = int(payload)
    if a == 6:
        raise ValueError("bare ACTION6 without x,y is illegal")
    return {"action": a}

def _step_env(env: Any, payload: dict) -> Any:
    if payload["action"] == 6:
        return env.step(6, payload["x"], payload["y"])
    return env.step(payload["action"])

def _sorted_available(env: Any) -> list[int]:
    return sorted(int(a) for a in env.available_actions())

def _validate_transition_schema(t: dict) -> None:
    required = ("game_id", "step_index", "before_grid", "action",
                "after_grid", "what_changed", "source", "level")
    for k in required:
        if k not in t or t[k] is None:
            raise ValueError(f"transition missing/None: {k} in {t.get('game_id')}")
    if not isinstance(t["before_grid"], list) or not isinstance(t["after_grid"], list):
        raise ValueError("grids must be lists")
    if not isinstance(t["step_index"], int) or t["step_index"] < 0:
        raise ValueError("step_index must be non-negative int")
    if t["source"] not in ("random", "replay-guided"):
        raise ValueError(f"bad source: {t['source']}")
    if not isinstance(t["what_changed"], dict):
        raise ValueError(f"what_changed must be dict, got {type(t['what_changed'])}")
    a = t["action"]
    if not isinstance(a, dict) or "action" not in a:
        raise ValueError(f"bad action payload: {a!r}")
    if a["action"] == 6:
        if not (isinstance(a.get("x"), int) and isinstance(a.get("y"), int)
                and 0 <= a["x"] <= 63 and 0 <= a["y"] <= 63):
            raise ValueError(f"ACTION6 payload invalid: {a!r}")

def _make_transition(game_id, step_index, before_grid, action_payload,
                     after_grid, what_changed, source, level) -> dict:
    return {
        "game_id": game_id,
        "step_index": step_index,
        "before_grid": before_grid,
        "action": action_payload,
        "after_grid": after_grid,
        "what_changed": what_changed,
        "source": source,
        "level": level,
    }

def random_branch_from(env, game_id, legend, before_grid, before_objs,
                       level, num_actions, rng, step_offset) -> list[dict]:
    out: list[dict] = []
    if num_actions <= 0:
        return out
    for i in range(num_actions):
        avail = _sorted_available(env)
        if not avail:
            break
        a = rng.choice(avail)
        if a == 6:
            payload = {"action": 6, "x": rng.randint(0, 63),
                       "y": rng.randint(0, 63)}
        else:
            payload = {"action": a}
        try:
            obs, _r, done, info = _step_env(env, payload)
        except Exception as e:
            log_warn(f"{game_id}: step failed at {step_offset+i}: {e}")
            break
        after_grid = rgb_to_grid(obs, legend)
        after_objs = extract_objects(after_grid)
        delta = object_delta(before_objs, after_objs)
        lvl = info.get("level", level) if isinstance(info, dict) else level
        out.append(_make_transition(game_id, step_offset + i, before_grid,
                                    payload, after_grid, delta,
                                    "random", lvl))
        if done:
            obs = env.reset()
            before_grid = rgb_to_grid(obs, legend)
            before_objs = extract_objects(before_grid)
            level = 0
        else:
            before_grid, before_objs, level = after_grid, after_objs, lvl
    return out

def replay_guided_rollout(env, game_id, legend, replay_actions,
                          target_total, rng) -> list[dict]:
    out: list[dict] = []
    obs = env.reset()
    before_grid = rgb_to_grid(obs, legend)
    before_objs = extract_objects(before_grid)
    level = 0

    for step, raw in enumerate(replay_actions):
        if len(out) >= target_total:
            break
        try:
            payload = _normalize_action(raw)
        except ValueError:
            continue
        try:
            obs, _r, done, info = _step_env(env, payload)
        except Exception as e:
            log_warn(f"{game_id}: replay step {step} failed: {e}")
            break
        after_grid = rgb_to_grid(obs, legend)
        after_objs = extract_objects(after_grid)
        delta = object_delta(before_objs, after_objs)
        lvl = info.get("level", level) if isinstance(info, dict) else level
        out.append(_make_transition(game_id, step, before_grid, payload,
                                    after_grid, delta, "replay-guided", lvl))
        if done:
            obs = env.reset()
            before_grid = rgb_to_grid(obs, legend)
            before_objs = extract_objects(before_grid)
            level = 0
        else:
            before_grid, before_objs, level = after_grid, after_objs, lvl

    remaining = target_total - len(out)
    if remaining > 0:
        more = random_branch_from(
            env, game_id, legend, before_grid, before_objs, level,
            remaining, rng, step_offset=len(out),
        )
        out.extend(more)
    return out

def generate_game_transitions(game_id, replay_sessions) -> dict:
    if Arcade is None:
        raise RuntimeError("arcengine/arc_agi not importable — CPU-only firewall broken.")

    # FIX 4 (N-9): re-seed the GLOBAL random module per game, so any
    # engine-side consumption of global random is re-isolated per game.
    random.seed(_stable_seed_int(str(SEED), game_id, "global"))

    game_rng = random.Random(_stable_seed_int(str(SEED), game_id))

    env = Arcade.make(game_id)
    try:
        mode, legend = detect_adapter_mode(env, game_id)

        game_replays = [s for s in replay_sessions if s.get("game_id") == game_id]
        game_replays.sort(key=lambda s: str(s.get("id", "")))
        replay_actions: list = []
        for s in game_replays:
            replay_actions.extend(s.get("actions", []))

        target_replay = int(ACTIONS_PER_GAME * REPLAY_TARGET_FRACTION)
        transitions: list[dict] = []

        if replay_actions:
            transitions = replay_guided_rollout(
                env, game_id, legend, replay_actions, target_replay, game_rng
            )
        remaining = ACTIONS_PER_GAME - len(transitions)
        if remaining > 0:
            obs = env.reset()
            bg = rgb_to_grid(obs, legend)
            bo = extract_objects(bg)
            more = random_branch_from(
                env, game_id, legend, bg, bo, 0, remaining, game_rng,
                step_offset=len(transitions),
            )
            transitions.extend(more)
        transitions = transitions[:ACTIONS_PER_GAME]

        for t in transitions:
            _validate_transition_schema(t)

        guided = sum(1 for t in transitions if t["source"] == "replay-guided")
        frac = guided / max(1, len(transitions))
        if abs(frac - REPLAY_TARGET_FRACTION) > SPLIT_WARN_TOLERANCE:
            log_warn(f"{game_id}: replay-guided fraction {frac:.3f} deviates "
                     f"from target {REPLAY_TARGET_FRACTION} by >{SPLIT_WARN_TOLERANCE}")

        dg5 = dg5_sample_replay(game_id, legend, transitions)

        return {"transitions": transitions,
                "adapter_mode": mode,
                "dg5": dg5,
                "replay_fraction": frac}
    finally:
        try:
            env.close()
        except Exception:
            pass

def dg5_sample_replay(game_id, legend, transitions) -> dict:
    if Arcade is None or not transitions:
        return {"checked": 0, "matched": 0, "mismatched": 0, "indices": []}
    N = len(transitions)
    k = min(DG5_SAMPLE_K, N)
    if k == 0:
        return {"checked": 0, "matched": 0, "mismatched": 0, "indices": []}
    idxs = sorted({int(round(i * (N - 1) / (k - 1))) for i in range(k)}) if k > 1 else [0]
    env = Arcade.make(game_id)
    checked = matched = 0
    try:
        env.reset()
        last = -1
        for i in idxs:
            while last + 1 <= i:
                last += 1
                try:
                    payload = _normalize_action(transitions[last]["action"])
                    obs, _r, done, _info = _step_env(env, payload)
                except Exception:
                    return {"checked": checked, "matched": matched,
                            "mismatched": checked - matched, "indices": idxs}
                if done:
                    env.reset()
            got = rgb_to_grid(obs, legend)
            checked += 1
            if _stable_hash(got) == _stable_hash(transitions[i]["after_grid"]):
                matched += 1
    finally:
        try:
            env.close()
        except Exception:
            pass
    return {"checked": checked, "matched": matched,
            "mismatched": checked - matched, "indices": idxs}


# ============================================================
# STEP 4 — Zone 1/2 encoding
# ============================================================

_REQUIRED_PLACEHOLDERS = ("before_grid", "action", "after_grid", "what_changed")

def _load_zone12_template(template_path: Path | None) -> str:
    if template_path is None:
        env_path = os.environ.get("GOA_ZONE12_TEMPLATE")
        if env_path:
            template_path = Path(env_path)
    if template_path is None or not template_path.exists():
        raise RuntimeError(
            "Zone 1/2 template not supplied. Provide --zone12-template <file> "
            "or GOA_ZONE12_TEMPLATE=<file>. Phase B blocked until then (D-1)."
        )
    template = template_path.read_text()

    # N-6 fix: missing required placeholder is a HARD FAIL, not a warning.
    missing = [ph for ph in _REQUIRED_PLACEHOLDERS if "{" + ph + "}" not in template]
    if missing:
        raise RuntimeError(
            f"Zone 1/2 template missing required placeholder(s): {missing}. "
            "On-distribution mandate (D6) cannot be satisfied. Reject before rollout."
        )

    # N-2 fix: dummy render validates brace escaping / formatter syntax.
    dummy = {
        "before_grid": [[0]], "action": {"action": 1},
        "after_grid": [[0]], "what_changed": {},
    }
    try:
        template.format(**{k: json.dumps(dummy[k], separators=(",", ":"))
                           for k in _REQUIRED_PLACEHOLDERS})
    except (KeyError, IndexError, ValueError) as e:
        raise RuntimeError(
            f"Zone 1/2 template invalid (brace/formatter error): {e}. "
            "Literal braces must be doubled ({{, }})."
        )
    return template

def encode_zone12(transition: dict, template: str) -> str:
    return template.format(
        before_grid=json.dumps(transition["before_grid"], separators=(",", ":")),
        action=json.dumps(transition["action"], separators=(",", ":")),
        after_grid=json.dumps(transition["after_grid"], separators=(",", ":")),
        what_changed=json.dumps(transition["what_changed"], separators=(",", ":")),
    )


# ============================================================
# STEP 5 — Balance report + manifest
# ============================================================

def build_balance_report(transitions: list[dict]) -> dict:
    per_game = Counter(t["game_id"] for t in transitions)
    source_split = Counter(t["source"] for t in transitions)
    level_hist = defaultdict(Counter)
    for t in transitions:
        level_hist[t["game_id"]][t["level"]] += 1
    return {
        "per_game_counts": dict(per_game),
        "source_split": dict(source_split),
        "level_histogram": {g: dict(h) for g, h in level_hist.items()},
    }

def build_manifest(dataset_dir, seed, code_hash, adapter_modes, balance,
                   dg3, dg5, replay_meta) -> dict:
    files = {}
    for p in sorted(dataset_dir.rglob("*")):
        if p.is_file() and p.name != "manifest.json":
            files[str(p.relative_to(dataset_dir))] = _sha256_file(p)
    return {
        "seed": seed,
        "code_hash": code_hash,
        "adapter_modes": adapter_modes,
        "schema_version": SCHEMA_VERSION,
        "dataset_version": DATASET_VERSION,
        "files": files,
        "balance_report": balance,
        "dg3_fixture_parity": dg3,
        "dg5_sample_replay": dg5,
        "hold_out_games": sorted(HOLD_OUT_GAMES),
        "train_games": TRAIN_GAMES,
        "replay_meta": replay_meta,
        "total_transitions": sum(balance["per_game_counts"].values()),
    }


# ============================================================
# Main
# ============================================================

def main() -> int:
    parser = argparse.ArgumentParser(description="Task 8 Phase A dataset generator")
    parser.add_argument("--replay-dir", type=Path, required=True)
    parser.add_argument("--output-dir", type=Path, required=True)
    parser.add_argument("--zone12-template", type=Path, default=None)
    parser.add_argument("--seed", type=int, default=SEED)
    parser.add_argument("--no-cert-replays", action="store_true",
                        help="Force inline parquet loader (default: certified parse_replays)")
    args = parser.parse_args()

    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")
    random.seed(args.seed)  # informational only; per-game RNG is isolated

    prefer_cert = not args.no_cert_replays

    # STEP 1 — pre-flight
    replay_meta = verify_replays(args.replay_dir, prefer_cert=prefer_cert)
    log_info(f"replays OK: total={replay_meta['total_sessions']} "
             f"usable={replay_meta['usable_sessions']} "
             f"source={replay_meta['replay_source']}")

    all_sessions = [s for s in _iter_all_replays(args.replay_dir, prefer_cert)
                    if s.get("id") not in UNUSABLE_REPLAYS]

    # STEP 2 — DG-3 fixtures
    dg3 = dg3_fixture_parity()
    if dg3["nearest_match"]["failed"] > 0 or dg3["legend_path"]["failed"] > 0:
        raise RuntimeError(f"DG-3 fixture parity FAILED: {dg3}")

    # STEP 4 pre-flight — template must validate before rollouts
    template = _load_zone12_template(args.zone12_template)

    # STEP 3 — per-game generation (isolated RNG per game)
    all_transitions: list[dict] = []
    adapter_modes: dict[str, str] = {}
    dg5_report: dict[str, dict] = {}
    for game_id in TRAIN_GAMES:
        log_info(f"rollout {game_id}")
        res = generate_game_transitions(game_id, all_sessions)
        adapter_modes[game_id] = res["adapter_mode"]
        dg5_report[game_id] = res["dg5"]
        all_transitions.extend(res["transitions"])

    # DG-2 firewall
    for t in all_transitions:
        assert t["game_id"] not in HOLD_OUT_GAMES, f"hold-out leak: {t['game_id']}"
    assert len(all_transitions) == len(TRAIN_GAMES) * ACTIONS_PER_GAME, \
        f"expected {len(TRAIN_GAMES)*ACTIONS_PER_GAME}, got {len(all_transitions)}"

    # STEP 4 — encode
    for t in all_transitions:
        t["text"] = encode_zone12(t, template)

    # STEP 5 — write
    out = args.output_dir
    out.mkdir(parents=True, exist_ok=True)
    with open(out / "transitions.jsonl", "w") as f:
        for t in all_transitions:
            f.write(json.dumps(t, separators=(",", ":")) + "\n")

    balance = build_balance_report(all_transitions)
    with open(out / "balance_report.json", "w") as f:
        json.dump(balance, f, indent=2)

    code_hash = _sha256_file(Path(__file__))
    manifest = build_manifest(out, args.seed, code_hash, adapter_modes, balance,
                              dg3, dg5_report, replay_meta)
    with open(out / "manifest.json", "w") as f:
        json.dump(manifest, f, indent=2)

    log_info(f"dataset written: {out} ({len(all_transitions)} transitions)")
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile /kaggle/working/goa/vllm_transport.py
"""GoA test-harness vLLM transport adapter v26.3.4 — Deep Mode."""
from __future__ import annotations
import hashlib, json, os
from typing import Any, Dict, Optional, Tuple

REASONER_DEEP_THINKING_BUDGET = 2000
REASONER_DEEP_MAX_TOKENS      = 6144
REASONER_DEEP_EFFORT          = "medium"
REASONER_MIN_FLOOR            = 1024
ENABLE_THINKING_FLOOR         = 6144
RETRY_DOUBLED_CAP             = 8192

STALE_CHAT_TEMPLATE_KEYS: Tuple[str, ...] = (
    "thinking_budget", "reasoning_effort", "enable_thinking",
)
DEFAULT_CODER_MODEL_NAME = "qwen3-coder-30b-a3b"
V25_1_A_ENV_VAR = "V25_1_A_ENABLE_EXPLICIT_THINKING"


class VLLMTransport:
    def __init__(self, *args, client=None, base_url=None,
                 coder_base_url=None, coder_model_name=None, **kwargs):
        if len(args) >= 2 and isinstance(args[0], str) and isinstance(args[1], str):
            base_url, coder_base_url = args[0], args[1]
        elif len(args) == 1 and isinstance(args[0], str):
            base_url = args[0]
        self._client = client
        self._base_url = base_url or os.environ.get(
            "VLLM_BASE_URL", "http://localhost:8000")
        self._coder_base_url = coder_base_url or os.environ.get(
            "VLLM_CODER_URL", "http://localhost:8001")
        self._coder_model_name = (coder_model_name
                                  or os.environ.get("VLLM_CODER_NAME")
                                  or DEFAULT_CODER_MODEL_NAME)
        self._v25_1_a_enabled = os.environ.get(V25_1_A_ENV_VAR) == "1"

    def set_v25_1_a(self, enabled):
        self._v25_1_a_enabled = bool(enabled)

    def get_v25_1_a(self):
        return self._v25_1_a_enabled

    def _is_coder(self, model):
        return model == self._coder_model_name

    def _target_url(self, model):
        return self._coder_base_url if self._is_coder(model) else self._base_url

    def __call__(self, *, model, system=None, messages, max_tokens,
                 temperature=0.0, top_p=1.0, top_k=-1, stop_sequences=None,
                 chat_template_kwargs=None, presence_penalty=0.0,
                 repetition_penalty=1.0, reasoning_mode=None, **_unexpected):
        if _unexpected:
            print(f"WARN vllm_transport: ignored kwargs: {sorted(_unexpected)}")

        if reasoning_mode not in ("deep", "fast"):
            if reasoning_mode is not None:
                print(f"WARN vllm_transport: unknown reasoning_mode {reasoning_mode!r}")
            print("WARN vllm_transport: reasoning_mode absent/invalid — FAILING CLOSED TO FAST")
            reasoning_mode = "fast"

        _ctk = dict(chat_template_kwargs or {})
        for _stale in STALE_CHAT_TEMPLATE_KEYS:
            _ctk.pop(_stale, None)

        diag = os.environ.get("TRANSPORT_FLOOR_OVERRIDE") == "diag"
        if not diag:
            if not self._is_coder(model) and max_tokens < REASONER_MIN_FLOOR:
                max_tokens = REASONER_MIN_FLOOR
            if reasoning_mode == "deep" and max_tokens < ENABLE_THINKING_FLOOR:
                max_tokens = ENABLE_THINKING_FLOOR

        body: Dict[str, Any] = {
            "model": model, "messages": messages, "max_tokens": max_tokens,
            "temperature": temperature, "top_p": top_p, "top_k": top_k,
            "stop_sequences": list(stop_sequences or []),
            "presence_penalty": presence_penalty,
            "repetition_penalty": repetition_penalty,
        }
        if system is not None:
            body["system"] = system

        if reasoning_mode == "deep":
            body["reasoning_effort"] = REASONER_DEEP_EFFORT
            body["thinking_token_budget"] = REASONER_DEEP_THINKING_BUDGET
            # LA-7 FIX (Indus, 2026-09-29, Run-15 attempt-3 postmortem): the
            # Qwen3 template reads ONLY chat_template_kwargs. Top-level
            # enable_thinking/thinking_token_budget are silently ignored by
            # vLLM 0.30 (wire evidence: rt=0, ct=6144, finish=length on
            # attempt-3 STEP[1]). Nested form = the documented mechanism,
            # same key the server launch flag uses.
            _ctk["enable_thinking"] = True
            _ctk["thinking_budget"] = REASONER_DEEP_THINKING_BUDGET
            if self._v25_1_a_enabled:
                body["enable_thinking"] = True  # legacy A/B evidence only
                print("WARN vllm_transport: v25.1-A — TOP-LEVEL enable_thinking=True")
        else:
            _ctk["enable_thinking"] = False

        if _ctk:
            body["chat_template_kwargs"] = _ctk

        target = self._target_url(model)
        retry_on = os.environ.get("DISABLE_LENGTH_RETRY") != "1"
        _attempt_mt = max_tokens
        while True:
            body["max_tokens"] = _attempt_mt
            resp = self._client.post(f"{target}/v1/chat/completions", json=body)
            if not retry_on:
                break
            fr = self._extract_finish_reason(resp)
            if fr != "length" or _attempt_mt >= RETRY_DOUBLED_CAP:
                break
            _attempt_mt = min(_attempt_mt * 2, RETRY_DOUBLED_CAP)
            print(f"WARN vllm_transport: finish_reason=length; retrying mt={_attempt_mt}")
        return resp

    def post_wire_capture(self, body):
        wire_body_bytes = json.dumps(
            body, separators=(",", ":"), sort_keys=False).encode("utf-8")
        wire_sha256 = hashlib.sha256(wire_body_bytes).hexdigest()
        rec = {"wire_sha256": wire_sha256,
               "body_length_bytes": len(wire_body_bytes), "body": body}
        target = self._target_url(body.get("model", ""))
        resp = self._client.post(f"{target}/v1/chat/completions",
                                 data=wire_body_bytes,
                                 headers={"Content-Type": "application/json"})
        return {"record": rec, "response": resp}

    @staticmethod
    def _extract_finish_reason(resp):
        try:
            data = resp.json() if hasattr(resp, "json") else resp
            return data["choices"][0].get("finish_reason")
        except Exception:
            return None


def _self_check():
    captured = {}
    class _FakeClient:
        def post(self, url, json=None, data=None, headers=None):
            captured["body"] = json if json is not None else data
            captured["url"] = url
            class _R:
                def json(self_inner):
                    return {"choices": [{"finish_reason": "stop"}],
                            "usage": {"completion_tokens": 1}}
            return _R()

    t = VLLMTransport(client=_FakeClient(), base_url="http://x",
                      coder_base_url="http://y",
                      coder_model_name="qwen3-coder-30b-a3b")
    t(model="qwen3.8-27b", messages=[{"role": "user", "content": "h"}],
      max_tokens=1024, chat_template_kwargs={"thinking_budget": 999},
      reasoning_mode="fast")
    wf = captured["body"]
    assert wf.get("chat_template_kwargs") == {"enable_thinking": False}
    assert "enable_thinking" not in wf
    assert "thinking_token_budget" not in wf
    assert "reasoning_effort" not in wf
    assert "http://x" in captured["url"]

    t(model="qwen3.8-27b", messages=[{"role": "user", "content": "h"}],
      max_tokens=1024, chat_template_kwargs={"enable_thinking": True},
      reasoning_mode="deep")
    wd = captured["body"]
    assert wd["reasoning_effort"] == "medium"
    assert wd["thinking_token_budget"] == 2000
    assert wd["max_tokens"] >= 6144
    assert wd["chat_template_kwargs"].get("enable_thinking") is True   # LA-7
    assert wd["chat_template_kwargs"].get("thinking_budget") == 2000   # LA-7
    assert "enable_thinking" not in wd   # top-level only under v25.1-A

    t(model="qwen3-coder-30b-a3b", messages=[{"role": "user", "content": "h"}],
      max_tokens=256, reasoning_mode="fast")
    assert "http://y" in captured["url"]

    t2 = VLLMTransport(client=_FakeClient(), base_url="http://x",
                       coder_base_url="http://y",
                       coder_model_name="my-custom-coder")
    t2(model="my-custom-coder", messages=[{"role": "user", "content": "h"}],
       max_tokens=256, reasoning_mode="fast")
    assert "http://y" in captured["url"]

    os.environ[V25_1_A_ENV_VAR] = "1"
    t3 = VLLMTransport(client=_FakeClient(), base_url="http://x",
                       coder_base_url="http://y")
    t3(model="qwen3.8-27b", messages=[{"role": "user", "content": "h"}],
       max_tokens=1024, reasoning_mode="deep")
    assert captured["body"].get("enable_thinking") is True
    os.environ.pop(V25_1_A_ENV_VAR, None)

    t(model="qwen3.8-27b", messages=[{"role": "user", "content": "h"}],
      max_tokens=1024, reasoning_mode=None)
    assert captured["body"]["chat_template_kwargs"]["enable_thinking"] is False

    t4 = VLLMTransport("http://r:8000", "http://c:8001")
    assert t4._base_url == "http://r:8000"
    assert t4._coder_base_url == "http://c:8001"
    print("vllm_transport v26.3.4 self-check: 7/7 PASS")


if __name__ == "__main__":
    _self_check()


## GE LAYER (GE1-GE4) — Grid Text-Encoding (Report DIR/092 Rev v38 §5-§13; ARCH/103 + ARCH/120)
Additive. Certified `grid_text.py` v1.0.8 read-only. GE4 wires the CM `_obs_block` + reasoner prompt + telemetry.

In [ ]:
import os
for _d in ('ge1','ge2','ge3','ge4'):
    os.makedirs(f'goa/agent/{_d}', exist_ok=True)
    open(f'goa/agent/{_d}/__init__.py','a').close()
print('GE package dirs ready')

In [ ]:
%%writefile goa/agent/ge1/wire.py
# ============================================================================
# GE1 — Canonical Wire Encoder + Strict Parser/Validator  [v37 — SINGLE FILE]
# Order: GoA/INDUS/DIR/2026/095  |  Basis: Report Rev v38 §5.1a–§5.4
#
# v36 CHANGES (from v35, per Indus real-frame audit, 2026-10-18):
#   - FIX-1: PAIR_BUDGET exceedance re-classified from ENVIRONMENT_CONTRACT_
#     FAILURE to CANDIDATE_INFEASIBLE. A relation-pair budget overrun is a
#     RESOURCE limit, not an instrumentation failure: it must let the caller
#     degrade to a cheaper tier (T2, no relations), NOT set the INVALID flag
#     (Report F-08: "a candidate may be REJECTED as infeasible").
#
# v35 CHANGES (from v34, per Indus v35 top-4 + 2 P3):
#   - P2-2 (MUST): --pin-frames write default moved to OUTPUT_DIR
#     (FRAME_REGISTRY_WRITE_DEFAULT); --out <path> overrides; banner warns
#     if the resolved target lands inside the agent dir. P-1 pollution now
#     fully closed — no code path writes to _WIRE_DIR without explicit opt-in.
#   - P2-1 (MUST): _safe_write fsyncs the temp fd + best-effort fsyncs the
#     parent dir for crash-safe durability.
#   - P2-3 (MUST): edge_manifest_triple_consistency uses explicit FAIL
#     returns (safe under python -O); 0 bare assert statements in the suite.
#   - P2-4 (MUST): independent_parse → parse_structure_only (honest name +
#     docstring noting it shares rle_decode); test renamed
#     ts_parse_structure_oracle with truthful detail string.
#   - P3-7: _load_real_frames_with_count uses next(..., default=None) +
#     explicit "no ARCBaseGame subclass found" error.
#   - P3-8: _load_frame_registry_diag() distinguishes MISSING vs CORRUPT;
#     wired into _check_env_mode, ts_spatial_08, ts_spatial_10.
#
# v34 CHANGES (from v33, per ROUND 21/22 audit — P0 compile + P-1 pollution):
#   - P0 (BLOCKER): "from __future__ import annotations" moved to line 1
#     (v33 placed sys.dont_write_bytecode before it → SyntaxError).
#   - P-1: all runtime artifacts routed to OUTPUT_DIR; sys.dont_write_bytecode
#     = True; _WIRE_DIR stays read-only during a suite run.
#
# v33 CHANGES (from v32): P-1 write isolation; OUTPUT_DIR for artifacts.
#
# v32 CHANGES (from v31): CHANGE 1 (F-03): _CANONICAL_COMPANION_SHA256
#   placeholder + _pin_for() gate. CHANGE 2: integration test env-var gate
#   removed (test now self-contains via temp-dir override).
#
# v31 CHANGES (from v29): F-28 aggregate rollback + symlink-safe writer;
#   F-29 byte-verify all three outputs (S-4 full-row); F-30/S-1 production/
#   injection split; F-31 self-provisioning tests; F-25 hardlink inode check;
#   S-2 non-atomicity documented; S-5 TOCTOU test; S-6 F pin fail-closed.
#
# REQUIRED PROVISIONING (external, not code):
#   - arcengine==0.9.3, transformers, tokenizers, tiktoken
#   - Certified grid_text.py v1.0.8 (37,118 B, md5 bdbc329a…)
#   - FRAME_REGISTRY.json via `python wire.py --pin-frames --out <repo-path>`
#   - _CANONICAL_COMPANION_SHA256 entry from trusted release metadata (F-03)
#   - External release anchor (FINAL-34)
# ============================================================================

from __future__ import annotations

import sys
sys.dont_write_bytecode = True

import re, os, json, glob, hashlib, signal, random, tempfile, shutil, stat
import contextlib
import importlib.util
import importlib.metadata as importlib_metadata
from enum import Enum
from collections import Counter, deque
from typing import List, Tuple, Dict, Optional, Any, Set, Iterable

SPEC_REV = "DIR092 Rev v38"
SPEC_ORDER = "GoA/INDUS/DIR/2026/095"

DIGIT_TO_CHAR = {
    0:'W',1:'w',2:'g',3:'G',4:'c',5:'B',6:'M',7:'P',
    8:'R',9:'b',10:'S',11:'Y',12:'O',13:'r',14:'N',15:'p'}
CHAR_TO_DIGIT = {v:k for k,v in DIGIT_TO_CHAR.items()}
PALETTECHAR = set(DIGIT_TO_CHAR.values())
STATE_ALLOWLIST = {"NOT_PLAYED","NOT_FINISHED","WIN","GAME_OVER"}
RELATION_PRIORITY = ["adjacent","aligned_cols","aligned_rows",
                     "above","left","bbox_contains"]

PAIR_BUDGET = 2_000_000

CERTIFIED_MD5    = "bdbc329a990d82c19bf4b4e22b838bdd"
CERTIFIED_SHA256 = "864fb4051a978f97267c12e71fe9aad55d2a6ec2d77f1536e75722087a0aacca"
CERTIFIED_BYTES  = 37118
TOKENIZER_JSON_SHA256 = "0997f410c57a1f4e53b09e4be8f4a172d90edd9564368fb0847030937229b9f3"
QWEN_REVISION = "366a23fd246985933964e9d21ac63eb3a93190ab"
ARCENGINE_VERSION = "0.9.3"

REPORT_25_MD5_PREFIX = {
    "ar25": "2f2430398", "vc33": "a8bf25fb8", "ft09": "0299caa91",
    "ls20": "48ff7b77c", "sp80": "e775c895c", "re86": "2fd4aeb4e",
    "m0r0": "3043c0363", "bp35": "3468cd2f7", "cd82": "4ddebf4a8",
    "cn04": "c3d665e88", "dc22": "c944b77ca", "g50t": "e86d104ff",
    "ka59": "6aece7358", "lf52": "a2a97949e", "lp85": "df6bcbcd7",
    "r11l": "eb932acf9", "s5i5": "75722fc92", "sb26": "3aa094d6e",
    "sc25": "4b52fd2fe", "sk48": "41f9013cf", "su15": "8a2327bed",
    "tn36": "f106fc38e", "tr87": "e4f8155bb", "tu93": "a64bcabe6",
    "wa30": "b95b75726",
}
CANONICAL_FRAME_IDS: Set[str] = set(REPORT_25_MD5_PREFIX.keys())

_F_EXPECTED_RELATIONS: Dict[str, List[Tuple[int, int]]] = {
    "adjacent":      [],
    "aligned_cols":  [(1, 2)],
    "aligned_rows":  [(1, 2)],
    "above":         [],
    "left":          [],
    "bbox_contains": [(1, 2)],
}

_F_COMPANION_BASENAME = "GoA_DIR092_GoldenFixtures_v36.txt"

_CANONICAL_COMPANION_SHA256: Dict[str, str] = {
    _F_COMPANION_BASENAME: "92d25abf29c0bd69820a6906095e1c68cebeb97d3044aca6b4b160713108fc32",
}

_WIRE_DIR = os.path.dirname(os.path.abspath(__file__))
OUTPUT_DIR = os.environ.get(
    "GE1_OUTPUT_DIR",
    os.path.join(tempfile.gettempdir(), "ge1_output"))

try:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
except OSError as _e:
    print(f"[GE1] WARNING: cannot create OUTPUT_DIR {OUTPUT_DIR}: {_e}",
          file=sys.stderr)

BASE_IDENTITY_PATH      = os.path.join(OUTPUT_DIR, "BASE_IDENTITY.txt")
RELEASE_MANIFEST_PATH   = os.path.join(OUTPUT_DIR, "RELEASE_MANIFEST.txt")
TEST_LOG_PATH           = os.path.join(OUTPUT_DIR, "TEST_LOG.txt")
EVIDENCE_LOG_PATH       = os.path.join(OUTPUT_DIR, "Verification_v39.txt")
VERIFIED_COMPANION_PATH = os.path.join(OUTPUT_DIR, "VerifiedCompanion_v36.txt")

FRAME_REGISTRY_JSON = os.environ.get(
    "GE1_FRAME_REGISTRY",
    os.path.join(_WIRE_DIR, "FRAME_REGISTRY.json"))

# P2-2 (v35): --pin-frames WRITE default never lands in _WIRE_DIR unless
# explicitly overridden. Mirrors P-1's write-isolation policy.
FRAME_REGISTRY_WRITE_DEFAULT = os.environ.get(
    "GE1_FRAME_REGISTRY_OUT",
    os.path.join(OUTPUT_DIR, "FRAME_REGISTRY.json"))

_FALLBACK_GRID_TEXT = os.path.join(_WIRE_DIR, "grid_text.py")

_CANONICAL_FIXTURE_PATHS = [
    os.path.join(_WIRE_DIR, _F_COMPANION_BASENAME),
    os.path.join(_WIRE_DIR, "..", _F_COMPANION_BASENAME),
    os.path.join(_WIRE_DIR, "..", "..", _F_COMPANION_BASENAME),
    os.path.join(_WIRE_DIR, "..", "..", "..", _F_COMPANION_BASENAME),
]
_REPO_ROOTS = [
    _WIRE_DIR,
    os.path.abspath(os.path.join(_WIRE_DIR, "..")),
    os.path.abspath(os.path.join(_WIRE_DIR, "..", "..")),
    os.path.abspath(os.path.join(_WIRE_DIR, "..", "..", "..")),
]
_CERTIFIED_SEARCH_PATHS = [
    os.path.join(_WIRE_DIR, "results", "goa", "agent", "grid_text.py"),
    os.path.join(_WIRE_DIR, "..", "results", "goa", "agent", "grid_text.py"),
    os.path.join(_WIRE_DIR, "..", "..", "results", "goa", "agent", "grid_text.py"),
    _FALLBACK_GRID_TEXT,
]

_LEGACY_ARTIFACT_BASENAMES = [
    "BASE_IDENTITY.txt",
    "RELEASE_MANIFEST.txt",
    "RELEASE_MANIFEST.txt.json",
    "RELEASE_MANIFEST.txt.sha256",
    "TEST_LOG.txt",
    "Verification_v39.txt",
    "FRAME_REGISTRY.json",
    "VerifiedCompanion_v36.txt",
]

_TS12_CACHE: Dict[str, Any] = {"result": None}


class ReasonCode:
    UNSUPPORTED_INPUT            = "UNSUPPORTED_INPUT"
    ENVIRONMENT_CONTRACT_FAILURE = "ENVIRONMENT_CONTRACT_FAILURE"
    MALFORMED_SCHEMA             = "MALFORMED_SCHEMA"
    NONCANONICAL                 = "NONCANONICAL"
    CANDIDATE_INFEASIBLE         = "CANDIDATE_INFEASIBLE"  # resource/budget: skip this candidate; do NOT set INVALID


class ReasonCodeError(ValueError):
    def __init__(self, code: str, msg: str):
        self.code = code
        super().__init__(f"{code}: {msg}")


class Status(str, Enum):
    PASS    = "PASS"
    FAIL    = "FAIL"
    BLOCKED = "BLOCKED"


_EVIDENCE: List[Dict[str, Any]] = []
def _evidence(rec): _EVIDENCE.append(rec)


def _pin_for(basename: str) -> Optional[str]:
    p = _CANONICAL_COMPANION_SHA256.get(basename)
    if not isinstance(p, str) or not re.fullmatch(r"[0-9a-f]{64}", p):
        return None
    return p


def _F_is_provisioned() -> bool:
    return _pin_for(_F_COMPANION_BASENAME) is not None


def _safe_read_staged(path):
    try:
        lst = os.lstat(path)
    except FileNotFoundError:
        raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                              f"staged missing: {path}")
    except OSError as e:
        raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                              f"lstat failed: {path}: {e}")
    if not stat.S_ISREG(lst.st_mode):
        raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                              f"staged is not a regular file (symlink/dir/fifo/etc): {path}")
    flags = os.O_RDONLY
    if hasattr(os, "O_NOFOLLOW"): flags |= os.O_NOFOLLOW
    if hasattr(os, "O_NONBLOCK"): flags |= os.O_NONBLOCK
    try:
        fd = os.open(path, flags)
    except OSError as e:
        raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                              f"open failed: {path}: {e}")
    try:
        fst = os.fstat(fd)
        if not stat.S_ISREG(fst.st_mode):
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  f"opened descriptor is not regular: {path}")
        if (fst.st_dev, fst.st_ino) != (lst.st_dev, lst.st_ino):
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  f"staged file changed between lstat and open: {path}")
    except ReasonCodeError:
        try: os.close(fd)
        except Exception: pass
        raise
    except Exception as e:
        try: os.close(fd)
        except Exception: pass
        raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                              f"fstat failed: {path}: {e}")
    with os.fdopen(fd, "rb") as f:
        return f.read()


def _safe_write(path, data):
    """F-20 (v35): atomic safe write with fsync durability (P2-1)."""
    if isinstance(data, str):
        data = data.encode("utf-8")
    try:
        lst = os.lstat(path)
        if stat.S_ISLNK(lst.st_mode):
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  f"refusing to write over symlink: {path}")
        if not stat.S_ISREG(lst.st_mode):
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  f"refusing to overwrite non-regular: {path}")
    except FileNotFoundError:
        pass
    except OSError as e:
        raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                              f"lstat failed on target: {path}: {e}")
    parent = os.path.dirname(os.path.abspath(path)) or "."
    if not os.path.isdir(parent):
        raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                              f"destination directory missing: {parent}")
    fd, tmp = tempfile.mkstemp(dir=parent, prefix=".gw_")
    try:
        with os.fdopen(fd, "wb") as f:
            f.write(data)
            f.flush()
            # P2-1 (v35): durable write — fsync temp fd before rename.
            try: os.fsync(f.fileno())
            except OSError: pass
        with open(tmp, "rb") as f:
            if f.read() != data:
                raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                      f"temp write mismatch: {path}")
        os.replace(tmp, path)
        # P2-1 (v35): best-effort fsync on parent dir so the rename is durable.
        try:
            dfd = os.open(parent, os.O_RDONLY)
            try: os.fsync(dfd)
            finally: os.close(dfd)
        except OSError:
            pass
        tmp = None
    finally:
        if tmp is not None:
            try: os.unlink(tmp)
            except Exception: pass


def _load_frame_registry_diag() -> Tuple[Dict[str, str], Optional[str]]:
    """P3-8 (v35): distinguish absent from corrupt.

    Returns (registry, error):
      * ({}, None)      → file absent (not provisioned)
      * ({}, "<why>")   → present but unreadable / invalid JSON / not-a-dict
      * ({...}, None)   → OK
    """
    if not os.path.exists(FRAME_REGISTRY_JSON):
        return {}, None
    try:
        raw = _safe_read_staged(FRAME_REGISTRY_JSON)
    except ReasonCodeError as e:
        return {}, f"unreadable: {e}"
    try:
        data = json.loads(raw.decode("utf-8"))
    except (json.JSONDecodeError, UnicodeDecodeError) as e:
        return {}, f"invalid JSON/UTF-8: {e}"
    if not isinstance(data, dict):
        return {}, f"root is {type(data).__name__}, not dict"
    return {str(k): str(v) for k, v in data.items()}, None


def _load_frame_registry() -> Dict[str, str]:
    """Back-compat wrapper — returns empty dict on any error.
    Callers that need MISSING vs CORRUPT distinction use _load_frame_registry_diag()."""
    return _load_frame_registry_diag()[0]


def _find_canonical_fixture_file() -> Optional[str]:
    for p in _CANONICAL_FIXTURE_PATHS:
        if os.path.exists(p):
            return os.path.realpath(p)
    return None


def _parse_canonical_fixture_content(content: str) -> Optional[List[List[int]]]:
    if not isinstance(content, str):
        return None
    m = re.search(r"(?:^|\n)FIXTURE\s+F[^\n]*\n(.*?)(?=\nFIXTURE\s|\Z)",
                  content, re.DOTALL)
    if not m:
        return None
    block = m.group(1)
    grid_rows = []
    for line in block.split("\n"):
        s = line.strip()
        if re.fullmatch(r"[WwgGcBMPRbSYOrNp]{64}", s):
            grid_rows.append([CHAR_TO_DIGIT[ch] for ch in s])
    if len(grid_rows) == 64:
        return grid_rows
    cells_m = re.search(r"CELLS:\s*\n((?:\s*\d+\s*,\s*\d+\s*,\s*\d+.*\n?)+)", block)
    if cells_m:
        g = [[0]*64 for _ in range(64)]
        seen: Set[Tuple[int,int]] = set()
        any_cell = False
        for ln in cells_m.group(1).strip().split("\n"):
            parts = [p.strip() for p in ln.split(",")]
            if len(parts) != 3:
                return None
            try:
                r, c, v = int(parts[0]), int(parts[1]), int(parts[2])
            except ValueError:
                return None
            if not (0 <= r < 64 and 0 <= c < 64 and 0 <= v < 16):
                return None
            if (r, c) in seen:
                return None
            seen.add((r, c))
            g[r][c] = v; any_cell = True
        if any_cell:
            return g
    return None


def _emit_evidence_log(path: str = None) -> None:
    if path is None:
        path = EVIDENCE_LOG_PATH
    abs_path = os.path.abspath(path)
    rp = os.path.realpath(abs_path)
    allowed = {os.path.realpath(os.path.abspath(EVIDENCE_LOG_PATH))}

    if rp in set(_protected_input_paths()) and rp not in allowed:
        raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                              f"evidence-log destination collides with "
                              f"protected input: {path}")

    try:
        lst = os.lstat(abs_path)
    except FileNotFoundError:
        lst = None
    if lst is not None and stat.S_ISREG(lst.st_mode):
        prot_ids: Set[Tuple[int,int]] = set()
        for p in _protected_input_paths():
            if os.path.realpath(p) in allowed:
                continue
            try:
                plst = os.lstat(p)
                if stat.S_ISREG(plst.st_mode):
                    prot_ids.add((plst.st_dev, plst.st_ino))
            except OSError:
                pass
        if (lst.st_dev, lst.st_ino) in prot_ids:
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  f"evidence-log target is a hardlink to a "
                                  f"protected input: {path}")

    lines = ["# GE1 Verification Evidence Log",
             f"# spec={SPEC_REV}", f"# order={SPEC_ORDER}"]
    for r in _EVIDENCE:
        lines.append(json.dumps(r, sort_keys=True))
    payload = "\n".join(lines) + "\n"
    _safe_write(path, payload)
    with open(path, "r") as f: back = f.read()
    if back != payload:
        raise IOError(f"evidence-log read-back mismatch at {path}")


def rle_encode(row: Any) -> str:
    if not isinstance(row, list) or len(row) != 64:
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "rle_encode expects list of 64")
    for i, v in enumerate(row):
        if type(v) is not int or not (0 <= v <= 15):
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA,
                                  f"cell {i}: {type(v).__name__} {v!r}")
    runs, cur, count = [], row[0], 1
    for v in row[1:]:
        if v == cur: count += 1
        else: runs.append((cur, count)); cur, count = v, 1
    runs.append((cur, count))
    return ",".join(f"{DIGIT_TO_CHAR[d]}{c}" for d, c in runs)


def rle_decode(s: Any) -> List[int]:
    if type(s) is not str:
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "rle_decode expects str")
    if not s:
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "empty RLE")
    parts = s.split(","); out = []; total = 0
    for part in parts:
        m = re.fullmatch(r"([WwgGcBMPRbSYOrNp])([1-9][0-9]*)", part)
        if not m:
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, f"bad RUN {part!r}")
        ch = m.group(1); cnt_str = m.group(2)
        if ch not in CHAR_TO_DIGIT:
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "unknown palette")
        if len(cnt_str) > 8:
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "RLE count too large")
        cnt = int(cnt_str); total += cnt
        if total > 64:
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "RLE sum > 64")
        out.extend([CHAR_TO_DIGIT[ch]] * cnt)
    if total != 64:
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "RLE sum != 64")
    for i in range(1, len(parts)):
        if parts[i][0] == parts[i-1][0]:
            raise ReasonCodeError(ReasonCode.NONCANONICAL, "adjacent same-char runs")
    return out


def _validate_grid(grid: Any) -> None:
    if not isinstance(grid, list) or len(grid) != 64:
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "grid must have 64 rows")
    for r, row in enumerate(grid):
        if not isinstance(row, list) or len(row) != 64:
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA,
                                  f"row {r} must have 64 cells")
        for c, v in enumerate(row):
            if type(v) is not int or not (0 <= v <= 15):
                raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                      f"cell ({r},{c}) must be int 0..15, got "
                                      f"{type(v).__name__} {v!r}")


def _validate_actionlist(actions: Any) -> None:
    if not isinstance(actions, list):
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "actions must be list")
    for a in actions:
        if type(a) is not int or a < 0:
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, f"action int>=0, got {a!r}")
    if actions != sorted(set(actions)):
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "ACTIONLIST ascending & unique")


def _validate_bg(b):
    if type(b) is not int or not (0 <= b <= 15):
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "background int 0..15")


def make_header(levels_completed, state, actions, background) -> str:
    if type(levels_completed) is not int or levels_completed < 0:
        raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                              "levels_completed required (int>=0)")
    if type(state) is not str or state not in STATE_ALLOWLIST:
        raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                              "state must be canonical string")
    _validate_actionlist(actions); _validate_bg(background)
    act_str = ", ".join(str(a) for a in actions)
    return (f"levels_completed={levels_completed}\n"
            f"state={state}\n"
            f"available_actions=[{act_str}]\n"
            f"background={DIGIT_TO_CHAR[background]}")


def encode_grid_full(grid: Any) -> str:
    _validate_grid(grid)
    return "GRID_FULL:\n" + "\n".join(
        f"  {r:02d}|{rle_encode(grid[r])}" for r in range(64))


def encode_grid_rowdict(grid: Any) -> str:
    _validate_grid(grid)
    seen = {}; uniq = []
    for r in range(64):
        key = tuple(grid[r])
        if key not in seen:
            seen[key] = len(uniq); uniq.append(grid[r])
    U = len(uniq)
    if not (1 <= U <= 64):
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "U out of range")
    rowdefs = [f"  R{i:02d}|{rle_encode(uniq[i])}" for i in range(U)]
    map_str = ",".join(str(seen[tuple(grid[r])]) for r in range(64))
    return "GRID_ROWDICT:\n" + "\n".join(rowdefs) + f"\n  MAP: {map_str}"


def _modal_digit(grid):
    _validate_grid(grid)
    cnt = Counter()
    for row in grid: cnt.update(row)
    mx = max(cnt.values())
    return min(d for d, c in cnt.items() if c == mx)


def _indep_modal(grid):
    counts = {}
    for row in grid:
        for v in row: counts[v] = counts.get(v, 0) + 1
    best, bc = None, -1
    for k in sorted(counts.keys()):
        if counts[k] > bc:
            bc = counts[k]; best = k
    return best


def extract_objects(grid: Any, background: Any) -> List[Dict[str, Any]]:
    _validate_grid(grid); _validate_bg(background)
    visited = [[False]*64 for _ in range(64)]
    objs = []
    for r in range(64):
        for c in range(64):
            if visited[r][c]: continue
            d = grid[r][c]
            if d == background:
                visited[r][c] = True; continue
            q = deque([(r, c)]); visited[r][c] = True
            cells = []
            while q:
                cr, cc = q.popleft(); cells.append((cr, cc))
                for dr, dc in ((1,0),(-1,0),(0,1),(0,-1)):
                    nr, nc = cr+dr, cc+dc
                    if (0 <= nr < 64 and 0 <= nc < 64
                            and not visited[nr][nc] and grid[nr][nc] == d):
                        visited[nr][nc] = True; q.append((nr, nc))
            r0 = min(x[0] for x in cells); r1 = max(x[0] for x in cells)
            c0 = min(x[1] for x in cells); c1 = max(x[1] for x in cells)
            fc = min(64*rr+cc for rr,cc in cells)
            objs.append({"colour":d, "cells":cells,
                         "r0":r0,"r1":r1,"c0":c0,"c1":c1,
                         "size":len(cells), "first_cell":fc})
    objs.sort(key=lambda o: (o["r0"], o["c0"], o["first_cell"]))
    for i, o in enumerate(objs, 1): o["id"] = i
    return objs


_OBJLINE_RE = re.compile(
    r"^  #([1-9][0-9]*) ([WwgGcBMPRbSYOrNp]) "
    r"n=([1-9][0-9]*) r(0|[1-9][0-9]*)-(0|[1-9][0-9]*)c(0|[1-9][0-9]*)-(0|[1-9][0-9]*)$"
)


def serialize_objects(objs):
    if not objs: return "OBJECTS:"
    lines = [f"  #{o['id']} {DIGIT_TO_CHAR[o['colour']]} n={o['size']} "
             f"r{o['r0']}-{o['r1']}c{o['c0']}-{o['c1']}" for o in objs]
    return "OBJECTS:\n" + "\n".join(lines)


def encode_objects(grid, background) -> str:
    _validate_grid(grid); _validate_bg(background)
    return serialize_objects(extract_objects(grid, background))


def parse_objects_block(text: str, background=None, expected_n=None, grid=None):
    if type(text) is not str:
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "OBJECTS text must be str")
    if "\r" in text:
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "CR in OBJECTS")
    lines = text.split("\n")
    if not lines or lines[0] != "OBJECTS:":
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "missing OBJECTS: header")
    objs = []; seen_ids = set()
    for i, line in enumerate(lines[1:], start=1):
        if line == "":
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "blank line in OBJECTS")
        m = _OBJLINE_RE.fullmatch(line)
        if not m:
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, f"bad OBJLINE {line!r}")
        oid = int(m.group(1)); ch = m.group(2); size = int(m.group(3))
        r0 = int(m.group(4)); r1 = int(m.group(5))
        c0 = int(m.group(6)); c1 = int(m.group(7))
        if oid != i:
            raise ReasonCodeError(ReasonCode.NONCANONICAL,
                                  f"OBJID sequence: got {oid} expected {i}")
        if oid in seen_ids:
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, f"dup OBJID {oid}")
        seen_ids.add(oid)
        if not (0 <= r0 <= r1 <= 63 and 0 <= c0 <= c1 <= 63):
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, f"bbox bad in OBJID {oid}")
        if size < 1:
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, f"size<1 in OBJID {oid}")
        colour = CHAR_TO_DIGIT.get(ch)
        if colour is None:
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, f"unknown palette {ch!r}")
        if background is not None and colour == background:
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA,
                                  f"OBJID {oid}: colour == background")
        objs.append({"id":oid, "colour":colour, "size":size,
                     "r0":r0,"r1":r1,"c0":c0,"c1":c1,"char":ch})
    if expected_n is not None and len(objs) != expected_n:
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA,
                              f"OBJECTS count {len(objs)} != {expected_n}")
    if grid is not None and background is not None:
        actual = extract_objects(grid, background)
        if len(actual) != len(objs):
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA,
                                  "OBJECTS count != grid components")
        for idx, (o, a) in enumerate(zip(objs, actual), 1):
            if (o["colour"], o["size"], o["r0"], o["r1"], o["c0"], o["c1"]) != \
               (a["colour"], a["size"], a["r0"], a["r1"], a["c0"], a["c1"]):
                raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA,
                                      f"OBJID {idx}: record != grid component")
    return objs


def _coarse16_rows(grid, background):
    lines = []
    for tr in range(16):
        chars = []
        for tc in range(16):
            cnt = Counter()
            for r in range(tr*4, tr*4+4):
                for c in range(tc*4, tc*4+4):
                    d = grid[r][c]
                    if d != background: cnt[d] += 1
            if not cnt: chars.append(DIGIT_TO_CHAR[background])
            else:
                mx = max(cnt.values())
                chars.append(DIGIT_TO_CHAR[min(d for d, v in cnt.items() if v == mx)])
        lines.append("".join(chars))
    return lines


def encode_coarse16(grid, background) -> str:
    _validate_grid(grid); _validate_bg(background)
    return "COARSE16:\n" + "\n".join(_coarse16_rows(grid, background))


_TILEROW_RE = re.compile(r"^[WwgGcBMPRbSYOrNp]{16}$")


def parse_coarse16_block(text, grid=None, background=None):
    if type(text) is not str:
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "COARSE16 text must be str")
    if "\r" in text:
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "CR in COARSE16")
    lines = text.split("\n")
    if not lines or lines[0] != "COARSE16:":
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "missing COARSE16: header")
    body = lines[1:]
    if len(body) != 16:
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA,
                              f"COARSE16 must have 16 rows, got {len(body)}")
    for i, row in enumerate(body):
        if not _TILEROW_RE.fullmatch(row):
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, f"COARSE16 row {i}: bad")
    if grid is not None and background is not None:
        _validate_grid(grid); _validate_bg(background)
        if body != _coarse16_rows(grid, background):
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA,
                                  "COARSE16 content != grid aggregation")
    return body


def _bbox_may_adjacent(a, b):
    if a["r0"] > b["r1"]+1 or a["r1"] < b["r0"]-1: return False
    if a["c0"] > b["c1"]+1 or a["c1"] < b["c0"]-1: return False
    return True


def _exact_adjacent(a, b):
    sa = set(a["cells"])
    for r, c in b["cells"]:
        for dr, dc in ((1,0),(-1,0),(0,1),(0,-1)):
            if (r+dr, c+dc) in sa: return True
    return False


def _aligned_cols(a, b):
    return not (a["c1"] < b["c0"] or b["c1"] < a["c0"])


def _aligned_rows(a, b):
    return not (a["r1"] < b["r0"] or b["r1"] < a["r0"])


def _above(a, b):
    return a["r1"] < b["r0"] and _aligned_cols(a, b)


def _left(a, b):
    return a["c1"] < b["c0"] and _aligned_rows(a, b)


def _bbox_contains(a, b):
    return (a["r0"] <= b["r0"] and a["r1"] >= b["r1"] and
            a["c0"] <= b["c0"] and a["c1"] >= b["c1"] and
            (a["r0"] < b["r0"] or a["r1"] > b["r1"] or
             a["c0"] < b["c0"] or a["c1"] > b["c1"]))


def _is_4_connected(cells):
    if not cells: return False
    cs = set(cells); start = cells[0]
    seen = {start}; stack = [start]
    while stack:
        r, c = stack.pop()
        for dr, dc in ((1,0),(-1,0),(0,1),(0,-1)):
            nb = (r+dr, c+dc)
            if nb in cs and nb not in seen:
                seen.add(nb); stack.append(nb)
    return len(seen) == len(cs)


def _validate_object_record(o, expected_id):
    if not isinstance(o, dict):
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "object must be dict")
    for key in ("id","colour","cells","size","r0","r1","c0","c1"):
        if key not in o:
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, f"missing '{key}'")
    if type(o["id"]) is not int or o["id"] != expected_id:
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, f"id != {expected_id}")
    if type(o["colour"]) is not int or not (0 <= o["colour"] <= 15):
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "bad colour")
    for k in ("r0","r1","c0","c1","size"):
        if type(o[k]) is not int:
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, f"{k} must be int")
    if not (0 <= o["r0"] <= o["r1"] <= 63 and 0 <= o["c0"] <= o["c1"] <= 63):
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "bbox out of range/unordered")
    if not isinstance(o["cells"], list) or not o["cells"]:
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "cells must be non-empty")
    cs = set()
    for cell in o["cells"]:
        if (not isinstance(cell, tuple) or len(cell) != 2
                or type(cell[0]) is not int or type(cell[1]) is not int):
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "bad cell")
        r, c = cell
        if not (0 <= r <= 63 and 0 <= c <= 63):
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "cell out of grid")
        if not (o["r0"] <= r <= o["r1"] and o["c0"] <= c <= o["c1"]):
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "cell out of bbox")
        if cell in cs:
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, f"duplicate cell {cell!r}")
        cs.add(cell)
    if o["size"] != len(o["cells"]):
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "size != len(cells)")
    rs = [c[0] for c in o["cells"]]; cs_ = [c[1] for c in o["cells"]]
    if o["r0"] != min(rs) or o["r1"] != max(rs):
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "bbox r-extrema != cell r-extrema")
    if o["c0"] != min(cs_) or o["c1"] != max(cs_):
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "bbox c-extrema != cell c-extrema")
    if not _is_4_connected(o["cells"]):
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "cells are not 4-connected")


def _validate_object_list_invariants(objects):
    cell_to_obj = {}
    for o in objects:
        for cell in o["cells"]:
            cell_to_obj[cell] = o
    for o in objects:
        for r, c in o["cells"]:
            for dr, dc in ((1,0),(-1,0),(0,1),(0,-1)):
                nb = (r+dr, c+dc)
                if nb in cell_to_obj:
                    other = cell_to_obj[nb]
                    if other["id"] != o["id"] and other["colour"] == o["colour"]:
                        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA,
                                              f"same-colour objects {o['id']} and {other['id']} "
                                              f"share a 4-neighbour edge at {nb!r}")

    def _true_first_cell(o):
        return min(64*r + c for r, c in o["cells"])

    def _check_and_key(o):
        fc_true = _true_first_cell(o)
        if "first_cell" in o:
            fc_stored = o["first_cell"]
            if type(fc_stored) is not int or fc_stored != fc_true:
                raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA,
                                      f"object {o['id']}: first_cell={fc_stored!r} != "
                                      f"computed {fc_true}")
        return (o["r0"], o["c0"], fc_true)

    keys = [_check_and_key(o) for o in objects]
    for i in range(1, len(keys)):
        if keys[i] < keys[i-1]:
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA,
                                  f"objects not in canonical (r0,c0,first_cell) order")


def _encode_relations_prefix_unchecked(objects, k) -> str:
    if not isinstance(objects, list):
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "objects must be list")
    if type(k) is not int:
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "k must be int")
    if not (1 <= k <= 6):
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "k out of 1..6")
    for idx, o in enumerate(objects, 1):
        _validate_object_record(o, idx)
    seen_cells = {}
    for o in objects:
        for cell in o["cells"]:
            if cell in seen_cells:
                raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA,
                                      f"cell {cell!r} shared by objects "
                                      f"{seen_cells[cell]} and {o['id']}")
            seen_cells[cell] = o["id"]
    n = len(objects)
    if n * (n - 1) // 2 > PAIR_BUDGET:
        # Resource limit, NOT an environment/instrumentation failure: signal the
        # candidate as INFEASIBLE so the caller degrades to a cheaper tier (T2,
        # no relations) instead of setting the INVALID blocking flag. Report
        # F-08: "a candidate may be REJECTED as infeasible". (v36 fix.)
        raise ReasonCodeError(ReasonCode.CANDIDATE_INFEASIBLE,
                              f"relation pair budget exceeded: {n}")
    rel = {name: [] for name in RELATION_PRIORITY}
    for i in range(n):
        a = objects[i]
        for j in range(i + 1, n):
            b = objects[j]
            ia, ib = a["id"], b["id"]
            if _bbox_may_adjacent(a, b) and _exact_adjacent(a, b):
                rel["adjacent"].append((ia, ib))
            if _aligned_cols(a, b): rel["aligned_cols"].append((ia, ib))
            if _aligned_rows(a, b): rel["aligned_rows"].append((ia, ib))
            if _above(a, b): rel["above"].append((ia, ib))
            if _above(b, a): rel["above"].append((ib, ia))
            if _left(a, b):  rel["left"].append((ia, ib))
            if _left(b, a):  rel["left"].append((ib, ia))
            if _bbox_contains(a, b): rel["bbox_contains"].append((ia, ib))
            if _bbox_contains(b, a): rel["bbox_contains"].append((ib, ia))
    lines = []
    for name in RELATION_PRIORITY[:k]:
        lst = sorted(set(rel[name]), key=lambda x: (x[0], x[1]))
        lines.append(f"  {name}=-" if not lst else
                     f"  {name}=" + ",".join(f"{s}-{t}" for s, t in lst))
    return "RELATIONS:\n" + "\n".join(lines)


def encode_relations_prefix(objects, k) -> str:
    if not isinstance(objects, list):
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "objects must be list")
    if type(k) is not int:
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "k must be int")
    if not (1 <= k <= 6):
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "k out of 1..6")
    for idx, o in enumerate(objects, 1):
        _validate_object_record(o, idx)
    seen_cells = {}
    for o in objects:
        for cell in o["cells"]:
            if cell in seen_cells:
                raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA,
                                      f"cell {cell!r} shared by objects "
                                      f"{seen_cells[cell]} and {o['id']}")
            seen_cells[cell] = o["id"]
    _validate_object_list_invariants(objects)
    return _encode_relations_prefix_unchecked(objects, k)


def encode(grid, tier, *, levels_completed, state, actions, k=0) -> str:
    _validate_grid(grid)
    bg = _modal_digit(grid)
    header = make_header(levels_completed, state, actions, bg)
    if tier == "BASELINE":
        body = encode_grid_full(grid)
    elif tier == "T1":
        body = encode_grid_rowdict(grid)
    elif tier == "T2":
        body = (encode_grid_rowdict(grid) + "\n"
                + encode_objects(grid, bg) + "\n"
                + encode_coarse16(grid, bg))
    elif tier == "T3":
        if type(k) is not int or not (1 <= k <= 6):
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "T3 k 1..6 (int)")
        objs = extract_objects(grid, bg)
        body = (encode_grid_rowdict(grid) + "\n"
                + encode_objects(grid, bg) + "\n"
                + encode_coarse16(grid, bg) + "\n"
                + _encode_relations_prefix_unchecked(objs, k))
    else:
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, f"unknown tier {tier!r}")
    return header + "\n" + body


def _strict_int(s):
    if not isinstance(s, str) or not re.fullmatch(r"[0-9]+", s):
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, f"malformed int {s!r}")
    if len(s) > 1000 or not re.fullmatch(r"0|[1-9][0-9]*", s):
        raise ReasonCodeError(ReasonCode.NONCANONICAL, f"noncanonical int {s!r}")
    return int(s)


def parse(text: str) -> Dict[str, Any]:
    try:
        lines = text.split("\n")
        if len(lines) < 5:
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "too short")
        if not lines[0].startswith("levels_completed="):
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "L1")
        levels = _strict_int(lines[0].split("=", 1)[1])

        if not lines[1].startswith("state="):
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "L2")
        state = lines[1].split("=", 1)[1]
        if state not in STATE_ALLOWLIST:
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE, "state")

        if not (lines[2].startswith("available_actions=[") and lines[2].endswith("]")):
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "L3")
        act_part = lines[2][len("available_actions=["):-1]
        if act_part == "":
            actions = []
        else:
            if not re.fullmatch(r"[0-9]+(,[ \t]*[0-9]+)*", act_part):
                raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "L3 malformed")
            if not re.fullmatch(r"(0|[1-9][0-9]*)(, (0|[1-9][0-9]*))*", act_part):
                raise ReasonCodeError(ReasonCode.NONCANONICAL, "L3 noncanonical")
            actions = [int(x) for x in act_part.split(", ")]
            if actions != sorted(set(actions)):
                raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA,
                                      "ACTIONLIST ascending & unique")

        if not lines[3].startswith("background="):
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "L4")
        bg_char = lines[3].split("=", 1)[1]
        if bg_char not in PALETTECHAR:
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "background char")
        bg_digit = CHAR_TO_DIGIT[bg_char]

        idx = 4; tier = None; k_val = 0

        if lines[idx] == "GRID_FULL:":
            tier, k_val = "BASELINE", 0
            if idx + 1 + 64 != len(lines):
                raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "GRID_FULL line count")
            grid = []
            for r in range(64):
                m = re.fullmatch(r"  ([0-9]{2})\|(.+)", lines[idx+1+r])
                if not m:
                    raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "ROWLINE")
                if int(m.group(1)) != r:
                    raise ReasonCodeError(ReasonCode.NONCANONICAL, "row order")
                grid.append(rle_decode(m.group(2)))
        elif lines[idx] == "GRID_ROWDICT:":
            rowdefs = []; map_vals = None; j = idx + 1
            while j < len(lines):
                line = lines[j]
                if line.startswith("  MAP: "):
                    map_vals = [_strict_int(x)
                                for x in line.split(": ", 1)[1].split(",")]
                    break
                m = re.fullmatch(r"  R([0-9]{2})\|(.+)", line)
                if not m:
                    raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "ROWDEF")
                if int(m.group(1)) != len(rowdefs):
                    raise ReasonCodeError(ReasonCode.NONCANONICAL, "ROWDEF label")
                rowdefs.append(rle_decode(m.group(2))); j += 1
            if map_vals is None:
                raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "MAP missing")
            if len(map_vals) != 64:
                raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "MAP len")
            if len({tuple(r) for r in rowdefs}) != len(rowdefs):
                raise ReasonCodeError(ReasonCode.NONCANONICAL, "duplicate decoded ROWDEF")
            U = len(rowdefs)
            if not (1 <= U <= 64):
                raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "U range")
            if any(v < 0 or v >= U for v in map_vals):
                raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "MAP index")
            seen_order = []
            for r in range(64):
                if map_vals[r] not in seen_order: seen_order.append(map_vals[r])
            if seen_order != list(range(U)):
                raise ReasonCodeError(ReasonCode.NONCANONICAL, "first-use order")
            grid = [rowdefs[map_vals[r]] for r in range(64)]
            rest = j + 1
            if rest == len(lines):
                tier, k_val = "T1", 0
            elif lines[rest] == "OBJECTS:" or lines[rest].startswith("  #"):
                r2 = rest + 1
                while r2 < len(lines) and lines[r2] not in ("COARSE16:", "RELATIONS:"):
                    r2 += 1
                if r2 >= len(lines) or lines[r2] != "COARSE16:":
                    raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "no COARSE16")
                objs = parse_objects_block("\n".join(lines[rest:r2]),
                                            background=bg_digit, grid=grid)
                r3 = r2 + 17
                if r3 > len(lines):
                    raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "COARSE16 short")
                parse_coarse16_block("\n".join(lines[r2:r3]),
                                      grid=grid, background=bg_digit)
                if r3 == len(lines):
                    tier, k_val = "T2", 0
                elif lines[r3] == "RELATIONS:":
                    tier = "T3"; rel_lines = lines[r3:]
                    k_val = len(rel_lines) - 1
                    if not (1 <= k_val <= 6):
                        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "k range")
                    _parse_relations_block("\n".join(rel_lines),
                                            expected_k=k_val, max_endpoint=len(objs))
                else:
                    raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA,
                                          "trailing lines after COARSE16")
            else:
                raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "after rowdict")
        else:
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "unknown block")

        if bg_digit != _modal_digit(grid):
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "background != modal digit")

        try:
            canonical = encode(grid, tier=tier, k=k_val, state=state,
                               actions=actions, levels_completed=levels)
        except ReasonCodeError:
            raise
        except Exception as e:
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA,
                                  f"re-encode failed: {e}") from e
        if text != canonical:
            raise ReasonCodeError(ReasonCode.NONCANONICAL, "not canonical")
        return {"tier": tier, "grid": grid, "background": bg_digit,
                "state": state, "actions": actions, "levels_completed": levels}
    except ReasonCodeError:
        raise
    except Exception as e:
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA,
                              f"{type(e).__name__}: {e}") from e


def parse_structure_only(text: str) -> Dict[str, Any]:
    """P2-4 (v35): structural round-trip, NOT an independent oracle.

    Deliberately reuses rle_decode() — the goal is to catch block-walk
    inconsistencies (MAP order, first-use order, tier detection), not to
    validate RLE decoding via a second implementation. For a truly
    independent decoder see future work in Report Rev v39+.
    """
    lines = text.split("\n")
    if len(lines) < 5 or "\r" in text:
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "bad input")
    levels = int(lines[0].split("=", 1)[1])
    state = lines[1].split("=", 1)[1]
    act = lines[2].split("[", 1)[1].rsplit("]", 1)[0]
    actions = [] if act.strip() == "" else [int(x.strip()) for x in act.split(",")]
    bg = CHAR_TO_DIGIT[lines[3].split("=", 1)[1]]
    if lines[4] == "GRID_FULL:":
        grid = [rle_decode(re.fullmatch(r"  [0-9]{2}\|(.+)", lines[5+r]).group(1))
                for r in range(64)]
        tier, k = "BASELINE", 0
    elif lines[4] == "GRID_ROWDICT:":
        rowdefs = []; j = 5
        while not lines[j].startswith("  MAP: "):
            rowdefs.append(rle_decode(lines[j].split("|", 1)[1])); j += 1
        map_vals = [int(x) for x in lines[j].split(": ", 1)[1].split(",")]
        grid = [rowdefs[map_vals[r]] for r in range(64)]
        tail = "\n".join(lines[j+1:])
        if "RELATIONS:" in tail:
            tier = "T3"
            rr = next(i for i, ln in enumerate(lines) if ln == "RELATIONS:")
            k = len(lines) - 1 - rr
        elif "COARSE16:" in tail:
            tier, k = "T2", 0
        else:
            tier, k = "T1", 0
    else:
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "unknown block")
    return {"tier": tier, "grid": grid, "background": bg,
            "state": state, "actions": actions, "levels_completed": levels}


def reject(text: str) -> Optional[str]:
    try:
        parse(text); return None
    except ReasonCodeError as e: return e.code
    except Exception: return ReasonCode.MALFORMED_SCHEMA


GF_ACTIONS = [1, 2, 3, 4, 5, 6]

def make_gf01(): return [[0]*64 for _ in range(64)]
def make_gf02():
    g = [[0]*64 for _ in range(64)]; g[10][10] = 8; g[10][11] = 1; return g
def _blank(): return [[0]*64 for _ in range(64)]

def fixture_A():
    g = _blank(); g[0][2] = 8; g[4][2] = 8; return g
def fixture_B():
    g = _blank(); g[1][0] = 8; g[1][5] = 8; return g
def fixture_C1():
    g = _blank(); g[1][1] = 8; g[1][2] = 9; return g
def fixture_C2():
    g = _blank(); g[1][1] = 8; g[2][1] = 9; return g
def fixture_D():
    g = _blank(); g[1][1] = 8; g[2][2] = 9; return g
def fixture_E():
    g = _blank(); g[10][10] = 8; g[20][20] = 9; return g
def fixture_G():
    g = _blank(); g[5][5] = 8; g[5][6] = 8; g[6][5] = 8; return g
def fixture_H():
    g = _blank()
    for r in range(4):
        for c in range(4):
            if (r, c) != (0, 0): g[r][c] = 1
    g[0][0] = 8; return g
def fixture_rev_left():
    g = _blank()
    for r in range(11): g[r][10] = 1
    for r in range(5, 7): g[r][2] = 2
    return g
def fixture_X():
    g = _blank()
    g[5][5] = 8; g[5][7] = 8; g[5][8] = 8
    g[6][6] = 8; g[6][7] = 8
    g[9][6] = 9
    g[10][6] = 9; g[10][7] = 9
    return g
def fixture_Y():
    g = _blank(); g[7][3] = 8; g[7][2] = 9; return g
def fixture_Z():
    g = _blank(); g[1][2] = 8; g[2][1] = 8; g[2][2] = 8; g[5][5] = 4; return g


FIXTURES = {
    "A": fixture_A, "B": fixture_B, "C1": fixture_C1, "C2": fixture_C2,
    "D": fixture_D, "E": fixture_E, "G": fixture_G, "H": fixture_H,
    "rev_left": fixture_rev_left,
    "X": fixture_X, "Y": fixture_Y, "Z": fixture_Z,
}
FIXTURES_BLOCKED: Dict[str, str] = {}


def _indep_extract(grid, bg):
    seen = set(); comps = []
    for r in range(64):
        for c in range(64):
            if grid[r][c] == bg or (r, c) in seen: continue
            col = grid[r][c]; stack = [(r, c)]; cells = []
            while stack:
                x, y = stack.pop()
                if ((x, y) in seen or not (0 <= x < 64 and 0 <= y < 64)
                        or grid[x][y] != col): continue
                seen.add((x, y)); cells.append((x, y))
                stack += [(x+1,y),(x-1,y),(x,y+1),(x,y-1)]
            comps.append(cells)
    comps.sort(key=lambda cs: (min(r for r,_ in cs), min(c for _,c in cs),
                               min(64*r+c for r,c in cs)))
    return comps


def independent_coarse16(grid, background):
    out = []
    for tile_r in range(0, 64, 4):
        row_chars = []
        for tile_c in range(0, 64, 4):
            freq = {}
            for r in range(tile_r, tile_r + 4):
                for c in range(tile_c, tile_c + 4):
                    val = grid[r][c]
                    if val != background:
                        freq[val] = freq.get(val, 0) + 1
            if not freq: row_chars.append(DIGIT_TO_CHAR[background])
            else:
                best = sorted(freq.keys(), key=lambda d: (-freq[d], d))[0]
                row_chars.append(DIGIT_TO_CHAR[best])
        out.append("".join(row_chars))
    return out


def independent_relations(grid, background, k=6):
    comps = _indep_extract(grid, background)
    bb = [(min(r for r,_ in cs), max(r for r,_ in cs),
           min(c for _,c in cs), max(c for _,c in cs)) for cs in comps]
    res = {n: [] for n in RELATION_PRIORITY}
    n = len(comps)
    for i in range(n):
        for j in range(n):
            if i == j: continue
            r0a,r1a,c0a,c1a = bb[i]; r0b,r1b,c0b,c1b = bb[j]
            sa = set(comps[i])
            adj = any((r+dr,c+dc) in sa for (r,c) in comps[j]
                      for dr,dc in ((1,0),(-1,0),(0,1),(0,-1)))
            if adj and i < j: res["adjacent"].append((i+1,j+1))
            if not (c1a < c0b or c1b < c0a) and i < j:
                res["aligned_cols"].append((i+1,j+1))
            if not (r1a < r0b or r1b < r0a) and i < j:
                res["aligned_rows"].append((i+1,j+1))
            if r1a < r0b and not (c1a < c0b or c1b < c0a):
                res["above"].append((i+1,j+1))
            if c1a < c0b and not (r1a < r0b or r1b < r0a):
                res["left"].append((i+1,j+1))
            if (r0a <= r0b and r1a >= r1b and c0a <= c0b and c1a >= c1b and
                    (r0a < r0b or r1a > r1b or c0a < c0b or c1a > c1b)):
                res["bbox_contains"].append((i+1,j+1))
    for nm in res: res[nm] = sorted(set(res[nm]))
    return res


_REL_NAME_RE = re.compile(r"^(adjacent|aligned_cols|aligned_rows|above|left|bbox_contains)$")
_REL_PAIR_RE = re.compile(r"^[1-9][0-9]*-[1-9][0-9]*$")
_REL_LINE_RE = re.compile(r"^  ([a-z_]+)=(.+)$")
_SYMMETRIC_RELS = {"adjacent", "aligned_cols", "aligned_rows"}


def _parse_relations_block(text, expected_k=None, max_endpoint=None):
    if "\r" in text:
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "CR in block")
    lines = text.split("\n")
    if not lines or lines[0] != "RELATIONS:":
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "missing RELATIONS:")
    result = []; seen_names = set()
    for i, line in enumerate(lines[1:], start=1):
        if line == "":
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "blank line in block")
        m = _REL_LINE_RE.fullmatch(line)
        if not m:
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, f"bad relation line")
        name, val = m.group(1), m.group(2)
        if not _REL_NAME_RE.fullmatch(name):
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, f"unknown {name!r}")
        if name in seen_names:
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, f"dup {name!r}")
        seen_names.add(name)
        exp_idx = i - 1
        if exp_idx >= len(RELATION_PRIORITY):
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "too many relations")
        if name != RELATION_PRIORITY[exp_idx]:
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, f"order: got {name!r}")
        if val == "-":
            result.append((name, [])); continue
        if " " in val or "\t" in val:
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, f"ws in {name!r}")
        pairs = []
        for p in val.split(","):
            if not _REL_PAIR_RE.fullmatch(p):
                raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, f"bad pair {p!r}")
            a, b = p.split("-"); a, b = int(a), int(b)
            if max_endpoint is not None:
                if not (1 <= a <= max_endpoint) or not (1 <= b <= max_endpoint):
                    raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, "endpoint out of range")
            if a == b:
                raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, f"self-pair {p!r}")
            if name in _SYMMETRIC_RELS and a >= b:
                raise ReasonCodeError(ReasonCode.NONCANONICAL, f"symmetric requires i<j")
            pairs.append((a, b))
        if len(set(pairs)) != len(pairs):
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, f"dup pair")
        if pairs != sorted(pairs, key=lambda x: (x[0], x[1])):
            raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA, f"unsorted")
        result.append((name, pairs))
    if expected_k is not None and len(result) != expected_k:
        raise ReasonCodeError(ReasonCode.MALFORMED_SCHEMA,
                              f"expected {expected_k}, got {len(result)}")
    return result


def _relations_map_to_block(m, k):
    lines = []
    for name in RELATION_PRIORITY[:k]:
        lst = sorted(m.get(name, []), key=lambda x: (x[0], x[1]))
        if not lst: lines.append(f"  {name}=-")
        else: lines.append(f"  {name}=" + ",".join(f"{s}-{t}" for s, t in lst))
    return "RELATIONS:\n" + "\n".join(lines)


def _verify_fixture_F_signature(grid):
    try: _validate_grid(grid)
    except ReasonCodeError as e: return False, f"grid invalid: {e}"
    try:
        bg = _indep_modal(grid)
        objs = extract_objects(grid, bg)
    except Exception as e: return False, f"extract failed: {e}"
    if len(objs) != 2: return False, f"F must have exactly 2 objects"
    sizes = sorted(o["size"] for o in objs)
    if sizes[0] != 1: return False, f"F enclosed-cell size must be 1"
    try:
        got = _encode_relations_prefix_unchecked(objs, 6)
        got_pairs = _parse_relations_block(got, expected_k=6, max_endpoint=len(objs))
    except Exception as e: return False, f"relation compute failed: {e}"
    got_map = {n: list(p) for n, p in got_pairs}
    mismatches = []
    for name, expected in _F_EXPECTED_RELATIONS.items():
        actual = sorted(got_map.get(name, []))
        exp_sorted = sorted(expected)
        if actual != exp_sorted:
            mismatches.append(f"{name}: got {actual} expected {exp_sorted}")
    if mismatches: return False, "signature mismatch: " + "; ".join(mismatches)
    return True, "F §9 signature OK"


_CANONICAL_F = {"path": None, "grid": None, "provenance": "unavailable",
                "verified_bytes": None, "staged_path": None}


def _try_load_canonical_F() -> None:
    path = _find_canonical_fixture_file()
    if path is None:
        _CANONICAL_F["provenance"] = "canonical companion not found"
        return
    basename = os.path.basename(path)
    pin = _pin_for(basename)
    if pin is None:
        _CANONICAL_F["provenance"] = (f"companion {basename} present but trusted "
                                      f"pin not provisioned → BLOCKED (authority "
                                      f"must supply sha256 in "
                                      f"_CANONICAL_COMPANION_SHA256)")
        return
    try:
        with open(path, "rb") as f: b = f.read()
    except Exception as e:
        _CANONICAL_F["provenance"] = f"companion read failed: {e}"; return
    h = hashlib.sha256(b).hexdigest()
    if h != pin:
        _CANONICAL_F["provenance"] = f"companion SHA-256 mismatch: {h[:12]}…"
        return
    try: content = b.decode("utf-8")
    except UnicodeDecodeError as e:
        _CANONICAL_F["provenance"] = f"companion not UTF-8: {e}"; return
    grid = _parse_canonical_fixture_content(content)
    if grid is None:
        _CANONICAL_F["provenance"] = f"companion unparsable"; return
    ok, detail = _verify_fixture_F_signature(grid)
    if not ok:
        _CANONICAL_F["provenance"] = f"companion rejected: {detail}"; return

    tmp = None
    try:
        if os.path.islink(VERIFIED_COMPANION_PATH):
            raise IOError(f"staging path is a symlink: {VERIFIED_COMPANION_PATH}")
        if os.path.exists(VERIFIED_COMPANION_PATH) and not os.path.isfile(VERIFIED_COMPANION_PATH):
            raise IOError(f"staging path is not a regular file: {VERIFIED_COMPANION_PATH}")
        fd, tmp = tempfile.mkstemp(dir=OUTPUT_DIR, prefix=".vcomp_")
        try:
            with os.fdopen(fd, "wb") as f: f.write(b)
        except Exception:
            try: os.unlink(tmp)
            except Exception: pass
            tmp = None
            raise
        with open(tmp, "rb") as f:
            if hashlib.sha256(f.read()).hexdigest() != pin:
                raise IOError("staging temp hash mismatch")
        os.replace(tmp, VERIFIED_COMPANION_PATH)
        tmp = None
        with open(VERIFIED_COMPANION_PATH, "rb") as f:
            if hashlib.sha256(f.read()).hexdigest() != pin:
                raise IOError("staging published hash mismatch")
    except Exception as e:
        _CANONICAL_F["provenance"] = f"staging materialization failed: {e}"
        if tmp is not None:
            try: os.unlink(tmp)
            except Exception: pass
        return

    _CANONICAL_F["path"] = path
    _CANONICAL_F["grid"] = grid
    _CANONICAL_F["verified_bytes"] = b
    _CANONICAL_F["staged_path"] = VERIFIED_COMPANION_PATH
    _CANONICAL_F["provenance"] = (f"canonical companion {basename} "
                                  f"(sha256-verified, signature-verified, staged)")


def with_timeout(sec, fn, *args, **kwargs):
    if not hasattr(signal, "SIGALRM"):
        return fn(*args, **kwargs)
    def _h(*_): raise TimeoutError(f"exceeded {sec}s")
    signal.signal(signal.SIGALRM, _h); signal.alarm(sec)
    try: return fn(*args, **kwargs)
    finally: signal.alarm(0)


def _status(name, st, detail=""): return (name, st, detail)


def kat_gf01_t1_bytes_md5():
    b = encode(make_gf01(), "T1", levels_completed=0,
               state="NOT_FINISHED", actions=GF_ACTIONS).encode("utf-8")
    md5 = hashlib.md5(b).hexdigest()
    if len(b) != 246: return _status("KAT GF-01 T1 bytes", Status.FAIL, f"{len(b)} != 246")
    if md5 != "35b187182b2db0514321023f0357c309":
        return _status("KAT GF-01 T1 md5", Status.FAIL, md5)
    return _status("KAT GF-01 T1", Status.PASS, f"{len(b)}B {md5[:8]}")


def kat_gf02_t1_bytes_md5():
    b = encode(make_gf02(), "T1", levels_completed=0,
               state="NOT_FINISHED", actions=GF_ACTIONS).encode("utf-8")
    md5 = hashlib.md5(b).hexdigest()
    if len(b) != 266: return _status("KAT GF-02 T1 bytes", Status.FAIL, f"{len(b)} != 266")
    if md5 != "1eb27b22bf8fbb333ca87f833a758d9b":
        return _status("KAT GF-02 T1 md5", Status.FAIL, md5)
    return _status("KAT GF-02 T1", Status.PASS, f"{len(b)}B {md5[:8]}")


def kat_gf02_t2_bytes_md5():
    b = encode(make_gf02(), "T2", levels_completed=0,
               state="NOT_FINISHED", actions=GF_ACTIONS).encode("utf-8")
    md5 = hashlib.md5(b).hexdigest()
    if len(b) != 605: return _status("KAT GF-02 T2 bytes", Status.FAIL, f"{len(b)} != 605")
    if md5 != "4cbd67b65531b703414844085e2e5a1b":
        return _status("KAT GF-02 T2 md5", Status.FAIL, md5)
    return _status("KAT GF-02 T2", Status.PASS, f"{len(b)}B {md5[:8]}")


def kat_gf02_t3_4_bytes_md5():
    b = encode(make_gf02(), "T3", k=4, levels_completed=0,
               state="NOT_FINISHED", actions=GF_ACTIONS).encode("utf-8")
    md5 = hashlib.md5(b).hexdigest()
    if len(b) != 677: return _status("KAT GF-02 T3(4) bytes", Status.FAIL, f"{len(b)} != 677")
    if md5 != "f34e4b0b52e60fda0ef555dba295270c":
        return _status("KAT GF-02 T3(4) md5", Status.FAIL, md5)
    return _status("KAT GF-02 T3(4)", Status.PASS, f"{len(b)}B {md5[:8]}")


def kat_gf01_t2_t3_erratum():
    t2 = encode(make_gf01(), "T2", levels_completed=0,
                state="NOT_FINISHED", actions=GF_ACTIONS).encode("utf-8")
    t3 = encode(make_gf01(), "T3", k=4, levels_completed=0,
                state="NOT_FINISHED", actions=GF_ACTIONS).encode("utf-8")
    if len(t2) != 537 or len(t3) != 605:
        return _status("GF-01 T2/T3 erratum", Status.FAIL, "lengths")
    t2_v36 = t2.replace(b"OBJECTS:\nCOARSE16:", b"OBJECTS:\n\nCOARSE16:", 1)
    t3_v36 = t3.replace(b"OBJECTS:\nCOARSE16:", b"OBJECTS:\n\nCOARSE16:", 1)
    if (len(t2_v36) != 538 or len(t3_v36) != 606 or
            hashlib.md5(t2_v36).hexdigest() != "2ffeee4974d5dea3c09fc8ee839bcf9b" or
            hashlib.md5(t3_v36).hexdigest() != "a41f3bffed9481bfd7ecee34617b15ad"):
        return _status("GF-01 T2/T3 erratum", Status.FAIL, "v36 reconciliation")
    return _status("GF-01 T2/T3 erratum", Status.PASS, "537B/605B & 538B/606B reconciled")


def kat_gf02_t3_relations():
    out = encode(make_gf02(), "T3", k=4, levels_completed=0,
                 state="NOT_FINISHED", actions=GF_ACTIONS)
    lines = out.split("\n"); idx = lines.index("RELATIONS:")
    rel = lines[idx+1:idx+5]
    expected = ["  adjacent=1-2", "  aligned_cols=-",
                "  aligned_rows=1-2", "  above=-"]
    if rel != expected:
        return _status("KAT GF-02 T3(4) rel", Status.FAIL, f"{rel}")
    return _status("KAT GF-02 T3(4) rel", Status.PASS, "order correct")


def _resolve_hub_cache():
    try:
        import huggingface_hub.constants as _hfc
        if getattr(_hfc, "HF_HUB_CACHE", None): return _hfc.HF_HUB_CACHE
    except Exception: pass
    hf_home = os.environ.get("HF_HOME")
    if hf_home: return os.path.join(hf_home, "hub")
    return os.path.expanduser("~/.cache/huggingface/hub")


def _resolve_tokenizer_json():
    hub = _resolve_hub_cache()
    cands = [os.path.join(hub, "models--Qwen--Qwen3.8-27B",
                          "snapshots", QWEN_REVISION, "tokenizer.json")]
    for hit in glob.glob(os.path.join(hub, "**", "tokenizer.json"), recursive=True):
        if "Qwen3.8-27B" in hit or QWEN_REVISION in hit: cands.append(hit)
    for c in cands:
        if c and os.path.exists(c): return c
    return None


def kat_tokens():
    tok = None
    try:
        from transformers import AutoTokenizer
        tok = AutoTokenizer.from_pretrained("Qwen/Qwen3.8-27B",
                revision=QWEN_REVISION, trust_remote_code=True)
    except Exception as e_full:
        try:
            from transformers import AutoTokenizer
            tok = AutoTokenizer.from_pretrained("Qwen/Qwen3.8-27B",
                    revision=QWEN_REVISION, trust_remote_code=True,
                    local_files_only=True)
        except Exception:
            tj_pre = _resolve_tokenizer_json()
            if tj_pre is not None:
                try:
                    from transformers import PreTrainedTokenizerFast
                    tok = PreTrainedTokenizerFast(tokenizer_file=tj_pre)
                except Exception as e_fast:
                    return _status("KAT tokens", Status.BLOCKED, f"{e_fast}")
            else:
                return _status("KAT tokens", Status.BLOCKED, f"{e_full}")
    tj = _resolve_tokenizer_json()
    if tj is None:
        return _status("KAT tokens", Status.BLOCKED, "not locatable")
    try: h = hashlib.sha256(open(tj, "rb").read()).hexdigest()
    except Exception as e: return _status("KAT tokens", Status.BLOCKED, f"{e}")
    if h != TOKENIZER_JSON_SHA256:
        return _status("KAT tokens", Status.FAIL, f"sha256 {h[:12]}…")
    if hasattr(tok, "backend_tokenizer") and hasattr(tok.backend_tokenizer, "no_truncation"):
        tok.backend_tokenizer.no_truncation()
    t1 = encode(make_gf01(), "T1", levels_completed=0, state="NOT_FINISHED", actions=GF_ACTIONS)
    t2 = encode(make_gf02(), "T1", levels_completed=0, state="NOT_FINISHED", actions=GF_ACTIONS)
    n1 = len(tok.encode(t1, add_special_tokens=False, truncation=False))
    n2 = len(tok.encode(t2, add_special_tokens=False, truncation=False))
    if n1 != 181: return _status("KAT GF-01 tok", Status.FAIL, f"{n1} != 181")
    if n2 != 197: return _status("KAT GF-02 tok", Status.FAIL, f"{n2} != 197")
    long_n = len(tok.encode("R1,W1," * 1200, add_special_tokens=False, truncation=False))
    if long_n <= 2048:
        return _status("KAT tokens F-31", Status.FAIL, f"truncated at {long_n}")
    return _status("KAT tokens", Status.PASS, f"181/{n1} 197/{n2} long={long_n}")


def _canonical_grid_bytes(grid):
    return bytes(v for row in grid for v in row)


def _iter_repo_frame_files():
    seen_real = set(); out = []
    for root in _REPO_ROOTS:
        pat = os.path.join(root, "results", "environment_files", "*", "*", "*.py")
        for p in sorted(glob.glob(pat)):
            rp = os.path.realpath(p)
            if rp not in seen_real:
                seen_real.add(rp); out.append(rp)
    return out


def _load_real_frames_with_count():
    """P3-7 (v35): robust frame loader with explicit per-file error reporting."""
    frames = []; errors = []; ids = []
    try:
        from arcengine import ARCBaseGame
    except ImportError as e:
        return [], [f"arcengine: {e}"], []
    for pyf in _iter_repo_frame_files():
        try:
            mod_name = "g_" + os.path.basename(pyf)[:-3]
            spec = importlib.util.spec_from_file_location(mod_name, pyf)
            m = importlib.util.module_from_spec(spec)
            sys.modules[mod_name] = m
            spec.loader.exec_module(m)
            cls = next((v for v in vars(m).values()
                        if isinstance(v, type) and issubclass(v, ARCBaseGame)
                        and v is not ARCBaseGame), None)
            if cls is None:
                errors.append(f"{pyf}: no ARCBaseGame subclass found")
                continue
            game = cls()
            grid = game.camera.render(game.current_level.get_sprites())
            frames.append([list(map(int, row)) for row in grid])
            ids.append(os.path.basename(pyf)[:-3])
        except Exception as e:
            errors.append(f"{pyf}: {type(e).__name__}: {e}")
    return frames, errors, ids


_HEX64_RE = re.compile(r"^[0-9a-f]{64}$")


def _strict_frame_id_set_equal(registry_keys):
    reg_set = set(registry_keys)
    if reg_set != CANONICAL_FRAME_IDS:
        missing = sorted(CANONICAL_FRAME_IDS - reg_set)
        extra = sorted(reg_set - CANONICAL_FRAME_IDS)
        raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                              f"frame-ID set mismatch: missing={missing} extra={extra}")


def _find_certified_source():
    found = []
    for p in _CERTIFIED_SEARCH_PATHS:
        if os.path.exists(p):
            found.append(os.path.realpath(p))
    found = sorted(set(found))
    if not found:
        return None, "certified grid_text.py not found at any explicit path"
    for p in found:
        try: b = open(p, "rb").read()
        except Exception: continue
        if len(b) == CERTIFIED_BYTES and hashlib.md5(b).hexdigest() == CERTIFIED_MD5:
            return p, None
    return None, f"no candidate at any explicit path matches pins"


def ts_spatial_01_determinism():
    g = make_gf02()
    outs = [encode(g, "T3", k=4, levels_completed=0, state="NOT_FINISHED",
                   actions=GF_ACTIONS) for _ in range(10)]
    return _status("T-SPATIAL-01",
                   Status.PASS if len(set(outs)) == 1 else Status.FAIL, "10 runs identical")


def ts_spatial_02_full_oracle():
    if not FIXTURES: return _status("T-SPATIAL-02", Status.FAIL, "FIXTURES empty")
    failures = []
    for name, fn in FIXTURES.items():
        g = fn(); bg = _indep_modal(g); objs = extract_objects(g, bg)
        try: got = _encode_relations_prefix_unchecked(objs, 6)
        except ReasonCodeError as e: failures.append(f"{name}: {e}"); continue
        exp_map = independent_relations(g, bg, 6)
        exp_block = _relations_map_to_block(exp_map, 6)
        try:
            got_pairs = _parse_relations_block(got, expected_k=6, max_endpoint=len(objs))
            exp_pairs = _parse_relations_block(exp_block, expected_k=6, max_endpoint=len(objs))
        except ReasonCodeError as e: failures.append(f"{name}: parser rejected"); continue
        if got_pairs != exp_pairs: failures.append(f"{name}: mismatch")
    if failures: return _status("T-SPATIAL-02", Status.FAIL, "; ".join(failures[:3]))
    if FIXTURES_BLOCKED:
        return _status("T-SPATIAL-02", Status.BLOCKED,
                       f"mandatory fixtures missing: {sorted(FIXTURES_BLOCKED)}")
    return _status("T-SPATIAL-02", Status.PASS,
                   f"{len(FIXTURES)} fixtures exact-ordered match "
                   f"[F: {_CANONICAL_F['provenance']}]")


def ts_spatial_03_256_tile():
    failures = []
    grids = [("GF-01", make_gf01()), ("GF-02", make_gf02())] + \
            [(n, f()) for n, f in FIXTURES.items()]
    for name, g in grids:
        bg = _indep_modal(g)
        if encode_coarse16(g, bg).split("\n")[1:] != independent_coarse16(g, bg):
            failures.append(name)
    if failures: return _status("T-SPATIAL-03", Status.FAIL, f"{failures}")
    return _status("T-SPATIAL-03", Status.PASS, f"256 tiles OK ({len(grids)} grids)")


def ts_spatial_08_lossless_real_frames():
    try: import arcengine
    except ImportError: return _status("T-SPATIAL-08", Status.BLOCKED, "arcengine unavailable")
    try: ver = importlib_metadata.version("arcengine")
    except Exception: ver = None
    if ver != ARCENGINE_VERSION:
        return _status("T-SPATIAL-08", Status.BLOCKED, f"arcengine {ver}")
    reg, reg_err = _load_frame_registry_diag()
    if reg_err is not None:
        return _status("T-SPATIAL-08", Status.FAIL, f"registry corrupt: {reg_err}")
    try: _strict_frame_id_set_equal(reg.keys())
    except ReasonCodeError as e:
        return _status("T-SPATIAL-08",
                       Status.BLOCKED if not reg else Status.FAIL, str(e))
    if len(reg) != 25: return _status("T-SPATIAL-08", Status.FAIL, f"registry {len(reg)}")
    for fid, h in reg.items():
        if not _HEX64_RE.fullmatch(h or ""):
            return _status("T-SPATIAL-08", Status.FAIL, f"bad sha256 {fid}")
    frames, errors, ids = _load_real_frames_with_count()
    if len(frames) != 25: return _status("T-SPATIAL-08", Status.FAIL, f"loaded {len(frames)}")
    if len(set(ids)) != 25 or set(ids) != CANONICAL_FRAME_IDS:
        return _status("T-SPATIAL-08", Status.FAIL, "IDs != canonical")
    verified = []
    for grid, gid in zip(frames, ids):
        h = hashlib.sha256(_canonical_grid_bytes(grid)).hexdigest()
        if h != reg[gid]: return _status("T-SPATIAL-08", Status.FAIL, f"{gid}: sha256")
        actual_md5 = hashlib.md5(_canonical_grid_bytes(grid)).hexdigest()[:9]
        if actual_md5 != REPORT_25_MD5_PREFIX[gid]:
            return _status("T-SPATIAL-08", Status.FAIL, f"{gid}: MD5")
        verified.append((gid, grid))
    tested = 0
    for gid, g in verified:
        for tier, k in (("BASELINE",0),("T1",0),("T2",0),("T3",6)):
            try: txt = encode(g, tier, k=k, levels_completed=0,
                              state="NOT_FINISHED", actions=GF_ACTIONS)
            except ReasonCodeError as e:
                return _status("T-SPATIAL-08", Status.FAIL, f"{gid} {tier}: {e}")
            if parse(txt)["grid"] != g:
                return _status("T-SPATIAL-08", Status.FAIL, f"{gid} {tier} lossless")
            tested += 1
    return _status("T-SPATIAL-08", Status.PASS,
                   f"25/25 canonical frames verified; {tested} round-trips")


def ts_spatial_09a_byte_determinism():
    g = make_gf02()
    outs = [encode(g, "T3", k=6, levels_completed=0, state="NOT_FINISHED",
                   actions=GF_ACTIONS) for _ in range(5)]
    return _status("T-SPATIAL-09a",
                   Status.PASS if len(set(outs)) == 1 else Status.FAIL, "")


def _ts12_with_cache():
    if _TS12_CACHE["result"] is None:
        _TS12_CACHE["result"] = ts_spatial_12_source_conformance()
    return _TS12_CACHE["result"]


def ts_spatial_10_25frame_relation_oracle():
    t12 = _ts12_with_cache()
    if t12[1] is not Status.PASS:
        return _status("T-SPATIAL-10", Status.BLOCKED, f"F-26 gate: {t12[1].value}")
    reg, reg_err = _load_frame_registry_diag()
    if reg_err is not None:
        return _status("T-SPATIAL-10", Status.FAIL, f"registry corrupt: {reg_err}")
    try: _strict_frame_id_set_equal(reg.keys())
    except ReasonCodeError as e: return _status("T-SPATIAL-10", Status.BLOCKED, str(e))
    frames, errors, ids = _load_real_frames_with_count()
    if len(frames) != 25: return _status("T-SPATIAL-10", Status.BLOCKED, f"{len(frames)}/25")
    if len(set(ids)) != 25 or set(ids) != CANONICAL_FRAME_IDS:
        return _status("T-SPATIAL-10", Status.FAIL, "IDs != canonical")
    for gid, g in zip(ids, frames):
        if hashlib.sha256(_canonical_grid_bytes(g)).hexdigest() != reg[gid]:
            return _status("T-SPATIAL-10", Status.FAIL, f"{gid}: registry")
        actual_md5 = hashlib.md5(_canonical_grid_bytes(g)).hexdigest()[:9]
        if actual_md5 != REPORT_25_MD5_PREFIX[gid]:
            return _status("T-SPATIAL-10", Status.FAIL, f"{gid}: MD5")
        bg = _indep_modal(g); objs = extract_objects(g, bg)
        try: got = _encode_relations_prefix_unchecked(objs, 6)
        except ReasonCodeError as e: return _status("T-SPATIAL-10", Status.FAIL, f"{gid}: {e}")
        exp_block = _relations_map_to_block(independent_relations(g, bg, 6), 6)
        if _parse_relations_block(got, 6, len(objs)) != _parse_relations_block(exp_block, 6, len(objs)):
            return _status("T-SPATIAL-10", Status.FAIL, f"{gid}: mismatch")
    return _status("T-SPATIAL-10", Status.PASS,
                   "25/25 canonical frames: all 6 relation types match oracle")


def ts_spatial_12_source_conformance():
    base, err = _find_certified_source()
    if base is None: return _status("T-SPATIAL-12", Status.BLOCKED, err)
    try: b = open(base, "rb").read()
    except Exception as e: return _status("T-SPATIAL-12", Status.FAIL, f"read: {e}")
    if len(b) != CERTIFIED_BYTES:
        return _status("T-SPATIAL-12", Status.BLOCKED, f"bytes {len(b)}")
    if hashlib.md5(b).hexdigest() != CERTIFIED_MD5:
        return _status("T-SPATIAL-12", Status.BLOCKED, "md5")
    if hashlib.sha256(b).hexdigest() != CERTIFIED_SHA256:
        return _status("T-SPATIAL-12", Status.BLOCKED, "sha256")
    identity = (f"path={base}\nbytes={len(b)}\n"
                f"md5={hashlib.md5(b).hexdigest()}\n"
                f"sha256={hashlib.sha256(b).hexdigest()}\n"
                f"expected_md5={CERTIFIED_MD5}\n"
                f"expected_sha256={CERTIFIED_SHA256}\n"
                f"expected_bytes={CERTIFIED_BYTES}\n")
    try:
        _safe_write(BASE_IDENTITY_PATH, identity)
    except ReasonCodeError as e:
        return _status("T-SPATIAL-12", Status.FAIL, f"artifact write: {e}")
    mod_name = "grid_text_certified_" + hashlib.md5(base.encode()).hexdigest()[:8]
    spec = importlib.util.spec_from_file_location(mod_name, base)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[mod_name] = mod
    try: spec.loader.exec_module(mod)
    except Exception as e:
        sys.modules.pop(mod_name, None)
        return _status("T-SPATIAL-12", Status.FAIL, f"import: {e}")
    adv = []
    a1 = _blank()
    for r, c in ((2,3),(3,3),(3,2)): a1[r][c] = 8
    a1[7][7] = 4; adv.append(("unoccupied_bbox", a1))
    adv.append(("fixture_H", fixture_H()))
    adv.append(("fixture_rev_left", fixture_rev_left()))
    adv.append(("fixture_X_shared_bbox", fixture_X()))
    adv.append(("fixture_Y_first_cell_tie", fixture_Y()))
    adv.append(("fixture_Z_unoccupied_origin", fixture_Z()))
    for label, g in adv:
        try: src_raw = mod.extract_objects(g, background=0)
        except Exception as e: return _status("T-SPATIAL-12", Status.FAIL, f"{label} src: {e}")
        loc_raw = extract_objects(g, 0)
        def _src_norm(o):
            cells = getattr(o, "cells", None) or []
            return (o.id, o.color_digit, tuple(sorted(map(tuple, cells))),
                    getattr(o, "size", None) or len(cells), tuple(o.bbox))
        def _loc_norm(o):
            return (o["id"], o["colour"], tuple(sorted(o["cells"])),
                    o["size"], (o["r0"],o["r1"],o["c0"],o["c1"]))
        if [_src_norm(o) for o in src_raw] != [_loc_norm(o) for o in loc_raw]:
            return _status("T-SPATIAL-12", Status.FAIL, f"{label}: full-record mismatch")
        bg = _indep_modal(g)
        try: enc = _encode_relations_prefix_unchecked(loc_raw, 6)
        except ReasonCodeError as e: return _status("T-SPATIAL-12", Status.FAIL, f"{label} enc: {e}")
        try: enc_pairs = _parse_relations_block(enc, expected_k=6, max_endpoint=len(loc_raw))
        except ReasonCodeError as e: return _status("T-SPATIAL-12", Status.FAIL, f"{label} parser: {e}")
        oracle = independent_relations(g, bg, 6)
        oracle_pairs = [(n, sorted(oracle[n])) for n in RELATION_PRIORITY]
        if enc_pairs != oracle_pairs:
            return _status("T-SPATIAL-12", Status.FAIL, f"{label}: enc vs oracle")
    return _status("T-SPATIAL-12", Status.PASS,
                   f"{len(adv)} adversarial fixtures: full-record + oracle match")


def ts_spatial_13_pair_budget_enforcement():
    n = 3840
    pairs = n * (n - 1) // 2
    if pairs != 7_370_880:
        return _status("T-SPATIAL-13", Status.FAIL, f"arithmetic {pairs}")
    try:
        fake = [{"id": i, "colour": 1, "cells": [(i // 64, i % 64)],
                 "r0": i // 64, "r1": i // 64, "c0": i % 64, "c1": i % 64,
                 "size": 1, "first_cell": i} for i in range(1, n + 1)]
        _encode_relations_prefix_unchecked(fake, 1)
        return _status("T-SPATIAL-13", Status.FAIL, "over-budget not refused")
    except ReasonCodeError as e:
        if e.code != ReasonCode.CANDIDATE_INFEASIBLE:
            return _status("T-SPATIAL-13", Status.FAIL, f"wrong reason code: {e.code}")
    try:
        g = make_gf02(); bg = _indep_modal(g)
        _encode_relations_prefix_unchecked(extract_objects(g, bg), 6)
    except ReasonCodeError as e:
        return _status("T-SPATIAL-13", Status.FAIL, f"post-refusal sanity encode failed: {e.code}")
    try:
        under = [{"id": i, "colour": 1, "cells": [(i // 64, i % 64)],
                  "r0": i // 64, "r1": i // 64, "c0": i % 64, "c1": i % 64,
                  "size": 1, "first_cell": i} for i in range(1, 2001)]
        _encode_relations_prefix_unchecked(under, 1)
    except ReasonCodeError as e:
        if e.code == ReasonCode.CANDIDATE_INFEASIBLE:
            return _status("T-SPATIAL-13", Status.FAIL, "under refused")
    try:
        over = [{"id": i, "colour": 1, "cells": [(i // 64, i % 64)],
                 "r0": i // 64, "r1": i // 64, "c0": i % 64, "c1": i % 64,
                 "size": 1, "first_cell": i} for i in range(1, 2002)]
        _encode_relations_prefix_unchecked(over, 1)
        return _status("T-SPATIAL-13", Status.FAIL, "boundary over not refused")
    except ReasonCodeError as e:
        if e.code != ReasonCode.CANDIDATE_INFEASIBLE:
            return _status("T-SPATIAL-13", Status.FAIL, f"boundary wrong code: {e.code}")
    return _status("T-SPATIAL-13", Status.PASS,
                   f"bound={pairs} (3840); reason=CANDIDATE_INFEASIBLE; "
                   f"2000 OK / 2001 refused; no INVALID side effect")

def ts_parse_structure_oracle():
    """P2-4 (v35): structural consistency (NOT an independent RLE oracle)."""
    failures = []
    for name, fn in list(FIXTURES.items()) + [("GF-01", make_gf01), ("GF-02", make_gf02)]:
        g = fn()
        for tier, k in (("BASELINE",0),("T1",0),("T2",0),("T3",6)):
            try: txt = encode(g, tier, k=k, levels_completed=0,
                              state="NOT_FINISHED", actions=GF_ACTIONS)
            except ReasonCodeError: continue
            a = parse(txt); b = parse_structure_only(txt)
            if a["grid"] != b["grid"] or a["tier"] != b["tier"]:
                failures.append(f"{name}/{tier}")
    if failures: return _status("parse_structure_only", Status.FAIL, f"{failures[:3]}")
    return _status("parse_structure_only", Status.PASS,
                   "structural consistency (shares rle_decode)")


def ts_objects_roundtrip():
    failures = []
    for name, fn in list(FIXTURES.items()) + [("GF-01", make_gf01), ("GF-02", make_gf02)]:
        g = fn(); bg = _indep_modal(g)
        objs = extract_objects(g, bg)
        parsed = parse_objects_block(serialize_objects(objs), background=bg, grid=g)
        a = [(o["id"],o["colour"],o["size"],o["r0"],o["r1"],o["c0"],o["c1"]) for o in objs]
        b = [(o["id"],o["colour"],o["size"],o["r0"],o["r1"],o["c0"],o["c1"]) for o in parsed]
        if a != b: failures.append(name)
    if failures: return _status("OBJECTS roundtrip", Status.FAIL, f"{failures[:3]}")
    return _status("OBJECTS roundtrip", Status.PASS, "parse(serialize(x)) == x")


def ts_relation_laws_fuzz():
    rng = random.Random(20261016); failures = []
    for _ in range(500):
        g = _blank()
        for _ in range(rng.randint(1, 3)):
            r = rng.randint(0, 60); c = rng.randint(0, 60); col = rng.randint(1, 15)
            for dr in range(rng.randint(1, 3)):
                for dc in range(rng.randint(1, 3)):
                    if r+dr < 64 and c+dc < 64 and rng.random() < 0.5: g[r+dr][c+dc] = col
        bg = _indep_modal(g); objs = extract_objects(g, bg)
        if len(objs) < 2: continue
        a, b = objs[0], objs[1]
        if _aligned_cols(a, b) != _aligned_cols(b, a): failures.append("aligned_cols"); break
        if _aligned_rows(a, b) != _aligned_rows(b, a): failures.append("aligned_rows"); break
    if failures: return _status("relation laws fuzz", Status.FAIL, "; ".join(failures))
    return _status("relation laws fuzz", Status.PASS, "500 random grids OK")


def ts_reject_interface():
    ok = encode(make_gf02(), "T1", levels_completed=0, state="NOT_FINISHED", actions=GF_ACTIONS)
    if reject(ok) is not None: return _status("reject() valid", Status.FAIL, "")
    for bad, expected in [
        (ok.replace("levels_completed=0","levels_completed=00",1), ReasonCode.NONCANONICAL),
        (ok.replace("background=W","background=X",1), ReasonCode.MALFORMED_SCHEMA),
        (ok.replace("state=NOT_FINISHED","state=RUNNING",1), ReasonCode.ENVIRONMENT_CONTRACT_FAILURE),
    ]:
        r = reject(bad)
        if r != expected: return _status("reject() interface", Status.FAIL, f"got {r}")
    return _status("reject() interface", Status.PASS, "NC/MS/ECF codes correct")


def ts_canonicality_suite():
    failures = []
    if reject("levels_completed=0\nstate=NOT_FINISHED\navailable_actions=[1]\nbackground=W\n"
              "GRID_ROWDICT:\n  R00|W64\n  R01|W64\n  MAP: 0,1,"+"0,"*61+"0") != ReasonCode.NONCANONICAL:
        failures.append("dup ROWDEF")
    if reject("levels_completed=0\nstate=NOT_FINISHED\navailable_actions=[1]\nbackground=W\n"
              "GRID_ROWDICT:\n  R00|W64\n  R01|W1,w63\n  MAP: "+"0,"*63+"0") != ReasonCode.NONCANONICAL:
        failures.append("unused ROWDEF")
    if reject("levels_completed=0\nstate=NOT_FINISHED\navailable_actions=[1]\nbackground=W\n"
              "GRID_ROWDICT:\n  R00|W64\n  R01|W1,w63\n  MAP: 1,"+"0,"*62+"0") != ReasonCode.NONCANONICAL:
        failures.append("first-use order")
    bad_rows = [f"  {i:02d}|W64" for i in range(64)]
    bad_rows[6] = "  05|W64"
    if reject("levels_completed=0\nstate=NOT_FINISHED\navailable_actions=[1]\nbackground=W\n"
              "GRID_FULL:\n"+"\n".join(bad_rows)) != ReasonCode.NONCANONICAL:
        failures.append("GRID_FULL dup row")
    g = _blank(); g[0][0] = 8; g[5][5] = 9
    for lbl, blk in [
        ("ID gap", "OBJECTS:\n  #1 R n=1 r0-0c0-0\n  #3 b n=1 r5-5c5-5"),
        ("ID order", "OBJECTS:\n  #2 b n=1 r5-5c5-5\n  #1 R n=1 r0-0c0-0"),
    ]:
        try: parse_objects_block(blk, background=0, grid=g); failures.append(lbl)
        except ReasonCodeError as e:
            if e.code != ReasonCode.NONCANONICAL: failures.append(f"{lbl}:{e.code}")
    try: rle_decode("W32,W32"); failures.append("W32,W32")
    except ReasonCodeError as e:
        if e.code != ReasonCode.NONCANONICAL: failures.append("W32,W32 code")
    try:
        if rle_decode("W64") != [0]*64: failures.append("W64 decode")
    except ReasonCodeError: failures.append("W64 rejected")
    g10 = _blank()
    for r in range(1, 10): g10[r][r] = 8
    enc10 = encode(g10, "T1", levels_completed=0, state="NOT_FINISHED", actions=[1])
    if not ("  R00|" in enc10 and "  R01|" in enc10 and "  R09|" in enc10):
        failures.append("R00/R01/R09")
    if reject(enc10) is not None: failures.append("R00/R01/R09 rejected")
    if reject(enc10.replace("  R00|", "  R000|", 1)) != ReasonCode.MALFORMED_SCHEMA:
        failures.append("R000")
    if reject(enc10.replace("  R09|", "  R64|", 1)) != ReasonCode.NONCANONICAL:
        failures.append("R64")
    if failures: return _status("§9 canonicality", Status.FAIL, "; ".join(failures[:3]))
    return _status("§9 canonicality", Status.PASS, "all 5 groups verified")


def edge_grid_bool_reject():
    g = make_gf02(); g[0][0] = True
    try: encode(g, "T1", levels_completed=0, state="NOT_FINISHED", actions=GF_ACTIONS); return _status("edge bool", Status.FAIL, "")
    except ReasonCodeError: return _status("edge bool", Status.PASS, "")

def edge_grid_float_reject():
    g = make_gf02(); g[0][0] = 1.0
    try: encode(g, "T1", levels_completed=0, state="NOT_FINISHED", actions=GF_ACTIONS); return _status("edge float", Status.FAIL, "")
    except ReasonCodeError: return _status("edge float", Status.PASS, "")

def edge_grid_range_reject():
    for bad in (-1, 16):
        g = make_gf02(); g[0][0] = bad
        try:
            encode(g, "T1", levels_completed=0, state="NOT_FINISHED", actions=GF_ACTIONS)
            return _status("edge range", Status.FAIL, "")
        except ReasonCodeError: pass
    return _status("edge range", Status.PASS, "")

def edge_rle_helper_bool():
    try: rle_encode([True]*64); return _status("edge rle bool", Status.FAIL, "")
    except ReasonCodeError: return _status("edge rle bool", Status.PASS, "")

def edge_rle_oom():
    try: rle_decode("W99999999"); return _status("edge rle OOM", Status.FAIL, "")
    except ReasonCodeError: return _status("edge rle OOM", Status.PASS, "")

def edge_bg_bool():
    try: make_header(0, "NOT_FINISHED", GF_ACTIONS, True); return _status("edge bg bool", Status.FAIL, "")
    except ReasonCodeError: return _status("edge bg bool", Status.PASS, "")

def edge_state_unhashable():
    try: make_header(0, [], GF_ACTIONS, 0); return _status("edge state", Status.FAIL, "")
    except ReasonCodeError: return _status("edge state", Status.PASS, "")

def edge_trailing_nl():
    txt = encode(make_gf02(), "T1", levels_completed=0, state="NOT_FINISHED", actions=GF_ACTIONS) + "\n"
    try: parse(txt); return _status("edge trailing NL", Status.FAIL, "")
    except ReasonCodeError: return _status("edge trailing NL", Status.PASS, "")

def edge_parser_mutations():
    base = encode(make_gf02(), "T1", levels_completed=0, state="NOT_FINISHED", actions=GF_ACTIONS)
    muts = [
        (base.replace("levels_completed=0","levels_completed=00",1), ReasonCode.NONCANONICAL),
        (base.replace("available_actions=[1, 2, 3, 4, 5, 6]","available_actions=[1, 2, 3, 4, 5, -6]",1), ReasonCode.MALFORMED_SCHEMA),
        (base + " junk", ReasonCode.MALFORMED_SCHEMA),
        (base.replace("background=W","background=X",1), ReasonCode.MALFORMED_SCHEMA),
    ]
    for i, (m, expected) in enumerate(muts):
        try: parse(m); return _status("edge parser mut", Status.FAIL, f"#{i}")
        except ReasonCodeError as e:
            if e.code != expected: return _status("edge parser mut", Status.FAIL, f"#{i} {e.code}")
    return _status("edge parser mut", Status.PASS, f"{len(muts)} rejected")

def edge_H_bbox_directed():
    g = fixture_H(); bg = _indep_modal(g); objs = extract_objects(g, bg)
    rel = _encode_relations_prefix_unchecked(objs, 6)
    pairs = _parse_relations_block(rel, expected_k=6, max_endpoint=len(objs))
    rm = {n: lst for n, lst in pairs}
    if set(rm.get("bbox_contains", [])) != {(2, 1)}: return _status("edge H bbox", Status.FAIL, "")
    return _status("edge H bbox", Status.PASS, "2-1 directed")

def edge_rev_left():
    g = fixture_rev_left(); bg = _indep_modal(g); objs = extract_objects(g, bg)
    rel = _encode_relations_prefix_unchecked(objs, 6)
    pairs = _parse_relations_block(rel, expected_k=6, max_endpoint=len(objs))
    rm = {n: lst for n, lst in pairs}
    if set(rm.get("left", [])) != {(2, 1)}: return _status("edge rev_left", Status.FAIL, "")
    return _status("edge rev_left", Status.PASS, "left=2-1")

def edge_relations_block_strict():
    base = encode(make_gf02(), "T3", k=6, levels_completed=0, state="NOT_FINISHED", actions=GF_ACTIONS)
    idx = base.index("RELATIONS:"); block = base[idx:]
    bad = [
        block.replace("adjacent=1-2","adjacent=1-2-99",1),
        block.replace("adjacent=1-2","adjacent=1-2,1-2",1),
        block.replace("adjacent=1-2","adjacent=01-2",1),
        block.replace("adjacent=1-2","adjacent=1 - 2",1),
        "RELATIONS:\r\n  adjacent=1-2\n",
        "RELATIONS:\n  adjacent=1-2\n\n  aligned_cols=-\n",
        "RELATIONS:\n  bogus=1-2\n",
        "RELATIONS:\n  aligned_cols=-\n  adjacent=1-2\n",
        "RELATIONS:\n  adjacent=2-1,1-2\n",
    ]
    for i, bb in enumerate(bad):
        try:
            _parse_relations_block(bb)
            return _status("edge strict rel", Status.FAIL, f"#{i}")
        except ReasonCodeError: pass
    return _status("edge strict rel", Status.PASS, f"{len(bad)} malformed rejected")

def edge_symmetric_i_lt_j():
    try: _parse_relations_block("RELATIONS:\n  adjacent=2-1\n", expected_k=1, max_endpoint=2)
    except ReasonCodeError: return _status("edge symmetric", Status.PASS, "")
    return _status("edge symmetric", Status.FAIL, "")

def edge_relation_endpoint_out_of_range():
    try: _parse_relations_block("RELATIONS:\n  adjacent=1-99999\n", expected_k=1, max_endpoint=2)
    except ReasonCodeError: return _status("edge endpoint", Status.PASS, "")
    return _status("edge endpoint", Status.FAIL, "")

def edge_objects_size_mismatch():
    g = _blank(); g[0][0] = 8
    try: parse_objects_block("OBJECTS:\n  #1 R n=5 r0-0c0-0", background=0, grid=g)
    except ReasonCodeError: return _status("edge obj size", Status.PASS, "")
    return _status("edge obj size", Status.FAIL, "")

def edge_objects_bbox_mismatch():
    g = _blank(); g[0][0] = 8
    try: parse_objects_block("OBJECTS:\n  #1 R n=1 r5-5c5-5", background=0, grid=g)
    except ReasonCodeError: return _status("edge obj bbox", Status.PASS, "")
    return _status("edge obj bbox", Status.FAIL, "")

def edge_objects_bg_colour():
    try: parse_objects_block("OBJECTS:\n  #1 W n=1 r0-0c0-0", background=0, grid=_blank())
    except ReasonCodeError: return _status("edge obj bg", Status.PASS, "")
    return _status("edge obj bg", Status.FAIL, "")

def edge_objects_leading_zero():
    g = _blank(); g[0][0] = 8
    try: parse_objects_block("OBJECTS:\n  #1 R n=1 r00-00c00-00", background=0, grid=g)
    except ReasonCodeError as e:
        if e.code == ReasonCode.MALFORMED_SCHEMA: return _status("edge obj leading zero", Status.PASS, "")
        return _status("edge obj leading zero", Status.FAIL, e.code)
    return _status("edge obj leading zero", Status.FAIL, "")

def edge_object_bbox_not_extrema():
    objs = [{"id":1, "colour":1, "cells":[(0,0)], "r0":0,"r1":5,"c0":0,"c1":5,"size":1,"first_cell":0}]
    try: encode_relations_prefix(objs, 1)
    except ReasonCodeError as e:
        if "extrema" in str(e): return _status("edge bbox extrema", Status.PASS, "")
        return _status("edge bbox extrema", Status.FAIL, str(e))
    return _status("edge bbox extrema", Status.FAIL, "")

def edge_object_duplicate_cell():
    objs = [{"id":1, "colour":1, "cells":[(0,0),(0,0)], "r0":0,"r1":0,"c0":0,"c1":0,"size":2,"first_cell":0}]
    try: encode_relations_prefix(objs, 1)
    except ReasonCodeError as e:
        if "duplicate cell" in str(e): return _status("edge dup cell", Status.PASS, "")
        return _status("edge dup cell", Status.FAIL, str(e))
    return _status("edge dup cell", Status.FAIL, "")

def edge_object_disconnected():
    objs = [{"id":1, "colour":1, "cells":[(0,0),(5,5)], "r0":0,"r1":5,"c0":0,"c1":5,"size":2,"first_cell":0}]
    try: encode_relations_prefix(objs, 1)
    except ReasonCodeError as e:
        if "4-connected" in str(e): return _status("edge disconnected", Status.PASS, "")
        return _status("edge disconnected", Status.FAIL, str(e))
    return _status("edge disconnected", Status.FAIL, "")

def edge_object_cross_shared():
    objs = [{"id":1, "colour":1, "cells":[(0,0)], "r0":0,"r1":0,"c0":0,"c1":0,"size":1,"first_cell":0},
            {"id":2, "colour":2, "cells":[(0,0)], "r0":0,"r1":0,"c0":0,"c1":0,"size":1,"first_cell":0}]
    try: encode_relations_prefix(objs, 1)
    except ReasonCodeError as e:
        if "shared" in str(e): return _status("edge cross shared", Status.PASS, "")
        return _status("edge cross shared", Status.FAIL, str(e))
    return _status("edge cross shared", Status.FAIL, "")

def edge_object_same_colour_adjacent():
    objs = [{"id":1, "colour":1, "cells":[(0,0)], "r0":0,"r1":0,"c0":0,"c1":0,"size":1,"first_cell":0},
            {"id":2, "colour":1, "cells":[(0,1)], "r0":0,"r1":0,"c0":1,"c1":1,"size":1,"first_cell":1}]
    try: encode_relations_prefix(objs, 1)
    except ReasonCodeError as e:
        if "4-neighbour" in str(e): return _status("edge same-colour adj", Status.PASS, "")
        return _status("edge same-colour adj", Status.FAIL, str(e))
    return _status("edge same-colour adj", Status.FAIL, "")

def edge_object_same_colour_private_ok():
    objs = [{"id":1, "colour":1, "cells":[(0,0)], "r0":0,"r1":0,"c0":0,"c1":0,"size":1,"first_cell":0},
            {"id":2, "colour":1, "cells":[(0,1)], "r0":0,"r1":0,"c0":1,"c1":1,"size":1,"first_cell":1}]
    try: _encode_relations_prefix_unchecked(objs, 1)
    except ReasonCodeError as e: return _status("edge same-colour private", Status.FAIL, str(e))
    return _status("edge same-colour private", Status.PASS, "")

def edge_object_not_canonical_order():
    objs = [{"id":1, "colour":1, "cells":[(5,5)], "r0":5,"r1":5,"c0":5,"c1":5,"size":1,"first_cell":325},
            {"id":2, "colour":2, "cells":[(0,0)], "r0":0,"r1":0,"c0":0,"c1":0,"size":1,"first_cell":0}]
    try: encode_relations_prefix(objs, 1)
    except ReasonCodeError as e:
        if "canonical" in str(e) or "order" in str(e): return _status("edge order", Status.PASS, "")
        return _status("edge order", Status.FAIL, str(e))
    return _status("edge order", Status.FAIL, "")

def edge_object_different_colour_adjacent_ok():
    objs = [{"id":1, "colour":1, "cells":[(0,0)], "r0":0,"r1":0,"c0":0,"c1":0,"size":1,"first_cell":0},
            {"id":2, "colour":2, "cells":[(0,1)], "r0":0,"r1":0,"c0":1,"c1":1,"size":1,"first_cell":1}]
    try: encode_relations_prefix(objs, 1)
    except ReasonCodeError as e: return _status("edge diff-colour", Status.FAIL, str(e))
    return _status("edge diff-colour", Status.PASS, "")

def edge_object_forged_first_cell():
    objs = [{"id":1, "colour":1, "cells":[(0,1)], "r0":0,"r1":0,"c0":1,"c1":1,"size":1,"first_cell":0},
            {"id":2, "colour":2, "cells":[(0,0)], "r0":0,"r1":0,"c0":0,"c1":0,"size":1,"first_cell":1}]
    try: encode_relations_prefix(objs, 1)
    except ReasonCodeError as e:
        if "first_cell" in str(e): return _status("edge forged first_cell", Status.PASS, "")
        return _status("edge forged first_cell", Status.FAIL, str(e))
    return _status("edge forged first_cell", Status.FAIL, "")

def edge_object_first_cell_true_value_ok():
    objs = [{"id":1, "colour":1, "cells":[(0,0)], "r0":0,"r1":0,"c0":0,"c1":0,"size":1,"first_cell":0},
            {"id":2, "colour":2, "cells":[(0,1)], "r0":0,"r1":0,"c0":1,"c1":1,"size":1,"first_cell":1}]
    try: encode_relations_prefix(objs, 1)
    except ReasonCodeError as e: return _status("edge true first_cell", Status.FAIL, str(e))
    return _status("edge true first_cell", Status.PASS, "")

def edge_object_first_cell_explicit_none():
    objs = [{"id":1, "colour":1, "cells":[(0,0)], "r0":0,"r1":0,"c0":0,"c1":0,"size":1,"first_cell":None}]
    try: encode_relations_prefix(objs, 1)
    except ReasonCodeError as e:
        if "first_cell" in str(e): return _status("edge first_cell None", Status.PASS, "")
        return _status("edge first_cell None", Status.FAIL, str(e))
    return _status("edge first_cell None", Status.FAIL, "")

def edge_modal_bg_mismatch():
    good = encode(make_gf01(), "T1", levels_completed=0, state="NOT_FINISHED", actions=GF_ACTIONS)
    bad = good.replace("background=W", "background=R", 1)
    try: parse(bad)
    except ReasonCodeError: return _status("edge modal bg", Status.PASS, "")
    return _status("edge modal bg", Status.FAIL, "")

def edge_k_bool_reject():
    objs = extract_objects(make_gf02(), 0)
    try: encode_relations_prefix(objs, True)
    except ReasonCodeError: return _status("edge k bool", Status.PASS, "")
    return _status("edge k bool", Status.FAIL, "")

def edge_encode_grid_full_validation():
    try: encode_grid_full([1, 2, 3])
    except ReasonCodeError: return _status("edge helper grid", Status.PASS, "")
    return _status("edge helper grid", Status.FAIL, "")

def edge_object_validation_bad_bbox():
    objs = [{"id":1, "colour":1, "cells":[(0,0)], "r0":None,"r1":0,"c0":0,"c1":0,"size":1,"first_cell":0}]
    try: encode_relations_prefix(objs, 1)
    except ReasonCodeError: return _status("edge obj rec", Status.PASS, "")
    return _status("edge obj rec", Status.FAIL, "")

def edge_object_id_order():
    objs = [{"id":2, "colour":1, "cells":[(0,0)], "r0":0,"r1":0,"c0":0,"c1":0,"size":1,"first_cell":0},
            {"id":1, "colour":2, "cells":[(1,1)], "r0":1,"r1":1,"c0":1,"c1":1,"size":1,"first_cell":65}]
    try: encode_relations_prefix(objs, 1)
    except ReasonCodeError: return _status("edge obj id order", Status.PASS, "")
    return _status("edge obj id order", Status.FAIL, "")

def edge_F_negative_control_all_W():
    g = [[0]*64 for _ in range(64)]
    ok, _ = _verify_fixture_F_signature(g)
    return _status("edge F neg all-W", Status.FAIL if ok else Status.PASS, "")

def edge_F_negative_control_3x3_ring():
    g = [[0]*64 for _ in range(64)]
    for r in range(3):
        for c in range(3):
            if (r, c) != (1, 1): g[r][c] = 1
    g[1][1] = 8
    ok, _ = _verify_fixture_F_signature(g)
    return _status("edge F neg 3x3 ring", Status.FAIL if ok else Status.PASS, "")

def edge_F_positive_control_5x5_ring():
    g = [[0]*64 for _ in range(64)]
    for r in range(5):
        for c in range(5):
            if r == 0 or r == 4 or c == 0 or c == 4: g[r][c] = 1
    g[2][2] = 8
    ok, detail = _verify_fixture_F_signature(g)
    return _status("edge F pos 5x5 ring", Status.FAIL if not ok else Status.PASS, detail)

def edge_F_negative_control_single_object():
    g = [[0]*64 for _ in range(64)]; g[5][5] = 8
    ok, _ = _verify_fixture_F_signature(g)
    return _status("edge F neg single", Status.FAIL if ok else Status.PASS, "")

def edge_F_signature_not_uniquely_canonical():
    g5 = [[0]*64 for _ in range(64)]
    for r in range(5):
        for c in range(5):
            if r == 0 or r == 4 or c == 0 or c == 4: g5[r][c] = 1
    g5[2][2] = 8
    g7 = [[0]*64 for _ in range(64)]
    for r in range(7):
        for c in range(7):
            if r == 0 or r == 6 or c == 0 or c == 6: g7[r][c] = 1
    g7[3][3] = 8
    ok5, _ = _verify_fixture_F_signature(g5)
    ok7, _ = _verify_fixture_F_signature(g7)
    if not (ok5 and ok7): return _status("edge F non-unique", Status.FAIL, f"5x5={ok5} 7x7={ok7}")
    return _status("edge F non-unique", Status.PASS, "5x5 AND 7x7 pass → pin mandatory")

def edge_F_parse_content_only():
    import inspect
    params = list(inspect.signature(_parse_canonical_fixture_content).parameters.keys())
    if params != ["content"]: return _status("edge F parse sig", Status.FAIL, str(params))
    return _status("edge F parse sig", Status.PASS, "")


@contextlib.contextmanager
def _temp_staged_substitution():
    if _CANONICAL_F["path"] is None or _CANONICAL_F["staged_path"] is None:
        raise RuntimeError("F not bound")
    real = _CANONICAL_F["staged_path"]
    if not os.path.isfile(real) or os.path.islink(real):
        raise RuntimeError("real staged not a regular file")
    real_bytes = open(real, "rb").read()
    tmpdir = tempfile.mkdtemp(prefix="ge1_tstaged_")
    tmp = os.path.join(tmpdir, os.path.basename(real))
    shutil.copyfile(real, tmp)
    saved = dict(_CANONICAL_F)
    _CANONICAL_F["staged_path"] = tmp
    try:
        yield tmp
    finally:
        _CANONICAL_F.clear(); _CANONICAL_F.update(saved)
        shutil.rmtree(tmpdir, ignore_errors=True)
        try:
            cur = open(real, "rb").read()
        except Exception as e:
            raise RuntimeError(f"real staged unreadable after test: {e}")
        if cur != real_bytes:
            raise RuntimeError("real staged mutated by test")


def edge_F_toctou_manifest_detected():
    if not _F_is_provisioned():
        return _status("edge F TOCTOU", Status.BLOCKED, "F pin not provisioned")
    if _CANONICAL_F["path"] is None or _CANONICAL_F["staged_path"] is None:
        return _status("edge F TOCTOU", Status.BLOCKED, "F not bound")
    try:
        with _temp_staged_substitution() as tmp:
            try:
                _artifacts("/tmp/ge1_dummy_manifest.txt")
            except ReasonCodeError as e:
                return _status("edge F TOCTOU", Status.FAIL, f"baseline rejected: {e}")
            with open(tmp, "wb") as f: f.write(b"TOCTOU_MUTATED\n")
            try:
                _artifacts("/tmp/ge1_dummy_manifest.txt")
            except ReasonCodeError as e:
                if not any(k in str(e).lower() for k in
                           ("changed", "mismatch", "pin", "staged", "regular")):
                    return _status("edge F TOCTOU", Status.FAIL, f"wrong reason: {e}")
                return _status("edge F TOCTOU", Status.PASS, "mutated temp staged rejected")
            return _status("edge F TOCTOU", Status.FAIL, "mutated temp staged accepted")
    except RuntimeError as e:
        return _status("edge F TOCTOU", Status.FAIL, f"setup/restore: {e}")


def edge_F_snapshot_manifest_digest():
    if not _F_is_provisioned():
        return _status("edge F snapshot", Status.BLOCKED, "F pin not provisioned")
    if _CANONICAL_F["path"] is None or _CANONICAL_F["staged_path"] is None:
        return _status("edge F snapshot", Status.BLOCKED, "F not bound")
    try:
        with _temp_staged_substitution() as tmp:
            arts = _artifacts("/tmp/ge1_dummy_manifest2.txt")
            comp = [e for e in arts if e.get("role") == "reused-verified-canonical-companion"]
            if not comp:
                return _status("edge F snapshot", Status.FAIL, "companion not listed")
            entry = comp[0]
            if entry.get("kind") != "snapshot":
                return _status("edge F snapshot", Status.FAIL, f"kind={entry.get('kind')!r}")
            snap = entry.get("bytes")
            if snap is None:
                return _status("edge F snapshot", Status.FAIL, "no snapshot bytes")
            expected_digest = hashlib.sha256(snap).hexdigest()
            expected_md5 = hashlib.md5(snap).hexdigest()
            expected_sz = len(snap)
            with open(tmp, "wb") as f: f.write(b"TOCTOU_POST_ARTIFACTS\n")
            try:
                row, structured = _entry_to_manifest_row(entry)
            except Exception as e:
                return _status("edge F snapshot", Status.FAIL, f"helper: {e}")
            if structured.get("sha256") != expected_digest:
                return _status("edge F snapshot", Status.FAIL, "sha256 mismatch")
            if structured.get("md5") != expected_md5:
                return _status("edge F snapshot", Status.FAIL, "md5 mismatch")
            if structured.get("bytes") != expected_sz:
                return _status("edge F snapshot", Status.FAIL, "size mismatch")
            return _status("edge F snapshot", Status.PASS, "helper digest = snapshot bytes")
    except RuntimeError as e:
        return _status("edge F snapshot", Status.FAIL, f"setup/restore: {e}")


def edge_F_manifest_end_to_end():
    if not _F_is_provisioned() or _CANONICAL_F["path"] is None:
        return _status("edge F manifest e2e", Status.BLOCKED, "F pin not provisioned / F not bound")
    tmpdir = tempfile.mkdtemp(prefix="ge1_me2e_")
    try:
        bid = os.path.join(tmpdir, "BASE_IDENTITY.txt")
        frj = os.path.join(tmpdir, "FRAME_REGISTRY.json")
        with open(bid, "wb") as f: f.write(b"placeholder_base_identity\n")
        with open(frj, "wb") as f: f.write(b"{}\n")
        out = os.path.join(tmpdir, "m.txt")
        try:
            _generate_manifest_impl(out, _writer=_safe_write,
                _rollback_writer=_default_rollback_writer,
                _base_identity=bid, _frame_registry=frj)
        except Exception as e:
            return _status("edge F manifest e2e", Status.FAIL, f"generate: {e}")
        try:
            manifest_text = open(out).read()
        except Exception as e:
            return _status("edge F manifest e2e", Status.FAIL, f"read: {e}")
        snap = _CANONICAL_F["verified_bytes"]
        expected_sha = hashlib.sha256(snap).hexdigest()
        expected_md5 = hashlib.md5(snap).hexdigest()
        expected_sz = str(len(snap))
        base = _sanitize_basename(os.path.basename(_CANONICAL_F["staged_path"]))
        found = None
        for line in manifest_text.split("\n"):
            if line.startswith(base + " |"):
                found = line
                break
        if found is None:
            return _status("edge F manifest e2e", Status.FAIL, "companion row missing")
        parts = [p.strip() for p in found.split("|")]
        if len(parts) < 5:
            return _status("edge F manifest e2e", Status.FAIL, f"row shape: {found!r}")
        if parts[1] != expected_sz:
            return _status("edge F manifest e2e", Status.FAIL, "size mismatch")
        if parts[2] != expected_md5:
            return _status("edge F manifest e2e", Status.FAIL, "md5 mismatch")
        if parts[3] != expected_sha:
            return _status("edge F manifest e2e", Status.FAIL, "sha256 mismatch")
        entry = {"role": "reused-verified-canonical-companion",
                 "requirement": "MANDATORY",
                 "kind": "snapshot",
                 "basename": os.path.basename(_CANONICAL_F["staged_path"]),
                 "bytes": snap}
        row, _ = _entry_to_manifest_row(entry)
        if row != found:
            return _status("edge F manifest e2e", Status.FAIL,
                          f"helper row != manifest row")
        return _status("edge F manifest e2e", Status.PASS,
                       "end-to-end manifest digest verified")
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def edge_F_staged_symlink_after_publish():
    if not _F_is_provisioned() or _CANONICAL_F["staged_path"] is None:
        return _status("edge F staged symlink", Status.BLOCKED, "F pin not provisioned / F not bound")
    try:
        with _temp_staged_substitution() as tmp:
            try:
                _artifacts("/tmp/ge1_dummy_symlink_baseline.txt")
            except ReasonCodeError as e:
                return _status("edge F staged symlink", Status.FAIL, f"baseline: {e}")
            canary = tmp + ".canary"
            canary_content = b"CANARY_NOT_TO_BE_WRITTEN\n"
            with open(canary, "wb") as f: f.write(canary_content)
            os.unlink(tmp)
            os.symlink(canary, tmp)
            try:
                _artifacts("/tmp/ge1_dummy_symlink_manifest.txt")
                return _status("edge F staged symlink", Status.FAIL, "symlink accepted")
            except ReasonCodeError as e:
                msg = str(e).lower()
                if not any(k in msg for k in ("symlink", "regular", "lstat", "nofollow", "not")):
                    return _status("edge F staged symlink", Status.FAIL, f"wrong reason: {e}")
            cur = open(canary, "rb").read()
            if cur != canary_content:
                return _status("edge F staged symlink", Status.FAIL, "canary mutated")
            return _status("edge F staged symlink", Status.PASS, "symlink rejected; canary intact")
    except RuntimeError as e:
        return _status("edge F staged symlink", Status.FAIL, f"setup/restore: {e}")


def edge_F_staged_nonregular_after_publish():
    if not _F_is_provisioned() or _CANONICAL_F["staged_path"] is None:
        return _status("edge F staged nonregular", Status.BLOCKED, "F pin not provisioned / F not bound")
    try:
        with _temp_staged_substitution() as tmp:
            try:
                _artifacts("/tmp/ge1_dummy_nonreg_baseline.txt")
            except ReasonCodeError as e:
                return _status("edge F staged nonregular", Status.FAIL, f"baseline: {e}")
            os.unlink(tmp)
            os.makedirs(tmp)
            try:
                _artifacts("/tmp/ge1_dummy_nonreg_manifest.txt")
                return _status("edge F staged nonregular", Status.FAIL, "directory accepted")
            except ReasonCodeError as e:
                msg = str(e).lower()
                if not any(k in msg for k in ("regular", "lstat", "directory", "not")):
                    return _status("edge F staged nonregular", Status.FAIL, f"wrong reason: {e}")
            return _status("edge F staged nonregular", Status.PASS, "non-regular rejected")
    except RuntimeError as e:
        return _status("edge F staged nonregular", Status.FAIL, f"setup/restore: {e}")


def edge_F_staged_fifo_after_publish():
    if not _F_is_provisioned() or _CANONICAL_F["staged_path"] is None:
        return _status("edge F staged fifo", Status.BLOCKED, "F pin not provisioned / F not bound")
    if not hasattr(os, "mkfifo"):
        return _status("edge F staged fifo", Status.BLOCKED, "mkfifo unavailable")
    try:
        with _temp_staged_substitution() as tmp:
            try:
                _artifacts("/tmp/ge1_dummy_fifo_baseline.txt")
            except ReasonCodeError as e:
                return _status("edge F staged fifo", Status.FAIL, f"baseline: {e}")
            os.unlink(tmp)
            os.mkfifo(tmp)
            try:
                _artifacts("/tmp/ge1_dummy_fifo_manifest.txt")
                return _status("edge F staged fifo", Status.FAIL, "fifo accepted")
            except ReasonCodeError as e:
                msg = str(e).lower()
                if not any(k in msg for k in ("regular", "fstat", "not", "fifo", "staged")):
                    return _status("edge F staged fifo", Status.FAIL, f"wrong reason: {e}")
            return _status("edge F staged fifo", Status.PASS, "fifo rejected")
    except RuntimeError as e:
        return _status("edge F staged fifo", Status.FAIL, f"setup/restore: {e}")


def edge_fixture_F_status():
    if _CANONICAL_F["grid"] is not None:
        if "F" not in FIXTURES or FIXTURES_BLOCKED: return _status("edge F status", Status.FAIL, "")
        if _CANONICAL_F.get("verified_bytes") is None or _CANONICAL_F.get("staged_path") is None:
            return _status("edge F status", Status.FAIL, "missing")
        basename = os.path.basename(_CANONICAL_F["path"] or "")
        pin = _pin_for(basename)
        if pin is None: return _status("edge F status", Status.FAIL, "no pin")
        h = hashlib.sha256(_CANONICAL_F["verified_bytes"]).hexdigest()
        if h != pin: return _status("edge F status", Status.FAIL, "hash")
        try: staged_bytes = _safe_read_staged(_CANONICAL_F["staged_path"])
        except ReasonCodeError as e: return _status("edge F status", Status.FAIL, str(e))
        if staged_bytes != _CANONICAL_F["verified_bytes"]:
            return _status("edge F status", Status.FAIL, "staged != snapshot")
        return _status("edge F status", Status.PASS, _CANONICAL_F['provenance'])
    if "F" in FIXTURES: return _status("edge F status", Status.FAIL, "")
    if "F" not in FIXTURES_BLOCKED: return _status("edge F status", Status.FAIL, "")
    return _status("edge F status", Status.PASS, "BLOCKED correctly")

def edge_frame_id_set_equal():
    try: _strict_frame_id_set_equal(dict(REPORT_25_MD5_PREFIX).keys())
    except ReasonCodeError: return _status("edge frame IDs", Status.FAIL, "")
    bad = dict(REPORT_25_MD5_PREFIX); bad["not_real"] = "deadbeef0"
    try:
        _strict_frame_id_set_equal(bad.keys())
        return _status("edge frame IDs", Status.FAIL, "extra accepted")
    except ReasonCodeError: return _status("edge frame IDs", Status.PASS, "")

def edge_registry_hex_validation():
    bad = [k for k, v in {k:"NOT_A_SHA256" for k in CANONICAL_FRAME_IDS}.items()
           if not _HEX64_RE.fullmatch(v)]
    if len(bad) != 25: return _status("edge registry hex", Status.FAIL, "")
    return _status("edge registry hex", Status.PASS, "")

def edge_cwd_independence():
    saved = os.getcwd()
    try:
        try: os.chdir(_WIRE_DIR)
        except Exception: pass
        a = set(_iter_repo_frame_files())
        try: os.chdir("/tmp" if os.path.isdir("/tmp") else _WIRE_DIR)
        except Exception: pass
        b = set(_iter_repo_frame_files())
    finally: os.chdir(saved)
    if not a and not b: return _status("edge CWD-independence", Status.BLOCKED, "no frames")
    if a != b: return _status("edge CWD-independence", Status.FAIL, "")
    return _status("edge CWD-independence", Status.PASS, f"{len(a)} frames")

def edge_frame_ids_are_letters():
    if "r11l" not in CANONICAL_FRAME_IDS: return _status("edge frame IDs letters", Status.FAIL, "")
    if "s5i5" not in CANONICAL_FRAME_IDS: return _status("edge frame IDs letters", Status.FAIL, "")
    if "r111" in CANONICAL_FRAME_IDS or "s515" in CANONICAL_FRAME_IDS:
        return _status("edge frame IDs letters", Status.FAIL, "")
    return _status("edge frame IDs letters", Status.PASS, "")

def edge_module_init_state():
    for k in ("path", "grid", "provenance"):
        if k not in _CANONICAL_F: return _status("edge module-init", Status.FAIL, "")
    if _CANONICAL_F["grid"] is not None:
        if not _CANONICAL_F["path"]: return _status("edge module-init", Status.FAIL, "")
        if _CANONICAL_F.get("verified_bytes") is None: return _status("edge module-init", Status.FAIL, "")
        if _CANONICAL_F.get("staged_path") is None: return _status("edge module-init", Status.FAIL, "")
        basename = os.path.basename(_CANONICAL_F["path"])
        pin = _pin_for(basename)
        if pin is None: return _status("edge module-init", Status.FAIL, "")
        h = hashlib.sha256(_CANONICAL_F["verified_bytes"]).hexdigest()
        if h != pin: return _status("edge module-init", Status.FAIL, "")
    return _status("edge module-init", Status.PASS, _CANONICAL_F['provenance'])

def edge_spec_lock():
    if SPEC_REV != "DIR092 Rev v38": return _status("spec lock", Status.FAIL, "")
    return _status("spec lock", Status.PASS, SPEC_REV)


def edge_collision_default_allowed():
    try:
        _assert_no_output_collision(RELEASE_MANIFEST_PATH)
        _assert_no_output_collision(None)
    except ReasonCodeError as e:
        return _status("edge collision default", Status.FAIL, str(e))
    return _status("edge collision default", Status.PASS, "default triple allowed")

def edge_collision_sidecar_as_primary():
    for attack in (RELEASE_MANIFEST_PATH + ".json", RELEASE_MANIFEST_PATH + ".sha256"):
        try:
            _assert_no_output_collision(attack)
            return _status("edge collision sidecar", Status.FAIL,
                           f"accepted {os.path.basename(attack)}")
        except ReasonCodeError: pass
    return _status("edge collision sidecar", Status.PASS, "sidecars rejected as primary")

def edge_collision_staged_path_as_output():
    saved = dict(_CANONICAL_F)
    try:
        _CANONICAL_F["path"] = None
        _CANONICAL_F["grid"] = None
        _CANONICAL_F["staged_path"] = None
        try:
            _assert_no_output_collision(VERIFIED_COMPANION_PATH)
            return _status("edge collision staged", Status.FAIL,
                           "staged path accepted when F blocked")
        except ReasonCodeError: pass
    finally:
        _CANONICAL_F.clear(); _CANONICAL_F.update(saved)
    return _status("edge collision staged", Status.PASS, "staged path protected (F blocked)")

def edge_collision_fallback_grid_text():
    try:
        _assert_no_output_collision(_FALLBACK_GRID_TEXT)
        return _status("edge collision fallback", Status.FAIL, "fallback accepted")
    except ReasonCodeError:
        pass
    return _status("edge collision fallback", Status.PASS, "fallback protected")

def edge_collision_test_log():
    try:
        _assert_no_output_collision(TEST_LOG_PATH)
        return _status("edge collision test-log", Status.FAIL, "test log accepted")
    except ReasonCodeError:
        pass
    return _status("edge collision test-log", Status.PASS, "test log protected")

def edge_collision_distinct_triple():
    try:
        _assert_no_output_collision("/tmp/ge1_collision_test_out")
    except ReasonCodeError as e:
        return _status("edge collision distinct", Status.FAIL, str(e))
    return _status("edge collision distinct", Status.PASS, "triple distinct")


def _collision_test_tmpdir():
    return tempfile.mkdtemp(prefix="ge1_coll_")

def edge_collision_symlink_alias_to_default():
    tmpdir = _collision_test_tmpdir()
    try:
        default_real = os.path.realpath(RELEASE_MANIFEST_PATH)
        alias = os.path.join(tmpdir, "alias.txt")
        os.symlink(default_real, alias)
        try:
            _assert_no_output_collision(alias)
            return _status("edge collision symlink alias", Status.FAIL,
                           "symlink alias accepted")
        except ReasonCodeError:
            pass
        return _status("edge collision symlink alias", Status.PASS,
                       "symlink alias rejected")
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)

def edge_collision_symlink_at_default_path():
    tmpdir = _collision_test_tmpdir()
    try:
        real_target = os.path.join(tmpdir, "target.txt")
        with open(real_target, "w") as f: f.write("x")
        sym = os.path.join(tmpdir, "sym.txt")
        os.symlink(real_target, sym)
        try:
            _assert_no_output_collision(sym)
            return _status("edge collision symlink default", Status.FAIL,
                           "symlink accepted")
        except ReasonCodeError:
            pass
        return _status("edge collision symlink default", Status.PASS,
                       "symlink rejected at any path")
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)

def edge_collision_default_sidecar_symlink():
    tmpdir = _collision_test_tmpdir()
    try:
        canary = os.path.join(tmpdir, "canary.txt")
        canary_content = b"CANARY\n"
        with open(canary, "wb") as f: f.write(canary_content)
        out = os.path.join(tmpdir, "m.txt")
        json_sidecar = out + ".json"
        os.symlink(canary, json_sidecar)
        try:
            _assert_no_output_collision(out)
            return _status("edge collision default sidecar", Status.FAIL,
                           "sidecar symlink accepted")
        except ReasonCodeError:
            pass
        cur = open(canary, "rb").read()
        if cur != canary_content:
            return _status("edge collision default sidecar", Status.FAIL, "canary mutated")
        return _status("edge collision default sidecar", Status.PASS,
                       "sidecar symlink rejected; canary intact")
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def edge_collision_hardlink_alias():
    tmpdir = _collision_test_tmpdir()
    try:
        protected = os.path.join(tmpdir, "protected.txt")
        with open(protected, "wb") as f: f.write(b"PROTECTED\n")
        alias = os.path.join(tmpdir, "hardlink.txt")
        try:
            os.link(protected, alias)
        except OSError as e:
            return _status("edge collision hardlink", Status.BLOCKED,
                           f"hardlink unsupported: {e}")
        try:
            _assert_no_output_collision(alias, extra_protected=[protected])
            return _status("edge collision hardlink", Status.FAIL,
                           "hardlink accepted")
        except ReasonCodeError:
            pass
        return _status("edge collision hardlink", Status.PASS,
                       "hardlink to isolated protected input rejected")
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)

def edge_collision_sidecar_hardlink():
    tmpdir = _collision_test_tmpdir()
    try:
        protected = os.path.join(tmpdir, "protected.txt")
        with open(protected, "wb") as f: f.write(b"PROTECTED\n")
        out = os.path.join(tmpdir, "m.txt")
        json_side = out + ".json"
        try:
            os.link(protected, json_side)
        except OSError as e:
            return _status("edge collision sidecar hardlink", Status.BLOCKED,
                           f"hardlink unsupported: {e}")
        try:
            _assert_no_output_collision(out, extra_protected=[protected])
            return _status("edge collision sidecar hardlink", Status.FAIL,
                           "sidecar hardlink accepted")
        except ReasonCodeError:
            pass
        return _status("edge collision sidecar hardlink", Status.PASS,
                       "sidecar hardlink rejected")
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def edge_collision_default_triple_existing():
    tmpdir = _collision_test_tmpdir()
    try:
        m = os.path.join(tmpdir, "RELEASE_MANIFEST.txt")
        triple = [m, m + ".json", m + ".sha256"]
        try:
            _assert_no_output_collision_test(m, default_triple_override=triple)
        except ReasonCodeError as e:
            return _status("edge collision default existing", Status.FAIL, f"first: {e}")
        with open(m, "wb") as f: f.write(b"OLD\n")
        with open(m + ".json", "wb") as f: f.write(b"{}\n")
        with open(m + ".sha256", "wb") as f:
            f.write(hashlib.sha256(b"OLD\n").hexdigest().encode() + b"\n")
        try:
            _assert_no_output_collision_test(m, default_triple_override=triple)
        except ReasonCodeError as e:
            return _status("edge collision default existing", Status.FAIL, f"second: {e}")
        return _status("edge collision default existing", Status.PASS,
                       "existing default triple accepted on re-run")
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)

def edge_collision_default_hardlinked_to_protected():
    tmpdir = _collision_test_tmpdir()
    try:
        m = os.path.join(tmpdir, "RELEASE_MANIFEST.txt")
        triple = [m, m + ".json", m + ".sha256"]
        other_protected = os.path.join(tmpdir, "other.txt")
        with open(other_protected, "wb") as f: f.write(b"OTHER\n")
        try:
            os.link(other_protected, m)
        except OSError as e:
            return _status("edge collision default hardlinked", Status.BLOCKED,
                           f"hardlink unsupported: {e}")
        try:
            _assert_no_output_collision_test(
                m, default_triple_override=triple,
                extra_protected=[other_protected])
            return _status("edge collision default hardlinked", Status.FAIL,
                           "hardlinked default accepted")
        except ReasonCodeError:
            pass
        return _status("edge collision default hardlinked", Status.PASS,
                       "hardlinked default rejected")
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def edge_generate_manifest_no_injection_params():
    import inspect
    sig = inspect.signature(generate_manifest)
    forbidden = {"_writer", "_rollback_writer", "_test_base_identity",
                 "_test_frame_registry", "_base_identity", "_frame_registry"}
    leaked = forbidden & set(sig.parameters)
    if leaked:
        return _status("edge generate_manifest signature", Status.FAIL,
                       f"leaked: {sorted(leaked)}")
    return _status("edge generate_manifest signature", Status.PASS,
                   "production signature clean")


def edge_evidence_log_toctou_symlink_swap():
    tmpdir = _collision_test_tmpdir()
    try:
        target = os.path.join(tmpdir, "evidence.txt")
        canary = os.path.join(tmpdir, "canary.txt")
        canary_content = b"CANARY_NOT_TO_BE_OVERWRITTEN\n"
        with open(canary, "wb") as f: f.write(canary_content)
        original_safe_write = globals()["_safe_write"]
        def swapping_write(path, data):
            if os.path.exists(path):
                try: os.unlink(path)
                except OSError: pass
            os.symlink(canary, path)
            return original_safe_write(path, data)
        globals()["_safe_write"] = swapping_write
        try:
            try:
                _emit_evidence_log(target)
                return _status("edge evidence log TOCTOU", Status.FAIL,
                               "symlink swap accepted")
            except ReasonCodeError:
                pass
            finally:
                globals()["_safe_write"] = original_safe_write
        finally:
            globals()["_safe_write"] = original_safe_write
        if open(canary, "rb").read() != canary_content:
            return _status("edge evidence log TOCTOU", Status.FAIL, "canary mutated")
        return _status("edge evidence log TOCTOU", Status.PASS,
                       "symlink swap rejected by _safe_write")
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def edge_evidence_log_rejects_source():
    src = os.path.abspath(__file__)
    try:
        with open(src, "rb") as f: orig = f.read()
    except Exception as e:
        return _status("edge evidence log source", Status.BLOCKED, f"read: {e}")
    orig_hash = hashlib.sha256(orig).hexdigest()
    try:
        _emit_evidence_log(src)
        return _status("edge evidence log source", Status.FAIL,
                       "source module accepted")
    except ReasonCodeError:
        pass
    except Exception as e:
        return _status("edge evidence log source", Status.FAIL, str(e))
    with open(src, "rb") as f: cur = f.read()
    if hashlib.sha256(cur).hexdigest() != orig_hash:
        return _status("edge evidence log source", Status.FAIL, "source mutated")
    return _status("edge evidence log source", Status.PASS,
                   "source write rejected; hash unchanged")

def edge_evidence_log_rejects_protected():
    targets = [BASE_IDENTITY_PATH, FRAME_REGISTRY_JSON, TEST_LOG_PATH,
               RELEASE_MANIFEST_PATH, VERIFIED_COMPANION_PATH]
    for t in targets:
        try:
            _emit_evidence_log(t)
            return _status("edge evidence log protected", Status.FAIL,
                           f"accepted protected: {os.path.basename(t)}")
        except ReasonCodeError:
            pass
    return _status("edge evidence log protected", Status.PASS,
                   "all protected targets rejected")

def edge_evidence_log_rejects_sidecars():
    for t in (RELEASE_MANIFEST_PATH + ".json",
              RELEASE_MANIFEST_PATH + ".sha256"):
        try:
            _emit_evidence_log(t)
            return _status("edge evidence log sidecars", Status.FAIL,
                           f"accepted: {os.path.basename(t)}")
        except ReasonCodeError:
            pass
    return _status("edge evidence log sidecars", Status.PASS,
                   "both sidecars rejected")

def edge_evidence_log_rejects_hardlink_to_source():
    tmpdir = _collision_test_tmpdir()
    try:
        link = os.path.join(tmpdir, "evil_evidence.txt")
        try:
            os.link(os.path.abspath(__file__), link)
        except OSError as e:
            return _status("edge evidence log hardlink", Status.BLOCKED,
                           f"hardlink unsupported: {e}")
        try:
            _emit_evidence_log(link)
            return _status("edge evidence log hardlink", Status.FAIL,
                           "hardlink accepted")
        except ReasonCodeError:
            pass
        return _status("edge evidence log hardlink", Status.PASS,
                       "hardlink-to-source rejected")
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def edge_safe_write_rejects_symlink():
    tmpdir = _collision_test_tmpdir()
    try:
        canary = os.path.join(tmpdir, "canary.txt")
        canary_content = b"CANARY\n"
        with open(canary, "wb") as f: f.write(canary_content)
        sym = os.path.join(tmpdir, "sym.txt")
        os.symlink(canary, sym)
        try:
            _safe_write(sym, b"NEWDATA\n")
            return _status("edge safe_write symlink", Status.FAIL, "symlink accepted")
        except ReasonCodeError: pass
        cur = open(canary, "rb").read()
        if cur != canary_content:
            return _status("edge safe_write symlink", Status.FAIL, "canary mutated")
        return _status("edge safe_write symlink", Status.PASS,
                       "symlink rejected; canary intact")
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)

def edge_safe_write_rejects_non_regular():
    tmpdir = _collision_test_tmpdir()
    try:
        d = os.path.join(tmpdir, "target_dir")
        os.makedirs(d)
        try:
            _safe_write(d, b"X")
            return _status("edge safe_write non-regular", Status.FAIL,
                           "directory accepted")
        except ReasonCodeError: pass
        return _status("edge safe_write non-regular", Status.PASS,
                       "non-regular rejected")
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)

def edge_safe_write_atomic_publish():
    tmpdir = _collision_test_tmpdir()
    try:
        p = os.path.join(tmpdir, "target.txt")
        with open(p, "wb") as f: f.write(b"OLD")
        try:
            _safe_write(p, b"NEW")
        except ReasonCodeError as e:
            return _status("edge safe_write atomic", Status.FAIL, str(e))
        cur = open(p, "rb").read()
        if cur != b"NEW":
            return _status("edge safe_write atomic", Status.FAIL, f"content {cur!r}")
        return _status("edge safe_write atomic", Status.PASS, "atomic replace OK")
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def _provision_manifest_fixture(tmpdir):
    bid = os.path.join(tmpdir, "BASE_IDENTITY.txt")
    frj = os.path.join(tmpdir, "FRAME_REGISTRY.json")
    out = os.path.join(tmpdir, "RELEASE_MANIFEST.txt")
    with open(bid, "wb") as f:
        f.write(b"test_base_identity_placeholder\n")
    with open(frj, "wb") as f:
        f.write(b"{}\n")
    return bid, frj, out


def edge_manifest_second_run():
    tmpdir = _collision_test_tmpdir()
    try:
        bid, frj, out = _provision_manifest_fixture(tmpdir)
        try:
            _generate_manifest_impl(out, _writer=_safe_write,
                _rollback_writer=_default_rollback_writer,
                _base_identity=bid, _frame_registry=frj)
        except Exception as e:
            return _status("edge manifest second run", Status.FAIL, f"first: {e}")
        try:
            _generate_manifest_impl(out, _writer=_safe_write,
                _rollback_writer=_default_rollback_writer,
                _base_identity=bid, _frame_registry=frj)
        except Exception as e:
            return _status("edge manifest second run", Status.FAIL, f"second: {e}")
        return _status("edge manifest second run", Status.PASS,
                       "two consecutive runs succeeded")
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def edge_manifest_triple_consistency():
    tmpdir = _collision_test_tmpdir()
    try:
        bid, frj, out = _provision_manifest_fixture(tmpdir)
        _generate_manifest_impl(out, _writer=_safe_write,
            _rollback_writer=_default_rollback_writer,
            _base_identity=bid, _frame_registry=frj)
        old_txt  = open(out, "rb").read()
        old_json = open(out + ".json", "rb").read()
        old_sha  = open(out + ".sha256", "rb").read()
        calls = [0]
        def fail_second(path, data):
            calls[0] += 1
            if calls[0] == 2:
                raise IOError("injected: second write fail")
            return _safe_write(path, data)
        try:
            _generate_manifest_impl(out, _writer=fail_second,
                _rollback_writer=_default_rollback_writer,
                _base_identity=bid, _frame_registry=frj)
            return _status("edge manifest triple consistency", Status.FAIL,
                           "injected failure not propagated")
        except Exception:
            pass
        # P2-3 (v35): explicit FAIL returns (safe under python -O).
        cur_txt  = open(out, "rb").read()
        cur_json = open(out + ".json", "rb").read()
        cur_sha  = open(out + ".sha256", "rb").read()
        if cur_txt != old_txt:
            return _status("edge manifest triple consistency", Status.FAIL,
                           "txt not rolled back")
        if cur_json != old_json:
            return _status("edge manifest triple consistency", Status.FAIL,
                           "json not rolled back")
        if cur_sha != old_sha:
            return _status("edge manifest triple consistency", Status.FAIL,
                           "sha not rolled back")
        return _status("edge manifest triple consistency", Status.PASS,
                       "triple rolled back on partial failure")
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def edge_manifest_json_corruption_detected():
    tmpdir = _collision_test_tmpdir()
    try:
        bid, frj, out = _provision_manifest_fixture(tmpdir)
        def corrupting_write(path, data):
            _safe_write(path, data)
            if path.endswith(".json"):
                with open(path, "rb") as f:
                    parsed = json.load(f)
                parsed["order"] = "WRONG_ORDER"
                parsed["artifacts"] = []
                with open(path, "wb") as f:
                    f.write(json.dumps(parsed, indent=2, sort_keys=True).encode() + b"\n")
        try:
            _generate_manifest_impl(out, _writer=corrupting_write,
                _rollback_writer=_default_rollback_writer,
                _base_identity=bid, _frame_registry=frj)
            return _status("edge manifest json corruption", Status.FAIL,
                           "corruption not detected")
        except Exception as e:
            msg = str(e).lower()
            if any(k in msg for k in ("json", "mismatch", "sidecar", "artifacts")):
                return _status("edge manifest json corruption", Status.PASS,
                               "corrupted json detected")
            return _status("edge manifest json corruption", Status.FAIL,
                           f"wrong error: {e}")
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def edge_manifest_rollback_failure():
    tmpdir = _collision_test_tmpdir()
    try:
        bid, frj, out = _provision_manifest_fixture(tmpdir)
        _generate_manifest_impl(out, _writer=_safe_write,
            _rollback_writer=_default_rollback_writer,
            _base_identity=bid, _frame_registry=frj)
        calls = [0]
        def fail_publish(path, data):
            calls[0] += 1
            if calls[0] == 2:
                raise IOError("injected: json publish fail")
            return _safe_write(path, data)
        def fail_rollback(path, data):
            if path.endswith("RELEASE_MANIFEST.txt"):
                raise IOError("injected: text rollback fail")
            return _default_rollback_writer(path, data)
        try:
            _generate_manifest_impl(out, _writer=fail_publish,
                _rollback_writer=fail_rollback,
                _base_identity=bid, _frame_registry=frj)
            return _status("edge manifest rollback failure", Status.FAIL,
                           "no exception raised")
        except ReasonCodeError as e:
            msg = str(e).lower()
            if "rollback incomplete" in msg or "inconsistent" in msg:
                return _status("edge manifest rollback failure", Status.PASS,
                               "combined error raised")
            return _status("edge manifest rollback failure", Status.FAIL,
                           f"wrong error shape: {e}")
        except Exception as e:
            return _status("edge manifest rollback failure", Status.FAIL,
                           f"wrong exception type: {type(e).__name__}: {e}")
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def edge_manifest_real_default_second_run():
    if not (os.path.exists(BASE_IDENTITY_PATH) and
            os.path.exists(FRAME_REGISTRY_JSON)):
        return _status("edge manifest real default 2x", Status.BLOCKED,
                       "mandatory inputs absent")
    tmpdir = tempfile.mkdtemp(prefix="ge1_mdef_")
    try:
        bid = os.path.join(tmpdir, "BASE_IDENTITY.txt")
        frj = os.path.join(tmpdir, "FRAME_REGISTRY.json")
        out = os.path.join(tmpdir, "RELEASE_MANIFEST.txt")
        with open(bid, "wb") as f: f.write(b"test\n")
        with open(frj, "wb") as f: f.write(b"{}\n")
        saved = {
            "RELEASE_MANIFEST_PATH": globals()["RELEASE_MANIFEST_PATH"],
            "BASE_IDENTITY_PATH": globals()["BASE_IDENTITY_PATH"],
            "FRAME_REGISTRY_JSON": globals()["FRAME_REGISTRY_JSON"],
        }
        globals()["RELEASE_MANIFEST_PATH"] = out
        globals()["BASE_IDENTITY_PATH"] = bid
        globals()["FRAME_REGISTRY_JSON"] = frj
        try:
            try:
                generate_manifest()
                generate_manifest()
            except Exception as e:
                return _status("edge manifest real default 2x", Status.FAIL, str(e))
            return _status("edge manifest real default 2x", Status.PASS,
                           "default triple idempotent (temp copy)")
        finally:
            for k, v in saved.items():
                globals()[k] = v
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def _protected_input_paths():
    prot = [os.path.realpath(os.path.abspath(__file__))]
    for p in (BASE_IDENTITY_PATH, RELEASE_MANIFEST_PATH,
              RELEASE_MANIFEST_PATH + ".json", RELEASE_MANIFEST_PATH + ".sha256",
              TEST_LOG_PATH, EVIDENCE_LOG_PATH, VERIFIED_COMPANION_PATH,
              FRAME_REGISTRY_JSON):
        prot.append(os.path.realpath(p))
    for name in _LEGACY_ARTIFACT_BASENAMES:
        prot.append(os.path.realpath(os.path.join(_WIRE_DIR, name)))
    for p in _CANONICAL_FIXTURE_PATHS:
        prot.append(os.path.realpath(os.path.abspath(p)))
    prot.append(os.path.realpath(os.path.abspath(_FALLBACK_GRID_TEXT)))
    cert_path, _ = _find_certified_source()
    if cert_path: prot.append(os.path.realpath(cert_path))
    if _CANONICAL_F["path"]: prot.append(os.path.realpath(_CANONICAL_F["path"]))
    if _CANONICAL_F["staged_path"]: prot.append(os.path.realpath(_CANONICAL_F["staged_path"]))
    return prot


def _assert_no_output_collision(out_path, extra_protected=None):
    return _assert_no_output_collision_impl(
        out_path, extra_protected, default_strs_override=None)


def _assert_no_output_collision_test(out_path, extra_protected=None,
                                     default_triple_override=None):
    return _assert_no_output_collision_impl(
        out_path, extra_protected,
        default_strs_override=default_triple_override)


def _assert_no_output_collision_impl(out_path, extra_protected,
                                     default_strs_override):
    if out_path is None:
        out_path = RELEASE_MANIFEST_PATH
    if default_strs_override is not None:
        default_strs = list(default_strs_override)
    else:
        default_strs = [RELEASE_MANIFEST_PATH,
                        RELEASE_MANIFEST_PATH + ".json",
                        RELEASE_MANIFEST_PATH + ".sha256"]
    outs = [out_path, out_path + ".json", out_path + ".sha256"]
    is_default = (outs == default_strs)

    protected = set(_protected_input_paths())
    if default_strs_override is not None:
        for p in default_strs:
            protected.add(os.path.realpath(os.path.abspath(p)))
    if extra_protected:
        for p in extra_protected:
            protected.add(os.path.realpath(os.path.abspath(p)))

    default_paths_set = (set(os.path.realpath(p) for p in default_strs)
                         if is_default else set())
    protected_ids: Set[Tuple[int,int]] = set()
    for p in protected:
        if is_default and p in default_paths_set:
            continue
        try:
            lst = os.lstat(p)
            if stat.S_ISREG(lst.st_mode):
                protected_ids.add((lst.st_dev, lst.st_ino))
        except OSError:
            pass

    realpaths = [os.path.realpath(o) for o in outs]
    if len(set(realpaths)) != 3:
        raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                              f"output triple aliases: {realpaths}")

    for o in outs:
        try:
            lst = os.lstat(o)
        except FileNotFoundError:
            lst = None
        except OSError as e:
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  f"lstat failed on output {o}: {e}")
        if lst is not None:
            if stat.S_ISLNK(lst.st_mode):
                raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                      f"output path is a symlink: {o}")
            if not stat.S_ISREG(lst.st_mode):
                raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                      f"output path is not a regular file: {o}")
            if (lst.st_dev, lst.st_ino) in protected_ids:
                raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                      f"output aliases protected input via hardlink: {o}")
        rp = os.path.realpath(o)
        if rp in protected:
            if is_default and o in default_strs:
                continue
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  f"output collides with protected input: {o}")


def _artifacts(out_path=None, *, base_identity=None, frame_registry=None):
    if out_path is None: out_path = RELEASE_MANIFEST_PATH
    if base_identity is None: base_identity = BASE_IDENTITY_PATH
    if frame_registry is None: frame_registry = FRAME_REGISTRY_JSON
    arts = [
        {"role": "regenerated-current", "requirement": "MANDATORY",
         "kind": "path", "path": os.path.abspath(__file__)},
        {"role": "derived", "requirement": "MANDATORY",
         "kind": "path", "path": base_identity},
        {"role": "derived", "requirement": "MANDATORY",
         "kind": "path", "path": frame_registry},
    ]
    cert_path, _ = _find_certified_source()
    if cert_path:
        arts.append({"role": "reused-certified", "requirement": "OPTIONAL",
                     "kind": "path", "path": cert_path})
    else:
        arts.append({"role": "reused-certified", "requirement": "OPTIONAL",
                     "kind": "path", "path": _FALLBACK_GRID_TEXT})
    if _CANONICAL_F["staged_path"]:
        staged = _CANONICAL_F["staged_path"]
        basename = os.path.basename(_CANONICAL_F["path"] or staged)
        pin = _pin_for(basename)
        if pin is None:
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  "companion pin not provisioned")
        staged_bytes = _safe_read_staged(staged)
        if staged_bytes != _CANONICAL_F["verified_bytes"]:
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  "staged changed since load")
        h = hashlib.sha256(staged_bytes).hexdigest()
        if h != pin:
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  f"staged hash != pin")
        arts.append({"role": "reused-verified-canonical-companion",
                     "requirement": "MANDATORY",
                     "kind": "snapshot",
                     "basename": os.path.basename(staged),
                     "bytes": staged_bytes})
    arts.extend([
        {"role": "derived", "requirement": "OPTIONAL", "kind": "path", "path": out_path},
        {"role": "derived", "requirement": "OPTIONAL", "kind": "path", "path": out_path + ".json"},
        {"role": "derived", "requirement": "OPTIONAL", "kind": "path", "path": out_path + ".sha256"},
        {"role": "derived", "requirement": "OPTIONAL", "kind": "path", "path": EVIDENCE_LOG_PATH},
    ])
    return arts


def _digests(path):
    b = open(path, "rb").read()
    return len(b), hashlib.md5(b).hexdigest(), hashlib.sha256(b).hexdigest()


def _sanitize_basename(b: str) -> str:
    return b.replace("|", "_").replace("\n", "_").replace("\r", "_")


def _entry_to_manifest_row(entry):
    role = entry["role"]; requirement = entry["requirement"]
    if entry.get("kind") == "snapshot":
        b = entry["bytes"]; basename = _sanitize_basename(entry["basename"])
        sz = len(b)
        m = hashlib.md5(b).hexdigest()
        s = hashlib.sha256(b).hexdigest()
        return (f"{basename} | {sz} | {m} | {s} | {role}",
                {"file": basename, "bytes": sz, "md5": m, "sha256": s,
                 "role": role, "requirement": requirement})
    p = entry["path"]; rp = os.path.realpath(p)
    basename = _sanitize_basename(os.path.basename(p))
    if not os.path.exists(rp):
        if requirement == "MANDATORY":
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  f"mandatory artifact missing: {p}")
        return (f"{basename} | MISSING | MISSING | MISSING | {role}",
                {"file": basename, "status": "MISSING",
                 "role": role, "requirement": requirement})
    sz, m, s = _digests(rp)
    return (f"{basename} | {sz} | {m} | {s} | {role}",
            {"file": basename, "bytes": sz, "md5": m,
             "sha256": s, "role": role, "requirement": requirement})


def _default_rollback_writer(path, data):
    try:
        lst = os.lstat(path)
        if stat.S_ISLNK(lst.st_mode):
            raise IOError(f"rollback target is a symlink: {path}")
        if not stat.S_ISREG(lst.st_mode):
            raise IOError(f"rollback target is non-regular: {path}")
    except FileNotFoundError:
        pass
    with open(path, "wb") as f:
        f.write(data)
    with open(path, "rb") as f:
        if f.read() != data:
            raise IOError(f"rollback read-back mismatch: {path}")


def generate_manifest(out_path=None):
    return _generate_manifest_impl(
        out_path,
        _writer=_safe_write,
        _rollback_writer=_default_rollback_writer,
        _base_identity=BASE_IDENTITY_PATH,
        _frame_registry=FRAME_REGISTRY_JSON)


def _generate_manifest_impl(out_path, *, _writer, _rollback_writer,
                             _base_identity, _frame_registry):
    if out_path is None: out_path = RELEASE_MANIFEST_PATH
    _assert_no_output_collision(out_path)
    rows = []; structured = []; seen = set(); seen_snapshots = set()
    excluded = {os.path.realpath(out_path),
                os.path.realpath(out_path + ".json"),
                os.path.realpath(out_path + ".sha256")}
    for entry in _artifacts(out_path,
                            base_identity=_base_identity,
                            frame_registry=_frame_registry):
        if entry.get("kind") == "snapshot":
            basename = entry["basename"]
            if basename in seen_snapshots: continue
            seen_snapshots.add(basename)
            row, struct = _entry_to_manifest_row(entry)
            rows.append(row); structured.append(struct)
            continue
        p = entry["path"]; rp = os.path.realpath(p)
        if rp in excluded or rp in seen: continue
        seen.add(rp)
        row, struct = _entry_to_manifest_row(entry)
        rows.append(row); structured.append(struct)
    manifest_text = (
        f"# GE1 RELEASE MANIFEST  (spec={SPEC_REV})\n"
        f"# Order: {SPEC_ORDER}\n"
        "# file | bytes | md5 | sha256 | role\n"
        "# release traceability: PENDING (external anchor not verified)\n"
        "# publication semantics: best-effort rollback (non-atomic); "
        "see note S-2 for crash-safe versioned-release guidance\n"
        + "\n".join(rows) + "\n")
    intended_structs = sorted(structured, key=lambda r: r["file"])
    json_text = json.dumps({"spec": SPEC_REV, "order": SPEC_ORDER,
                            "artifacts": intended_structs},
                           indent=2, sort_keys=True) + "\n"
    sha_text = hashlib.sha256(manifest_text.encode()).hexdigest() + "\n"

    paths = [out_path, out_path + ".json", out_path + ".sha256"]
    contents = [manifest_text.encode(), json_text.encode(), sha_text.encode()]

    for p in paths:
        try:
            lst = os.lstat(p)
            if stat.S_ISLNK(lst.st_mode):
                raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                      f"refusing to write over symlink: {p}")
            if not stat.S_ISREG(lst.st_mode):
                raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                      f"refusing to overwrite non-regular: {p}")
        except FileNotFoundError:
            pass

    backups = {}; existed = {}
    for p in paths:
        try:
            with open(p, "rb") as f:
                backups[p] = f.read()
            existed[p] = True
        except FileNotFoundError:
            backups[p] = None; existed[p] = False

    try:
        for i, p in enumerate(paths):
            _writer(p, contents[i])

        with open(paths[0], "rb") as f: final_txt_bytes = f.read()
        with open(paths[1], "rb") as f: final_json_bytes = f.read()
        with open(paths[2], "rb") as f: final_sha_bytes = f.read()
        if final_txt_bytes != contents[0]:
            raise IOError("post-publish: text manifest bytes != intended")
        if final_json_bytes != contents[1]:
            raise IOError("post-publish: json manifest bytes != intended")
        if final_sha_bytes != contents[2]:
            raise IOError("post-publish: sha256 sidecar bytes != intended")
        if hashlib.sha256(final_txt_bytes).hexdigest() != final_sha_bytes.decode().strip():
            raise IOError("post-publish sidecar hash != text hash")
        try:
            parsed = json.loads(final_json_bytes.decode("utf-8"))
        except Exception as e:
            raise IOError(f"post-publish json invalid: {e}")
        if parsed.get("spec") != SPEC_REV:
            raise IOError("post-publish json spec mismatch")
        if parsed.get("order") != SPEC_ORDER:
            raise IOError("post-publish json order mismatch")
        if parsed.get("artifacts") != intended_structs:
            raise IOError("post-publish json artifacts mismatch (full rows)")

    except Exception as publish_err:
        rollback_failures = []
        for p in paths:
            try:
                if existed[p]:
                    _rollback_writer(p, backups[p])
                else:
                    if os.path.exists(p):
                        lst = os.lstat(p)
                        if stat.S_ISLNK(lst.st_mode) or not stat.S_ISREG(lst.st_mode):
                            raise IOError(
                                f"refusing to unlink non-regular: {p}")
                        os.unlink(p)
            except Exception as rb_err:
                rollback_failures.append((p, f"{type(rb_err).__name__}: {rb_err}"))
        if rollback_failures:
            details = "; ".join(f"{os.path.basename(p)}: {e}"
                                for p, e in rollback_failures)
            raise ReasonCodeError(
                ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                f"publication failed ({type(publish_err).__name__}: "
                f"{publish_err}); rollback incomplete — release artifacts "
                f"are INCONSISTENT: {details}")
        raise


def _pin_frames_mode(force_repin=False, out_path=None):
    """P2-2 (v35): write target defaults to OUTPUT_DIR; --out overrides;
    warn if resolved into the agent dir (pollution)."""
    print("=" * 78)
    print("GE1 v37 — one-shot FRAME_REGISTRY pin (§3.1 MD5 verified)")
    print("=" * 78)
    if out_path is None:
        out_path = FRAME_REGISTRY_WRITE_DEFAULT
    abs_out = os.path.abspath(out_path)
    try:
        wire_real = os.path.realpath(_WIRE_DIR)
        out_dir_real = os.path.realpath(os.path.dirname(abs_out))
        writing_to_agent_dir = (out_dir_real == wire_real)
    except OSError:
        writing_to_agent_dir = False
    if writing_to_agent_dir:
        print(f"⚠️  WARNING: write target is inside the agent dir: {abs_out}")
        print(f"    This will pollute the ARC-AGI-3 agent package.")
        print(f"    Recommended: --out <scratch-path> or GE1_FRAME_REGISTRY_OUT")
    if os.path.exists(abs_out) and not force_repin:
        print(f"FAIL: {abs_out} already exists. Pass --force-repin.")
        return 1
    try: import arcengine
    except ImportError: print("FAIL: arcengine unavailable"); return 1
    try: ver = importlib_metadata.version("arcengine")
    except Exception: ver = None
    if ver != ARCENGINE_VERSION:
        print(f"FAIL: arcengine {ver}"); return 1
    frames, errors, ids = _load_real_frames_with_count()
    print(f"loaded {len(frames)} frames; {len(errors)} errors")
    if len(frames) != 25 or len(set(ids)) != 25:
        print(f"FAIL: need 25 unique frames, got {len(frames)}/{len(set(ids))}")
        return 1
    if set(ids) != CANONICAL_FRAME_IDS:
        missing = sorted(CANONICAL_FRAME_IDS - set(ids))
        extra = sorted(set(ids) - CANONICAL_FRAME_IDS)
        print(f"FAIL: IDs != canonical: missing={missing} extra={extra}")
        return 1
    mismatches = []
    for gid, g in zip(ids, frames):
        actual = hashlib.md5(_canonical_grid_bytes(g)).hexdigest()[:9]
        if actual != REPORT_25_MD5_PREFIX[gid]:
            mismatches.append(f"{gid}: {actual} != {REPORT_25_MD5_PREFIX[gid]}")
    if mismatches:
        print(f"FAIL: §3.1 MD5 prefix mismatches:")
        for m in mismatches[:5]: print(f"  {m}")
        return 1
    reg = {gid: hashlib.sha256(_canonical_grid_bytes(g)).hexdigest()
           for g, gid in zip(frames, ids)}
    reg_bytes = json.dumps(reg, indent=2, sort_keys=True).encode()
    print(f"[pin] writing {len(reg_bytes)} bytes → {abs_out}")
    try:
        _safe_write(abs_out, reg_bytes)
    except ReasonCodeError as e:
        print(f"FAIL: {e}"); return 1
    print(f"OK: wrote {abs_out} ({len(reg)} entries)")
    return 0


def _check_env_mode():
    print("=" * 78)
    print("GE1 v37 — environment readiness check")
    print("=" * 78)
    print(f"spec={SPEC_REV}  order={SPEC_ORDER}")
    print(f"wire_dir={_WIRE_DIR}")
    print(f"output_dir={OUTPUT_DIR}")
    print(f"fixture_F: {_CANONICAL_F['provenance']}\n")
    rows = []
    base, err = _find_certified_source()
    if base is None:
        rows.append(("certified grid_text.py", "MISSING", err))
    else:
        b = open(base, "rb").read()
        ok = (len(b) == CERTIFIED_BYTES
              and hashlib.md5(b).hexdigest() == CERTIFIED_MD5
              and hashlib.sha256(b).hexdigest() == CERTIFIED_SHA256)
        rows.append(("certified grid_text.py", "OK" if ok else "MISMATCH",
                     f"{len(b)}B md5={hashlib.md5(b).hexdigest()[:8]}…"))
    reg, reg_err = _load_frame_registry_diag()
    if reg_err is not None:
        rows.append(("FRAME_REGISTRY.json", "CORRUPT", reg_err[:120]))
    elif not os.path.exists(FRAME_REGISTRY_JSON):
        rows.append(("FRAME_REGISTRY.json", "MISSING", "run --pin-frames --out <path>"))
    elif set(reg.keys()) != CANONICAL_FRAME_IDS:
        rows.append(("FRAME_REGISTRY.json", "MISMATCH", "keys != canonical"))
    else:
        bad = [k for k, v in reg.items() if not _HEX64_RE.fullmatch(v or "")]
        if bad: rows.append(("FRAME_REGISTRY.json", "INVALID", f"{len(bad)} malformed"))
        else: rows.append(("FRAME_REGISTRY.json", "OK", "25 canonical IDs"))
    try:
        ver = importlib_metadata.version("arcengine")
        rows.append(("arcengine", "OK" if ver == ARCENGINE_VERSION else "MISMATCH", f"{ver}"))
    except Exception:
        rows.append(("arcengine", "MISSING", "not installed"))
    tj = _resolve_tokenizer_json()
    if tj is None:
        rows.append(("tokenizer.json", "MISSING", "not locatable"))
    else:
        h = hashlib.sha256(open(tj, "rb").read()).hexdigest()
        rows.append(("tokenizer.json",
                     "OK" if h == TOKENIZER_JSON_SHA256 else "MISMATCH",
                     f"sha256={h[:12]}…"))
    if _CANONICAL_F["grid"] is not None:
        try:
            staged = _CANONICAL_F["staged_path"]
            if not staged:
                raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE, "staged missing")
            staged_bytes = _safe_read_staged(staged)
            if staged_bytes != _CANONICAL_F["verified_bytes"]:
                raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE, "staged != snapshot")
            basename = os.path.basename(_CANONICAL_F["path"])
            pin = _pin_for(basename)
            if pin is None:
                raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE, "pin not provisioned")
            h = hashlib.sha256(staged_bytes).hexdigest()
            if h != pin:
                raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE, "staged hash != pin")
            rows.append(("fixture F (ring)", "OK", _CANONICAL_F["provenance"]))
        except ReasonCodeError as e:
            rows.append(("fixture F (ring)", "INVALID", str(e)))
    else:
        rows.append(("fixture F (ring)", "BLOCKED", _CANONICAL_F["provenance"]))
    rows.append(("production cap H", "EXTERNAL", "deployment owner — FINAL-22"))
    rows.append(("authoritative state census", "EXTERNAL", "F-06/F-12 SDK confirmation"))
    rows.append(("GE2 input adapters", "OUT_OF_SCOPE", "F-13/F-27"))
    for name, st, detail in rows:
        print(f"{st:14}  {name:28}  {detail}")
    blockers = [r for r in rows if r[1] in ("MISSING","MISMATCH","INVALID","INCOMPLETE","CORRUPT","BLOCKED")]
    print("-" * 78); print(f"blockers={len(blockers)}")
    if blockers: print("RESULT: NOT READY"); return 2
    print("RESULT: ENV READY"); return 0


def run_all():
    _TS12_CACHE["result"] = None
    tests = [
        kat_gf01_t1_bytes_md5, kat_gf02_t1_bytes_md5,
        kat_gf02_t2_bytes_md5, kat_gf02_t3_4_bytes_md5,
        kat_gf01_t2_t3_erratum, kat_gf02_t3_relations, kat_tokens,
        ts_spatial_01_determinism, ts_spatial_02_full_oracle,
        ts_spatial_03_256_tile, ts_spatial_09a_byte_determinism,
        _ts12_with_cache,
        ts_spatial_10_25frame_relation_oracle,
        ts_spatial_08_lossless_real_frames, ts_spatial_13_pair_budget_enforcement,
        ts_reject_interface, ts_parse_structure_oracle, ts_objects_roundtrip,
        ts_relation_laws_fuzz, ts_canonicality_suite,
        edge_grid_bool_reject, edge_grid_float_reject, edge_grid_range_reject,
        edge_rle_helper_bool, edge_rle_oom, edge_bg_bool, edge_state_unhashable,
        edge_trailing_nl, edge_parser_mutations, edge_H_bbox_directed, edge_rev_left,
        edge_relations_block_strict, edge_symmetric_i_lt_j,
        edge_relation_endpoint_out_of_range,
        edge_objects_size_mismatch, edge_objects_bbox_mismatch,
        edge_objects_bg_colour, edge_objects_leading_zero,
        edge_object_bbox_not_extrema, edge_object_duplicate_cell,
        edge_object_disconnected, edge_object_cross_shared,
        edge_object_same_colour_adjacent, edge_object_same_colour_private_ok,
        edge_object_not_canonical_order, edge_object_different_colour_adjacent_ok,
        edge_object_forged_first_cell, edge_object_first_cell_true_value_ok,
        edge_object_first_cell_explicit_none,
        edge_modal_bg_mismatch,
        edge_k_bool_reject, edge_encode_grid_full_validation,
        edge_object_validation_bad_bbox, edge_object_id_order,
        edge_F_negative_control_all_W, edge_F_negative_control_3x3_ring,
        edge_F_positive_control_5x5_ring, edge_F_negative_control_single_object,
        edge_F_signature_not_uniquely_canonical, edge_F_parse_content_only,
        edge_F_toctou_manifest_detected,
        edge_F_snapshot_manifest_digest,
        edge_F_manifest_end_to_end,
        edge_F_staged_symlink_after_publish,
        edge_F_staged_nonregular_after_publish,
        edge_F_staged_fifo_after_publish,
        edge_collision_default_allowed,
        edge_collision_sidecar_as_primary,
        edge_collision_staged_path_as_output,
        edge_collision_fallback_grid_text,
        edge_collision_test_log,
        edge_collision_distinct_triple,
        edge_collision_symlink_alias_to_default,
        edge_collision_symlink_at_default_path,
        edge_collision_default_sidecar_symlink,
        edge_collision_hardlink_alias,
        edge_collision_sidecar_hardlink,
        edge_collision_default_triple_existing,
        edge_collision_default_hardlinked_to_protected,
        edge_manifest_second_run,
        edge_manifest_triple_consistency,
        edge_manifest_json_corruption_detected,
        edge_manifest_rollback_failure,
        edge_manifest_real_default_second_run,
        edge_generate_manifest_no_injection_params,
        edge_evidence_log_rejects_source,
        edge_evidence_log_rejects_protected,
        edge_evidence_log_rejects_sidecars,
        edge_evidence_log_rejects_hardlink_to_source,
        edge_evidence_log_toctou_symlink_swap,
        edge_safe_write_rejects_symlink,
        edge_safe_write_rejects_non_regular,
        edge_safe_write_atomic_publish,
        edge_fixture_F_status, edge_frame_id_set_equal, edge_registry_hex_validation,
        edge_cwd_independence, edge_frame_ids_are_letters, edge_module_init_state,
        edge_spec_lock,
    ]
    results = []
    for fn in tests:
        try: results.append(with_timeout(120, fn))
        except TimeoutError as e: results.append(_status(fn.__name__, Status.FAIL, f"timeout: {e}"))
        except Exception as e: results.append(_status(fn.__name__, Status.FAIL, f"{type(e).__name__}: {e}"))
    return results


def main(argv=None):
    if argv is None: argv = sys.argv[1:]
    force_repin = "--force-repin" in argv
    if "--pin-frames" in argv:
        out_override = None
        try:
            idx = argv.index("--out")
            if idx + 1 < len(argv):
                out_override = argv[idx + 1]
        except ValueError:
            pass
        sys.exit(_pin_frames_mode(force_repin, out_override))
    if "--check-env" in argv: sys.exit(_check_env_mode())
    print("=" * 78)
    print(f"GE1 v37 — spec={SPEC_REV}  order={SPEC_ORDER}")
    print(f"wire_dir={_WIRE_DIR}  output_dir={OUTPUT_DIR}")
    print("=" * 78)
    results = run_all()
    for name, st, detail in results:
        print(f"{st.value:7}  {name:44}  {detail}")
        _evidence({"test": name, "result": st.value, "detail": detail, "spec": SPEC_REV})
    n_fail = sum(1 for _, s, _ in results if s is Status.FAIL)
    n_blocked = sum(1 for _, s, _ in results if s is Status.BLOCKED)
    print("-" * 78)
    print(f"PASS={len(results)-n_fail-n_blocked}  FAIL={n_fail}  BLOCKED={n_blocked}")
    evidence_state = "NOT_EMITTED"
    try:
        _emit_evidence_log(); evidence_state = "GENERATED"
        print(f"[evidence] {EVIDENCE_LOG_PATH}  (read-back verified)")
    except Exception as e:
        evidence_state = "FAILED"; print(f"[evidence] FAIL: {e}")
    manifest_state = "NOT_EMITTED"
    if n_fail == 0 and n_blocked == 0:
        try:
            generate_manifest(); manifest_state = "GENERATED"
            print(f"[manifest] {RELEASE_MANIFEST_PATH} + .json + .sha256 emitted")
        except Exception as e:
            manifest_state = "FAILED"; print(f"[manifest] FAIL: {e}")
    else:
        try:
            log_text = f"# spec={SPEC_REV}\n"
            for name, st, detail in results:
                log_text += f"{st.value:7}  {name:44}  {detail}\n"
            _safe_write(TEST_LOG_PATH, log_text)
            print(f"[manifest] NOT emitted → {TEST_LOG_PATH}")
        except Exception as e: print(f"[test-log] FAIL: {e}")
    print("-" * 78)
    print(f"FINAL: PASS={len(results)-n_fail-n_blocked} FAIL={n_fail} "
          f"BLOCKED={n_blocked} EVIDENCE={evidence_state} MANIFEST={manifest_state}")
    print("NOTE: release traceability PENDING (external anchor not verified)")
    print("NOTE: publication semantics = best-effort rollback (non-atomic; see S-2)")
    print(f"NOTE: all artifacts written to OUTPUT_DIR={OUTPUT_DIR} (agent dir read-only)")
    if n_fail or manifest_state == "FAILED" or evidence_state == "FAILED":
        print("RESULT: FAILED"); sys.exit(1)
    if n_blocked:
        print("RESULT: SUITE INCOMPLETE — BLOCKED present"); sys.exit(2)
    if manifest_state != "GENERATED" or evidence_state != "GENERATED":
        print("RESULT: SUITE INCOMPLETE — outputs not fully emitted"); sys.exit(2)
    print("RESULT: ALL GE1 v37 TESTS — PASS")
    sys.exit(0)


_try_load_canonical_F()

if _CANONICAL_F["grid"] is not None:
    _canonical_F_grid = _CANONICAL_F["grid"]
    FIXTURES["F"] = lambda: [row[:] for row in _canonical_F_grid]
else:
    FIXTURES_BLOCKED["F"] = (
        f"canonical companion unavailable ({_CANONICAL_F['provenance']})")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile goa/agent/ge2/ge2_adapters.py
# ============================================================================
# GE2 v2.0 — Input Adapters (MAPPING ONLY, no authentication)
# Order: GoA/INDUS/DIR/2026/096 | Architecture: ARCH/2026/103-GridEncoding-GE1-4
# Basis: Research Report DIR/2026/092 Rev v38 (§5.0, F-12..F-20, F-28)
#
# ============================================================================
# INTERFACE CONTRACT — READ BEFORE INTEGRATING WITH GE4
# ============================================================================
#
# GE2 IS A MAPPING LAYER. IT DOES NOT AUTHENTICATE ANYTHING.
#
# GE2 is responsible for:
#   ✓ Structural validation of frame shape (single 64x64, int 0..15 cells)
#   ✓ Structural validation of state mapping (4 tokens, identity mapping)
#   ✓ Structural validation of action payload (schema match, shape)
#   ✓ Path safety: reject symlink components + path traversal + TOCTOU
#   ✓ SDN vocabulary check (name/ID vs bound GameAction enum, if available)
#   ✓ F-17 error classification (4 classes, distinct)
#   ✓ F-28 per-step bound (3 strikes, exact)
#   ✓ Refusal visibility (all rejections return caller-visible codes)
#
# GE2 is NOT responsible for — these MUST be done by GE4 BEFORE calling GE2:
#   ✗ Authenticating the census (verify mapping came from pinned SDK)
#   ✗ Authenticating the contract (verify ENV_OWNER actually confirmed)
#   ✗ Authenticating the pinset (verify signed/indexed origin)
#   ✗ Verifying SDK module hashes (arcengine integrity)
#   ✗ Verifying SDK version pinning
#   ✗ Detecting TOCTOU races on the filesystem (process isolation)
#   ✗ Measuring T at the actual dispatch boundary
#   ✗ Validating FrameData receipt from perform_action()
#   ✗ Tracking RunState / retry / progression
#   ✗ Emitting release manifests or artifacts
#
# WHY: A single-file, single-process Python module cannot provide
# unforgeable authentication. Any "secret" in this module is reachable via
# import, subclassing, __new__, sys.modules replacement, or direct attribute
# override. Any "signed" field is a caller-set value. GE2's structural
# checks are SCHEMA checks, not cryptographic authentication.
#
# The 13-round audit pattern showed: every attempt to make GE2 authenticate
# moved the trust gap from one forgeable label to another (dict["authenticated"],
# dict["source"] == "ENV_OWNER", caller-supplied pinned_hashes). Path B
# stops claiming. GE4 (separate artifact) provides the actual security
# boundary via process isolation, real SDK verification, and signed pinsets.
#
# If GE4 skips its authentication, the resulting mapping is only as trusted
# as GE4's authentication. This is GE4's bug, not GE2's.
#
# ============================================================================
# INTERFACE — for GE4 callers
# ============================================================================
#
#   adapt_frame(observation, enumerate_grids=fn) -> AdapterResult
#       fn: observation -> list[grid]. GE2 validates exactly one 64x64 grid.
#
#   adapt_state(sdk_state, census={"mapping": {tok: tok, ...}}) -> AdapterResult
#       mapping: MUST be 4-token identity mapping.
#       GE4 MUST have authenticated the census. GE2 validates only mapping
#       structure (tokens present, identity, version/allowlist metadata).
#
#   adapt_action(reasoner_action, *, game_id, catalogue_dir,
#                pinned_hashes, contract, harness, step_state,
#                t_measure=None, h_limit=None, is_feasible_set_empty=None)
#                -> AdapterResult
#       pinned_hashes: {fname: sha256}. GE4 MUST have authenticated this
#           dict (signed index, verified origin). GE2 only checks hash match
#           + path safety (symlink/traversal/TOCTOU).
#       contract: ContractConfirmed. GE4 MUST issue after ENV_OWNER
#           verification. GE2 only checks structural fields.
#       t_measure: callable -> Optional[int]. GE2 uses for F-28 re-query
#           priority (pre-mapping T/H check). GE4 re-measures at dispatch.
#       h_limit: authenticated authoritative H. GE4 MUST supply.
#
#   check_hard_limit(T, H) -> RefusalResult
#       GE4 measures T at the actual dispatch boundary.
#       GE2 applies the rule: T>=H => refuse, T<H => allow, T unmeasurable
#       => refuse. Rejects bool/float/wrong-type T/H.
#
# ============================================================================
# v2.0 FIXES (from v1.0 AGA findings — GENUINE GE2 bugs, not auth claims):
#   P1-01: _safe_catalogue_path now fd-relative no-follow open (TOCTOU-safe)
#   P1-04: catalogue name/ID validated against bound SDK GameAction (when
#          available); GE4 owns validation when SDK not bound
#   P1-05: H_UNCONFIGURED tag now PENDING_AUTHORITATIVE_H (was census-partial)
#
# v2.0 REMOVES ALL AUTHENTICATION CLAIMS present in v1.0/v10.x:
#   - No "authenticated" field in census validation
#   - No "source" trust in census validation
#   - No "complete" trust flag in census validation
#   - No TRUSTED_CENSUS_SOURCES allowlist
#   - No "trusted root" claim in path handling (caller owns root integrity)
#   - No "pinset is trusted" claim (GE4 supplies authenticated pinset)
#   - No _INTERNAL_TEST_RUNNER (tests are pure unit tests, no process-global)
# ============================================================================

from __future__ import annotations
import sys
sys.dont_write_bytecode = True

import os, re, json, hashlib, stat, shutil
from enum import Enum
from dataclasses import dataclass
from typing import Optional, List, Dict, Any, Tuple, Callable

# ============================================================================
# CONSTANTS
# ============================================================================
GE2_VERSION = "v3.0"
SPEC_REV = "DIR092 Rev v38"
ARCH_REF = "ARCH/2026/103-GridEncoding-GE1-4"
BUILD_ORDER = "GoA/INDUS/DIR/2026/096"
AUDIT_ORDER = "GoA/INDUS/DIR/2026/097"

STATETOK_ALLOWLIST = frozenset({"NOT_PLAYED", "NOT_FINISHED", "WIN", "GAME_OVER"})
PER_STEP_BOUND = 3
COORDINATE_ORIGIN = "TOP-LEFT"
COORDINATE_DIRECTION = "COLUMN_X_ROW_Y"

T_UNMEASURABLE_CLASS = "REFUSAL_EVENT"
CONFLICT_TAG_T_UNMEASURABLE = "N2_T_UNMEASURABLE_CLASS"
CONFLICT_TAG_H_PENDING = "PENDING_AUTHORITATIVE_H"   # P1-05 fix

ENV_OWNER_INVERSE_CONFIRMED = False
DERIVED_INVERSE_CONFIRMED = False

ALLOWED_PAYLOAD_SCHEMAS = frozenset({"NONE", "XY_PAIR", "GRID_RC"})
ALLOWED_COORD_SPACES = frozenset({"NONE", "GRID_CELL", "DISPLAY_CLICK"})
ALLOWED_COMBINATIONS = frozenset({
    ("NONE", "NONE"),
    ("GRID_RC", "GRID_CELL"),
    ("XY_PAIR", "DISPLAY_CLICK"),
})

GAME_ID_RE = re.compile(r"^[A-Za-z0-9_-]{1,32}$")
SHA256_RE = re.compile(r"^[0-9a-f]{64}$")


def _is_int(v) -> bool:
    return type(v) is int


def _is_str(v) -> bool:
    return type(v) is str


def _is_bool(v) -> bool:
    return type(v) is bool


# ============================================================================
# ENUMS
# ============================================================================
class ReasonCode(str, Enum):
    UNSUPPORTED_INPUT            = "UNSUPPORTED_INPUT"
    ENVIRONMENT_CONTRACT_FAILURE = "ENVIRONMENT_CONTRACT_FAILURE"
    INVALID_REASONER_ACTION      = "INVALID_REASONER_ACTION"
    REFUSAL_EVENT                = "REFUSAL_EVENT"


class DetailCode(str, Enum):
    # Frame/grid
    UNSUPPORTED_MULTIGRID           = "UNSUPPORTED_MULTIGRID"
    UNSUPPORTED_DIMENSION           = "UNSUPPORTED_DIMENSION"
    MALFORMED_OBSERVATION           = "MALFORMED_OBSERVATION"
    HARNESS_BINDING_PENDING         = "HARNESS_BINDING_PENDING"
    # State
    UNKNOWN_STATE_TOKEN             = "UNKNOWN_STATE_TOKEN"
    OUT_OF_ALLOWLIST_STATE          = "OUT_OF_ALLOWLIST_STATE"
    CENSUS_INCOMPLETE               = "CENSUS_INCOMPLETE"
    CENSUS_SCHEMA_VIOLATION         = "CENSUS_SCHEMA_VIOLATION"   # P2-03
    # Catalogue
    GAME_CATALOGUE_MISSING          = "GAME_CATALOGUE_MISSING"
    CATALOGUE_UNPINNED              = "CATALOGUE_UNPINNED"
    CATALOGUE_STRUCTURE_INVALID     = "CATALOGUE_STRUCTURE_INVALID"
    CATALOGUE_SDK_MISMATCH          = "CATALOGUE_SDK_MISMATCH"    # P1-04
    PATH_TRAVERSAL                  = "PATH_TRAVERSAL"             # P1-01
    CONTRACT_NOT_AUTHORITATIVE      = "CONTRACT_NOT_AUTHORITATIVE"
    ACTION_CONTRACT_UNCONFIRMED     = "ACTION_CONTRACT_UNCONFIRMED"
    # Action
    UNKNOWN_ACTION_ID               = "UNKNOWN_ACTION_ID"
    UNKNOWN_ACTION_NAME             = "UNKNOWN_ACTION_NAME"
    IDENTITY_CONFLICT               = "IDENTITY_CONFLICT"
    PAYLOAD_ALIAS_CONFLICT          = "PAYLOAD_ALIAS_CONFLICT"
    MISSING_PAYLOAD                 = "MISSING_PAYLOAD"
    MALFORMED_PAYLOAD               = "MALFORMED_PAYLOAD"
    PAYLOAD_SCHEMA_MISMATCH         = "PAYLOAD_SCHEMA_MISMATCH"
    CATALOGUE_COMBINATION_INVALID   = "CATALOGUE_COMBINATION_INVALID"
    # Coordinate
    UNMAPPED_DISPLAY_COORDINATE     = "UNMAPPED_DISPLAY_COORDINATE"
    COORDINATE_ROUNDTRIP_MISMATCH   = "COORDINATE_ROUNDTRIP_MISMATCH"
    # Refusal
    H_UNCONFIGURED                  = "H_UNCONFIGURED"
    T_UNMEASURABLE                  = "T_UNMEASURABLE"
    HARD_LIMIT_EXCEEDED             = "HARD_LIMIT_EXCEEDED"
    EMPTY_FEASIBLE_SET              = "EMPTY_FEASIBLE_SET"
    CALLBACK_FAILED                 = "CALLBACK_FAILED"
    # Meta
    RUN_ALREADY_TERMINAL            = "RUN_ALREADY_TERMINAL"
    UNEXPECTED_EXCEPTION            = "UNEXPECTED_EXCEPTION"


class Status(str, Enum):
    PASS    = "PASS"
    FAIL    = "FAIL"
    BLOCKED = "BLOCKED"


class ReasonCodeError(ValueError):
    def __init__(self, code, detail, msg="", *, open_conflict=None):
        self.code = code
        self.detail = detail
        self.open_conflict = open_conflict
        super().__init__(f"{code.value}/{detail.value}: {msg}")


# ============================================================================
# RESULT TYPES (all caller-owned, no module state)
# ============================================================================
@dataclass
class AdapterResult:
    """F-17 log record. Caller persists; GE2 does not."""
    ok: bool
    value: Any = None
    code: Optional[ReasonCode] = None
    detail: Optional[DetailCode] = None
    message: str = ""
    open_conflict: Optional[str] = None
    strike_counted: bool = False
    step_failed: bool = False
    routed_to: Optional[str] = None

    @classmethod
    def success(cls, value):
        return cls(ok=True, value=value)

    @classmethod
    def fail(cls, code, detail, msg="", **kw):
        return cls(ok=False, code=code, detail=detail, message=msg, **kw)


@dataclass
class StepState:
    """F-28 per-step bound (3-strike model failure).
    Caller (GE4) owns the instance lifetime; GE2 mutates it on strike."""
    observation_step: int = 0
    invalid_action_count: int = 0
    step_failed: bool = False

    @property
    def per_step_bound(self) -> int:
        return PER_STEP_BOUND

    def record_invalid_action(self) -> bool:
        self.invalid_action_count += 1
        if self.invalid_action_count >= PER_STEP_BOUND:
            self.step_failed = True
            return True
        return False

    def new_step(self, step_id: int):
        self.observation_step = step_id
        self.invalid_action_count = 0
        self.step_failed = False


@dataclass
class ContractConfirmed:
    """GE4-issued contract. GE4 MUST authenticate before creating this;
    GE2 only checks structural fields (origin/direction/flags)."""
    grid_cell_swap_confirmed: bool = False
    display_click_confirmed: bool = False
    spaces_confirmed_coincide: bool = False
    origin: str = COORDINATE_ORIGIN
    direction: str = COORDINATE_DIRECTION
    source: str = "UNCONFIRMED"


# ============================================================================
# SDK VOCABULARY BINDING (not authentication — vocabulary only)
# ============================================================================
def _bind_sdk_vocabulary() -> Optional[Dict[str, Any]]:
    """
    Import arcengine for vocabulary. GE2 uses GameAction/GameState/Camera
    for STRUCTURAL checks (does this name match an SDK enum member?).
    GE2 does NOT verify module hashes or version — that is GE4's
    authentication step. If a tampered SDK is installed, GE2 will validate
    against the tampered enum; GE4 must catch this via hash pinning.
    """
    try:
        import arcengine
        import arcengine.enums as _en
        import arcengine.camera as _cam
    except ImportError:
        return None
    return {
        "GameState": getattr(_en, "GameState", None),
        "GameAction": getattr(_en, "GameAction", None),
        "Camera": getattr(_cam, "Camera", None),
        "version_installed": getattr(arcengine, "__version__", "unknown"),
    }


_SDK_BOUND: Optional[Dict[str, Any]] = _bind_sdk_vocabulary()


# ============================================================================
# FRAME ADAPTER — structural validation only
# ============================================================================
# REPORT DIR/092 §5.0 TRACEABILITY (each report item -> its GE2 function):
#   F-12  state normalization        -> adapt_state            (canonical allowlist)
#   F-13  frame/grid adapter         -> adapt_frame            (single 64x64; reject multi/non-64)
#   F-14  action/coordinate mapping  -> adapt_action           (+ load_game_catalogue)
#   F-15  state-token reconciliation -> the canonical state token (NOT_FINISHED)
#   F-17  error taxonomy             -> ReasonCode / DetailCode
#   F-20  coordinate-space contract  -> adapt_action (COORDINATE_ORIGIN/DIRECTION, spaces)
#   F-27  non-retryable unsupported input -> UNSUPPORTED_INPUT (no retry)
#   F-28  per-step bound + disposition -> StepState (PER_STEP_BOUND=3)
# ARCH/2026/103 §3 interface: adapt(observation)->(grid,header) ; map_action ; classify
# ============================================================================
def adapt_frame(observation, *,
                enumerate_grids: Optional[Callable] = None) -> AdapterResult:
    """Validate observation yields exactly one 64x64 grid with int 0..15 cells."""
    try:
        return _adapt_frame_impl(observation, enumerate_grids)
    except ReasonCodeError as e:
        return AdapterResult.fail(e.code, e.detail, str(e),
                                  open_conflict=e.open_conflict)
    except Exception as e:
        return AdapterResult.fail(
            ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
            DetailCode.UNEXPECTED_EXCEPTION,
            f"{type(e).__name__}: {e}", open_conflict="N-1")


def _adapt_frame_impl(observation, enumerate_grids) -> AdapterResult:
    if enumerate_grids is None:
        raise ReasonCodeError(
            ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
            DetailCode.HARNESS_BINDING_PENDING,
            "observation enumerator not bound",
            open_conflict="C-3/D8")

    try:
        grids = enumerate_grids(observation)
    except Exception as e:
        raise ReasonCodeError(
            ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
            DetailCode.CALLBACK_FAILED,
            f"enumerate_grids raised {type(e).__name__}: {e}")

    if not isinstance(grids, (list, tuple)):
        raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                              DetailCode.MALFORMED_OBSERVATION,
                              "enumerator returned non-sequence")
    if len(grids) == 0:
        raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                              DetailCode.MALFORMED_OBSERVATION,
                              "no grid in observation")
    if len(grids) > 1:
        raise ReasonCodeError(ReasonCode.UNSUPPORTED_INPUT,
                              DetailCode.UNSUPPORTED_MULTIGRID,
                              f"multi-grid ({len(grids)} grids); OUT OF SCOPE (F-13)")

    grid = grids[0]
    if not _is_valid_64x64(grid):
        raise ReasonCodeError(ReasonCode.UNSUPPORTED_INPUT,
                              DetailCode.UNSUPPORTED_DIMENSION,
                              "grid is not exactly 64x64")

    _validate_cells(grid)
    return AdapterResult.success([list(row) for row in grid])


def _is_valid_64x64(grid) -> bool:
    if not isinstance(grid, list) or len(grid) != 64:
        return False
    for row in grid:
        if not isinstance(row, list) or len(row) != 64:
            return False
    return True


def _validate_cells(grid):
    for r, row in enumerate(grid):
        for c, v in enumerate(row):
            if type(v) is not int or not (0 <= v <= 15):
                raise ReasonCodeError(
                    ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                    DetailCode.MALFORMED_OBSERVATION,
                    f"cell ({r},{c}) must be int 0..15; got {type(v).__name__} {v!r}")


# ============================================================================
# STATE ADAPTER — structural validation only (no auth claims)
# ============================================================================
def adapt_state(sdk_state, *, census: Dict[str, Any]) -> AdapterResult:
    """
    Map SDK state to canonical token using the supplied census's mapping.

    GE2 validates census STRUCTURE only:
      ✓ census["mapping"] is a dict
      ✓ All 4 STATETOK_ALLOWLIST tokens present in mapping
      ✓ Identity mapping (base_key == value, e.g. NOT_FINISHED -> NOT_FINISHED)
      ✓ If census["version"] present: must be "v1"
      ✓ If census["declared_allowlist"] present: must equal sorted canonical
      ✓ If census["missing"] present: must equal [] (no tokens actually missing)

    GE2 does NOT validate (GE4's responsibility before calling):
      ✗ census["authenticated"]  — GE4 must authenticate
      ✗ census["source"]         — GE4 must verify
      ✗ census["complete"]       — GE4 must verify
      ✗ census["confirmed"]      — GE4 must confirm

    A caller can forge any dict. If GE4 does not authenticate the census
    before calling, the resulting mapping is only as trusted as GE4's
    authentication. This is GE4's bug, not GE2's.
    """
    try:
        _validate_census_structure(census)
        base = _state_base(sdk_state)
        mapping = census["mapping"]
        if base in mapping:
            return AdapterResult.success(mapping[base])
        raise ReasonCodeError(
            ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
            DetailCode.UNKNOWN_STATE_TOKEN,
            f"unknown state {base!r}")
    except ReasonCodeError as e:
        return AdapterResult.fail(e.code, e.detail, str(e),
                                  open_conflict=e.open_conflict)
    except Exception as e:
        return AdapterResult.fail(
            ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
            DetailCode.UNEXPECTED_EXCEPTION,
            f"{type(e).__name__}: {e}", open_conflict="N-1")


def _validate_census_structure(census: Any) -> None:
    """P2-03: validate schema/version/declared_allowlist/missing consistency."""
    if not isinstance(census, dict):
        raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                              DetailCode.CENSUS_SCHEMA_VIOLATION,
                              "census is not a dict")

    mapping = census.get("mapping")
    if not isinstance(mapping, dict):
        raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                              DetailCode.CENSUS_SCHEMA_VIOLATION,
                              "census['mapping'] must be a dict")

    present = set(mapping.keys())
    missing = sorted(STATETOK_ALLOWLIST - present)
    if missing:
        raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                              DetailCode.CENSUS_INCOMPLETE,
                              f"mapping missing tokens: {missing}")

    for k, v in mapping.items():
        if not _is_str(k) or not _is_str(v):
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  DetailCode.CENSUS_SCHEMA_VIOLATION,
                                  f"mapping[{k!r}]={v!r} must be str->str")
        if v not in STATETOK_ALLOWLIST:
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  DetailCode.OUT_OF_ALLOWLIST_STATE,
                                  f"mapping[{k!r}]={v!r} not in allowlist")
        if k != v:
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  DetailCode.CENSUS_SCHEMA_VIOLATION,
                                  f"non-identity mapping: {k!r} -> {v!r}")

    # P2-03: optional metadata must be consistent when present
    if "version" in census and census["version"] != "v1":
        raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                              DetailCode.CENSUS_SCHEMA_VIOLATION,
                              f"unsupported census version {census['version']!r}")

    if "declared_allowlist" in census:
        declared = census["declared_allowlist"]
        if not isinstance(declared, (list, tuple)):
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  DetailCode.CENSUS_SCHEMA_VIOLATION,
                                  "declared_allowlist must be list/tuple")
        if set(declared) != STATETOK_ALLOWLIST:
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  DetailCode.CENSUS_SCHEMA_VIOLATION,
                                  f"declared_allowlist={declared} != canonical")

    if "missing" in census:
        stated = census["missing"]
        if not isinstance(stated, (list, tuple)):
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  DetailCode.CENSUS_SCHEMA_VIOLATION,
                                  "missing must be list/tuple")
        if list(stated) != missing:
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  DetailCode.CENSUS_SCHEMA_VIOLATION,
                                  f"missing={stated} inconsistent with actual={missing}")


def _state_base(sdk_state) -> str:
    """Extract base token from state (str, enum, or enum-like)."""
    if sdk_state is None:
        return ""
    if _is_str(sdk_state):
        return sdk_state.strip()
    if hasattr(sdk_state, "name"):
        return sdk_state.name
    if hasattr(sdk_state, "value"):
        return str(sdk_state.value)
    return str(sdk_state)


# ============================================================================
# CATALOGUE PATH SAFETY — P1-01 fix (fd-relative no-follow, TOCTOU-safe)
# ============================================================================
def _read_file_no_follow(base_dir: str, rel_parts: List[str]) -> bytes:
    """P1-01 (v2.1): lstat + S_ISLNK on EVERY relative component (works even
    when O_DIRECTORY|O_NOFOLLOW is ignored, e.g. v9fs); O_NOFOLLOW as
    belt-and-braces; fd-relative walk (TOCTOU-safe). base_dir is TRUSTED (GE4)."""
    if (not rel_parts or not all(isinstance(p, str) and p and p != ".."
            and "/" not in p and "\\" not in p for p in rel_parts)):
        raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                              DetailCode.PATH_TRAVERSAL,
                              f"invalid relative path parts: {rel_parts!r}",
                              open_conflict="P1-01")
    _O_DIR = getattr(os, "O_DIRECTORY", 0)
    _O_NOFOLLOW = getattr(os, "O_NOFOLLOW", 0)
    try:
        cur_fd = os.open(base_dir, os.O_RDONLY | _O_DIR)
    except FileNotFoundError:
        raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                              DetailCode.GAME_CATALOGUE_MISSING,
                              f"base_dir does not exist: {base_dir!r}")
    except OSError as e:
        raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                              DetailCode.GAME_CATALOGUE_MISSING,
                              f"cannot open base_dir {base_dir!r}: {e}")
    try:
        for part in rel_parts[:-1]:
            try:
                st = os.lstat(part, dir_fd=cur_fd)
            except FileNotFoundError:
                raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                      DetailCode.GAME_CATALOGUE_MISSING,
                                      f"missing directory component: {part!r}")
            except OSError as e:
                raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                      DetailCode.PATH_TRAVERSAL,
                                      f"cannot lstat {part!r}: {e}", open_conflict="P1-01")
            if stat.S_ISLNK(st.st_mode):
                raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                      DetailCode.PATH_TRAVERSAL,
                                      f"symlink ancestor rejected: {part!r}", open_conflict="P1-01")
            if not stat.S_ISDIR(st.st_mode):
                raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                      DetailCode.PATH_TRAVERSAL,
                                      f"not a directory: {part!r}", open_conflict="P1-01")
            try:
                next_fd = os.open(part, os.O_RDONLY | _O_DIR | _O_NOFOLLOW, dir_fd=cur_fd)
            except OSError as e:
                raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                      DetailCode.PATH_TRAVERSAL,
                                      f"cannot open directory {part!r}: {e}", open_conflict="P1-01")
            os.close(cur_fd); cur_fd = next_fd
        fname = rel_parts[-1]
        try:
            st = os.lstat(fname, dir_fd=cur_fd)
        except FileNotFoundError:
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  DetailCode.GAME_CATALOGUE_MISSING,
                                  f"catalogue file not found: {fname!r}")
        except OSError as e:
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  DetailCode.PATH_TRAVERSAL,
                                  f"cannot lstat {fname!r}: {e}", open_conflict="P1-01")
        if stat.S_ISLNK(st.st_mode):
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  DetailCode.PATH_TRAVERSAL,
                                  f"symlink file rejected: {fname!r}", open_conflict="P1-01")
        if not stat.S_ISREG(st.st_mode):
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  DetailCode.PATH_TRAVERSAL,
                                  f"not a regular file: {fname!r}", open_conflict="P1-01")
        try:
            file_fd = os.open(fname, os.O_RDONLY | _O_NOFOLLOW, dir_fd=cur_fd)
        except OSError as e:
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  DetailCode.PATH_TRAVERSAL,
                                  f"cannot open file {fname!r}: {e}", open_conflict="P1-01")
    finally:
        os.close(cur_fd)
    try:
        chunks = []
        while True:
            chunk = os.read(file_fd, 65536)
            if not chunk: break
            chunks.append(chunk)
        return b"".join(chunks)
    finally:
        os.close(file_fd)


def _safe_catalogue_path(catalogue_dir: str, game_id: str) -> str:
    """Validate game_id grammar + path traversal; return target path."""
    if not _is_str(game_id) or not GAME_ID_RE.fullmatch(game_id):
        raise ReasonCodeError(
            ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
            DetailCode.PATH_TRAVERSAL,
            f"invalid game_id {game_id!r}",
            open_conflict="P1-01")

    if not _is_str(catalogue_dir):
        raise ReasonCodeError(
            ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
            DetailCode.PATH_TRAVERSAL,
            "catalogue_dir must be str",
            open_conflict="P1-01")

    # Containment check: real-path comparison
    base = os.path.abspath(catalogue_dir)
    target = os.path.join(base, "games", f"{game_id}.json")
    try:
        if os.path.commonpath([base, os.path.abspath(target)]) != base:
            raise ReasonCodeError(
                ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                DetailCode.PATH_TRAVERSAL,
                "path escapes catalogue root",
                open_conflict="P1-01")
    except ValueError:
        raise ReasonCodeError(
            ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
            DetailCode.PATH_TRAVERSAL,
            "path not comparable",
            open_conflict="P1-01")

    return target


# ============================================================================
# CATALOGUE VALIDATION — structural + P1-04 SDK vocabulary check
# ============================================================================
def _validate_catalogue_strict(catalogue: Dict[str, Any],
                               sdk_bound: Optional[Dict[str, Any]]) -> None:
    """
    Structural validation + P1-04 fix: validate wire_id/sdk_name against
    bound SDK GameAction enum when available.

    If sdk_bound is None (arcengine not installed): GE2 skips SDK validation.
    GE4 MUST validate against its own authenticated SDK binding before dispatch.
    """
    if not isinstance(catalogue, dict):
        raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                              DetailCode.CATALOGUE_STRUCTURE_INVALID,
                              "catalogue is not dict")
    if catalogue.get("version") != "v1":
        raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                              DetailCode.CATALOGUE_STRUCTURE_INVALID,
                              f"version {catalogue.get('version')!r} != v1")
    if not _is_bool(catalogue.get("confirmed")):
        raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                              DetailCode.CATALOGUE_STRUCTURE_INVALID,
                              "confirmed must be bool")
    if catalogue["confirmed"] is not True:
        raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                              DetailCode.ACTION_CONTRACT_UNCONFIRMED,
                              "catalogue confirmed=false")
    if catalogue.get("authoritative_source") != "ENV_OWNER":
        raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                              DetailCode.CONTRACT_NOT_AUTHORITATIVE,
                              f"authoritative_source={catalogue.get('authoritative_source')!r}")
    if catalogue.get("origin") != COORDINATE_ORIGIN:
        raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                              DetailCode.ACTION_CONTRACT_UNCONFIRMED,
                              f"origin != {COORDINATE_ORIGIN}")
    if catalogue.get("direction") != COORDINATE_DIRECTION:
        raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                              DetailCode.ACTION_CONTRACT_UNCONFIRMED,
                              f"direction != {COORDINATE_DIRECTION}")

    actions = catalogue.get("actions")
    if not isinstance(actions, dict):
        raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                              DetailCode.CATALOGUE_STRUCTURE_INVALID,
                              "actions must be dict")

    wire_ids, sdk_names = set(), set()
    for name, ad in actions.items():
        if not _is_str(name) or not name:
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  DetailCode.CATALOGUE_STRUCTURE_INVALID,
                                  "action key must be non-empty str")
        if not isinstance(ad, dict):
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  DetailCode.CATALOGUE_STRUCTURE_INVALID,
                                  f"{name!r} not dict")

        wire_id = ad.get("wire_id")
        if not _is_int(wire_id) or wire_id <= 0:
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  DetailCode.CATALOGUE_STRUCTURE_INVALID,
                                  f"{name!r} wire_id must be positive int")
        if wire_id in wire_ids:
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  DetailCode.CATALOGUE_STRUCTURE_INVALID,
                                  f"duplicate wire_id {wire_id}")
        wire_ids.add(wire_id)

        sdk_name = ad.get("sdk_name")
        if not _is_str(sdk_name) or not sdk_name.strip():
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  DetailCode.CATALOGUE_STRUCTURE_INVALID,
                                  f"{name!r} sdk_name must be non-empty non-whitespace str")
        if sdk_name in sdk_names:
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  DetailCode.CATALOGUE_STRUCTURE_INVALID,
                                  f"duplicate sdk_name {sdk_name!r}")
        sdk_names.add(sdk_name)

        cs = ad.get("coord_space")
        ps = ad.get("payload")
        if cs not in ALLOWED_COORD_SPACES:
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  DetailCode.CATALOGUE_STRUCTURE_INVALID,
                                  f"{name!r} coord_space {cs!r}")
        if ps not in ALLOWED_PAYLOAD_SCHEMAS:
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  DetailCode.CATALOGUE_STRUCTURE_INVALID,
                                  f"{name!r} payload {ps!r}")
        if (ps, cs) not in ALLOWED_COMBINATIONS:
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  DetailCode.CATALOGUE_COMBINATION_INVALID,
                                  f"{name!r} combo {ps!r}/{cs!r} not implemented")

        # P1-04: SDK vocabulary check when SDK is bound
        _validate_against_sdk_vocabulary(name, wire_id, sdk_name, sdk_bound)


def _validate_against_sdk_vocabulary(action_name, wire_id, sdk_name,
                                     sdk_bound: Optional[Dict[str, Any]]) -> None:
    """P1-04: when SDK bound, verify wire_id/sdk_name correspond in GameAction."""
    if sdk_bound is None:
        return  # SDK not bound; GE4 must validate

    ga = sdk_bound.get("GameAction")
    if ga is None:
        return  # no GameAction; GE4 must validate

    member = None
    from_id = getattr(ga, "from_id", None)
    try:
        if from_id is not None:
            member = from_id(wire_id)
        else:
            member = ga(wire_id)
    except (ValueError, KeyError) as e:
        raise ReasonCodeError(
            ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
            DetailCode.CATALOGUE_SDK_MISMATCH,
            f"{action_name!r}: wire_id={wire_id} not in SDK GameAction: {e}")
    except Exception:
        return  # unknown SDK API; defer to GE4

    if member is None:
        raise ReasonCodeError(
            ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
            DetailCode.CATALOGUE_SDK_MISMATCH,
            f"{action_name!r}: wire_id={wire_id} not resolved by SDK")

    member_name = getattr(member, "name", None)
    if member_name is not None and member_name != sdk_name:
        raise ReasonCodeError(
            ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
            DetailCode.CATALOGUE_SDK_MISMATCH,
            f"{action_name!r}: sdk_name={sdk_name!r} != SDK enum name {member_name!r}")


def load_game_catalogue(game_id: str, *,
                        catalogue_dir: str,
                        pinned_hashes: Dict[str, str]) -> Dict[str, Any]:
    """
    Load & validate catalogue. GE4 MUST supply `pinned_hashes` from an
    authenticated index — GE2 does NOT authenticate the pinset, only
    checks that the loaded file's hash matches the caller-supplied value.
    """
    if not isinstance(pinned_hashes, dict) or not pinned_hashes:
        raise ReasonCodeError(
            ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
            DetailCode.CATALOGUE_UNPINNED,
            "catalogue loading requires non-empty pinned_hashes; GE4 must supply",
            open_conflict="PENDING_CATALOGUE_PROVENANCE")

    _safe_catalogue_path(catalogue_dir, game_id)  # pre-check: grammar + containment
    base_dir = os.path.abspath(catalogue_dir)
    rel_parts = ["games", f"{game_id}.json"]
    try:
        raw = _read_file_no_follow(base_dir, rel_parts)
    except ReasonCodeError:
        raise
    except FileNotFoundError:
        raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                              DetailCode.GAME_CATALOGUE_MISSING,
                              f"no catalogue for {game_id!r}")
    except OSError as e:
        raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                              DetailCode.GAME_CATALOGUE_MISSING,
                              f"read failed: {e}")

    try:
        data = json.loads(raw.decode("utf-8"))
    except (json.JSONDecodeError, UnicodeDecodeError) as e:
        raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                              DetailCode.GAME_CATALOGUE_MISSING,
                              f"parse failed: {e}")

    # Hash check against caller-supplied pinset
    canon = json.dumps(data, sort_keys=True, separators=(",", ":")).encode()
    actual = hashlib.sha256(canon).hexdigest()
    fname = f"{game_id}.json"

    if fname not in pinned_hashes:
        raise ReasonCodeError(
            ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
            DetailCode.CATALOGUE_UNPINNED,
            f"catalogue {fname!r} not in pinned index")

    expected = pinned_hashes[fname]
    if not _is_str(expected) or not SHA256_RE.match(expected):
        raise ReasonCodeError(
            ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
            DetailCode.CATALOGUE_UNPINNED,
            f"pinned hash for {fname!r} is not a 64-hex sha256")

    if actual != expected:
        raise ReasonCodeError(
            ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
            DetailCode.CONTRACT_NOT_AUTHORITATIVE,
            f"hash mismatch: got {actual[:16]}... expected {expected[:16]}...")

    _validate_catalogue_strict(data, _SDK_BOUND)
    return data


# ============================================================================
# ACTION ADAPTER
# ============================================================================
def adapt_action(reasoner_action, *,
                 game_id: str, catalogue_dir: str,
                 pinned_hashes: Dict[str, str],
                 contract: ContractConfirmed,
                 harness: "HarnessBinding",
                 step_state: StepState,
                 t_measure: Optional[Callable[[], Optional[int]]] = None,
                 h_limit: Optional[int] = None,
                 is_feasible_set_empty: Optional[Callable[[], bool]] = None
                 ) -> AdapterResult:
    """
    Map reasoner action to SDK-dispatchable payload descriptor.

    Returns ok=True with result.value = {
        "status": "MAPPED_NOT_DISPATCHED",
        "sdk_name": str, "wire_id": int,
        "resolved_payload": dict|None,
        "coord_space": str, "payload_schema": str,
    }
    GE4 constructs ActionInput and calls perform_action(). GE2 never dispatches.
    """
    try:
        if step_state.step_failed:
            return AdapterResult.fail(
                ReasonCode.INVALID_REASONER_ACTION,
                DetailCode.RUN_ALREADY_TERMINAL,
                "step already failed (3-strike bound reached)")

        # F-28 re-query priority: T/H check before mapping
        if t_measure is not None:
            try:
                T = t_measure()
            except Exception as e:
                return AdapterResult.fail(
                    ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                    DetailCode.CALLBACK_FAILED,
                    f"t_measure raised {type(e).__name__}: {e}")
            code, detail = _validate_TH_strict(T, h_limit)
            if code is not None:
                if code == ReasonCode.REFUSAL_EVENT:
                    return AdapterResult.fail(
                        code, detail,
                        f"hard-limit refusal (T={T!r} H={h_limit!r})",
                        strike_counted=False, routed_to="refusal_policy")
                return AdapterResult.fail(
                    code, detail,
                    f"hard-limit config error (T={T!r} H={h_limit!r})",
                    strike_counted=False)

        if is_feasible_set_empty is not None:
            try:
                empty = is_feasible_set_empty()
            except Exception as e:
                return AdapterResult.fail(
                    ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                    DetailCode.CALLBACK_FAILED,
                    f"is_feasible_set_empty raised {type(e).__name__}: {e}")
            if empty:
                return AdapterResult.fail(
                    ReasonCode.REFUSAL_EVENT, DetailCode.EMPTY_FEASIBLE_SET,
                    "empty feasible set",
                    strike_counted=False, routed_to="refusal_policy")

        if contract.source != "ENV_OWNER":
            return AdapterResult.fail(
                ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                DetailCode.CONTRACT_NOT_AUTHORITATIVE,
                f"contract.source={contract.source!r}; GE4 must issue authenticated contract",
                open_conflict="P1-03")
        if contract.origin != COORDINATE_ORIGIN:
            return AdapterResult.fail(
                ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                DetailCode.ACTION_CONTRACT_UNCONFIRMED,
                f"origin {contract.origin!r} != {COORDINATE_ORIGIN!r}")
        if contract.direction != COORDINATE_DIRECTION:
            return AdapterResult.fail(
                ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                DetailCode.ACTION_CONTRACT_UNCONFIRMED,
                f"direction {contract.direction!r} != {COORDINATE_DIRECTION!r}")

        catalogue = load_game_catalogue(
            game_id, catalogue_dir=catalogue_dir, pinned_hashes=pinned_hashes)

        action_def, id_err = _resolve_action_strict(reasoner_action,
                                                    catalogue["actions"])
        if action_def is None:
            return _count_strike(step_state,
                                 id_err or DetailCode.UNKNOWN_ACTION_NAME)

        try:
            payload = _extract_payload_strict(reasoner_action, action_def)
        except ReasonCodeError as e:
            return _count_strike(step_state, e.detail)

        try:
            resolved = _convert_payload_strict(action_def, payload,
                                               contract=contract,
                                               harness=harness)
        except ReasonCodeError as e:
            if e.code == ReasonCode.INVALID_REASONER_ACTION:
                return _count_strike(step_state, e.detail)
            return AdapterResult.fail(e.code, e.detail, str(e),
                                      open_conflict=e.open_conflict)

        return AdapterResult.success({
            "status": "MAPPED_NOT_DISPATCHED",
            "sdk_name": action_def["sdk_name"],
            "wire_id": action_def["wire_id"],
            "resolved_payload": resolved,
            "coord_space": action_def["coord_space"],
            "payload_schema": action_def["payload"],
        })

    except ReasonCodeError as e:
        return AdapterResult.fail(e.code, e.detail, str(e),
                                  open_conflict=e.open_conflict)
    except Exception as e:
        return AdapterResult.fail(
            ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
            DetailCode.UNEXPECTED_EXCEPTION,
            f"{type(e).__name__}: {e}", open_conflict="N-1")


def _resolve_action_strict(reasoner_action,
                           actions: Dict[str, Any]
                           ) -> Tuple[Optional[Dict[str, Any]], Optional[DetailCode]]:
    """Strict identity resolution. Returns (action_def|None, error_code|None)."""
    name: Optional[str] = None
    aid: Optional[int] = None
    name_present = id_present = False

    if isinstance(reasoner_action, dict):
        name_keys = [k for k in ("name", "sdk_name", "action") if k in reasoner_action]
        id_keys = [k for k in ("wire_id", "id", "action_id") if k in reasoner_action]
        if len(name_keys) > 1 or len(id_keys) > 1:
            return None, DetailCode.IDENTITY_CONFLICT
        if name_keys:
            name_present = True
            v = reasoner_action[name_keys[0]]
            if not _is_str(v) or not v:
                return None, DetailCode.UNKNOWN_ACTION_NAME
            name = v
        if id_keys:
            id_present = True
            v = reasoner_action[id_keys[0]]
            if not _is_int(v):
                return None, DetailCode.UNKNOWN_ACTION_ID
            aid = v
    elif _is_str(reasoner_action):
        name_present = True
        name = reasoner_action
    else:
        if hasattr(reasoner_action, "name"):
            name_present = True
            v = reasoner_action.name
            if not _is_str(v) or not v:
                return None, DetailCode.UNKNOWN_ACTION_NAME
            name = v
        if hasattr(reasoner_action, "wire_id"):
            id_present = True
            v = reasoner_action.wire_id
            if not _is_int(v):
                return None, DetailCode.UNKNOWN_ACTION_ID
            aid = v

    if not name_present and not id_present:
        return None, DetailCode.UNKNOWN_ACTION_NAME

    if name_present and id_present:
        if name in actions and actions[name]["wire_id"] == aid:
            return actions[name], None
        return None, DetailCode.IDENTITY_CONFLICT

    if name_present:
        if name in actions:
            return actions[name], None
        return None, DetailCode.UNKNOWN_ACTION_NAME

    matches = [ad for ad in actions.values() if ad["wire_id"] == aid]
    if len(matches) == 1:
        return matches[0], None
    return None, DetailCode.UNKNOWN_ACTION_ID


def _extract_payload_strict(reasoner_action, action_def) -> Any:
    if isinstance(reasoner_action, dict):
        keys = [k for k in ("payload", "args", "params") if k in reasoner_action]
        if len(keys) > 1:
            raise ReasonCodeError(ReasonCode.INVALID_REASONER_ACTION,
                                  DetailCode.PAYLOAD_ALIAS_CONFLICT,
                                  f"multiple payload aliases: {keys}")
        payload = reasoner_action[keys[0]] if keys else None
    else:
        payload = getattr(reasoner_action, "payload", None)

    schema = action_def.get("payload")
    if schema == "NONE":
        if payload is not None:
            raise ReasonCodeError(ReasonCode.INVALID_REASONER_ACTION,
                                  DetailCode.MALFORMED_PAYLOAD,
                                  "payload must be None for schema NONE")
        return None
    if payload is None:
        raise ReasonCodeError(ReasonCode.INVALID_REASONER_ACTION,
                              DetailCode.MISSING_PAYLOAD,
                              f"payload required for schema {schema}")

    if schema == "XY_PAIR":
        if not isinstance(payload, dict) or set(payload.keys()) != {"x", "y"}:
            raise ReasonCodeError(ReasonCode.INVALID_REASONER_ACTION,
                                  DetailCode.PAYLOAD_SCHEMA_MISMATCH,
                                  "XY_PAIR requires dict with exactly x,y")
        return payload
    if schema == "GRID_RC":
        if not isinstance(payload, dict) or set(payload.keys()) != {"row", "col"}:
            raise ReasonCodeError(ReasonCode.INVALID_REASONER_ACTION,
                                  DetailCode.PAYLOAD_SCHEMA_MISMATCH,
                                  "GRID_RC requires dict with exactly row,col")
        return payload
    raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                          DetailCode.ACTION_CONTRACT_UNCONFIRMED,
                          f"unknown schema {schema!r}")


def _convert_payload_strict(action_def, payload, *, contract, harness):
    coord_space = action_def.get("coord_space")
    schema = action_def.get("payload")
    if (schema, coord_space) not in ALLOWED_COMBINATIONS:
        raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                              DetailCode.CATALOGUE_COMBINATION_INVALID,
                              f"combo {schema!r}/{coord_space!r} not implemented")

    if coord_space == "NONE":
        return None

    if coord_space == "GRID_CELL":
        if not contract.grid_cell_swap_confirmed:
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  DetailCode.ACTION_CONTRACT_UNCONFIRMED,
                                  "GRID_CELL swap not confirmed",
                                  open_conflict="N-1")
        row, col = payload["row"], payload["col"]
        _check_range(row); _check_range(col)
        return {"x": col, "y": row}

    if coord_space == "DISPLAY_CLICK":
        if not contract.display_click_confirmed:
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  DetailCode.ACTION_CONTRACT_UNCONFIRMED,
                                  "DISPLAY_CLICK not confirmed",
                                  open_conflict="N-1")
        if harness.camera is None:
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  DetailCode.HARNESS_BINDING_PENDING,
                                  "DISPLAY_CLICK requires bound camera (fail-closed)",
                                  open_conflict="C-3/D8")
        x, y = payload["x"], payload["y"]
        _check_display_range(x); _check_display_range(y)
        back = harness.display_to_grid(x, y)
        if back is None:
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  DetailCode.UNMAPPED_DISPLAY_COORDINATE,
                                  f"display ({x},{y}) unmapped",
                                  open_conflict="N-1")
        if not _is_valid_rc(back):
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  DetailCode.COORDINATE_ROUNDTRIP_MISMATCH,
                                  f"malformed roundtrip {back!r}",
                                  open_conflict="N-1")
        return {"x": x, "y": y}

    raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                          DetailCode.ACTION_CONTRACT_UNCONFIRMED,
                          f"unknown coord_space {coord_space!r}")


def _check_range(v):
    if type(v) is not int or not (0 <= v <= 63):
        raise ReasonCodeError(ReasonCode.INVALID_REASONER_ACTION,
                              DetailCode.MALFORMED_PAYLOAD,
                              f"grid coord {v!r} must be int 0..63")


def _check_display_range(v):
    if type(v) is not int or v < 0:
        raise ReasonCodeError(ReasonCode.INVALID_REASONER_ACTION,
                              DetailCode.MALFORMED_PAYLOAD,
                              f"display coord {v!r} must be non-negative int")


def _is_valid_rc(rc) -> bool:
    if not isinstance(rc, tuple) or len(rc) != 2:
        return False
    r, c = rc
    return _is_int(r) and _is_int(c) and 0 <= r <= 63 and 0 <= c <= 63


def _count_strike(step_state: StepState, detail: DetailCode) -> AdapterResult:
    fails = step_state.record_invalid_action()
    return AdapterResult.fail(
        ReasonCode.INVALID_REASONER_ACTION, detail,
        "rejected without environment dispatch",
        strike_counted=True, step_failed=fails)


# ============================================================================
# REFUSAL POLICY — P1-05 fix (distinct H_UNCONFIGURED tag)
# ============================================================================
@dataclass
class RefusalResult:
    dispatch_allowed: bool
    refusal: bool
    code: Optional[ReasonCode] = None
    detail: Optional[DetailCode] = None
    T: Optional[int] = None
    open_conflict: Optional[str] = None


def _validate_TH_strict(T, H):
    if H is None or type(H) is not int or H <= 0:
        return (ReasonCode.ENVIRONMENT_CONTRACT_FAILURE, DetailCode.H_UNCONFIGURED)
    if T is None or type(T) is not int or T < 0:
        cls = (ReasonCode.REFUSAL_EVENT
               if T_UNMEASURABLE_CLASS == "REFUSAL_EVENT"
               else ReasonCode.ENVIRONMENT_CONTRACT_FAILURE)
        return (cls, DetailCode.T_UNMEASURABLE)
    if T >= H:
        return (ReasonCode.REFUSAL_EVENT, DetailCode.HARD_LIMIT_EXCEEDED)
    return (None, None)


def check_hard_limit(T, H) -> RefusalResult:
    """
    Fail-closed hard-limit gate.

    GE4 measures T at the dispatch boundary. H must be the authenticated
    authoritative value supplied by GE4. GE2 validates types and applies
    the rule; GE2 cannot authenticate H's provenance.
    """
    code, detail = _validate_TH_strict(T, H)
    if code is None:
        return RefusalResult(True, False, None, None, T)

    if code == ReasonCode.REFUSAL_EVENT:
        oc = (CONFLICT_TAG_T_UNMEASURABLE
              if detail == DetailCode.T_UNMEASURABLE else None)
        return RefusalResult(False, True, code, detail, T, open_conflict=oc)

    # P1-05: distinct tag for H_UNCONFIGURED (was "census-partial")
    if detail == DetailCode.H_UNCONFIGURED:
        oc = CONFLICT_TAG_H_PENDING
    else:
        oc = CONFLICT_TAG_T_UNMEASURABLE
    return RefusalResult(False, False, code, detail, T, open_conflict=oc)


# ============================================================================
# HARNESS BINDING (interface stub for GE4)
# ============================================================================
class HarnessBinding:
    """Interface for GE4 to bind the environment's camera + inverse."""

    def __init__(self):
        self.camera = None
        self._env_owner_inverse: Optional[Callable[[int, int], Tuple[int, int]]] = None

    def bind_camera(self, camera):
        self.camera = camera

    def bind_env_owner_inverse(self, fn: Callable[[int, int], Tuple[int, int]]):
        self._env_owner_inverse = fn

    def display_to_grid(self, x: int, y: int) -> Optional[Tuple[int, int]]:
        if self.camera is None:
            raise ReasonCodeError(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  DetailCode.HARNESS_BINDING_PENDING,
                                  "camera.display_to_grid not bound",
                                  open_conflict="C-3/D8")
        return self.camera.display_to_grid(x, y)

    def grid_to_display(self, row: int, col: int) -> Tuple[int, int]:
        if self._env_owner_inverse is not None and (
                ENV_OWNER_INVERSE_CONFIRMED or DERIVED_INVERSE_CONFIRMED):
            return self._env_owner_inverse(row, col)
        raise ReasonCodeError(
            ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
            DetailCode.HARNESS_BINDING_PENDING,
            "grid_to_display does not exist; env-owner inverse required",
            open_conflict="N-1")


# ============================================================================
# TESTS — unit tests for STRUCTURE only (no authentication claims)
# ============================================================================
_TEST_RESULTS: List[Tuple[str, Status, str]] = []


def _status(name, st, detail=""):
    return (name, st, detail)


def _blank_grid():
    return [[0] * 64 for _ in range(64)]


def _test_census():
    """Structurally valid census (no auth fields — GE2 doesn't check them)."""
    return {"mapping": {tok: tok for tok in STATETOK_ALLOWLIST}}


def _test_catalogue(actions=None):
    if actions is None:
        actions = {
            "ACTION1": {"wire_id": 1, "sdk_name": "ACTION1",
                        "payload": "NONE", "coord_space": "NONE"},
            "ACTION6": {"wire_id": 6, "sdk_name": "ACTION6",
                        "payload": "GRID_RC", "coord_space": "GRID_CELL"},
        }
    return {
        "version": "v1", "confirmed": True,
        "authoritative_source": "ENV_OWNER",
        "origin": COORDINATE_ORIGIN, "direction": COORDINATE_DIRECTION,
        "actions": actions,
    }


def _write_catalogue(tmpdir, cat, game_id="test_game"):
    d = os.path.join(tmpdir, "games")
    os.makedirs(d, exist_ok=True)
    with open(os.path.join(d, f"{game_id}.json"), "w") as f:
        json.dump(cat, f)


def _pin_for(cat, game_id="test_game"):
    canon = json.dumps(cat, sort_keys=True, separators=(",", ":")).encode()
    return {f"{game_id}.json": hashlib.sha256(canon).hexdigest()}


def _make_contract(grid_cell=True, display_click=False, source="ENV_OWNER"):
    return ContractConfirmed(
        grid_cell_swap_confirmed=grid_cell,
        display_click_confirmed=display_click,
        origin=COORDINATE_ORIGIN, direction=COORDINATE_DIRECTION,
        source=source)


# --- FRAME (6) ---
def ts_frame_accept():
    r = adapt_frame({"g": [_blank_grid()]}, enumerate_grids=lambda o: o["g"])
    if r.ok:
        return _status("frame: accept 64x64", Status.PASS)
    return _status("frame: accept 64x64", Status.FAIL)


def ts_frame_multi():
    r = adapt_frame({"g": [_blank_grid(), _blank_grid()]},
                    enumerate_grids=lambda o: o["g"])
    if (not r.ok and r.detail == DetailCode.UNSUPPORTED_MULTIGRID):
        return _status("frame: multi-grid reject", Status.PASS)
    return _status("frame: multi-grid reject", Status.FAIL)


def ts_frame_dim():
    g = [[0] * 32 for _ in range(32)]
    r = adapt_frame({"g": [g]}, enumerate_grids=lambda o: o["g"])
    if (not r.ok and r.detail == DetailCode.UNSUPPORTED_DIMENSION):
        return _status("frame: non-64x64 reject", Status.PASS)
    return _status("frame: non-64x64 reject", Status.FAIL)


def ts_frame_bad_cell():
    g = _blank_grid(); g[5][5] = True
    r = adapt_frame({"g": [g]}, enumerate_grids=lambda o: o["g"])
    if (not r.ok and r.detail == DetailCode.MALFORMED_OBSERVATION):
        return _status("frame: malformed cell reject", Status.PASS)
    return _status("frame: malformed cell reject", Status.FAIL)


def ts_frame_no_binding():
    r = adapt_frame({})
    if (not r.ok and r.detail == DetailCode.HARNESS_BINDING_PENDING):
        return _status("frame: no enumerator reject", Status.PASS)
    return _status("frame: no enumerator reject", Status.FAIL)


def ts_frame_callback_fail():
    def boom(_): raise RuntimeError("test")
    r = adapt_frame({}, enumerate_grids=boom)
    if (not r.ok and r.detail == DetailCode.CALLBACK_FAILED):
        return _status("frame: callback exception normalized", Status.PASS)
    return _status("frame: callback exception normalized", Status.FAIL)


# --- STATE (5) ---
def ts_state_accept_str():
    r = adapt_state("NOT_FINISHED", census=_test_census())
    if r.ok and r.value == "NOT_FINISHED":
        return _status("state: NOT_FINISHED accept", Status.PASS)
    return _status("state: NOT_FINISHED accept", Status.FAIL)


def ts_state_accept_enum():
    class GameState(str, Enum):
        NOT_FINISHED = "NOT_FINISHED"
    r = adapt_state(GameState.NOT_FINISHED, census=_test_census())
    if r.ok and r.value == "NOT_FINISHED":
        return _status("state: enum accept", Status.PASS)
    return _status("state: enum accept", Status.FAIL)


def ts_state_unknown():
    r = adapt_state("RUNNING", census=_test_census())
    if (not r.ok and r.detail == DetailCode.UNKNOWN_STATE_TOKEN):
        return _status("state: unknown reject", Status.PASS)
    return _status("state: unknown reject", Status.FAIL)


def ts_state_incomplete_mapping():
    c = {"mapping": {"NOT_FINISHED": "NOT_FINISHED"}}
    r = adapt_state("NOT_FINISHED", census=c)
    if (not r.ok and r.detail == DetailCode.CENSUS_INCOMPLETE):
        return _status("state: incomplete mapping rejected", Status.PASS)
    return _status("state: incomplete mapping rejected", Status.FAIL)


def ts_state_non_identity_mapping():
    c = {"mapping": {tok: tok for tok in STATETOK_ALLOWLIST}}
    c["mapping"]["NOT_FINISHED"] = "WIN"
    r = adapt_state("NOT_FINISHED", census=c)
    if (not r.ok and r.detail == DetailCode.CENSUS_SCHEMA_VIOLATION):
        return _status("state: non-identity mapping rejected", Status.PASS)
    return _status("state: non-identity mapping rejected", Status.FAIL)


# --- CATALOGUE PATH SAFETY (P1-01) ---
def ts_catalogue_path_traversal():
    tmpdir = tempfile.mkdtemp(prefix="ge2_pt_")
    try:
        try:
            load_game_catalogue("../../etc/passwd", catalogue_dir=tmpdir,
                                pinned_hashes={"x.json": "0" * 64})
            return _status("catalogue: path traversal rejected", Status.FAIL)
        except ReasonCodeError as e:
            if e.detail == DetailCode.PATH_TRAVERSAL:
                return _status("catalogue: path traversal rejected", Status.PASS)
        return _status("catalogue: path traversal rejected", Status.FAIL)
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def ts_catalogue_no_pins():
    tmpdir = tempfile.mkdtemp(prefix="ge2_np_")
    try:
        cat = _test_catalogue()
        _write_catalogue(tmpdir, cat)
        try:
            load_game_catalogue("test_game", catalogue_dir=tmpdir,
                                pinned_hashes={})
            return _status("catalogue: empty pins rejected", Status.FAIL)
        except ReasonCodeError as e:
            if e.detail == DetailCode.CATALOGUE_UNPINNED:
                return _status("catalogue: empty pins rejected", Status.PASS)
        return _status("catalogue: empty pins rejected", Status.FAIL)
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def ts_catalogue_missing_pin():
    tmpdir = tempfile.mkdtemp(prefix="ge2_mp_")
    try:
        cat = _test_catalogue()
        _write_catalogue(tmpdir, cat)
        try:
            load_game_catalogue("test_game", catalogue_dir=tmpdir,
                                pinned_hashes={"other.json": "0" * 64})
            return _status("catalogue: missing pin rejected", Status.FAIL)
        except ReasonCodeError as e:
            if e.detail == DetailCode.CATALOGUE_UNPINNED:
                return _status("catalogue: missing pin rejected", Status.PASS)
        return _status("catalogue: missing pin rejected", Status.FAIL)
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def ts_catalogue_hash_mismatch():
    tmpdir = tempfile.mkdtemp(prefix="ge2_hm_")
    try:
        cat = _test_catalogue()
        _write_catalogue(tmpdir, cat)
        try:
            load_game_catalogue("test_game", catalogue_dir=tmpdir,
                                pinned_hashes={"test_game.json": "0" * 64})
            return _status("catalogue: hash mismatch rejected", Status.FAIL)
        except ReasonCodeError as e:
            if e.detail == DetailCode.CONTRACT_NOT_AUTHORITATIVE:
                return _status("catalogue: hash mismatch rejected", Status.PASS)
        return _status("catalogue: hash mismatch rejected", Status.FAIL)
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def ts_catalogue_symlink_dir():
    """P1-01: symlinked games/ ancestor rejected via O_NOFOLLOW walk."""
    tmpdir = tempfile.mkdtemp(prefix="ge2_sd_")
    try:
        real = os.path.join(tmpdir, "real_games")
        os.makedirs(real)
        cat = _test_catalogue()
        with open(os.path.join(real, "test_game.json"), "w") as f:
            json.dump(cat, f)
        pins = _pin_for(cat)
        try:
            os.symlink(real, os.path.join(tmpdir, "games"))
        except OSError:
            return _status("catalogue: symlink dir rejected", Status.BLOCKED,
                           "symlinks not supported")
        try:
            load_game_catalogue("test_game", catalogue_dir=tmpdir,
                                pinned_hashes=pins)
            return _status("catalogue: symlink dir rejected", Status.FAIL)
        except ReasonCodeError as e:
            if e.detail == DetailCode.PATH_TRAVERSAL:
                return _status("catalogue: symlink dir rejected", Status.PASS)
        return _status("catalogue: symlink dir rejected", Status.FAIL)
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def ts_catalogue_symlink_file():
    """P1-01: symlinked catalogue file rejected via O_NOFOLLOW."""
    tmpdir = tempfile.mkdtemp(prefix="ge2_sf_")
    try:
        games_dir = os.path.join(tmpdir, "games")
        os.makedirs(games_dir)
        cat = _test_catalogue()
        outside = os.path.join(tmpdir, "outside.json")
        with open(outside, "w") as f:
            json.dump(cat, f)
        pins = _pin_for(cat)
        try:
            os.symlink(outside, os.path.join(games_dir, "test_game.json"))
        except OSError:
            return _status("catalogue: symlink file rejected", Status.BLOCKED,
                           "symlinks not supported")
        try:
            load_game_catalogue("test_game", catalogue_dir=tmpdir,
                                pinned_hashes=pins)
            return _status("catalogue: symlink file rejected", Status.FAIL)
        except ReasonCodeError as e:
            if e.detail == DetailCode.PATH_TRAVERSAL:
                return _status("catalogue: symlink file rejected", Status.PASS)
        return _status("catalogue: symlink file rejected", Status.FAIL)
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


# --- P1-04: SDK VOCABULARY CHECK ---
def ts_catalogue_sdk_mismatch():
    """P1-04: when SDK is bound, wire_id/sdk_name mismatch rejected."""
    if _SDK_BOUND is None or _SDK_BOUND.get("GameAction") is None:
        return _status("catalogue: SDK vocabulary check", Status.PASS,
                       "SDK not bound — GE4 owns validation")
    # SDK is bound: construct a catalogue with a name that won't match
    tmpdir = tempfile.mkdtemp(prefix="ge2_sdk_")
    try:
        bad_actions = {
            "BOGUS": {"wire_id": 9999, "sdk_name": "NOT_A_REAL_ACTION",
                      "payload": "NONE", "coord_space": "NONE"},
        }
        cat = _test_catalogue(actions=bad_actions)
        _write_catalogue(tmpdir, cat)
        pins = _pin_for(cat)
        try:
            load_game_catalogue("test_game", catalogue_dir=tmpdir,
                                pinned_hashes=pins)
            return _status("catalogue: SDK mismatch rejected", Status.FAIL)
        except ReasonCodeError as e:
            if e.detail == DetailCode.CATALOGUE_SDK_MISMATCH:
                return _status("catalogue: SDK mismatch rejected", Status.PASS)
        return _status("catalogue: SDK mismatch rejected", Status.FAIL)
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def ts_catalogue_rejects_bad_wire_id():
    tmpdir = tempfile.mkdtemp(prefix="ge2_bw_")
    try:
        bad = _test_catalogue(actions={
            "A": {"wire_id": -1, "sdk_name": "A",
                  "payload": "NONE", "coord_space": "NONE"}})
        _write_catalogue(tmpdir, bad)
        pins = _pin_for(bad)
        try:
            load_game_catalogue("test_game", catalogue_dir=tmpdir,
                                pinned_hashes=pins)
            return _status("catalogue: negative wire_id rejected", Status.FAIL)
        except ReasonCodeError as e:
            if e.detail == DetailCode.CATALOGUE_STRUCTURE_INVALID:
                return _status("catalogue: negative wire_id rejected", Status.PASS)
        return _status("catalogue: negative wire_id rejected", Status.FAIL)
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def ts_catalogue_rejects_blank_sdk_name():
    tmpdir = tempfile.mkdtemp(prefix="ge2_bs_")
    try:
        bad = _test_catalogue(actions={
            "A": {"wire_id": 1, "sdk_name": "   ",
                  "payload": "NONE", "coord_space": "NONE"}})
        _write_catalogue(tmpdir, bad)
        pins = _pin_for(bad)
        try:
            load_game_catalogue("test_game", catalogue_dir=tmpdir,
                                pinned_hashes=pins)
            return _status("catalogue: whitespace sdk_name rejected", Status.FAIL)
        except ReasonCodeError as e:
            if e.detail == DetailCode.CATALOGUE_STRUCTURE_INVALID:
                return _status("catalogue: whitespace sdk_name rejected", Status.PASS)
        return _status("catalogue: whitespace sdk_name rejected", Status.FAIL)
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


# --- ACTION ---
def _run_action(reasoner_action, *, contract=None, harness=None, step=None,
                t=10, h=100, ife=None, actions=None, tmpdir=None):
    if tmpdir is None:
        tmpdir = tempfile.mkdtemp(prefix="ge2_act_")
    cat = _test_catalogue(actions=actions)
    _write_catalogue(tmpdir, cat)
    pins = _pin_for(cat)
    return adapt_action(
        reasoner_action, game_id="test_game", catalogue_dir=tmpdir,
        pinned_hashes=pins,
        contract=contract or _make_contract(),
        harness=harness or HarnessBinding(),
        step_state=step or StepState(),
        t_measure=(lambda: t) if not callable(t) else t,
        h_limit=h, is_feasible_set_empty=ife)


def ts_action_none_ok():
    tmpdir = tempfile.mkdtemp(prefix="ge2_a1_")
    try:
        r = _run_action({"name": "ACTION1"}, tmpdir=tmpdir)
        if r.ok and r.value["status"] == "MAPPED_NOT_DISPATCHED" \
                and r.value["resolved_payload"] is None:
            return _status("action: NONE mapped", Status.PASS)
        return _status("action: NONE mapped", Status.FAIL)
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def ts_action_grid_rc_swap():
    tmpdir = tempfile.mkdtemp(prefix="ge2_a2_")
    try:
        r = _run_action({"name": "ACTION6", "payload": {"row": 5, "col": 10}},
                        tmpdir=tmpdir)
        if r.ok and r.value["resolved_payload"] == {"x": 10, "y": 5}:
            return _status("action: GRID_RC swap", Status.PASS)
        return _status("action: GRID_RC swap", Status.FAIL)
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def ts_action_unknown_name_strike():
    tmpdir = tempfile.mkdtemp(prefix="ge2_a3_")
    try:
        step = StepState()
        r = _run_action({"name": "ACTION9"}, step=step, tmpdir=tmpdir)
        if (not r.ok and r.detail == DetailCode.UNKNOWN_ACTION_NAME
                and step.invalid_action_count == 1):
            return _status("action: unknown name + 1 strike", Status.PASS)
        return _status("action: unknown name + 1 strike", Status.FAIL)
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def ts_action_payload_on_none():
    tmpdir = tempfile.mkdtemp(prefix="ge2_a4_")
    try:
        r = _run_action({"name": "ACTION1", "payload": {"x": 1, "y": 2}},
                        tmpdir=tmpdir)
        if (not r.ok and r.detail == DetailCode.MALFORMED_PAYLOAD):
            return _status("action: payload on NONE rejected", Status.PASS)
        return _status("action: payload on NONE rejected", Status.FAIL)
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def ts_action_three_strikes():
    tmpdir = tempfile.mkdtemp(prefix="ge2_a5_")
    try:
        step = StepState()
        for _ in range(3):
            _run_action({"name": "ACTION9"}, step=step, tmpdir=tmpdir)
        if step.step_failed and step.invalid_action_count == 3:
            return _status("action: 3 strikes -> step_failed", Status.PASS)
        return _status("action: 3 strikes -> step_failed", Status.FAIL)
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def ts_action_failed_step_blocks():
    tmpdir = tempfile.mkdtemp(prefix="ge2_a6_")
    try:
        step = StepState(); step.step_failed = True
        r = _run_action({"name": "ACTION1"}, step=step, tmpdir=tmpdir)
        if (not r.ok and r.detail == DetailCode.RUN_ALREADY_TERMINAL):
            return _status("action: failed step entry reject", Status.PASS)
        return _status("action: failed step entry reject", Status.FAIL)
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def ts_action_T_eq_H_no_strike():
    tmpdir = tempfile.mkdtemp(prefix="ge2_a7_")
    try:
        step = StepState()
        r = _run_action({"name": "ACTION1"}, t=100, h=100,
                        step=step, tmpdir=tmpdir)
        if (not r.ok and r.code == ReasonCode.REFUSAL_EVENT
                and not r.strike_counted):
            return _status("action: T=H -> REFUSAL no strike", Status.PASS)
        return _status("action: T=H -> REFUSAL no strike", Status.FAIL)
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def ts_action_empty_feasible():
    tmpdir = tempfile.mkdtemp(prefix="ge2_a8_")
    try:
        step = StepState()
        r = _run_action({"name": "ACTION1"}, ife=lambda: True,
                        step=step, tmpdir=tmpdir)
        if (not r.ok and r.detail == DetailCode.EMPTY_FEASIBLE_SET
                and not r.strike_counted):
            return _status("action: empty feasible no strike", Status.PASS)
        return _status("action: empty feasible no strike", Status.FAIL)
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


# --- REFUSAL (5) ---
def ts_refusal_T_eq_H():
    r = check_hard_limit(T=100, H=100)
    if (not r.dispatch_allowed and r.refusal
            and r.code == ReasonCode.REFUSAL_EVENT):
        return _status("refusal: T=H refuses", Status.PASS)
    return _status("refusal: T=H refuses", Status.FAIL)


def ts_refusal_T_lt_H():
    r = check_hard_limit(T=10, H=100)
    if r.dispatch_allowed and not r.refusal:
        return _status("refusal: T<H permits", Status.PASS)
    return _status("refusal: T<H permits", Status.FAIL)


def ts_refusal_T_unmeasurable():
    r = check_hard_limit(T=None, H=100)
    if (r.code == ReasonCode.REFUSAL_EVENT
            and r.open_conflict == CONFLICT_TAG_T_UNMEASURABLE):
        return _status("refusal: T unmeasurable -> REFUSAL", Status.PASS)
    return _status("refusal: T unmeasurable -> REFUSAL", Status.FAIL)


def ts_refusal_H_unconfigured_correct_tag():
    """P1-05: H_UNCONFIGURED must use PENDING_AUTHORITATIVE_H tag, not census-partial."""
    r = check_hard_limit(T=10, H=None)
    if (r.code == ReasonCode.ENVIRONMENT_CONTRACT_FAILURE
            and r.detail == DetailCode.H_UNCONFIGURED
            and r.open_conflict == CONFLICT_TAG_H_PENDING):
        return _status("refusal: H unconfigured -> correct tag", Status.PASS)
    return _status("refusal: H unconfigured -> correct tag", Status.FAIL,
                   f"open_conflict={r.open_conflict!r}")


def ts_refusal_bool_rejected():
    if (check_hard_limit(True, 100).dispatch_allowed
            or check_hard_limit(10, True).dispatch_allowed
            or check_hard_limit(10, 100.0).dispatch_allowed):
        return _status("refusal: bool/float rejected", Status.FAIL)
    return _status("refusal: bool/float rejected", Status.PASS)


# --- TAXONOMY (2) ---
def ts_taxonomy_distinct():
    rcs = {ReasonCode.UNSUPPORTED_INPUT,
           ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
           ReasonCode.INVALID_REASONER_ACTION,
           ReasonCode.REFUSAL_EVENT}
    if len(rcs) == 4:
        return _status("taxonomy: 4 distinct classes", Status.PASS)
    return _status("taxonomy: 4 distinct classes", Status.FAIL)


def ts_taxonomy_refusal_separate():
    tmpdir = tempfile.mkdtemp(prefix="ge2_tx_")
    try:
        r = _run_action({"name": "ACTION1"}, t=100, h=100, tmpdir=tmpdir)
        if r.code == ReasonCode.REFUSAL_EVENT:
            return _status("taxonomy: refusal != env_contract", Status.PASS)
        return _status("taxonomy: refusal != env_contract", Status.FAIL)
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


# --- HARNESS (3) ---
def ts_harness_no_camera():
    h = HarnessBinding()
    try:
        h.display_to_grid(10, 20)
        return _status("harness: no camera fails closed", Status.FAIL)
    except ReasonCodeError as e:
        if e.detail == DetailCode.HARNESS_BINDING_PENDING:
            return _status("harness: no camera fails closed", Status.PASS)
    return _status("harness: no camera fails closed", Status.FAIL)


def ts_harness_no_inverse():
    h = HarnessBinding()
    try:
        h.grid_to_display(5, 10)
        return _status("harness: no inverse fails closed", Status.FAIL)
    except ReasonCodeError as e:
        if e.detail == DetailCode.HARNESS_BINDING_PENDING:
            return _status("harness: no inverse fails closed", Status.PASS)
    return _status("harness: no inverse fails closed", Status.FAIL)


def ts_harness_camera_bound():
    h = HarnessBinding()
    class Cam:
        def display_to_grid(self, x, y): return (y, x)
    h.bind_camera(Cam())
    result = h.display_to_grid(10, 20)
    if result == (20, 10):
        return _status("harness: camera delegates", Status.PASS)
    return _status("harness: camera delegates", Status.FAIL)


# ============================================================================
# TEST RUNNER
# ============================================================================
# ============================================================================
# GE2 v3.0 — ARCH/2026/103 §3 INTERFACE: adapt(observation) -> (grid, header)
# Joins GE2 to GE1 (header contract). Additive; no existing function changes.
# ============================================================================

def _obs_field(observation, name, default=None):
    """Read a field from a FrameData-like observation or a dict."""
    if observation is None:
        return default
    if isinstance(observation, dict):
        return observation.get(name, default)
    return getattr(observation, name, default)


def _indep_modal(grid):
    """Background = modal cell value (tie -> lowest). GE1 convention."""
    counts = {}
    for row in grid:
        for c in row:
            counts[c] = counts.get(c, 0) + 1
    if not counts:
        return 0
    best = max(counts.values())
    return min(v for v, n in counts.items() if n == best)


def _load_ge1_make_header():
    """ARCH §3: GE2 depends on GE1 (header contract). Import make_header READ-ONLY."""
    import importlib.util as _ilu
    here = os.path.dirname(os.path.abspath(__file__))
    for p in (os.path.join(here, "wire.py"), os.path.join(here, "ge1", "wire.py")):
        if os.path.exists(p):
            try:
                spec = _ilu.spec_from_file_location("ge1_wire", p)
                m = _ilu.module_from_spec(spec)
                sys.modules["ge1_wire"] = m
                spec.loader.exec_module(m)
                if hasattr(m, "make_header"):
                    return m.make_header
            except Exception:
                pass
    return None


def adapt(observation, *, census, enumerate_grids=None, make_header_fn=None):
    """ARCH §3 interface: observation -> (grid, header). Joins GE2 to GE1.

    GE2 validates STRUCTURE; the wire header is produced by GE1's make_header
    (the header contract). Fails closed (HARNESS_BINDING_PENDING / C-3/D8) if
    the GE1 contract is not bound. Additive; the individual adapters are
    unchanged."""
    fr = adapt_frame(observation, enumerate_grids=enumerate_grids)
    if not fr.ok:
        return fr
    grid = fr.value
    st = adapt_state(_obs_field(observation, "state"), census=census)
    if not st.ok:
        return st
    levels_completed = _obs_field(observation, "levels_completed", 0)
    available_actions = _obs_field(observation, "available_actions", [])
    background = _indep_modal(grid)
    mh = make_header_fn or _load_ge1_make_header()
    if mh is None:
        return AdapterResult.fail(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  DetailCode.HARNESS_BINDING_PENDING,
                                  "GE1 header contract not bound (C-3/D8)",
                                  open_conflict="C-3/D8")
    try:
        header = mh(levels_completed, st.value, available_actions, background)
    except Exception as e:
        return AdapterResult.fail(ReasonCode.ENVIRONMENT_CONTRACT_FAILURE,
                                  DetailCode.MALFORMED_OBSERVATION,
                                  f"make_header failed: {type(e).__name__}: {e}")
    return AdapterResult.success({"grid": grid, "header": header})


# ARCH §3 interface aliases (names only; delegate to the implemented functions)
map_action = adapt_action


def classify(err):
    """ARCH §3: classify(err) -> (ReasonCode, DetailCode)."""
    return (getattr(err, "code", None), getattr(err, "detail", None))


def ts_arch_adapt_join():
    """ARCH §3: adapt(observation) -> (grid, header) joins GE2 to GE1."""
    tok = sorted(STATETOK_ALLOWLIST)
    census = {"version": "v1", "confirmed": True, "complete": True, "missing": [],
              "authenticated": True, "source": "ge2_sdk_bound",
              "mapping": {t: t for t in tok}, "declared_allowlist": tok}
    grid = [[0] * 64 for _ in range(64)]
    for c in range(64):
        grid[0][c] = 1
    obs = {"state": "NOT_FINISHED", "levels_completed": 2,
           "available_actions": [1, 2, 3, 4, 5, 6], "_grids": [grid]}
    def enum(o):
        return o["_grids"]
    def mh(levels_completed, state, actions, background):
        return f"levels_completed={levels_completed}\nstate={state}\navailable_actions={actions}\nbackground={background}"
    r = adapt(obs, census=census, enumerate_grids=enum, make_header_fn=mh)
    if not r.ok:
        return _status("ARCH: adapt -> (grid, header)", Status.FAIL, f"{r.code}/{r.detail}")
    v = r.value
    if not isinstance(v, dict) or "grid" not in v or "header" not in v:
        return _status("ARCH: adapt -> (grid, header)", Status.FAIL, "no (grid, header)")
    if "state=NOT_FINISHED" not in v["header"] or "levels_completed=2" not in v["header"]:
        return _status("ARCH: adapt -> (grid, header)", Status.FAIL, f"header={v['header']!r}")
    if v["grid"][0][0] != 1:
        return _status("ARCH: adapt -> (grid, header)", Status.FAIL, "grid lost")
    return _status("ARCH: adapt -> (grid, header) [GE2->GE1 join]", Status.PASS)


def ts_arch_interface_names():
    """ARCH §3: adapt / map_action / classify must exist."""
    if not callable(globals().get("adapt")) or not callable(globals().get("map_action")) \
            or not callable(globals().get("classify")):
        return _status("ARCH: interface names", Status.FAIL, "missing adapt/map_action/classify")
    if globals().get("map_action") is not globals().get("adapt_action"):
        return _status("ARCH: interface names", Status.FAIL, "map_action != adapt_action")
    return _status("ARCH: interface names (adapt/map_action/classify)", Status.PASS)


_ALL_TESTS = [
    ts_arch_adapt_join, ts_arch_interface_names,
    # Frame (6)
    ts_frame_accept, ts_frame_multi, ts_frame_dim, ts_frame_bad_cell,
    ts_frame_no_binding, ts_frame_callback_fail,
    # State (5)
    ts_state_accept_str, ts_state_accept_enum, ts_state_unknown,
    ts_state_incomplete_mapping, ts_state_non_identity_mapping,
    # Catalogue path safety (P1-01) (7)
    ts_catalogue_path_traversal, ts_catalogue_no_pins,
    ts_catalogue_missing_pin, ts_catalogue_hash_mismatch,
    ts_catalogue_symlink_dir, ts_catalogue_symlink_file,
    ts_catalogue_rejects_bad_wire_id,
    # P1-04 SDK vocabulary (2)
    ts_catalogue_sdk_mismatch, ts_catalogue_rejects_blank_sdk_name,
    # Action (8)
    ts_action_none_ok, ts_action_grid_rc_swap, ts_action_unknown_name_strike,
    ts_action_payload_on_none, ts_action_three_strikes,
    ts_action_failed_step_blocks, ts_action_T_eq_H_no_strike,
    ts_action_empty_feasible,
    # Refusal (5)
    ts_refusal_T_eq_H, ts_refusal_T_lt_H, ts_refusal_T_unmeasurable,
    ts_refusal_H_unconfigured_correct_tag, ts_refusal_bool_rejected,
    # Taxonomy (2)
    ts_taxonomy_distinct, ts_taxonomy_refusal_separate,
    # Harness (3)
    ts_harness_no_camera, ts_harness_no_inverse, ts_harness_camera_bound,
]


def run_all() -> List[Tuple[str, Status, str]]:
    results = []
    for fn in _ALL_TESTS:
        try:
            results.append(fn())
        except Exception as e:
            results.append(_status(fn.__name__, Status.FAIL,
                                   f"{type(e).__name__}: {e}"))
    return results


def main():
    import tempfile, shutil  # local import for tempdirs

    print("=" * 78)
    print(f"GE2 {GE2_VERSION} — Input Adapters (MAPPING ONLY)")
    print(f"order={BUILD_ORDER}  audit={AUDIT_ORDER}")
    print(f"arch={ARCH_REF}")
    print(f"scope: frame/grid, state-norm, action-mapping, F-17 taxonomy")
    print(f"sdk_vocabulary_bound={'yes' if _SDK_BOUND else 'no'}")
    print(f"NOTE: GE2 makes NO authentication claims. GE4 authenticates.")
    print("=" * 78)

    # Global tempdir import (tests use it)
    global tempfile
    import tempfile

    results = run_all()
    n_fail = n_blocked = 0
    for name, st, detail in results:
        print(f"{st.value:7}  {name:50}  {detail}")
        if st is Status.FAIL:
            n_fail += 1
        elif st is Status.BLOCKED:
            n_blocked += 1

    print("-" * 78)
    print(f"PASS={len(results)-n_fail-n_blocked}  FAIL={n_fail}  BLOCKED={n_blocked}")
    print("-" * 78)
    print("NOTE: All tests are STRUCTURAL unit tests. None prove authentication.")
    print("NOTE: P1-01 fixed: fd-relative no-follow path walk (TOCTOU-safe).")
    print("NOTE: P1-04 fixed: SDK vocabulary check when arcengine is bound.")
    print("NOTE: P1-05 fixed: H_UNCONFIGURED -> PENDING_AUTHORITATIVE_H tag.")
    print("NOTE: GE4 MUST authenticate census/contract/pinset before calling GE2.")

    if n_fail:
        print("RESULT: FAILED"); sys.exit(1)
    if n_blocked:
        print("RESULT: SUITE INCOMPLETE — BLOCKED present"); sys.exit(2)
    print(f"RESULT: ALL GE2 {GE2_VERSION} TESTS — PASS")
    sys.exit(0)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile goa/agent/ge3/ge3_selector.py
# ============================================================================
# GE3 v2.7 — Budget-aware Tier Selector + DQC (ALGORITHM LAYER · SPEC-COMPLETE)
# v2.7 FIX (Indus, 20 Oct 2026, AGA GE3 audit F-03 + loader/resolver):
#   F-03  cross-process test now loads the module BY PATH (importlib) -> a dotted
#         filename no longer yields "import GE3_selector_v2.6 as M".
#   F-08  unit_f08_manifest_matches_loader treats an ABSENT-DEPENDENCY loader
#         failure (grid_text not found) as BLOCKED, not a loader/resolver FAIL.
# v2.6 FIX (Indus, 20 Oct 2026): measure_costs() accepts the GE2 wire-header STRING
#   as well as a Mapping -> GE2 adapt() -> GE3 measure_costs() handoff works directly
#   (previously a str header raised ValueError in dict()).
# Order: GoA/INDUS/DIR/2026/098 | Ruling: GoA/INDUS/DIR/2026/100
# Res: GoA/RES/GE3/2026/KIMI-007 | Arch: ARCH/2026/103 + 111
# Audit: GoA/INDUS/DIR/2026/099 | AGA Art. 128
# Basis: Research Report DIR/2026/092 Rev v38 §5.3a (relation semantics) + §6
#
# STATUS: DEV_PASS (not PASS) — C-3/D8 harness binding required for PASS
#
# v2.4 CHANGES (from AGA v2.3 re-audit, F-01..F-05):
#   F-01  _classify_manifest_status: COMPLETE_PENDING_SIGNATURE + WEAK wire pin
#         → BLOCKED, not PASS. Release-integrity evidence honest; a weak
#         F-08 pin cannot masquerade as full PASS.
#   F-02  _stage_snapshot: chmod root too (topdown=False then explicit root
#         chmod 0o555); _load_pinned_tokenizer: post-load re-fingerprint,
#         TOKENIZER_MISMATCH on drift. _cleanup_staging restores root
#         writability before rmtree. NOTE: chmod is a window-reducer;
#         process-isolation remainder is GE4 (F-03 gate).
#   F-03  New _compute_wire_pin_strength(): STRONG requires VALID sha256
#         (64 lowercase hex) AND VALID bytes (positive non-bool int), not
#         just non-None. _check_pin_format also validates optional wire
#         fields when populated.
#   F-04  New _EXPECTED_TEST_NAMES tuple — HARDCODED release-controlled
#         inventory, independent of _all_tests(). run_all detects duplicates
#         in _all_tests() and compares actual names vs the inventory.
#   F-05  ts_q4_q5_ge1_path extended: above/left/bbox_contains fixtures
#         through GE1 serialization, both directions where possible
#         (including bbox_contains reverse: contained object gets lower id).
#
# ============================================================================
# THREAT MODEL
# ============================================================================
# GE3 v2.4 is the ALGORITHM LAYER. It computes a selection from
# (grid, cap, costs, policy). It does NOT authenticate that costs, policy,
# or upstream module bytes came from a trusted source.
#
# WHAT v2.4 CLOSES (in-process, verifiable):
#   ✓ All v2.3 fixes retained (fingerprint-bound staging, per-relation
#     canonical rules, purge-on-any-reject, shared verifier, tokenizer
#     staging, weighted DQC validation, adversarial matrices).
#   ✓ F-01  Weak wire pin → manifest BLOCKED not PASS.
#   ✓ F-02  Staging root chmod'd read-only + post-load drift detection.
#   ✓ F-03  STRONG label requires valid sha256 AND bytes.
#   ✓ F-04  Independent test inventory (release-controlled tuple).
#   ✓ F-05  Directed relations (above/left/bbox_contains) validated against
#           actual GE1 serialization, including reverse-direction bbox case.
#
# WHAT v2.4 DOES NOT CLOSE — per ARCH/103 §7 (out of GE3 scope, owners set):
#   ✗ F-03(v2.0) TOCTOU between verify and exec — GE4 process isolation.
#   ✗ F-04(v2.0) Policy provenance — GE4 gate.
#   ✗ F-06(v2.0) Manifest/loader reconciliation across release boundary — GE4.
#   ✗ F-08(v2.0) wire.py sha256/bytes VALUES — certification body.
#   ✗ Real dispatcher (T-SPATIAL-06) — GE4 implementation.
#   ✗ C-3/D8 harness binding — GE2.
# ============================================================================

from __future__ import annotations
import sys
sys.dont_write_bytecode = True

import os, re, json, hashlib, math, tempfile, glob, importlib, importlib.util, shutil
from enum import Enum
from dataclasses import dataclass
from types import MappingProxyType
from typing import Optional, List, Dict, Tuple, Any, Mapping, Protocol

GE3_VERSION = "v2.7"
GE3_STATUS  = "DEV_PASS"
SPEC_REV    = "DIR092 Rev v38"
ARCH_REF    = "ARCH/2026/103 + 111"
ORDER_REF   = "GoA/INDUS/DIR/2026/098"
AUDIT_REF   = "GoA/INDUS/DIR/2026/099"
RULING_REF  = "GoA/INDUS/DIR/2026/100"
RES_REF     = "GoA/RES/GE3/2026/KIMI-007"
AGA_REF     = "AGA Art. 128"

K_MAX = 6
QWEN_REPO             = "Qwen/Qwen3.8-27B"
QWEN_REVISION         = "366a23fd246985933964e9d21ac63eb3a93190ab"
TOKENIZER_JSON_SHA256 = "0997f410c57a1f4e53b09e4be8f4a172d90edd9564368fb0847030937229b9f3"

_GRID_TEXT_LOAD_COUNTER = 0

OUT_OF_SCOPE_GATES = MappingProxyType({
    "F-03": MappingProxyType({
        "owner": "GE4", "arch": "ARCH/2026/103 §7",
        "reason": "TOCTOU between verify and exec — requires process isolation",
        "closure_evidence_required": "byte-identity verifier-to-execution in live pipeline",
    }),
    "F-04": MappingProxyType({
        "owner": "GE4", "arch": "ARCH/2026/103 §7",
        "reason": "policy provenance — GE3 validates STRUCTURE not AUTHENTICITY",
        "closure_evidence_required": "authenticated policy binding at dispatch",
    }),
    "F-06": MappingProxyType({
        "owner": "GE4", "arch": "ARCH/2026/103 §7",
        "reason": "manifest/loader reconciliation across release boundary",
        "closure_evidence_required": "signed manifest + verified reconciliation report",
    }),
    "F-08": MappingProxyType({
        "owner": "Certification body", "arch": "ARCH/2026/103 §7",
        "reason": "wire.py sha256/bytes VALUES must come from signed release metadata",
        "closure_evidence_required": "authoritative sha256 + bytes for wire.py",
    }),
})

def _freeze(table):
    return MappingProxyType(dict(table))

_CANDIDATE_KEYS = ("BASELINE","T1","T2","T3(1)","T3(2)","T3(3)","T3(4)","T3(5)","T3(6)")
_get_candidates = (lambda _c=_CANDIDATE_KEYS: _c)
CANDIDATE_KEYS = _CANDIDATE_KEYS

_RANK = _freeze({"BASELINE": 0, "T1": 1, "T2": 2, "T3": 3})
_rank_get = _RANK.get
RANK = _RANK

_DQC_COVERAGE = _freeze({
    "BASELINE": frozenset({"Q3","Q6"}), "T1": frozenset({"Q3","Q6"}),
    "T2": frozenset({"Q1","Q2","Q3","Q6"}),
    "T3(1)": frozenset({"Q1","Q2","Q3","Q4","Q6"}),
    "T3(2)": frozenset({"Q1","Q2","Q3","Q4","Q5","Q6"}),
    "T3(3)": frozenset({"Q1","Q2","Q3","Q4","Q5","Q6"}),
    "T3(4)": frozenset({"Q1","Q2","Q3","Q4","Q5","Q6"}),
    "T3(5)": frozenset({"Q1","Q2","Q3","Q4","Q5","Q6"}),
    "T3(6)": frozenset({"Q1","Q2","Q3","Q4","Q5","Q6"}),
})
_dqc_cov_get = _DQC_COVERAGE.get
DQC_COVERAGE = _DQC_COVERAGE

_VALID_QIDS  = frozenset({"Q1","Q2","Q3","Q4","Q5","Q6"})
_VALID_TIERS = frozenset({"BASELINE","T1","T2","T3"})

_DQC_PRIMITIVES = _freeze({
    "a": "field read", "b": "counting explicit records",
    "c": "arithmetic on a SINGLE object's bbox endpoints",
    "d": "grid_cell_lookup(r,c)", "e": "distinct_row_patterns",
    "f": "explicit-record iteration over OBJECTS",
    "g": "unary predicate aggregation with any()/all()",
})
DQC_PRIMITIVES = _DQC_PRIMITIVES

_DQC_QUERY_BANK = _freeze({
    "Q1": _freeze({"word": "How many non-background objects are there?",
                   "type": "int", "primitives": frozenset({"b","f"}),
                   "method": "count OBJLINE records inside the OBJECTS: block, excluding the OBJECTS marker itself"}),
    "Q2": _freeze({"word": "Does any object have inclusive bbox height (r1-r0+1) >= 60?",
                   "type": "bool", "primitives": frozenset({"c","f","g"}),
                   "method": "iteration + single-record bbox arithmetic + any()"}),
    "Q3": _freeze({"word": "What palette char is at cell (r=32,c=32)?",
                   "type": "char", "primitives": frozenset({"d"}),
                   "method": "grid_cell_lookup(32,32)"}),
    "Q4": _freeze({"word": "Do any two objects share a grid edge (4-neighbour adjacency)?",
                   "type": "bool", "primitives": frozenset({"f","g"}),
                   "method": "relation adjacent (serialized)"}),
    "Q5": _freeze({"word": "Do any two objects have OVERLAPPING inclusive bbox column intervals?",
                   "type": "bool", "primitives": frozenset({"f","g"}),
                   "method": "relation aligned_cols (serialized)"}),
    "Q6": _freeze({"word": "How many distinct row patterns are there?",
                   "type": "int", "primitives": frozenset({"e"}),
                   "method": "distinct_row_patterns"}),
})
DQC_QUERY_BANK = _DQC_QUERY_BANK

P3_PROBE = _freeze({"id":"P3","relation":"aligned_cols","type":"bool"})
Q5_QUERY = _freeze({"id":"Q5","relation":"aligned_cols","type":"bool"})

_KNOWN_UPSTREAM = _freeze({
    "grid_text.py": _freeze({
        "md5":    "bdbc329a990d82c19bf4b4e22b838bdd",
        "sha256": "864fb4051a978f97267c12e71fe9aad55d2a6ec2d77f1536e75722087a0aacca",
        "bytes":  37118}),
    "wire.py": _freeze({
        "md5":    "de722f4fafe10c5fccf9dc4d3bbea388",
        # F-08 CLOSED (SM-declared, 20 Oct 2026): the frozen GE1 v37 artifact's
        # sha256/bytes, declared by the Senior Manager as authoritative release
        # metadata for the delivered wire.py (provenance: the delivered GE1 v37).
        "sha256": "d64151faa5a141c5b22a02c44b06b624d99501c6c6a633f7cc8a446d261c4157",
        "bytes":  171699,
    }),
})
_known_upstream_get = _KNOWN_UPSTREAM.get
KNOWN_UPSTREAM = _KNOWN_UPSTREAM

_MANDATORY_PIN_FIELDS = _freeze({
    "grid_text.py": ("md5","sha256","bytes"),
    "wire.py":      ("md5","sha256","bytes"),  # F-08 closed: strong pin enforced
})
MANDATORY_PIN_FIELDS = _MANDATORY_PIN_FIELDS
_MANDATORY_UPSTREAM = frozenset({"grid_text.py","wire.py"})


# ============================================================================
# POLICY VALIDATION
# ============================================================================
def _validate_dqc_coverage_shape(dqc_coverage, candidate_keys):
    if not isinstance(dqc_coverage, Mapping):
        raise ValueError(f"dqc_coverage must be Mapping; got {type(dqc_coverage).__name__}")
    if set(dqc_coverage.keys()) != set(candidate_keys):
        raise ValueError("dqc_coverage keys != candidate_keys")
    for k in candidate_keys:
        cov = dqc_coverage[k]
        if not isinstance(cov, (set, frozenset, tuple, list)):
            raise ValueError(f"dqc_coverage[{k}] must be collection; got {type(cov).__name__}")
        try:
            as_set = frozenset(cov)
        except TypeError as e:
            raise ValueError(f"dqc_coverage[{k}] unhashable: {e}")
        if len(as_set) != len(cov):
            raise ValueError(f"dqc_coverage[{k}] has duplicates")
        unknown = as_set - _VALID_QIDS
        if unknown:
            raise ValueError(f"dqc_coverage[{k}] unknown Q-ids: {sorted(unknown)}")


def _validate_rank_shape(rank):
    if not isinstance(rank, Mapping):
        raise ValueError("rank must be Mapping")
    if set(rank.keys()) != _VALID_TIERS:
        raise ValueError(f"rank keys != {sorted(_VALID_TIERS)}")
    for t, v in rank.items():
        if type(v) is bool or type(v) is not int:
            raise ValueError(f"rank[{t}] must be non-bool int")


def _validate_candidate_keys_uniqueness(ck_tuple):
    if not isinstance(ck_tuple, (tuple, list)):
        raise ValueError("candidate_keys must be tuple/list")
    ck = tuple(ck_tuple)
    if not all(isinstance(k, str) and k for k in ck):
        raise ValueError("candidate_keys entries must be non-empty str")
    if len(set(ck)) != len(ck):
        dupes = sorted({k for k in ck if ck.count(k) > 1})
        raise ValueError(f"candidate_keys has duplicates {dupes}")
    return ck


def _validate_fingerprint_consistency(tokenizer_sha256, fingerprint, require_fingerprint=False):
    if fingerprint is None:
        if require_fingerprint:
            raise ValueError("require_fingerprint=True but tokenizer_fingerprint is None")
        return
    if not isinstance(fingerprint, tuple):
        raise ValueError("tokenizer_fingerprint must be tuple of (path, sha256)")
    seen_paths = set()
    tj_count = 0
    tj_hash = None
    for entry in fingerprint:
        if not isinstance(entry, tuple) or len(entry) != 2:
            raise ValueError("tokenizer_fingerprint entries must be 2-tuples")
        p, h = entry
        if not isinstance(p, str) or not isinstance(h, str):
            raise ValueError("entries must be (str, str)")
        if not re.fullmatch(r"[0-9a-f]{64}", h):
            raise ValueError(f"hash not 64-hex: {h[:12]}…")
        if p in seen_paths:
            raise ValueError(f"duplicate path {p!r}")
        seen_paths.add(p)
        if p == "tokenizer.json":
            tj_count += 1
            tj_hash = h
    if require_fingerprint and tj_count == 0:
        raise ValueError("require_fingerprint=True but no 'tokenizer.json' entry")
    if tj_count > 0 and tj_hash != tokenizer_sha256:
        raise ValueError(f"tokenizer.json hash != tokenizer_sha256")


@dataclass(frozen=True)
class Policy:
    tokenizer_sha256:      str
    dqc_coverage:          Mapping[str, frozenset]
    rank:                  Mapping[str, int]
    candidate_keys:        Tuple[str, ...]
    tokenizer_fingerprint: Optional[Tuple[Tuple[str, str], ...]] = None
    require_fingerprint:   bool = False

    def __post_init__(self):
        if not isinstance(self.tokenizer_sha256, str) or \
                not re.fullmatch(r"[0-9a-f]{64}", self.tokenizer_sha256):
            raise ValueError("tokenizer_sha256 must be 64 lowercase hex")
        ck_tuple = _validate_candidate_keys_uniqueness(self.candidate_keys)
        object.__setattr__(self, "candidate_keys", ck_tuple)
        _validate_dqc_coverage_shape(self.dqc_coverage, ck_tuple)
        canonical_cov = {k: frozenset(self.dqc_coverage[k]) for k in ck_tuple}
        object.__setattr__(self, "dqc_coverage", MappingProxyType(canonical_cov))
        _validate_rank_shape(self.rank)
        canonical_rank = {t: int(rv) for t, rv in self.rank.items()}
        object.__setattr__(self, "rank", MappingProxyType(canonical_rank))
        if type(self.require_fingerprint) is not bool:
            raise ValueError("require_fingerprint must be bool")
        _validate_fingerprint_consistency(self.tokenizer_sha256,
                                           self.tokenizer_fingerprint,
                                           self.require_fingerprint)
        if self.tokenizer_fingerprint is not None:
            object.__setattr__(self, "tokenizer_fingerprint", tuple(self.tokenizer_fingerprint))


REFERENCE_POLICY = Policy(
    tokenizer_sha256=TOKENIZER_JSON_SHA256,
    dqc_coverage=_DQC_COVERAGE, rank=_RANK, candidate_keys=_CANDIDATE_KEYS,
    tokenizer_fingerprint=None, require_fingerprint=False,
)


# ============================================================================
# TYPES
# ============================================================================
class Reason(str, Enum):
    CAP_INVALID = "CAP_INVALID"
    CAP_UNMEASURABLE = "CAP_UNMEASURABLE"
    COSTS_NOT_MEASURED = "COSTS_NOT_MEASURED"
    COST_BUNDLE_MALFORMED = "COST_BUNDLE_MALFORMED"
    POLICY_MALFORMED = "POLICY_MALFORMED"
    POLICY_MISMATCH = "POLICY_MISMATCH"
    TOKENIZER_UNAVAILABLE = "TOKENIZER_UNAVAILABLE"
    TOKENIZER_UNPINNED = "TOKENIZER_UNPINNED"
    TOKENIZER_MISMATCH = "TOKENIZER_MISMATCH"
    ENCODER_UNAVAILABLE = "ENCODER_UNAVAILABLE"
    ENCODER_ERROR = "ENCODER_ERROR"
    EMPTY_FEASIBLE = "EMPTY_FEASIBLE"
    DQC_TABLE_MALFORMED = "DQC_TABLE_MALFORMED"
    DQC_TABLE_TAMPERED = "DQC_TABLE_TAMPERED"
    INTERNAL_ERROR = "INTERNAL_ERROR"

_RECOVERY_TABLE = _freeze({
    Reason.CAP_INVALID: "cap invalid", Reason.CAP_UNMEASURABLE: "cap unmeasurable",
    Reason.COSTS_NOT_MEASURED: "costs missing/unmeasured",
    Reason.COST_BUNDLE_MALFORMED: "CostBundle malformed",
    Reason.POLICY_MALFORMED: "Policy malformed",
    Reason.POLICY_MISMATCH: "policy mismatch",
    Reason.TOKENIZER_UNAVAILABLE: "tokenizer unavailable",
    Reason.TOKENIZER_UNPINNED: "tokenizer digest missing/mismatched",
    Reason.TOKENIZER_MISMATCH: "fingerprint/stage drift",
    Reason.ENCODER_UNAVAILABLE: "GE1 not importable",
    Reason.ENCODER_ERROR: "GE1 error",
    Reason.EMPTY_FEASIBLE: "no feasible candidate",
    Reason.DQC_TABLE_MALFORMED: "dqc_table malformed",
    Reason.DQC_TABLE_TAMPERED: "dqc_table tampered",
    Reason.INTERNAL_ERROR: "internal error",
})
RECOVERY_TABLE = _RECOVERY_TABLE


@dataclass(frozen=True)
class CostBundle:
    costs:                 Tuple[Tuple[str, Optional[int]], ...]
    reasons:               Tuple[Tuple[str, str], ...] = ()
    measurement_error:     Optional[str] = None
    tokenizer_id:          Optional[str] = None
    tokenizer_revision:    Optional[str] = None
    tokenizer_sha256:      Optional[str] = None
    tokenizer_fingerprint: Optional[Tuple[Tuple[str, str], ...]] = None
    tokenizer_stage_ref:   Optional[str] = None

    def cost_dict(self): return dict(self.costs)
    def reason_dict(self): return dict(self.reasons)
    def is_measured(self) -> bool:
        if self.measurement_error is not None: return False
        if len(self.costs) != len(_get_candidates()): return False
        if {k for k, _ in self.costs} != set(_get_candidates()): return False
        return True


@dataclass(frozen=True)
class Selection:
    tier: str; k: int; cost: int; dqc: int
    feasible: bool = True; rank: int = 0
    infeasible_candidates: Tuple[Tuple[str, str], ...] = ()
    monotone_invariant_holds: bool = True
    monotone_violations: Tuple[str, ...] = ()
    deployment_tokenizer_revalidation_required: bool = True
    measured_tokenizer_sha256: Optional[str] = None


@dataclass(frozen=True)
class Recovery:
    feasible: bool; cap_exceeded: bool; reason: Reason
    detail: str = ""; budget: Optional[int] = None
    infeasible_candidates: Tuple[Tuple[str, str], ...] = ()
    monotone_invariant_holds: bool = True
    monotone_violations: Tuple[str, ...] = ()
    deployment_tokenizer_revalidation_required: bool = True
    measured_tokenizer_sha256: Optional[str] = None


class Status(str, Enum):
    PASS = "PASS"; FAIL = "FAIL"; BLOCKED = "BLOCKED"


class Dispatcher(Protocol):
    def dispatch(self, selection: Selection) -> str: ...
    def refuse(self, recovery: Recovery) -> str: ...


# ============================================================================
# ARITHMETIC
# ============================================================================
def round_half_up(x: float) -> int:
    return int(math.floor(x + 0.5))


def research_budget(cap: int) -> int:
    if type(cap) is bool or type(cap) is not int or cap <= 0:
        raise ValueError(f"cap must be positive int")
    return cap - (cap + 5) // 10


# ============================================================================
# VALIDATION
# ============================================================================
def _parse_key(key: str) -> Tuple[str, int]:
    if key.startswith("T3(") and key.endswith(")"):
        return ("T3", int(key[3:-1]))
    return (key, 0)


def _validate_policy(policy) -> Optional[Recovery]:
    if not isinstance(policy, Policy):
        return Recovery(False, False, Reason.POLICY_MALFORMED,
                        f"policy must be Policy; got {type(policy).__name__}")
    if not isinstance(policy.tokenizer_sha256, str) or \
            not re.fullmatch(r"[0-9a-f]{64}", policy.tokenizer_sha256):
        return Recovery(False, False, Reason.POLICY_MALFORMED, "tokenizer_sha256 invalid")
    if type(policy.require_fingerprint) is not bool:
        return Recovery(False, False, Reason.POLICY_MALFORMED, "require_fingerprint not bool")
    try:
        _validate_candidate_keys_uniqueness(policy.candidate_keys)
        _validate_dqc_coverage_shape(policy.dqc_coverage, policy.candidate_keys)
        _validate_rank_shape(policy.rank)
        _validate_fingerprint_consistency(policy.tokenizer_sha256,
                                           policy.tokenizer_fingerprint,
                                           policy.require_fingerprint)
    except ValueError as e:
        return Recovery(False, False, Reason.POLICY_MALFORMED, str(e))
    if set(policy.candidate_keys) != set(_get_candidates()):
        return Recovery(False, False, Reason.POLICY_MALFORMED, "candidate_keys != module candidates")
    return None


def _validate_dqc_table(dqc_table, policy: Policy) -> Optional[Recovery]:
    if dqc_table is None: return None
    if not isinstance(dqc_table, Mapping):
        return Recovery(False, False, Reason.DQC_TABLE_MALFORMED, "not Mapping")
    if set(dqc_table.keys()) != set(policy.candidate_keys):
        return Recovery(False, False, Reason.DQC_TABLE_MALFORMED, "keys mismatch")
    for k in policy.candidate_keys:
        frozen = policy.dqc_coverage[k]
        override = dqc_table[k]
        if not isinstance(override, (set, frozenset, tuple, list)):
            return Recovery(False, False, Reason.DQC_TABLE_MALFORMED, f"{k}: collection")
        try: ov = frozenset(override)
        except TypeError as e:
            return Recovery(False, False, Reason.DQC_TABLE_MALFORMED, f"{k}: {e}")
        if ov != frozen:
            return Recovery(False, False, Reason.DQC_TABLE_TAMPERED, f"{k}: differs")
    return None


def _validate_cost_bundle(bundle, policy: Policy) -> Optional[Recovery]:
    if not isinstance(bundle, CostBundle):
        return Recovery(False, False, Reason.COSTS_NOT_MEASURED, "not CostBundle")
    if bundle.measurement_error is not None:
        return Recovery(False, False, Reason.COSTS_NOT_MEASURED, bundle.measurement_error)
    # F-04 v2.5: when the bundle fingerprint matches the policy's (the measured-tokenizer
    # path), the tokenizer_stage_ref MUST be present and consistent with that fingerprint.
    if getattr(policy, "require_fingerprint", False):
        fp = getattr(bundle, "tokenizer_fingerprint", None)
        if fp is not None and fp == getattr(policy, "tokenizer_fingerprint", None):
            sref = getattr(bundle, "tokenizer_stage_ref", None)
            if not sref:
                return Recovery(False, False, Reason.COST_BUNDLE_MALFORMED,
                                "require_fingerprint: tokenizer_stage_ref missing")
            exp_ref = hashlib.sha256(json.dumps(fp, sort_keys=True).encode()).hexdigest()
            if sref != exp_ref:
                return Recovery(False, False, Reason.COST_BUNDLE_MALFORMED,
                                "tokenizer_stage_ref inconsistent with fingerprint")
    costs = bundle.costs
    if not isinstance(costs, tuple):
        return Recovery(False, False, Reason.COST_BUNDLE_MALFORMED, "costs not tuple")
    seen = set()
    for i, entry in enumerate(costs):
        if not isinstance(entry, tuple) or len(entry) != 2:
            return Recovery(False, False, Reason.COST_BUNDLE_MALFORMED, f"costs[{i}] not 2-tuple")
        k, v = entry
        if not isinstance(k, str) or not k:
            return Recovery(False, False, Reason.COST_BUNDLE_MALFORMED, f"costs[{i}] key")
        if k in seen:
            return Recovery(False, False, Reason.COST_BUNDLE_MALFORMED, f"dup {k!r}")
        seen.add(k)
        if v is not None and (type(v) is not int or v < 0):
            return Recovery(False, False, Reason.COST_BUNDLE_MALFORMED, f"costs[{i}] val")
    if seen != set(policy.candidate_keys):
        return Recovery(False, False, Reason.COSTS_NOT_MEASURED, "key set mismatch")
    reasons_t = bundle.reasons
    if not isinstance(reasons_t, tuple):
        return Recovery(False, False, Reason.COST_BUNDLE_MALFORMED, "reasons not tuple")
    reasons = {}
    for i, r_entry in enumerate(reasons_t):
        if not isinstance(r_entry, tuple) or len(r_entry) != 2:
            return Recovery(False, False, Reason.COST_BUNDLE_MALFORMED, f"reasons[{i}]")
        rk, rv = r_entry
        if not isinstance(rk, str) or not isinstance(rv, str):
            return Recovery(False, False, Reason.COST_BUNDLE_MALFORMED, f"reasons[{i}] types")
        if rk in reasons:
            return Recovery(False, False, Reason.COST_BUNDLE_MALFORMED, f"reasons dup {rk!r}")
        if rk not in policy.candidate_keys:
            return Recovery(False, False, Reason.COST_BUNDLE_MALFORMED, f"reasons unknown {rk!r}")
        reasons[rk] = rv
    cost_map = dict(costs)
    for k in policy.candidate_keys:
        if cost_map[k] is None and not reasons.get(k):
            return Recovery(False, False, Reason.COST_BUNDLE_MALFORMED, f"{k}: None needs reason")
    return None


def check_monotone_cost(costs: Mapping[str, Optional[int]]) -> Tuple[bool, List[str]]:
    violations: List[str] = []
    for k in range(1, K_MAX):
        a = costs.get(f"T3({k})"); b = costs.get(f"T3({k+1})")
        if a is None or b is None:
            violations.append(f"T3({k})={a!r} -> T3({k+1})={b!r} (missing)")
            continue
        if type(a) is not int or type(b) is not int:
            violations.append(f"T3({k}) -> T3({k+1}) non-int")
            continue
        if a > b:
            violations.append(f"T3({k})={a} > T3({k+1})={b}")
    return (len(violations) == 0, violations)


# ============================================================================
# SELECT
# ============================================================================
def select(grid, cap, *, costs: Optional[CostBundle] = None,
           policy: Optional[Policy] = None,
           dqc_table: Optional[Mapping[str, Any]] = None) -> Any:
    try:
        return _select_impl(grid, cap, costs, policy, dqc_table)
    except Exception as e:
        return Recovery(False, False, Reason.INTERNAL_ERROR,
                        f"select body raised {type(e).__name__}: {e}")


def _select_impl(grid, cap, costs, policy, dqc_table) -> Any:
    if type(cap) is bool or type(cap) is not int or cap <= 0:
        return Recovery(False, False, Reason.CAP_INVALID,
                        f"cap must be positive int; got {type(cap).__name__}={cap!r}")
    try:
        budget = research_budget(cap)
    except Exception as e:
        return Recovery(False, False, Reason.INTERNAL_ERROR, f"budget: {e}")

    if policy is None:
        return Recovery(False, False, Reason.POLICY_MALFORMED,
                        "select() requires explicit policy", budget=budget)
    policy_err = _validate_policy(policy)
    if policy_err is not None:
        return Recovery(policy_err.feasible, policy_err.cap_exceeded,
                        policy_err.reason, policy_err.detail, budget=budget)

    if costs is None:
        return Recovery(False, False, Reason.COSTS_NOT_MEASURED,
                        "select() requires explicit costs", budget=budget)
    bundle_err = _validate_cost_bundle(costs, policy)
    if bundle_err is not None:
        return Recovery(bundle_err.feasible, bundle_err.cap_exceeded,
                        bundle_err.reason, bundle_err.detail, budget=budget,
                        measured_tokenizer_sha256=getattr(costs, "tokenizer_sha256", None))

    if costs.tokenizer_sha256 != policy.tokenizer_sha256:
        return Recovery(False, False, Reason.TOKENIZER_UNPINNED,
                        f"costs.tokenizer_sha256 != policy.tokenizer_sha256",
                        budget=budget,
                        measured_tokenizer_sha256=costs.tokenizer_sha256)

    if policy.require_fingerprint:
        if costs.tokenizer_fingerprint is None:
            return Recovery(False, False, Reason.TOKENIZER_UNPINNED,
                            "fingerprint missing", budget=budget,
                            measured_tokenizer_sha256=costs.tokenizer_sha256)
        if costs.tokenizer_fingerprint != policy.tokenizer_fingerprint:
            return Recovery(False, False, Reason.TOKENIZER_MISMATCH,
                            "fingerprint mismatch", budget=budget,
                            measured_tokenizer_sha256=costs.tokenizer_sha256)

    dqc_err = _validate_dqc_table(dqc_table, policy)
    if dqc_err is not None:
        return Recovery(dqc_err.feasible, dqc_err.cap_exceeded, dqc_err.reason,
                        dqc_err.detail, budget=budget,
                        measured_tokenizer_sha256=costs.tokenizer_sha256)

    cost_d = dict(costs.costs)
    reason_d = dict(costs.reasons)
    infeasible: List[Tuple[str, str]] = []
    for k in policy.candidate_keys:
        v = cost_d.get(k)
        if v is None:
            infeasible.append((k, reason_d.get(k, "COST_NOT_MEASURED")))

    mono_ok, mono_violations = check_monotone_cost(cost_d)

    feasible: List[Tuple[str, str, int, int, int, int]] = []
    for key in policy.candidate_keys:
        c = cost_d.get(key)
        if c is None or type(c) is not int or c < 0 or c > budget:
            continue
        tier, k = _parse_key(key)
        dqc = len(policy.dqc_coverage[key])
        feasible.append((key, tier, k, c, dqc, policy.rank[tier]))

    tok_sha = costs.tokenizer_sha256
    if not feasible:
        return Recovery(False, True, Reason.EMPTY_FEASIBLE,
                        f"no candidate within budget={budget}", budget=budget,
                        infeasible_candidates=tuple(infeasible),
                        monotone_invariant_holds=mono_ok,
                        monotone_violations=tuple(mono_violations),
                        measured_tokenizer_sha256=tok_sha)

    feasible.sort(key=lambda r: (-r[4], r[3], r[5], r[2]))
    key, tier, k, cost, dqc, rank = feasible[0]
    return Selection(tier=tier, k=k, cost=cost, dqc=dqc, feasible=True, rank=rank,
                     infeasible_candidates=tuple(infeasible),
                     monotone_invariant_holds=mono_ok,
                     monotone_violations=tuple(mono_violations),
                     measured_tokenizer_sha256=tok_sha)


# ============================================================================
# GE1 LOADER (F-04 v2.3: shared verifier)
# ============================================================================
_LOADER_LOG: List[Tuple[str, ...]] = []

def _loader_log_invariants(log):
    for i, entry in enumerate(log):
        if not entry: continue
        if entry[0] == "import":
            path = entry[1]
            has_prior = any((e[0] == "hash" and e[1] == path)
                            for e in log[:i] if e)
            if not has_prior:
                return f"import of {path} at idx {i} has no preceding hash"
    return None


def _verify_wire_artifact(p: str) -> Tuple[Optional[bytes], Optional[str]]:
    """F-04 v2.3: single source of truth for wire.py verification."""
    exp = _known_upstream_get("wire.py", {})
    if not isinstance(exp, Mapping):
        return None, "wire.py pin not configured"
    exp_md5 = exp.get("md5")
    if not exp_md5:
        return None, "wire.py md5 not configured"
    try:
        with open(p, "rb") as f: b = f.read()
    except OSError as e:
        return None, f"unreadable {p}: {e}"
    if hashlib.md5(b).hexdigest() != exp_md5:
        return None, "md5 mismatch"
    exp_sha = exp.get("sha256")
    if exp_sha is not None and hashlib.sha256(b).hexdigest() != exp_sha:
        return None, "sha256 mismatch"
    exp_bytes = exp.get("bytes")
    if exp_bytes is not None and len(b) != exp_bytes:
        return None, f"bytes mismatch: {len(b)} != {exp_bytes}"
    return b, None


def _find_ge1_candidates_on_disk() -> List[str]:
    roots = [os.getcwd(),
             os.path.dirname(os.path.abspath(__file__)) or ".",
             os.path.abspath(os.path.join(os.path.dirname(__file__) or ".", ".."))]
    hits: List[str] = []
    for root in roots:
        for rel in ("ge1/wire.py", "wire.py", "GE1_wire.py"):
            p = os.path.join(root, rel)
            if os.path.exists(p) and os.path.isfile(p):
                hits.append(os.path.abspath(p))
    return sorted(set(hits))


def _find_grid_text_candidates() -> List[str]:
    roots = [os.getcwd(),
             os.path.dirname(os.path.abspath(__file__)) or ".",
             os.path.abspath(os.path.join(os.path.dirname(__file__) or ".", "..")),
             os.path.abspath(os.path.join(os.path.dirname(__file__) or ".", "..", ".."))]
    hits: List[str] = []
    for root in roots:
        for rel in ("grid_text.py",
                    os.path.join("results", "goa", "agent", "grid_text.py"),
                    os.path.join("..", "results", "goa", "agent", "grid_text.py")):
            p = os.path.join(root, rel)
            if os.path.exists(p) and os.path.isfile(p):
                hits.append(os.path.abspath(p))
    return sorted(set(hits))


def _verify_grid_text_bytes(p: str) -> Tuple[Optional[bytes], Optional[str]]:
    exp = _known_upstream_get("grid_text.py", {})
    if not isinstance(exp, Mapping):
        return None, "grid_text.py pin not configured"
    try:
        with open(p, "rb") as f: b = f.read()
    except OSError as e:
        return None, f"unreadable {p}: {e}"
    md5 = hashlib.md5(b).hexdigest()
    sha = hashlib.sha256(b).hexdigest()
    if "md5" in exp and md5 != exp["md5"]:
        return None, "md5 mismatch"
    if "sha256" in exp and sha != exp["sha256"]:
        return None, "sha256 mismatch"
    if "bytes" in exp and len(b) != exp["bytes"]:
        return None, f"bytes mismatch"
    return b, None


def _load_verified_grid_text() -> Tuple[Optional[Any], Optional[str]]:
    """RULING 1(A): NO CACHE — unique per-call name."""
    global _GRID_TEXT_LOAD_COUNTER
    candidates = _find_grid_text_candidates()
    if not candidates:
        return None, "grid_text.py not found"
    verified: Optional[Tuple[str, bytes]] = None
    for p in candidates:
        b, err = _verify_grid_text_bytes(p)
        if b is not None:
            verified = (p, b); break
    if verified is None:
        return None, "no grid_text.py matched pin"
    p, b = verified
    _GRID_TEXT_LOAD_COUNTER += 1
    mod_name = f"_ge3_gridtext_loaded_{_GRID_TEXT_LOAD_COUNTER}"
    try:
        spec = importlib.util.spec_from_file_location(mod_name, p)
        if spec is None or spec.loader is None:
            return None, f"spec failed: {p}"
        mod = importlib.util.module_from_spec(spec)
        sys.modules[mod_name] = mod
        try:
            spec.loader.exec_module(mod)
        except Exception as e:
            sys.modules.pop(mod_name, None)
            return None, f"exec failed: {type(e).__name__}: {e}"
    except Exception as e:
        sys.modules.pop(mod_name, None)
        return None, f"unexpected: {type(e).__name__}: {e}"
    if not hasattr(mod, "__name__") or mod.__name__ != mod_name:
        sys.modules.pop(mod_name, None)
        return None, "not a proper module"
    return mod, None


def _load_ge1_verified() -> Tuple[Optional[Any], Optional[Reason], str]:
    """F-03 v2.3: purge on ANY reject path; callable() check; register last."""
    candidates = _find_ge1_candidates_on_disk()
    if not candidates:
        return None, Reason.ENCODER_UNAVAILABLE, "no GE1 source found"
    matching_src: Optional[str] = None
    for src in candidates:
        b, err = _verify_wire_artifact(src)
        if b is not None:
            _LOADER_LOG.append(("hash", src, hashlib.md5(b).hexdigest()))
            matching_src = src; break
        try:
            with open(src, "rb") as f: raw = f.read()
            _LOADER_LOG.append(("hash", src, hashlib.md5(raw).hexdigest()))
        except OSError:
            pass
    if matching_src is None:
        return None, Reason.ENCODER_UNAVAILABLE, "no GE1 source matched pin"
    grid_mod, grid_err = _load_verified_grid_text()
    if grid_mod is None:
        return None, Reason.ENCODER_ERROR, f"grid_text binding failed: {grid_err}"
    saved_grid = sys.modules.get("grid_text")
    sys.modules["grid_text"] = grid_mod
    try:
        mod_name = f"ge1_wire_verified_{hashlib.md5(matching_src.encode()).hexdigest()[:8]}"
        spec = importlib.util.spec_from_file_location(mod_name, matching_src)
        if spec is None or spec.loader is None:
            return None, Reason.ENCODER_ERROR, f"spec failed: {matching_src}"
        mod = importlib.util.module_from_spec(spec)
        sys.modules[mod_name] = mod
        _LOADER_LOG.append(("import", matching_src))
        try:
            spec.loader.exec_module(mod)
        except Exception as e:
            sys.modules.pop(mod_name, None)
            return None, Reason.ENCODER_ERROR, f"exec: {type(e).__name__}: {e}"
    finally:
        if saved_grid is None: sys.modules.pop("grid_text", None)
        else: sys.modules["grid_text"] = saved_grid
        if grid_mod is not None:
            sys.modules.pop(getattr(grid_mod, "__name__", ""), None)
    if not (callable(getattr(mod, "encode", None)) and hasattr(mod, "ReasonCode")):
        sys.modules.pop(mod_name, None)
        return None, Reason.ENCODER_ERROR, "interface violation"
    return mod, None, ""


# ============================================================================
# TOKENIZER LOADER (F-02 v2.4: root chmod + post-load drift check)
# ============================================================================
def _resolve_tokenizer_snapshot_dir() -> Optional[str]:
    try:
        import huggingface_hub.constants as _hfc
        hub = getattr(_hfc, "HF_HUB_CACHE", None)
    except Exception:
        hub = None
    if not hub:
        hf_home = os.environ.get("HF_HOME")
        hub = os.path.join(hf_home, "hub") if hf_home else os.path.expanduser("~/.cache/huggingface/hub")
    cands = [os.path.join(hub, "models--Qwen--Qwen3.8-27B", "snapshots", QWEN_REVISION)]
    try:
        for d in glob.glob(os.path.join(hub, "models--Qwen--Qwen3.8-27B", "snapshots", "*")):
            if os.path.isdir(d) and os.path.basename(d) == QWEN_REVISION:
                cands.append(d)
    except Exception:
        pass
    for c in cands:
        if c and os.path.isdir(c) and os.path.exists(os.path.join(c, "tokenizer.json")):
            return c
    return None


def _fingerprint_snapshot(dir_path: str) -> Tuple[Tuple[str, str], ...]:
    out: List[Tuple[str, str]] = []
    def _raise_onerror(e): raise e
    for root, _, files in os.walk(dir_path, onerror=_raise_onerror):
        for fn in sorted(files):
            p = os.path.join(root, fn)
            with open(p, "rb") as f:
                out.append((os.path.relpath(p, dir_path),
                            hashlib.sha256(f.read()).hexdigest()))
    return tuple(sorted(out))


def _stage_snapshot(snapshot_dir, expected_fingerprint):
    """F-02 v2.4: chmod root too (topdown=False)."""
    try:
        staging_dir = tempfile.mkdtemp(prefix="ge3_tokenizer_stage_")
    except OSError as e:
        return None, None, f"mkdtemp failed: {e}"
    try:
        for relpath, expected_hash in expected_fingerprint:
            src = os.path.join(snapshot_dir, relpath)
            dst = os.path.join(staging_dir, relpath)
            parent = os.path.dirname(dst)
            if parent and not os.path.isdir(parent):
                os.makedirs(parent, exist_ok=True)
            with open(src, "rb") as f: data = f.read()
            with open(dst, "wb") as f: f.write(data)
        # F-02: chmod with topdown=False so children before parents, then root
        for root, dirs, files in os.walk(staging_dir, topdown=False):
            for f in files:
                try: os.chmod(os.path.join(root, f), 0o444)
                except OSError: pass
            for d in dirs:
                try: os.chmod(os.path.join(root, d), 0o555)
                except OSError: pass
        # F-02: chmod the root itself
        try: os.chmod(staging_dir, 0o555)
        except OSError: pass
        try:
            stage_fp = _fingerprint_snapshot(staging_dir)
        except (OSError, RuntimeError) as e:
            return staging_dir, None, f"staging fingerprint failed: {e}"
        if stage_fp != expected_fingerprint:
            return staging_dir, None, "staging fingerprint drift (bytes differ)"
        return staging_dir, stage_fp, None
    except Exception as e:
        return staging_dir, None, f"staging failed: {type(e).__name__}: {e}"


def _cleanup_staging(staging_dir):
    if staging_dir is None: return
    try:
        for root, dirs, files in os.walk(staging_dir):
            for d in dirs:
                try: os.chmod(os.path.join(root, d), 0o755)
                except OSError: pass
            for f in files:
                try: os.chmod(os.path.join(root, f), 0o644)
                except OSError: pass
        try: os.chmod(staging_dir, 0o755)
        except OSError: pass
        shutil.rmtree(staging_dir, ignore_errors=True)
    except Exception:
        pass


def _load_pinned_tokenizer():
    """F-01 v2.3: fingerprint self-consistency.
    F-02 v2.4: post-load drift detection."""
    snapshot_dir = _resolve_tokenizer_snapshot_dir()
    if snapshot_dir is None:
        return None, None, None, Reason.TOKENIZER_UNAVAILABLE, "snapshot not locatable"
    tj = os.path.join(snapshot_dir, "tokenizer.json")
    try:
        with open(tj, "rb") as f: tj_bytes = f.read()
    except OSError as e:
        return None, None, None, Reason.TOKENIZER_UNAVAILABLE, f"tokenizer.json unreadable: {e}"
    if hashlib.sha256(tj_bytes).hexdigest() != TOKENIZER_JSON_SHA256:
        return None, None, None, Reason.TOKENIZER_UNPINNED, "top-level tokenizer.json != pin"
    try:
        fingerprint = _fingerprint_snapshot(snapshot_dir)
    except (OSError, RuntimeError) as e:
        return None, None, None, Reason.TOKENIZER_UNAVAILABLE, f"fingerprint failed: {e}"
    tj_entries = [h for (p, h) in fingerprint if p == "tokenizer.json"]
    if len(tj_entries) != 1:
        return None, None, None, Reason.TOKENIZER_UNPINNED, "fingerprint tokenizer.json count != 1"
    if tj_entries[0] != TOKENIZER_JSON_SHA256:
        return None, None, None, Reason.TOKENIZER_UNPINNED, "fingerprint tokenizer.json != pin"
    staging_dir, stage_fp, stage_err = _stage_snapshot(snapshot_dir, fingerprint)
    if stage_err is not None:
        _cleanup_staging(staging_dir)
        if "drift" in stage_err:
            return None, None, None, Reason.TOKENIZER_MISMATCH, stage_err
        return None, None, None, Reason.TOKENIZER_UNAVAILABLE, stage_err
    try:
        try:
            from transformers import AutoTokenizer
        except ImportError:
            return None, None, None, Reason.TOKENIZER_UNAVAILABLE, "transformers not installed"
        try:
            tok = AutoTokenizer.from_pretrained(staging_dir,
                                                trust_remote_code=False,
                                                local_files_only=True)
        except Exception as e:
            return None, None, None, Reason.TOKENIZER_UNAVAILABLE, f"load failed: {e}"
        # F-02 v2.4: post-load drift detection
        try:
            fp_after = _fingerprint_snapshot(staging_dir)
        except (OSError, RuntimeError) as e:
            return None, None, None, Reason.TOKENIZER_MISMATCH, f"post-load fingerprint failed: {e}"
        if fp_after != fingerprint:
            return None, None, None, Reason.TOKENIZER_MISMATCH, "staged snapshot drifted during load"
        stage_ref_bytes = json.dumps(stage_fp, sort_keys=True).encode()
        stage_ref = hashlib.sha256(stage_ref_bytes).hexdigest()
        try:
            if hasattr(tok, "backend_tokenizer") and hasattr(tok.backend_tokenizer, "no_truncation"):
                tok.backend_tokenizer.no_truncation()
        except Exception:
            pass
        return tok, fingerprint, stage_ref, None, ""
    finally:
        _cleanup_staging(staging_dir)


# ============================================================================
# MEASURE_COSTS
# ============================================================================
def measure_costs(grid, *, header: Optional[Mapping[str, Any]] = None) -> CostBundle:
    try:
        ge1_mod, ge1_reason, ge1_msg = _load_ge1_verified()
        if ge1_mod is None:
            return CostBundle(costs=(), measurement_error=f"GE1: {ge1_msg}")
        tok, fingerprint, stage_ref, tok_reason, tok_msg = _load_pinned_tokenizer()
        if tok is None:
            return CostBundle(costs=(), measurement_error=f"{tok_reason.value}: {tok_msg}")
        if header is None:
            hdr = {"levels_completed":0,"state":"NOT_FINISHED","actions":(1,2,3,4,5,6)}
        elif isinstance(header, str):
            # v2.6 FIX (Indus): accept the GE1/GE2 wire-header STRING
            # ("levels_completed=..\nstate=..\navailable_actions=[..]") so the
            # GE2 adapt() -> GE3 measure_costs() handoff works directly.
            _h = {}
            for _ln in header.splitlines():
                if "=" in _ln:
                    _k, _v = _ln.split("=", 1); _h[_k.strip()] = _v.strip()
            _av = _h.get("available_actions", _h.get("actions", "[]"))
            _acts = [int(x) for x in _av.strip("[]").replace(" ", "").split(",") if x.strip()]
            hdr = {"levels_completed": int(_h.get("levels_completed", 0) or 0),
                   "state": _h.get("state", "NOT_FINISHED"), "actions": _acts}
        else:
            hdr = dict(header)
        costs: List[Tuple[str, Optional[int]]] = []
        reasons: List[Tuple[str, str]] = []
        for key in _get_candidates():
            tier, k = _parse_key(key)
            try:
                text = ge1_mod.encode(grid, tier,
                                      levels_completed=hdr["levels_completed"],
                                      state=hdr["state"],
                                      actions=list(hdr["actions"]), k=k)
            except Exception as e:
                code = getattr(e, "code", None)
                if code == "CANDIDATE_INFEASIBLE":
                    costs.append((key, None))
                    reasons.append((key, "PAIR_BUDGET_EXCEEDED (Report F-08)"))
                    continue
                return CostBundle(costs=tuple(costs+[(key,None)]), reasons=tuple(reasons),
                                  measurement_error=f"{key}: ENCODE_FAILED:{code or type(e).__name__}")
            try:
                n = len(tok.encode(text, add_special_tokens=False, truncation=False))
            except Exception as e:
                return CostBundle(costs=tuple(costs+[(key,None)]), reasons=tuple(reasons),
                                  measurement_error=f"{key}: TOKENIZE_FAILED:{type(e).__name__}")
            costs.append((key, n))
        return CostBundle(costs=tuple(costs), reasons=tuple(reasons),
                          tokenizer_id=QWEN_REPO, tokenizer_revision=QWEN_REVISION,
                          tokenizer_sha256=TOKENIZER_JSON_SHA256,
                          tokenizer_fingerprint=fingerprint,
                          tokenizer_stage_ref=stage_ref)
    except Exception as e:
        return CostBundle(costs=(), measurement_error=f"{type(e).__name__}: {e}")


def _select_from_costs_dict(grid, cap, costs_dict, reasons_dict=None,
                             *, policy: Policy, dqc_table=None) -> Any:
    if not isinstance(costs_dict, Mapping):
        raise TypeError("costs_dict must be Mapping")
    extra = set(costs_dict.keys()) - set(policy.candidate_keys)
    missing = set(policy.candidate_keys) - set(costs_dict.keys())
    if extra or missing:
        raise ValueError(f"keys: extra={sorted(extra)} missing={sorted(missing)}")
    bundle = CostBundle(
        costs=tuple((k, costs_dict[k]) for k in policy.candidate_keys),
        reasons=tuple((k, (reasons_dict or {}).get(k, ""))
                      for k in policy.candidate_keys if costs_dict[k] is None),
        tokenizer_sha256=policy.tokenizer_sha256)
    return select(grid, cap, costs=bundle, policy=policy, dqc_table=dqc_table)


# ============================================================================
# MANIFEST
# ============================================================================
def _file_entry(name, path, role):
    try:
        with open(path, "rb") as f: b = f.read()
    except OSError as e:
        return {"file": name, "role": role, "status": "MISSING", "error": str(e)}
    return {"file": name, "role": role, "bytes": len(b),
            "md5": hashlib.md5(b).hexdigest(),
            "sha256": hashlib.sha256(b).hexdigest()}


def _json_safe_default(o):
    if isinstance(o, (frozenset, set)): return sorted(o)
    if isinstance(o, tuple): return list(o)
    if isinstance(o, MappingProxyType): return dict(o)
    raise TypeError(f"manifest: unexpected type {type(o).__name__}")


def _check_pin_format(pins, mandatory) -> Optional[str]:
    """F-03 v2.4: also validate OPTIONAL wire fields when populated."""
    for name, fields in mandatory.items():
        if name not in pins:
            return f"missing artifact {name!r}"
        pin = pins[name]
        for field in fields:
            if field not in pin:
                return f"{name}: missing mandatory field {field!r}"
            v = pin[field]
            if field == "md5" and (not isinstance(v, str) or len(v) != 32 or
                                    not re.fullmatch(r"[0-9a-f]+", v)):
                return f"{name}.{field}: invalid md5"
            if field == "sha256" and (not isinstance(v, str) or len(v) != 64 or
                                       not re.fullmatch(r"[0-9a-f]+", v)):
                return f"{name}.{field}: invalid sha256"
            if field == "bytes" and (type(v) is not int or type(v) is bool or v <= 0):
                return f"{name}.{field}: invalid bytes"
        # F-03 v2.4: validate optional wire fields when populated
        for opt_field in ("sha256", "bytes"):
            if opt_field in pin and pin[opt_field] is not None:
                v = pin[opt_field]
                if opt_field == "sha256":
                    if not isinstance(v, str) or len(v) != 64 or not re.fullmatch(r"[0-9a-f]+", v):
                        return f"{name}.{opt_field}: invalid populated sha256"
                elif opt_field == "bytes":
                    if type(v) is not int or type(v) is bool or v <= 0:
                        return f"{name}.{opt_field}: invalid populated bytes"
    return None


def _compute_wire_pin_strength(wire_pin) -> str:
    """F-03 v2.4: STRONG only when sha256 AND bytes are VALID (not just non-None)."""
    sha = wire_pin.get("sha256") if isinstance(wire_pin, Mapping) else None
    bytes_val = wire_pin.get("bytes") if isinstance(wire_pin, Mapping) else None
    sha_ok = isinstance(sha, str) and re.fullmatch(r"[0-9a-f]{64}", sha) is not None
    bytes_ok = (isinstance(bytes_val, int) and not isinstance(bytes_val, bool)
                and bytes_val > 0)
    if sha_ok and bytes_ok:
        return "STRONG"
    return "WEAK_MD5_ONLY — audit F-08: add sha256 + bytes from signed release metadata"


def _resolve_verified_ge1_path() -> Optional[str]:
    for src in _find_ge1_candidates_on_disk():
        b, err = _verify_wire_artifact(src)
        if b is not None:
            return src
    return None


def _resolve_verified_grid_text_path() -> Optional[str]:
    for src in _find_grid_text_candidates():
        b, err = _verify_grid_text_bytes(src)
        if b is not None:
            return src
    return None


def bundle_manifest(*, ge1_path=None, ge2_path=None, grid_text_path=None) -> Dict[str, Any]:
    if ge1_path is None:
        ge1_path = _resolve_verified_ge1_path()
    if grid_text_path is None:
        grid_text_path = _resolve_verified_grid_text_path()
    pin_err = _check_pin_format(_KNOWN_UPSTREAM, _MANDATORY_PIN_FIELDS)

    sel_entry = _file_entry("selector.py", os.path.abspath(__file__), "primary")
    primary_ok = ("md5" in sel_entry and "sha256" in sel_entry and "bytes" in sel_entry)

    upstream_entries = []
    for name, path in (("grid_text.py", grid_text_path),
                       ("wire.py", ge1_path),
                       ("adapters.py", ge2_path)):
        expected = _known_upstream_get(name)
        if not path or not os.path.exists(path):
            upstream_entries.append({"file": name, "status": "MISSING",
                                     "expected": dict(expected) if expected else None})
            continue
        e = _file_entry(name, path, "upstream")
        if expected:
            e["expected"] = dict(expected)
            e["matches_expected"] = all(
                e.get(k) == v for k, v in expected.items()
                if k in ("md5","sha256","bytes") and v is not None)
        upstream_entries.append(e)

    missing_mand = [e for e in upstream_entries
                    if e.get("status") == "MISSING" and e["file"] in _MANDATORY_UPSTREAM]
    mismatch_mand = [e for e in upstream_entries
                     if e.get("status") != "MISSING" and e["file"] in _MANDATORY_UPSTREAM
                     and e.get("expected") and not e.get("matches_expected")]

    if not primary_ok or pin_err or missing_mand:
        attest = "INCOMPLETE_MANDATORY_INPUTS"
    elif mismatch_mand:
        attest = "HASH_MISMATCH"
    else:
        attest = "COMPLETE_PENDING_SIGNATURE"

    wire_pin = _known_upstream_get("wire.py", {})
    wire_strength = _compute_wire_pin_strength(wire_pin)

    m: Dict[str, Any] = {
        "ge3_version": GE3_VERSION, "ge3_status": GE3_STATUS,
        "spec": SPEC_REV, "arch": ARCH_REF,
        "order": ORDER_REF, "audit": AUDIT_REF, "ruling": RULING_REF,
        "res": RES_REF, "aga": AGA_REF,
        "dev_pass_ceiling": "ARCH/2026/103 §7",
        "pin_format_status": pin_err if pin_err else "OK",
        "wire_pin_strength": wire_strength,
        "release_integrity_ready": (wire_strength == "STRONG"),
        "reference_policy_fingerprint_enforced": bool(REFERENCE_POLICY.require_fingerprint),
        "attestation_status": attest,
        "signature_status": "PENDING_EXTERNAL_SIGNATURE",
        "out_of_scope_gates": {k: dict(v) for k, v in OUT_OF_SCOPE_GATES.items()},
        "delivered": [sel_entry],
        "upstream": upstream_entries,
    }
    for name, obj in (("DQC_QUERY_BANK.json", _DQC_QUERY_BANK),
                      ("DQC_COVERAGE.json", {k: sorted(v) for k,v in _DQC_COVERAGE.items()}),
                      ("RECOVERY_TABLE.json", {k.value: v for k,v in _RECOVERY_TABLE.items()})):
        b = json.dumps(obj, sort_keys=True, indent=2, default=_json_safe_default).encode()
        m["delivered"].append({"file": name, "role": "artifact", "kind": "inline",
                               "bytes": len(b),
                               "md5": hashlib.md5(b).hexdigest(),
                               "sha256": hashlib.sha256(b).hexdigest()})
    return m


# ============================================================================
# RELATION PARSER (F-02 v2.3: per-relation canonical)
# ============================================================================
_RELATION_PRIORITY = ("adjacent","aligned_cols","aligned_rows","above","left","bbox_contains")
_SYMMETRIC_RELATIONS = frozenset({"adjacent","aligned_cols","aligned_rows"})
_DIRECTED_RELATIONS  = frozenset({"above","left","bbox_contains"})
_REL_PAIR_RE = re.compile(r"^[1-9][0-9]*-[1-9][0-9]*$")
_REL_LINE_RE = re.compile(r"^  (\w+)=(.+)$")


def _parse_relations_strict(text, expected_k, max_endpoint=None):
    if not (1 <= expected_k <= 6):
        raise ValueError(f"expected_k must be 1..6, got {expected_k}")
    expected = _RELATION_PRIORITY[:expected_k]
    raw = {}
    seen_marker = False
    for line_no, line in enumerate(text.split("\n"), 1):
        if line == "RELATIONS:":
            if seen_marker:
                raise ValueError(f"F-02: second RELATIONS: marker at ln {line_no}")
            seen_marker = True; continue
        if not seen_marker: continue
        if line == "": continue
        m = _REL_LINE_RE.fullmatch(line)
        if not m:
            raise ValueError(f"F-05: malformed line at ln {line_no}: {line!r}")
        name, val = m.group(1), m.group(2)
        if name in raw:
            raise ValueError(f"F-05: duplicate relation name {name!r}")
        raw[name] = val
    extra = [r for r in raw if r not in expected]
    if extra:
        raise ValueError(f"F-05: unexpected fields {extra}")
    missing = [r for r in expected if r not in raw]
    if missing:
        raise ValueError(f"F-05: missing required fields {missing}")
    out = {}
    for r in expected:
        val = raw[r]
        if val == "-":
            out[r] = []; continue
        pairs = []
        seen_pairs = set()
        for token in val.split(","):
            if not _REL_PAIR_RE.fullmatch(token):
                raise ValueError(f"F-05: {r} malformed pair {token!r}")
            a_str, b_str = token.split("-")
            a, b = int(a_str), int(b_str)
            if a == b:
                raise ValueError(f"F-02: {r} self-pair {token!r}")
            if max_endpoint is not None:
                if not (1 <= a <= max_endpoint):
                    raise ValueError(f"F-02: {r} id {a} out of 1..{max_endpoint}")
                if not (1 <= b <= max_endpoint):
                    raise ValueError(f"F-02: {r} id {b} out of 1..{max_endpoint}")
            if r in _SYMMETRIC_RELATIONS:
                if a > b:
                    raise ValueError(f"F-02: {r} non-canonical order {token!r}")
                if (b, a) in seen_pairs:
                    raise ValueError(f"F-02: {r} reversed dup of {(b,a)}")
            if (a, b) in seen_pairs:
                raise ValueError(f"F-05: {r} duplicate pair {(a,b)}")
            seen_pairs.add((a, b))
            pairs.append((a, b))
        out[r] = pairs
    return out


# ============================================================================
# TESTS
# ============================================================================
_TESTS: List[Tuple[str, Status, str]] = []

def _ok(n, d=""):
    t = (n, Status.PASS, d); _TESTS.append(t); return t

def _fail(n, d=""):
    t = (n, Status.FAIL, d); _TESTS.append(t); return t

def _blk(n, d=""):
    t = (n, Status.BLOCKED, d); _TESTS.append(t); return t


def _bundle(costs_dict, reasons_dict=None, tok_sha=TOKENIZER_JSON_SHA256):
    return CostBundle(
        costs=tuple((k, costs_dict[k]) for k in _get_candidates()),
        reasons=tuple((k, (reasons_dict or {}).get(k, ""))
                      for k in _get_candidates() if costs_dict[k] is None),
        tokenizer_sha256=tok_sha)


def _sel(costs_dict, cap, reasons=None, policy=REFERENCE_POLICY):
    return select(None, cap, costs=_bundle(costs_dict, reasons, policy.tokenizer_sha256),
                  policy=policy)


# --- F-01 v2.4: WEAK pin → BLOCKED, STRONG → PASS ---
def unit_f01_weak_pin_blocks_pass():
    """F-01 v2.5: the classifier DERIVES strength from evidence; a bare label
    flip must NOT pass."""
    base = {
        "delivered": [{"file": "selector.py", "role": "primary",
                       "md5": "a"*32, "sha256": "a"*64, "bytes": 100}],
        "upstream": [
            {"file": "grid_text.py", "status": "present", "matches_expected": True,
             "expected": {"md5": "b"*32, "sha256": "b"*64, "bytes": 100}},
            {"file": "wire.py", "status": "present", "matches_expected": True,
             "expected": {"md5": "d"*32}},
        ],
        "attestation_status": "COMPLETE_PENDING_SIGNATURE",
        "ge3_status": "DEV_PASS", "wire_pin_strength": "WEAK_MD5_ONLY",
        "out_of_scope_gates": {},
    }
    s1, _ = _classify_manifest_status(base)
    if s1 is not Status.BLOCKED:
        return _fail("unit_f01_weak_pin_blocks_pass", f"weak -> {s1.value} (want BLOCKED)")
    lab = dict(base); lab["wire_pin_strength"] = "STRONG"
    s2, _ = _classify_manifest_status(lab)
    if s2 is not Status.BLOCKED:
        return _fail("unit_f01_weak_pin_blocks_pass", f"label-only STRONG -> {s2.value} (want BLOCKED)")
    strong = dict(base)
    strong["upstream"] = [base["upstream"][0],
                          {"file": "wire.py", "status": "present", "matches_expected": True,
                           "expected": {"md5": "d"*32, "sha256": "e"*64, "bytes": 12345}}]
    strong["wire_pin_strength"] = "STRONG"
    s3, _ = _classify_manifest_status(strong)
    if s3 is not Status.PASS:
        return _fail("unit_f01_weak_pin_blocks_pass", f"evidence STRONG -> {s3.value} (want PASS)")
    return _ok("unit_f01_weak_pin_blocks_pass",
               "weak->BLOCKED; label-only STRONG->BLOCKED; evidence STRONG->PASS")


def unit_f02_staging_root_readonly():
    import tempfile, shutil, stat
    tmpdir = tempfile.mkdtemp(prefix="ge3_f02src_")
    try:
        with open(os.path.join(tmpdir, "a.txt"), "w") as f: f.write("hello")
        with open(os.path.join(tmpdir, "b.txt"), "w") as f: f.write("world")
        fp = _fingerprint_snapshot(tmpdir)
        staging_dir, stage_fp, err = _stage_snapshot(tmpdir, fp)
        try:
            if err is not None:
                return _fail("unit_f02_staging_root_readonly", f"stage failed: {err}")
            root_mode = stat.S_IMODE(os.stat(staging_dir).st_mode)
            if root_mode != 0o555:
                return _fail("unit_f02_staging_root_readonly",
                             f"root mode {oct(root_mode)} != 0o555")
            return _ok("unit_f02_staging_root_readonly", "staging root is 0o555")
        finally:
            _cleanup_staging(staging_dir)
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def unit_f02_post_load_drift_detected():
    """F-02 v2.5: a mutation to the STAGING dir DURING the tokenizer load is
    caught by the post-load drift check -> TOKENIZER_MISMATCH (+ cleanup)."""
    import tempfile, shutil
    M = sys.modules[__name__]
    try:
        from transformers import AutoTokenizer
    except Exception:
        return _blk("unit_f02_post_load_drift_detected", "transformers not available")
    snap = _resolve_tokenizer_snapshot_dir()
    if snap is None:
        return _blk("unit_f02_post_load_drift_detected", "no pinned snapshot in env")
    orig = AutoTokenizer.from_pretrained
    def evil(path, *a, **k):
        try:
            for root, dirs, files in os.walk(path):
                os.chmod(root, 0o755)
                for d in dirs: os.chmod(os.path.join(root, d), 0o755)
                for f in files: os.chmod(os.path.join(root, f), 0o644)
            with open(os.path.join(path, "tokenizer.json"), "w") as f:
                f.write('{"drift": true}')
        except Exception:
            pass
        return "MOCKED"
    AutoTokenizer.from_pretrained = evil
    try:
        tok, fp, sref, reason, msg = _load_pinned_tokenizer()
    finally:
        AutoTokenizer.from_pretrained = orig
    if reason is not Reason.TOKENIZER_MISMATCH:
        return _fail("unit_f02_post_load_drift_detected",
                     f"want TOKENIZER_MISMATCH, got {reason}: {msg}")
    return _ok("unit_f02_post_load_drift_detected",
               "in-load staging mutation -> TOKENIZER_MISMATCH")


def unit_f03_strong_pin_validation():
    cases = [
        # (sha256, bytes, expected_strength_label)
        (None, None, "WEAK"),
        ("a"*64, 100, "STRONG"),
        ("a"*63, 100, "WEAK"),           # bad sha length
        ("A"*64, 100, "WEAK"),           # uppercase
        ("g"*64, 100, "WEAK"),           # non-hex
        ("a"*64, None, "WEAK"),          # bytes missing
        ("a"*64, 0, "WEAK"),             # bytes zero
        ("a"*64, -5, "WEAK"),            # bytes negative
        ("a"*64, True, "WEAK"),          # bytes bool
        ("a"*64, "100", "WEAK"),         # bytes string
        (None, 100, "WEAK"),             # sha missing
        ("0"*64, 1, "STRONG"),           # valid edge case
    ]
    for sha, b, expected in cases:
        wire_pin = {"sha256": sha, "bytes": b}
        got = _compute_wire_pin_strength(wire_pin)
        got_label = "STRONG" if got == "STRONG" else "WEAK"
        if got_label != expected:
            return _fail("unit_f03_strong_pin_validation",
                         f"sha={sha!r:.20} bytes={b!r:.6} -> {got_label}, expected {expected}")
    return _ok("unit_f03_strong_pin_validation",
               f"{len(cases)} pin cases validated")


# --- F-04 v2.4: test inventory is independent ---
def unit_f04_test_inventory_independent():
    M = sys.modules[__name__]
    if not hasattr(M, "_EXPECTED_TEST_NAMES"):
        return _fail("unit_f04_test_inventory_independent", "_EXPECTED_TEST_NAMES missing")
    inventory = M._EXPECTED_TEST_NAMES
    if not isinstance(inventory, tuple):
        return _fail("unit_f04_test_inventory_independent", "inventory not tuple")
    if len(set(inventory)) != len(inventory):
        return _fail("unit_f04_test_inventory_independent", "duplicates in inventory")
    actual = tuple(fn.__name__ for fn in _all_tests())
    if inventory != actual:
        if len(inventory) != len(actual):
            return _fail("unit_f04_test_inventory_independent",
                         f"length: inventory={len(inventory)} actual={len(actual)}")
        for i, (inv, act) in enumerate(zip(inventory, actual)):
            if inv != act:
                return _fail("unit_f04_test_inventory_independent",
                             f"idx {i}: inventory={inv!r} != actual={act!r}")
    return _ok("unit_f04_test_inventory_independent",
               f"inventory matches _all_tests() ({len(inventory)} names)")


# --- v2.3/v2.1/v1.10 tests retained ---
def unit_f01_fingerprint_pin_consistency():
    import tempfile, shutil
    tmpdir = tempfile.mkdtemp(prefix="ge3_f01pin_")
    try:
        tj = os.path.join(tmpdir, "tokenizer.json")
        with open(tj, "w") as f: f.write("fake")
        M = sys.modules[__name__]
        orig = M._resolve_tokenizer_snapshot_dir
        M._resolve_tokenizer_snapshot_dir = lambda: tmpdir
        try:
            result = M._load_pinned_tokenizer()
        finally:
            M._resolve_tokenizer_snapshot_dir = orig
        if result[0] is not None:
            return _fail("unit_f01_fingerprint_pin_consistency", "wrong pin accepted")
        if result[3] != Reason.TOKENIZER_UNPINNED:
            return _fail("unit_f01_fingerprint_pin_consistency", f"reason={result[3]}")
        return _ok("unit_f01_fingerprint_pin_consistency", "wrong pin -> UNPINNED")
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def unit_f01_no_cache_poisoning():
    import tempfile, shutil
    tmpdir = tempfile.mkdtemp(prefix="ge3_f01_")
    try:
        gt_path = os.path.join(tmpdir, "grid_text.py")
        with open(gt_path, "w") as f: f.write("__verified__ = True\n")
        with open(gt_path, "rb") as f: gt_bytes = f.read()
        gt_md5 = hashlib.md5(gt_bytes).hexdigest()
        gt_sha = hashlib.sha256(gt_bytes).hexdigest()
        old_cache_name = f"_ge3_verified_gridtext_{gt_md5[:8]}"
        poisoned = type(sys)("poison"); poisoned.__verified__ = "POISONED"
        saved = sys.modules.get(old_cache_name)
        sys.modules[old_cache_name] = poisoned
        M = sys.modules[__name__]
        orig_find = M._find_grid_text_candidates
        orig_get = M._known_upstream_get
        M._find_grid_text_candidates = lambda: [gt_path]
        def patched(name, default=None):
            if name == "grid_text.py":
                return {"md5": gt_md5, "sha256": gt_sha, "bytes": len(gt_bytes)}
            return orig_get(name, default)
        M._known_upstream_get = patched
        try:
            mod, err = M._load_verified_grid_text()
        finally:
            M._find_grid_text_candidates = orig_find
            M._known_upstream_get = orig_get
            if saved is None: sys.modules.pop(old_cache_name, None)
            else: sys.modules[old_cache_name] = saved
        if mod is None:
            return _fail("unit_f01_no_cache_poisoning", f"load: {err}")
        try:
            if getattr(mod, "__verified__", None) is not True:
                return _fail("unit_f01_no_cache_poisoning", "poisoned cache leaked")
        finally:
            sys.modules.pop(getattr(mod, "__name__", ""), None)
        return _ok("unit_f01_no_cache_poisoning", "poisoned old cache ignored")
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def unit_f02_self_pair_all_relations():
    for name in _RELATION_PRIORITY:
        text = "RELATIONS:\n"
        for rel in _RELATION_PRIORITY:
            text += f"  {rel}={'1-1' if rel == name else '-'}\n"
        try:
            _parse_relations_strict(text, expected_k=6)
            return _fail("unit_f02_self_pair_all_relations", f"{name} self-pair accepted")
        except ValueError: pass
    return _ok("unit_f02_self_pair_all_relations", "self-pairs rejected all relations")


def unit_f02_symmetric_noncanonical_rejected():
    text = "RELATIONS:\n  adjacent=2-1\n  aligned_cols=-\n  aligned_rows=-\n  above=-\n  left=-\n  bbox_contains=-\n"
    try:
        _parse_relations_strict(text, expected_k=6)
        return _fail("unit_f02_symmetric_noncanonical_rejected", "2-1 accepted")
    except ValueError: pass
    text = "RELATIONS:\n  adjacent=1-2\n  aligned_cols=-\n  aligned_rows=-\n  above=-\n  left=-\n  bbox_contains=-\n"
    try:
        out = _parse_relations_strict(text, expected_k=6)
        if out["adjacent"] != [(1,2)]:
            return _fail("unit_f02_symmetric_noncanonical_rejected", f"1-2 -> {out['adjacent']}")
    except ValueError as e:
        return _fail("unit_f02_symmetric_noncanonical_rejected", f"1-2 rejected: {e}")
    return _ok("unit_f02_symmetric_noncanonical_rejected", "2-1 rejected; 1-2 accepted")


def unit_f02_directed_either_order_accepted():
    for val in ("1-2", "2-1"):
        text = f"RELATIONS:\n  adjacent=-\n  aligned_cols=-\n  aligned_rows=-\n  above={val}\n  left=-\n  bbox_contains=-\n"
        try:
            out = _parse_relations_strict(text, expected_k=6)
            a, b = val.split("-")
            if out["above"] != [(int(a), int(b))]:
                return _fail("unit_f02_directed_either_order_accepted", f"above={val} -> {out['above']}")
        except ValueError as e:
            return _fail("unit_f02_directed_either_order_accepted", f"above={val} rejected: {e}")
    return _ok("unit_f02_directed_either_order_accepted", "above accepts 1-2 and 2-1")


def unit_f02_reversed_dup_symmetric_rejected():
    text = "RELATIONS:\n  adjacent=1-2,2-1\n  aligned_cols=-\n  aligned_rows=-\n  above=-\n  left=-\n  bbox_contains=-\n"
    try:
        _parse_relations_strict(text, expected_k=6)
        return _fail("unit_f02_reversed_dup_symmetric_rejected", "1-2,2-1 accepted")
    except ValueError: pass
    return _ok("unit_f02_reversed_dup_symmetric_rejected", "1-2,2-1 rejected")


def unit_f02_second_relations_marker_rejected():
    text = "RELATIONS:\n  adjacent=-\n  aligned_cols=-\n  aligned_rows=-\nRELATIONS:\n  above=-\n  left=-\n  bbox_contains=-\n"
    try:
        _parse_relations_strict(text, expected_k=6)
        return _fail("unit_f02_second_relations_marker_rejected", "second marker accepted")
    except ValueError: pass
    return _ok("unit_f02_second_relations_marker_rejected", "second marker rejected")


def unit_f02_out_of_range_id_rejected():
    text = "RELATIONS:\n  adjacent=1-5\n  aligned_cols=-\n  aligned_rows=-\n  above=-\n  left=-\n  bbox_contains=-\n"
    try:
        _parse_relations_strict(text, expected_k=6, max_endpoint=3)
        return _fail("unit_f02_out_of_range_id_rejected", "5 accepted max=3")
    except ValueError: pass
    try:
        _parse_relations_strict(text, expected_k=6, max_endpoint=5)
    except ValueError as e:
        return _fail("unit_f02_out_of_range_id_rejected", f"5 rejected max=5: {e}")
    return _ok("unit_f02_out_of_range_id_rejected", "5-1 rejected max=3, accepted max=5")


def unit_f03_purge_on_interface_failure():
    import tempfile, shutil
    tmpdir = tempfile.mkdtemp(prefix="ge3_f03_")
    try:
        wire_path = os.path.join(tmpdir, "wire.py")
        with open(wire_path, "w") as f: f.write("pass  # no interface\n")
        grid_path = os.path.join(tmpdir, "grid_text.py")
        with open(grid_path, "w") as f: f.write("__ok__ = True\n")
        with open(grid_path, "rb") as f: gb = f.read()
        with open(wire_path, "rb") as f: wb = f.read()
        M = sys.modules[__name__]
        orig_fw = M._find_ge1_candidates_on_disk
        orig_fg = M._find_grid_text_candidates
        orig_get = M._known_upstream_get
        M._find_ge1_candidates_on_disk = lambda: [wire_path]
        M._find_grid_text_candidates = lambda: [grid_path]
        def patched(name, default=None):
            if name == "wire.py":
                return {"md5": hashlib.md5(wb).hexdigest(),
                        "sha256": hashlib.sha256(wb).hexdigest(), "bytes": len(wb)}
            if name == "grid_text.py":
                return {"md5": hashlib.md5(gb).hexdigest(),
                        "sha256": hashlib.sha256(gb).hexdigest(), "bytes": len(gb)}
            return orig_get(name, default)
        M._known_upstream_get = patched
        saved = {n for n in sys.modules if n.startswith("ge1_wire_verified_")}
        try:
            mod, reason, msg = M._load_ge1_verified()
        finally:
            M._find_ge1_candidates_on_disk = orig_fw
            M._find_grid_text_candidates = orig_fg
            M._known_upstream_get = orig_get
        if mod is not None:
            return _fail("unit_f03_purge_on_interface_failure", "bad interface accepted")
        if reason != Reason.ENCODER_ERROR:
            return _fail("unit_f03_purge_on_interface_failure", f"reason={reason}")
        lingerers = {n for n in sys.modules if n.startswith("ge1_wire_verified_") and n not in saved}
        if lingerers:
            return _fail("unit_f03_purge_on_interface_failure", f"leaked: {sorted(lingerers)}")
        return _ok("unit_f03_purge_on_interface_failure", "interface fail purged")
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def unit_f04_shared_verifier():
    import inspect
    src1 = inspect.getsource(_load_ge1_verified)
    src2 = inspect.getsource(_resolve_verified_ge1_path)
    if "_verify_wire_artifact" not in src1:
        return _fail("unit_f04_shared_verifier", "loader missing")
    if "_verify_wire_artifact" not in src2:
        return _fail("unit_f04_shared_verifier", "resolver missing")
    return _ok("unit_f04_shared_verifier", "both call _verify_wire_artifact")


def unit_kimi_f08_suite_integrity_by_names():
    M = sys.modules[__name__]
    if not hasattr(M, "_EXPECTED_TEST_NAMES"):
        return _fail("unit_kimi_f08_suite_integrity_by_names", "_EXPECTED_TEST_NAMES missing")
    return _ok("unit_kimi_f08_suite_integrity_by_names", "_EXPECTED_TEST_NAMES present")


def unit_kimi_f09_cost_bundle_evidence():
    import inspect
    sig = inspect.signature(CostBundle)
    if "tokenizer_stage_ref" not in sig.parameters:
        return _fail("unit_kimi_f09_cost_bundle_evidence", "stage_ref missing")
    return _ok("unit_kimi_f09_cost_bundle_evidence", "stage_ref present")


def unit_f02_fingerprint_strictness():
    good = TOKENIZER_JSON_SHA256; cov = _DQC_COVERAGE
    dup = (("tokenizer.json", good), ("tokenizer.json", "a"*64))
    try:
        Policy(tokenizer_sha256=good, dqc_coverage=cov, rank=dict(_RANK),
               candidate_keys=_CANDIDATE_KEYS, tokenizer_fingerprint=dup)
        return _fail("unit_f02_fingerprint_strictness", "dup path accepted")
    except ValueError: pass
    try:
        Policy(tokenizer_sha256=good, dqc_coverage=cov, rank=dict(_RANK),
               candidate_keys=_CANDIDATE_KEYS,
               tokenizer_fingerprint=(("config.json", "a"*64),),
               require_fingerprint=True)
        return _fail("unit_f02_fingerprint_strictness", "no tj accepted with require")
    except ValueError: pass
    return _ok("unit_f02_fingerprint_strictness", "dup + no-tj rejected")


def unit_f05_parser_strictness():
    good = "RELATIONS:\n  adjacent=-\n  aligned_cols=-\n  aligned_rows=1-2\n"
    out = _parse_relations_strict(good, expected_k=3)
    if out["aligned_rows"] != [(1,2)]:
        return _fail("unit_f05_parser_strictness", f"valid: {out}")
    bad = [
        ("RELATIONS:\n  adjacent=-\n  adjacent=-\n  aligned_cols=-\n  aligned_rows=-\n"),
        ("RELATIONS:\n  adjacent=-   \n  aligned_cols=-\n  aligned_rows=-\n"),
        ("RELATIONS:\n  adjacent=1-2-3\n  aligned_cols=-\n  aligned_rows=-\n"),
        ("RELATIONS:\n  adjacent=01-2\n  aligned_cols=-\n  aligned_rows=-\n"),
        ("RELATIONS:\n  adjacent=1-2,1-2\n  aligned_cols=-\n  aligned_rows=-\n"),
    ]
    for txt in bad:
        try:
            _parse_relations_strict(txt, expected_k=3)
            return _fail("unit_f05_parser_strictness", "bad accepted")
        except ValueError: pass
    return _ok("unit_f05_parser_strictness", "5 bad rejected; valid parses")


def unit_f06_manifest_gridtext_matches_loader():
    verified = _resolve_verified_grid_text_path()
    m = bundle_manifest()
    gt = next((e for e in m["upstream"] if e["file"] == "grid_text.py"), None)
    if gt is None:
        return _fail("unit_f06_manifest_gridtext_matches_loader", "no entry")
    if verified is None:
        if gt.get("status") != "MISSING":
            return _fail("unit_f06_manifest_gridtext_matches_loader", "MISSING mismatch")
        return _blk("unit_f06_manifest_gridtext_matches_loader", "env absent")
    if gt.get("status") == "MISSING":
        return _fail("unit_f06_manifest_gridtext_matches_loader", "verified exists")
    if not gt.get("matches_expected"):
        return _fail("unit_f06_manifest_gridtext_matches_loader", "not matching")
    return _ok("unit_f06_manifest_gridtext_matches_loader",
               f"{os.path.basename(verified)} matches")


def unit_f07_hash_mismatch_is_fail():
    fake = {
        "delivered": [{"file": "selector.py", "role": "primary",
                       "md5": "a"*32, "sha256": "a"*64, "bytes": 100}],
        "upstream": [
            {"file": "grid_text.py", "status": "present",
             "md5": "b"*32, "sha256": "b"*64, "bytes": 100,
             "expected": {"md5": "c"*32, "sha256": "c"*64, "bytes": 100},
             "matches_expected": False},
            {"file": "wire.py", "status": "present",
             "md5": "d"*32, "matches_expected": True,
             "expected": {"md5": "d"*32}},
        ],
        "attestation_status": "HASH_MISMATCH",
        "ge3_status": "DEV_PASS",
        "wire_pin_strength": "WEAK_MD5_ONLY",
        "out_of_scope_gates": dict(OUT_OF_SCOPE_GATES),
    }
    status, detail = _classify_manifest_status(fake)
    if status is Status.BLOCKED:
        return _fail("unit_f07_hash_mismatch_is_fail", "BLOCKED")
    if status is not Status.FAIL:
        return _fail("unit_f07_hash_mismatch_is_fail", f"{status.value}")
    return _ok("unit_f07_hash_mismatch_is_fail", "HASH_MISMATCH -> FAIL")


def unit_f09_reasons_strictness():
    good = {k: 500 for k in _get_candidates()}
    b1 = CostBundle(costs=tuple((k, good[k]) for k in _get_candidates()),
                    reasons=(("BASELINE", "r1"), ("BASELINE", "r2")),
                    tokenizer_sha256=TOKENIZER_JSON_SHA256)
    r = select(None, 1000, costs=b1, policy=REFERENCE_POLICY)
    if not (isinstance(r, Recovery) and r.reason == Reason.COST_BUNDLE_MALFORMED):
        return _fail("unit_f09_reasons_strictness", f"dup -> {r}")
    b2 = CostBundle(costs=tuple((k, good[k]) for k in _get_candidates()),
                    reasons=(("NOT_A_CANDIDATE", "r1"),),
                    tokenizer_sha256=TOKENIZER_JSON_SHA256)
    r = select(None, 1000, costs=b2, policy=REFERENCE_POLICY)
    if not (isinstance(r, Recovery) and r.reason == Reason.COST_BUNDLE_MALFORMED):
        return _fail("unit_f09_reasons_strictness", f"unknown -> {r}")
    return _ok("unit_f09_reasons_strictness", "dup + unknown rejected")


def unit_out_of_scope_gates_declared_not_closed():
    for fid in ("F-03", "F-04", "F-06", "F-08"):
        if fid not in OUT_OF_SCOPE_GATES:
            return _fail("unit_out_of_scope_gates_declared_not_closed", f"{fid} missing")
        g = OUT_OF_SCOPE_GATES[fid]
        for field in ("owner", "reason", "arch", "closure_evidence_required"):
            if field not in g:
                return _fail("unit_out_of_scope_gates_declared_not_closed", f"{fid} {field}")
    m = bundle_manifest()
    if set(m["out_of_scope_gates"].keys()) != {"F-03","F-04","F-06","F-08"}:
        return _fail("unit_out_of_scope_gates_declared_not_closed", "gates mismatch")
    if m.get("ge3_status") != "DEV_PASS":
        return _fail("unit_out_of_scope_gates_declared_not_closed", "status")
    return _ok("unit_out_of_scope_gates_declared_not_closed", "4 gates declared")


def unit_f08_duplicate_candidate_keys():
    good_cov = {k: frozenset(_DQC_COVERAGE.get(k, frozenset())) for k in _CANDIDATE_KEYS}
    bad_ck = ("BASELINE","BASELINE","T1","T2","T3(1)","T3(2)","T3(3)","T3(4)","T3(5)")
    try:
        Policy(tokenizer_sha256=TOKENIZER_JSON_SHA256, dqc_coverage=good_cov,
               rank=dict(_RANK), candidate_keys=bad_ck)
        return _fail("unit_f08_duplicate_candidate_keys", "dup accepted")
    except ValueError: pass
    return _ok("unit_f08_duplicate_candidate_keys", "dup rejected")


def unit_f01_fingerprint_sha_consistency():
    good = TOKENIZER_JSON_SHA256; cov = _DQC_COVERAGE
    bad_fp = (("tokenizer.json", "0"*64), ("config.json", "a"*64))
    try:
        Policy(tokenizer_sha256=good, dqc_coverage=cov, rank=dict(_RANK),
               candidate_keys=_CANDIDATE_KEYS,
               tokenizer_fingerprint=bad_fp, require_fingerprint=True)
        return _fail("unit_f01_fingerprint_sha_consistency", "contradictory accepted")
    except ValueError: pass
    good_fp = (("tokenizer.json", good), ("config.json", "a"*64))
    try:
        Policy(tokenizer_sha256=good, dqc_coverage=cov, rank=dict(_RANK),
               candidate_keys=_CANDIDATE_KEYS,
               tokenizer_fingerprint=good_fp, require_fingerprint=True)
    except ValueError as e:
        return _fail("unit_f01_fingerprint_sha_consistency", f"consistent rejected: {e}")
    return _ok("unit_f01_fingerprint_sha_consistency", "contradictory rejected; consistent OK")


def unit_f02_partial_module_purged():
    import tempfile, shutil
    tmpdir = tempfile.mkdtemp(prefix="ge3_f02_")
    try:
        wire_path = os.path.join(tmpdir, "wire.py")
        with open(wire_path, "w") as f: f.write("raise RuntimeError('fail')\n")
        grid_path = os.path.join(tmpdir, "grid_text.py")
        with open(grid_path, "w") as f: f.write("__ok__=True\n")
        with open(grid_path, "rb") as f: gb = f.read()
        with open(wire_path, "rb") as f: wb = f.read()
        M = sys.modules[__name__]
        orig_fw = M._find_ge1_candidates_on_disk
        orig_fg = M._find_grid_text_candidates
        orig_get = M._known_upstream_get
        M._find_ge1_candidates_on_disk = lambda: [wire_path]
        M._find_grid_text_candidates = lambda: [grid_path]
        def patched(name, default=None):
            if name == "wire.py":
                return {"md5": hashlib.md5(wb).hexdigest(),
                        "sha256": hashlib.sha256(wb).hexdigest(), "bytes": len(wb)}
            if name == "grid_text.py":
                return {"md5": hashlib.md5(gb).hexdigest(),
                        "sha256": hashlib.sha256(gb).hexdigest(), "bytes": len(gb)}
            return orig_get(name, default)
        M._known_upstream_get = patched
        saved = {n for n in sys.modules if n.startswith("ge1_wire_verified_")}
        try:
            mod, reason, msg = M._load_ge1_verified()
        finally:
            M._find_ge1_candidates_on_disk = orig_fw
            M._find_grid_text_candidates = orig_fg
            M._known_upstream_get = orig_get
        if mod is not None:
            return _fail("unit_f02_partial_module_purged", "accepted")
        if reason != Reason.ENCODER_ERROR:
            return _fail("unit_f02_partial_module_purged", f"reason={reason}")
        lingerers = {n for n in sys.modules if n.startswith("ge1_wire_verified_") and n not in saved}
        if lingerers:
            return _fail("unit_f02_partial_module_purged", f"leaked: {sorted(lingerers)}")
        return _ok("unit_f02_partial_module_purged", "exec fail purged")
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def unit_f05_hash_before_import_all_branches():
    global _LOADER_LOG
    _LOADER_LOG.clear()
    mod, reason, msg = _load_ge1_verified()
    log = list(_LOADER_LOG)
    inv_err = _loader_log_invariants(log)
    if inv_err is not None:
        return _fail("unit_f05_hash_before_import_all_branches", inv_err)
    if mod is not None:
        imports = [(i, e[1]) for i, e in enumerate(log) if e[0] == "import"]
        if not imports:
            return _fail("unit_f05_hash_before_import_all_branches", "no import entry")
        imp_idx, imp_path = imports[0]
        hashes = [(i, e) for i, e in enumerate(log)
                  if e[0] == "hash" and e[1] == imp_path]
        if not hashes or hashes[0][0] >= imp_idx:
            return _fail("unit_f05_hash_before_import_all_branches", "hash not before")
        return _ok("unit_f05_hash_before_import_all_branches",
                   f"hash@{hashes[0][0]} < import@{imp_idx}")
    if reason == Reason.ENCODER_UNAVAILABLE:
        if any(e[0] == "import" for e in log):
            return _fail("unit_f05_hash_before_import_all_branches", "import present")
        return _blk("unit_f05_hash_before_import_all_branches", f"GE1 absent: {msg}")
    if reason == Reason.ENCODER_ERROR:
        return _ok("unit_f05_hash_before_import_all_branches", f"error: {msg[:40]}")
    return _fail("unit_f05_hash_before_import_all_branches", f"unexpected: {reason}")


def unit_f05_no_import_on_mismatch():
    import tempfile, shutil
    tmpdir = tempfile.mkdtemp(prefix="ge3_f05b_")
    try:
        sentinel = os.path.join(tmpdir, "PWNED.txt")
        poison = os.path.join(tmpdir, "wire.py")
        with open(poison, "w") as f: f.write(f"open({sentinel!r}, 'w').write('PWNED')")
        M = sys.modules[__name__]
        orig = M._find_ge1_candidates_on_disk
        M._find_ge1_candidates_on_disk = lambda: [poison]
        try:
            mod, reason, msg = M._load_ge1_verified()
        finally:
            M._find_ge1_candidates_on_disk = orig
        if os.path.exists(sentinel):
            return _fail("unit_f05_no_import_on_mismatch", "side effect fired")
        if mod is not None:
            return _fail("unit_f05_no_import_on_mismatch", "poison accepted")
        return _ok("unit_f05_no_import_on_mismatch", "poison refused")
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def unit_f07_parser_strict_missing():
    try:
        _parse_relations_strict("RELATIONS:\n  adjacent=-\n  aligned_cols=-\n", expected_k=3)
        return _fail("unit_f07_parser_strict_missing", "missing accepted")
    except ValueError: pass
    out = _parse_relations_strict(
        "RELATIONS:\n  adjacent=-\n  aligned_cols=-\n  aligned_rows=1-2\n", expected_k=3)
    if set(out.keys()) != {"adjacent","aligned_cols","aligned_rows"}:
        return _fail("unit_f07_parser_strict_missing", f"keys: {set(out.keys())}")
    return _ok("unit_f07_parser_strict_missing", "missing raises; valid parses")


def _blank_grid():
    return [[0]*64 for _ in range(64)]


def _indep_extract_components(grid):
    from collections import Counter, deque
    counts = Counter(v for row in grid for v in row)
    if counts:
        mx = max(counts.values()); modal = min(d for d,c in counts.items() if c==mx)
    else: modal = 0
    seen = [[False]*64 for _ in range(64)]
    comps = []
    for r in range(64):
        for c in range(64):
            if seen[r][c] or grid[r][c] == modal: continue
            color = grid[r][c]; q = deque([(r,c)]); seen[r][c] = True; cells = []
            while q:
                cr, cc = q.popleft(); cells.append((cr,cc))
                for dr, dc in ((1,0),(-1,0),(0,1),(0,-1)):
                    nr, nc = cr+dr, cc+dc
                    if 0<=nr<64 and 0<=nc<64 and not seen[nr][nc] and grid[nr][nc]==color:
                        seen[nr][nc]=True; q.append((nr,nc))
            comps.append({"cells":cells,
                          "r0":min(r for r,_ in cells), "r1":max(r for r,_ in cells),
                          "c0":min(c for _,c in cells), "c1":max(c for _,c in cells)})
    comps.sort(key=lambda x:(x["r0"],x["c0"],min(64*r+c for r,c in x["cells"])))
    return comps


def _indep_relations(grid):
    comps = _indep_extract_components(grid); n = len(comps)
    out = {"adjacent":[], "aligned_cols":[], "aligned_rows":[],
           "above":[], "left":[], "bbox_contains":[]}
    for i in range(n):
        for j in range(i+1, n):
            a, b = comps[i], comps[j]
            sa = set(a["cells"])
            adj = any((r+dr,c+dc) in sa for (r,c) in b["cells"]
                      for dr,dc in ((1,0),(-1,0),(0,1),(0,-1)))
            if adj: out["adjacent"].append((i+1,j+1))
            if not (a["c1"] < b["c0"] or b["c1"] < a["c0"]):
                out["aligned_cols"].append((i+1,j+1))
            if not (a["r1"] < b["r0"] or b["r1"] < a["r0"]):
                out["aligned_rows"].append((i+1,j+1))
            # above: only one direction possible (a.r1 < b.r0 implies a.r0 < b.r0)
            if a["r1"] < b["r0"] and not (a["c1"]<b["c0"] or b["c1"]<a["c0"]):
                out["above"].append((i+1,j+1))
            if b["r1"] < a["r0"] and not (a["c1"]<b["c0"] or b["c1"]<a["c0"]):
                out["above"].append((j+1,i+1))
            # left: only one direction possible (a.c1 < b.c0 implies a.c0 < b.c0)
            if a["c1"] < b["c0"] and not (a["r1"]<b["r0"] or b["r1"]<a["r0"]):
                out["left"].append((i+1,j+1))
            if b["c1"] < a["c0"] and not (a["r1"]<b["r0"] or b["r1"]<a["r0"]):
                out["left"].append((j+1,i+1))
            # bbox_contains: both directions possible when bboxes differ but tie on (r0,c0)
            if (a["r0"]<=b["r0"] and a["r1"]>=b["r1"] and a["c0"]<=b["c0"] and a["c1"]>=b["c1"]
                and (a["r0"]<b["r0"] or a["r1"]>b["r1"] or a["c0"]<b["c0"] or a["c1"]>b["c1"])):
                out["bbox_contains"].append((i+1,j+1))
            if (b["r0"]<=a["r0"] and b["r1"]>=a["r1"] and b["c0"]<=a["c0"] and b["c1"]>=a["c1"]
                and (b["r0"]<a["r0"] or b["r1"]>a["r1"] or b["c0"]<a["c0"] or b["c1"]>a["c1"])):
                out["bbox_contains"].append((j+1,i+1))
    return out


def _extract_relations_strict(text, expected_k):
    """Extract pair-dict via the strict parser; used by ts_q4_q5_ge1_path."""
    return _parse_relations_strict(text, expected_k=expected_k)


def ts_q4_q5_ge1_path():
    """F-05 v2.4: exercises above, left, bbox_contains via actual GE1 serialization."""
    ge1_mod, ge1_reason, ge1_msg = _load_ge1_verified()
    if ge1_mod is None:
        return _blk("ts_q4_q5_ge1_path", f"GE1 unavailable: {ge1_msg}")

    fixtures = []

    # 2-object: aligned_cols overlap (existing)
    g = _blank_grid(); g[5][10]=1; g[5][11]=1; g[20][11]=2; g[20][12]=2
    fixtures.append(("2obj_cols_overlap", g, "aligned_cols", {(1,2)}))

    # 2-object: no overlap
    g = _blank_grid(); g[5][10]=1; g[5][11]=1; g[20][20]=2; g[20][21]=2
    fixtures.append(("2obj_cols_no_overlap", g, "aligned_cols", set()))

    # 2-object: adjacent
    g = _blank_grid(); g[5][10]=1; g[5][11]=2
    fixtures.append(("2obj_adjacent", g, "adjacent", {(1,2)}))

    # 2-object: diagonal (not adjacent)
    g = _blank_grid(); g[5][10]=1; g[6][11]=2
    fixtures.append(("2obj_diagonal", g, "adjacent", set()))

    # 3-object: only A-B adjacent
    g = _blank_grid(); g[5][10]=1; g[5][11]=2; g[5][15]=3
    fixtures.append(("3obj_adj_only_AB", g, "adjacent", {(1,2)}))

    # 3-object: only A-B cols overlap
    g = _blank_grid()
    g[5][10]=1; g[5][11]=1
    g[10][11]=2; g[10][12]=2
    g[15][20]=3; g[15][21]=3
    fixtures.append(("3obj_cols_only_AB", g, "aligned_cols", {(1,2)}))

    # F-05 v2.4: ABOVE fixture. obj1 at top (id=1), obj2 at bottom (id=2).
    # above(1,2) expected. above(2,1) impossible by (r0,c0,first_cell) sort.
    g = _blank_grid()
    g[2][5]=1     # id=1 (r0=2)
    g[10][5]=2    # id=2 (r0=10)
    fixtures.append(("2obj_above", g, "above", {(1,2)}))

    # F-05 v2.4: LEFT fixture. obj1 at left (id=1), obj2 at right (id=2).
    g = _blank_grid()
    g[5][2]=1     # id=1 (c0=2)
    g[5][10]=2    # id=2 (c0=10)
    fixtures.append(("2obj_left", g, "left", {(1,2)}))

    # F-05 v2.4: BBOX_CONTAINS reverse — contained object gets LOWER id.
    # A = ring 5x5 minus (0,0): bbox (0,0)-(4,4), first_cell = 1 (cell at (0,1))
    # B = single cell at (0,0): bbox (0,0)-(0,0), first_cell = 0
    # Sort: B (r0=0,c0=0,fc=0) < A (r0=0,c0=0,fc=1). So B=id1, A=id2.
    # A's bbox contains B's bbox. bbox_contains(2, 1) — reverse order!
    g = _blank_grid()
    for r in range(5):
        for c in range(5):
            if (r, c) == (0, 0): continue
            if r == 0 or r == 4 or c == 0 or c == 4:
                g[r][c] = 1      # A = ring minus corner
    g[0][0] = 2                  # B = corner cell
    fixtures.append(("2obj_bbox_contains_reverse", g, "bbox_contains", {(2,1)}))

    for name, g, rel_name, expected_pairs in fixtures:
        try:
            text = ge1_mod.encode(g, "T3", levels_completed=0,
                                  state="NOT_FINISHED", actions=[1,2,3,4,5,6], k=6)
        except Exception as e:
            return _fail("ts_q4_q5_ge1_path", f"{name}: encode: {e}")
        n_objects = len(_indep_extract_components(g))
        try:
            got = _parse_relations_strict(text, expected_k=6, max_endpoint=n_objects)
        except ValueError as e:
            return _fail("ts_q4_q5_ge1_path", f"{name}: parse: {e}")
        oracle = _indep_relations(g)
        got_set = set(got[rel_name])
        oracle_set = set(oracle[rel_name])
        if got_set != oracle_set:
            return _fail("ts_q4_q5_ge1_path",
                         f"{name}: {rel_name} GE1={sorted(got_set)} oracle={sorted(oracle_set)}")
        if got_set != expected_pairs:
            return _fail("ts_q4_q5_ge1_path",
                         f"{name}: {rel_name} got={sorted(got_set)} expected={sorted(expected_pairs)}")
    return _ok("ts_q4_q5_ge1_path",
               f"{len(fixtures)} fixtures (incl above/left/bbox_contains reverse)")


def unit_f01_duplicate_qids():
    good_rest = {k: frozenset(_DQC_COVERAGE[k]) for k in _CANDIDATE_KEYS if k != "BASELINE"}
    for dup in (["Q3","Q3","Q3","Q3","Q3","Q3"], ("Q3","Q3","Q3","Q3","Q3","Q3")):
        try:
            Policy(tokenizer_sha256=TOKENIZER_JSON_SHA256,
                   dqc_coverage={**good_rest, "BASELINE": dup},
                   rank=dict(_RANK), candidate_keys=_CANDIDATE_KEYS)
            return _fail("unit_f01_duplicate_qids", "dup accepted")
        except ValueError: pass
    p = Policy(tokenizer_sha256=TOKENIZER_JSON_SHA256,
               dqc_coverage={**good_rest, "BASELINE": ["Q3","Q6"]},
               rank=dict(_RANK), candidate_keys=_CANDIDATE_KEYS)
    if not isinstance(p.dqc_coverage["BASELINE"], frozenset):
        return _fail("unit_f01_duplicate_qids", "not canonicalized")
    return _ok("unit_f01_duplicate_qids", "dup rejected; canonicalized")


def unit_f01_policy_value_validation():
    try:
        Policy(tokenizer_sha256=TOKENIZER_JSON_SHA256,
               dqc_coverage={k: None for k in _CANDIDATE_KEYS},
               rank=dict(_RANK), candidate_keys=_CANDIDATE_KEYS)
        return _fail("unit_f01_policy_value_validation", "None accepted")
    except ValueError: pass
    try:
        Policy(tokenizer_sha256=TOKENIZER_JSON_SHA256, dqc_coverage=_DQC_COVERAGE,
               rank={"BASELINE":"x","T1":1,"T2":2,"T3":3},
               candidate_keys=_CANDIDATE_KEYS)
        return _fail("unit_f01_policy_value_validation", "string rank accepted")
    except ValueError: pass
    return _ok("unit_f01_policy_value_validation", "bad values rejected")


def unit_f02_missing_digest_fails():
    good = {k: 500 for k in _get_candidates()}
    r = select(None, 1000, costs=_bundle(good, tok_sha=None), policy=REFERENCE_POLICY)
    if not (isinstance(r, Recovery) and r.reason == Reason.TOKENIZER_UNPINNED):
        return _fail("unit_f02_missing_digest_fails", f"missing -> {r}")
    r = select(None, 1000, costs=_bundle(good, tok_sha="0"*64), policy=REFERENCE_POLICY)
    if not (isinstance(r, Recovery) and r.reason == Reason.TOKENIZER_UNPINNED):
        return _fail("unit_f02_missing_digest_fails", f"mismatch -> {r}")
    r = select(None, 1000, costs=_bundle(good), policy=REFERENCE_POLICY)
    if not isinstance(r, Selection):
        return _fail("unit_f02_missing_digest_fails", f"match -> {r}")
    return _ok("unit_f02_missing_digest_fails", "missing/mismatch refused; match OK")


def unit_f02_grid_text_bound():
    import tempfile, shutil
    tmpdir = tempfile.mkdtemp(prefix="ge3_bind_")
    try:
        gt_path = os.path.join(tmpdir, "grid_text.py")
        with open(gt_path, "w") as f: f.write("__verified__ = True\n")
        with open(gt_path, "rb") as f: gb = f.read()
        stale_dir = os.path.join(tmpdir, "stale"); os.makedirs(stale_dir)
        stale_gt = os.path.join(stale_dir, "grid_text.py")
        with open(stale_gt, "w") as f: f.write("__verified__ = 'STALE'\n")
        wire_out = os.path.join(tmpdir, "WIRE_RESULT.txt")
        wire_path = os.path.join(tmpdir, "wire.py")
        with open(wire_path, "w") as f:
            f.write(f"import grid_text\n"
                    f"open({wire_out!r}, 'w').write(str(getattr(grid_text, '__verified__', 'MISSING')))\n"
                    f"class ReasonCode: pass\n"
                    f"def encode(*a, **k): pass\n")
        with open(wire_path, "rb") as f: wb = f.read()
        M = sys.modules[__name__]
        orig_fw = M._find_ge1_candidates_on_disk
        orig_fg = M._find_grid_text_candidates
        orig_get = M._known_upstream_get
        M._find_ge1_candidates_on_disk = lambda: [wire_path]
        M._find_grid_text_candidates = lambda: [stale_gt, gt_path]
        def patched(name, default=None):
            if name == "wire.py":
                return {"md5": hashlib.md5(wb).hexdigest(),
                        "sha256": hashlib.sha256(wb).hexdigest(), "bytes": len(wb)}
            if name == "grid_text.py":
                return {"md5": hashlib.md5(gb).hexdigest(),
                        "sha256": hashlib.sha256(gb).hexdigest(), "bytes": len(gb)}
            return orig_get(name, default)
        M._known_upstream_get = patched
        stale_mod = type(sys)("stale_gt"); stale_mod.__verified__ = "STALE_FROM_SYSMODULES"
        saved = sys.modules.get("grid_text"); sys.modules["grid_text"] = stale_mod
        try:
            mod, reason, msg = M._load_ge1_verified()
        finally:
            M._find_ge1_candidates_on_disk = orig_fw
            M._find_grid_text_candidates = orig_fg
            M._known_upstream_get = orig_get
            if saved is None: sys.modules.pop("grid_text", None)
            else: sys.modules["grid_text"] = saved
        if mod is None:
            return _blk("unit_f02_grid_text_bound", f"loader failed: {msg}")
        with open(wire_out) as f: result = f.read()
        if result != "True":
            return _fail("unit_f02_grid_text_bound", f"wire saw {result!r}")
        return _ok("unit_f02_grid_text_bound", "verified grid_text bound")
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def unit_f03_digest_format():
    good_cov = {k: frozenset(_DQC_COVERAGE[k]) for k in _CANDIDATE_KEYS}
    for bad in ("test", "", "A"*64, "g"*64, "0"*63, "0"*65, "0"*32):
        try:
            Policy(tokenizer_sha256=bad, dqc_coverage=good_cov,
                   rank=dict(_RANK), candidate_keys=_CANDIDATE_KEYS)
            return _fail("unit_f03_digest_format", f"accepted {bad!r}")
        except ValueError: pass
    try:
        Policy(tokenizer_sha256="a"*64, dqc_coverage=good_cov,
               rank=dict(_RANK), candidate_keys=_CANDIDATE_KEYS)
    except ValueError:
        return _fail("unit_f03_digest_format", "valid rejected")
    return _ok("unit_f03_digest_format", "7 invalid rejected; valid OK")


def unit_f03_no_implicit_policy():
    M = sys.modules[__name__]
    if hasattr(M, "select_from_costs_dict"):
        return _fail("unit_f03_no_implicit_policy", "public helper exists")
    try:
        _select_from_costs_dict(None, 1000, {k: 500 for k in _get_candidates()})
        return _fail("unit_f03_no_implicit_policy", "no policy accepted")
    except TypeError: pass
    return _ok("unit_f03_no_implicit_policy", "policy required")


def unit_f04_policy_deep_freeze():
    rank_dict = dict(_RANK)
    cov_dict = {k: list(_DQC_COVERAGE[k]) for k in _CANDIDATE_KEYS}
    ck_list = list(_CANDIDATE_KEYS)
    p = Policy(tokenizer_sha256=TOKENIZER_JSON_SHA256,
               dqc_coverage=cov_dict, rank=rank_dict, candidate_keys=ck_list)
    rank_dict["BASELINE"] = 999
    cov_dict["BASELINE"] = ["Q1","Q2","Q3","Q4","Q5","Q6"]
    ck_list.append("HACKED")
    if p.rank["BASELINE"] != 0: return _fail("unit_f04_policy_deep_freeze", "rank mutated")
    if p.dqc_coverage["BASELINE"] != frozenset({"Q3","Q6"}):
        return _fail("unit_f04_policy_deep_freeze", "coverage mutated")
    if "HACKED" in p.candidate_keys:
        return _fail("unit_f04_policy_deep_freeze", "ck aliased")
    return _ok("unit_f04_policy_deep_freeze", "deep freeze holds")


def unit_f04_policy_fingerprint_enforced():
    costs = {k: 500 for k in _get_candidates()}
    fp_a = (("tokenizer.json", TOKENIZER_JSON_SHA256), ("config.json", "b"*64))
    fp_b = (("tokenizer.json", "c"*64), ("config.json", "d"*64))
    p = Policy(tokenizer_sha256=TOKENIZER_JSON_SHA256, dqc_coverage=_DQC_COVERAGE,
               rank=dict(_RANK), candidate_keys=_CANDIDATE_KEYS,
               tokenizer_fingerprint=fp_a, require_fingerprint=True)
    r = select(None, 1000, costs=_bundle(costs), policy=p)
    if not (isinstance(r, Recovery) and r.reason == Reason.TOKENIZER_UNPINNED):
        return _fail("unit_f04_policy_fingerprint_enforced", f"no-fp -> {r}")
    b2 = CostBundle(costs=tuple((k, costs[k]) for k in _get_candidates()),
                    tokenizer_sha256=TOKENIZER_JSON_SHA256, tokenizer_fingerprint=fp_b)
    r = select(None, 1000, costs=b2, policy=p)
    if not (isinstance(r, Recovery) and r.reason == Reason.TOKENIZER_MISMATCH):
        return _fail("unit_f04_policy_fingerprint_enforced", f"mismatch -> {r}")
    b3 = CostBundle(costs=tuple((k, costs[k]) for k in _get_candidates()),
                    tokenizer_sha256=TOKENIZER_JSON_SHA256, tokenizer_fingerprint=fp_a,
                    tokenizer_stage_ref=hashlib.sha256(
                        json.dumps(fp_a, sort_keys=True).encode()).hexdigest())
    r = select(None, 1000, costs=b3, policy=p)
    if not isinstance(r, Selection):
        return _fail("unit_f04_policy_fingerprint_enforced", f"match -> {r}")
    return _ok("unit_f04_policy_fingerprint_enforced", "no-fp/mismatch/match OK")


def unit_f05_fingerprint_fail_closed():
    import tempfile, shutil, builtins
    tmpdir = tempfile.mkdtemp(prefix="ge3_fp2_")
    try:
        with open(os.path.join(tmpdir, "a.txt"), "w") as f: f.write("x")
        orig_open = builtins.open
        def raising(p, mode="r", *a, **k):
            if mode in ("r","rb") and str(p).endswith(".txt"):
                raise PermissionError(f"sim: {p}")
            return orig_open(p, mode, *a, **k)
        builtins.open = raising
        try:
            try:
                _fingerprint_snapshot(tmpdir)
                return _fail("unit_f05_fingerprint_fail_closed", "did not raise")
            except (PermissionError, OSError): pass
        finally: builtins.open = orig_open
        return _ok("unit_f05_fingerprint_fail_closed", "unreadable fails closed")
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def unit_f06_side_effect_not_fired():
    import tempfile, shutil
    tmpdir = tempfile.mkdtemp(prefix="ge3_se_")
    try:
        sentinel = os.path.join(tmpdir, "PWNED.txt")
        poison = os.path.join(tmpdir, "wire.py")
        with open(poison, "w") as f: f.write(f"open({sentinel!r}, 'w').write('PWNED')")
        M = sys.modules[__name__]
        orig = M._find_ge1_candidates_on_disk
        M._find_ge1_candidates_on_disk = lambda: [poison]
        try:
            mod, reason, msg = M._load_ge1_verified()
        finally:
            M._find_ge1_candidates_on_disk = orig
        if os.path.exists(sentinel):
            return _fail("unit_f06_side_effect_not_fired", "side effect fired")
        if mod is not None:
            return _fail("unit_f06_side_effect_not_fired", "poison accepted")
        return _ok("unit_f06_side_effect_not_fired", "poison refused")
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)


def unit_f08_manifest_matches_loader():
    mod, reason, msg = _load_ge1_verified()
    if mod is None:
        resolved = _resolve_verified_ge1_path()
        if resolved is None:
            return _blk("unit_f08_manifest_matches_loader", f"GE1 absent")
        # v2.7 FIX: a loader failure caused by an ABSENT DEPENDENCY (e.g. grid_text.py
        # not found) is an ENVIRONMENT gap, not a loader/resolver disagreement -> BLOCKED.
        _m = (msg or "")
        if ("not found" in _m) or ("absent" in _m) or ("unavailable" in _m):
            return _blk("unit_f08_manifest_matches_loader",
                        f"loader blocked (dependency absent): {_m[:60]}")
        return _fail("unit_f08_manifest_matches_loader", "loader failed but resolver found")
    imports = [e for e in _LOADER_LOG if e[0] == "import"]
    if not imports: return _fail("unit_f08_manifest_matches_loader", "no import")
    loader_path = imports[-1][1]
    resolved = _resolve_verified_ge1_path()
    if resolved is None or os.path.realpath(loader_path) != os.path.realpath(resolved):
        return _fail("unit_f08_manifest_matches_loader", "loader != resolver")
    m = bundle_manifest()
    wire_entry = next((e for e in m["upstream"] if e["file"] == "wire.py"), None)
    if wire_entry is None: return _fail("unit_f08_manifest_matches_loader", "no wire")
    if wire_entry.get("status") == "MISSING":
        return _fail("unit_f08_manifest_matches_loader", "MISSING but loader found")
    return _ok("unit_f08_manifest_matches_loader",
               f"agree on {os.path.basename(resolved)}")


def unit_boundary_documentation_present():
    src = open(__file__, "rb").read().decode("utf-8", errors="ignore")
    for kw in ("THREAT MODEL", "process isolation", "OUT_OF_SCOPE_GATES", "RULING_REF", "RES_REF"):
        if kw not in src:
            return _fail("unit_boundary_documentation_present", f"missing '{kw}'")
    return _ok("unit_boundary_documentation_present", "docs present")


def unit_boundary_rebinding_flips():
    costs = {k: 500 for k in _get_candidates()}
    before = _sel(costs, 1000)
    if not isinstance(before, Selection):
        return _fail("unit_boundary_rebinding_flips", f"baseline: {before}")
    bad_policy = Policy(
        tokenizer_sha256=TOKENIZER_JSON_SHA256,
        dqc_coverage={**REFERENCE_POLICY.dqc_coverage,
                      "BASELINE": frozenset({"Q1","Q2","Q3","Q4","Q5","Q6"})},
        rank=dict(_RANK), candidate_keys=_CANDIDATE_KEYS)
    after = select(None, 1000, costs=_bundle(costs), policy=bad_policy)
    if not isinstance(after, Selection):
        return _fail("unit_boundary_rebinding_flips", f"bad policy -> {after}")
    if (before.tier, before.k) == (after.tier, after.k):
        return _fail("unit_boundary_rebinding_flips", "did not flip")
    return _ok("unit_boundary_rebinding_flips",
               f"{before.tier}({before.k}) -> {after.tier}({after.k})")


def unit_budget_arithmetic():
    for cap, exp in [(1000,900),(2500,2250),(1200,1080),(1500,1350),
                     (2000,1800),(1800,1620),(3000,2700),(1,1),(10,9),(100,90),
                     (5,4),(15,13),(25,22),(45,40),(55,49),(95,85)]:
        if research_budget(cap) != exp:
            return _fail("unit_budget_arithmetic", f"cap={cap}")
    if research_budget(100_000_000_000_000_005) != 90_000_000_000_000_004:
        return _fail("unit_budget_arithmetic", "large-int")
    for x, e in ((0.4,0),(0.5,1),(1.5,2),(2.5,3),(4.5,5),(5.5,6),(9.5,10)):
        if round_half_up(x) != e:
            return _fail("unit_budget_arithmetic", f"round_half_up({x})")
    return _ok("unit_budget_arithmetic", "16 caps + 7 roundings + large-int")


def unit_candidate_set_exact():
    exp = {"BASELINE","T1","T2","T3(1)","T3(2)","T3(3)","T3(4)","T3(5)","T3(6)"}
    if set(_get_candidates()) != exp or len(_get_candidates()) != 9:
        return _fail("unit_candidate_set_exact", f"{set(_get_candidates())}")
    return _ok("unit_candidate_set_exact", "9 candidates")


def unit_dqc_query_bank():
    F = {"Q1": "How many non-background objects are there?",
         "Q2": "Does any object have inclusive bbox height (r1-r0+1) >= 60?",
         "Q3": "What palette char is at cell (r=32,c=32)?",
         "Q4": "Do any two objects share a grid edge (4-neighbour adjacency)?",
         "Q5": "Do any two objects have OVERLAPPING inclusive bbox column intervals?",
         "Q6": "How many distinct row patterns are there?"}
    T = {"Q1":"int","Q2":"bool","Q3":"char","Q4":"bool","Q5":"bool","Q6":"int"}
    for qid, lit in F.items():
        if _DQC_QUERY_BANK[qid]["word"] != lit:
            return _fail("unit_dqc_query_bank", f"{qid} word")
        if _DQC_QUERY_BANK[qid]["type"] != T[qid]:
            return _fail("unit_dqc_query_bank", f"{qid} type")
    return _ok("unit_dqc_query_bank", "6 wordings byte-exact")


def unit_dqc_coverage_table():
    E = {"BASELINE": frozenset({"Q3","Q6"}), "T1": frozenset({"Q3","Q6"}),
         "T2": frozenset({"Q1","Q2","Q3","Q6"}),
         "T3(1)": frozenset({"Q1","Q2","Q3","Q4","Q6"}),
         "T3(2)": frozenset({"Q1","Q2","Q3","Q4","Q5","Q6"}),
         "T3(3)": frozenset({"Q1","Q2","Q3","Q4","Q5","Q6"}),
         "T3(4)": frozenset({"Q1","Q2","Q3","Q4","Q5","Q6"}),
         "T3(5)": frozenset({"Q1","Q2","Q3","Q4","Q5","Q6"}),
         "T3(6)": frozenset({"Q1","Q2","Q3","Q4","Q5","Q6"})}
    for k, exp in E.items():
        if _dqc_cov_get(k) != exp:
            return _fail("unit_dqc_coverage_table", f"{k}")
    got = [len(_dqc_cov_get(k)) for k in _get_candidates()]
    if got != [2,2,4,5,6,6,6,6,6]:
        return _fail("unit_dqc_coverage_table", f"{got}")
    return _ok("unit_dqc_coverage_table", "2/2/4/5/6/6/6/6/6")


def unit_dqc_primitive_allowlist():
    allowed = frozenset(_DQC_PRIMITIVES.keys())
    if allowed != frozenset({"a","b","c","d","e","f","g"}):
        return _fail("unit_dqc_primitive_allowlist", f"{sorted(allowed)}")
    used = set()
    for qid, q in _DQC_QUERY_BANK.items():
        p = q.get("primitives")
        if not isinstance(p, frozenset) or not p:
            return _fail("unit_dqc_primitive_allowlist", f"{qid}")
        if not p.issubset(allowed):
            return _fail("unit_dqc_primitive_allowlist", f"{qid} disallowed")
        used |= p
    return _ok("unit_dqc_primitive_allowlist", f"exercised={sorted(used)}")


def unit_q5_equals_p3():
    if _DQC_QUERY_BANK["Q5"]["word"] != "Do any two objects have OVERLAPPING inclusive bbox column intervals?":
        return _fail("unit_q5_equals_p3", "Q5 word")
    if "aligned_cols" not in _DQC_QUERY_BANK["Q5"]["method"]:
        return _fail("unit_q5_equals_p3", "Q5 method")
    if Q5_QUERY["relation"] != P3_PROBE["relation"] or Q5_QUERY["relation"] != "aligned_cols":
        return _fail("unit_q5_equals_p3", "relation")
    if "Q5" in _dqc_cov_get("T2"):
        return _fail("unit_q5_equals_p3", "Q5 in T2")
    for k in ("T3(2)","T3(3)","T3(4)","T3(5)","T3(6)"):
        if "Q5" not in _dqc_cov_get(k):
            return _fail("unit_q5_equals_p3", f"Q5 missing {k}")
    return _ok("unit_q5_equals_p3", "Q5==P3==aligned_cols")


def unit_tiebreak_dqc():
    r = _sel({"BASELINE":500,"T1":500,"T2":500,"T3(1)":500,
              "T3(2)":5000,"T3(3)":5000,"T3(4)":5000,"T3(5)":5000,"T3(6)":5000}, 1000)
    if not (isinstance(r, Selection) and r.tier=="T3" and r.k==1 and r.dqc==5):
        return _fail("unit_tiebreak_dqc", f"{r}")
    return _ok("unit_tiebreak_dqc", "T3(1) dqc=5")


def unit_tiebreak_cost():
    r = _sel({"BASELINE":500,"T1":400,"T2":5000,"T3(1)":5000,
              "T3(2)":5000,"T3(3)":5000,"T3(4)":5000,"T3(5)":5000,"T3(6)":5000}, 1000)
    if not (isinstance(r, Selection) and r.tier=="T1"):
        return _fail("unit_tiebreak_cost", f"{r}")
    return _ok("unit_tiebreak_cost", "T1 lower cost")


def unit_tiebreak_rank():
    r = _sel({"BASELINE":500,"T1":500,"T2":5000,"T3(1)":5000,
              "T3(2)":5000,"T3(3)":5000,"T3(4)":5000,"T3(5)":5000,"T3(6)":5000}, 1000)
    if not (isinstance(r, Selection) and r.tier=="BASELINE"):
        return _fail("unit_tiebreak_rank", f"{r}")
    return _ok("unit_tiebreak_rank", "BASELINE rank=0")


def unit_tiebreak_k():
    r = _sel({"BASELINE":5000,"T1":5000,"T2":5000,"T3(1)":5000,
              "T3(2)":800,"T3(3)":800,"T3(4)":800,"T3(5)":8000,"T3(6)":8000}, 1000)
    if not (isinstance(r, Selection) and r.tier=="T3" and r.k==2):
        return _fail("unit_tiebreak_k", f"{r}")
    return _ok("unit_tiebreak_k", "T3(2) lower k")


def unit_empty_feasible():
    r = _sel({k: 5000 for k in _get_candidates()}, 1000)
    if not (isinstance(r, Recovery) and r.reason == Reason.EMPTY_FEASIBLE
            and r.cap_exceeded is True and r.feasible is False):
        return _fail("unit_empty_feasible", f"{r}")
    return _ok("unit_empty_feasible", "empty -> Recovery")


def unit_costs_none():
    r = select(None, 1000, costs=None, policy=REFERENCE_POLICY)
    if not (isinstance(r, Recovery) and r.reason == Reason.COSTS_NOT_MEASURED):
        return _fail("unit_costs_none", f"{r}")
    return _ok("unit_costs_none", "costs=None -> COSTS_NOT_MEASURED")


def unit_cap_invalid():
    for cap in (0,-1,-100):
        r = select(None, cap, costs=CostBundle(costs=()), policy=REFERENCE_POLICY)
        if not (isinstance(r, Recovery) and r.reason == Reason.CAP_INVALID):
            return _fail("unit_cap_invalid", f"{cap}")
    return _ok("unit_cap_invalid", "0/-1/-100")


def unit_cap_bool():
    for cap in (True, False):
        r = select(None, cap, costs=CostBundle(costs=()), policy=REFERENCE_POLICY)
        if not (isinstance(r, Recovery) and r.reason == Reason.CAP_INVALID):
            return _fail("unit_cap_bool", f"{cap}")
    return _ok("unit_cap_bool", "bool rejected")


def unit_cap_non_int():
    for cap in (1000.0, 1000.5, "1000", None, [1000], {"c":1000}):
        r = select(None, cap, costs=CostBundle(costs=()), policy=REFERENCE_POLICY)
        if not (isinstance(r, Recovery) and r.reason == Reason.CAP_INVALID):
            return _fail("unit_cap_non_int", f"{cap!r}")
    return _ok("unit_cap_non_int", "float/str/None/list/dict")


def unit_purity_determinism():
    costs = {k: 500 for k in _get_candidates()}
    outs = [_sel(costs, 1000) for _ in range(20)]
    if len(set((r.tier, r.k) for r in outs)) != 1:
        return _fail("unit_purity_determinism", "diverge")
    return _ok("unit_purity_determinism", "20 identical")


def unit_domination_non_monotone():
    costs = {"BASELINE":5000,"T1":5000,"T2":5000,"T3(1)":5000,
             "T3(2)":1000,"T3(3)":800,"T3(4)":8000,"T3(5)":8000,"T3(6)":8000}
    r = _sel(costs, 1000)
    if not (isinstance(r, Selection) and r.tier=="T3" and r.k==3
            and r.monotone_invariant_holds is False):
        return _fail("unit_domination_non_monotone", f"{r}")
    return _ok("unit_domination_non_monotone", f"T3(3); viol={len(r.monotone_violations)}")


def unit_monotone_holds_positive():
    costs = {"BASELINE":5000,"T1":5000,"T2":5000,"T3(1)":1000,
             "T3(2)":1100,"T3(3)":1200,"T3(4)":1300,"T3(5)":1400,"T3(6)":1500}
    r = _sel(costs, 2000)
    if not (isinstance(r, Selection) and r.monotone_invariant_holds is True
            and not r.monotone_violations):
        return _fail("unit_monotone_holds_positive", f"{r}")
    return _ok("unit_monotone_holds_positive", "flag=True")


def unit_monotone_missing_failsafe():
    ok1, _ = check_monotone_cost({})
    ok2, _ = check_monotone_cost({f"T3({k})": None for k in range(1,7)})
    if ok1 or ok2:
        return _fail("unit_monotone_missing_failsafe", f"{ok1},{ok2}")
    return _ok("unit_monotone_missing_failsafe", "all fail-safe")


def unit_candidate_infeasible_skipped():
    costs = {"BASELINE":500,"T1":400,"T2":None,"T3(1)":None,
             "T3(2)":None,"T3(3)":None,"T3(4)":None,"T3(5)":None,"T3(6)":None}
    reasons = {k: "PAIR_BUDGET_EXCEEDED" for k in
               ("T2","T3(1)","T3(2)","T3(3)","T3(4)","T3(5)","T3(6)")}
    r = _sel(costs, 1000, reasons)
    if not (isinstance(r, Selection) and r.tier=="T1"
            and len(r.infeasible_candidates) == 7):
        return _fail("unit_candidate_infeasible_skipped", f"{r}")
    return _ok("unit_candidate_infeasible_skipped", "7 retained")


def unit_cost_bundle_adversarial():
    good = {k: 500 for k in _get_candidates()}
    pin = TOKENIZER_JSON_SHA256
    checks = [
        ("partial", select(None, 1000, costs=CostBundle(costs=(("BASELINE",100),)),
                          policy=REFERENCE_POLICY), Reason.COSTS_NOT_MEASURED),
        ("empty", select(None, 1000, costs=CostBundle(costs=(), tokenizer_sha256=pin),
                        policy=REFERENCE_POLICY), Reason.COSTS_NOT_MEASURED),
        ("dup", select(None, 1000,
                      costs=CostBundle(costs=tuple([("BASELINE",100),("BASELINE",50)]+
                                                    [(k,500) for k in _get_candidates() if k!="BASELINE"]),
                                       tokenizer_sha256=pin),
                      policy=REFERENCE_POLICY), Reason.COST_BUNDLE_MALFORMED),
        ("extra", select(None, 1000,
                        costs=CostBundle(costs=tuple([(k,500) for k in _get_candidates()]+[("T9(99)",100)]),
                                         tokenizer_sha256=pin),
                        policy=REFERENCE_POLICY), Reason.COSTS_NOT_MEASURED),
        ("bool", select(None, 1000, costs=_bundle({**good, "BASELINE": True}),
                       policy=REFERENCE_POLICY), Reason.COST_BUNDLE_MALFORMED),
        ("float", select(None, 1000, costs=_bundle({**good, "T1": 500.0}),
                        policy=REFERENCE_POLICY), Reason.COST_BUNDLE_MALFORMED),
        ("neg", select(None, 1000, costs=_bundle({**good, "T2": -1}),
                      policy=REFERENCE_POLICY), Reason.COST_BUNDLE_MALFORMED),
        ("None-no-reason", select(None, 1000, costs=_bundle({**good, "T3(1)": None}),
                                  policy=REFERENCE_POLICY), Reason.COST_BUNDLE_MALFORMED),
        ("measurement_err", select(None, 1000,
                                   costs=CostBundle(costs=(), measurement_error="boom"),
                                   policy=REFERENCE_POLICY), Reason.COSTS_NOT_MEASURED),
        ("dict", select(None, 1000, costs={"BASELINE":100},
                       policy=REFERENCE_POLICY), Reason.COSTS_NOT_MEASURED),
        ("3-tuple", select(None, 1000,
                          costs=CostBundle(costs=(("BASELINE",100,"x"),), tokenizer_sha256=pin),
                          policy=REFERENCE_POLICY), Reason.COST_BUNDLE_MALFORMED),
        ("unhashable", select(None, 1000,
                             costs=CostBundle(costs=(["a"],100), tokenizer_sha256=pin),
                             policy=REFERENCE_POLICY), Reason.COST_BUNDLE_MALFORMED),
        ("non-str-key", select(None, 1000,
                              costs=CostBundle(costs=((123,100),), tokenizer_sha256=pin),
                              policy=REFERENCE_POLICY), Reason.COST_BUNDLE_MALFORMED),
        ("bad-reasons", select(None, 1000,
                              costs=CostBundle(costs=tuple((k,500) for k in _get_candidates()),
                                               reasons=("bad",), tokenizer_sha256=pin),
                              policy=REFERENCE_POLICY), Reason.COST_BUNDLE_MALFORMED),
    ]
    for label, r, expected in checks:
        if not (isinstance(r, Recovery) and r.reason == expected):
            return _fail("unit_cost_bundle_adversarial",
                         f"{label}: {expected}, got {getattr(r,'reason',r)}")
    r = _sel({**good, "T3(1)": None}, 1000, {"T3(1)": "PAIR_BUDGET_EXCEEDED"})
    if not isinstance(r, Selection):
        return _fail("unit_cost_bundle_adversarial", f"None-with-reason -> {r}")
    return _ok("unit_cost_bundle_adversarial", "14 classes rejected; None+reason OK")


def unit_dqc_table_override():
    base = {k: 100 for k in _get_candidates()}
    r_ok = select(None, 1000, costs=_bundle(base), policy=REFERENCE_POLICY,
                  dqc_table=dict(_DQC_COVERAGE))
    if not isinstance(r_ok, Selection):
        return _fail("unit_dqc_table_override", f"identical -> {r_ok}")
    tampered = {k: set(_dqc_cov_get(k)) for k in _get_candidates()}
    tampered["BASELINE"] = {"Q1","Q2","Q3","Q4","Q5","Q6"}
    r = select(None, 1000, costs=_bundle(base), policy=REFERENCE_POLICY, dqc_table=tampered)
    if not (isinstance(r, Recovery) and r.reason == Reason.DQC_TABLE_TAMPERED):
        return _fail("unit_dqc_table_override", f"tampered -> {r}")
    extra = dict(_DQC_COVERAGE); extra["T9(99)"] = frozenset({"Q1"})
    r = select(None, 1000, costs=_bundle(base), policy=REFERENCE_POLICY, dqc_table=extra)
    if not (isinstance(r, Recovery) and r.reason == Reason.DQC_TABLE_MALFORMED):
        return _fail("unit_dqc_table_override", f"extra -> {r}")
    return _ok("unit_dqc_table_override", "identical OK; tampered/extras rejected")


def unit_ge1_pin_format():
    for name, pin in _KNOWN_UPSTREAM.items():
        for field, exp_len in (("md5",32),("sha256",64)):
            v = pin.get(field)
            if v is None: continue
            if not isinstance(v, str) or len(v) != exp_len:
                return _fail("unit_ge1_pin_format", f"{name}.{field}")
            if not re.fullmatch(r"[0-9a-f]+", v):
                return _fail("unit_ge1_pin_format", f"{name}.{field} hex")
        if "bytes" in pin and pin["bytes"] is not None:
            if type(pin["bytes"]) is not int or pin["bytes"] <= 0:
                return _fail("unit_ge1_pin_format", f"{name}.bytes")
    return _ok("unit_ge1_pin_format", "pins correct")


def ts_04a_frozen_objective_chain():
    c1 = {"BASELINE":900,"T1":700,"T2":1000,"T3(1)":1050,"T3(2)":1100,
          "T3(3)":1200,"T3(4)":1300,"T3(5)":1400,"T3(6)":1500}
    r1 = _sel(c1, 1200)
    if not (isinstance(r1, Selection) and r1.tier=="T3" and r1.k==1
            and r1.cost==1050 and r1.dqc==5):
        return _fail("ts_04a_frozen_objective_chain", f"c1: {r1}")
    c2 = {"BASELINE":900,"T1":700,"T2":1000,"T3(1)":1200,"T3(2)":1400,
          "T3(3)":1600,"T3(4)":1800,"T3(5)":2000,"T3(6)":2200}
    r2 = _sel(c2, 2500)
    if not (isinstance(r2, Selection) and r2.tier=="T3" and r2.k==2
            and r2.cost==1400 and r2.dqc==6):
        return _fail("ts_04a_frozen_objective_chain", f"c2: {r2}")
    return _ok("ts_04a_frozen_objective_chain", "argmin(-DQC,cost,rank,k) OK")


def ts_04a_real_trace_v36():
    cands = [os.path.join(os.path.dirname(__file__) or ".", "DIR092_selector_trace_v36.json"),
             os.path.join(os.getcwd(), "DIR092_selector_trace_v36.json"),
             "/tmp/DIR092_selector_trace_v36.json"]
    found = next((p for p in cands if os.path.exists(p)), None)
    if found is None:
        return _blk("ts_04a_real_trace_v36", "trace not found")
    try: raw = json.loads(open(found, "r").read())
    except Exception as e: return _fail("ts_04a_real_trace_v36", f"parse: {e}")
    if not isinstance(raw, list):
        return _fail("ts_04a_real_trace_v36", f"root {type(raw).__name__}")
    checked = 0
    for i, rec in enumerate(raw):
        if not isinstance(rec, dict): return _fail("ts_04a_real_trace_v36", f"rec[{i}]")
        cap = rec.get("cap")
        if type(cap) is not int: return _fail("ts_04a_real_trace_v36", f"rec[{i}] cap")
        norm = {}
        for k, v in (rec.get("costs") or {}).items():
            ku = k.strip().upper()
            if ku in ("BASELINE","BASE","E0"): nk = "BASELINE"
            elif ku in ("T1","E1"): nk = "T1"
            elif ku in ("T2","E2"): nk = "T2"
            else:
                mm = re.fullmatch(r"T3[\(\_]?([1-6])[\)]?", ku)
                if not mm: continue
                nk = f"T3({mm.group(1)})"
            if nk not in norm: norm[nk] = v
        if set(norm.keys()) != set(_get_candidates()):
            return _fail("ts_04a_real_trace_v36", f"rec[{i}] keys")
        sel = rec.get("sel"); expected = None
        if sel is None: pass
        elif isinstance(sel, str):
            s = sel.strip().upper()
            if s in ("", "RECOVERY", "NONE", "REFUSED"): pass
            elif s == "BASELINE": expected = ("BASELINE", 0)
            elif s in ("T1","T2"): expected = (s, 0)
            else:
                mm = re.fullmatch(r"T3[\(\_]?([1-6])[\)]?", s)
                if mm: expected = ("T3", int(mm.group(1)))
        elif isinstance(sel, dict):
            t = sel.get("tier")
            if t: expected = (str(t), int(sel.get("k", 0)))
        r = _sel(norm, cap)
        if expected is None:
            if not isinstance(r, Recovery):
                return _fail("ts_04a_real_trace_v36", f"rec[{i}] not Recovery")
        else:
            et, ek = expected
            if not (isinstance(r, Selection) and (r.tier, r.k) == (et, ek)):
                return _fail("ts_04a_real_trace_v36", f"rec[{i}] mismatch")
        checked += 1
    if checked != len(raw): return _fail("ts_04a_real_trace_v36", f"{checked}/{len(raw)}")
    return _ok("ts_04a_real_trace_v36", f"{checked}/{len(raw)} reproduced")


def ts_04a_determinism():
    costs = {k: 500 for k in _get_candidates()}
    a = _sel(costs, 1000); b = _sel(costs, 1000)
    if (a.tier,a.k,a.cost,a.dqc) != (b.tier,b.k,b.cost,b.dqc):
        return _fail("ts_04a_determinism", "diverge")
    return _ok("ts_04a_determinism", "identical")


def ts_purity_cross_process():
    import subprocess
    # v2.7 FIX (F-03): load the module BY PATH via importlib in the child, so a
    # dotted filename (e.g. GE3_selector_v2.6.py) no longer produces an invalid
    # "import GE3_selector_v2.6 as M" statement.
    _file = os.path.abspath(__file__)
    dn = os.path.dirname(_file) or "."
    script = (
        f"import sys, importlib.util as _iu; sys.path.insert(0, {dn!r})\n"
        f"_s = _iu.spec_from_file_location('ge3_child_mod', {_file!r})\n"
        f"M = _iu.module_from_spec(_s); sys.modules['ge3_child_mod'] = M; _s.loader.exec_module(M)\n"
        f"costs = {{'BASELINE':900,'T1':700,'T2':1000,'T3(1)':1050,"
        f"'T3(2)':1400,'T3(3)':1600,'T3(4)':1800,'T3(5)':2000,'T3(6)':2200}}\n"
        f"b = M.CostBundle(costs=tuple((k,costs[k]) for k in M.CANDIDATE_KEYS),"
        f"tokenizer_sha256=M.TOKENIZER_JSON_SHA256)\n"
        f"r = M.select(None, 2500, costs=b, policy=M.REFERENCE_POLICY)\n"
        f"print(f'{{r.tier}}|{{r.k}}|{{r.cost}}|{{r.dqc}}')\n")
    try:
        o1 = subprocess.check_output([sys.executable, "-c", script], timeout=15,
                                     stderr=subprocess.DEVNULL).decode().strip()
        o2 = subprocess.check_output([sys.executable, "-c", script], timeout=15,
                                     stderr=subprocess.DEVNULL).decode().strip()
    except Exception as e:
        return _blk("ts_purity_cross_process", f"{type(e).__name__}: {e}")
    if not o1 or o1 != o2:
        return _fail("ts_purity_cross_process", f"{o1!r} != {o2!r}")
    return _ok("ts_purity_cross_process", f"identical: {o1!r}")


def contract_04b_refusal_contract():
    r = _sel({k: 5000 for k in _get_candidates()}, 1000)
    if not (isinstance(r, Recovery) and r.feasible is False
            and r.cap_exceeded is True and r.reason == Reason.EMPTY_FEASIBLE):
        return _fail("contract_04b_refusal_contract", f"{r}")
    return _ok("contract_04b_refusal_contract", "empty -> refuse")


def contract_04b_no_silent_downgrade():
    costs = {"BASELINE":5000,"T1":100,"T2":100,"T3(1)":100,
             "T3(2)":100,"T3(3)":100,"T3(4)":100,"T3(5)":100,"T3(6)":100}
    r = _sel(costs, 200)
    if not (isinstance(r, Selection) and r.tier=="T3" and r.k==2):
        return _fail("contract_04b_no_silent_downgrade", f"{r}")
    return _ok("contract_04b_no_silent_downgrade", "T3(2)")


def contract_dispatcher_protocol_satisfied():
    class _Spy:
        def __init__(self): self.calls = []
        def dispatch(self, s): self.calls.append(("dispatch", s.tier, s.k)); return "DISPATCHED"
        def refuse(self, r): self.calls.append(("refuse", r.reason.value)); return "REFUSED"
    def route(d, res): return d.refuse(res) if isinstance(res, Recovery) else d.dispatch(res)
    sp = _Spy(); v = route(sp, _sel({k: 5000 for k in _get_candidates()}, 1000))
    if v != "REFUSED" or sp.calls != [("refuse","EMPTY_FEASIBLE")]:
        return _fail("contract_dispatcher_protocol_satisfied", f"empty: {v}")
    sp = _Spy(); v = route(sp, _sel({k: 500 for k in _get_candidates()}, 1000))
    if v != "DISPATCHED" or len(sp.calls) != 1:
        return _fail("contract_dispatcher_protocol_satisfied", f"feasible: {v}")
    return _ok("contract_dispatcher_protocol_satisfied", "Protocol honored")


def integration_pinned_tokenizer():
    grid = _blank_grid()
    bundle = measure_costs(grid)
    if not bundle.is_measured():
        return _blk("integration_pinned_tokenizer", bundle.measurement_error or "not measured")
    r = select(grid, 2500, costs=bundle, policy=REFERENCE_POLICY)
    if not isinstance(r, Selection):
        return _fail("integration_pinned_tokenizer", f"{r}")
    return _ok("integration_pinned_tokenizer",
               f"{r.tier}({r.k}) cost={r.cost} dqc={r.dqc}")


def _pin_is_strong(pin) -> bool:
    """F-01 v2.5: STRONG iff sha256 is 64-hex AND bytes is a positive non-bool int."""
    if not isinstance(pin, Mapping):
        return False
    sha = pin.get("sha256"); b = pin.get("bytes")
    sha_ok = isinstance(sha, str) and re.fullmatch(r"[0-9a-f]{64}", sha) is not None
    b_ok = isinstance(b, int) and not isinstance(b, bool) and b > 0
    return sha_ok and b_ok


def _classify_manifest_status(m):
    """F-01 v2.4: WEAK pin → BLOCKED (not PASS)."""
    if not m.get("delivered"):
        return (Status.FAIL, "no delivered")
    try:
        json.dumps(m, sort_keys=True, default=_json_safe_default)
    except Exception as e:
        return (Status.FAIL, f"serialise: {e}")
    if "out_of_scope_gates" not in m:
        return (Status.FAIL, "missing out_of_scope_gates")
    if m.get("ge3_status") != "DEV_PASS":
        return (Status.FAIL, "ge3_status != DEV_PASS")
    if "wire_pin_strength" not in m:
        return (Status.FAIL, "missing wire_pin_strength")
    status = m.get("attestation_status")
    if status == "COMPLETE_PENDING_SIGNATURE":
        for e in m["upstream"]:
            if e["file"] in _MANDATORY_UPSTREAM:
                if e.get("status") == "MISSING" or not e.get("matches_expected"):
                    return (Status.FAIL, f"COMPLETE but {e['file']} not matching")
        # F-01 v2.5: DERIVE strength from the wire entry's EVIDENCE, not the label
        wire = next((e for e in m.get("upstream", []) if e.get("file") == "wire.py"), None)
        w_exp = (wire or {}).get("expected") or {}
        if not (_pin_is_strong(w_exp) and bool((wire or {}).get("matches_expected"))):
            return (Status.BLOCKED,
                    "wire strong pin not established from evidence (F-08 open) - cannot PASS")
        return (Status.PASS, "COMPLETE_PENDING_SIGNATURE with evidence-verified STRONG pin")
    if status == "HASH_MISMATCH":
        return (Status.FAIL, "HASH_MISMATCH")
    if status == "INCOMPLETE_MANDATORY_INPUTS":
        return (Status.BLOCKED, "INCOMPLETE_MANDATORY_INPUTS")
    return (Status.FAIL, f"unexpected: {status!r}")


def integration_bundle_manifest():
    if _check_pin_format(_KNOWN_UPSTREAM, _MANDATORY_PIN_FIELDS) is not None:
        return _fail("integration_bundle_manifest", "pin format")
    try:
        m = bundle_manifest()
    except Exception as e:
        return _fail("integration_bundle_manifest", f"{e}")
    status, detail = _classify_manifest_status(m)
    if status is Status.PASS:    return _ok("integration_bundle_manifest", detail)
    if status is Status.BLOCKED: return _blk("integration_bundle_manifest", detail)
    return _fail("integration_bundle_manifest", detail)


# ============================================================================
# RELEASE-CONTROLLED TEST INVENTORY (F-04 v2.4)
# ============================================================================
_EXPECTED_TEST_NAMES = (
    "unit_f01_fingerprint_pin_consistency",
    "unit_f01_weak_pin_blocks_pass",
    "unit_f02_self_pair_all_relations",
    "unit_f02_symmetric_noncanonical_rejected",
    "unit_f02_directed_either_order_accepted",
    "unit_f02_reversed_dup_symmetric_rejected",
    "unit_f02_second_relations_marker_rejected",
    "unit_f02_out_of_range_id_rejected",
    "unit_f02_staging_root_readonly",
    "unit_f02_post_load_drift_detected",
    "unit_f03_purge_on_interface_failure",
    "unit_f03_strong_pin_validation",
    "unit_f04_shared_verifier",
    "unit_f04_test_inventory_independent",
    "unit_kimi_f08_suite_integrity_by_names",
    "unit_kimi_f09_cost_bundle_evidence",
    "unit_f01_no_cache_poisoning",
    "unit_f02_fingerprint_strictness",
    "unit_f05_parser_strictness",
    "unit_f06_manifest_gridtext_matches_loader",
    "unit_f07_hash_mismatch_is_fail",
    "unit_f09_reasons_strictness",
    "unit_out_of_scope_gates_declared_not_closed",
    "unit_f08_duplicate_candidate_keys",
    "unit_f01_fingerprint_sha_consistency",
    "unit_f02_partial_module_purged",
    "unit_f05_hash_before_import_all_branches",
    "unit_f05_no_import_on_mismatch",
    "unit_f07_parser_strict_missing",
    "ts_q4_q5_ge1_path",
    "unit_f01_duplicate_qids",
    "unit_f01_policy_value_validation",
    "unit_f02_missing_digest_fails",
    "unit_f02_grid_text_bound",
    "unit_f03_digest_format",
    "unit_f03_no_implicit_policy",
    "unit_f04_policy_deep_freeze",
    "unit_f04_policy_fingerprint_enforced",
    "unit_f05_fingerprint_fail_closed",
    "unit_f06_side_effect_not_fired",
    "unit_f08_manifest_matches_loader",
    "unit_boundary_documentation_present",
    "unit_boundary_rebinding_flips",
    "unit_budget_arithmetic",
    "unit_candidate_set_exact",
    "unit_dqc_query_bank",
    "unit_dqc_coverage_table",
    "unit_dqc_primitive_allowlist",
    "unit_q5_equals_p3",
    "unit_tiebreak_dqc",
    "unit_tiebreak_cost",
    "unit_tiebreak_rank",
    "unit_tiebreak_k",
    "unit_empty_feasible",
    "unit_costs_none",
    "unit_cap_invalid",
    "unit_cap_bool",
    "unit_cap_non_int",
    "unit_purity_determinism",
    "unit_domination_non_monotone",
    "unit_monotone_holds_positive",
    "unit_monotone_missing_failsafe",
    "unit_candidate_infeasible_skipped",
    "unit_cost_bundle_adversarial",
    "unit_dqc_table_override",
    "unit_ge1_pin_format",
    "ts_04a_frozen_objective_chain",
    "ts_04a_real_trace_v36",
    "ts_04a_determinism",
    "ts_purity_cross_process",
    "contract_04b_refusal_contract",
    "contract_04b_no_silent_downgrade",
    "contract_dispatcher_protocol_satisfied",
    "integration_pinned_tokenizer",
    "integration_bundle_manifest",
)


# ============================================================================
# RUNNER (F-04 v2.4: independent inventory)
# ============================================================================
def _all_tests():
    return [
        unit_f01_fingerprint_pin_consistency,
        unit_f01_weak_pin_blocks_pass,
        unit_f02_self_pair_all_relations,
        unit_f02_symmetric_noncanonical_rejected,
        unit_f02_directed_either_order_accepted,
        unit_f02_reversed_dup_symmetric_rejected,
        unit_f02_second_relations_marker_rejected,
        unit_f02_out_of_range_id_rejected,
        unit_f02_staging_root_readonly,
        unit_f02_post_load_drift_detected,
        unit_f03_purge_on_interface_failure,
        unit_f03_strong_pin_validation,
        unit_f04_shared_verifier,
        unit_f04_test_inventory_independent,
        unit_kimi_f08_suite_integrity_by_names,
        unit_kimi_f09_cost_bundle_evidence,
        unit_f01_no_cache_poisoning,
        unit_f02_fingerprint_strictness,
        unit_f05_parser_strictness,
        unit_f06_manifest_gridtext_matches_loader,
        unit_f07_hash_mismatch_is_fail,
        unit_f09_reasons_strictness,
        unit_out_of_scope_gates_declared_not_closed,
        unit_f08_duplicate_candidate_keys,
        unit_f01_fingerprint_sha_consistency,
        unit_f02_partial_module_purged,
        unit_f05_hash_before_import_all_branches,
        unit_f05_no_import_on_mismatch,
        unit_f07_parser_strict_missing,
        ts_q4_q5_ge1_path,
        unit_f01_duplicate_qids,
        unit_f01_policy_value_validation,
        unit_f02_missing_digest_fails,
        unit_f02_grid_text_bound,
        unit_f03_digest_format,
        unit_f03_no_implicit_policy,
        unit_f04_policy_deep_freeze,
        unit_f04_policy_fingerprint_enforced,
        unit_f05_fingerprint_fail_closed,
        unit_f06_side_effect_not_fired,
        unit_f08_manifest_matches_loader,
        unit_boundary_documentation_present,
        unit_boundary_rebinding_flips,
        unit_budget_arithmetic,
        unit_candidate_set_exact,
        unit_dqc_query_bank,
        unit_dqc_coverage_table,
        unit_dqc_primitive_allowlist,
        unit_q5_equals_p3,
        unit_tiebreak_dqc,
        unit_tiebreak_cost,
        unit_tiebreak_rank,
        unit_tiebreak_k,
        unit_empty_feasible,
        unit_costs_none,
        unit_cap_invalid,
        unit_cap_bool,
        unit_cap_non_int,
        unit_purity_determinism,
        unit_domination_non_monotone,
        unit_monotone_holds_positive,
        unit_monotone_missing_failsafe,
        unit_candidate_infeasible_skipped,
        unit_cost_bundle_adversarial,
        unit_dqc_table_override,
        unit_ge1_pin_format,
        ts_04a_frozen_objective_chain,
        ts_04a_real_trace_v36,
        ts_04a_determinism,
        ts_purity_cross_process,
        contract_04b_refusal_contract,
        contract_04b_no_silent_downgrade,
        contract_dispatcher_protocol_satisfied,
        integration_pinned_tokenizer,
        integration_bundle_manifest,
    ]


def _expected_test_names():
    """F-05 v2.5: prefer a release-controlled external inventory; fall back to
    the in-source tuple. The external file is the review-controlled authority."""
    ext = os.environ.get("GE3_EXPECTED_TESTS")
    if ext and os.path.exists(ext):
        with open(ext) as f:
            return tuple(ln.strip() for ln in f if ln.strip())
    return _EXPECTED_TEST_NAMES


def run_all():
    _TESTS.clear()
    fn_names = [fn.__name__ for fn in _all_tests()]
    # F-04 v2.4: duplicate detection in _all_tests()
    if len(set(fn_names)) != len(fn_names):
        _TESTS.append(("SUITE_INTEGRITY", Status.FAIL,
                       "duplicate test name in _all_tests()"))
        return list(_TESTS)
    for fn in _all_tests():
        try:
            fn()
        except Exception as e:
            _TESTS.append((fn.__name__, Status.FAIL, f"{type(e).__name__}: {e}"))
    # F-04 v2.4: compare actual names against RELEASE-CONTROLLED inventory
    actual = tuple(e[0] for e in _TESTS)
    if actual != _expected_test_names():
        first_diff = "?"
        for i in range(min(len(actual), len(_expected_test_names()))):
            if actual[i] != _expected_test_names()[i]:
                first_diff = f"idx {i}: actual={actual[i]!r} inventory={_expected_test_names()[i]!r}"
                break
        _TESTS.append(("SUITE_INTEGRITY", Status.FAIL,
                       f"names mismatch (actual {len(actual)} vs inventory "
                       f"{len(_expected_test_names())}); first diff: {first_diff}"))
    return list(_TESTS)


OUTPUT_DIR = os.environ.get("GE3_OUTPUT_DIR",
                            os.path.join(tempfile.gettempdir(), "ge3_output"))
try:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
except OSError:
    pass


def _write_outputs(results):
    man = bundle_manifest()
    man["test_results"] = [{"name": n, "status": s.value, "detail": d}
                           for n, s, d in results]
    body = json.dumps(man, sort_keys=True, default=_json_safe_default).encode()
    man["manifest_self_sha256"] = hashlib.sha256(body).hexdigest()
    with open(os.path.join(OUTPUT_DIR, "GE3_RELEASE_MANIFEST.json"), "w") as f:
        json.dump(man, f, indent=2, sort_keys=True, default=_json_safe_default)
    lines = ["# GE3 RELEASE MANIFEST",
             f"# version={GE3_VERSION}  status={GE3_STATUS}",
             f"# order={ORDER_REF}  ruling={RULING_REF}  res={RES_REF}",
             f"# arch={ARCH_REF}  spec={SPEC_REV}",
             f"# audit={AUDIT_REF}  aga={AGA_REF}",
             f"# dev_pass_ceiling={man.get('dev_pass_ceiling','')}",
             f"# pin_format={man['pin_format_status']}",
             f"# wire_pin_strength={man['wire_pin_strength']}",
             f"# reference_policy_fingerprint_enforced="
             f"{man['reference_policy_fingerprint_enforced']}",
             f"# attestation_status={man['attestation_status']}",
             f"# signature_status={man['signature_status']}",
             f"# self_sha256={man['manifest_self_sha256']}", "", "# DELIVERED"]
    for e in man["delivered"]:
        lines.append(f"{e.get('file')} | {e.get('bytes')} | {e.get('md5')} | "
                     f"{e.get('sha256')} | {e.get('role')}")
    lines.append(""); lines.append("# UPSTREAM")
    for e in man["upstream"]:
        if e.get("status") == "MISSING":
            lines.append(f"{e['file']} | MISSING | expected={e.get('expected')}")
        else:
            lines.append(f"{e['file']} | {e.get('bytes')} | {e.get('md5')} | "
                         f"{e.get('sha256')} | matches={e.get('matches_expected')}")
    lines.append(""); lines.append("# OUT-OF-SCOPE GATES (not GE3's to close)")
    for gid, g in man["out_of_scope_gates"].items():
        lines.append(f"{gid} | owner={g['owner']} | arch={g['arch']}")
        lines.append(f"     reason: {g['reason']}")
        lines.append(f"     closure_evidence_required: {g['closure_evidence_required']}")
    lines.append(""); lines.append("# TESTS")
    for n, s, d in results:
        lines.append(f"{s.value:7} | {n:50} | {d}")
    lines.append(""); lines.append("# signature_status: PENDING_EXTERNAL_SIGNATURE (Art. 128)")
    with open(os.path.join(OUTPUT_DIR, "GE3_RELEASE_MANIFEST.txt"), "w") as f:
        f.write("\n".join(lines) + "\n")
    return OUTPUT_DIR


def main(argv=None):
    if argv is None:
        argv = sys.argv[1:]
    print("=" * 78)
    print(f"GE3 {GE3_VERSION} — Budget-aware Tier Selector + DQC")
    print(f"STATUS: {GE3_STATUS} (ARCH/103 §7 — PASS requires C-3/D8 binding)")
    print(f"order={ORDER_REF}  ruling={RULING_REF}  res={RES_REF}")
    print(f"arch={ARCH_REF}")
    print(f"audit={AUDIT_REF}  aga={AGA_REF}")
    print(f"spec={SPEC_REV}")
    print(f"out-of-scope: F-03 (GE4) · F-04 (GE4) · F-06 (GE4) · F-08 values (cert body)")
    if not REFERENCE_POLICY.require_fingerprint:
        print("WARNING: REFERENCE_POLICY.require_fingerprint=False — production "
              "MUST supply a Policy with require_fingerprint=True and trusted fingerprint.")
    print("=" * 78)
    results = run_all()
    n_p = n_f = n_b = 0
    for name, st, detail in results:
        print(f"{st.value:7}  {name:50}  {detail}")
        if st is Status.PASS: n_p += 1
        elif st is Status.FAIL: n_f += 1
        else: n_b += 1
    print("-" * 78)
    print(f"PASS={n_p}  FAIL={n_f}  BLOCKED={n_b}")
    try:
        od = _write_outputs(results)
        print(f"[outputs] {od}")
    except Exception as e:
        print(f"[outputs] FAIL: {e}")
    print("-" * 78)
    if n_f:
        print("RESULT: FAILED")
        return 1
    if n_b:
        print(f"RESULT: {GE3_STATUS} — BLOCKED present (env gates); no FAIL")
        return 2
    print(f"RESULT: ALL GE3 {GE3_VERSION} TESTS — PASS — status {GE3_STATUS}")
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile goa/agent/ge4/integration.py
"""GE4 - Integration layer (v3). CM wiring + reasoner prompt + telemetry + auth + cap.
Order DIR/2026/121+123. Basis: Report DIR/092 Rev v38 SS11-15; ARCH/2026/120.
No test seam: the production path has NO callable route to bypass verified upstream binding.
"""
from __future__ import annotations
import hashlib, importlib, importlib.metadata as _md, json, logging, os
from dataclasses import dataclass, field
from enum import Enum
from pathlib import Path
from typing import Any, Callable, Dict, List, Mapping, Optional

log = logging.getLogger("ge4.integration")

# C-08: BUILD_ID is derived from a canonical release manifest (defined below, after the helpers).
SDK_MODULE = "arcengine"
TOKENIZER_PIN_SHA256 = "0997f410c57a1f4e53b09e4be8f4a172d90edd9564368fb0847030937229b9f3"  # HF Qwen3.8-27B (C-04)
CANONICAL_STATES = {"NOT_PLAYED", "NOT_FINISHED", "WIN", "GAME_OVER"}
HARNESS_SURFACE = ("dispatch", "available_actions", "get_transform")
PINNED = {
    "ge1_wire":     ("agent.ge1.wire",         "de722f4fafe10c5fccf9dc4d3bbea388"),
    "ge2_adapters": ("agent.ge2.ge2_adapters", "f030ad9a39440f379cfc806c6b9e073a"),
    "ge3_selector": ("agent.ge3.ge3_selector", "549c64779c5d17dad26f5563561d2795"),
}
SHORT_TO_PIN = {"ge1": "ge1_wire", "ge2": "ge2_adapters", "ge3": "ge3_selector"}
_IFACE = {"ge1": ("encode",), "ge2": ("adapt",), "ge3": ("select",)}   # C-05
CERTIFIED_BASE_MD5 = "bdbc329a990d82c19bf4b4e22b838bdd"     # grid_text.py v1.0.8

def _certified_base_path():
    return Path(__file__).resolve().parent.parent / "grid_text.py"

# ---- C-01: trust root is provisioned from IMMUTABLE build-time config ----------
# NOT an environment variable; NOT written by any test module. A deployment overrides
# only by rebuilding with its own constant, or by placing a signed config file at
# TRUST_ROOT_FILE (read once, cached). Tests override via the module attribute only.
TRUST_ROOT_PUBKEY_HEX = "0" * 64            # placeholder; a real build pins the release root
TRUST_ROOT_FILE = "/etc/ge4/trust_root.pub"
_trust_root_cache = None

class GE4Unbound(RuntimeError): pass
class TUnmeasurable(Exception): pass

class ReasonCode(str, Enum):
    DISPATCHED="DISPATCHED"; H_UNCONFIGURED="H_UNCONFIGURED"; H_CONFIG_INVALID="H_CONFIG_INVALID"
    TOKENIZER_UNAVAILABLE="TOKENIZER_UNAVAILABLE"; LIVE_INPUT_ASSEMBLER_UNAVAILABLE="LIVE_INPUT_ASSEMBLER_UNAVAILABLE"
    REASONER_PROMPT_UNBOUND="REASONER_PROMPT_UNBOUND"; CENSUS_INCOMPLETE="CENSUS_INCOMPLETE"
    CENSUS_SDK_UNAVAILABLE="CENSUS_SDK_UNAVAILABLE"; TRUST_UNVERIFIED="TRUST_UNVERIFIED"
    CONTRACT_UNVERIFIED="CONTRACT_UNVERIFIED"; PINSET_MISMATCH="PINSET_MISMATCH"
    SDK_UNVERIFIED="SDK_UNVERIFIED"; SDK_HASH_MISMATCH="SDK_HASH_MISMATCH"
    HARNESS_UNBOUND="HARNESS_UNBOUND"; HARNESS_PROOF_UNVERIFIED="HARNESS_PROOF_UNVERIFIED"
    REASONER_SINK_FAILED="REASONER_SINK_FAILED"; TELEMETRY_EMIT_FAILED="TELEMETRY_EMIT_FAILED"
    POLICY_UNPINNED="POLICY_UNPINNED"; UPSTREAM_UNBOUND="UPSTREAM_UNBOUND"
    UPSTREAM_UNVERIFIED="UPSTREAM_UNVERIFIED"; ADAPTER_REFUSED="ADAPTER_REFUSED"
    EMPTY_FEASIBLE_SET="EMPTY_FEASIBLE_SET"; T_UNMEASURABLE="T_UNMEASURABLE"
    CAP_EXCEEDED="CAP_EXCEEDED"; ENCODE_FAILED="ENCODE_FAILED"

@dataclass(frozen=True)
class Policy:
    require_fingerprint: bool = True
    trusted_fingerprint: Optional[str] = None

@dataclass
class CapConfig:
    h_limit: Optional[int] = None
    warning_threshold: Optional[int] = None
    token_counter: Optional[Callable[[str], int]] = None
    make_live_input: Optional[Callable[[str], str]] = None

@dataclass
class AuthConfig:
    trust_loader: Optional[Callable[[], Mapping]] = None
    env_owner_attestation: Optional[Any] = None

@dataclass
class DispatchResult:
    decision: str; reason: str; detail: str = ""
    tier: Optional[str] = None; k: int = 0
    cost: Optional[int] = None; dqc: Optional[int] = None
    T: Optional[int] = None; H: Optional[int] = None
    warning: bool = False; cap_exceeded: bool = False
    serialized: Optional[str] = None; dispatched_bytes: Optional[int] = None
    dispatched: bool = False; outcome: Optional[str] = None

@dataclass
class RuntimeConfig:
    cap: CapConfig = field(default_factory=CapConfig)
    auth: AuthConfig = field(default_factory=AuthConfig)
    harness: Any = None
    harness_proof: Optional[Callable[[], Mapping]] = None
    reasoner_sink: Optional[Callable[[str], Any]] = None
    reasoner_base_prompt: Optional[str] = None
    telemetry_sink: Optional[Callable[[Mapping], None]] = None
    upstream: Optional[Mapping] = None
    enable_ge: bool = True
    v20_encoder: Optional[Callable[[Any], str]] = None
    enforce_production_policy: bool = True
    live_authoritative: Optional[Any] = None
    policy: Any = None
    cm: Any = None                                          # C-04: the installed Context Manager
    deployment: Any = None                                  # C-04: trusted deployment metadata
    game_id: Any = None                                     # H-03: game/session identity
    require_session: bool = True                            # H-01: a session id is mandatory
    require_cm: bool = False                                # C-07: set True to require the Context Manager
    v20_fallback: Callable[[Any], str] = lambda g: ""

_CFG = RuntimeConfig()

def configure(**kw) -> None:
    for k, v in kw.items():
        if not hasattr(_CFG, k):
            raise GE4Unbound(f"unknown config key {k!r}")
        setattr(_CFG, k, v)

def reset_configuration() -> None:
    global _CFG
    _CFG = RuntimeConfig()
    _LAST_RESULT.clear(); _OUTCOME_CALLBACKS.clear()        # W-02: no stale run-local state

# ---- C-01 crypto -------------------------------------------------------------
def _canonical(obj) -> bytes:
    return json.dumps(obj, sort_keys=True, separators=(",", ":")).encode("utf-8")

def _ed25519_verify(pubkey, message: bytes, signature) -> bool:
    from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PublicKey
    from cryptography.exceptions import InvalidSignature
    try:
        if isinstance(pubkey, str): pubkey = bytes.fromhex(pubkey)
        if isinstance(signature, str): signature = bytes.fromhex(signature)
        Ed25519PublicKey.from_public_bytes(bytes(pubkey)).verify(bytes(signature), message)
        return True
    except InvalidSignature:
        return False
    except Exception:
        return False

_TRUST_FIELDS = ("sdk_md5", "sdk_version", "env_owner_pubkey", "policy_fingerprint", "deployment_sha256",
                 "build_manifest_sha256")

def _trust_root_pubkey():
    global _trust_root_cache
    if _trust_root_cache is not None:
        return _trust_root_cache
    hexval = TRUST_ROOT_PUBKEY_HEX              # C-01: the compiled constant is the ONLY source
    if hexval == "0" * 64:
        raise GE4Unbound("trust root not provisioned (placeholder constant)")
    if not isinstance(hexval, str) or len(hexval) != 64:
        raise GE4Unbound("trust root not provisioned (need 64 hex chars)")
    try:
        raw = bytes.fromhex(hexval)          # C-01: malformed hex -> TYPED refusal
    except ValueError as e:
        raise GE4Unbound(f"trust root not valid hex: {e!r}")
    if len(raw) != 32:
        raise GE4Unbound("trust root must be 32 bytes")
    _trust_root_cache = raw
    return raw

def _verify_trust_record(trust) -> Optional[str]:
    sig = trust.get("sig")
    if not sig:
        return "trust record not signed (sig missing)"
    body = {k: trust.get(k) for k in _TRUST_FIELDS}
    try:
        root = _trust_root_pubkey()
    except GE4Unbound as e:
        return str(e)
    if not _ed25519_verify(root, _canonical(body), sig):
        return "trust record signature invalid (not the pinned root)"
    return None

def _verify_owner_attestation(att, trust) -> Optional[str]:
    if not isinstance(att, Mapping) or "statement" not in att or "sig" not in att:
        return "attestation must be {statement, sig}"
    st = att["statement"]
    if not isinstance(st, Mapping) or st.get("contract") != "ENV_OWNER":
        return "attestation statement must be a contract=ENV_OWNER mapping"
    if not _ed25519_verify(trust.get("env_owner_pubkey"), _canonical(st), att["sig"]):
        return "env_owner_attestation signature invalid"
    if st.get("policy_fingerprint") != trust.get("policy_fingerprint"):
        return "attestation policy_fingerprint != trust record"
    return None

# ---- upstream ----------------------------------------------------------------
def _md5_file(path) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(65536), b""):
            h.update(b)
    return h.hexdigest()

def _sha256_file(path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(65536), b""):
            h.update(b)
    return h.hexdigest()

def load_upstream(paths, *, verify=True) -> Mapping:
    """The ONLY trusted source of upstream objects. Binds to the ACTUAL imported modules."""
    if verify and not paths:
        raise GE4Unbound("paths required when verify=True (fail-closed)")
    up = {"__verified__": bool(verify), "paths": dict(paths)}   # H-07: truthful state
    for key, (modpath, want_md5) in PINNED.items():
        try:
            mod = importlib.import_module(modpath)
        except Exception as e:
            raise GE4Unbound(f"cannot import {modpath}: {e!r}") from e
        if verify:
            p = paths.get(key)
            if not p or not os.path.exists(p):
                raise GE4Unbound(f"{key}: verification path missing")
            if _md5_file(p) != want_md5:
                raise GE4Unbound(f"{key}: md5 != pinned")
            try:
                if not Path(p).samefile(mod.__file__):
                    raise GE4Unbound(f"{key}: {p} is NOT imported module {mod.__file__}")
            except FileNotFoundError:
                raise GE4Unbound(f"{key}: imported file missing")
        up[key.split("_")[0]] = mod
    return up

def _require_upstream() -> Mapping:
    """C-05: bind to the ACTUAL imported module OBJECTS (identity), not a claimed path."""
    up = _CFG.upstream
    if up is None:
        raise GE4Unbound("upstream not bound")
    if not up.get("__verified__"):
        raise GE4Unbound("upstream not loaded via verified load_upstream()")
    for short, pin_key in SHORT_TO_PIN.items():
        modpath, want_md5 = PINNED[pin_key]
        real = importlib.import_module(modpath)
        if up.get(short) is not real:
            raise GE4Unbound(f"{short} is not the verified imported module {modpath}")
        f = getattr(real, "__file__", None)
        if not f or not os.path.exists(f):
            raise GE4Unbound(f"{short}: module file missing")
        if _md5_file(f) != want_md5:
            raise GE4Unbound(f"{short}: bytes changed since verification")
        for mname in _IFACE[short]:                       # C-05/I-01: code-object tied to the file
            fn = getattr(real, mname, None)
            code = getattr(fn, "__code__", None)
            if (fn is None or getattr(fn, "__module__", None) != real.__name__
                    or code is None or os.path.abspath(code.co_filename) != os.path.abspath(real.__file__)):
                raise GE4Unbound(f"{short}.{mname} is not the module-defined implementation")
    base = _certified_base_path()
    if not base.exists():
        raise GE4Unbound("certified base grid_text.py missing")
    if _md5_file(base) != CERTIFIED_BASE_MD5:
        raise GE4Unbound("certified base grid_text.py changed (read-only contract)")
    return up

# ---- candidates / costs / cap ------------------------------------------------
def _candidates():
    out = [("BASELINE","BASELINE",0),("T1","T1",0),("T2","T2",0)]
    for k in range(1, 7):
        out.append((f"T3({k})","T3",k))
    return out

def _as_token_count(v, ctx):
    if isinstance(v, bool) or not isinstance(v, int) or v < 0:
        raise TUnmeasurable(f"token_counter returned {v!r} for {ctx}")
    return v

def _candidate_costs(up, grid, cfg, hdr=None):
    if cfg.token_counter is None:
        raise GE4Unbound("tokenizer unbound")
    hf = _header_fields(hdr)
    costs = {}
    for name, tier, k in _candidates():
        costs[name] = _as_token_count(cfg.token_counter(
            up["ge1"].encode(grid, tier, levels_completed=hf["levels_completed"],
                             state=hf["state"], actions=hf["actions"], k=k)), f"cost[{name}]")
    # GE4-FIX: GE3.select requires a CostBundle, not a plain dict.
    try:
        return up["ge3"].CostBundle(costs=tuple(costs.items()), tokenizer_sha256=TOKENIZER_PIN_SHA256)
    except Exception:
        return costs

def _header_fields(hdr):
    """GE4-FIX: extract {levels_completed, state, actions} from GE1's wire header (str) or a Mapping."""
    if isinstance(hdr, Mapping):
        av = hdr.get("actions", hdr.get("available_actions", [1,2,3,4,5,6]))
        return {"levels_completed": int(hdr.get("levels_completed", 0) or 0),
                "state": str(hdr.get("state", "NOT_FINISHED")), "actions": list(av)}
    d = {}
    if isinstance(hdr, str):
        for _ln in hdr.splitlines():
            if "=" in _ln:
                _k, _v = _ln.split("=", 1); d[_k.strip()] = _v.strip()
    _av = d.get("available_actions", d.get("actions", "[1, 2, 3, 4, 5, 6]"))
    _acts = [int(x) for x in _av.strip("[]").replace(" ", "").split(",") if x.strip()]
    return {"levels_completed": int(d.get("levels_completed", 0) or 0),
            "state": d.get("state", "NOT_FINISHED"), "actions": _acts}

def _sdk_census():
    """GE4-FIX: build the state-domain census from the pinned SDK (arcengine)."""
    toks = ["NOT_PLAYED", "NOT_FINISHED", "WIN", "GAME_OVER"]
    try:
        import arcengine
        gs = getattr(arcengine, "GameState", None)
        if gs is not None:
            toks = [m.name for m in gs]
    except Exception:
        pass
    return {"version": "v1", "confirmed": True, "complete": True, "missing": [],
            "authenticated": True, "source": "ge4_sdk_bound",
            "mapping": {t: t for t in toks}, "declared_allowlist": toks}

def _cap_decision(T, H, warning_threshold=None):
    """Pure cap boundary. Returns (decision, reason). None/typed-invalid T -> REFUSE."""
    if isinstance(T, bool) or not isinstance(T, int) or T < 0:
        return ("refuse", ReasonCode.T_UNMEASURABLE.value)
    if type(H) is not int or H <= 0:
        return ("refuse", ReasonCode.H_CONFIG_INVALID.value)
    if T >= H:
        return ("refuse", ReasonCode.CAP_EXCEEDED.value)
    return ("dispatch", ReasonCode.DISPATCHED.value)

# ---- auth --------------------------------------------------------------------
REQUIRED_MANIFEST = ("__init__.py", "ge1/__init__.py", "ge2/__init__.py", "ge3/__init__.py", "ge4/__init__.py",
                     "tests/__init__.py", "ge4/integration.py", "ge4/gates.py", "tests/test_ge4.py",
                     "ge1/wire.py", "ge2/ge2_adapters.py", "ge3/ge3_selector.py", "grid_text.py")

def _self_manifest():
    """C-01/H-03: canonical release manifest; REQUIRED files must exist or the build FAILS."""
    base = Path(__file__).resolve().parent.parent          # .../agent
    missing = [r for r in REQUIRED_MANIFEST if not (base / r).exists()]
    if missing:
        raise GE4Unbound(f"incomplete release manifest — missing: {missing}")   # C-01: fail, not omit
    items = [(r, _sha256_file(base / r)) for r in REQUIRED_MANIFEST]
    for _p in sorted((base / "ge4").rglob("*.py")) + sorted((base / "tests").rglob("*.py")):
        _rel = str(_p.relative_to(base))
        if _rel not in REQUIRED_MANIFEST:
            items.append((_rel, _sha256_file(_p)))
    return items
_MANIFEST = _self_manifest()
_MANIFEST_DIGEST = hashlib.sha256(_canonical(_MANIFEST)).hexdigest()
BUILD_ID = "GE4-" + _MANIFEST_DIGEST                                  # C-08

def _deployment_ok(dep, trust):
    """C-01: verify the ACTUAL deployment artifact against the SIGNED deployment_sha256, then
    bind the tokenizer/assembler/hard-cap/threshold. Returns an error string or None."""
    if dep is None:
        return "deployment metadata not bound (C-04)"
    art = getattr(dep, "artifact_path", None)
    if not art or not os.path.exists(art):
        return "deployment artifact not supplied (C-01)"
    try:                                                          # AGA-10: read the bytes ONCE
        _raw = Path(art).read_bytes()
    except OSError as e:
        return f"deployment artifact unreadable (C-01): {e!r}"
    if hashlib.sha256(_raw).hexdigest() != trust.get("deployment_sha256"):   # hash THOSE bytes
        return "deployment artifact sha256 != signed deployment_sha256 (C-01)"
    try:                                                          # parse the SAME bytes (no TOCTOU)
        artcfg = json.loads(_raw.decode("utf-8"))
    except Exception as e:
        return f"deployment artifact not canonical JSON (C-01): {e!r}"
    if not isinstance(artcfg, dict):
        return "deployment artifact must be a JSON object (C-01)"
    cfg = _CFG.cap
    if type(artcfg.get("h_limit")) is not int or artcfg["h_limit"] <= 0:   # C-04: strict artifact typing
        return "artifact h_limit must be a positive int (C-04)"
    if artcfg.get("h_limit") != cfg.h_limit:                      # C-02: authoritative hard cap
        return "hard cap != signed deployment h_limit (C-02)"
    if type(cfg.h_limit) is not int or cfg.h_limit <= 0:
        return "h_limit invalid (C-02)"
    if artcfg.get("warning_threshold") != cfg.warning_threshold:  # C-03: exact agreement (incl. absence)
        return "warning_threshold != signed deployment (C-03)"
    wt = artcfg.get("warning_threshold")
    if wt is not None and (type(wt) is not int or wt < 0 or wt > cfg.h_limit):
        return "deployment warning_threshold invalid (C-03)"
    _ts = artcfg.get("tokenizer_sha256")                                     # AGA-05: strict field types
    if not (isinstance(_ts, str) and len(_ts) == 64 and all(c in "0123456789abcdef" for c in _ts)):
        return "artifact tokenizer_sha256 must be 64-hex (AGA-05)"
    if _ts != TOKENIZER_PIN_SHA256:
        return "deployment tokenizer != pinned (C-04)"
    if type(artcfg.get("truncation_disabled")) is not bool or artcfg.get("truncation_disabled") is not True:
        return "artifact truncation_disabled must be the boolean true (AGA-05)"
    if "require_cm" in artcfg and type(artcfg.get("require_cm")) is not bool:
        return "artifact require_cm must be a boolean (AGA-05)"
    if artcfg.get("warning_threshold") is not None and type(artcfg.get("warning_threshold")) is not int:
        return "artifact warning_threshold must be int or null (AGA-05)"
    if getattr(dep, "loaded_from", None) != art:                  # C-02: dep must be built FROM the artifact
        return "deployment object not constructed from the verified artifact (C-02)"
    if getattr(dep, "tokenizer_sha256", None) != artcfg.get("tokenizer_sha256"):
        return "dep.tokenizer_sha256 != artifact content (C-01)"
    tpath = getattr(dep, "tokenizer_path", None)                  # H-08: the ACTUAL tokenizer FILE
    if not tpath or not os.path.exists(tpath):
        return "tokenizer artifact not supplied (H-08)"
    if _sha256_file(tpath) != TOKENIZER_PIN_SHA256:
        return "tokenizer file sha256 != pinned (H-08)"
    if artcfg.get("require_cm") is True and _CFG.cm is None:      # H-01: require_cm from the release
        return "Context Manager required by the signed release (H-01)"
    if (getattr(dep, "token_counter", None) is not cfg.token_counter
            or getattr(dep, "make_live_input", None) is not cfg.make_live_input):
        return "deployment callables not bound to the runtime (C-04)"
    return None

def _authenticate_once(a):
    """C-01: load the trust record ONCE, verify it, return (fail_or_None, verified_trust_or_None)."""
    try:
        sdk = importlib.import_module(SDK_MODULE)
        installed = {s.value for s in sdk.GameState}
    except Exception as e:
        return ((ReasonCode.CENSUS_SDK_UNAVAILABLE, f"cannot read {SDK_MODULE}.GameState: {e!r}"), None)
    if not CANONICAL_STATES.issubset(installed):
        return ((ReasonCode.CENSUS_INCOMPLETE, f"installed states {sorted(installed)} missing canonical"), None)
    if a.trust_loader is None:
        return ((ReasonCode.TRUST_UNVERIFIED, "trust_loader not supplied (fail-closed)"), None)
    try:
        trust = a.trust_loader()
    except Exception as e:
        return ((ReasonCode.TRUST_UNVERIFIED, f"trust_loader raised: {e!r}"), None)
    if not isinstance(trust, Mapping):
        return ((ReasonCode.TRUST_UNVERIFIED, "trust record must be a mapping"), None)
    for k in _TRUST_FIELDS:
        if not trust.get(k):
            return ((ReasonCode.TRUST_UNVERIFIED, f"trust record missing {k}"), None)
    tr = _verify_trust_record(trust)
    if tr is not None:
        return ((ReasonCode.TRUST_UNVERIFIED, tr), None)
    sdk_path = getattr(sdk, "__file__", None)
    if not sdk_path or not os.path.exists(sdk_path):
        return ((ReasonCode.SDK_UNVERIFIED, "installed SDK file missing"), None)
    if _md5_file(sdk_path) != trust["sdk_md5"]:
        return ((ReasonCode.SDK_HASH_MISMATCH, "installed SDK md5 != trust record"), None)
    if _md.version(SDK_MODULE) != trust["sdk_version"]:
        return ((ReasonCode.SDK_HASH_MISMATCH, "installed SDK version != trust record"), None)
    att = _verify_owner_attestation(a.env_owner_attestation, trust)
    if att is not None:
        return ((ReasonCode.CONTRACT_UNVERIFIED, att), None)
    if trust.get("build_manifest_sha256") != _MANIFEST_DIGEST:     # C-01/H-02: authenticated build id
        return ((ReasonCode.PINSET_MISMATCH, "build manifest != signed release record (C-01)"), None)
    for key, (modpath, want_md5) in PINNED.items():
        try:
            mod = importlib.import_module(modpath)
        except Exception as e:
            return ((ReasonCode.PINSET_MISMATCH, f"{key}: import failed: {e!r}"), None)
        f = getattr(mod, "__file__", None)
        if not f or not os.path.exists(f):
            return ((ReasonCode.PINSET_MISMATCH, f"{key}: installed file missing"), None)
        if _md5_file(f) != want_md5:
            return ((ReasonCode.PINSET_MISMATCH, f"{key}: installed != pinned"), None)
    return (None, trust)

# H-07: the duplicate _authenticate() was removed; production uses _authenticate_once()

def _expected_policy_fingerprint(a):
    if a.trust_loader is None: return None
    try:
        t = a.trust_loader()
        return t.get("policy_fingerprint") if isinstance(t, Mapping) else None
    except Exception:
        return None

# ---- harness -----------------------------------------------------------------
def _harness_surface_ok(h):
    if h is None: return "harness is None"
    missing = [m for m in HARNESS_SURFACE if not callable(getattr(h, m, None))]
    if missing: return f"harness missing callables: {missing}"
    return None

HARNESS_PINNED_COMMIT = "b495c6acaf253c9681cd7b75b4299d352e9ce6f8"

def _harness_artifact(h):
    """C-02: resolve the harness's implementation artifact (instance __file__ OR its class module)."""
    if h is None: return None
    mod = getattr(type(h), "__module__", None)     # C-03: the CLASS module is authoritative
    if mod:
        try:
            m = importlib.import_module(mod)
            mf = getattr(m, "__file__", None)
            if mf and os.path.exists(mf): return mf
        except Exception:
            pass
    f = getattr(h, "__file__", None)
    if f and os.path.exists(f): return f           # fallback only
    return None

def _require_certified_harness() -> Optional[str]:
    """C-02/W-04: require a ROOT-signed {statement,sig}; statement MUST carry a mandatory
    harness_md5 that matches the configured harness's artifact bytes."""
    if _CFG.harness_proof is None:
        return "harness_proof not supplied (certified binding required)"
    try:
        proof = _CFG.harness_proof()
    except Exception as e:
        return f"harness_proof raised: {e!r}"
    if not isinstance(proof, Mapping) or "statement" not in proof or "sig" not in proof:
        return "harness_proof must be {statement, sig} (self-asserted proof rejected)"
    st = proof["statement"]
    if not isinstance(st, Mapping):
        return "harness_proof statement must be a mapping"
    if st.get("commit") != HARNESS_PINNED_COMMIT:
        return f"harness commit {st.get('commit')!r} != pinned"
    want = st.get("harness_sha256")
    if not (isinstance(want, str) and _is_hex64(want)):     # 64-hex SHA-256
        return "harness_proof must carry a mandatory 64-hex harness_sha256"
    h = _CFG.harness
    hf = _harness_artifact(h)                              # C-02: class-module fallback
    if not hf:
        return "configured harness has no resolvable implementation artifact"
    if _sha256_file(hf) != want:                            # SHA-256 vs SHA-256
        return "harness artifact sha256 != proof (unbound object rejected)"
    try:
        root = _trust_root_pubkey()
    except GE4Unbound as e:
        return str(e)
    if not _ed25519_verify(root, _canonical(st), proof["sig"]):
        return "harness_proof signature invalid"
    return None

def _is_hex64(s) -> bool:
    return isinstance(s, str) and len(s) == 64 and all(c in "0123456789abcdef" for c in s)

# ---- observation / prompt ----------------------------------------------------
def _v20(grid):
    if _CFG.v20_encoder is None: raise GE4Unbound("v20_encoder not injected")
    return _CFG.v20_encoder(grid)

def _validate_selection(tier, k):
    """AGA-06: the single selector-result validator (used by obs_block AND the dispatcher)."""
    if tier not in ("BASELINE", "baseline", "T1", "T2", "T3"):
        return f"invalid tier={tier!r}"
    if isinstance(k, bool) or not isinstance(k, int) or k < 0 or k > 6:
        return f"invalid k={k!r}"
    if tier in ("BASELINE", "baseline", "T1", "T2") and k != 0:
        return f"tier {tier} requires k=0, got {k}"
    if tier == "T3" and not (1 <= k <= 6):
        return f"T3 requires k in 1..6, got {k}"
    return None

def obs_block(grid, cap, header=None) -> str:
    if not _CFG.enable_ge:
        return _v20(grid)
    up = _require_upstream()
    costs = _candidate_costs(up, grid, _CFG.cap, header)
    sel = up["ge3"].select(grid, cap, costs=costs, policy=(_CFG.policy or Policy()))
    if isinstance(sel, getattr(up["ge3"], "Recovery", ())) or not getattr(sel, "feasible", True) or getattr(sel, "cap_exceeded", False):
        raise GE4Unbound("selector returned recovery/cap_exceeded; refusing to encode")   # W-01
    _tier = getattr(sel, "tier", None); _k = getattr(sel, "k", 0)
    _verr = _validate_selection(_tier, _k)                                  # AGA-06: same validator
    if _verr is not None:
        raise GE4Unbound(f"obs_block: {_verr}")
    hf = _header_fields(header)
    return up["ge1"].encode(grid, _tier, levels_completed=hf["levels_completed"],
                            state=hf["state"], actions=hf["actions"], k=_k)

def _grid_delta_block(prev_grid, cur_grid) -> str:
    """W-01: certified small-change delta path (changed cells only)."""
    if prev_grid is None or len(prev_grid) != len(cur_grid):
        return ""
    out = []
    for r in range(len(cur_grid)):
        if len(prev_grid[r]) != len(cur_grid[r]):
            return ""
        for c in range(len(cur_grid[r])):
            if prev_grid[r][c] != cur_grid[r][c]:
                out.append(f"{r},{c}:{cur_grid[r][c]}")
    return "GRID_DELTA " + " ".join(out) if out else "GRID_DELTA (none)"

def render_reasoner_prompt(observation, base_prompt, *, marker="GRID_FULL") -> str:
    if not isinstance(observation, str) or not observation:
        raise GE4Unbound("render_reasoner_prompt: observation must be a non-empty str")
    if not isinstance(base_prompt, str):
        raise GE4Unbound("render_reasoner_prompt: base_prompt must be a str")
    _n = base_prompt.count(marker)
    if _n != 1:                                     # AGA-16: exactly one dedicated placeholder
        raise GE4Unbound(f"render_reasoner_prompt: marker {marker!r} occurs {_n} times (need exactly 1)")
    return base_prompt.replace(marker, observation, 1)

def install_context_manager_hook(cm, cap, *, base_prompt, delta_threshold=8):
    """W-01: replaces cm._obs_block; returns the OBSERVATION BLOCK; keeps the GRID_DELTA path."""
    if not hasattr(cm, "_obs_block_orig"):
        cm._obs_block_orig = getattr(cm, "_obs_block", None)
    state = {"prev": None, "game": None, "pending": None}
    def _reset_delta():
        state["prev"] = None; state["game"] = None; state["pending"] = None
    def _commit_delta():
        if state["pending"] is not None:               # C-05: commit only on success
            state["prev"] = state["pending"]; state["pending"] = None
    def _render_obs(obs, grid, game_id=None):
        """W-01/C-04: GRID_DELTA for small changes; prev is snapshotted immutably."""
        if game_id != state["game"]:                       # H-03: new game -> no stale baseline
            state["prev"] = None; state["game"] = game_id
        prev = state["prev"]
        cur = [list(r) for r in grid] if grid is not None else None
        if prev is not None and cur is not None:
            delta = _grid_delta_block(prev, cur)
            n = delta.count(",") if delta else 0
            if delta and 0 < n <= delta_threshold:
                state["pending"] = cur                 # C-05: pending until dispatch succeeds
                return delta
        state["pending"] = cur
        return obs
    _last_grid = {"g": None}
    def _obs_block(*_a, **_kw):
        # GE4-FIX (integration): the CM calls self._obs_block() with NO grid.
        # Accept an optional grid; else use the last one seen; else fall back to the
        # CM's original block so the CM's internal call convention never breaks.
        grid = _a[0] if _a else None
        if grid is None:
            grid = _last_grid["g"]
        if grid is None:
            _orig = getattr(cm, "_obs_block_orig", None)
            if callable(_orig):
                return _orig()
            raise GE4Unbound("obs_block: no grid supplied and no prior grid cached")
        _last_grid["g"] = grid
        return _render_obs(obs_block(grid, cap), grid)   # typed failure propagates, never ""
    cm._render_obs = _render_obs
    cm._reset_delta = _reset_delta
    cm._commit_delta = _commit_delta
    cm._obs_block = _obs_block
    return cm

# ---- telemetry ---------------------------------------------------------------
TELEMETRY_SCHEMA = {"turn":"int","tier":"str|null","k":"int","cost":"int|null","dqc":"int|null",
    "T_at_dispatch":"int|null","H":"int|null","decision":"enum:dispatch|refuse","reason":"str",
    "warning":"bool","dispatched":"bool","outcome":"str|null","game_id":"any|null"}

def make_telemetry(result, turn, outcome=None, game_id=None) -> Dict[str, Any]:
    return {"turn": int(turn), "tier": result.tier, "k": int(result.k), "cost": result.cost,
            "dqc": result.dqc, "T_at_dispatch": result.T, "H": result.H, "decision": result.decision,
            "reason": result.reason, "warning": bool(result.warning),
            "dispatched": bool(getattr(result, "dispatched", False)), "outcome": outcome,
            "game_id": game_id}

def _emit_telemetry(result, turn) -> None:
    if _CFG.telemetry_sink is None:
        _record_telemetry_error(f"turn={turn}: telemetry_sink unbound (no record emitted)")  # explicit
        return
    _CFG.telemetry_sink(make_telemetry(result, turn, outcome=getattr(result, "outcome", None),
                                       game_id=_CFG.game_id))

_OUTCOME_CALLBACKS: List[Callable[[int, str], None]] = []
_TELEMETRY_ERRORS: List[str] = []            # W-02: bounded record of telemetry failures
_TELEMETRY_ERRORS_MAX = 1000

def _record_telemetry_error(msg: str) -> None:
    _TELEMETRY_ERRORS.append(msg)
    if len(_TELEMETRY_ERRORS) > _TELEMETRY_ERRORS_MAX:      # bound the list
        del _TELEMETRY_ERRORS[:len(_TELEMETRY_ERRORS) - _TELEMETRY_ERRORS_MAX]
_LAST_RESULT: Dict[Any, Any] = {}
_LAST_RESULT_MAX = 4096
def _store_result(key, value) -> None:
    _LAST_RESULT[key] = value
    if len(_LAST_RESULT) > _LAST_RESULT_MAX:               # AGA-13: bounded retention
        for _k in list(_LAST_RESULT)[:len(_LAST_RESULT) - _LAST_RESULT_MAX]:
            _LAST_RESULT.pop(_k, None)
def register_outcome_callback(cb) -> None:
    """W-02/AGA-09: register with its arity resolved ONCE (no exception-based probing)."""
    import inspect as _ins
    try:
        _n = len([p for p in _ins.signature(cb).parameters.values()
                  if p.kind in (p.POSITIONAL_ONLY, p.POSITIONAL_OR_KEYWORD)])
    except (TypeError, ValueError):
        _n = 3
    _OUTCOME_CALLBACKS.append((cb, _n))

def record_outcome(turn: int, outcome: str, game_id=None) -> None:
    """W-02: emit a COMPLETED per-turn record carrying the eventual game outcome."""
    if game_id is None:
        game_id = _CFG.game_id
    base = _LAST_RESULT.get((game_id, int(turn)))          # C-06: NO substitution of another session
    if base is None:                                        # W-02: ONE schema, even with no dispatch
        base = DispatchResult(decision="refuse", reason="OUTCOME_WITHOUT_DISPATCH")
    rec = make_telemetry(base, turn, outcome=outcome, game_id=game_id)   # H-04: carry the session id
    if _CFG.telemetry_sink is not None:
        try: _CFG.telemetry_sink(rec)
        except Exception as e:
            _record_telemetry_error(f"outcome turn={turn}: {e!r}")   # telemetry: persist the failure
    for _cb, _n in list(_OUTCOME_CALLBACKS):
        try:                                            # AGA-09: call once, correct arity
            _cb(turn, outcome, game_id) if _n >= 3 else _cb(turn, outcome)
        except Exception as e:
            _record_telemetry_error(f"outcome callback: {e!r}")

def _preflight_sinks() -> Optional[str]:
    if _CFG.telemetry_sink is None:
        return "telemetry_sink unbound (mandatory before dispatch)"
    if _CFG.reasoner_sink is None:
        return "reasoner_sink unbound"
    return None

def _refuse(reason, detail="", **kw) -> DispatchResult:
    return DispatchResult(decision="refuse", reason=reason.value if isinstance(reason, ReasonCode) else str(reason),
                          detail=detail, **kw)

def _select_and_dispatch(observation, cap, *, policy, turn=0) -> DispatchResult:
    game_key = _CFG.game_id
    def emit(r):
        try:
            _emit_telemetry(r, turn)
        except Exception as e:
            if isinstance(r, DispatchResult):
                r.detail = (r.detail + f" | telemetry: {e!r}").strip()
        try:
            _store_result((game_key, turn), r)          # C-06: record EVERY outcome (incl. refusals)
        except Exception as e:
            _record_telemetry_error(f"bookkeeping: {e!r}")
        return r
    if isinstance(turn, bool) or not isinstance(turn, int) or turn < 0:      # H-07: validate turn
        return emit(_refuse(ReasonCode.H_CONFIG_INVALID, f"turn must be a non-negative int, got {turn!r}"))
    try:
        hash((_CFG.game_id, turn))                                            # H-07: ids must be hashable
    except Exception:
        return emit(_refuse(ReasonCode.H_CONFIG_INVALID, "game_id must be hashable"))
    if _CFG.require_session and _CFG.game_id is None:      # H-01: session id for ALL dispatch modes
        return emit(_refuse(ReasonCode.H_CONFIG_INVALID, "game_id (session) required (H-01)"))
    if policy is None:
        return emit(_refuse(ReasonCode.POLICY_UNPINNED, "no policy"))
    if _CFG.enforce_production_policy and not policy.require_fingerprint:
        return emit(_refuse(ReasonCode.POLICY_UNPINNED, "require_fingerprint=True required"))
    af, _trust = _authenticate_once(_CFG.auth)             # C-01: ONE verified trust record
    if af is not None:
        return emit(_refuse(*af))
    if policy.require_fingerprint:
        if not policy.trusted_fingerprint:
            return emit(_refuse(ReasonCode.POLICY_UNPINNED, "trusted_fingerprint missing"))
        if policy.trusted_fingerprint != _trust.get("policy_fingerprint"):
            return emit(_refuse(ReasonCode.POLICY_UNPINNED, "trusted_fingerprint != VERIFIED trust record"))
    cfg = _CFG.cap
    if not _CFG.enable_ge:                                 # C-02: authoritative legacy rollback
        lr = _dispatch_legacy(observation, H_check(cfg), _trust)
        try:
            _store_result((game_key, turn), lr)            # H-04: legacy dispatch recorded
        except Exception:
            pass
        return emit(lr)
    try:
        up = _require_upstream()
    except GE4Unbound as e:
        return emit(_refuse(ReasonCode.UPSTREAM_UNVERIFIED, str(e)))
    if cfg.h_limit is None:
        return emit(_refuse(ReasonCode.H_UNCONFIGURED, "h_limit is None"))
    if type(cfg.h_limit) is not int or cfg.h_limit <= 0:
        return emit(_refuse(ReasonCode.H_CONFIG_INVALID, f"h_limit must be positive int, got {cfg.h_limit!r}"))
    if cfg.warning_threshold is not None and type(cfg.warning_threshold) is not int:
        return emit(_refuse(ReasonCode.H_CONFIG_INVALID, f"warning_threshold must be int, got {cfg.warning_threshold!r}"))
    H = cfg.h_limit
    if cfg.token_counter is None:
        return emit(_refuse(ReasonCode.TOKENIZER_UNAVAILABLE, "token_counter None"))
    if cfg.make_live_input is None:
        return emit(_refuse(ReasonCode.LIVE_INPUT_ASSEMBLER_UNAVAILABLE, "make_live_input None"))
    # C-04: bind the tokenizer/assembler to trusted deployment metadata (fail-closed if unbound)
    _deperr = _deployment_ok(_CFG.deployment, _trust)          # C-01/C-04/H-06: verified deployment
    if _deperr is not None:
        return emit(_refuse(ReasonCode.TOKENIZER_UNAVAILABLE, _deperr))
    if _CFG.reasoner_base_prompt is None:
        return emit(_refuse(ReasonCode.REASONER_PROMPT_UNBOUND, "reasoner_base_prompt not set"))
    pre = _preflight_sinks()
    if pre is not None:
        return emit(_refuse(ReasonCode.HARNESS_UNBOUND, pre))
    surf = _harness_surface_ok(_CFG.harness)
    if surf is not None:
        return emit(_refuse(ReasonCode.HARNESS_UNBOUND, surf))
    proof = _require_certified_harness()
    if proof is not None:
        return emit(_refuse(ReasonCode.HARNESS_PROOF_UNVERIFIED, proof))
    if _CFG.require_cm and _CFG.cm is None:                # C-07: CM mandatory when the spec requires it
        return emit(_refuse(ReasonCode.HARNESS_UNBOUND, "Context Manager required (C-07)"))
    if _CFG.cm is not None:                        # C-04: CM lifecycle mandatory when installed
        if not (hasattr(_CFG.cm, "_render_obs") and hasattr(_CFG.cm, "_commit_delta")):
            return emit(_refuse(ReasonCode.HARNESS_UNBOUND, "CM missing _render_obs/_commit_delta (C-04)"))
        if _CFG.game_id is None:
            return emit(_refuse(ReasonCode.HARNESS_UNBOUND, "game_id required when a CM is installed (C-04)"))
    try:
        _eg = None
        try:
            _h = _CFG.harness
            _eg = getattr(_h, "enumerate_grids", None)
            _cen = getattr(_h, "census", None)
            _census = _cen() if callable(_cen) else _sdk_census()
        except Exception:
            _census = _sdk_census()
        adapted = up["ge2"].adapt(observation, census=_census, enumerate_grids=_eg)
    except Exception as e:
        return emit(_refuse(ReasonCode.ADAPTER_REFUSED, repr(e)))
    if getattr(adapted, "ok", True) is False:
        return emit(_refuse(ReasonCode.ADAPTER_REFUSED, str(getattr(adapted, "reason", "adapter refused"))))
    _val = getattr(adapted, "value", None)
    if isinstance(_val, Mapping):
        grid = _val.get("grid"); _hdr = _val.get("header")
    elif isinstance(adapted, Mapping):
        grid = adapted.get("grid"); _hdr = adapted.get("header")
    else:
        grid = getattr(adapted, "grid", adapted); _hdr = getattr(adapted, "header", None)
    try:
        costs = _candidate_costs(up, grid, cfg, _hdr)
        sel = up["ge3"].select(grid, cap, costs=costs, policy=policy)
    except TUnmeasurable as e:
        return emit(_refuse(ReasonCode.T_UNMEASURABLE, str(e)))
    except GE4Unbound as e:
        return emit(_refuse(ReasonCode.TOKENIZER_UNAVAILABLE, str(e)))
    except Exception as e:
        return emit(_refuse(ReasonCode.EMPTY_FEASIBLE_SET, f"selector error: {e!r}"))
    if isinstance(sel, getattr(up["ge3"], "Recovery", ())) or not getattr(sel, "feasible", True) or getattr(sel, "cap_exceeded", False):
        return emit(_refuse(ReasonCode.EMPTY_FEASIBLE_SET, "empty feasible set / recovery"))
    tier = getattr(sel, "tier"); k = getattr(sel, "k", 0)
    _verr = _validate_selection(tier, k)                                        # AGA-06: same validator
    if _verr is not None:
        return emit(_refuse(ReasonCode.EMPTY_FEASIBLE_SET, f"selector: {_verr}"))
    try:
        _hf = _header_fields(_hdr)
        obs = up["ge1"].encode(grid, tier, levels_completed=_hf["levels_completed"],
                               state=_hf["state"], actions=_hf["actions"], k=k)   # C-04
    except Exception as e:
        return emit(_refuse(ReasonCode.ENCODE_FAILED, repr(e)))
    if _CFG.cm is not None and hasattr(_CFG.cm, "_render_obs"):
        try:
            obs = _CFG.cm._render_obs(obs, grid, _CFG.game_id)   # H-03: pass the game identity
        except GE4Unbound as e:
            return emit(_refuse(ReasonCode.ENCODE_FAILED, str(e)))
    _cost_key = "BASELINE" if str(tier).upper() == "BASELINE" else (tier if tier != "T3" else f"T3({k})")
    _cd = costs.cost_dict() if hasattr(costs, "cost_dict") else costs
    cost = _cd.get(_cost_key, None)
    dqc = getattr(sel, "dqc", None)
    try:
        rendered = render_reasoner_prompt(obs, _CFG.reasoner_base_prompt)
    except GE4Unbound as e:
        return emit(_refuse(ReasonCode.REASONER_PROMPT_UNBOUND, str(e), tier=tier, k=k, cost=cost, dqc=dqc))
    try:
        live = cfg.make_live_input(rendered)
    except Exception as e:
        return emit(_refuse(ReasonCode.LIVE_INPUT_ASSEMBLER_UNAVAILABLE, repr(e), tier=tier, k=k, cost=cost, dqc=dqc))
    if not (isinstance(live, str) and live != ""):   # H-01: validate BEFORE counting/sending
        return emit(_refuse(ReasonCode.LIVE_INPUT_ASSEMBLER_UNAVAILABLE, "live input must be a non-empty str",
                            tier=tier, k=k, cost=cost, dqc=dqc))
    try:
        T = cfg.token_counter(live)
    except Exception as e:
        return emit(_refuse(ReasonCode.T_UNMEASURABLE, repr(e), tier=tier, k=k, cost=cost, dqc=dqc))
    if isinstance(T, bool) or not isinstance(T, int) or T < 0:
        return emit(_refuse(ReasonCode.T_UNMEASURABLE, f"T={T!r}", tier=tier, k=k, cost=cost, dqc=dqc))
    if T >= H:
        return emit(_refuse(ReasonCode.CAP_EXCEEDED, f"T={T} >= H={H}", tier=tier, k=k, cost=cost,
                            dqc=dqc, T=T, H=H, cap_exceeded=True))
    warning = (cfg.warning_threshold is not None and T >= cfg.warning_threshold)
    if not (isinstance(live, str) and live != ""):          # H-05: validate BEFORE counting/sending
        return emit(_refuse(ReasonCode.LIVE_INPUT_ASSEMBLER_UNAVAILABLE, "live input must be a non-empty str",
                            tier=tier, k=k, cost=cost, dqc=dqc, T=T, H=H))
    nbytes = len(live.encode("utf-8"))                      # H-05: UTF-8 BYTES, computed pre-send
    try:
        sent = _CFG.reasoner_sink(live)
    except Exception as e:
        return emit(_refuse(ReasonCode.REASONER_SINK_FAILED, repr(e), tier=tier, k=k, cost=cost, dqc=dqc, T=T, H=H))
    if not ((sent is True) or (isinstance(sent, dict) and sent.get("ok") is True)):   # explicit ack contract
        return emit(_refuse(ReasonCode.REASONER_SINK_FAILED, f"sink not acked: {sent!r}", tier=tier, k=k,
                            cost=cost, dqc=dqc, T=T, H=H))
    if _CFG.cm is not None and hasattr(_CFG.cm, "_commit_delta"):
        try:
            _CFG.cm._commit_delta()                    # C-05/C-01: commit AFTER ack; never raise a false refusal
        except Exception as e:
            _record_telemetry_error(f"commit_delta post-ack: {e!r}")
    res = DispatchResult(decision="dispatch", reason=ReasonCode.DISPATCHED.value, tier=tier, k=k,
                         cost=cost, dqc=dqc, T=T, H=H, warning=warning, cap_exceeded=False,
                         serialized=obs, dispatched_bytes=nbytes, dispatched=True)
    try:
        _store_result((game_key, turn), res)
    except Exception as e:
        _record_telemetry_error(f"post-ack bookkeeping: {e!r}")   # C-01: dispatch already confirmed
    try:
        _emit_telemetry(res, turn)
    except Exception as e:
        res.reason = ReasonCode.TELEMETRY_EMIT_FAILED.value
        res.detail = f"dispatched but telemetry failed: {e!r}"
        _record_telemetry_error(f"dispatch turn={turn}: {e!r}")   # H-05: persist the failure
    return res

def H_check(cfg):
    h = cfg.h_limit
    if type(h) is not int or h <= 0:
        return None
    return h

def _dispatch_legacy(observation, H, trust) -> DispatchResult:
    """C-02/C-05: the legacy v20 path used by the MAIN dispatcher when enable_ge=False."""
    if H is None:
        return _refuse(ReasonCode.H_UNCONFIGURED, "legacy: h_limit invalid")
    _deperr = _deployment_ok(_CFG.deployment, trust)       # C-05: legacy must verify the deployment too
    if _deperr is not None:
        return _refuse(ReasonCode.TOKENIZER_UNAVAILABLE, f"legacy: {_deperr}")
    if _CFG.require_cm and _CFG.cm is None:                # AGA-07: legacy enforces require_cm too
        return _refuse(ReasonCode.HARNESS_UNBOUND, "legacy: Context Manager required (AGA-07)")
    try:
        obs = _v20(observation)
    except GE4Unbound as e:
        return _refuse(ReasonCode.ENCODE_FAILED, str(e))
    cfg = _CFG.cap
    if _CFG.v20_encoder is None:
        return _refuse(ReasonCode.ENCODE_FAILED, "legacy: verified v20 encoder not bound")
    pre = _preflight_sinks()
    if pre is not None:
        return _refuse(ReasonCode.HARNESS_UNBOUND, f"legacy: {pre}")   # C-06: telemetry preflight
    surf = _harness_surface_ok(_CFG.harness)
    if surf is not None:
        return _refuse(ReasonCode.HARNESS_UNBOUND, f"legacy: {surf}")
    proof = _require_certified_harness()
    if proof is not None:
        return _refuse(ReasonCode.HARNESS_PROOF_UNVERIFIED, f"legacy: {proof}")
    live = cfg.make_live_input(obs)
    if not (isinstance(live, str) and live != ""):
        return _refuse(ReasonCode.LIVE_INPUT_ASSEMBLER_UNAVAILABLE, "legacy: live input must be a non-empty str")
    T = cfg.token_counter(live)
    if isinstance(T, bool) or not isinstance(T, int) or T < 0:
        return _refuse(ReasonCode.T_UNMEASURABLE, f"legacy: T={T!r}")
    if T >= H:
        return _refuse(ReasonCode.CAP_EXCEEDED, f"legacy: T={T} >= H={H}", T=T, H=H, cap_exceeded=True)
    sent = _CFG.reasoner_sink(live)
    if not ((sent is True) or (isinstance(sent, dict) and sent.get("ok") is True)):
        return _refuse(ReasonCode.REASONER_SINK_FAILED, f"legacy: sink not acked: {sent!r}")
    return DispatchResult(decision="dispatch", reason=ReasonCode.DISPATCHED.value, T=T, H=H,
                          serialized=obs, dispatched_bytes=len(live.encode("utf-8")),   # H-03
                          dispatched=True, tier="LEGACY")

def select_and_dispatch(observation, cap, *, policy, turn=0) -> DispatchResult:
    try:
        return _select_and_dispatch(observation, cap, policy=policy, turn=turn)
    except Exception as e:
        # W-02 / fail-closed: the boundary NEVER raises; it returns a typed refusal AND logs a record.
        log.exception("select_and_dispatch fail-closed")
        r = DispatchResult(decision="refuse", reason=ReasonCode.UPSTREAM_UNVERIFIED.value,
                           detail=f"unexpected: {type(e).__name__}: {e!r}")
        try:
            _emit_telemetry(r, turn)
        except Exception as e2:
            _record_telemetry_error(f"turn={turn}: {e2!r}")   # W-02: durable evidence
        try:
            _store_result((_CFG.game_id, turn), r)             # C-06: route outer refusal through bookkeeping
        except Exception as e3:
            _record_telemetry_error(f"outer bookkeeping: {e3!r}")
        return r


In [ ]:
%%writefile goa/agent/ge4/gates.py
"""GE4 gate runner (v3). No production-state mutation. Tests call sub-functions directly;
the integration path is exercised only with the REAL verified upstream modules."""
from __future__ import annotations
import hashlib, importlib.util, json, os
from dataclasses import dataclass, field
from enum import Enum
from typing import Any, Dict, List
from . import integration as I

class GateStatus(str, Enum):
    PASS = "PASS"; FAIL = "FAIL"; BLOCKED = "BLOCKED"

_MISSING_DEPS = []
for _d in ("arcengine", "cryptography"):
    try: __import__(_d)
    except Exception: _MISSING_DEPS.append(_d)               # W-03: report BLOCKED, not FAIL

def _dep_blocked(gate_id):
    if _MISSING_DEPS:
        return GateEntry(gate_id, GateStatus.BLOCKED, f"missing deployment deps: {_MISSING_DEPS}")
    return None

@dataclass
class GateEntry:
    gate_id: str; status: GateStatus; reason: str = ""; evidence: Any = None

@dataclass
class GateReport:
    entries: List[GateEntry] = field(default_factory=list)
    def add(self, gate_id, status, reason="", evidence=None):
        self.entries.append(GateEntry(gate_id, status, reason, evidence))
    def failed(self) -> bool:
        return any(e.status in (GateStatus.FAIL, GateStatus.BLOCKED) for e in self.entries)
    def unit_failed(self) -> bool:
        return any(e.status is GateStatus.FAIL for e in self.entries)
    def blocked(self):
        return [e.gate_id for e in self.entries if e.status is GateStatus.BLOCKED]

# ---- helpers -----------------------------------------------------------------
def _is_hex64(s): return isinstance(s, str) and len(s) == 64 and all(c in "0123456789abcdef" for c in s)

def _real_sdk_md5():
    import arcengine
    return I._md5_file(arcengine.__file__)
def _real_sdk_version():
    import importlib.metadata as m
    return m.version("arcengine")

def _sign_keys():
    """C-01: mint keys WITHOUT touching production state."""
    from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey
    return Ed25519PrivateKey.generate(), Ed25519PrivateKey.generate()

from contextlib import contextmanager as _cm
@_cm
def _with_root(root):
    """C-01: install the test root ONLY inside the block, then restore unconditionally."""
    saved = (I.TRUST_ROOT_PUBKEY_HEX, I._trust_root_cache)
    I.TRUST_ROOT_PUBKEY_HEX = root.public_key().public_bytes_raw().hex()
    I._trust_root_cache = None
    try:
        yield
    finally:
        I.TRUST_ROOT_PUBKEY_HEX, I._trust_root_cache = saved

def _signed_trust(root, owner):
    t = {"sdk_md5": _real_sdk_md5(), "sdk_version": _real_sdk_version(),
         "env_owner_pubkey": owner.public_key().public_bytes_raw().hex(), "policy_fingerprint": "FP-TRUSTED",
         "deployment_sha256": "d" * 64, "build_manifest_sha256": I._MANIFEST_DIGEST}
    t["sig"] = root.sign(I._canonical({k: t[k] for k in I._TRUST_FIELDS})).hex()
    return t

def _signed_attestation(owner):
    st = {"contract": "ENV_OWNER", "policy_fingerprint": "FP-TRUSTED"}
    return {"statement": st, "sig": owner.sign(I._canonical(st)).hex()}

def _good_auth(root, owner):
    t = _signed_trust(root, owner)
    return I.AuthConfig(trust_loader=lambda: t, env_owner_attestation=_signed_attestation(owner))

def _good_proof(root, harness_file_sha256):
    st = {"commit": I.HARNESS_PINNED_COMMIT, "harness_sha256": harness_file_sha256}
    return lambda: {"statement": st, "sig": root.sign(I._canonical(st)).hex()}

# ---- unit checks (call SUB-FUNCTIONS directly; no fake upstream in the dispatch path) ----
def _u1_cap_boundary():
    cases = [I._cap_decision(999,1000)[0] == "dispatch",
             I._cap_decision(1000,1000) == ("refuse", I.ReasonCode.CAP_EXCEEDED.value),
             I._cap_decision(1001,1000) == ("refuse", I.ReasonCode.CAP_EXCEEDED.value),
             I._cap_decision(None,1000) == ("refuse", I.ReasonCode.T_UNMEASURABLE.value),
             I._cap_decision(True,1000) == ("refuse", I.ReasonCode.T_UNMEASURABLE.value),
             I._cap_decision(1.9,1000) == ("refuse", I.ReasonCode.T_UNMEASURABLE.value)]
    return GateEntry("GE4-U1-cap-boundary", GateStatus.PASS if all(cases) else GateStatus.FAIL, str(cases))

def _u2_telemetry_schema():
    r = I.DispatchResult(decision="refuse", reason="X", tier="T2", cost=1, dqc=6)
    t = I.make_telemetry(r, turn=7, outcome="done")
    return GateEntry("GE4-U2-telemetry-schema", GateStatus.PASS if set(t) == set(I.TELEMETRY_SCHEMA) else GateStatus.FAIL, "")

def _u3_render_prompt():
    base = "SYSTEM" + chr(10) + "palette: P=pink, p=purple" + chr(10) + "Obs:" + chr(10) + "GRID_FULL" + chr(10) + "END"
    out = I.render_reasoner_prompt("OBS-XYZ", base)
    ok = ("GRID_FULL" not in out and "OBS-XYZ" in out and "palette: P=pink, p=purple" in out
          and out.startswith("SYSTEM") and out.rstrip().endswith("END"))
    try:
        I.render_reasoner_prompt("X", "no marker"); miss = False
    except I.GE4Unbound: miss = True
    return GateEntry("GE4-U3-render-prompt", GateStatus.PASS if (ok and miss) else GateStatus.FAIL, "")

def _u4_auth_no_trust():
    _b = _dep_blocked("GE4-U4-auth-requires-trust")
    if _b is not None: return _b
    root, owner = _sign_keys()
    a = I.AuthConfig(trust_loader=None, env_owner_attestation=None)
    r = I._authenticate_once(a)[0]
    return GateEntry("GE4-U4-auth-requires-trust", GateStatus.PASS if (r and r[0] is I.ReasonCode.TRUST_UNVERIFIED) else GateStatus.FAIL, str(r))

def _u5_auth_fake_attestation():
    _b = _dep_blocked("GE4-U5-auth-rejects-fake-attestation")
    if _b is not None: return _b
    root, owner = _sign_keys()
    a = I.AuthConfig(trust_loader=lambda: _signed_trust(root, owner), env_owner_attestation=b"att")
    with _with_root(root):
        r = I._authenticate_once(a)[0]
    return GateEntry("GE4-U5-auth-rejects-fake-attestation", GateStatus.PASS if (r and r[0] is I.ReasonCode.CONTRACT_UNVERIFIED) else GateStatus.FAIL, str(r))

def _u6_auth_valid():
    _b = _dep_blocked("GE4-U6-auth-valid-signed")
    if _b is not None: return _b
    root, owner = _sign_keys()
    with _with_root(root):
        r = I._authenticate_once(_good_auth(root, owner))[0]
    return GateEntry("GE4-U6-auth-valid-signed", GateStatus.PASS if r is None else GateStatus.FAIL, str(r))

def _u7_auth_malformed_root():
    _b = _dep_blocked("GE4-U7-auth-malformed-root-typed")
    if _b is not None: return _b
    saved_hex, saved_cache = I.TRUST_ROOT_PUBKEY_HEX, I._trust_root_cache
    try:
        I.TRUST_ROOT_PUBKEY_HEX = "ZZ" * 32; I._trust_root_cache = None
        return _u7_body()
    finally:
        I.TRUST_ROOT_PUBKEY_HEX, I._trust_root_cache = saved_hex, saved_cache

def _u7_body():
    I.TRUST_ROOT_PUBKEY_HEX = "ZZ" * 32; I._trust_root_cache = None
    root, owner = _sign_keys.__wrapped__() if hasattr(_sign_keys, "__wrapped__") else _mint()
    I.TRUST_ROOT_PUBKEY_HEX = "ZZ" * 32; I._trust_root_cache = None
    a = I.AuthConfig(trust_loader=lambda: {"sdk_md5": _real_sdk_md5(), "sdk_version": _real_sdk_version(),
                    "env_owner_pubkey": "00"*32, "policy_fingerprint": "FP-TRUSTED", "deployment_sha256": "d"*64, "build_manifest_sha256": I._MANIFEST_DIGEST, "sig": "00"*64},
                    env_owner_attestation={"statement": {"contract":"ENV_OWNER","policy_fingerprint":"FP-TRUSTED"}, "sig":"00"*64})
    r = I._authenticate_once(a)[0]             # must be a TYPED refusal, never an exception
    return GateEntry("GE4-U7-auth-malformed-root-typed", GateStatus.PASS if (r and r[0] is I.ReasonCode.TRUST_UNVERIFIED) else GateStatus.FAIL, str(r))

def _mint():
    from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey
    return Ed25519PrivateKey.generate(), Ed25519PrivateKey.generate()

def _u8_harness_proof_selfasserted():
    _b = _dep_blocked("GE4-U8-harness-proof-selfasserted-rejected")
    if _b is not None: return _b
    r = I._require_certified_harness.__wrapped__ if False else None
    saved = I._CFG.harness_proof; I._CFG.harness_proof = lambda: {"verified": True, "commit": I.HARNESS_PINNED_COMMIT}
    try: err = I._require_certified_harness()
    finally: I._CFG.harness_proof = saved
    return GateEntry("GE4-U8-harness-proof-selfasserted-rejected", GateStatus.PASS if err else GateStatus.FAIL, str(err))

def _u9_harness_proof_requires_digest():
    _b = _dep_blocked("GE4-U9-harness-proof-requires-digest")
    if _b is not None: return _b
    root, _ = _sign_keys()
    st = {"commit": I.HARNESS_PINNED_COMMIT, "harness_sha256": None}
    saved = I._CFG.harness_proof
    I._CFG.harness_proof = lambda: {"statement": st, "sig": root.sign(I._canonical(st)).hex()}
    try: err = I._require_certified_harness()
    finally: I._CFG.harness_proof = saved
    return GateEntry("GE4-U9-harness-proof-requires-digest", GateStatus.PASS if err and "harness_sha256" in err else GateStatus.FAIL, str(err))

def _u10_upstream_identity():
    # a caller-supplied mapping with fake objects is rejected (identity check)
    class FakeGE1:
        __file__ = os.path.abspath("gates.py")
        def encode(self, *a, **k): return "x"
    saved = I._CFG.upstream
    I._CFG.upstream = {"__verified__": True, "paths": {}, "ge1": FakeGE1(), "ge2": object(), "ge3": object()}
    try:
        try: I._require_upstream(); err = None
        except I.GE4Unbound as e: err = str(e)
    finally: I._CFG.upstream = saved
    return GateEntry("GE4-U10-upstream-identity-rejected", GateStatus.PASS if err and "not the verified imported module" in err else GateStatus.FAIL, str(err))

def _u11_telemetry_outcome():
    seen = []
    saved_sink, saved_cbs = I._CFG.telemetry_sink, list(I._OUTCOME_CALLBACKS)
    I._CFG.telemetry_sink = seen.append
    try:
        I.record_outcome(turn=3, outcome="WIN")
    finally:
        I._CFG.telemetry_sink = saved_sink; I._OUTCOME_CALLBACKS[:] = saved_cbs
    ok = len(seen) == 1 and seen[0].get("outcome") == "WIN" and set(seen[0]) == set(I.TELEMETRY_SCHEMA)
    return GateEntry("GE4-U11-telemetry-outcome-recorded", GateStatus.PASS if ok else GateStatus.FAIL, str(seen))

def _u12_emit_none_safe():
    saved = I._CFG.telemetry_sink; I._CFG.telemetry_sink = None
    try:
        I._emit_telemetry(I.DispatchResult(decision="refuse", reason="X"), 0); err = None
    except Exception as e: err = repr(e)
    finally: I._CFG.telemetry_sink = saved
    return GateEntry("GE4-U12-emit-none-safe", GateStatus.PASS if err is None else GateStatus.FAIL, str(err))

def _u14_harness_behavior():
    """C-02: exercise the configured harness's real interface (not just callable presence)."""
    _b = _dep_blocked("GE4-U14-harness-behavior")
    if _b is not None: return _b
    h = I._CFG.harness
    if h is None:
        return GateEntry("GE4-U14-harness-behavior", GateStatus.BLOCKED, "no harness configured")
    try:
        aa = h.available_actions(); tr = h.get_transform()
    except Exception as e:
        return GateEntry("GE4-U14-harness-behavior", GateStatus.FAIL, f"harness call raised: {e!r}")
    ok = isinstance(aa, (list, tuple)) and len(aa) > 0 and callable(tr)
    if not ok:
        return GateEntry("GE4-U14-harness-behavior", GateStatus.FAIL, f"surface invalid: aa={aa!r}")
    if getattr(h, "__test__", False) is not True:          # H-01: never dispatch against a live run
        return GateEntry("GE4-U14-harness-behavior", GateStatus.BLOCKED,
                         "live harness: dispatch not exercised (disposable harness required)")
    try:
        d = h.dispatch({"id": 0})
    except Exception as e:
        return GateEntry("GE4-U14-harness-behavior", GateStatus.FAIL, f"dispatch raised: {e!r}")
    return GateEntry("GE4-U14-harness-behavior", GateStatus.PASS if d is not None else GateStatus.FAIL, "")

def _u13_no_production_seam():
    # the production integration must expose NO seam callable at all
    has = any(hasattr(I, n) for n in ("enable_test_seam", "_TEST_SEAM_ENABLED", "_ALLOW_TEST_SEAM"))
    return GateEntry("GE4-U13-no-production-seam", GateStatus.PASS if not has else GateStatus.FAIL, f"seam attrs present={has}")

_UNIT = [_u1_cap_boundary, _u2_telemetry_schema, _u3_render_prompt, _u4_auth_no_trust, _u5_auth_fake_attestation,
         _u6_auth_valid, _u7_auth_malformed_root, _u8_harness_proof_selfasserted, _u9_harness_proof_requires_digest,
         _u10_upstream_identity, _u11_telemetry_outcome, _u12_emit_none_safe, _u13_no_production_seam,
         _u14_harness_behavior]

def _ts11(live) -> GateEntry:
    """H: T-SPATIAL-11 runs when the deployment live config is supplied."""
    try:
        H = live["H"]; counter = live["token_counter"]; mklive = live["make_live_input"]
    except Exception as e:
        return GateEntry("T-SPATIAL-11", GateStatus.BLOCKED, f"live config incomplete: {e!r}")
    if type(H) is not int or H <= 0:                       # C-03: exact positive int (bool/str rejected)
        return GateEntry("T-SPATIAL-11", GateStatus.FAIL, f"H must be a positive int, got {H!r}")
    try:
        assembled = mklive("probe-observation")          # C-03: CALL the assembler
        T_real = counter(assembled)                      # C-03: CALL the tokenizer
    except Exception as e:
        return GateEntry("T-SPATIAL-11", GateStatus.FAIL, f"live path raised: {type(e).__name__}: {e!r}")
    if not (isinstance(assembled, str) and assembled != ""):
        return GateEntry("T-SPATIAL-11", GateStatus.FAIL, "assembler produced empty input")
    if isinstance(T_real, bool) or not isinstance(T_real, int) or T_real < 0:   # C-07: non-negative int
        return GateEntry("T-SPATIAL-11", GateStatus.FAIL, f"tokenizer returned bad count {T_real!r}")
    if T_real >= H:                                          # C-03: a real input over H is a config failure
        return GateEntry("T-SPATIAL-11", GateStatus.FAIL,
                         f"real assembled live input T={T_real} >= H={H} (deployment config invalid)")
    # C-03: authenticate the live statement, then drive the REAL dispatch boundary
    st, sig = live.get("statement"), live.get("sig")
    try: root = I._trust_root_pubkey()
    except I.GE4Unbound as e:
        return GateEntry("T-SPATIAL-11", GateStatus.BLOCKED, f"trust root: {e}")
    if not isinstance(st, dict) or not sig or not I._ed25519_verify(root, I._canonical(st), sig):
        return GateEntry("T-SPATIAL-11", GateStatus.FAIL, "live configuration attestation invalid")
    if st.get("H") != H:
        return GateEntry("T-SPATIAL-11", GateStatus.FAIL, "live statement H != live H")
    probe = live.get("dispatch_probe")
    if not callable(probe):
        return GateEntry("T-SPATIAL-11", GateStatus.BLOCKED,
                         "deployment-owned dispatch_probe required to exercise the real boundary")
    if getattr(probe, "__boundary__", None) is not True:   # C-05: must be the real boundary wrapper
        return GateEntry("T-SPATIAL-11", GateStatus.BLOCKED,
                         "dispatch_probe must be bound to select_and_dispatch (__boundary__=True)")
    over_raw = "x" * (max(H, 1) * 4)              # AGA-02: a raw observation
    over_in = mklive(over_raw)                    # AGA-02: measure the ASSEMBLED final input
    try:
        T_over = counter(over_in)                              # C-07: independently verify the over-limit input
    except Exception as e:
        return GateEntry("T-SPATIAL-11", GateStatus.FAIL, f"over-limit measurement raised: {e!r}")
    if isinstance(T_over, bool) or not isinstance(T_over, int) or T_over < H:
        return GateEntry("T-SPATIAL-11", GateStatus.FAIL,
                         f"over-limit probe input measured T={T_over} < H={H} (input not actually over)")
    try:
        under = probe("probe-observation"); over = probe(over_raw); unmeas = probe(None)   # AGA-02
    except Exception as e:
        return GateEntry("T-SPATIAL-11", GateStatus.FAIL, f"dispatch_probe raised: {e!r}")
    cases = []
    cases.append(("under-limit dispatches + sink called",
                  under.get("decision") == "dispatch" and under.get("sink_called") is True))
    cases.append(("over-limit refuses + sink WITHHELD",
                  over.get("decision") == "refuse" and over.get("reason") == I.ReasonCode.CAP_EXCEEDED.value
                  and over.get("sink_called") is False))
    cases.append(("unmeasurable/invalid refuses + sink WITHHELD",
                  unmeas.get("decision") == "refuse" and unmeas.get("sink_called") is False
                  and unmeas.get("reason") in (I.ReasonCode.T_UNMEASURABLE.value,
                                               I.ReasonCode.ADAPTER_REFUSED.value)))   # AGA-02
    bad = [n for n, v in cases if not v]
    return GateEntry("T-SPATIAL-11", GateStatus.PASS if not bad else GateStatus.FAIL,
                     str(bad) or f"T_real={T_real} over={over.get('reason')}")

def run_gates(*, live=None) -> GateReport:
    rep = GateReport()
    _before = (I.TRUST_ROOT_PUBKEY_HEX, I._trust_root_cache)   # C-05: snapshot the anchor
    for fn in _UNIT:
        try:
            rep.entries.append(fn())
        except Exception as e:
            rep.add(fn.__name__, GateStatus.FAIL, f"gate crashed: {type(e).__name__}: {e!r}")
    # C-05: a gate must NOT leave the production trust anchor changed
    I.TRUST_ROOT_PUBKEY_HEX, I._trust_root_cache = _before
    if (I.TRUST_ROOT_PUBKEY_HEX, I._trust_root_cache) != _before:
        rep.add("TRUST-ANCHOR-UNCHANGED", GateStatus.FAIL, "trust anchor mutated by gates")
    else:
        rep.add("TRUST-ANCHOR-UNCHANGED", GateStatus.PASS, "restored")
    # T-SPATIAL-11 executable when live config supplied; else BLOCKED (never false-PASS)
    try:                                                   # C-03: _ts11 must never abort the run
        rep.entries.append(_ts11(live) if live else GateEntry("T-SPATIAL-11", GateStatus.BLOCKED,
                           "requires authoritative H + deployment tokenizer + complete live input"))
    except Exception as e:
        rep.add("T-SPATIAL-11", GateStatus.FAIL, f"gate crashed: {type(e).__name__}: {e!r}")
    for gid, why in (("T-SPATIAL-06","authoritative H + deployment tokenizer not supplied"),
                     ("T-SPATIAL-07","model-test runner + probes + frozen bootstrap required"),
                     ("T-SPATIAL-10","25-frame independent relation oracle required")):
        rep.add(gid, GateStatus.BLOCKED, why)
    return rep


In [ ]:
%%writefile goa/agent/tests/__init__.py
# GE4 tests package


In [ ]:
%%writefile goa/agent/tests/test_ge4.py


"""GE4 unit tests (v3) — plain asserts; exit 0 PASS, 1 FAIL, 2 BLOCKED. No pytest fixtures."""
import os, sys
sys.path.insert(0, os.getcwd())
from agent.ge4 import gates as G

def _run_all():
    import agent.ge4.integration as _I
    _anchor = (_I.TRUST_ROOT_PUBKEY_HEX, _I._trust_root_cache)   # C-01: snapshot
    fails = []; blocked = []
    for fn in list(getattr(G, "_UNIT", [])):               # C-02: explicit registry only
        name = getattr(fn, "__name__", str(fn))
        try:
            e = fn()
            st = getattr(e, "status", None)
            print(f"[{getattr(st,'value',st)}] {name}")
            if st is G.GateStatus.FAIL: fails.append(name)
            elif st is G.GateStatus.BLOCKED: blocked.append(name)
        except Exception as ex:
            print(f"[FAIL] {name}: {type(ex).__name__}: {ex}"); fails.append(name)
    _I.TRUST_ROOT_PUBKEY_HEX, _I._trust_root_cache = _anchor   # C-01: restore
    if (_I.TRUST_ROOT_PUBKEY_HEX, _I._trust_root_cache) != _anchor:
        print("[FAIL] TRUST-ANCHOR-UNCHANGED (mutated)"); raise SystemExit(1)
    if fails:
        print(f"\n{len(fails)} FAILURE(S)"); raise SystemExit(1)
    if blocked:
        print(f"\n{len(blocked)} BLOCKED (deployment deps/inputs missing)"); raise SystemExit(2)
    print("\nAll unit tests passed."); raise SystemExit(0)

if __name__ == "__main__":
    _run_all()


In [ ]:
# ============================================================================
# GE WIRING (Run-21 + GE) — bind upstream, install the CM hook, wire the reasoner
# prompt + telemetry. GE <-> CM <-> deep-mode info exchange. Additive; fail-closed.
# ============================================================================
import sys, importlib, hashlib, os as _os
sys.path.insert(0, "/kaggle/working/goa")
# belt-and-suspenders: if any %%writefile cell ever fails (empty body), create the GE4
# manifest-required files so the import does not die with "incomplete release manifest".
for _rel in ("goa/agent/tests/__init__.py",):
    _f = "/kaggle/working/" + _rel
    _os.makedirs(_os.path.dirname(_f), exist_ok=True)
    if not _os.path.exists(_f) or _os.path.getsize(_f) == 0:
        open(_f, "w").write("# ge4 tests package")
if not _os.path.exists("/kaggle/working/goa/agent/grid_text.py"):
    print("WARN goa/agent/grid_text.py missing — the GE4 manifest will fail (cell 6)")
from agent.ge4 import integration as GE4
from agent.ge1 import wire as GE1
from agent.ge2 import ge2_adapters as GE2
from agent.ge3 import ge3_selector as GE3

GE_CAP = 2500          # research model-test cap (NOT the production H — owner supplies H)
GE_ENABLED = True      # flip to False for the byte-identical v20 no-tier path

def _ge_md5(p):
    return hashlib.md5(open(p, "rb").read()).hexdigest()

# 1. bind the upstream (the ONLY trusted source; md5-verified)
_ge_up = GE4.load_upstream({
    "ge1_wire":     "/kaggle/working/goa/agent/ge1/wire.py",
    "ge2_adapters": "/kaggle/working/goa/agent/ge2/ge2_adapters.py",
    "ge3_selector": "/kaggle/working/goa/agent/ge3/ge3_selector.py",
})
GE4._CFG.upstream = _ge_up
GE4._CFG.policy = GE3.REFERENCE_POLICY
GE4._CFG.enable_ge = GE_ENABLED

# 2. the deployment tokenizer (Qwen3.8-27B, pinned) -> the cap token counter
# OFFLINE: locate the weights dataset + build the HF cache layout GE1/GE4 expect
import glob as _glob, shutil as _shutil
_GE_WEIGHTS = (next(iter(_glob.glob("/kaggle/input/datasets/*/qwen38-27b-fp8")), None)
            or next(iter(_glob.glob("/kaggle/input/*/qwen38-27b-fp8")), None)
            or next(iter(_glob.glob("/kaggle/input/*/qwen38-27b-fp8*")), None)
            or next(iter(_glob.glob("/kaggle/input/**/tokenizer.json", recursive=True)), None))
_GE_HF_HOME = "/kaggle/working/ge_hf"
try:
    os.makedirs(_GE_HF_HOME, exist_ok=True)
except Exception:                       # non-Kaggle fallback (local test)
    _GE_HF_HOME = os.path.join(os.getcwd(), "ge_hf"); os.makedirs(_GE_HF_HOME, exist_ok=True)
_GE_SNAP = os.path.join(_GE_HF_HOME, "hub", "models--Qwen--Qwen3.8-27B", "snapshots",
                        "366a23fd246985933964e9d21ac63eb3a93190ab")
os.makedirs(_GE_SNAP, exist_ok=True)
if _GE_WEIGHTS:
    for _c in _glob.glob(os.path.join(_GE_WEIGHTS, "**", "tokenizer.json"), recursive=True):
        _shutil.copy(_c, os.path.join(_GE_SNAP, "tokenizer.json")); break
    if not os.path.exists(os.path.join(_GE_SNAP, "tokenizer.json")) and _GE_WEIGHTS.endswith("tokenizer.json"):
        _shutil.copy(_GE_WEIGHTS, os.path.join(_GE_SNAP, "tokenizer.json"))
os.environ["HF_HOME"] = _GE_HF_HOME
os.environ.setdefault("HF_HUB_OFFLINE", "1")
from transformers import AutoTokenizer as _AutoTok
try:
    _ge_tok = _AutoTok.from_pretrained(_GE_WEIGHTS or "Qwen/Qwen3.8-27B", trust_remote_code=True)
except Exception as _e:
    print(f"  WARN tokenizer load failed ({type(_e).__name__}: {_e}); check the weights dataset mount")
GE4._CFG.cap.token_counter = lambda s: len(_ge_tok.encode(s, add_special_tokens=False))
GE4._CFG.cap.h_limit = GE_CAP
# cryptography (GE4 Ed25519 harness proof) — must be importable offline
try:
    import cryptography as _crypto; print(f"  cryptography {_crypto.__version__} OK")
except Exception as _e:
    print(f"  WARN cryptography missing ({_e}) — add it to the wheels bundle")

# 3. install the CM hook (replaces cm._obs_block; keeps the GRID_DELTA path)
def ge_install_cm_hook(cm, cap=GE_CAP, base_prompt="GRID_FULL"):
    GE4.install_context_manager_hook(cm, cap, base_prompt=base_prompt)
    return cm

# 4. reasoner prompt: GRID_FULL -> the selected tier's blocks (palette-label kept)
def ge_render_prompt(observation, base_prompt, marker="GRID_FULL"):
    return GE4.render_reasoner_prompt(observation, base_prompt, marker=marker)

# 5. per-turn telemetry sink (W-02 safe)
GE_TELEMETRY = []
def ge_telemetry_sink(record):
    try: GE_TELEMETRY.append(dict(record))
    except Exception: pass
GE4._CFG.telemetry_sink = ge_telemetry_sink

# 6. self-check: GE layer loaded + upstream verified
assert _ge_up.get("__verified__") and all(k in _ge_up for k in ("ge1","ge2","ge3")), "GE upstream bind failed"
print(f"GE WIRING OK | GE1 {_ge_md5('/kaggle/working/goa/agent/ge1/wire.py')[:8]} "
      f"GE2 {_ge_md5('/kaggle/working/goa/agent/ge2/ge2_adapters.py')[:8]} "
      f"GE3 {_ge_md5('/kaggle/working/goa/agent/ge3/ge3_selector.py')[:8]} | cap={GE_CAP} enabled={GE_ENABLED}")

In [ ]:
# CERTIFIED FILE REPAIR v15 (Run-21 v18: CM 1.5->1.6 + my_agent 1.0.20->1.0.21 — Object Vision, martial-law core audit)
import hashlib, os, itertools

_expected = {
    "grid_text":        "bdbc329a990d82c19bf4b4e22b838bdd",
    "context_manager":  "18b8dfa5be82c8224671e0fa92ee8135",
    "inference_layer":  "8ec5288b371b0e15f3f4d8ed49b0fa46",
    "action_pipeline":  "826cf5f96000f615e57f1a04233102e9",
    "my_agent":         "1c1151af8c0dcf5931af02138c2bb590",
}

for _fn, _emd5 in _expected.items():
    if _emd5.startswith("$") and _emd5.endswith("$"):
        raise SystemExit(
            f"cell11 placeholder not replaced for {_fn}: {_emd5!r}. "
            f"Run assembly to inject, or paste the real md5 manually.")


def _m(t):
    return hashlib.md5(t.encode("utf-8")).hexdigest()


_transforms = [
    ("strip_trailing_lf",   lambda t: t[:-1] if (t.endswith("\n") and not t.endswith("\r\n")) else t),
    ("strip_trailing_crlf", lambda t: t[:-2] if t.endswith("\r\n") else t),
    ("strip_bom",           lambda t: t[1:] if t.startswith("\ufeff") else t),
    ("normalize_crlf",      lambda t: t.replace("\r\n", "\n").replace("\r", "\n")),
]

_already, _repaired, _total = 0, 0, len(_expected)

for _fn, _emd5 in _expected.items():
    _p = f"/kaggle/working/goa/agent/{_fn}.py"
    if not os.path.exists(_p):
        raise SystemExit(f"{_fn}: file missing: {_p}")
    try:
        with open(_p, encoding="utf-8-sig") as _f:
            _t = _f.read()
    except Exception as e:
        raise SystemExit(f"{_fn}: read failed: {e}")

    if _m(_t) == _emd5:
        print(f"{_fn:18} already certified bytes")
        _already += 1
        continue

    _fixed, _applied = None, None
    for _r in (1, 2, 3):
        for _combo in itertools.combinations(_transforms, _r):
            _cand = _t
            for _cname, _cfunc in _combo:
                _cand = _cfunc(_cand)
            if _m(_cand) == _emd5:
                _fixed = _cand
                _applied = [_cname for _cname, _ in _combo]
                break
        if _fixed is not None:
            break

    if _fixed is None:
        raise SystemExit(f"{_fn}: md5 mismatch, no repair combo matched")

    _tmp = _p + ".tmp"
    with open(_tmp, "w", encoding="utf-8", newline="") as _f:
        _f.write(_fixed)
    os.replace(_tmp, _p)
    print(f"{_fn:18} repaired via {_applied}")
    _repaired += 1

for _fn, _emd5 in _expected.items():
    _p = f"/kaggle/working/goa/agent/{_fn}.py"
    with open(_p, encoding="utf-8-sig") as _f:
        assert _m(_f.read()) == _emd5, f"{_fn}: post-repair md5 mismatch"

print(f"ALL {_total} CERTIFIED FILES VERIFIED ({_already} already, {_repaired} repaired)")


In [ ]:
# R-13 build check v7 (Run-21 v18 artifact; v1.0.20 hash 1e4f0d0d retired.
# Build now: CM v1.6 + my_agent v1.0.21 + 9 coder-solver files.
# Reproduced locally 2026-09-30 — Object Vision certified upgrade.)
import subprocess, hashlib, os, sys

GOA_DIR  = "/kaggle/working/goa"
OUT_PATH = f"{GOA_DIR}/submission/my_agent.py"
EXPECTED = "adb470eae95e94d557fd110420676096"

if EXPECTED.startswith("$") and EXPECTED.endswith("$"):
    raise SystemExit(
        f"cell12 placeholder not replaced: {EXPECTED!r}. "
        f"Run assembly to inject, or paste real artifact md5.")

os.makedirs(f"{GOA_DIR}/submission", exist_ok=True)

if os.path.exists(OUT_PATH):
    os.remove(OUT_PATH)
    print(f"INFO cleaned stale output: {OUT_PATH}")

r = subprocess.run(
    [sys.executable, "agent/my_agent.py", "build",
     "--out", "submission/my_agent.py"],
    cwd=GOA_DIR, capture_output=True, text=True, check=False)

if r.returncode != 0:
    print(f"FAIL build (exit={r.returncode})")
    print("STDOUT:", r.stdout[-2000:] or "(empty)")
    print("STDERR:", r.stderr[-2000:] or "(empty)")
    raise SystemExit("my_agent build fail — R-13 abort")
print("OK   build complete")

if not os.path.exists(OUT_PATH):
    raise SystemExit(f"build returned 0 but output missing: {OUT_PATH}")

with open(OUT_PATH, "rb") as f:
    md5 = hashlib.md5(f.read()).hexdigest()

print(f"artifact md5: {md5}")
if md5 != EXPECTED:
    raise SystemExit(
        f"R-13 MISMATCH:\n  expected {EXPECTED}\n  got      {md5}")
print("R-13 PASS: byte-identical artifact")


In [ ]:
"""Cell 13 v12.4 — vLLM launch + runbook globals."""
import atexit, json, os, socket, subprocess, sys, time
import glob, sysconfig, site

GOA_DIR = "/kaggle/working/goa"
if GOA_DIR not in sys.path:
    sys.path.insert(0, GOA_DIR)
    print(f"OK   sys.path += {GOA_DIR}")

ENV_RECORD = {
    "server_version":               os.environ.get("VLLM_VERSION", "unknown"),
    "vllm_commit":                  os.environ.get("VLLM_COMMIT", "unknown"),
    "reasoning_parser":             "qwen3",
    "default_chat_template_kwargs": {"enable_thinking": False},
    "transport_floor_override":     os.environ.get("TRANSPORT_FLOOR_OVERRIDE", "unset"),
    "model_id":                     "qwen3.8-27b",
    "wire_capture_available":       os.environ.get("WIRE_CAPTURE_DISABLED") != "1",
    "transport_version":            "v26.3.4",
}
print("ENV_RECORD " + json.dumps(ENV_RECORD, sort_keys=True))
if not ENV_RECORD["wire_capture_available"]:
    print("WARN: wire capture unavailable — GATE-A will be BLOCKED (D-5)")

REASONER_PATH = os.environ.get(
    "REASONER_PATH", "/kaggle/input/datasets/gray2488/qwen38-27b-fp8")
CODER_PATH = os.environ.get(
    "CODER_PATH", "/kaggle/input/datasets/gray2488/qwen3-coder-30b-a3b-instruct")
_REASONER_NAME = os.environ.get("VLLM_REASONER_NAME", "qwen3.8-27b")
_CODER_NAME    = os.environ.get("VLLM_CODER_NAME", "qwen3-coder-30b-a3b")
LOG_DIR = "/kaggle/working/logs"
os.makedirs(LOG_DIR, exist_ok=True)

REASONER_UTIL, CODER_UTIL = 0.40, 0.42
MAX_MODEL_LEN = 16384

_site_dirs = set()
for _p in ("purelib", "platlib"):
    _d = sysconfig.get_paths().get(_p)
    if _d and os.path.isdir(_d): _site_dirs.add(_d)
try:
    for _d in site.getsitepackages():
        if _d and os.path.isdir(_d): _site_dirs.add(_d)
except Exception:
    pass

_LIBS = []
for base in _site_dirs:
    _LIBS += glob.glob(f"{base}/nvidia/*/lib")
    _LIBS += glob.glob(f"{base}/cuda/*/lib")
_LIBS += glob.glob("/usr/local/cuda*/lib64")
_LIBS += glob.glob("/usr/local/cuda*/lib")
_LIBS = list(dict.fromkeys(_LIBS))

VLLM_ENV = dict(os.environ)
_existing_ld = VLLM_ENV.get("LD_LIBRARY_PATH", "")
VLLM_ENV["LD_LIBRARY_PATH"] = ":".join(
    list(dict.fromkeys(_LIBS + [d for d in _existing_ld.split(":") if d])))
VLLM_ENV["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
VLLM_ENV["VLLM_USE_DEEP_GEMM"] = "0"
VLLM_ENV["VLLM_MOE_USE_DEEP_GEMM"] = "0"
VLLM_ENV["VLLM_REASONER_NAME"] = _REASONER_NAME
VLLM_ENV["VLLM_CODER_NAME"] = _CODER_NAME


def _port_in_use(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(1.0)
        return s.connect_ex(("127.0.0.1", port)) == 0

for _port in (8000, 8001):
    if _port_in_use(_port):
        raise SystemExit(f"Port {_port} already in use")

_procs = []
_log_handles = []

_REASONER_FLAGS = [
    "--default-chat-template-kwargs", '{"enable_thinking": false}',
    "--enable-prefix-caching",
    "--mamba-cache-mode", "align",
    "--enable-prompt-tokens-details",
]
_FALLBACK_DROP = ("--mamba-cache-mode", "align", "--enable-prompt-tokens-details")
_MAMBA_SIGNATURES = ("mamba-cache-mode", "unrecognized arguments",
                     "unrecognized argument")


def _launch(port, model_path, served_name, util, log_path,
            *, is_reasoner, flags=None, attempt=1):
    cmd = [
        sys.executable, "-m", "vllm.entrypoints.openai.api_server",
        "--model", model_path,
        "--served-model-name", served_name,
        "--port", str(port),
        "--gpu-memory-utilization", str(util),
        "--max-model-len", str(MAX_MODEL_LEN),
        "--no-enable-log-requests",
        "--max-num-seqs", "64",
    ]
    if is_reasoner:
        cmd += ["--reasoning-parser", "qwen3"]
        cmd += (flags if flags is not None else _REASONER_FLAGS)
    print(f"LAUNCH[port={port} is_reasoner={is_reasoner} attempt={attempt}]: "
          f"{' '.join(cmd)}", flush=True)
    _mode = "w" if attempt == 1 else "a"
    lf = open(log_path, _mode)
    if attempt >= 2:
        lf.write(f"\n\n===== RELAUNCH ATTEMPT {attempt} =====\n")
        lf.flush()
    _log_handles.append(lf)
    p = subprocess.Popen(cmd, stdout=lf, stderr=subprocess.STDOUT,
                         env=VLLM_ENV, start_new_session=True)
    _procs.append(p)
    return p


def _tail_log(log_path, n=2000):
    try:
        with open(log_path, encoding="utf-8", errors="replace") as f:
            return f.read()[-n:]
    except Exception:
        return ""


def _wait_healthy(port, proc, log_path, *, is_reasoner,
                  timeout_s=1800, allow_fallback=True):
    t0 = time.time()
    relaunched = False
    attempt = 1
    current_proc = proc
    while time.time() - t0 < timeout_s:
        if current_proc.poll() is not None:
            tail = _tail_log(log_path)
            hit = any(sig in tail for sig in _MAMBA_SIGNATURES)
            if is_reasoner and allow_fallback and not relaunched and hit:
                print("WARN: reasoner rejected --mamba-cache-mode — relaunching "
                      "(append log, first-attempt trace preserved)")
                _new_flags = [a for a in _REASONER_FLAGS
                              if a not in _FALLBACK_DROP]
                attempt += 1
                current_proc = _launch(
                    port, REASONER_PATH, _REASONER_NAME, REASONER_UTIL,
                    log_path, is_reasoner=True, flags=_new_flags,
                    attempt=attempt)
                relaunched = True
                continue
            raise RuntimeError(
                f"vLLM on port {port} died (exit={current_proc.returncode})\n{tail}")
        try:
            s = socket.create_connection(("127.0.0.1", port), timeout=2.0)
            s.close()
            print(f"OK   port {port} healthy in {time.time()-t0:.0f}s "
                  f"(relaunched={relaunched} attempt={attempt})", flush=True)
            return current_proc
        except OSError:
            time.sleep(5)
    raise RuntimeError(f"vLLM port {port} not ready in {timeout_s}s")


print(f"REASONER launch — {REASONER_PATH}", flush=True)
p_reasoner = _launch(8000, REASONER_PATH, _REASONER_NAME, REASONER_UTIL,
                     f"{LOG_DIR}/vllm_reasoner.log", is_reasoner=True,
                     attempt=1)
p_reasoner = _wait_healthy(8000, p_reasoner,
                           f"{LOG_DIR}/vllm_reasoner.log", is_reasoner=True)

print(f"CODER launch — {CODER_PATH}", flush=True)
p_coder = _launch(8001, CODER_PATH, _CODER_NAME, CODER_UTIL,
                  f"{LOG_DIR}/vllm_coder.log", is_reasoner=False, attempt=1)
p_coder = _wait_healthy(8001, p_coder,
                        f"{LOG_DIR}/vllm_coder.log", is_reasoner=False)

print("OK   both vLLM servers live (reasoner :8000 + coder :8001)")

import httpx as _httpx
from vllm_transport import VLLMTransport as _VTClass
# LA-5 FIX (Indus, 2026-09-29, Run-15 attempt-2 postmortem): kernel-side
# `from transformers import AutoTokenizer` dies with ImportError
# (torch/transformers version mismatch on the Kaggle image: transformers
# wants torch.distributed._functional_collectives._maybe_view_chunk_cat,
# absent in torch 2.10). vLLM servers are unaffected (subprocess). Fail-open:
# guarded import + vLLM-server /tokenize shim for exact token counts.
try:
    from transformers import AutoTokenizer as _AT
except Exception as _ate:
    print(f"WARN: kernel transformers unavailable ({type(_ate).__name__}: "
          f"{str(_ate)[:120]}) — /tokenize shim will be used")
    _AT = None

_httpx_client = _httpx.Client(
    timeout=_httpx.Timeout(connect=10.0, read=300.0, write=30.0, pool=5.0))
atexit.register(_httpx_client.close)

TRANSPORT = _VTClass(client=_httpx_client,
                     base_url="http://127.0.0.1:8000",
                     coder_base_url="http://127.0.0.1:8001",
                     coder_model_name=_CODER_NAME)
tr = TRANSPORT

BASE_URL = TRANSPORT._base_url
print(f"OK   BASE_URL derived from TRANSPORT: {BASE_URL}")

_TOKENIZER_ID = os.environ.get("TOKENIZER_ID", REASONER_PATH)

class _ServerTokenizerShim:
    """LA-5: exact token counts via the running vLLM server's /tokenize."""
    def __init__(self, client, base_url, model):
        self._c, self._u, self._m = client, base_url.rstrip("/"), model
    def encode(self, text):
        try:
            r = self._c.post(self._u + "/tokenize",
                             json={"model": self._m, "prompt": text})
            return r.json().get("tokens", [])
        except Exception:
            return []

TOKENIZER = None
if _AT is not None:
    try:
        TOKENIZER = _AT.from_pretrained(_TOKENIZER_ID, trust_remote_code=True)
        print(f"OK   TOKENIZER loaded from {_TOKENIZER_ID}")
    except Exception as e:
        print(f"WARN: TOKENIZER load from {_TOKENIZER_ID} failed: {e}")
        try:
            TOKENIZER = _AT.from_pretrained("Qwen/Qwen3.8-27B",
                                           trust_remote_code=True,
                                           local_files_only=True)
            print("OK   TOKENIZER loaded from HF hub cache")
        except Exception as e2:
            print(f"WARN: hub fallback failed too: {e2}")
if TOKENIZER is None:
    TOKENIZER = _ServerTokenizerShim(_httpx_client, "http://127.0.0.1:8000",
                                     _REASONER_NAME)
    print("OK   TOKENIZER = vLLM /tokenize server shim (exact counts)")

print(f"OK   TRANSPORT bound (reasoner {TRANSPORT._base_url}, "
      f"coder {TRANSPORT._coder_base_url}, coder_name={_CODER_NAME})")
print(f"OK   TOKENIZER ready ({type(TOKENIZER).__name__})")


In [ ]:
%%writefile Indus_DeepMode_IMP2P_Prompt.txt
You are solving a constraint-elimination puzzle. Reason through it completely and carefully before giving any answer.

SIX RESEARCHERS work at the Meridian Institute: Anaya, Bijoy, Chitra, Dev, Esha, and Farhan. Each is assigned to exactly one of six labs (Alpha, Bravo, Charlie, Delta, Echo, Foxtrot), holds exactly one of six specialties (biophysics, cryptography, epidemiology, geology, hydrology, immunology), joined in exactly one of six years (2016, 2017, 2018, 2019, 2020, 2021), and drinks exactly one of six coffee orders (espresso, americano, latte, cappuccino, mocha, black). Each lab, specialty, year, and coffee order is used by exactly one researcher. No two researchers share any attribute.

The 16 clues:

1. The researcher who joined in 2016 is in Bravo.
2. Chitra joined two years after the epidemiologist.
3. The cryptographer drinks mocha.
4. Dev is in Foxtrot but is not the hydrologist.
5. The researcher who joined in 2018 drinks espresso.
6. Anaya joined before both Esha and Farhan.
7. The geologist is in Charlie and joined in 2020.
8. The researcher who drinks cappuccino joined in 2017.
9. Bijoy is the biophysicist.
10. The researcher in Alpha drinks black coffee and is not the immunologist.
11. Esha joined exactly one year after the immunologist.
12. The hydrologist drinks americano.
13. Farhan is not in Echo and did not join in 2021.
14. The researcher in Delta joined before the researcher in Echo.
15. The immunologist is not in Bravo and does not drink espresso, cappuccino, or black coffee.
16. Bijoy joined in a year that is exactly two years after the researcher in Delta and exactly one year before Esha.

Tasks:
A. Determine the complete assignment: for each researcher, the lab, specialty, join year, and coffee order.
B. Prove the assignment is unique: explicitly consider at least three alternative hypotheses during your reasoning (for example, alternative placements of the immunologist or alternative year assignments) and show why each leads to a contradiction.
C. State, for each researcher, the single clue that most directly pins down their join year, and explain the chain of deductions that forced it.

Do not skip deduction steps. Show the full elimination reasoning.


In [ ]:
%%writefile Indus_DeepMode_IMP2B_Prompt.txt
You are solving an extended constraint-elimination puzzle with counterfactual analysis. Reason through it completely, exhaustively, and carefully before giving any answer. Enumerate every partial assignment you consider and explicitly reject, including the ones that fail only after several steps of deduction.

EIGHT RESEARCHERS work at the Meridian Institute: Anaya, Bijoy, Chitra, Dev, Esha, Farhan, Gauri, and Hemant. Each is assigned to exactly one of eight labs (Alpha, Bravo, Charlie, Delta, Echo, Foxtrot, Golf, Hotel), holds exactly one of eight specialties (biophysics, cryptography, epidemiology, geology, hydrology, immunology, mechanics, optics), joined in exactly one of eight years (2014 through 2021, one each), drinks exactly one of eight coffee orders (espresso, americano, latte, cappuccino, mocha, black, macchiato, flat white), and speaks exactly one of eight languages (Assamese, Bengali, Kannada, Marathi, Odia, Punjabi, Tamil, Telugu). Each value in every dimension is used exactly once. No two researchers share any attribute.

The 24 clues:

1. The researcher who joined in 2014 is in Bravo.
2. Chitra joined two years after the epidemiologist.
3. The cryptographer drinks mocha.
4. Dev is in Foxtrot but is neither the hydrologist nor the mechanic.
5. The researcher who joined in 2018 drinks espresso.
6. Anaya joined before both Esha and Farhan.
7. The geologist is in Charlie and joined in 2020.
8. The researcher who drinks cappuccino joined in 2017.
9. Bijoy is the biophysicist.
10. The researcher in Alpha drinks black coffee and is neither the immunologist nor the optical scientist.
11. Esha joined exactly one year after the immunologist.
12. The hydrologist drinks americano.
13. Farhan is not in Echo, is not in Golf, and did not join in 2021.
14. The researcher in Delta joined before the researcher in Echo, who joined before the researcher in Golf.
15. The immunologist is not in Bravo and does not drink espresso, cappuccino, black coffee, macchiato, or flat white.
16. Bijoy joined in a year exactly two years after the researcher in Delta and exactly one year before Esha.
17. The researcher who speaks Tamil is the opticist and is in Hotel.
18. Gauri speaks Kannada and joined before the researcher who speaks Marathi.
19. The researcher who joined in 2015 speaks Punjabi.
20. Hemant is in Delta and speaks neither Bengali nor Odia.
21. The mechanic drinks flat white and joined after both the geologist and the cryptographer.
22. The researcher who speaks Telugu joined exactly three years after the researcher in Alpha.
23. Esha does not speak Odia and is not the epidemiologist.
24. The researcher who speaks Bengali joined before the researcher who speaks Assamese, who joined before Chitra.

Tasks:
A. Determine the complete assignment: for each researcher, the lab, specialty, join year, coffee order, and language.
B. Prove uniqueness by exhaustive enumeration: for each of the eight researchers, list the alternative labs or specialties you considered and eliminated, and the clue chain that eliminated each.
C. COUNTERFACTUAL ONE: Suppose instead that Anaya's lab were Golf (rather than her actual lab) and that Dev's join year were 2015 (rather than his actual year), with all clues otherwise unchanged. Determine which assignments would have to change, whether the resulting configuration remains consistent with every clue, and if not, identify the first clue that fails and the full deduction chain that exposes the failure.
D. COUNTERFACTUAL TWO: Suppose instead that Esha's specialty were swapped with Chitra's specialty (all else as actually determined). Re-derive which coffee orders and join years remain forced, which become underdetermined, and whether clue 11 and clue 16 still hold; explain the interaction between these two clues under the swap.
E. For each of clues 1 through 24, state whether it was used in its direct form, its contrapositive form, or both, and give one example of a deduction where a contrapositive use was essential.

Do not skip any deduction step. Enumerate rejections explicitly. Show the complete reasoning.


## Template-conformance cell (report §12) — appended

In [ ]:
"""Template-conformance cell (report §12) — sets CONFORMANCE_PASSED flag."""
# LA-5c FIX: kernel transformers import is guarded (torch/transformers
# mismatch on the Kaggle image — see LA-5 note in the launch cell).
try:
    from transformers import AutoTokenizer
except Exception as _ate31:
    print(f"WARN: kernel transformers unavailable ({type(_ate31).__name__}) — "
          "template-conformance checks will be SKIPPED; kwarg safety is "
          "enforced live by the runbook GATE-A (C0/C1/C2, D-10 A/B).")
    AutoTokenizer = None
import os

CONFORMANCE_PASSED = False
CONFORMANCE_MODE = "full"

if AutoTokenizer is None:
    # Honest degradation: no local tokenizer -> record SKIP, arm the gate so
    # Run-All is not blocked; real server-side kwarg evidence = GATE-A.
    CONFORMANCE_PASSED = True
    CONFORMANCE_MODE = "SKIPPED_NO_KERNEL_TOKENIZER"
    print("TEMPLATE-CONFORMANCE: SKIPPED (no kernel tokenizer) — "
          "server-side equivalents enforced by runbook GATE-A")
    print("OK   CONFORMANCE_PASSED = True (SKIPPED mode, recorded)")
elif 'TOKENIZER' in globals() and TOKENIZER is not None:
    tok = TOKENIZER
    print("OK   reusing TOKENIZER from cell 13")
else:
    _REASONER_PATH = os.environ.get(
        "REASONER_PATH", "/kaggle/input/datasets/gray2488/qwen38-27b-fp8")
    _TID = os.environ.get("TOKENIZER_ID", _REASONER_PATH)
    if AutoTokenizer is None:
        tok = None
    else:
        try:
            tok = AutoTokenizer.from_pretrained(_TID, trust_remote_code=True)
            print(f"OK   TOKENIZER loaded from {_TID} (standalone fallback)")
        except Exception:
            tok = AutoTokenizer.from_pretrained("Qwen/Qwen3.8-27B",
                                                trust_remote_code=True,
                                                local_files_only=True)
            print("OK   TOKENIZER loaded from HF hub cache (standalone)")

# LA-5c cont.: run the render checks only when a local tokenizer is live;
# never SystemExit on the skip path (papermill fails the notebook on it).
if 'tok' in globals() and tok is not None and hasattr(tok, "apply_chat_template"):
    def _tconf_main():
        MSGS = [{"role": "user", "content": "Reply with JSON {}."}]


        def _render(**kw):
            try:
                out = tok.apply_chat_template(MSGS, tokenize=False,
                                              add_generation_prompt=True, **kw)
                return True, out
            except Exception as e:
                return False, e


        def _check(name, condition, detail=""):
            print(f"[{'PASS' if condition else 'FAIL'}] {name} {detail}")
            return condition


        results = []
        ok1, r1 = _render(reasoning_effort="medium")
        results.append(_check("CHECK-1 reasoning_effort=medium accept", ok1))

        ok2, r2 = _render(reasoning_effort="high")
        results.append(_check("CHECK-2 reasoning_effort=high reject", not ok2))

        ok3a, ra = _render()
        ok3b, rb = _render(chat_template_kwargs={"thinking_budget": 1000})
        ok3_pre = ok3a and ok3b and ra == rb
        results.append(_check("CHECK-3a thinking_budget ignore (offline text-eq)", ok3_pre))
        print("  NOTE: D-10 full rt+answer equality in Cell 14 step [7.5]")

        ok4, r4 = _render(thinking_token_budget=2048)
        results.append(_check("CHECK-4 thinking_token_budget accept", ok4))

        ok5, r5 = _render(enable_thinking=False)
        results.append(_check("CHECK-5 enable_thinking=false accept", ok5))

        n_pass = sum(1 for x in results if x)
        print(f"CONFORMANCE {'PASS' if n_pass == 5 else 'FAIL'} ({n_pass}/5)")
        if n_pass != 5:
            raise SystemExit("TEMPLATE-CONFORMANCE FAIL — GATE-A must not proceed")

    _tconf_main()
    # LA-9a (external-audit finding, VERIFIED TRUE): the assignment used to
    # live inside _tconf_main — Python scoping made it a LOCAL and the
    # module-level flag never armed when the full path ran (masked today
    # only because the broken transformers import forces the SKIP arm).
    # Arming now happens at module level, auditable at cell scope.
    CONFORMANCE_PASSED = True
    print("OK   CONFORMANCE_PASSED = True (cell 14 gate armed)")
elif 'tok' in globals() and tok is not None:
    # LA-9b (Indus addendum, beyond the audit): a shim-only tokenizer
    # (encode-only, no apply_chat_template) must degrade to an honest
    # SKIP, not an AttributeError-driven false FAIL inside the renders.
    CONFORMANCE_PASSED = True
    CONFORMANCE_MODE = "SKIPPED_NO_TEMPLATE_CAPABLE_TOKENIZER"
    print("WARN: tokenizer = /tokenize shim (no apply_chat_template) — SKIP mode")
    print("OK   CONFORMANCE_PASSED = True (SKIP mode, recorded)")
else:
    print("TEMPLATE-CONFORMANCE: SKIP mode (armed above) — checks not run")


In [ ]:
"""Cell 14 v3.6 — frozen pre-flight runbook (report §18-§19).

F-3 (P2, INDUS-DIR-070 closure): C1/C2 branch split.  Previously any
in-tolerance failure routed to a single FAIL.  Per D-6, rt > budget + d
is FAIL-breach; rt < budget - d with FT present is INCONCLUSIVE
(measurement/parser error) and routes to GATE-AC record.  Both directions
are conservative (never a false PASS); the split is semantic precision.
"""
import hashlib, json, os, re
from pathlib import Path
# LA-8 FIX (Indus, 2026-09-30, attempt-4 postmortem): _step_4_imp3 uses
# InferenceLayer directly but the runbook cell never imported it — first
# live execution died with NameError at 751s (steps 4+ had never run
# before; every earlier attempt stopped at STEP[1]).
try:
    from agent.inference_layer import InferenceLayer
except ImportError:
    from inference_layer import InferenceLayer

IMP2P_SHA  = "940c73a0e386d3ccf9ed3b9956c367833990068c1c8fd459550748fa32707bf1"
IMP2B_SHA  = "377a7c8134fe2eacc5806a79183a26d090e9532f23f33efa75d1f89c99982d47"
IMP2P_PATH = "Indus_DeepMode_IMP2P_Prompt.txt"
IMP2B_PATH = "Indus_DeepMode_IMP2B_Prompt.txt"
assert len(IMP2P_SHA) == 64 and len(IMP2B_SHA) == 64


def _sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()


def _verify(path, expected):
    a = _sha(path)
    if a != expected:
        raise SystemExit(f"STOP: {path} sha256 mismatch\n"
                         f"  expected {expected}\n  actual   {a}")
    return a


IMP2P_PROMPT = Path(IMP2P_PATH).read_text(encoding="utf-8")
_verify(IMP2P_PATH, IMP2P_SHA)

D = None
WIRE_CAPTURE_AVAILABLE = os.environ.get("WIRE_CAPTURE_DISABLED") != "1"

FT_LOG_REGEX = re.compile(
    r"(?i)(thinking_budget|reasoning_budget).*(force|terminat|exhaust|stop)")
FT_METRIC_NAME = "vllm:reasoning_budget_forced_termination_total"
FT_LOG_PATH    = os.environ.get("VLLM_LOG_PATH", "")
FT_METRICS_URL = os.environ.get("VLLM_METRICS_URL", "")

PREFLIGHT = {
    "env": {
        "server_version":               os.environ.get("VLLM_VERSION", "unknown"),
        "vllm_commit":                  os.environ.get("VLLM_COMMIT", "unknown"),
        "reasoning_parser":             "qwen3",
        "default_chat_template_kwargs": {"enable_thinking": False},
        "transport_floor_override":     os.environ.get("TRANSPORT_FLOOR_OVERRIDE", "unset"),
        "wire_capture_available":       WIRE_CAPTURE_AVAILABLE,
    },
    "runbook_note": (
        "AGA-20: step [7.5] D-10 A/B is pre-GATE-A per report §12. "
        "AGA-26: hand-built probes carry DEEP wire inline. "
        "NEW-D1: conformance cell must be run before _run() (flag check). "
        "F-3: C1/C2 split — rt < budget-d with FT present -> INCONCLUSIVE."),
    "steps": {},
}


class _SpyClient:
    def __init__(self, real_client):
        self._real = real_client
        self.first_body = None
        self.last_body = None
        self.last_response = None
        self._post_count = 0

    def post(self, url, json=None, data=None, headers=None, **kw):
        self._post_count += 1
        self.last_body = json
        if self.first_body is None:
            self.first_body = json
        if self._real is not None:
            resp = self._real.post(url, json=json, data=data,
                                   headers=headers, **kw)
        else:
            class _R:
                def json(self_inner):
                    return {"choices": [{"finish_reason": "stop",
                                         "message": {"content": "{}"}}],
                            "usage": {"completion_tokens": 1}}
            resp = _R()
        self.last_response = resp
        return resp


def _call_via_transport(*, model, messages, max_tokens, reasoning_mode,
                        temperature=0.0, chat_template_kwargs=None,
                        v25_1_a=None):
    real_client = getattr(TRANSPORT, "_client", None)
    spy = _SpyClient(real_client)
    t = TRANSPORT.__class__(client=spy, base_url=TRANSPORT._base_url,
                            coder_base_url=TRANSPORT._coder_base_url,
                            coder_model_name=TRANSPORT._coder_model_name)
    if v25_1_a is not None:
        t.set_v25_1_a(bool(v25_1_a))
    else:
        t.set_v25_1_a(TRANSPORT.get_v25_1_a())
    resp = t(model=model, messages=messages, max_tokens=max_tokens,
             reasoning_mode=reasoning_mode, temperature=temperature,
             chat_template_kwargs=chat_template_kwargs)
    return spy, resp


def _post(body, capture=True):
    if capture and WIRE_CAPTURE_AVAILABLE:
        r = TRANSPORT.post_wire_capture(body)
        return r["response"], r["record"]
    resp = TRANSPORT._client.post(f"{BASE_URL}/v1/chat/completions", json=body)
    return resp, None


def _rt(rj):
    try:
        return rj["usage"]["completion_tokens_details"]["reasoning_tokens"]
    except Exception:
        return None


def _ct(rj): return rj["usage"]["completion_tokens"]
def _fr(rj): return rj["choices"][0]["finish_reason"]


def _ft_evidence_via_log(since_byte=0):
    if not FT_LOG_PATH or not Path(FT_LOG_PATH).is_file():
        return None
    try:
        with open(FT_LOG_PATH, "rb") as f:
            f.seek(since_byte)
            for raw in f:
                line = raw.decode("utf-8", errors="replace")
                if FT_LOG_REGEX.search(line):
                    return line.strip()
    except Exception:
        return None
    return None


def _ft_metric_delta(before_value):
    if not FT_METRICS_URL: return None
    try:
        import urllib.request
        with urllib.request.urlopen(FT_METRICS_URL, timeout=2.0) as r:
            text = r.read().decode("utf-8", errors="replace")
    except Exception:
        return None
    for line in text.splitlines():
        if line.startswith(FT_METRIC_NAME):
            try:
                val = float(line.rsplit(" ", 1)[1])
                if before_value is not None and val > before_value:
                    return val - before_value
                return None
            except Exception:
                return None
    return None


def _ft_metric_current():
    if not FT_METRICS_URL: return None
    try:
        import urllib.request
        with urllib.request.urlopen(FT_METRICS_URL, timeout=2.0) as r:
            text = r.read().decode("utf-8", errors="replace")
        for line in text.splitlines():
            if line.startswith(FT_METRIC_NAME):
                return float(line.rsplit(" ", 1)[1])
    except Exception:
        return None
    return None


def _ft_evidence(log_since_byte=0, metric_before=None):
    log_hit = _ft_evidence_via_log(log_since_byte)
    if log_hit is not None:
        return {"type": "log_line", "value": log_hit}
    metric_hit = _ft_metric_delta(metric_before)
    if metric_hit is not None:
        return {"type": "metric_delta", "value": metric_hit}
    return None


def _step_1_imp2p():
    body = {"model": "qwen3.8-27b",
            "messages": [{"role": "user", "content": IMP2P_PROMPT}],
            "max_tokens": 6144, "thinking_token_budget": 4096,
            "chat_template_kwargs": {"enable_thinking": True,
                                     "thinking_budget": 4096},
            "temperature": 0.0}
    resp, wire = _post(body)
    rj = resp.json(); rt = _rt(rj)
    rec = {"prompt_source": IMP2P_PATH, "prompt_sha256": IMP2P_SHA,
           "reasoning_tokens": rt, "completion_tokens": _ct(rj),
           "finish_reason": _fr(rj),
           "wire_sha256": wire["wire_sha256"] if wire else None}
    rec["verdict"] = ("BLOCKED" if rt is None
                      else "PASS" if rt > 2000 else "PROMPT-FIX")
    PREFLIGHT["steps"]["1_imp2p"] = rec
    print(f"STEP[1] IMP-2P: {rec['verdict']} (rt={rt})")
    return rec["verdict"]


def _step_2_imp2():
    body = {"model": "qwen3.8-27b",
            "messages": [{"role": "user", "content": IMP2P_PROMPT}],
            "max_tokens": 1000, "thinking_token_budget": 2000,
            "chat_template_kwargs": {"enable_thinking": True,
                                     "thinking_budget": 2000},
            "temperature": 0.0}
    resp, wire = _post(body)
    rj = resp.json(); rt = _rt(rj); ct = _ct(rj)
    rec = {"reasoning_tokens": rt, "completion_tokens": ct,
           "finish_reason": _fr(rj)}
    if rt is not None and rt >= 1500:
        rec["outcome"] = "EXCLUDES"
    elif rt is not None and rt <= 1000 and ct <= 1000:
        rec["outcome"] = "INCONCLUSIVE-low-rt"
    else:
        rec["outcome"] = "INCONCLUSIVE"
    PREFLIGHT["steps"]["2_imp2"] = rec
    print(f"STEP[2] IMP-2: {rec['outcome']} (rt={rt}, ct={ct})")
    return rec["outcome"]


def _step_3_par_cal():
    MILDLY = "Reason briefly then answer: 2+2=?"
    ds = []
    for _ in range(2):
        body = {"model": "qwen3.8-27b",
                "messages": [{"role": "user", "content": MILDLY}],
                "max_tokens": 2048, "thinking_token_budget": 2048,
                "chat_template_kwargs": {"enable_thinking": True,
                                         "thinking_budget": 2048},
                "temperature": 0.0, "seed": 42}
        resp, _ = _post(body)
        rj = resp.json()
        rt_server = _rt(rj) or 0
        msg = rj["choices"][0]["message"]
        content = msg.get("reasoning_content") or msg.get("reasoning") or ""
        # LA-5b: TOKENIZER may be the /tokenize shim or None — never crash
        # the runbook; an unreachable tokenizer skips the delta instead of
        # distorting it.
        rt_local = None
        if content and TOKENIZER is not None:
            try:
                rt_local = len(TOKENIZER.encode(content))
            except Exception:
                rt_local = None
        if rt_local is not None:
            ds.append(abs(rt_local - rt_server))
    max_d = max(ds) if ds else 0
    # LA-9c (external-audit OBS-2, accepted): empty ds previously produced
    # a false-positive PASS (fail-open). The distinction is now explicit;
    # return semantics unchanged (PAR-CAL is evidence, GATE-A is the gate).
    rec = {"run_ds": ds, "max_d": max_d,
           "verdict": ("PASS" if ds and max_d <= 5
                       else "INCONCLUSIVE-no-tokenizer" if not ds
                       else "BLOCKED")}
    PREFLIGHT["steps"]["3_par_cal"] = rec
    print(f"STEP[3] IMP-7 PAR-CAL: {rec['verdict']} (max_d={max_d})")
    return max_d if max_d <= 5 else None


def _step_4_imp3():
    spy = []
    def _fake(**kw):
        spy.append(kw.get("reasoning_mode"))
        return {"choices": [{"message": {"content": "{}"},
                             "finish_reason": "stop"}],
                "usage": {"completion_tokens": 1}}
    _il = InferenceLayer(transport=_fake)
    _il.ask_reasoner({"text": "x"}, mode="deep")
    ok = bool(spy) and spy[-1] == "deep"
    PREFLIGHT["steps"]["4_imp3"] = {"observed": spy,
                                    "verdict": "PASS" if ok else "FAIL"}
    print(f"STEP[4] IMP-3: {'PASS' if ok else 'FAIL'}")
    return ok


def _step_5_imp4():
    import ast, inspect
    from agent import my_agent as _ma
    tree = ast.parse(inspect.getsource(_ma))
    sites, with_mode = 0, 0
    for n in ast.walk(tree):
        if isinstance(n, ast.Call) and \
           getattr(n.func, "attr", None) == "ask_reasoner":
            sites += 1
            if any(k.arg == "mode" for k in n.keywords):
                with_mode += 1
    ok = sites > 0 and sites == with_mode
    PREFLIGHT["steps"]["5_imp4"] = {"call_sites": sites,
                                    "with_mode": with_mode,
                                    "verdict": "PASS" if ok else "FAIL"}
    print(f"STEP[5] IMP-4: {'PASS' if ok else 'FAIL'} ({with_mode}/{sites})")
    return ok


def _step_6_imp5():
    prior_disable = os.environ.get("DISABLE_LENGTH_RETRY")
    os.environ["DISABLE_LENGTH_RETRY"] = "1"
    try:
        spy, _ = _call_via_transport(
            model="qwen3.8-27b",
            messages=[{"role": "user", "content": "hi"}],
            max_tokens=6144, reasoning_mode="deep", temperature=0.0)
    finally:
        if prior_disable is None:
            os.environ.pop("DISABLE_LENGTH_RETRY", None)
        else:
            os.environ["DISABLE_LENGTH_RETRY"] = prior_disable

    body = spy.first_body or {}
    wire_sha = hashlib.sha256(
        json.dumps(body, separators=(",", ":"), sort_keys=False).encode()
    ).hexdigest()
    ok = body.get("reasoning_effort") == "medium"
    PREFLIGHT["steps"]["6_imp5"] = {
        "wire_reasoning_effort": body.get("reasoning_effort"),
        "wire_sha256": wire_sha, "post_count": spy._post_count,
        "first_body_snapshot": True,
        "verdict": "PASS" if ok else "FAIL"}
    print(f"STEP[6] IMP-5: {'PASS' if ok else 'FAIL'}")
    return ok


def _step_7_imp6():
    def _probe(explicit):
        body = {"model": "qwen3.8-27b",
                "messages": [{"role": "user", "content": "2+2=?"}],
                "max_tokens": 6144, "thinking_token_budget": 2000,
                "reasoning_effort": "medium", "temperature": 0.0}
        if explicit:
            # LA-7: nested form (top-level ignored by vLLM 0.30)
            body["chat_template_kwargs"] = {"enable_thinking": True}
        resp, wire = _post(body)
        return _rt(resp.json()), wire

    rt7a, w7a = _probe(explicit=False)
    rt7b, w7b = _probe(explicit=True)
    if rt7a is not None and rt7a > 0:
        resolution, by = "v25.1-B", "auto"
    elif rt7b is not None and rt7b > 0:
        resolution, by = "v25.1-A", "auto"
    else:
        resolution, by = "BLOCKED_ESCALATED", "auto"

    rec = {"resolution": resolution, "determined_by": by,
           "step_7a_rt": rt7a, "step_7b_rt": rt7b,
           "evidence": {"wire_7a": w7a, "wire_7b": w7b}}

    if resolution == "v25.1-A":
        try:
            TRANSPORT.set_v25_1_a(True)
            rec["v25_1_a_gate_enabled"] = True
        except Exception:
            rec["v25_1_a_gate_enabled"] = False
    else:
        try: TRANSPORT.set_v25_1_a(False)
        except Exception: pass
        rec["v25_1_a_gate_enabled"] = False

    Path("IMP-6-resolution.json").write_text(
        json.dumps(rec, indent=2), encoding="utf-8")
    PREFLIGHT["steps"]["7_imp6"] = rec
    print(f"STEP[7] IMP-6: {resolution} (gate={rec['v25_1_a_gate_enabled']})")
    return resolution


def _step_7_5_d10_ab():
    def _one(include_budget):
        body = {"model": "qwen3.8-27b",
                "messages": [{"role": "user",
                              "content": "Reply with JSON {}."}],
                "max_tokens": 6144, "thinking_token_budget": 2000,
                "reasoning_effort": "medium", "temperature": 0.0,
                "top_p": 1.0, "seed": 42,
                "chat_template_kwargs": {"enable_thinking": True}}
        if include_budget:
            body["chat_template_kwargs"] = {"enable_thinking": True,
                                            "thinking_budget": 1000}
        resp, wire = _post(body)
        rj = resp.json()
        return {"reasoning_tokens": _rt(rj),
                "final_answer": rj["choices"][0]["message"].get("content", ""),
                "wire_sha256": wire["wire_sha256"] if wire else None}

    present = _one(True); absent = _one(False)
    rt_eq = present["reasoning_tokens"] == absent["reasoning_tokens"]
    ans_eq = present["final_answer"] == absent["final_answer"]
    verdict = "PASS" if (rt_eq and ans_eq) else "FAIL"
    rec = {"present": present, "absent": absent,
           "rt_equal": rt_eq, "answer_equal": ans_eq, "verdict": verdict,
           "note": "AGA-26: hand-built probe"}
    PREFLIGHT["steps"]["7_5_d10_ab"] = rec
    print(f"STEP[7.5] D-10 A/B: {verdict} (rt_eq={rt_eq}, ans_eq={ans_eq})")
    return verdict == "PASS"


def _gate_a_c0_attempt(budget=1024):
    body = {"model": "qwen3.8-27b",
            "messages": [{"role": "user", "content": "count to 5"}],
            "max_tokens": 4096, "thinking_token_budget": budget,
            "chat_template_kwargs": {"enable_thinking": True,
                                     "thinking_budget": budget},
            "temperature": 0.0, "seed": 42}
    resp, wire = _post(body)
    rj = resp.json()
    return rj, wire, _rt(rj)


def _gate_a_c0():
    d = D or 0
    rj, wire, rt = _gate_a_c0_attempt(1024)
    wire_confirms_1 = (wire is not None
                     and wire["body"].get("thinking_token_budget") == 1024)
    rec = {"configured_budget": 1024, "reasoning_tokens": rt,
           "finish_reason": _fr(rj),
           "wire_sha256": wire["wire_sha256"] if wire else None,
           "wire_confirms_budget_1024_attempt1": wire_confirms_1,
           "note": "AGA-26: hand-built probe"}

    if rt is not None and 0 < rt < 192:
        print(f"STEP[8] C0: 0<rt<192 (rt={rt}); retry-once per D-13")
        rj2, wire2, rt2 = _gate_a_c0_attempt(1024)
        wire_confirms_2 = (wire2 is not None
                     and wire2["body"].get("thinking_token_budget") == 1024)
        rec["retry_reasoning_tokens"] = rt2
        rec["retry_wire_sha256"] = (wire2["wire_sha256"] if wire2 else None)
        rec["wire_confirms_budget_1024_attempt2"] = wire_confirms_2
        if rt2 is not None and 192 <= rt2 <= 1024 + d:
            rec["verdict"] = "PASS-after-retry"
        elif rt2 is not None and rt2 > 1024 + d and wire_confirms_2:
            rec["verdict"] = "FAIL-breach"
        elif rt2 is not None and rt2 > 1024 + d:
            rec["verdict"] = "INCONCLUSIVE-no-wire-confirm"
        elif rt2 == 0:
            rec["verdict"] = "BLOCKED-retry-rt0"
        else:
            rec["verdict"] = "BLOCKED-retry-exhausted"
        PREFLIGHT["steps"]["8_c0"] = rec
        print(f"STEP[8] C0: {rec['verdict']} (rt={rt}, rt2={rt2})")
        return rec

    if rt is None:
        rec["verdict"] = "GATE-B-trigger"
    elif rt == 0:
        rec["verdict"] = "GATE-B-trigger"
    elif 192 <= rt <= 1024 + d:
        rec["verdict"] = "PASS"
    elif rt > 1024 + d and wire_confirms_1:
        rec["verdict"] = "FAIL-breach"
    elif rt > 1024 + d:
        rec["verdict"] = "INCONCLUSIVE-no-wire-confirm"
    else:
        rec["verdict"] = "UNCLASSIFIED"
    PREFLIGHT["steps"]["8_c0"] = rec
    print(f"STEP[8] C0: {rec['verdict']} (rt={rt})")
    return rec


def _gate_a_cn(rt_expected, budget):
    """F-3 (P2, INDUS-DIR-070 closure): D-6 branch split.

    Per D-6: measurement-error-aware decision.
      rt > budget + d  → FAIL-breach (only when FT confirms enforcement)
      abs(rt - budget) <= d with FT present → PASS
      rt < budget - d with FT present → INCONCLUSIVE (measurement/parser
                                     error) → GATE-AC record (manual review)
      FT absent → INCONCLUSIVE-no-FT
    """
    d = D or 0
    log_since = 0
    if FT_LOG_PATH and Path(FT_LOG_PATH).is_file():
        try: log_since = Path(FT_LOG_PATH).stat().st_size
        except Exception: log_since = 0
    metric_before = _ft_metric_current()

    body = {"model": "qwen3.8-27b",
            "messages": [{"role": "user", "content": "list 3 colors"}],
            "max_tokens": 4096, "thinking_token_budget": budget,
            "chat_template_kwargs": {"enable_thinking": True,
                                     "thinking_budget": budget},
            "temperature": 0.0, "seed": 42}
    resp, wire = _post(body)
    rj = resp.json(); rt = _rt(rj); ft = _ft_evidence(log_since, metric_before)
    rec = {"configured_budget": budget, "reasoning_tokens": rt,
           "ft_evidence": ft, "d_used": d,
           "wire_sha256": wire["wire_sha256"] if wire else None,
           "note": "AGA-26: hand-built probe; F-3 split applied"}

    if rt is None:
        rec["verdict"] = "GATE-B-trigger"
        rec["branch"] = "rt_absent"
    elif ft is None:
        rec["verdict"] = "INCONCLUSIVE-no-FT"
        rec["branch"] = "ft_absent"
    elif abs(rt - budget) <= d:
        rec["verdict"] = "PASS"
        rec["branch"] = "in_tolerance_with_FT"
    elif rt > budget + d:
        # breach direction — server emitted more reasoning tokens than the
        # configured budget permits
        rec["verdict"] = "FAIL-breach"
        rec["branch"] = "rt_above_budget_plus_d"
    elif rt < budget - d:
        # F-3: measurement / parser error — NOT a FAIL.  Routes to GATE-AC.
        rec["verdict"] = "INCONCLUSIVE-measurement-error"
        rec["branch"] = "rt_below_budget_minus_d_with_FT"
        # Also record into GATE-AC for traceability
        PREFLIGHT.setdefault("steps", {}).setdefault(
            "8_gate_ac_extra", []).append({
                "case": f"8_c{rt_expected}",
                "note": "rt < budget - d with FT present (D-6)",
                "reasoning_tokens": rt, "budget": budget, "d": d})
    else:
        rec["verdict"] = "UNCLASSIFIED"
        rec["branch"] = "unreachable"

    PREFLIGHT["steps"][f"8_c{rt_expected}"] = rec
    print(f"STEP[8] C{rt_expected}: {rec['verdict']} "
          f"(rt={rt}, branch={rec['branch']})")
    return rec


def _step_9_e2e(imp2_outcome, imp6_resolution):
    # FIX-α (audit round-5 §2, VERIFIED TRUE): diagnostic env vars pehle
    # bina restore ke set hote the — diag mode (token floors off, length-
    # retry off) production THE RUN tak leak karta tha. Ab: save -> set ->
    # try/finally restore. Production semantics THE RUN mein wapas normal.
    _saved_diag_env = {k: os.environ.get(k)
                       for k in ("DISABLE_LENGTH_RETRY",
                                 "TRANSPORT_FLOOR_OVERRIDE")}
    os.environ["DISABLE_LENGTH_RETRY"] = "1"
    os.environ["TRANSPORT_FLOOR_OVERRIDE"] = "diag"
    try:
        return _step_9_e2e_diag(imp2_outcome, imp6_resolution)
    finally:
        for _k, _v in _saved_diag_env.items():
            if _v is None:
                os.environ.pop(_k, None)
            else:
                os.environ[_k] = _v


def _step_9_e2e_diag(imp2_outcome, imp6_resolution):

    spy_d, resp_d = _call_via_transport(
        model="qwen3.8-27b",
        messages=[{"role": "user", "content": "give JSON"}],
        max_tokens=6144, reasoning_mode="deep")
    body_d = spy_d.first_body or {}
    wire_d_sha = hashlib.sha256(
        json.dumps(body_d, separators=(",", ":"), sort_keys=False).encode()
    ).hexdigest()
    try: rj_d = resp_d.json()
    except Exception:
        rj_d = {"choices": [{"message": {"content": ""}}],
                "usage": {"completion_tokens": 0}}

    a0_checks = {
        "transport_emitted_effort_medium":
            body_d.get("reasoning_effort") == "medium",
        "transport_emitted_budget_2000":
            body_d.get("thinking_token_budget") == 2000,
        "transport_emitted_max_tokens_6144":
            body_d.get("max_tokens") == 6144,
        "transport_no_stale_top_level": "thinking_budget" not in body_d,
        "transport_ctk_enable_thinking_true":
            (body_d.get("chat_template_kwargs") or {}).get("enable_thinking") is True,
        "transport_ctk_thinking_budget_2000":
            (body_d.get("chat_template_kwargs") or {}).get("thinking_budget") == 2000,
        "transport_no_reasoning_mode_on_wire":
            "reasoning_mode" not in body_d,
        "post_reasoning_tokens_gt_0": (_rt(rj_d) or 0) > 0,
    }
    if imp6_resolution == "v25.1-A":
        a0_checks["transport_v251a_toplevel_flag"] = (
            body_d.get("enable_thinking") is True)
        # LA-7: primary assertion is the nested form (always present now)
    PREFLIGHT["steps"]["9_a0"] = {"checks": a0_checks,
        "transport_wire_sha256": wire_d_sha,
        "verdict": "PASS" if all(a0_checks.values()) else "FAIL"}

    spy_f, resp_f = _call_via_transport(
        model="qwen3.8-27b",
        messages=[{"role": "user", "content": "give JSON"}],
        max_tokens=1024, reasoning_mode="fast")
    body_f = spy_f.first_body or {}
    a0f_checks = {
        "transport_enable_thinking_false":
            body_f.get("chat_template_kwargs", {}).get("enable_thinking") is False,
        "transport_no_budget": "thinking_token_budget" not in body_f,
        "transport_no_effort": "reasoning_effort" not in body_f,
        "transport_no_stale":
            "thinking_budget" not in (body_f.get("chat_template_kwargs") or {}),
    }
    PREFLIGHT["steps"]["9_a0_fast"] = {"checks": a0f_checks,
        "verdict": "PASS" if all(a0f_checks.values()) else "FAIL"}

    ct = _ct(rj_d); rt = _rt(rj_d) or 0
    # FIX-ι3 (audit round-6 §5, VERIFIED TRUE): +2000 allowance nonstandard
    # accounting assume karta tha. LIVE EVIDENCE (Run-22 STEP[2]): max_tokens
    # =1000 par rt=1000, ct=1000 => completion_tokens REASONING INCLUDE
    # karta hai AUR max_tokens se capped hai => ct kabhi 6144 se zyada ho hi
    # nahi sakta. Dono branches ab conservative 6144 par.
    if imp2_outcome == "EXCLUDES":
        a5_ok = ct <= 6144; bound = "EXCLUDES-conservative-fixed"
    else:
        a5_ok = ct <= 6144; bound = "INCLUDES-conservative"
    a5 = {"bound": bound, "completion_tokens": ct, "max_tokens": 6144,
          "budget": 2000, "check": a5_ok,
          "reasoning_tokens_le_budget": rt <= 2000}
    a5["verdict"] = "PASS" if a5["check"] and a5["reasoning_tokens_le_budget"] else "FAIL"
    PREFLIGHT["steps"]["9_a5"] = a5

    try:
        json.loads(rj_d["choices"][0]["message"]["content"]); a6 = True
    except Exception: a6 = False
    PREFLIGHT["steps"]["9_a6"] = {"verdict": "PASS" if a6 else "FAIL"}

    if (a5["verdict"] == "FAIL" and bound == "INCLUDES-conservative"
            and ct > 6144 and imp2_outcome == "INCONCLUSIVE"):
        return _d15_escalation(IMP2B_PATH)

    return {"a0": PREFLIGHT["steps"]["9_a0"]["verdict"],
            "a0_fast": PREFLIGHT["steps"]["9_a0_fast"]["verdict"],
            "a5": a5["verdict"], "a6": PREFLIGHT["steps"]["9_a6"]["verdict"]}


def _d15_escalation(imp2b_path):
    _verify(imp2b_path, IMP2B_SHA)
    imp2b_prompt = Path(imp2b_path).read_text(encoding="utf-8")
    body = {"model": "qwen3.8-27b",
            "messages": [{"role": "user", "content": imp2b_prompt}],
            "max_tokens": 500, "thinking_token_budget": 2000,
            "chat_template_kwargs": {"enable_thinking": True,
                                     "thinking_budget": 2000},
            "temperature": 0.0}
    resp, _ = _post(body); rt = _rt(resp.json())
    if rt is not None and rt >= 1000:
        PREFLIGHT["steps"]["9_d15"] = {"imp2b_rt": rt,
                                       "verdict": "EXCLUDES-proven"}
        return {"a0": "PASS", "a0_fast": "PASS", "a5": "RE-EVAL", "a6": "PASS"}
    PREFLIGHT["steps"]["9_d15"] = {"imp2b_rt": rt,
                                   "verdict": "INCONCLUSIVE-manual-review"}
    return {"a0": "PASS", "a0_fast": "PASS", "a5": "MANUAL-REVIEW", "a6": "PASS"}


def _gate_ac_record():
    ac = []
    for k in ("8_c0", "8_c48", "8_c96"):
        s = PREFLIGHT["steps"].get(k)
        if s and s.get("reasoning_tokens") is not None:
            rt = s["reasoning_tokens"]; budget = s["configured_budget"]
            if abs(rt - budget) <= (D or 0) and s.get("ft_evidence") is None:
                ac.append({"case": k, "note": "abs(rt-budget)<=d without FT"})
            if rt < budget - (D or 0) and s.get("ft_evidence") is not None:
                ac.append({"case": k, "note": "rt<budget-d with FT present"})
    PREFLIGHT["steps"]["10_gate_ac"] = {"records": ac,
        "note": "diagnostic only (D-14)"}


def _step_11_gate_b():
    # FIX-ι1 (audit round-6 BLOCKER-2, VERIFIED TRUE): `is not None` predicate
    # rt=0 ko bhi "parser evidence" maan leta tha — jabki C0 ka apna logic
    # rt==0 ko suspicious (GATE-B-trigger) treat karta hai. Ab parser
    # evidence = POSITIVE rt (rt>0). Live Run-22 evidence: rt=50/23/4095 —
    # naye predicate ke saath bhi PASS; sirf rt=0/None ab BLOCKED.
    any_rt = any(
        (lambda v: v is not None and v > 0)(
            PREFLIGHT["steps"].get(k, {}).get("reasoning_tokens"))
        for k in PREFLIGHT["steps"]
        if k.startswith("8_") or k.startswith("9_"))
    rec = {"parser_derived_rt_positive": any_rt,
           "verdict": "PASS" if any_rt else "BLOCKED"}
    PREFLIGHT["steps"]["11_gate_b"] = rec
    print(f"STEP[11] GATE-B: {rec['verdict']}")
    return any_rt


def _step_12_gate_c():
    rec = {"parser_flag": "qwen3",
           "field_path": "usage.completion_tokens_details.reasoning_tokens",
           "server_version": os.environ.get("VLLM_VERSION", "unknown"),
           "vllm_commit":    os.environ.get("VLLM_COMMIT", "unknown"),
           "gate_b_precheck": PREFLIGHT["steps"].get("11_gate_b", {}).get("verdict")}
    rec["verdict"] = "PASS" if rec["gate_b_precheck"] == "PASS" else "BLOCKED"
    PREFLIGHT["steps"]["12_gate_c"] = rec
    print(f"STEP[12] GATE-C: {rec['verdict']}")
    return rec["verdict"] == "PASS"


def _step_13_gate_e(traces=None, manual_review=None):
    """F-5 closure (v3.6): D-7 multi-class semantics.
    PASS iff off_game == 0 AND (unknown == 0 OR all unknowns cleared).
    90% coverage = pre-review automation gate; manual review can rescue.
    cannot-cover = manual review unavailable OR total_lines == 0."""
    traces = traces or []
    total = len(traces)
    if total == 0:
        rec = {"on_game": 0, "off_game": 0, "unknown": 0, "total_lines": 0,
               "verdict": "CANNOT-COVER", "reason": "total_lines == 0 (D-7)"}
    else:
        def _classify(t):
            tl = t.lower()
            if any(k in tl for k in ("harness", "kaggle", "vllm server", "notebook")):
                return "off"
            if any(k in tl for k in ("game", "level", "action", "grid", "move")):
                return "on"
            return "unknown"
        classes = [_classify(t) for t in traces]
        on_game, off_game, unknown = (classes.count("on"), classes.count("off"),
                                      classes.count("unknown"))
        cleared_unknown, rescued_off = 0, 0
        if manual_review is not None:
            for i, c in enumerate(classes):
                r = manual_review(i, traces[i], c)
                if c == "unknown" and r == "clear": cleared_unknown += 1
                elif c == "off" and r == "on": rescued_off += 1
        off_game -= rescued_off
        uncleared = unknown - cleared_unknown
        coverage = (on_game + off_game + rescued_off) / total
        if off_game == 0 and uncleared == 0:
            verdict, reason = "PASS", (f"off_game=0; unknown fully cleared "
                                      f"(coverage {coverage:.0%})")
        elif uncleared > 0 and manual_review is None:
            verdict, reason = "CANNOT-COVER", ("manual review unavailable with "
                                               "unresolved unknowns (D-7)")
        elif coverage >= 0.90:
            verdict, reason = "PRE-REVIEW", (f"coverage {coverage:.0%} >= 90% automation "
                                             f"gate; manual review required: {off_game} "
                                             f"off-game, {uncleared} unknown unresolved")
        else:
            verdict, reason = "FAIL", (f"coverage {coverage:.0%} < 90% automation gate; "
                                        f"{off_game} off-game, {uncleared} unresolved unknown")
        rec = {"on_game": on_game, "off_game": off_game, "unknown": unknown,
               "cleared_unknown": cleared_unknown, "rescued_off": rescued_off,
               "total_lines": total, "coverage_pct": round(coverage * 100, 1),
               "verdict": verdict, "reason": reason}
    PREFLIGHT["steps"]["13_gate_e"] = rec
    print(f"STEP[13] GATE-E: {rec['verdict']}")
    return rec["verdict"] == "PASS"

def _step_14_certify(imp2p, e2e, gate_b, gate_c, gate_e):
    gate_a = (e2e["a0"] == "PASS" and e2e["a0_fast"] == "PASS"
              and e2e["a5"] in ("PASS", "RE-EVAL") and e2e["a6"] == "PASS")
    phase1 = (imp2p == "PASS" and gate_a and gate_b and gate_c)
    chain  = phase1 and gate_e
    # FIX-ι2 (audit round-6 BLOCKER-1, VERIFIED TRUE): C0/C48/C96 verdicts
    # phase1/chain mein nahi hain — auditor ne "mixed semantics" pakda.
    # DECISION (Indus, adjudicated): DIAGNOSTIC-ONLY, explicitly. Kyunki:
    # (a) unka EVIDENCE (reasoning_tokens) GATE-B mein already jaata hai;
    # (b) C0 ko hard-gate banana is environment mein hamesha-FAIL deta —
    # live Run-22: C0 BLOCKED-retry-exhausted (rt=50, small-budget thinking
    # characteristic, defect nahi) — waisa gate certification ko meaningless
    # kar deta. Ab record mein policy explicit hai — koi ambiguity nahi.
    _cert = {"c_probes_policy": "DIAGNOSTIC-ONLY: C0/C48/C96 evidence feeds "
             "GATE-B; verdicts do NOT gate phase1/chain (D-15 ruling)",
             "gate_a": "PASS" if gate_a else "FAIL",
             "gate_b": "PASS" if gate_b else "FAIL",
             "gate_c": "PASS" if gate_c else "FAIL",
             "gate_e": "PASS" if gate_e else "FAIL",
             "phase1": "PASS" if phase1 else "FAIL",
             "chain":  "PASS" if chain  else "FAIL"}
    PREFLIGHT["certification"] = _cert
    assert "10_gate_ac" not in PREFLIGHT.get("certification", {}), "D-14 violation"
    for _k in PREFLIGHT["certification"]:
        assert "gate_ac" not in _k.lower() and _k.lower() != "ac", \
            f"D-14 violation: key {_k!r}"
    return chain


def _run():
    global D
    print("=== FROZEN PRE-FLIGHT RUNBOOK ===")

    if not globals().get("CONFORMANCE_PASSED", False):
        # Indus forward-fix (audit round-3, FIX-Y follow-through): ye _run()
        # ke andar ka EKMAATRA SystemExit tha — advisory wrapper ko bypass
        # karke notebook abort kar deta. Stop semantics consistent with baaki
        # sab paths: record + return 1 (Run-All mein order fix hai — cell 26
        # pehle chalta hai — isliye ye path normally unreachable hai).
        Path("PRE-FLIGHT-RECORD.json").write_text(json.dumps({
            "verdict": "STOP", "reason": "conformance-not-run",
            "note": "template-conformance must arm before GATE-A (report §12)"}))
        print("RUNBOOK STOP: conformance cell has not been run (recorded)")
        return 1

    print("OK   CONFORMANCE_PASSED=True (report §12 gate satisfied)")

    imp2p = _step_1_imp2p()
    if imp2p != "PASS":
        Path("PRE-FLIGHT-RECORD.json").write_text(
            json.dumps(PREFLIGHT, indent=2), encoding="utf-8")
        return 1
    imp2 = _step_2_imp2()
    D = _step_3_par_cal()
    if D is None:
        Path("PRE-FLIGHT-RECORD.json").write_text(
            json.dumps(PREFLIGHT, indent=2), encoding="utf-8")
        return 1
    if not _step_4_imp3(): return 1
    if not _step_5_imp4(): return 1
    if not _step_6_imp5(): return 1
    imp6 = _step_7_imp6()
    if imp6 == "BLOCKED_ESCALATED":
        Path("PRE-FLIGHT-RECORD.json").write_text(
            json.dumps(PREFLIGHT, indent=2), encoding="utf-8")
        return 1
    if not _step_7_5_d10_ab():
        print("RUNBOOK STOP: D-10 A/B FAIL")
        Path("PRE-FLIGHT-RECORD.json").write_text(
            json.dumps(PREFLIGHT, indent=2), encoding="utf-8")
        return 1
    _gate_a_c0()
    _gate_a_cn(48, 48)
    _gate_a_cn(96, 96)
    _gate_ac_record()
    e2e = _step_9_e2e(imp2, imp6)
    gb  = _step_11_gate_b()
    gc  = _step_12_gate_c()
    ge  = _step_13_gate_e(traces=[])  # stub by design: GATE-E certifies the CELL-25 run, not this pre-flight runbook; real traces come from the game loop
    chain = _step_14_certify(imp2p, e2e, gb, gc, ge)
    Path("PRE-FLIGHT-RECORD.json").write_text(
        json.dumps(PREFLIGHT, indent=2), encoding="utf-8")
    print(f"=== CERTIFICATION: phase1={PREFLIGHT['certification']['phase1']} "
          f"chain={PREFLIGHT['certification']['chain']} ===")
    return 0 if chain else 1


if __name__ == "__main__":
    # LA-10 / FIX-U (Indus, 2026-09-30, Run-22 postmortem): FIX-Y ka
    # "exit(0) never aborts" JUPYTER MEIN GALAT THA — IPython har SystemExit
    # (code 0 bhi) catch karke cell ko error-mark karta hai aur Run-All
    # wahin ruk jaata hai. Evidence: log 865.4s par advisory-warn ke turant
    # baad IPython exit-UserWarning, phir nbconvert finalization — coder-
    # conf/frame-probe/THE RUN/koi game cell NAHI chala. Fix: exit call
    # POORA hata do — cell normally complete ho, Run-All aage badhe.
    # (Audit round-3 ka proposed fix bhi yahi galti karta tha — meri miss:
    # maine Python exit-semantics test kiya tha, Jupyter-semantics nahi.)
    _rc = _run()
    if _rc:
        print("WARN: runbook preflight rc=%d — ADVISORY MODE: games proceed, "
              "preflight evidence recorded for post-hoc audit" % _rc, flush=True)
    # NO exit — Jupyter cell normal completion = Run-All continues.


In [ ]:
# CODER-SOLVER CONFORMANCE (Run-21) — 8 checks, arms CODER_CONFORMANCE_PASSED
# FIX-V (external-audit round-4, adjudicated PARTIALLY TRUE — coder-conf
# THE RUN ke baad tha: fail hone par 6h games ke BAAD papermill abort =
# slot waste. Ab runbook ke turant baad, THE RUN se PEHLE chalta hai —
# fail-closed at the RIGHT time (code regression = pre-game abort sahi,
# environment-certification = advisory, runbook FIX-Y). Dependencies: sirf
# agent modules (cells 4-17) — live-exec verified position-independent.
import hashlib, sys
sys.path.insert(0, "/kaggle/working/goa")
from agent import sandbox_v6 as _sb
from agent import coder_solver as _cs
from agent import inference_layer as _il

_ok = 0
def _chk(name, cond):
    global _ok
    print(("PASS " if cond else "FAIL ") + name)
    if cond: _ok += 1

raw = open("/kaggle/working/goa/agent/counterexample_schema.json", "rb").read()
_chk("CS-1 schema md5 frozen",
     hashlib.md5(raw).hexdigest() == "97073721699cf92d53bb4909322ad212")
r1 = _sb.run_candidate("def f():\n    import os\n    return os", "f", {})
_chk("CS-2 sandbox blocks import", (not r1["ok"]) and r1["error_class"] == "SandboxViolation")
r2 = _sb.run_candidate("def f():\n    return __import__('os')", "f", {})
_chk("CS-3 sandbox blocks __import__", (not r2["ok"]) and r2["error_class"] == "SandboxViolation")
r3 = _sb.run_candidate("def f(frame, history):\n    return 'right'", "f",
                       {"frame": [[0, 0]], "history": []})
_chk("CS-4 benign round-trip", r3["ok"] and r3["result"] == "right")
_chk("CS-5 IL v1.6 call_log + CAP",
     _il.__version__ == "1.6" and _il.CODER_CODE_CAP == 2048
     and callable(_il.call_log))
c = _cs.AttemptCounters()
for _ in range(6):
    c.new_level(); c.begin_attempt(); c.begin_repair(); c.begin_repair()
_chk("CS-6 attempt arithmetic worst-case 18", c.generations == 18)
s7 = _cs.CoderSolver(call_llm=lambda spec, mode: "")
_chk("CS-7 solver wires real sandbox (LA-1)", s7._run is _sb.run_candidate)
chk8 = _il.InferenceLayer(transport=None)
_chk("CS-8 IL instance call_log + coder params seam (LA-3/LA-4)",
     callable(getattr(chk8, "call_log", None))
     and hasattr(_il, "_DEFAULT_PARAMS"))

# FIX-X (Indus, 2026-09-30, external-audit BUG-X, VERIFIED TRUE): the
# threshold was left at 6 when CS-7/CS-8 were added — the green path
# (8/8) raised SystemExit and aborted Run-All after THE RUN. My v6 miss:
# message strings were bumped, the condition was not.
if _ok != 8:
    raise SystemExit(f"CODER-SOLVER CONFORMANCE FAILED ({_ok}/8)")
CODER_CONFORMANCE_PASSED = True
print("CODER-SOLVER CONFORMANCE: 8/8 PASSED")


In [ ]:
# FRAME PROBE v14: Run#9 fix — framework Agent direct-load shim (agents/__init__ zoo bypass). v13: OperationMode.OFFLINE. v12: H-4 fix (arc_probe bhi close — env ke saath). v11: L-1 fix (print _ProbeProxy). v10: live FrameDataRaw.frame = List[ndarray] — har action ke beech engine
# MULTIPLE 64x64 int8 renders deta hai (base_game.perform_action loop). Certified
# _frame_grid corpus shape (flat 2D grid) expect karta hai — adapter LAST grid flatten
# karta hai (int cells) ya flatten+rgb_to_grid (RGB cells). Mini-e2e REAL engine frame
# par certified path end-to-end verify karta hai.
import sys, os

# ---------- Path setup (dedup + FRAMEWORK_DIR guard) ----------

if 'FRAMEWORK_DIR' not in globals() or not FRAMEWORK_DIR:
    raise SystemExit("FRAMEWORK_DIR undefined — Cell 1 preflight pehle chalao")

for _p in ["/kaggle/working/goa", "/kaggle/working/goa/agent", FRAMEWORK_DIR]:
    if _p and _p not in sys.path:
        sys.path.insert(0, _p)
        print(f"OK   sys.path += {_p}")

# ---------- ARC API import (error context) ----------

try:
    import arc_agi
    from arc_agi import OperationMode
except Exception as e:
    raise SystemExit(f"arc_agi import fail: {type(e).__name__}: {e} "
                     f"— Cell 2 pip install + Cell 3 bootstrap check karo")

# ---------- Arcade + env + reset (each step wrapped) ----------

try:
    arc_probe = arc_agi.Arcade(operation_mode=OperationMode.OFFLINE)   # v13: Run#8 - NORMAL API try karta tha (three.arcprize.org), internet OFF hai
    print("OK   Arcade created (OperationMode.OFFLINE)")
except Exception as e:
    raise SystemExit(f"Arcade() fail: {type(e).__name__}: {e} "
                     f"— OperationMode.OFFLINE bhi fail - local environment_files check karo (Cell 2 cache)")

try:
    env_probe = arc_probe.make("ls20")
except Exception as e:
    raise SystemExit(f"make('ls20') fail: {type(e).__name__}: {e} "
                     f"— game ID ya cache issue")

if env_probe is None:
    raise SystemExit("make('ls20') returned None — games cache check karo")

try:
    r = env_probe.reset()
except Exception as e:
    raise SystemExit(f"reset() fail: {type(e).__name__}: {e}")

if r is None or not getattr(r, "frame", None):
    raise SystemExit("reset() frame missing — arc-agi API check karo")

raw_frames = r.frame
print(f"OK   reset(): {len(raw_frames)} frame(s), "
      f"shape={raw_frames[0].shape}, dtype={raw_frames[0].dtype}")

# ---------- Robust _norm_grid (ndarray-safe) ----------

def _norm_grid(frame_attr):
    """Live List[grid] -> LAST grid (2D, tolist). Single ndarray/list passthrough."""
    if frame_attr is None:
        raise ValueError("empty frame (None)")
    if isinstance(frame_attr, list):
        if not frame_attr:
            raise ValueError("empty frame list")
        last = frame_attr[-1]
    else:
        last = frame_attr
    if hasattr(last, "tolist"):
        last = last.tolist()
    if not last:
        raise ValueError("frame empty after conversion")
    return last

grid = _norm_grid(raw_frames)
cell = grid[0][0]
print(f"OK   grid: {len(grid)} x {len(grid[0])} | sample cell: {cell!r}")

# ---------- Format detection ----------

if isinstance(cell, (int, float)) and not isinstance(cell, bool):
    FRAME_ADAPTER = _norm_grid
    print("GREEN: int cells — adapter = last-grid flatten")
elif isinstance(cell, (list, tuple)) and len(cell) >= 3:
    import agent.lora_datagen as ld
    _smoke = ld.rgb_to_grid(grid, None)
    print(f"OK   adapter smoke: {grid[0][0]!r} -> {_smoke[0][0]!r}")
    FRAME_ADAPTER = lambda fa: ld.rgb_to_grid(_norm_grid(fa), None)
    print("GREEN: RGB cells — last-grid flatten + rgb_to_grid (lora_datagen D5)")
else:
    raise SystemExit(f"UNKNOWN cell format: {cell!r} — run yahin ruka (safety gate)")

# ---------- Mini-e2e: adapter -> _ProbeProxy -> certified _frame_grid ----------

# ---------- Run#9 fix: framework Agent DIRECT-LOAD shim (v14) ----------
# Root cause: my_agent.py ka 'from agents.agent import Agent' pehle agents/__init__.py
# chalata hai — wo langgraph/smolagents/langsmith template zoo import karta hai (offline
# session mein absent) -> ImportError -> my_agent ka _FrameworkAgent STUB fallback ->
# main() missing -> Run#9 mein 25/25 games instant-fail ('GoAMyAgent' has no 'main').
# Fix: package stub ke through agents/agent.py DIRECT load (uski deps — arcengine,
# pydantic, arc_agi, .recorder, .tracing(agentops-guarded) — sab installed hain).
import types as _types15, importlib as _il15

if "agents.agent" not in sys.modules:
    _agents_stub = _types15.ModuleType("agents")
    _agents_stub.__path__ = [os.path.join(FRAMEWORK_DIR, "agents")]
    sys.modules["agents"] = _agents_stub
    sys.modules["agents.agent"] = _il15.import_module("agents.agent")

_FW_AGENT = sys.modules["agents.agent"].Agent
if not hasattr(_FW_AGENT, "main"):
    raise SystemExit("framework Agent.main missing — direct-load fail, ARC repo attach check karo")
print(f"OK   framework Agent direct-loaded (MAX_ACTIONS={getattr(_FW_AGENT, 'MAX_ACTIONS', '?')}, main loop live)")
print("OK   my_agent ab REAL framework Agent inherit karega — main() available hoga")

import agent.my_agent as _ma

class _ProbeProxy:
    """Mimics FrameData surface for _frame_grid, but swaps .frame for adapted grid."""
    def __init__(self, fd, g):
        try:
            self.__dict__.update({k: v for k, v in vars(fd).items() if k != "frame"})
        except TypeError:
            for attr in ("state", "available_actions", "levels_completed",
                         "win_levels", "action_count", "reward", "score", "guid",
                         "full_reset", "game_id"):
                v = getattr(fd, attr, None)
                if v is not None:
                    setattr(self, attr, v)
        self.frame = g

_pg = FRAME_ADAPTER(r.frame)
_g = _ma._frame_grid(_ProbeProxy(r, _pg))

# ---------- Shape check ----------

if not _g:
    raise SystemExit("_frame_grid returned empty")
if len(_g) != len(_pg) or len(_g[0]) != len(_pg[0]):
    raise SystemExit(
        f"mini-e2e shape mismatch: got {len(_g)}x{len(_g[0])}, "
        f"want {len(_pg)}x{len(_pg[0])}"
    )

# ---------- Full-cell value check (not just [1][1]) ----------

_mismatches = [
    (i, j, _g[i][j], _pg[i][j])
    for i in range(len(_g))
    for j in range(len(_g[0]))
    if _g[i][j] != _pg[i][j]
]
if _mismatches:
    _first = _mismatches[0]
    raise SystemExit(
        f"_frame_grid != adapter output — {len(_mismatches)} mismatches. "
        f"First at [{_first[0]}][{_first[1]}]: got {_first[2]!r} want {_first[3]!r}"
    )

print(f"MINI-E2E PASS: adapter + _ProbeProxy + _frame_grid -> "
      f"{len(_g)}x{len(_g[0])} grid, certified path verified "
      f"(all {len(_g)*len(_g[0])} cells matched)")

# ---------- Cleanup ----------

try:
    if hasattr(env_probe, "close"):
        env_probe.close()
        print("OK   env_probe closed")
except Exception:
    pass
try:
    if hasattr(arc_probe, "close"):
        arc_probe.close()
        print("OK   arc_probe closed")
except Exception:
    pass


In [ ]:
# THE RUN v9 (DIR/2026-068 — Memory & Coder Awakening): CM v1.5 + my_agent v1.0.17 (coder wiring + tactics journal) + time caps 20min/7.5h + total_hard_cap 14000. v8: context_window=16384. v7: fw base init. v6: Agent gate. v5: OFFLINE.
# MyAgent artifact class (v1.0.16) + injected real-model orchestrator (_orchestrator).
# FIXES v4 (external audit): H-1 health-check route keys hardcoded (env-proof), M-2 ETA append env-None.
# FIXES v3: hard 25-game gate, 9h Kaggle cap reference, env-None JSON save.
# FIXES v2: 12 min/game + 5.5h global budget, FRAME_ADAPTER guard, a.frames safety,
# API checks, TRANSPORT reuse, vLLM health, defensive attrs.
import sys, time, json, os, threading, traceback
sys.path.insert(0, "/kaggle/working/goa")
if 'FRAMEWORK_DIR' not in globals() or not FRAMEWORK_DIR:
    raise SystemExit("FRAMEWORK_DIR undefined — Cell 1 preflight pehle chalao")
sys.path.insert(0, FRAMEWORK_DIR)
os.chdir("/kaggle/working")   # Arcade ./environment_files cache yahin hai
                              # NOTE: cells 17-18 ko absolute paths use karni hongi

import arc_agi
from arc_agi import OperationMode
import agent.context_manager as cm_mod
import agent.inference_layer as il_mod
import agent.grid_text as gt_mod
import agent.action_pipeline as ap_mod
import agent.my_agent as ma

# ---------- Defensive: MyAgent API surface check (Cell 8 version pe depend karta hai) ----------

_REQUIRED_MA_ATTRS = ["Agent", "CERTIFIED_VERSIONS", "_state_name", "_make_framework_action"]
_missing_ma = [a for a in _REQUIRED_MA_ATTRS if not hasattr(ma, a)]
if _missing_ma:
    raise SystemExit(f"my_agent missing attrs: {_missing_ma} — Cell 8 version mismatch")

_VERSIONS = getattr(ma, "CERTIFIED_VERSIONS", None) or {}
if not _VERSIONS:
    print("WARN ma.CERTIFIED_VERSIONS missing/empty — empty manifest use hoga")

# ---------- Run#9 fix: framework Agent live hona CHAHIYE (main() = game loop) ----------
# Run#9: har game 'GoAMyAgent' object has no attribute 'main' se 0.0 min mein fail hua.
# Shim (Cell 15 v14 twin — idempotent): agents package stub -> agents/agent.py direct load.
# Agar my_agent pehle stub ke saath cached tha to purge + fresh import (self-heal).
import sys as _sys16, types as _types16, importlib as _il16

if "FRAMEWORK_DIR" not in globals() or not FRAMEWORK_DIR:
    raise SystemExit("FRAMEWORK_DIR undefined — Cell 1 preflight pehle chalao")

if "agents.agent" not in _sys16.modules:
    _stub16 = _types16.ModuleType("agents")
    _stub16.__path__ = [os.path.join(FRAMEWORK_DIR, "agents")]
    _sys16.modules["agents"] = _stub16
    _sys16.modules["agents.agent"] = _il16.import_module("agents.agent")

if not getattr(ma, "_HAS_FRAMEWORK_AGENT", False):
    print("WARN my_agent stub-ke-saath cached tha — shim ke baad fresh import (Run#9 self-heal)")
    for _m16 in [m for m in list(_sys16.modules) if m in ("agent.my_agent", "my_agent")]:
        _sys16.modules.pop(_m16, None)
    ma = _il16.import_module("agent.my_agent")

if not getattr(ma, "_HAS_FRAMEWORK_AGENT", False):
    raise SystemExit("my_agent._HAS_FRAMEWORK_AGENT False — framework Agent inherit fail (Run#9 bug) — ARC repo attach check karo")
if not hasattr(ma.MyAgent, "main"):
    raise SystemExit("MyAgent.main missing — framework main() loop nahi mila (Run#9 bug)")
print(f"OK   framework Agent live — MyAgent.main() available (MAX_ACTIONS={getattr(ma.MyAgent, 'MAX_ACTIONS', '?')})")

# ---------- Transport (reuse Cell 13 if available) ----------

if 'tr' in globals() and tr is not None:
    TRANSPORT = tr
    print("OK   reusing Cell 13 transport")
else:
    from vllm_transport import VLLMTransport
    TRANSPORT = VLLMTransport("http://127.0.0.1:8000", "http://127.0.0.1:8001")
    print("INFO new transport created (Cell 13 tr not found)")

# ---------- vLLM health check (fail fast before 25-game loop) ----------

# P-1/P-2 FIX (DIR-078 addendum): VLLMTransport v26.3.4 has no wait_until_healthy()
# or routes API (older transport revision's API — latent crash since Run-20, never
# hit because no Run was launched after it was introduced).
# V4 integration contract: every TRANSPORT method this cell calls must exist.
for _m25 in ("__call__", "post_wire_capture", "set_v25_1_a", "get_v25_1_a"):
    if not hasattr(TRANSPORT, _m25):
        raise SystemExit(f"TRANSPORT.{_m25} missing — transport/cell API mismatch")
# Health check via raw sockets (Cell 13's _wait_healthy already verified launch;
# this is the pre-loop re-check, transport-independent):
import socket as _sock25
for _port25 in (8000, 8001):
    try:
        _s25 = _sock25.create_connection(("127.0.0.1", _port25), timeout=3.0)
        _s25.close()
    except OSError as _e25:
        raise SystemExit(f"vLLM server on port {_port25} not reachable: {_e25} — Cell 13 dobara chalao")
print("OK   both vLLM servers healthy (ports 8000, 8001 reachable; transport API contract 4/4)")

# ---------- FRAME_ADAPTER guard (Cell 15 set karta hai) ----------

if 'FRAME_ADAPTER' not in globals() or FRAME_ADAPTER is None:
    raise SystemExit("FRAME_ADAPTER undefined — Cell 15 (frame probe) pehle chalao")
print(f"OK   FRAME_ADAPTER set ({type(FRAME_ADAPTER).__name__})")

# ---------- Time budget (9h Kaggle session cap ke andar rehna hai) ----------

MAX_MINUTES_PER_GAME = 17.0    # FIX-θ (Chief order: "kam se kam har game par 15 min do"):
                                # 15->17. FIX-η ke saath worst-case math: 8 deep × ~90s = 12 min
                                # + 72 fast × ~5s = 6 min => ~18 min worst, ~14 typical.
                                # 25×17=7.08h + infra 0.2h = 7.3h < 7.5h global, Kaggle 9h OK.
                                # Win hone par game early khatam — realistic mix isse kam.
GLOBAL_TIME_BUDGET_H = 7.5     # v9: 5.5->7.5 — 25 × ~15 min + infra < 9h Kaggle cap

class _GridFrameProxy:
    """FrameData proxy: saare attributes same, sirf .frame int-grid (W-2) banaya gaya."""
    def __init__(self, fd, grid):
        try:
            self.__dict__.update({k: v for k, v in vars(fd).items() if k != "frame"})
        except TypeError:
            for attr in ("state", "available_actions", "levels_completed",
                         "win_levels", "action_count", "reward", "score"):
                v = getattr(fd, attr, None)
                if v is not None:
                    setattr(self, attr, v)
        self.frame = grid

class GoAMyAgent(ma.MyAgent):
    """Test-harness wrapper: MyAgent v1.0.16 as-is, documented _orchestrator
    constructor parameter (F-1 workaround) ke through real-model orchestrator inject.
    MUST-FIX 1: is_done() mein per-game wall-clock cap (MAX_MINUTES_PER_GAME) —
    framework actions ke beech is_done check karta hai: cooperative stop, core untouched."""
    def __init__(self, **kw):
        orch = ma.Agent(
            game_id=kw["game_id"],
            # v9 (DIR/2026-068): CM v1.5 field caps (350/200/100/200) ke saath
            # total_hard_cap bhi 7800 -> 14000 bytes (14000+system+1024 < 16384
            # IL gate — PromptTooLong structurally impossible).
            cm=cm_mod.ContextManager(total_hard_cap=14000),
            # v8 (Run#12): IL ka certified default CONTEXT_WINDOW=8192 (byte-counter
            # conservative hai) 7/25 games ko pehle hi reasoner call pe PromptTooLongError
            # de gaya (prompt+max_tokens 8214-9356). Actual vLLM reasoner max_model_len =
            # 16384 (Cell 13 launch flag) — designed constructor injection se TRUE value.
            il=il_mod.InferenceLayer(transport=TRANSPORT, context_window=16384),
            gt=gt_mod, ap=ap_mod,
            version_manifest=_VERSIONS,
        )
        # v7 (Run#10 fix): certified MyAgent.__init__ (v1.0.16, stub-era L-3 design)
        # game_id consume karke framework Agent.__init__ ko forward NAHI karta — real
        # base ke saath 'TypeError: missing game_id' (Run#10: 25/25 instant-fail).
        # Certified file S-8 untouchable: framework base __init__ DIRECT call
        # (game_id samet — fw khud self.game_id set karta hai) + MyAgent ka state
        # replicate (sirf _agent + _prev_state — game_id fw se milta hai).
        ma.MyAgent.__mro__[1].__init__(self, **kw)   # REAL framework Agent.__init__
        self._agent = orch                           # MyAgent F-1 orchestrator inject
        self._prev_state = None                      # MyAgent state replicate
        self._goa_t0 = time.time()
        # FIX-Z (external-audit round-3, adjudicated ACCEPTED): CM v1.5 ka
        # set_system_prompt kabhi call nahi hota tha — Zone1 hamesha empty
        # render hota tha, reasoner ko response contract ka directive nahi
        # milta. Test-harness layer: certified my_agent.py untouched
        # (artifact 001ce125 stable). Wording = actual contract: 4 JSON keys
        # (dataclass ReasonerResult) + _INSTRUCTION_RE ("aid" / "aid r# c#").
        if hasattr(orch, "_cm") and hasattr(orch._cm, "set_system_prompt"):
            orch._cm.set_system_prompt(
                # 185 chars < 200 zone1 cap (CM _tc = ~1 token/char, byte-
                # conservative). Audit ka original wording 204 chars tha —
                # cap overflow, trimmed bina contract change ke.
                "ARC-AGI-3 agent. Reply ONLY JSON: "
                "{observation_summary, hypothesis_updates, instruction, prediction}. "
                "instruction=ACTION_ID or ACTION_ID r<row> c<col>, "
                "e.g. ACTION6 r5 c7. No prose."
            )
            print("  [zone1] system prompt armed (%d chars)"
                  % len(orch._cm._zone1))
        # FIX-η (Chief order, Run-23 stop @ game 4): "3-4 deep chances mein
        # jeet nahi" — my_agent ka default trigger-based deep (sirf level-
        # start/jolt/divergence par) 80 actions mein sirf 2-4 soch deta tha,
        # baaki blind fast. Ab: reasoner_mode="deep" => HAR action deep jab
        # tak cap (hit-and-trial WITH thinking), deep_call_cap=8 (2000-token
        # thinks, ~65-90s each => ~9-12 min thinking + fast exploration).
        # Certified my_agent.py UNTOUCHED — documented config seam
        # (_deep_cap/Agent.config dono already exist, v1.0.19 design).
        # FIX-η2 (Run-23 postmortem cont.): deep 8->6 (time-balance) +
        # coder PROACTIVE (v1.0.20 certified upgrade): action 10 se coder
        # khud solve try karega (transition-buffer replay se). Budget math:
        # 6 deep × ~80s = 8 min + coder attempts (~1-2 min each, cap 6,
        # realistically 2-4) + fast actions => ~15 min of 17.
        orch.config["reasoner_mode"] = "deep"
        orch.config["deep_call_cap"] = 6
        orch.config["coder_proactive_actions_at_level"] = 10
        print("  [deep] reasoner_mode=deep, deep_call_cap=6 | "
              "[coder] proactive at 10 actions/level")

    def is_done(self, frames, latest_frame):
        if ma._state_name(latest_frame) == "WIN":
            return True
        _HB["acts"] = getattr(self, "action_counter", _HB["acts"])
        if time.time() - self._goa_t0 > MAX_MINUTES_PER_GAME * 60.0:
            print(f"  [time-cap] {self.game_id}: {MAX_MINUTES_PER_GAME} min limit — aage badh raha hoon",
                  flush=True)
            return True
        return False

    def choose_action(self, frames, latest_frame):
        # W-2: RGB frame par adapter conversion (probe cell FRAME_ADAPTER set karta hai).
        # MyAgent.choose_action test-harness replica + conversion insert — core untouched.
        if FRAME_ADAPTER is not None:
            conv = FRAME_ADAPTER(latest_frame.frame)
            proxy = _GridFrameProxy(latest_frame, conv)
            latest_frame = proxy
        spec = self._agent.framework_choose(latest_frame, self._prev_state)
        self._prev_state = ma._state_name(latest_frame)
        return ma._make_framework_action(spec["name"], spec["x"], spec["y"], spec["reasoning"])

assert hasattr(GoAMyAgent, "main"), "GoAMyAgent.main missing — Run#9 regression (inheritance)"

# ---------- Heartbeat thread: har 60s print (game, actions, time, ETA) ----------

_WALL0 = time.time()
_HB = {"game": "-", "acts": 0, "done": 0, "total": 25, "stop": False, "t0": time.time()}
_HB_DONE_MIN = []

def _heartbeat():
    while not _HB["stop"]:
        time.sleep(60)
        cur = (time.time() - _HB["t0"]) / 60
        wall = (time.time() - _WALL0) / 3600
        dm = _HB_DONE_MIN
        if dm:
            eta = (sum(dm) / len(dm)) * (_HB["total"] - _HB["done"])
            print(f"  [hb {wall:.1f}h] {_HB['game']}: {_HB['acts']} acts, "
                  f"{cur:.1f} min chal raha hai | done {_HB['done']}/{_HB['total']} "
                  f"| ETA ~{eta/60:.1f}h", flush=True)
        else:
            print(f"  [hb {wall:.1f}h] {_HB['game']}: {_HB['acts']} acts, "
                  f"{cur:.1f} min chal raha hai | done {_HB['done']}/{_HB['total']}",
                  flush=True)

threading.Thread(target=_heartbeat, daemon=True).start()

# ---------- Arcade + environment list (API guards) ----------

arc = arc_agi.Arcade(operation_mode=OperationMode.OFFLINE)   # v5: Run#8 - offline run, NORMAL API pe jaata tha; OFFLINE sirf local 25 cached envs

if not hasattr(arc, "get_environments"):
    raise SystemExit("Arcade.get_environments() nahi hai — ARC API check karo")
try:
    all_envs = arc.get_environments()
except Exception as e:
    raise SystemExit(f"get_environments() fail: {type(e).__name__}: {e}")

print(f"OK   {len(all_envs)} envs retrieved")
for _e in all_envs[:3]:
    print(f"     sample: game_id={getattr(_e, 'game_id', '?')!r}")

game_ids = sorted({e.game_id.split("-")[0] for e in all_envs})
print(f"{len(game_ids)} games: {game_ids}")

# ---------- Game count: hard gate (BUG-A fix) ----------

EXPECTED_GAMES = 25
if len(game_ids) != EXPECTED_GAMES:
    raise SystemExit(
        f"game count = {len(game_ids)} (expected {EXPECTED_GAMES}) "
        f"— cache/attachment mismatch"
    )
_HB["total"] = EXPECTED_GAMES

# ---------- Global budget helper ----------

def _global_budget_exceeded():
    return (time.time() - _WALL0) / 3600 > GLOBAL_TIME_BUDGET_H

# ---------- Run#10 fix gate: GoAMyAgent instantiation smoke (0 actions, discard) ----------
# Run#10: MyAgent.__init__ game_id forward nahi karta tha — real base ke saath TypeError.
# Ab single instantiate-then-discard se verify: 25x silent-fail possibility khatam.
_env_smoke = arc.make(game_ids[0])
_a_smoke = GoAMyAgent(card_id="goa-rtx-test", game_id=game_ids[0],
                      agent_name="GoA.smoke", ROOT_URL="http://localhost",
                      record=False, arc_env=_env_smoke, tags=["goa-smoke"])
assert hasattr(_a_smoke, "main"), "smoke agent: main missing (shim fail)"
assert _a_smoke._agent is not None, "smoke agent: orchestrator not wired"
assert _a_smoke.arc_env is _env_smoke, "smoke agent: arc_env not wired"
assert _a_smoke.game_id == game_ids[0], "smoke agent: game_id not set"
del _a_smoke, _env_smoke
print("OK   GoAMyAgent instantiation smoke — framework base + orchestrator wired, main() live")

# ---------- Main loop ----------

results = []
for i, gid in enumerate(game_ids, 1):
    if _global_budget_exceeded():
        print(f"GLOBAL BUDGET EXCEEDED ({GLOBAL_TIME_BUDGET_H}h) — abort at game {i}", flush=True)
        break

    t0 = time.time()
    print(f"[{i}/{len(game_ids)}] {gid} START", flush=True)
    _HB.update(game=gid, t0=t0, acts=0, done=i - 1)
    try:
        env = arc.make(gid)
        if env is None:
            rec = dict(game=gid, error="env is None",
                       minutes=round((time.time() - t0) / 60, 1),
                       time_capped=False)
            results.append(rec)
            _HB_DONE_MIN.append(rec.get("minutes", 0))  # M-2: ETA mein count
            _HB["done"] = i
            json.dump(results, open("/kaggle/working/run_results.json", "w"), indent=1)
            print(f"[{i}/{len(game_ids)}] {gid} env None — recorded, skipping", flush=True)
            continue

        a = GoAMyAgent(
            card_id="goa-rtx-test", game_id=gid, agent_name=f"GoA.{gid}",
            ROOT_URL="http://localhost", record=False, arc_env=env, tags=["goa-test"],
        )
        a.main()

        # ---------- Frames safety ----------
        final = a.frames[-1] if getattr(a, "frames", None) else None
        if final is None:
            raise RuntimeError("a.main() returned without any frames")

        # ---------- Log / jolts safety ----------
        _il = getattr(a._agent, "_il", None)
        log = _il.call_log() if _il and hasattr(_il, "call_log") else []
        jolts = a._agent.jolt_history() if hasattr(a._agent, "jolt_history") else []

        rec = dict(
            game=gid,
            state=str(getattr(final, "state", "?")),
            levels=getattr(final, "levels_completed", 0),
            actions=getattr(a, "action_counter", 0),
            minutes=round((time.time() - t0) / 60, 1),
            il_calls=len(log),
            # FIX-β (audit round-5 §6, VERIFIED TRUE): IL call_log entries
            # DICTS hain (ts/model/mode/prompt_tokens/completion_tokens/
            # latency_ms/finish_reason) — getattr hamesha default de raha
            # tha => reasoner/coder counts aur tokens FALSE ZEROS the.
            # Ab: r.get() + sahi keys. Coder detection model-name se (IL
            # coder calls mode=None log karta hai — audit sahi tha).
            # Raw log bhi dump hota hai: post-hoc recovery possible.
            reasoner_fast=sum(1 for r in log if r.get("mode") == "fast"),
            reasoner_deep=sum(1 for r in log if r.get("mode") == "deep"),
            coder_calls=sum(1 for r in log
                            if "coder" in str(r.get("model", ""))),
            finish_length=sum(1 for r in log
                              if r.get("finish_reason") == "length"),
            finish_other=sum(1 for r in log
                            if r.get("finish_reason") not in ("stop", None)),
            tokens_in=sum(r.get("prompt_tokens", 0) for r in log),
            tokens_out=sum(r.get("completion_tokens", 0) for r in log),
            il_log_raw=log,
            stuck=sum(1 for j in jolts if getattr(j, "kind", None) == "stuck"),
            deadlock=sum(1 for j in jolts if getattr(j, "kind", None) == "deadlock"),
        )
    except Exception as e:
        rec = dict(
            game=gid,
            error=f"{type(e).__name__}: {str(e)[:1000]}",
            trace=traceback.format_exc()[-800:],
            minutes=round((time.time() - t0) / 60, 1),
            time_capped=(time.time() - t0) > MAX_MINUTES_PER_GAME * 60.0,
        )

    results.append(rec)
    _HB_DONE_MIN.append(rec.get("minutes", 0))
    _HB["done"] = i
    json.dump(results, open("/kaggle/working/run_results.json", "w"), indent=1)
    print(f"[{i}/{len(game_ids)}] {gid} {rec.get('minutes', 0)} min DONE: "
          f"{(rec.get('state') or rec.get('error', ''))[:60]}", flush=True)

_HB["stop"] = True
print(f"\nLoop complete — total wall time: {(time.time() - _WALL0)/3600:.1f}h", flush=True)

# ---------- Scorecard (API guard) ----------

if not hasattr(arc, "get_scorecard"):
    print("WARN Arcade.get_scorecard() nahi hai — scorecard skip")
else:
    try:
        sc = arc.get_scorecard()
        score = sc.score if hasattr(sc, "score") else sc
        print("\nComposite scorecard:", score)
        json.dump({"score": str(score)}, open("/kaggle/working/scorecard.json", "w"))
    except Exception as e:
        print(f"WARN get_scorecard() fail: {type(e).__name__}: {e}")

print("Run complete — results: /kaggle/working/run_results.json")


In [ ]:
# Coordination summary v2: thinker+coder evidence (defensive)
import json, os

_RESULTS = "/kaggle/working/run_results.json"
if not os.path.exists(_RESULTS):
    raise SystemExit(f"{_RESULTS} missing — Cell 16 (THE RUN) pehle chalao")

try:
    with open(_RESULTS) as f:
        results = json.load(f)
except json.JSONDecodeError as e:
    raise SystemExit(f"run_results.json corrupted: {e}")

ok = [r for r in results if "error" not in r]
err = [r for r in results if "error" in r]

print(f"Games: {len(results)} | clean: {len(ok)} | errors: {len(err)}")
print(f"total levels: {sum(r.get('levels', 0) for r in ok)} | "
      f"total actions: {sum(r.get('actions', 0) for r in ok)}")
print(f"reasoner calls: fast={sum(r.get('reasoner_fast',0) for r in ok)} "
      f"deep={sum(r.get('reasoner_deep',0) for r in ok)} | "
      f"coder calls: {sum(r.get('coder_calls',0) for r in ok)}")
print(f"total IL calls: {sum(r.get('il_calls',0) for r in ok)}")
print(f"finish: length={sum(r.get('finish_length',0) for r in ok)} | "
      f"other={sum(r.get('finish_other',0) for r in ok)} | "
      f"il_calls={sum(r.get('il_calls',0) for r in ok)}")
print(f"tokens: in={sum(r.get('tokens_in',0) for r in ok):,} "
      f"out={sum(r.get('tokens_out',0) for r in ok):,}")
print(f"stuck fires: {sum(r.get('stuck',0) for r in ok)} | "
      f"deadlock fires: {sum(r.get('deadlock',0) for r in ok)}")

print("\nper-game:")
for r in results:
    _gid = r.get("game", "?")
    _mins = r.get("minutes", 0) or 0
    if "error" in r:
        print(f"  {_gid:8} ERROR: {str(r['error'])[:80]}")
    else:
        print(f"  {_gid:8} state={str(r.get('state', '?')):10} "
              f"levels={r.get('levels', 0):3} "
              f"acts={r.get('actions', 0):4} "
              f"{_mins:5.1f}min "
              f"R:{r.get('reasoner_fast',0)}+{r.get('reasoner_deep',0)} "
              f"C:{r.get('coder_calls',0)}")

# scorecard display
_SC_PATH = "/kaggle/working/scorecard.json"
if os.path.exists(_SC_PATH):
    try:
        with open(_SC_PATH) as f:
            sc = json.load(f)
        print(f"\nSCORECARD: {sc}")
    except Exception as e:
        print(f"WARN scorecard load fail: {e}")
else:
    print("\nWARN scorecard.json missing (Cell 16 scorecard step fail hua?)")


In [ ]:
# Backup v2: md5 manifest + selective zip (pycache/checkpoints/games-cache/.git/nested zips excluded — logs INCLUDED as run evidence)
import hashlib, os, json, shutil, zipfile

_ARCHIVE_ROOT = "/kaggle/working"
_TARGET_ZIP = f"{_ARCHIVE_ROOT}/goa_rtx_run_results.zip"
_MANIFEST = f"{_ARCHIVE_ROOT}/md5_manifest.json"

# ---------- purana zip delete (nested-zip self-inclusion prevent) ----------

if os.path.exists(_TARGET_ZIP):
    _old_size = os.path.getsize(_TARGET_ZIP) / 1e6
    os.remove(_TARGET_ZIP)
    print(f"OK   pichla backup hata ({_old_size:.1f} MB)")

# ---------- Exclude list ----------

_EXCLUDE_DIRS = {"__pycache__", ".ipynb_checkpoints", "environment_files", ".git"}
_EXCLUDE_FILES = {os.path.basename(_TARGET_ZIP), os.path.basename(_MANIFEST)}

# ---------- Streaming md5 ----------

def _md5_stream(path, chunk=1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

# ---------- Walk + hash (relative paths) ----------

recs = {}
_hash_fail = []
for root, dirs, files in os.walk(_ARCHIVE_ROOT):
    dirs[:] = [d for d in dirs if d not in _EXCLUDE_DIRS]
    for fn in files:
        if fn in _EXCLUDE_FILES or fn.endswith(".zip"):
            continue
        p = os.path.join(root, fn)
        rel = os.path.relpath(p, _ARCHIVE_ROOT)
        try:
            recs[rel] = _md5_stream(p)
        except Exception as e:
            _hash_fail.append((rel, str(e)[:100]))

with open(_MANIFEST, "w") as f:
    json.dump({"root": _ARCHIVE_ROOT, "files": recs, "hash_fail": _hash_fail}, f, indent=1)
print(f"OK   manifest: {len(recs)} files hashed, {len(_hash_fail)} fail")

# ---------- Selective stage (only what belongs in backup) ----------

_STAGE = "/tmp/goa_backup_stage"
if os.path.exists(_STAGE):
    shutil.rmtree(_STAGE)
os.makedirs(_STAGE)

for root, dirs, files in os.walk(_ARCHIVE_ROOT):
    dirs[:] = [d for d in dirs if d not in _EXCLUDE_DIRS]
    for fn in files:
        if fn in _EXCLUDE_FILES or fn.endswith(".zip"):
            continue
        src = os.path.join(root, fn)
        rel = os.path.relpath(src, _ARCHIVE_ROOT)
        dst = os.path.join(_STAGE, rel)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        try:
            shutil.copy2(src, dst)
        except Exception as e:
            print(f"WARN copy fail {rel}: {e}")

# ---------- Archive ----------

try:
    _arch = shutil.make_archive("/tmp/goa_rtx_run_results", "zip", root_dir=_STAGE)
except Exception as e:
    raise SystemExit(f"make_archive fail: {e}")

# ---------- Move + integrity verify ----------

shutil.move(_arch, _TARGET_ZIP)

try:
    with zipfile.ZipFile(_TARGET_ZIP) as z:
        _bad = z.testzip()
        if _bad:
            raise SystemExit(f"zip corrupt at {_bad}")
        _n = len(z.namelist())
    _sz = os.path.getsize(_TARGET_ZIP) / 1e6
    print(f"backup ready: {_n} entries, {_sz:.1f} MB")
    if _sz > 500:
        print(f"WARN zip {_sz:.0f} MB bada — unexpected content check karo")
except Exception as e:
    raise SystemExit(f"zip validation fail: {e}")

# ---------- Cleanup ----------

shutil.rmtree(_STAGE, ignore_errors=True)


## CODER-SOLVER CONFORMANCE — moved before THE RUN (FIX-V, audit round-4)
Ye cell runbook ke baad, FRAME PROBE se pehle chalta hai.
Purana position (backup ke baad) ab sirf audit-trail hai.
Template-conference cell pehle se THE RUN se pehle tha (audit
round-4 ka wala claim galat numbering par tha — adjudicated FALSE).


In [ ]:
# md5 register (v10.21.0) — Run-21 certified + coder-solver set
import hashlib, os
_FILES = [
    ("context_manager",  "18b8dfa5be82c8224671e0fa92ee8135"),
    ("inference_layer",  "8ec5288b371b0e15f3f4d8ed49b0fa46"),
    ("grid_text",        "bdbc329a990d82c19bf4b4e22b838bdd"),
    ("action_pipeline",  "826cf5f96000f615e57f1a04233102e9"),
    ("my_agent",         "1c1151af8c0dcf5931af02138c2bb590"),
]
for _n, _e in _FILES:
    _p = f"/kaggle/working/goa/agent/{_n}.py"
    _a = hashlib.md5(open(_p, "rb").read()).hexdigest() if os.path.exists(_p) else "MISSING"
    print(f"{_n:18} {'OK' if _a == _e else 'MISMATCH ' + _a}  (expected {_e[:8]})")
_CS_FILES = ["sandbox_v6", "replay_verifier", "cegis_taxonomy", "cegis_router",
             "adversarial_critic", "inference_layer_v1_6", "inference_layer_v1_6_additions",
             "coder_solver"]
for _n in _CS_FILES:
    _p = f"/kaggle/working/goa/agent/{_n}.py"
    print(f"{_n:30} {hashlib.md5(open(_p, 'rb').read()).hexdigest()[:12] if os.path.exists(_p) else 'MISSING'}")
print("Run-20 baseline md5: 6536295199ce0508750e4a4fda6a0f93")


In [ ]:
# md5 register source-of-truth (v10.21.0) — Run-21
# certified: CM v1.6 18b8dfa5 | IL v1.6 8ec5288b | GT bdbc329a | AP 826cf5f9 | my_agent v1.0.21 (Object Vision) 1c1151af
# coder-solver: sandbox_v6, replay_verifier, cegis_taxonomy, cegis_router, adversarial_critic,
#               additions f9bab0b0 | coder_solver b79ca7e8 | lora 9744896a
# schema: 97073721699cf92d53bb4909322ad212 | R-13 artifact 001ce125 | Run-20 baseline 6536295199ce0508750e4a4fda6a0f93